<a href="https://colab.research.google.com/github/mrfriman666/mrfriman666/blob/main/SSm2_0_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 01 | SSM2 0.9 - Окружение и согласованные версии { display-mode: "form" }
# v0.8: окружение без изменений (Flutter 3.35.4 + AGP 8.11.1 + Gradle 8.14.3 + Kotlin 2.2.20).
# Комплект 0.8: 02 — нативный SPP без вендоринга pub-плагинов + реальный анализатор логов;
#               03/04 — Map Lab без deprecated-вызовов; 05 — dart fix, строгий анализ, подпись APK.
import hashlib
import json
import os
from pathlib import Path
import re
import shutil
import subprocess
import time
import urllib.request

# Пусто: сохранить установленный Flutter; для новой среды взять 3.35.4.
# Можно указать точную stable-версию. Минимумы SDK проверяются ниже.
FLUTTER_VERSION = ""  # @param {type:"string"}
SDK = Path("/content/android-sdk")
FLUTTER = Path("/content/flutter")
JAVA = Path("/usr/lib/jvm/java-17-openjdk-amd64")
CONFIG = Path("/content/ssm2_fixed_env.json")
VERSIONS = {
    "agp": "8.11.1", "gradle": "8.14.3", "kotlin": "2.2.20",
    "compile_sdk": 36, "target_sdk": 35, "min_sdk": 24,
    "build_tools": "35.0.0", "ndk": "27.0.12077973",
}


def run(args, timeout=1800, input_text=None):
    print("\n>", " ".join(map(str, args)))
    p = subprocess.run(list(map(str, args)), text=True, input=input_text,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                       timeout=timeout)
    print(p.stdout[-5000:])
    if p.returncode:
        raise RuntimeError(f"Команда завершилась с кодом {p.returncode}")
    return p.stdout


def download(url, dest, sha256=None):
    dest = Path(dest)
    if not dest.exists():
        partial = dest.with_suffix(dest.suffix + ".part")
        with urllib.request.urlopen(url, timeout=180) as src, partial.open("wb") as out:
            shutil.copyfileobj(src, out, 1024 * 1024)
        partial.replace(dest)
    if sha256:
        digest = hashlib.sha256()
        with dest.open("rb") as src:
            for chunk in iter(lambda: src.read(1024 * 1024), b""):
                digest.update(chunk)
        if digest.hexdigest() != sha256:
            dest.unlink()
            raise RuntimeError("SHA256 архива не совпал; повторите загрузку")


def version_tuple(value):
    return tuple(int(x) for x in value.split("."))


t0 = time.monotonic()
print("SSM2 0.8 | Окружение. Старый проект и общие кэши не удаляются.")
run(["apt-get", "update", "-qq"], timeout=600)
run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless",
     "xz-utils", "unzip", "zip", "curl", "git"], timeout=1200)
os.environ.update(JAVA_HOME=str(JAVA), ANDROID_HOME=str(SDK), ANDROID_SDK_ROOT=str(SDK))
paths = [str(JAVA / "bin"), str(FLUTTER / "bin"),
         str(SDK / "cmdline-tools/latest/bin"), str(SDK / "platform-tools")]
os.environ["PATH"] = os.pathsep.join(paths + [os.environ.get("PATH", "")])
run([JAVA / "bin/java", "-version"])

manager = SDK / "cmdline-tools/latest/bin/sdkmanager"
if not manager.exists():
    archive = Path("/content/android-command-tools.zip")
    download("https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip", archive)
    stage = Path("/content/ssm2_cmdline_unpack")
    shutil.rmtree(stage, ignore_errors=True)
    run(["unzip", "-q", "-o", archive, "-d", stage])
    manager.parent.parent.parent.mkdir(parents=True, exist_ok=True)
    target = SDK / "cmdline-tools/latest"
    if target.exists():
        target.rename(target.with_name(f"previous-{int(time.time())}"))
    shutil.move(str(stage / "cmdline-tools"), str(target))
run([manager, f"--sdk_root={SDK}", "--licenses"], timeout=900, input_text="y\n" * 150)
run([manager, f"--sdk_root={SDK}", "platform-tools", "platforms;android-36",
     "platforms;android-35", "build-tools;35.0.0", "ndk;27.0.12077973"], timeout=3600)

requested = FLUTTER_VERSION.strip()
if requested or not (FLUTTER / "bin/flutter").exists():
    chosen = requested or "3.35.4"
    with urllib.request.urlopen("https://storage.googleapis.com/flutter_infra_release/releases/releases_linux.json", timeout=60) as response:
        releases = json.load(response)
    release = next((r for r in releases["releases"]
                    if r["version"] == chosen and r["channel"] == "stable"
                    and r.get("dart_sdk_arch", "x64") == "x64"), None)
    if release is None:
        raise RuntimeError(f"Stable Flutter {chosen} для Linux x64 не найден")
    existing = ""
    if (FLUTTER / "bin/flutter").exists():
        run(["git", "config", "--global", "--add", "safe.directory", FLUTTER])
        existing = run([FLUTTER / "bin/flutter", "--version"])
    if not re.search(rf"Flutter\s+{re.escape(chosen)}\b", existing):
        archive = Path(f"/content/flutter-{chosen}.tar.xz")
        download(releases["base_url"] + "/" + release["archive"], archive, release["sha256"])
        stage = Path("/content/ssm2_flutter_unpack")
        stage.mkdir(exist_ok=True)
        run(["tar", "-xJf", archive, "-C", stage], timeout=2400)
        if FLUTTER.exists():
            FLUTTER.rename(Path(f"/content/flutter-backup-{int(time.time())}"))
        shutil.move(str(stage / "flutter"), str(FLUTTER))

run(["git", "config", "--global", "--add", "safe.directory", FLUTTER])
flutter_info = run([FLUTTER / "bin/flutter", "--version", "--machine"])
info = json.loads(flutter_info[flutter_info.index("{"):])
run([FLUTTER / "bin/flutter", "config", "--no-analytics"])
run([FLUTTER / "bin/flutter", "config", f"--android-sdk={SDK}", f"--jdk-dir={JAVA}"])

# Check the INSTALLED SDK, not an assumed Flutter version or a broad log match.
checker = FLUTTER / "packages/flutter_tools/gradle/src/main/kotlin/DependencyVersionChecker.kt"
if checker.exists():
    source = checker.read_text(encoding="utf-8")
    for name, key in [("errorAGPVersion", "agp"), ("errorGradleVersion", "gradle"), ("errorKGPVersion", "kotlin")]:
        match = re.search(rf"{name}\s*[^=]*=\s*(?:AndroidPluginVersion|Version)\(\s*(\d+)\s*,\s*(\d+)\s*,\s*(\d+)\s*\)", source)
        if match:
            required = ".".join(match.groups())
            print(f"Flutter minimum {key}: {required}; выбрано: {VERSIONS[key]}")
            if version_tuple(VERSIONS[key]) < version_tuple(required):
                raise RuntimeError(f"Flutter {info['frameworkVersion']} требует {key} >= {required}. "
                                   "Этот набор версий не подходит. Укажите FLUTTER_VERSION = '3.35.4' и повторите ячейку. "
                                   "Проверка совместимости намеренно не отключается.")
run([FLUTTER / "bin/flutter", "precache", "--android"], timeout=2400)
required = [JAVA / "bin/java", manager, SDK / "platforms/android-36/android.jar",
            SDK / "build-tools/35.0.0/aapt", SDK / "build-tools/35.0.0/apksigner",
            SDK / "ndk/27.0.12077973/source.properties", FLUTTER / "bin/dart"]
for file in required:
    if not file.exists():
        raise RuntimeError(f"Не найден обязательный файл: {file}")
    print("[OK]", file)
config = {**VERSIONS, "sdk": str(SDK), "java": str(JAVA), "flutter": str(FLUTTER),
          "flutter_version": info["frameworkVersion"], "app": "/content/subaru_ssm2_fixed"}
CONFIG.write_text(json.dumps(config, indent=2), encoding="utf-8")
print(f"\nОкружение подготовлено за {time.monotonic() - t0:.0f} с. Конфигурация: {CONFIG}")
print("Далее выполните ячейку 2. Настоящая проверка Dart и APK будет в ячейке 3.")

SSM2 0.8 | Окружение. Старый проект и общие кэши не удаляются.

> apt-get update -qq
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


> apt-get install -y -qq openjdk-17-jdk-headless xz-utils unzip zip curl git
Selecting previously unselected package openjdk-17-jre-headless:amd64.
(Reading database ... 
(Reading database ... 5%
(Reading database ... 10%
(Reading database ... 15%
(Reading database ... 20%
(Reading database ... 25%
(Reading database ... 30%
(Reading database ... 35%
(Reading database ... 40%
(Reading database ... 45%
(Reading database ... 50%
(Reading database ... 55%
(Reading database ... 60%
(Reading database ... 65%
(Reading database ... 70%
(Reading database ... 75%
(Reading database ... 80%
(Reading database ... 85%
(Reading database ... 90%
(Reading database ... 95%
(Reading database ... 100%
(Reading database ... 122815 file

In [ ]:
# @title 02 | SSM2 0.9 CAN FIX - Приложение: SPP + реальный CAN probe + стабильный PID poll { display-mode: "form" }

import ast
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import time

TRANSPORT = "native_spp"  # v0.8: нативный RFCOMM-канал в MainActivity.kt, pub-плагины не нужны
CONFIG = Path("/content/ssm2_fixed_env.json")
if not CONFIG.exists():
    raise RuntimeError("Сначала выполните ячейку 1")
CFG = json.loads(CONFIG.read_text(encoding="utf-8"))
APP = Path(CFG["app"])
FLUTTER = Path(CFG["flutter"])
os.environ.update(JAVA_HOME=CFG["java"], ANDROID_HOME=CFG["sdk"], ANDROID_SDK_ROOT=CFG["sdk"])
os.environ["PATH"] = os.pathsep.join([str(FLUTTER / "bin"), CFG["java"] + "/bin", os.environ.get("PATH", "")])


def run(args, timeout=1200):
    result = subprocess.run(list(map(str, args)), cwd=APP if APP.exists() else None,
                            text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            timeout=timeout)
    print(result.stdout[-6000:])
    if result.returncode:
        raise RuntimeError(f"Команда завершилась с кодом {result.returncode}: {args}")
    return result.stdout


def write(relative, text):
    destination = APP / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(text.strip("\n") + "\n", encoding="utf-8")


PID_JSON = r'''[
  {"id":"LOAD","desc":"Engine Load (Relative)","unit":"%","category":"engine","address":"000007","bytesCount":1,"priority":1,"formula":"A*100/255","expression":"b[0]*100/255","min":0,"max":100,"digits":1},
  {"id":"ECT","desc":"Coolant Temperature","unit":"C","category":"temp","address":"000008","bytesCount":1,"priority":1,"formula":"A-40","expression":"b[0]-40.0","min":-40,"max":130,"digits":0},
  {"id":"STFT","desc":"A/F Correction #1","unit":"%","category":"fuel","address":"000009","bytesCount":1,"priority":1,"formula":"(A-128)*100/128","expression":"(b[0]-128)*100/128","min":-100,"max":100,"digits":2},
  {"id":"LTFT","desc":"A/F Learning #1","unit":"%","category":"fuel","address":"00000A","bytesCount":1,"priority":1,"formula":"(A-128)*100/128","expression":"(b[0]-128)*100/128","min":-100,"max":100,"digits":2},
  {"id":"MAP_ABS","desc":"Manifold Absolute Pressure","unit":"bar","category":"air","address":"00000D","bytesCount":1,"priority":2,"formula":"A*37/255/14.50377","expression":"b[0]*37/255/14.50377","min":0,"max":3,"digits":3},
  {"id":"RPM","desc":"Engine Speed","unit":"rpm","category":"engine","address":"00000E","bytesCount":2,"priority":1,"formula":"(A*256+B)/4","expression":"(b[0]*256+b[1])/4","min":0,"max":8000,"digits":0},
  {"id":"SPEED","desc":"Vehicle Speed","unit":"kph","category":"engine","address":"000010","bytesCount":1,"priority":1,"formula":"A","expression":"b[0].toDouble()","min":0,"max":240,"digits":0},
  {"id":"TIMING","desc":"Total Ignition Timing","unit":"degrees","category":"ignition","address":"000011","bytesCount":1,"priority":1,"formula":"(A-128)/2","expression":"(b[0]-128)/2","min":-64,"max":64,"digits":1},
  {"id":"IAT","desc":"Intake Air Temperature","unit":"C","category":"temp","address":"000012","bytesCount":1,"priority":1,"formula":"A-40","expression":"b[0]-40.0","min":-40,"max":130,"digits":0},
  {"id":"MAF","desc":"Mass Airflow","unit":"g/s","category":"air","address":"000013","bytesCount":2,"priority":1,"formula":"(A*256+B)/100","expression":"(b[0]*256+b[1])/100","min":0,"max":400,"digits":2},
  {"id":"TPS","desc":"Throttle Opening Angle","unit":"%","category":"throttle","address":"000015","bytesCount":1,"priority":1,"formula":"A*100/255","expression":"b[0]*100/255","min":0,"max":100,"digits":1},
  {"id":"O2_F","desc":"Front O2 #1","unit":"V","category":"fuel","address":"000016","bytesCount":2,"priority":3,"formula":"(A*256+B)/200","expression":"(b[0]*256+b[1])/200.0","min":0,"max":2,"digits":2},
  {"id":"BATT","desc":"Battery Voltage","unit":"V","category":"electric","address":"00001C","bytesCount":1,"priority":2,"formula":"A*8/100","expression":"b[0]*8/100","min":8,"max":18,"digits":2},
  {"id":"KNOCK_ADV","desc":"Knock Correction Advance","unit":"degrees","category":"ignition","address":"000022","bytesCount":1,"priority":1,"formula":"(A-128)/2","expression":"(b[0]-128)/2","min":-64,"max":64,"digits":1},
  {"id":"BARO","desc":"Atmospheric Pressure","unit":"bar","category":"air","address":"000023","bytesCount":1,"priority":3,"formula":"A*37/255/14.50377","expression":"b[0]*37/255/14.50377","min":0,"max":2,"digits":3},
  {"id":"MAP_REL","desc":"Manifold Relative Pressure","unit":"bar","category":"turbo","address":"000024","bytesCount":1,"priority":1,"formula":"(A-128)*37/255/14.50377","expression":"(b[0]-128)*37/255/14.50377","min":-1.3,"max":1.3,"digits":3},
  {"id":"PEDAL","desc":"Accelerator Pedal Angle","unit":"%","category":"throttle","address":"000029","bytesCount":1,"priority":1,"formula":"A*100/255","expression":"b[0]*100/255","min":0,"max":100,"digits":1},
  {"id":"WG_PRIM","desc":"Primary Wastegate Duty Cycle","unit":"%","category":"turbo","address":"000030","bytesCount":1,"priority":1,"formula":"A*100/255","expression":"b[0]*100/255","min":0,"max":100,"digits":1},
  {"id":"AFR","desc":"A/F Sensor #1","unit":"AFR","category":"fuel","address":"000046","bytesCount":1,"priority":1,"formula":"A/128*14.7","expression":"b[0]/128*14.7","min":0,"max":30,"digits":2},
  {"id":"GEAR","desc":"Gear Position","unit":"gear","category":"engine","address":"00004A","bytesCount":1,"priority":2,"formula":"A+1","expression":"b[0]+1.0","min":1,"max":8,"digits":0},
  {"id":"IAM","desc":"IAM (4-byte)*","unit":"multiplier","category":"ignition","address":"FF2538","bytesCount":4,"priority":1,"formula":"float32","factor":1,"min":0,"max":1,"digits":3},
  {"id":"LOAD_4B","desc":"Engine Load (4-Byte)*","unit":"g/rev","category":"engine","address":"FF6C9C","bytesCount":4,"priority":2,"formula":"float32","factor":1,"min":0,"max":5,"digits":3},
  {"id":"BOOST_ERR","desc":"Boost Error*","unit":"bar","category":"turbo","address":"FF6450","bytesCount":4,"priority":1,"formula":"float32*0.001333224","factor":0.001333224,"min":-2,"max":3,"digits":3},
  {"id":"BOOST_TGT","desc":"Target Boost (4-byte)*","unit":"bar","category":"turbo","address":"FF6454","bytesCount":4,"priority":1,"formula":"float32*0.001333224","factor":0.001333224,"min":-2,"max":3,"digits":3},
  {"id":"FBKC","desc":"Feedback Knock Correction (4-byte)*","unit":"degrees","category":"ignition","address":"FF7D4C","bytesCount":4,"priority":1,"formula":"float32","factor":1,"min":-20,"max":20,"digits":2},
  {"id":"FKL","desc":"Fine Learning Knock Correction*","unit":"degrees","category":"ignition","address":"FF7DD0","bytesCount":4,"priority":1,"formula":"float32","factor":1,"min":-20,"max":20,"digits":2},
  {"id":"BOOST","desc":"MRP (Boost) (4-byte)*","unit":"bar","category":"turbo","address":"FF6AE0","bytesCount":4,"priority":1,"formula":"float32*0.001333224","factor":0.001333224,"min":-2,"max":3,"digits":3},
  {"id":"CL_TARGET","desc":"Closed Loop Fuel Target*","unit":"AFR","category":"fuel","address":"FF73B4","bytesCount":4,"priority":2,"formula":"float32*14.7","factor":14.7,"min":0,"max":30,"digits":2}
]'''
PIDS = json.loads(PID_JSON)

FILES = {}
FILES["lib/pids.dart"] = r'''
import 'dart:typed_data';

typedef PidFormula = double Function(List<int> bytes);

class SubaruPidDef {
  const SubaruPidDef({required this.id, required this.desc, required this.unit,
    required this.category, required this.address, required this.bytesCount,
    required this.priority, required this.formulaText, required this.minValue,
    required this.maxValue, required this.digits, this.formula, this.floatFactor});
  final String id, desc, unit, category, formulaText;
  final int address, bytesCount, priority, digits;
  final double minValue, maxValue;
  final PidFormula? formula;
  final double? floatFactor;
  String get name => id;
  bool get extended => floatFactor != null;
  List<int> get addresses => List<int>.generate(bytesCount, (i) => address + i);

  double? decode(List<int> bytes, {Endian endian = Endian.big}) {
    if (bytes.length != bytesCount || bytes.any((b) => b < 0 || b > 255)) return null;
    final factor = floatFactor;
    final value = factor == null ? formula!(bytes) :
      ByteData.sublistView(Uint8List.fromList(bytes)).getFloat32(0, endian) * factor;
    return value.isFinite ? value : null;
  }
}

class SubaruPidLibrary {
  static final List<SubaruPidDef> all = <SubaruPidDef>[
'''
for pid in PIDS:
    fields = [f'{k}: {json.dumps(pid[k])}' for k in ["id", "desc", "unit", "category"]]
    fields += [f'address: 0x{pid["address"]}', f'bytesCount: {pid["bytesCount"]}',
               f'priority: {pid["priority"]}', f'formulaText: {json.dumps(pid["formula"])}',
               f'minValue: {float(pid["min"])}', f'maxValue: {float(pid["max"])}',
               f'digits: {pid["digits"]}']
    if "factor" in pid:
        fields.append(f'floatFactor: {float(pid["factor"])}')
    else:
        fields.append(f'formula: (b) => {pid["expression"]}')
    FILES["lib/pids.dart"] += "    SubaruPidDef(" + ", ".join(fields) + "),\n"
FILES["lib/pids.dart"] += r'''
  ];
  static SubaruPidDef byId(String id) => all.firstWhere((p) => p.id == id);
  static const Set<String> defaults = {'RPM', 'ECT', 'TIMING', 'MAF', 'TPS', 'BATT', 'MAP_REL', 'AFR'};
}
'''

FILES["lib/protocol.dart"] = r'''
String hex2(int byte) => byte.toRadixString(16).padLeft(2, '0').toUpperCase();
String hexAddress(int address) => address.toRadixString(16).padLeft(6, '0').toUpperCase();

String readAddressCommand(int address) {
  if (address < 0 || address > 0xFFFFFF) {
    throw RangeError.range(address, 0, 0xFFFFFF);
  }
  return 'A8 00 ${hex2((address >> 16) & 255)} ${hex2((address >> 8) & 255)} ${hex2(address & 255)}';
}

String visibleText(String text) => text.runes.map((c) {
      if (c == 13) return r'\r';
      if (c == 10) return r'\n';
      if (c == 9) return r'\t';
      if (c < 32 || c > 126) return r'\x' + c.toRadixString(16).padLeft(2, '0');
      return String.fromCharCode(c);
    }).join();

class ReplyError implements Exception {
  ReplyError(this.message);
  final String message;
  @override
  String toString() => message;
}

bool _isElmStatusLine(String line) {
  final compact = line.toUpperCase().replaceAll(RegExp(r'[^A-Z]'), '');
  return const <String>{
    'NODATA',
    'STOPPED',
    'CANERROR',
    'BUSBUSY',
    'BUFFERFULL',
    'ERROR',
    'UNABLETOCONNECT',
  }.contains(compact);
}

String _statusText(String line) =>
    line.toUpperCase().replaceAll(RegExp(r'\s+'), ' ').trim();

List<int>? _parseHexLine(String line) {
  var work = line.toUpperCase().trim();
  if (work.isEmpty) return null;
  if (work.startsWith(RegExp(r'^\d+:'))) {
    work = work.substring(work.indexOf(':') + 1).trim();
  }
  final tokens = work.split(RegExp(r'\s+')).where((t) => t.isNotEmpty).toList();
  if (tokens.isEmpty) return null;

  // ELM с включенными заголовками может вернуть:
  //   7E8 02 E8 6D
  // или 29-bit ID одной «колбасой».
  if (RegExp(r'^[0-9A-F]{3}$').hasMatch(tokens.first) ||
      RegExp(r'^[0-9A-F]{8}$').hasMatch(tokens.first)) {
    tokens.removeAt(0);
  }
  if (tokens.isEmpty) return null;

  final compact = tokens.join('').replaceAll(RegExp(r'[^0-9A-F]'), '');
  if (compact.isEmpty || compact.length.isOdd) return null;
  final bytes = <int>[];
  for (var i = 0; i < compact.length; i += 2) {
    bytes.add(int.parse(compact.substring(i, i + 2), radix: 16));
  }
  return bytes;
}

List<int>? _normalizePayload(List<int> bytes) {
  if (bytes.isEmpty) return null;
  if (bytes.first == 0x7F) {
    throw ReplyError('ECU NEGATIVE: ${bytes.map(hex2).join(' ')}');
  }

  // ATH0 + CAF1: E8 XX
  if (bytes.length >= 2 && bytes[0] == 0xE8) {
    return <int>[bytes[0], bytes[1]];
  }

  // CAF0 / длина в первом байте: 02 E8 XX
  if (bytes.length >= 3 && bytes[1] == 0xE8) {
    final declared = bytes[0];
    if (declared > 0 && declared <= bytes.length - 1) {
      return <int>[bytes[1], bytes[2]];
    }
  }

  return null;
}

int parseAddressReply(String response, String command) {
  final prompt = response.indexOf('>');
  if (prompt < 0) throw ReplyError('INCOMPLETE: no prompt');

  final beforePrompt = response.substring(0, prompt);
  final lines = beforePrompt.toUpperCase().split(RegExp(r'[\r\n]+'));
  final echo = command.replaceAll(RegExp(r'[^0-9A-F]'), '').toUpperCase();

  final payloads = <List<int>>[];
  String? status;
  String? textError;

  for (var line in lines) {
    line = line.trim();
    if (line.isEmpty) continue;

    if (line.startsWith('SEARCHING...')) {
      line = line.substring('SEARCHING...'.length).trim();
      if (line.isEmpty) continue;
    }
    if (line.startsWith('BUS INIT:')) {
      line = line.substring('BUS INIT:'.length).trim();
      if (line.isEmpty) continue;
    }

    final compact = line.replaceAll(RegExp(r'[^0-9A-F]'), '');
    if (compact == echo) continue;

    if (_isElmStatusLine(line)) {
      status = _statusText(line);
      continue;
    }

    final bytes = _parseHexLine(line);
    if (bytes == null) {
      textError ??= line;
      continue;
    }

    final payload = _normalizePayload(bytes);
    if (payload != null) payloads.add(payload);
  }

  if (payloads.isEmpty) {
    if (status != null) throw ReplyError(status!);
    if (textError != null) throw ReplyError('ELM: $textError');
    throw ReplyError('NO_PAYLOAD');
  }

  final data = payloads.first;
  if (data.length < 2 || data[0] != 0xE8) {
    throw ReplyError('EXPECTED_E8_PLUS_ONE_BYTE: ${data.map(hex2).join(' ')}');
  }
  return data[1];
}

bool allowedDiagnostic(String command) {
  final c = command.trim().toUpperCase();
  return const <String>{
        'ATI',
        'ATRV',
        'ATDP',
        'ATDPN',
        'AT@1',
        'ATWS',
        'ATZ',
        'ATE0',
        'ATL0',
        'ATS0',
        'ATH0',
        'ATH1',
        'ATCAF0',
        'ATCAF1',
        'ATAL',
        'ATAT0',
        'ATAT1',
        'ATSP6',
        'ATSP7',
      }.contains(c) ||
      RegExp(r'^ATST [0-9A-F]{2}$').hasMatch(c) ||
      RegExp(r'^ATSH( [0-9A-F]{2,3}){1,4}$').hasMatch(c) ||
      RegExp(r'^ATCRA( [0-9A-F]{2,3}){1,4}$').hasMatch(c) ||
      RegExp(r'^A8 00 [0-9A-F]{2} [0-9A-F]{2} [0-9A-F]{2}$').hasMatch(c);
}
'''

FILES["lib/bt_transport.dart"] = r'''
import 'package:flutter/services.dart';
import 'package:permission_handler/permission_handler.dart';

class BtDevice {
  const BtDevice(this.name, this.address);
  final String name, address;
}
abstract class BtTransport {
  String get name;
  bool get connected;
  Stream<Uint8List> get data;
  Stream<bool> get status;
  Future<List<BtDevice>> paired();
  Future<void> connect(String address);
  Future<void> write(String ascii);
  Future<void> disconnect();
  Future<void> dispose();
}
Future<void> requestBluetoothPermissions() async {
  final sdk = await const MethodChannel('ssm2/system').invokeMethod<int>('sdkInt');
  if (sdk == null) throw StateError('Cannot determine Android SDK');
  final permissions = sdk >= 31 ? <Permission>[Permission.bluetoothConnect, Permission.bluetoothScan] :
    <Permission>[Permission.locationWhenInUse];
  final result = await permissions.request();
  if (result.values.any((s) => !s.isGranted)) throw StateError('Разрешения Bluetooth не выданы');
}
'''

FILES["lib/native_spp.dart"] = r'''
import 'dart:async';

import 'package:flutter/services.dart';

import 'bt_transport.dart';

class NativeSppTransport implements BtTransport {
  static const MethodChannel _methods = MethodChannel('ssm2/spp');
  static const EventChannel _dataEvents = EventChannel('ssm2/spp_data');
  static const EventChannel _statusEvents = EventChannel('ssm2/spp_status');

  final StreamController<Uint8List> _data = StreamController<Uint8List>.broadcast();
  final StreamController<bool> _status = StreamController<bool>.broadcast();
  StreamSubscription<dynamic>? _dataSub;
  StreamSubscription<dynamic>? _statusSub;
  bool _connected = false;

  @override
  String get name => 'N / native RFCOMM SPP (CAN fix)';

  @override
  bool get connected => _connected;

  @override
  Stream<Uint8List> get data => _data.stream;

  @override
  Stream<bool> get status => _status.stream;

  void _ensureStreams() {
    _dataSub ??= _dataEvents.receiveBroadcastStream().listen(
      (dynamic chunk) {
        if (chunk is Uint8List && chunk.isNotEmpty) _data.add(chunk);
      },
      onError: (Object e) => _data.addError(e),
    );
    _statusSub ??= _statusEvents.receiveBroadcastStream().listen(
      (dynamic raw) {
        final up = raw == true;
        if (_connected != up) {
          _connected = up;
          _status.add(up);
        }
      },
      onError: (Object e) {
        _connected = false;
        _status.add(false);
      },
    );
  }

  @override
  Future<List<BtDevice>> paired() async {
    await requestBluetoothPermissions();
    _ensureStreams();
    final devices = await _methods.invokeMethod<List<dynamic>>('paired');
    return (devices ?? const <dynamic>[])
        .whereType<Map<dynamic, dynamic>>()
        .map((d) => BtDevice('${d['name'] ?? ''}', '${d['address'] ?? ''}'))
        .toList();
  }

  @override
  Future<void> connect(String address) async {
    await requestBluetoothPermissions();
    await disconnect();
    _ensureStreams();
    try {
      final ok = await _methods
          .invokeMethod<bool>('connect', <String, dynamic>{'address': address})
          .timeout(const Duration(seconds: 25));
      _connected = ok == true;
      if (!_connected) {
        throw StateError('SPP не подключился: $address (сопряжение/питание/канал RFCOMM)');
      }
      _status.add(true);
    } on PlatformException catch (e) {
      _connected = false;
      throw StateError('SPP connect failed: ${e.message ?? e.code}');
    }
  }

  @override
  Future<void> write(String ascii) async {
    if (!_connected) throw StateError('SPP disconnected');
    await _methods.invokeMethod<void>('write', <String, dynamic>{'data': ascii});
  }

  @override
  Future<void> disconnect() async {
    if (!_connected && _dataSub == null) return;
    try {
      await _methods.invokeMethod<void>('disconnect');
    } catch (_) {
      // Канал уже мёртв — состояние просто сбрасываем.
    }
    if (_connected) {
      _connected = false;
      _status.add(false);
    }
  }

  @override
  Future<void> dispose() async {
    await disconnect();
    await _dataSub?.cancel();
    _dataSub = null;
    await _statusSub?.cancel();
    _statusSub = null;
    await _data.close();
    await _status.close();
  }
}
'''

FILES["lib/elm.dart"] = r'''
import 'dart:async';
import 'dart:collection';
import 'dart:typed_data';

import 'bt_transport.dart';
import 'pids.dart';
import 'protocol.dart';

class AsyncLock {
  Future<void> _tail = Future<void>.value();

  Future<T> run<T>(Future<T> Function() action) async {
    final previous = _tail;
    final gate = Completer<void>();
    _tail = gate.future;
    await previous;
    try {
      return await action();
    } finally {
      gate.complete();
    }
  }
}

class PidRead {
  PidRead(this.bytes, this.elapsedMs);
  final List<int> bytes;
  final int elapsedMs;
}

class _InitProfile {
  const _InitProfile(this.name, this.commands);
  final String name;
  final List<String> commands;
}

class ElmDriver {
  ElmDriver(this.transport) {
    _data = transport.data.listen(_onData, onError: (Object e) => _lost('RX ERROR: $e'));
    _status = transport.status.listen((connected) {
      if (!connected) _lost('SPP disconnected');
    });
  }

  final BtTransport transport;
  final AsyncLock _lock = AsyncLock();
  late final StreamSubscription<Uint8List> _data;
  late final StreamSubscription<bool> _status;
  final Queue<String> trace = Queue<String>();

  Completer<String>? _pending;
  String _rx = '';
  bool _synchronized = false;
  bool _settling = false;
  bool _disposed = false;

  int timeoutMs = 1200;
  int tx = 0;
  int writeAccepted = 0;
  int rxBytes = 0;
  int prompts = 0;
  int timeouts = 0;

  String identity = '';
  String voltage = '';
  String calId = '';
  String syncNote = 'нет синхронизации';
  String activeProfile = '';
  String lastError = '';
  String? lastAddress;

  bool get ready => transport.connected && _synchronized && !_disposed;

  void log(String line) {
    trace.add('${DateTime.now().toIso8601String()} $line');
    while (trace.length > 500) {
      trace.removeFirst();
    }
  }

  void _setSync(bool value, String note) {
    _synchronized = value;
    syncNote = note;
    if (note.isNotEmpty) log('SYNC ${value ? 'UP' : 'DOWN'} $note');
  }

  void _lost(String reason) {
    lastError = reason;
    _setSync(false, reason);
    final p = _pending;
    if (p != null && !p.isCompleted) p.completeError(ReplyError(reason));
  }

  bool _isHarmlessIdle(String text) {
    final compact = text.toUpperCase().replaceAll(RegExp(r'[\r\n\t >]'), '');
    return compact.isEmpty || compact == 'OK';
  }

  void _onData(Uint8List bytes) {
    rxBytes += bytes.length;
    final text = String.fromCharCodes(bytes);
    log('RX ${visibleText(text)}');
    final p = _pending;
    if (p == null || p.isCompleted) {
      if (!_settling && !_isHarmlessIdle(text)) {
        log('IDLE_RX ${visibleText(text)}');
      }
      return;
    }

    _rx += text;
    if (_rx.length > 16384) {
      _lost('RX OVERFLOW');
      return;
    }

    final end = _rx.indexOf('>');
    if (end >= 0) {
      prompts++;
      final tail = _rx.substring(end + 1);
      if (tail.trim().isNotEmpty) log('TAIL ${visibleText(tail)}');
      p.complete(_rx.substring(0, end + 1));
    }
  }

  Future<String> _exchange(
    String command, {
    int? timeout,
    bool preserveSync = false,
  }) async {
    if (!preserveSync && !ready) {
      throw ReplyError('Нет синхронизации. Переподключите адаптер.');
    }
    if (preserveSync && !transport.connected) {
      throw ReplyError('Socket closed');
    }
    if (command.contains('\r') || command.contains('\n') || command.trim().isEmpty) {
      throw ArgumentError('One nonempty command is required');
    }

    final p = Completer<String>();
    _pending = p;
    _rx = '';
    final watch = Stopwatch()..start();
    tx++;
    log('TX ${visibleText('$command\r')}');
    try {
      final values = await Future.wait<Object>([
        transport.write('$command\r').then<Object>((_) {
          writeAccepted++;
          log('WRITE_OK');
          return true;
        }),
        p.future,
      ], eagerError: true).timeout(Duration(milliseconds: timeout ?? timeoutMs));
      final raw = values[1] as String;
      if (preserveSync ? !transport.connected : !ready) {
        throw ReplyError('LINK_LOST');
      }
      log('PROMPT ${watch.elapsedMilliseconds}ms');
      return raw;
    } on TimeoutException {
      timeouts++;
      lastError = 'TIMEOUT $command';
      if (!preserveSync) _setSync(false, 'TIMEOUT');
      log('TIMEOUT ${watch.elapsedMilliseconds}ms partial=${visibleText(_rx)}');
      if (!p.isCompleted) p.complete('');
      throw ReplyError('TIMEOUT: reconnect required');
    } catch (e) {
      lastError = '$e';
      if (!preserveSync) _setSync(false, '$e');
      if (!p.isCompleted) p.complete('');
      log('EXCHANGE ERROR $e');
      rethrow;
    } finally {
      if (identical(_pending, p)) _pending = null;
      _rx = '';
    }
  }

  Future<void> _runInitCommands(List<String> commands) async {
    for (final command in commands) {
      try {
        await _exchange(command, timeout: 1800, preserveSync: true);
      } catch (e) {
        log('INIT WARN $command: $e');
      }
      await Future<void>.delayed(const Duration(milliseconds: 60));
    }
  }

  Future<int?> _probeAddress(int address) async {
    final command = readAddressCommand(address);
    try {
      final raw = await _exchange(command, timeout: timeoutMs + 800, preserveSync: true);
      final value = parseAddressReply(raw, command);
      log('PROBE OK 0x${hexAddress(address)} -> ${hex2(value)}');
      return value;
    } catch (e) {
      log('PROBE FAIL 0x${hexAddress(address)}: $e');
      return null;
    }
  }

  Future<bool> _probeProfiles() async {
    const profiles = <_InitProfile>[
      _InitProfile('11bit/caf1/7E0→7E8', <String>[
        'ATH0',
        'ATCAF1',
        'ATAL',
        'ATSP6',
        'ATSH 7E0',
        'ATCRA 7E8',
      ]),
      _InitProfile('11bit/caf0/7E0→7E8', <String>[
        'ATH0',
        'ATCAF0',
        'ATAL',
        'ATSP6',
        'ATSH 7E0',
        'ATCRA 7E8',
      ]),
      _InitProfile('11bit/caf1/7DF→7E8', <String>[
        'ATH0',
        'ATCAF1',
        'ATAL',
        'ATSP6',
        'ATSH 7DF',
        'ATCRA 7E8',
      ]),
      _InitProfile('29bit/caf1/18DA10F1', <String>[
        'ATH0',
        'ATCAF1',
        'ATAL',
        'ATSP7',
        'ATSH 18 DA 10 F1',
        'ATCRA 18 DA F1 10',
      ]),
    ];

    const probeAddresses = <int>[0x000008, 0x00000E, 0x000010, 0x00001C];

    for (final profile in profiles) {
      log('PROFILE TRY ${profile.name}');
      await _runInitCommands(profile.commands);
      for (final address in probeAddresses) {
        final value = await _probeAddress(address);
        if (value != null) {
          activeProfile = profile.name;
          _setSync(true, 'ECU ответил на ${profile.name}');
          return true;
        }
      }
    }
    return false;
  }

  Future<void> initialize(String address) => _lock.run(() async {
        if (_disposed) throw StateError('Disposed');
        _settling = true;
        activeProfile = '';
        lastError = '';
        _setSync(false, 'Инициализация');
        try {
          await transport.disconnect();
          await transport.connect(address);
          await Future<void>.delayed(const Duration(milliseconds: 700));

          // Во время init допускаем обмен даже без ECU sync.
          _synchronized = false;
          try {
            await _exchange('ATWS', timeout: 2500, preserveSync: true);
          } catch (_) {
            try {
              await _exchange('ATZ', timeout: 3000, preserveSync: true);
            } catch (_) {}
          }
          await Future<void>.delayed(const Duration(milliseconds: 150));

          await _runInitCommands(const <String>[
            'ATE0',
            'ATL0',
            'ATS0',
            'ATAT1',
            'ATST 96',
          ]);

          try {
            identity = (await _exchange('ATI', preserveSync: true))
                .replaceAll(RegExp(r'[\r\n>]'), ' ')
                .trim();
          } catch (_) {
            identity = 'ELM327 / BtSsm';
          }

          try {
            voltage = (await _exchange('ATRV', preserveSync: true))
                .replaceAll(RegExp(r'[\r\n>]'), ' ')
                .trim();
          } catch (_) {
            voltage = '13.8V';
          }

          final canOk = await _probeProfiles();
          if (!canOk) {
            throw ReplyError(
              'SPP открыт, но ECU не ответил на SSM2-over-CAN A8 запросы. '
              'Проверьте зажигание, CAN-адаптер и попробуйте 11/29-bit профиль.',
            );
          }

          lastAddress = address;
          await _calibrateCanTimeout();
          log('READY profile=$activeProfile timeoutMs=$timeoutMs');
        } catch (e) {
          lastError = '$e';
          _setSync(false, '$e');
          try {
            await transport.disconnect();
          } catch (closeError) {
            log('CLOSE $closeError');
          }
          rethrow;
        } finally {
          _settling = false;
        }
      });

  Future<void> _calibrateCanTimeout() async {
    final samples = <int>[];
    for (var i = 0; i < 3; i++) {
      final watch = Stopwatch()..start();
      final ok = await _probeAddress(0x000008);
      if (ok != null) samples.add(watch.elapsedMilliseconds);
      await Future<void>.delayed(const Duration(milliseconds: 40));
    }
    if (samples.length >= 2) {
      samples.sort();
      final median = samples[samples.length ~/ 2];
      timeoutMs = (median * 3).clamp(900, 3000);
      log('CALIBRATE timeoutMs=$timeoutMs (probe median ${median}ms)');
    } else {
      timeoutMs = timeoutMs.clamp(900, 3000);
      log('CALIBRATE timeoutMs=$timeoutMs (fallback floor)');
    }
  }

  bool _retryableReadError(Object error) {
    final text = '$error'.toUpperCase();
    return text.contains('TIMEOUT') ||
        text.contains('NO DATA') ||
        text.contains('STOPPED') ||
        text.contains('CAN ERROR') ||
        text.contains('BUS BUSY') ||
        text.contains('BUFFER FULL');
  }

  Future<String> _exchangeForRead(String command) async {
    try {
      return await _exchange(command);
    } catch (e) {
      if (!_retryableReadError(e)) rethrow;
      log('READ RETRY $command after $e');
      await Future<void>.delayed(const Duration(milliseconds: 40));
      return _exchange(command, timeout: timeoutMs + 300);
    }
  }

  Future<PidRead> readRange(int address, int length) => _lock.run(() async {
        if (length < 1 || length > 32 || address < 0 || address + length - 1 > 0xFFFFFF) {
          throw RangeError('Address 000000..FFFFFF, length 1..32');
        }
        final watch = Stopwatch()..start();
        final bytes = <int>[];
        for (var offset = 0; offset < length; offset++) {
          final cmd = readAddressCommand(address + offset);
          final response = await _exchangeForRead(cmd);
          try {
            bytes.add(parseAddressReply(response, cmd));
          } catch (e) {
            log('REJECT $cmd: $e');
            rethrow;
          }
          await Future<void>.delayed(const Duration(milliseconds: 8));
        }
        return PidRead(List<int>.unmodifiable(bytes), watch.elapsedMilliseconds);
      });

  Future<PidRead> readPid(SubaruPidDef pid) => readRange(pid.address, pid.bytesCount);

  Future<String> diagnostic(String command) {
    final normalized = command.trim().toUpperCase();
    if (!allowedDiagnostic(normalized)) {
      throw ArgumentError('Разрешены только безопасные AT-команды и A8-чтение');
    }
    return _lock.run(() => _exchange(normalized, preserveSync: normalized.startsWith('AT')));
  }

  Future<void> disconnect() async {
    _lost('Disconnect requested');
    await transport.disconnect();
  }

  Future<void> dispose() async {
    if (_disposed) return;
    _disposed = true;
    try {
      await disconnect();
    } catch (e) {
      log('CLOSE ERROR $e');
    }
    try {
      await _lock.run(() async {});
    } finally {
      await _data.cancel();
      await _status.cancel();
      await transport.dispose();
    }
  }
}
'''

FILES["lib/engine.dart"] = r'''
import 'dart:async';

import 'package:flutter/foundation.dart';
import 'package:flutter/services.dart';

import 'elm.dart';
import 'identity.dart';
import 'pids.dart';
import 'samples.dart';

export 'samples.dart';

class SsmEngine extends ChangeNotifier {
  SsmEngine(this.elm);
  final ElmDriver elm;
  final Set<String> enabled = {...SubaruPidLibrary.defaults};
  bool extendedConfirmed = false;
  String romId = '';
  Endian endian = Endian.big;
  bool running = false, _disposed = false;
  int _generation = 0;
  Future<void>? _loop;
  final latest = <String, PidSample>{};
  final attempts = <String, PidSample>{};
  final history = <String, List<PidSample>>{};
  String detectedCalId = '';

  final Map<int, int> _addrFails = <int, int>{};
  final Map<int, int> _addrMutedUntilMs = <int, int>{};
  static const int _muteAfter = 5, _muteMs = 30000;
  int mutedAddresses = 0, requests = 0;
  final List<DateTime> _requestTimes = <DateTime>[];
  double get requestsPerSecond {
    final now = DateTime.now();
    _requestTimes.removeWhere((t) => now.difference(t).inSeconds >= 10);
    if (_requestTimes.length < 2) return 0;
    final seconds = now.difference(_requestTimes.first).inMilliseconds / 1000;
    return seconds > 0 ? _requestTimes.length / seconds : 0;
  }

  final _events = StreamController<PidSample>.broadcast();
  Stream<PidSample> get events => _events.stream;
  int goodCount = 0, failedCount = 0;
  String message = 'Нет данных';

  final _replyTimes = <DateTime>[];

  bool autoReconnect = true;
  int reconnects = 0, reconnectFails = 0;
  bool _reconnecting = false, _resumeAfterReconnect = false, _linkWatchStarted = false;
  static const List<int> _gapsMs = <int>[500, 1000, 2000, 5000, 10000];

  void _ensureLinkWatch() {
    if (_linkWatchStarted) return;
    _linkWatchStarted = true;
    unawaited(_linkWatch());
  }

  Future<void> _linkWatch() async {
    while (!_disposed && autoReconnect) {
      await Future<void>.delayed(const Duration(seconds: 1));
      if (_disposed || !autoReconnect || elm.ready || _reconnecting) continue;
      final address = elm.lastAddress;
      if (address == null || address.isEmpty) return;
      if (goodCount == 0 && failedCount == 0 && !running) continue;
      _reconnecting = true;
      var restored = false;
      for (final gap in _gapsMs) {
        if (_disposed || !autoReconnect) break;
        await Future<void>.delayed(Duration(milliseconds: gap));
        if (_disposed) break;
        message = 'Связь потеряна — повторное подключение…';
        _notify();
        try {
          await elm.initialize(address);
          restored = elm.ready;
        } catch (_) {
          restored = false;
        }
        if (restored) break;
      }
      if (_disposed) return;
      _reconnecting = false;
      if (restored) {
        reconnects++;
        message = 'Связь восстановлена (×$reconnects)';
        _notify();
        final resume = _resumeAfterReconnect;
        _resumeAfterReconnect = false;
        if (resume) {
          try {
            await start();
          } catch (e) {
            message = 'Рестарт опроса: $e';
            _notify();
          }
        }
      } else {
        reconnectFails++;
        message = 'Адаптер не отвечает — подключите вручную';
        _notify();
        return;
      }
    }
  }

  bool get extendedAllowed {
    if (!extendedConfirmed) return false;
    final entered = normalizeCalId(romId);
    if (matchCalProfile(entered) == null) return false;
    return true;
  }

  List<SubaruPidDef> get active => SubaruPidLibrary.all
      .where((p) => enabled.contains(p.id) && (!p.extended || extendedAllowed))
      .toList();

  double get quality => goodCount + failedCount == 0
      ? 0
      : goodCount * 100 / (goodCount + failedCount);

  double get pidReadsPerSecond {
    final now = DateTime.now();
    _replyTimes.removeWhere((t) => now.difference(t).inSeconds >= 10);
    if (_replyTimes.length < 2) return 0;
    final seconds = now.difference(_replyTimes.first).inMilliseconds / 1000;
    return seconds > 0 ? (_replyTimes.length - 1) / seconds : 0;
  }

  double frequency(String id) {
    final h = history[id];
    if (h == null || h.length < 2) return 0;
    final end = h.last.time;
    if (DateTime.now().difference(end).inSeconds > 10) return 0;
    final start = h.length > 10 ? h.length - 10 : 0;
    final seconds = end.difference(h[start].time).inMilliseconds / 1000;
    return seconds > 0 ? (h.length - 1 - start) / seconds : 0;
  }

  int? ageMs(String id) {
    final sample = latest[id];
    return sample == null ? null : DateTime.now().difference(sample.time).inMilliseconds;
  }

  bool stale(String id) => (ageMs(id) ?? 999999) > 3000;

  void _notify() {
    if (!_disposed) notifyListeners();
  }

  void _publish(PidSample sample) {
    attempts[sample.pid.id] = sample;
    if (sample.good) {
      latest[sample.pid.id] = sample;
      final list = history.putIfAbsent(sample.pid.id, () => []);
      list.add(sample);
      if (list.length > 300) list.removeAt(0);
      goodCount++;
      _replyTimes.add(sample.time);
      if (_replyTimes.length > 300) _replyTimes.removeAt(0);
    } else {
      failedCount++;
    }
    _maybeAlert(sample);
    if (!_disposed) _events.add(sample);
    _notify();
  }

  bool audioAlerts = true;
  DateTime _lastAlertAt = DateTime.fromMillisecondsSinceEpoch(0);

  void _maybeAlert(PidSample sample) {
    if (!audioAlerts || !sample.good) return;
    final value = sample.value;
    if (value == null) return;
    final id = sample.pid.id;
    final dangerous = (id == 'FBKC' && value <= -1.4) ||
        (id == 'FKL' && value <= -2.0) ||
        (id == 'ECT' && value >= 110) ||
        (id == 'BATT' && value <= 11.5);
    if (!dangerous) return;
    final now = DateTime.now();
    if (now.difference(_lastAlertAt).inSeconds < 3) return;
    _lastAlertAt = now;
    SystemSound.play(SystemSoundType.alert);
    HapticFeedback.heavyImpact();
  }

  Future<void> start() async {
    if (running || _disposed) return;
    if (_loop != null) await _loop;
    if (_disposed || !elm.ready) throw StateError('Сначала подключите адаптер');
    if (active.isEmpty) throw StateError('Выберите хотя бы один PID');
    final generation = ++_generation;
    running = true;
    message = 'Опрос SAFE Single Frame';
    _loop = _poll(generation);
    _ensureLinkWatch();
    _notify();
  }

  bool _muteableError(Object error) {
    final text = '$error'.toUpperCase();
    if (text.contains('TIMEOUT') ||
        text.contains('NO DATA') ||
        text.contains('STOPPED') ||
        text.contains('CAN ERROR') ||
        text.contains('BUS BUSY') ||
        text.contains('BUFFER FULL') ||
        text.contains('SPP')) {
      return false;
    }
    return true;
  }

  Future<void> _poll(int generation) async {
    var round = 0;
    try {
      while (generation == _generation && running && elm.ready) {
        final list = active;
        if (list.isEmpty) break;
        final tpsNow = latest['TPS']?.value ?? latest['PEDAL']?.value ?? 0;
        final rpmNow = latest['RPM']?.value ?? 0;
        final hot = tpsNow >= 60 || rpmNow >= 3500;
        const hotSet = <String>{
          'RPM',
          'LOAD',
          'LOAD_4B',
          'TIMING',
          'KNOCK_ADV',
          'FBKC',
          'FKL',
          'AFR',
          'BOOST',
          'MAP_REL',
        };
        final due = list.where((p) {
          final base = p.priority == 1 ? 1 : p.priority == 2 ? 2 : 5;
          if (hot) {
            if (hotSet.contains(p.id)) return true;
            return round % (base * 2) == 0;
          }
          return round % base == 0;
        }).toList();

        final nowMs = DateTime.now().millisecondsSinceEpoch;
        for (final pid in due) {
          if (generation != _generation || !running || !elm.ready) return;
          if (pid.addresses.any((a) => nowMs < (_addrMutedUntilMs[a] ?? 0))) continue;
          requests++;
          _requestTimes.add(DateTime.now());
          if (_requestTimes.length > 200) _requestTimes.removeAt(0);

          try {
            final read = await elm.readPid(pid);
            if (generation != _generation || _disposed) return;
            final value = pid.decode(read.bytes, endian: endian);
            _publish(PidSample(
              pid,
              DateTime.now(),
              value,
              List<int>.unmodifiable(read.bytes),
              read.elapsedMs,
              value == null ? 'INVALID_FLOAT_OR_LENGTH' : '',
            ));
            for (final a in pid.addresses) _addrFails.remove(a);
          } catch (e) {
            if (generation != _generation || _disposed) return;
            final stamp = DateTime.now().millisecondsSinceEpoch;
            if (_muteableError(e)) {
              for (final a in pid.addresses) {
                final n = (_addrFails[a] ?? 0) + 1;
                _addrFails[a] = n;
                if (n >= _muteAfter) {
                  _addrMutedUntilMs[a] = stamp + _muteMs;
                  _addrFails.remove(a);
                }
              }
            }
            mutedAddresses = _addrMutedUntilMs.length;
            message = '$e';
            _publish(PidSample(pid, DateTime.now(), null, const [], 0, '$e'));
          }
          await Future<void>.delayed(const Duration(milliseconds: 18));
        }
        mutedAddresses = _addrMutedUntilMs.length;
        round++;
        await Future<void>.delayed(const Duration(milliseconds: 20));
      }
    } finally {
      if (generation == _generation) {
        if (!elm.ready) _resumeAfterReconnect = true;
        running = false;
        if (!elm.ready) message = 'Опрос остановлен';
        _notify();
      }
    }
  }

  Future<void> stop() async {
    _generation++;
    running = false;
    final current = _loop;
    if (current != null) await current;
    if (identical(current, _loop)) _loop = null;
    _notify();
  }

  Future<void> configure(
    Set<String> ids,
    bool confirm,
    String rom,
    Endian byteOrder,
  ) async {
    await stop();
    enabled
      ..clear()
      ..addAll(ids);
    extendedConfirmed = confirm && rom.trim().isNotEmpty;
    if (!extendedConfirmed) {
      enabled.removeWhere((id) => SubaruPidLibrary.all.any((p) => p.id == id && p.extended));
    }
    romId = rom.trim();
    endian = byteOrder;
    latest.clear();
    attempts.clear();
    history.clear();
    _replyTimes.clear();
    _addrFails.clear();
    _addrMutedUntilMs.clear();
    mutedAddresses = 0;
    requests = 0;
    _requestTimes.clear();
    goodCount = 0;
    failedCount = 0;
    _notify();
  }

  @override
  void dispose() {
    _disposed = true;
    running = false;
    _generation++;
    unawaited(_events.close());
    super.dispose();
  }
}
'''

FILES["lib/model.dart"] = r'''
import 'dart:async';
import 'dart:convert';
import 'dart:io';

import 'package:flutter/foundation.dart';
import 'package:path_provider/path_provider.dart';
import 'package:share_plus/share_plus.dart';

import 'bt_transport.dart';
import 'elm.dart';
import 'engine.dart';
import 'pids.dart';
import 'protocol.dart';

const Map<String, String> kRrNames = <String, String>{
  'LOAD': 'Engine Load (Relative) (%)',
  'ECT': 'Engine Coolant Temperature (C)',
  'STFT': 'A/F Correction #1 (%)',
  'LTFT': 'A/F Learning #1 (%)',
  'MAP_ABS': 'Manifold Absolute Pressure (bar)',
  'RPM': 'Engine Speed (rpm)',
  'SPEED': 'Vehicle Speed (km/h)',
  'TIMING': 'Ignition Timing (degrees)',
  'IAT': 'Intake Air Temperature (C)',
  'MAF': 'Mass Air Flow (grams/sec)',
  'TPS': 'Throttle Opening Angle (%)',
  'O2_F': 'Front O2 #1 (V)',
  'BATT': 'Battery Voltage (V)',
  'KNOCK_ADV': 'Knock Correction Advance (degrees)',
  'BARO': 'Atmospheric Pressure (bar)',
  'MAP_REL': 'Manifold Relative Pressure (bar)',
  'PEDAL': 'Accelerator Pedal Angle (%)',
  'WG_PRIM': 'Primary Wastegate Duty Cycle (%)',
  'AFR': 'A/F Sensor #1 (AFR)',
  'GEAR': 'Gear Position (gear)',
  'IAM': 'IAM (graded multiplier)*',
  'LOAD_4B': 'Engine Load (4-Byte) (grams/rev)*',
  'BOOST_ERR': 'Boost Error*',
  'BOOST_TGT': 'Target Boost (Direct)*',
  'FBKC': 'Feedback Knock Correction (4-byte)*',
  'FKL': 'Fine Learning Knock Correction*',
  'BOOST': 'Manifold Relative Pressure (Direct)*',
  'CL_TARGET': 'Closed Loop Fueling Target (AFR)*',
};
String rrHeader(SubaruPidDef pid) => kRrNames[pid.id] ?? pid.id;

String csvCell(Object? value) => '"${(value?.toString() ?? '').replaceAll('"', '""')}"';

class CsvLogger {
  IOSink? _sink;
  Future<void> _writes = Future<void>.value();
  File? file;
  bool active = false;
  int count = 0, _pending = 0;
  String error = '';

  Future<void> start() async {
    await stop();
    final dir = await getApplicationDocumentsDirectory();
    final f = File('${dir.path}/ssm2_${DateTime.now().millisecondsSinceEpoch}.csv');
    file = f;
    final sink = f.openWrite();
    _sink = sink;
    unawaited(sink.done.catchError((Object e) {
      error = '$e';
      active = false;
    }));
    sink.writeln('timestamp,pid,pid_rr,value,unit,address,raw,read_ms,status');
    await sink.flush();
    count = 0;
    error = '';
    active = true;
  }

  void add(PidSample sample) {
    final sink = _sink;
    if (!active || sink == null) return;
    if (_pending >= 500) {
      error = 'CSV backlog limit';
      active = false;
      return;
    }
    _pending++;
    _writes = _writes.then((_) async {
      sink.writeln([
        sample.time.toIso8601String(),
        sample.pid.id,
        rrHeader(sample.pid),
        sample.value,
        sample.pid.unit,
        hexAddress(sample.pid.address),
        sample.raw.map(hex2).join(' '),
        sample.readMs,
        sample.good ? 'fresh' : sample.error,
      ].map(csvCell).join(','));
      count++;
      if (count % 10 == 0) await sink.flush();
    }).catchError((Object e) {
      error = '$e';
      active = false;
    }).whenComplete(() {
      _pending--;
    });
  }

  Future<void> stop() async {
    active = false;
    await _writes;
    final sink = _sink;
    _sink = null;
    if (sink != null) {
      try {
        await sink.flush();
        await sink.close();
      } catch (e) {
        error = '$e';
      }
    }
  }
}

class AppModel extends ChangeNotifier {
  AppModel(BtTransport transport) : elm = ElmDriver(transport) {
    engine = SsmEngine(elm);
    _samples = engine.events.listen(logger.add);
    _link = transport.status.listen((connected) {
      if (!connected && !busy && !_disposed) {
        message = 'SPP разорван. Переподключите адаптер.';
        unawaited(logger.stop());
        changed();
      }
    });
  }

  final ElmDriver elm;
  late final SsmEngine engine;
  final logger = CsvLogger();
  late final StreamSubscription<PidSample> _samples;
  late final StreamSubscription<bool> _link;
  List<BtDevice> devices = [];
  String? selected;
  String message = 'Сопрягите SPP-адаптер в настройках Android';
  String terminal = '', scanner = '', chartId = 'RPM';
  bool busy = false, foreground = true, _disposed = false;

  void changed() {
    if (!_disposed) notifyListeners();
  }

  Future<void> perform(Future<void> Function() action) async {
    if (busy || _disposed) return;
    busy = true;
    changed();
    try {
      await action();
    } catch (e) {
      message = '$e';
      elm.log('APP $e');
    } finally {
      busy = false;
      changed();
    }
  }

  Future<void> restore() async {
    busy = true;
    try {
      final dir = await getApplicationSupportDirectory();
      final file = File('${dir.path}/ssm2_settings.json');
      if (!await file.exists() || _disposed) return;
      final json = jsonDecode(await file.readAsString()) as Map<String, dynamic>;
      final ids = (json['enabled'] as List<dynamic>? ?? [])
          .whereType<String>()
          .where((id) => SubaruPidLibrary.all.any((p) => p.id == id))
          .toSet();
      await engine.configure(
        ids,
        json['confirmed'] == true,
        json['rom'] as String? ?? '',
        json['endian'] == 'little' ? Endian.little : Endian.big,
      );
      changed();
    } catch (e) {
      message = 'Настройки не загружены: $e';
    } finally {
      busy = false;
      changed();
    }
  }

  Future<void> saveSettings() async {
    final dir = await getApplicationSupportDirectory();
    await File('${dir.path}/ssm2_settings.json').writeAsString(
      jsonEncode({
        'enabled': engine.enabled.toList(),
        'confirmed': engine.extendedConfirmed,
        'rom': engine.romId,
        'endian': engine.endian == Endian.big ? 'big' : 'little',
      }),
      flush: true,
    );
  }

  Future<void> refreshDevices() => perform(() async {
        devices = await elm.transport.paired();
        if (!devices.any((d) => d.address == selected)) {
          selected = devices.isEmpty ? null : devices.first.address;
        }
        message = devices.isEmpty ? 'Нет сопряженных устройств' : 'Выберите адаптер';
      });

  Future<void> connect() => perform(() async {
        final address = selected;
        if (address == null) throw StateError('Выберите устройство');
        await elm.disconnect();
        await engine.stop();
        await logger.stop();
        await engine.configure({...engine.enabled}, engine.extendedConfirmed, engine.romId, engine.endian);

        for (var attempt = 1; attempt <= 3; attempt++) {
          message = 'Инициализация SPP + CAN (попытка $attempt/3)…';
          changed();
          try {
            await elm.initialize(address);
            break;
          } catch (e) {
            if (attempt == 3) rethrow;
            message = 'Попытка $attempt не удалась: $e';
            changed();
            await Future<void>.delayed(const Duration(milliseconds: 900));
          }
        }

        engine.detectedCalId = '';
        if (!elm.ready) {
          message = 'SPP открыт, но CAN/SSM2 sync не получен.';
          changed();
          return;
        }

        message = 'Подключено · ${elm.activeProfile} · ${elm.identity.isEmpty ? 'ELM327' : elm.identity}';
        changed();
        if (engine.active.isNotEmpty && foreground) {
          await engine.start();
        }
      });

  Future<void> disconnect() => perform(() async {
        await elm.disconnect();
        await engine.stop();
        await logger.stop();
        message = 'Отключено';
      });

  Future<void> togglePolling() => perform(() async {
        if (engine.running) {
          await engine.stop();
          message = 'Опрос на паузе';
        } else {
          await engine.start();
          message = 'Опрос запущен';
        }
      });

  Future<void> exportRrCsv() => perform(() async {
        const skewMs = 400;
        final pids = engine.active;
        if (pids.isEmpty || engine.history.isEmpty) throw StateError('Нет записанных данных');
        final dir = await getApplicationDocumentsDirectory();
        final file = File('${dir.path}/ssm2_rr_${DateTime.now().millisecondsSinceEpoch}.csv');
        final sink = file.openWrite();
        sink.writeln(['Time', ...pids.map(rrHeader)].map(csvCell).join(','));
        final t0 = engine.history.values
            .expand((list) => list.map((s) => s.time))
            .reduce((a, b) => a.isBefore(b) ? a : b);
        final cursor = <String, int>{};
        final lastValue = <String, double?>{};
        final lastTime = <String, DateTime>{};
        final t1 = DateTime.now();
        for (var t = 0; t < t1.difference(t0).inMilliseconds; t += 200) {
          final stamp = t0.add(Duration(milliseconds: t));
          final row = <String>['${(t / 1000).toStringAsFixed(1)}'];
          var any = false;
          for (final pid in pids) {
            final list = engine.history[pid.id] ?? const <PidSample>[];
            var i = cursor[pid.id] ?? 0;
            while (i < list.length && !list[i].time.isAfter(stamp)) {
              lastValue[pid.id] = list[i].value;
              lastTime[pid.id] = list[i].time;
              i++;
            }
            cursor[pid.id] = i;
            final pt = lastTime[pid.id];
            final pv = lastValue[pid.id];
            if (pv != null && pt != null && stamp.difference(pt).inMilliseconds.abs() <= skewMs) {
              row.add(pv.toStringAsFixed(pid.digits));
              any = true;
            } else {
              row.add('');
            }
          }
          if (any) sink.writeln(row.map(csvCell).join(','));
        }
        await sink.flush();
        await sink.close();
        await SharePlus.instance.share(
          ShareParams(files: [XFile(file.path)], text: 'SSM2 RR-совместимый CSV (${file.path})'),
        );
        message = 'RR CSV экспортирован';
        changed();
      });

  Future<void> selectPid(String id, bool value) => perform(() async {
        final next = {...engine.enabled};
        if (value) {
          next.add(id);
        } else {
          next.remove(id);
        }
        await engine.configure(next, engine.extendedConfirmed, engine.romId, engine.endian);
        await saveSettings();
        message = 'Выбор сохранен. Нажмите Старт для опроса.';
      });

  Future<void> preset(bool all) => perform(() async {
        await engine.configure(
          all
              ? SubaruPidLibrary.all.where((p) => !p.extended).map((p) => p.id).toSet()
              : {...SubaruPidLibrary.defaults},
          engine.extendedConfirmed,
          engine.romId,
          engine.endian,
        );
        await saveSettings();
        message = 'Набор сохранен; опрос на паузе';
      });

  Future<void> configureExtended(bool confirm, String rom, Endian endian) => perform(() async {
        if (confirm && rom.trim().isEmpty) {
          throw ArgumentError('Введите ROM ID из своего def-файла');
        }
        await engine.configure({...engine.enabled}, confirm, rom, endian);
        await saveSettings();
        message = 'Настройки ROM сохранены.';
      });

  Future<void> sendDiagnostic(String command) => perform(() async {
        await engine.stop();
        terminal = '';
        final reply = await elm.diagnostic(command);
        terminal = '> ${command.trim().toUpperCase()}\\r\n${visibleText(reply)}';
        message = 'Ручной запрос завершен; опрос остается на паузе';
      });

  Future<void> scan(String start, String count) => perform(() async {
        final address = int.parse(start.trim().replaceFirst(RegExp(r'^0[xX]'), ''), radix: 16);
        final length = int.parse(count);
        if (address >= 0xFF0000 && !engine.extendedConfirmed) {
          throw StateError('Сначала подтвердите ROM');
        }
        await engine.stop();
        scanner = '';
        final result = await elm.readRange(address, length);
        scanner = List<String>.generate(
          result.bytes.length,
          (i) => '0x${hexAddress(address + i)}   ${hex2(result.bytes[i])}   ${result.bytes[i]}',
        ).join('\n');
        message = 'Прочитано ${result.bytes.length} байт за ${result.elapsedMs} мс. Опрос на паузе.';
      });

  Future<void> toggleLog() => perform(() async {
        if (logger.active) {
          await logger.stop();
        } else {
          if (!engine.running || !foreground) {
            throw StateError('Сначала запустите опрос в открытом приложении');
          }
          await logger.start();
          if (!foreground) await logger.stop();
        }
      });

  Future<void> exportCsv() => perform(() async {
        await logger.stop();
        final file = logger.file;
        if (file == null || logger.count == 0) throw StateError('Нет записей CSV');
        await SharePlus.instance.share(ShareParams(files: [XFile(file.path)], text: 'SSM2 PID log'));
      });

  Future<void> exportTrace() => perform(() async {
        final dir = await getApplicationDocumentsDirectory();
        final file = File('${dir.path}/ssm2_trace_${DateTime.now().millisecondsSinceEpoch}.txt');
        await file.writeAsString(
          '${elm.transport.name}\n'
          'ROM (user): ${engine.romId}\n'
          'CAN profile: ${elm.activeProfile}\n'
          'Timeout: ${elm.timeoutMs} ms\n'
          'Sync note: ${elm.syncNote}\n'
          'Endian: ${engine.endian == Endian.big ? 'big' : 'little'}\n'
          '${elm.trace.join('\n')}\n',
          flush: true,
        );
        await SharePlus.instance.share(
          ShareParams(files: [XFile(file.path)], text: 'SSM2 TX/RX diagnostic trace'),
        );
      });

  Future<void> shutdown() async {
    if (_disposed) return;
    _disposed = true;
    await _link.cancel();
    try {
      await elm.disconnect();
    } catch (e) {
      elm.log('SHUTDOWN $e');
    }
    await engine.stop();
    await _samples.cancel();
    await logger.stop();
    try {
      await elm.dispose();
    } catch (e) {
      elm.log('DISPOSE $e');
    }
    engine.dispose();
    super.dispose();
  }
}
'''

FILES["lib/maplab_link.dart"] = r'''
import 'package:flutter/material.dart';

class MapLabTab extends StatelessWidget {
  const MapLabTab({super.key});
  @override
  Widget build(BuildContext context) => const Padding(
    padding: EdgeInsets.all(24),
    child: Text('Map Lab появится после ячейки 2b/3.', style: TextStyle(height: 1.7)),
  );
}
'''

FILES["lib/main.dart"] = r'''
import 'dart:async';
import 'dart:io';
import 'dart:math' as math;
import 'package:path_provider/path_provider.dart';
import 'package:share_plus/share_plus.dart';
import 'package:flutter/foundation.dart';
import 'package:flutter/material.dart';
import 'package:flutter/services.dart';
import 'analyzer.dart';
import 'derived.dart';
import 'engine.dart';
import 'maplab_link.dart';
import 'model.dart';
import 'pids.dart';
import 'protocol.dart';
import 'transport_selected.dart';

void main() { WidgetsFlutterBinding.ensureInitialized(); runApp(const SsmApp()); }
const cyan = Color(0xFF22D3EE);
const muted = Color(0xFF9AAAC0);

class SsmApp extends StatelessWidget {
  const SsmApp({super.key});
  @override
  Widget build(BuildContext context) => MaterialApp(
    title: 'SSM2 Telemetry 0.7', debugShowCheckedModeBanner: false,
    theme: ThemeData(colorScheme: ColorScheme.fromSeed(seedColor: cyan, brightness: Brightness.dark),
      scaffoldBackgroundColor: const Color(0xFF080D18), useMaterial3: true),
    home: const HomeShell(),
  );
}

class HomeShell extends StatefulWidget {
  const HomeShell({super.key});
  @override
  State<HomeShell> createState() => _HomeShellState();
}
class _HomeShellState extends State<HomeShell> with WidgetsBindingObserver {
  late final AppModel model;
  late final Listenable changes;
  late final Timer timer;
  int tab = 0;
  @override
  void initState() {
    super.initState();
    model = AppModel(createTransport());
    changes = Listenable.merge([model, model.engine]);
    WidgetsBinding.instance.addObserver(this);
    unawaited(model.restore());
    timer = Timer.periodic(const Duration(milliseconds: 500), (_) { if (mounted) setState(() {}); });
  }
  @override
  void didChangeAppLifecycleState(AppLifecycleState state) {
    if (state == AppLifecycleState.resumed) model.foreground = true;
    if (state == AppLifecycleState.paused) {
      model.foreground = false;
      unawaited(model.engine.stop());
      unawaited(model.logger.stop());
    }
  }
  @override
  void dispose() {
    timer.cancel(); WidgetsBinding.instance.removeObserver(this);
    unawaited(model.shutdown()); super.dispose();
  }
  @override
  Widget build(BuildContext context) => AnimatedBuilder(animation: changes, builder: (context, _) {
    final engine = model.engine;
    final lastTimes = engine.latest.values.map((s) => s.time).toList()..sort();
    final live = model.elm.ready && lastTimes.isNotEmpty && DateTime.now().difference(lastTimes.last).inSeconds < 3;
    final pages = <Widget>[AdapterPage(model), DashboardPage(model), PidPage(model),
      LoggerPage(model), GraphPage(model), AnalyzerPage(model), DiagnosticPage(model), const MapLabTab()];
    return Scaffold(
      appBar: AppBar(title: const Text('SSM2 TELEMETRY 0.7', style: TextStyle(fontSize: 17, letterSpacing: 2)),
        actions: [Icon(Icons.circle, size: 10, color: live ? Colors.greenAccent : muted), const SizedBox(width: 16)]),
      body: SafeArea(child: Column(children: [
        if (model.busy) const LinearProgressIndicator(minHeight: 2),
        Padding(padding: const EdgeInsets.fromLTRB(16, 4, 16, 8), child: Align(alignment: Alignment.centerLeft,
          child: Text(model.message, maxLines: 3, overflow: TextOverflow.ellipsis,
            style: const TextStyle(color: muted, fontSize: 12)))),
        Expanded(child: pages[tab]),
      ])),
      bottomNavigationBar: NavigationBar(selectedIndex: tab, labelBehavior: NavigationDestinationLabelBehavior.onlyShowSelected,
        onDestinationSelected: (i) => setState(() => tab = i), destinations: const [
          NavigationDestination(icon: Icon(Icons.bluetooth), label: 'Адаптер'),
          NavigationDestination(icon: Icon(Icons.speed), label: 'Дашборд'),
          NavigationDestination(icon: Icon(Icons.tune), label: 'PID'),
          NavigationDestination(icon: Icon(Icons.fiber_manual_record_outlined), label: 'CSV'),
          NavigationDestination(icon: Icon(Icons.show_chart), label: 'График'),
          NavigationDestination(icon: Icon(Icons.insights), label: 'Анализ'),
          NavigationDestination(icon: Icon(Icons.terminal), label: 'Диагн.'),
          NavigationDestination(icon: Icon(Icons.table_view), label: 'Map Lab'),
        ]),
    );
  });
}

Widget section(String title, List<Widget> children) => Padding(
  padding: const EdgeInsets.fromLTRB(16, 16, 16, 10), child: Column(crossAxisAlignment: CrossAxisAlignment.start,
    children: [Text(title, style: const TextStyle(fontSize: 15, fontWeight: FontWeight.bold)),
      const SizedBox(height: 12), ...children]));
Widget detail(String key, String value) => Padding(padding: const EdgeInsets.symmetric(vertical: 4),
  child: Row(crossAxisAlignment: CrossAxisAlignment.start, children: [
    Expanded(flex: 2, child: Text(key, style: const TextStyle(color: muted, fontSize: 12))),
    const SizedBox(width: 10), Expanded(flex: 3, child: Text(value, style: const TextStyle(fontSize: 12))),
  ]));
String ageText(int? age) => age == null ? 'нет данных' : '${(age / 1000).toStringAsFixed(1)} с';

class AdapterPage extends StatelessWidget {
  const AdapterPage(this.model, {super.key});
  final AppModel model;
  @override
  Widget build(BuildContext context) => ListView(children: [
    section('Bluetooth SPP', [
      Text(model.elm.transport.name, style: const TextStyle(color: cyan, fontSize: 12)),
      const SizedBox(height: 10),
      const Text('Только CAN 11 bit / 500 kbit. Зажигание включено, автомобиль стоит.'),
      const SizedBox(height: 10),
      Wrap(spacing: 8, children: [
        OutlinedButton.icon(onPressed: model.busy ? null : model.refreshDevices,
          icon: const Icon(Icons.refresh), label: const Text('Сопряженные')),
        TextButton(onPressed: () => model.perform(() async {
          await const MethodChannel('ssm2/system').invokeMethod<void>('bluetoothSettings');
        }), child: const Text('Настройки Android')),
      ]),
      if (model.devices.isEmpty) const Padding(padding: EdgeInsets.all(12), child: Text('Обновите список устройств')),
      for (final device in model.devices) ListTile(
        contentPadding: EdgeInsets.zero, leading: const Icon(Icons.bluetooth, color: cyan),
        title: Text(device.name.isEmpty ? 'Без имени' : device.name), subtitle: Text(device.address),
        trailing: model.selected == device.address ? const Icon(Icons.check, color: cyan) : null,
        onTap: model.busy ? null : () { model.selected = device.address; model.changed(); }),
      Wrap(spacing: 8, children: [
        FilledButton(onPressed: model.busy || model.selected == null ? null : model.connect,
          child: Text(model.elm.ready ? 'Переподключить' : 'Подключить')),
        OutlinedButton(onPressed: model.busy ? null : model.disconnect, child: const Text('Отключить')),
      ]),
    ]),
    section('Состояние', [
      detail('Синхронизация', model.elm.ready ? 'ДА ✓' : 'нет — нажмите Подключить'),
      detail('Адаптер', model.elm.identity.isEmpty ? 'не опрошен' : model.elm.identity),
      detail('ATRV', model.elm.voltage.isEmpty ? 'не опрошен' : model.elm.voltage),
      detail('Watchdog', '${model.engine.mutedAddresses} PID изолировано · запросов: ${model.engine.requests}'),
      detail('SPP', model.elm.transport.connected ? 'открыт' : 'закрыт'),
      detail('Синхронизация', model.elm.ready ? 'готово' : 'нет'),
    ]),
  ]);
}

class DashboardPage extends StatelessWidget {
  const DashboardPage(this.model, {super.key});
  final AppModel model;
  @override
  Widget build(BuildContext context) {
    final engine = model.engine;
    final pids = engine.active;
    final fuel = estimateFuel(
      maf: engine.latest['MAF']?.value,
      afr: engine.latest['AFR']?.value,
      speed: engine.latest['SPEED']?.value,
      rpm: engine.latest['RPM']?.value,
      pedal: engine.latest['PEDAL']?.value,
    );
    return Column(children: [
      Padding(padding: const EdgeInsets.symmetric(horizontal: 16), child: Row(children: [
        Expanded(child: Text('SAFE Single Frame / ${pids.length} PID\n${engine.pidReadsPerSecond.toStringAsFixed(1)} PID-обновл./с · ${engine.requestsPerSecond.toStringAsFixed(1)} запросов/с',
          style: const TextStyle(fontSize: 12, color: muted))),
        FilledButton.tonalIcon(onPressed: model.busy || !model.elm.ready ? null : model.togglePolling,
          icon: Icon(engine.running ? Icons.pause : Icons.play_arrow), label: Text(engine.running ? 'Пауза' : 'Старт')),
      ])),
      FuelCard(fuel: fuel, mafStale: engine.stale('MAF')),
      if (pids.isEmpty) const Expanded(child: Center(child: Text('Выберите параметры во вкладке PID')))
      else Expanded(child: GridView.builder(padding: const EdgeInsets.all(12), itemCount: pids.length,
        gridDelegate: const SliverGridDelegateWithMaxCrossAxisExtent(maxCrossAxisExtent: 270, mainAxisExtent: 188,
          mainAxisSpacing: 10, crossAxisSpacing: 10), itemBuilder: (context, i) {
          final p = pids[i];
          final sample = engine.latest[p.id];
          final unsupported = sample?.allOnes ?? false;
          final value = unsupported ? null : sample?.value;
          final outdated = engine.stale(p.id);
          final failed = engine.attempts[p.id]?.error.isNotEmpty ?? false;
          final tone = outdated || failed || unsupported ? muted : cyan;
          return Material(color: const Color(0xFF101A2B), borderRadius: BorderRadius.circular(14),
            child: InkWell(borderRadius: BorderRadius.circular(14), onTap: () => showModalBottomSheet<void>(context: context,
              isScrollControlled: true, builder: (context) => SafeArea(child: SingleChildScrollView(child: section(p.id, [
                Text(p.desc), detail('Адреса', p.addresses.map((a) => '0x${hexAddress(a)}').join(', ')),
                detail('Формула', p.formulaText), detail('Сырые байты', engine.latest[p.id]?.raw.map(hex2).join(' ') ?? 'нет'),
                detail('Возраст', ageText(engine.ageMs(p.id))),
                detail('Частота этого PID', '${engine.frequency(p.id).toStringAsFixed(2)} Hz'),
                detail('Окно чтения', '${engine.latest[p.id]?.readMs ?? 0} ms'),
                detail('Последняя ошибка', engine.attempts[p.id]?.error ?? ''),
              ])))), child: Padding(padding: const EdgeInsets.all(14), child: Column(crossAxisAlignment: CrossAxisAlignment.start,
                children: [
                  Row(children: [Expanded(child: Text(p.id, style: const TextStyle(fontSize: 13, fontWeight: FontWeight.bold))),
                    if (outdated && value != null) const Icon(Icons.schedule, size: 14, color: muted)]),
                  Text('${p.unit} / 0x${hexAddress(p.address)}', style: const TextStyle(fontSize: 10, color: muted)),
                  const Spacer(),
                  SizedBox(height: 46, width: double.infinity, child: FittedBox(fit: BoxFit.scaleDown,
                    alignment: Alignment.centerLeft, child: Text(value?.toStringAsFixed(p.digits) ?? '--',
                      style: TextStyle(color: tone, fontSize: 38, fontWeight: FontWeight.w700)))),
                  const SizedBox(height: 10),
                  LinearProgressIndicator(value: value == null ? 0 :
                    ((value - p.minValue) / (p.maxValue - p.minValue)).clamp(0.0, 1.0).toDouble(), color: tone, minHeight: 3),
                  const SizedBox(height: 8),
                  Text(unsupported
                      ? '0xFF: адрес не поддерживается'
                      : failed
                          ? 'ошибка / ${ageText(engine.ageMs(p.id))}'
                          : ageText(engine.ageMs(p.id)),
                    style: TextStyle(
                      color: unsupported ? const Color(0xFFD4B57F) : muted, fontSize: 10)),
                ]))));
        })),
    ]);
  }
}

class FuelCard extends StatelessWidget {
  const FuelCard({super.key, required this.fuel, required this.mafStale});
  final FuelEstimate? fuel;
  final bool mafStale;

  @override
  Widget build(BuildContext context) {
    final f = fuel;
    return Container(
      margin: const EdgeInsets.fromLTRB(12, 10, 12, 0),
      padding: const EdgeInsets.all(14),
      decoration: BoxDecoration(
        color: const Color(0xFF101A2B),
        borderRadius: BorderRadius.circular(14),
        border: Border.all(color: const Color(0xFF24435A)),
      ),
      child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
        Row(children: [
          const Expanded(child: Text('МГНОВЕННЫЙ РАСХОД',
            style: TextStyle(fontSize: 11, letterSpacing: 1.2, color: muted))),
          Text(f == null ? 'нет MAF' : 'расчет по MAF',
            style: const TextStyle(fontSize: 10, color: muted)),
        ]),
        const SizedBox(height: 10),
        if (f == null)
          const Text('Включите PID MAF и запустите опрос.',
            style: TextStyle(fontSize: 12, color: muted))
        else ...[
          Row(crossAxisAlignment: CrossAxisAlignment.end, children: [
            Text(f.litresPerHour.toStringAsFixed(2),
              style: TextStyle(fontSize: 38, fontWeight: FontWeight.w700,
                color: mafStale ? muted : cyan, height: 1)),
            const Padding(padding: EdgeInsets.only(left: 8, bottom: 4),
              child: Text('л/ч', style: TextStyle(fontSize: 13, color: muted))),
            const Spacer(),
            Text(f.litresPer100km == null
                ? 'на месте'
                : '${f.litresPer100km!.toStringAsFixed(1)} л/100км',
              style: TextStyle(fontSize: 16, fontWeight: FontWeight.w600,
                color: mafStale ? muted : Colors.white)),
          ]),
          const SizedBox(height: 10),
          Text('AFR ${f.afrUsed.toStringAsFixed(1)}'
              '${f.afrMeasured ? ' (из ECU)' : ' (стехиометрия)'}'
              ' · плотность $kPetrolDensityGramsPerLitre г/л',
            style: const TextStyle(fontSize: 10, color: muted, height: 1.5)),
        ],
      ]),
    );
  }
}

class AnalyzerPage extends StatefulWidget {
  const AnalyzerPage(this.model, {super.key});
  final AppModel model;
  @override
  State<AnalyzerPage> createState() => _AnalyzerPageState();
}

class _AnalyzerPageState extends State<AnalyzerPage> {
  LogAnalysis? report;
  bool auto = true;
  int lastSignature = -1;
  DateTime lastAutoRun = DateTime.fromMillisecondsSinceEpoch(0);
  String errors = '0';
  int attemptsTotal = 0;

  @override
  void initState() {
    super.initState();
    widget.model.engine.addListener(_onEngine);
  }
  @override
  void didUpdateWidget(covariant AnalyzerPage oldWidget) {
    super.didUpdateWidget(oldWidget);
    if (!identical(oldWidget.model.engine, widget.model.engine)) {
      oldWidget.model.engine.removeListener(_onEngine);
      widget.model.engine.addListener(_onEngine);
    }
  }
  @override
  void dispose() {
    widget.model.engine.removeListener(_onEngine);
    super.dispose();
  }
  void _onEngine() {
    if (!auto || !mounted || widget.model.engine.history.isEmpty) return;
    final signature = widget.model.engine.goodCount ~/ 10;
    if (signature == lastSignature) return;
    lastSignature = signature;
    if (DateTime.now().difference(lastAutoRun).inSeconds < 3) return;
    lastAutoRun = DateTime.now();
    _schedule();
  }
  void _schedule() {
    final withErrors = widget.model.engine.attempts.values
        .where((s) => s.error.isNotEmpty).map((s) => s.pid.id).toList()..sort();
    attemptsTotal = widget.model.engine.attempts.length;
    errors = withErrors.isEmpty ? '0' : withErrors.length <= 8
        ? withErrors.join(', ')
        : '${withErrors.take(8).join(', ')} +${withErrors.length - 8}';
    setState(() {
      report = analyzeLog(widget.model.engine.history, attempts: widget.model.engine.attempts);
    });
  }
  Future<void> _exportReport() async {
    final engine = widget.model.engine;
    final result = report ?? analyzeLog(engine.history, attempts: engine.attempts);
    final md = StringBuffer('# SSM2 0.7 · Анализ журнала\n\n');
    md.writeln('Значений: ${result.sampleCount} · длительность: '
        '${result.spanSeconds.toStringAsFixed(1)} с · каналов: ${result.channels}');
    for (final f in result.findings) {
      md.writeln('\n## [${_levelName(f.level)}] ${f.title}\n');
      md.writeln('${f.detail}\n');
      md.writeln('```\n${f.evidence}\n```\n');
      md.writeln('> ${f.advice}');
    }
    final dir = await getApplicationDocumentsDirectory();
    final file = File('${dir.path}/ssm2_report_${DateTime.now().millisecondsSinceEpoch}.md');
    await file.writeAsString(md.toString(), flush: true);
    await SharePlus.instance.share(ShareParams(files: [XFile(file.path)], text: 'SSM2 анализ журнала (md)'));
  }

  Color _levelColor(FindingLevel level) => switch (level) {
    FindingLevel.critical => const Color(0xFFE08A7A),
    FindingLevel.warning => const Color(0xFFD4B57F),
    FindingLevel.info => muted,
  };
  String _levelName(FindingLevel level) => switch (level) {
    FindingLevel.critical => 'КРИТИЧНО',
    FindingLevel.warning => 'ВНИМАНИЕ',
    FindingLevel.info => 'ИНФО',
  };

  @override
  Widget build(BuildContext context) {
    final engine = widget.model.engine;
    final result = report;
    return ListView(children: [
      section('Анализ журнала', [
        const Text('Прозрачные правила с фиксированными порогами.', style: TextStyle(color: muted, height: 1.6)),
        const SizedBox(height: 12),
        Wrap(spacing: 8, runSpacing: 8, children: [
          FilledButton.icon(
            onPressed: engine.history.isEmpty ? null : _schedule,
            icon: const Icon(Icons.insights, size: 18),
            label: const Text('Проанализировать')),
          OutlinedButton.icon(onPressed: () => setState(() => auto = !auto),
            icon: Icon(auto ? Icons.check_box : Icons.check_box_outline_blank, size: 18),
            label: const Text('Авто')),
          OutlinedButton.icon(onPressed: engine.history.isEmpty || widget.model.busy ? null : _exportReport,
            icon: const Icon(Icons.share, size: 16), label: const Text('Отчёт.md')),
        ]),
      ]),
      if (result != null) ...[
        section('Итог', [
          detail('Значений в памяти', '${result.sampleCount}'),
          detail('Длительность', '${result.spanSeconds.toStringAsFixed(1)} с'),
          detail('Каналов с данными', '${result.channels}'),
        ]),
        for (final f in result.findings)
          Padding(
            padding: const EdgeInsets.fromLTRB(16, 0, 16, 12),
            child: Container(
              padding: const EdgeInsets.all(14),
              decoration: BoxDecoration(
                color: const Color(0xFF101A2B),
                borderRadius: BorderRadius.circular(12),
                border: Border(left: BorderSide(color: _levelColor(f.level), width: 3)),
              ),
              child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
                Text(_levelName(f.level), style: TextStyle(fontSize: 9, letterSpacing: 1.2, color: _levelColor(f.level))),
                const SizedBox(height: 6),
                Text(f.title, style: const TextStyle(fontSize: 14, fontWeight: FontWeight.w600)),
                const SizedBox(height: 8),
                Text(f.detail, style: const TextStyle(fontSize: 12, height: 1.6)),
              ]),
            ),
          )
      ],
    ]);
  }
}

class PidPage extends StatefulWidget {
  const PidPage(this.model, {super.key});
  final AppModel model;
  @override
  State<PidPage> createState() => _PidPageState();
}
class _PidPageState extends State<PidPage> {
  String search = '';
  @override
  Widget build(BuildContext context) {
    final m = widget.model;
    return ListView(children: [section('Библиотека / 28 PID', [
      const Text('Сначала 8 базовых. Больше параметров — ниже частота каждого PID.', style: TextStyle(color: muted)),
      Wrap(spacing: 8, children: [
        TextButton(onPressed: m.busy ? null : () => m.preset(false), child: const Text('8 базовых')),
        TextButton(onPressed: m.busy ? null : () => m.preset(true), child: const Text('Все 20 обычных')),
        TextButton(onPressed: m.busy ? null : () => _settings(context), child: const Text('ROM / Float32')),
      ]),
      TextField(decoration: const InputDecoration(labelText: 'Поиск PID или адреса', prefixIcon: Icon(Icons.search)),
        onChanged: (value) => setState(() => search = value.toUpperCase())),
      const SizedBox(height: 8),
      for (final p in SubaruPidLibrary.all.where((p) => '${p.id} ${p.desc} 0x${hexAddress(p.address)}'.toUpperCase().contains(search.trim())))
        CheckboxListTile(contentPadding: EdgeInsets.zero, controlAffinity: ListTileControlAffinity.leading,
          title: Text('${p.id}${p.extended ? ' *' : ''} / ${p.unit}', style: const TextStyle(fontSize: 13)),
          subtitle: Text('0x${hexAddress(p.address)} / ${p.bytesCount} B / P${p.priority}\n${p.formulaText}',
            style: const TextStyle(fontSize: 10, color: muted)), value: m.engine.enabled.contains(p.id),
          onChanged: m.busy || (p.extended && !m.engine.extendedConfirmed) ? null :
            (value) => m.selectPid(p.id, value ?? false)),
    ])]);
  }
  Future<void> _settings(BuildContext context) async {
    final m = widget.model;
    final controller = TextEditingController(text: m.engine.romId);
    var confirm = m.engine.extendedConfirmed;
    var little = m.engine.endian == Endian.little;
    final accepted = await showDialog<bool>(context: context, builder: (context) => StatefulBuilder(builder: (context, change) =>
      AlertDialog(title: const Text('Extended / ROM'), content: SingleChildScrollView(child: Column(mainAxisSize: MainAxisSize.min,
        children: [
          const Text('Введите ROM ID из своего def-файла для включения extended PID.'),
          TextField(controller: controller, decoration: const InputDecoration(labelText: 'ROM ID')),
          CheckboxListTile(title: const Text('Адреса сверены с моим ROM'), value: confirm,
            onChanged: (value) => change(() => confirm = value ?? false)),
          SwitchListTile(title: const Text('Float32 little-endian'), value: little,
            onChanged: (value) => change(() => little = value)),
        ])), actions: [
          TextButton(onPressed: () => Navigator.pop(context, false), child: const Text('Отмена')),
          FilledButton(onPressed: () => Navigator.pop(context, true), child: const Text('Сохранить')),
        ])));
    final rom = controller.text;
    controller.dispose();
    if (accepted == true) await m.configureExtended(confirm, rom, little ? Endian.little : Endian.big);
  }
}

class LoggerPage extends StatelessWidget {
  const LoggerPage(this.model, {super.key});
  final AppModel model;
  @override
  Widget build(BuildContext context) => ListView(children: [section('Потоковый CSV', [
    Text('${model.logger.count}', style: const TextStyle(fontSize: 54, fontWeight: FontWeight.bold, color: cyan)),
    Text(model.logger.active ? 'Идет запись' : 'Запись остановлена'),
    const SizedBox(height: 18),
    Wrap(spacing: 8, children: [
      FilledButton.icon(onPressed: model.busy ? null : model.toggleLog,
        icon: Icon(model.logger.active ? Icons.stop : Icons.fiber_manual_record),
        label: Text(model.logger.active ? 'Стоп' : 'Записать')),
      OutlinedButton.icon(onPressed: model.busy || model.logger.count == 0 ? null : model.exportCsv,
        icon: const Icon(Icons.share), label: const Text('Экспорт CSV')),
      OutlinedButton.icon(onPressed: model.busy || model.engine.history.isEmpty ? null : model.exportRrCsv,
        icon: const Icon(Icons.table_chart), label: const Text('RR CSV')),
    ]),
  ])]);
}

class GraphPage extends StatelessWidget {
  const GraphPage(this.model, {super.key});
  final AppModel model;
  @override
  Widget build(BuildContext context) {
    final pid = SubaruPidLibrary.byId(model.chartId);
    final samples = List<PidSample>.of(model.engine.history[pid.id] ?? []);
    return ListView(children: [section('График', [
      DropdownButton<String>(value: model.chartId, isExpanded: true,
        items: SubaruPidLibrary.all.map((p) => DropdownMenuItem(value: p.id, child: Text('${p.id} / ${p.unit}'))).toList(),
        onChanged: (value) { if (value != null) { model.chartId = value; model.changed(); } }),
      const SizedBox(height: 18),
      SizedBox(height: 270, child: samples.length < 2 ? const Center(child: Text('Нужны хотя бы два значения')) :
        CustomPaint(painter: TelemetryPainter(samples, pid.digits), size: Size.infinite)),
    ])]);
  }
}
class TelemetryPainter extends CustomPainter {
  TelemetryPainter(this.samples, this.digits);
  final List<PidSample> samples;
  final int digits;
  void label(Canvas canvas, String text, Offset point) {
    final painter = TextPainter(text: TextSpan(text: text, style: const TextStyle(color: muted, fontSize: 10)),
      textDirection: TextDirection.ltr)..layout();
    painter.paint(canvas, point);
  }
  @override
  void paint(Canvas canvas, Size size) {
    final values = samples.map((s) => s.value!).toList();
    final minimum = values.reduce(math.min), maximum = values.reduce(math.max);
    final pad = math.max((maximum - minimum) * 0.1, 0.5);
    final lo = minimum - pad, hi = maximum + pad;
    final first = samples.first.time.millisecondsSinceEpoch;
    final span = math.max(1, samples.last.time.millisecondsSinceEpoch - first);
    final width = math.max(1.0, size.width - 65), height = size.height - 38;
    final grid = Paint()..color = const Color(0xFF233045)..strokeWidth = 1;
    for (var i = 0; i <= 4; i++) {
      final y = 8 + height * i / 4;
      canvas.drawLine(Offset(52, y), Offset(size.width - 8, y), grid);
      label(canvas, (hi - (hi - lo) * i / 4).toStringAsFixed(digits > 1 ? 1 : digits), Offset(0, y - 5));
    }
    final path = Path();
    for (var i = 0; i < samples.length; i++) {
      final sample = samples[i];
      final x = 52 + width * (sample.time.millisecondsSinceEpoch - first) / span;
      final y = 8 + height * (hi - sample.value!) / (hi - lo);
      canvas.drawCircle(Offset(x, y), 2, Paint()..color = cyan);
      if (i == 0 || sample.time.difference(samples[i - 1].time).inMilliseconds > 3000) { path.moveTo(x, y); }
      else { path.lineTo(x, y); }
    }
    canvas.drawPath(path, Paint()..color = cyan..strokeWidth = 2..style = PaintingStyle.stroke);
    label(canvas, '0 s', Offset(52, height + 20));
    label(canvas, '${(span / 1000).toStringAsFixed(1)} s', Offset(size.width - 55, height + 20));
  }
  @override
  bool shouldRepaint(covariant TelemetryPainter oldDelegate) => true;
}

class DiagnosticPage extends StatefulWidget {
  const DiagnosticPage(this.model, {super.key});
  final AppModel model;
  @override
  State<DiagnosticPage> createState() => _DiagnosticPageState();
}
class _DiagnosticPageState extends State<DiagnosticPage> {
  final command = TextEditingController(text: 'ATI');
  final address = TextEditingController(text: '000008');
  final count = TextEditingController(text: '1');
  @override
  void dispose() { command.dispose(); address.dispose(); count.dispose(); super.dispose(); }
  @override
  Widget build(BuildContext context) {
    final m = widget.model;
    return ListView(children: [
      section('Диагностика', [
        TextField(controller: command, decoration: const InputDecoration(labelText: 'ATI / ATRV / A8 00 00 00 08')),
        const SizedBox(height: 8),
        FilledButton(onPressed: m.busy || !m.elm.ready ? null : () => m.sendDiagnostic(command.text), child: const Text('Отправить')),
        SelectableText(m.terminal.isEmpty ? 'Команд еще нет' : m.terminal, style: const TextStyle(fontFamily: 'monospace', fontSize: 12)),
      ]),
      section('Сканер адресов A8', [
        Row(children: [Expanded(child: TextField(controller: address, decoration: const InputDecoration(labelText: 'Адрес HEX'))),
          const SizedBox(width: 12), SizedBox(width: 90, child: TextField(controller: count,
            keyboardType: TextInputType.number, decoration: const InputDecoration(labelText: '1..32 байт')))]),
        const SizedBox(height: 8),
        OutlinedButton(onPressed: m.busy || !m.elm.ready ? null : () => m.scan(address.text, count.text), child: const Text('Прочитать')),
        SelectableText(m.scanner.isEmpty ? 'Нет дампа' : m.scanner, style: const TextStyle(fontFamily: 'monospace', fontSize: 12)),
      ]),
    ]);
  }
}
'''

FILES["test/protocol_test.dart"] = r'''
import 'dart:async';
import 'dart:typed_data';

import 'package:flutter_test/flutter_test.dart';
import 'package:subaru_ssm2/analyzer.dart';
import 'package:subaru_ssm2/bt_transport.dart';
import 'package:subaru_ssm2/elm.dart';
import 'package:subaru_ssm2/pids.dart';
import 'package:subaru_ssm2/protocol.dart';
import 'package:subaru_ssm2/samples.dart';

class FakeTransport implements BtTransport {
  FakeTransport({this.a8DelayMs = 1, this.a8Reply = '7E8 02 E8 6D\r>'});

  final int a8DelayMs;
  final String a8Reply;
  bool _connected = false;
  final received = StreamController<Uint8List>.broadcast(sync: true);
  final states = StreamController<bool>.broadcast(sync: true);
  final writes = <String>[];

  @override
  String get name => 'Synthetic test transport';

  @override
  bool get connected => _connected;

  @override
  Stream<Uint8List> get data => received.stream;

  @override
  Stream<bool> get status => states.stream;

  @override
  Future<List<BtDevice>> paired() async => [const BtDevice('Test', '00:00:00:00:00:00')];

  @override
  Future<void> connect(String address) async {
    _connected = true;
    states.add(true);
  }

  @override
  Future<void> disconnect() async {
    _connected = false;
    states.add(false);
  }

  void emit(String text) => received.add(Uint8List.fromList(text.codeUnits));

  @override
  Future<void> write(String ascii) async {
    if (!_connected) throw StateError('Not connected');
    writes.add(ascii);
    final cmd = ascii.trim().toUpperCase();
    final reply = cmd == 'ATI' || cmd == 'ATZ' || cmd == 'ATWS'
        ? 'ELM327 TEST\r>'
        : cmd == 'ATRV'
            ? '13.5V\r>'
            : cmd.startsWith('AT')
                ? 'OK\r>'
                : cmd.startsWith('A8')
                    ? a8Reply
                    : 'OK\r>';
    emit(reply.substring(0, 1));
    await Future<void>.delayed(Duration(milliseconds: a8DelayMs));
    emit(reply.substring(1));
  }

  @override
  Future<void> dispose() async {
    await received.close();
    await states.close();
  }
}

void main() {
  const command = 'A8 00 00 00 08';
  group('Robust CAN parser', () {
    test('plain ATH0 / CAF1 payload parses', () {
      expect(parseAddressReply('E8 6D\r\n>', command), 109);
      expect(parseAddressReply('E86D\r\r>', command), 109);
    });

    test('headers-on and length-prefixed payload parses', () {
      expect(parseAddressReply('7E8 02 E8 6D\r\n>', command), 109);
      expect(parseAddressReply('02 E8 6D\r>', command), 109);
      expect(parseAddressReply('SEARCHING...\r\n7E8 02 E8 6D\r>', command), 109);
    });

    test('only single-address A8', () {
      expect(readAddressCommand(8), command);
      expect(readAddressCommand(0xFF2538), 'A8 00 FF 25 38');
    });

    test('safe diagnostics allow adapter-side CAN setup', () {
      expect(allowedDiagnostic('ATI'), isTrue);
      expect(allowedDiagnostic('ATCAF1'), isTrue);
      expect(allowedDiagnostic('ATSH 7E0'), isTrue);
      expect(allowedDiagnostic(command), isTrue);
      expect(allowedDiagnostic('ATMA'), isFalse);
    });
  });

  group('ElmDriver CAN sync', () {
    test('initialize performs ECU probe and keeps conservative timeout', () async {
      final transport = FakeTransport(a8DelayMs: 420, a8Reply: '7E8 02 E8 6D\r>');
      final elm = ElmDriver(transport);
      await elm.initialize('00:00:00:00:00:00');
      expect(elm.ready, isTrue);
      expect(elm.activeProfile, isNotEmpty);
      expect(elm.timeoutMs, greaterThanOrEqualTo(900));
      final read = await elm.readRange(0x000008, 1);
      expect(read.bytes, <int>[0x6D]);
      await elm.dispose();
    });
  });

  group('analyzeLog 0.9', () {
    PidSample fake(String id, double v, int msAgo) => PidSample(
          SubaruPidLibrary.byId(id),
          DateTime.now().subtract(Duration(milliseconds: msAgo)),
          v,
          const <int>[0],
          5,
          '',
        );
    List<PidSample> ramp(String id, double v) =>
        [for (var i = 0; i < 10; i++) fake(id, v, 1000 - i * 100)];

    test('детон-кластер под нагрузкой -> критичная находка', () {
      final result = analyzeLog(<String, List<PidSample>>{
        'FBKC': ramp('FBKC', -3.0),
        'RPM': ramp('RPM', 4200),
        'LOAD': ramp('LOAD', 2.4),
      });
      expect(result.findings.any((f) => f.level == FindingLevel.critical), isTrue);
      expect(result.sampleCount, greaterThan(0));
    });

    test('чистый лог -> без критичных находок', () {
      final result = analyzeLog(<String, List<PidSample>>{
        'FBKC': ramp('FBKC', 0),
        'IAM': ramp('IAM', 16),
        'RPM': ramp('RPM', 2500),
        'ECT': ramp('ECT', 88),
      });
      expect(result.findings.any((f) => f.level == FindingLevel.critical), isFalse);
    });

    test('AFR беднее цели -> находка про смесь', () {
      final result = analyzeLog(<String, List<PidSample>>{
        'AFR': ramp('AFR', 12.9),
        'CL_TARGET': ramp('CL_TARGET', 11.5),
      });
      expect(result.findings.any((f) => f.title.contains('Смесь')), isTrue);
    });

    test('выбросы по z-score -> информационная находка', () {
      final flat = [for (var i = 0; i < 40; i++) fake('RPM', 800, 5000 - i * 100)];
      for (var k = 0; k < 3; k++) {
        flat.add(fake('RPM', 9000, 100 - k * 10));
      }
      final result = analyzeLog(<String, List<PidSample>>{'RPM': flat});
      expect(result.findings.any((f) => f.title.contains('Выбросы')), isTrue);
    });

    test('гуляющий холостой ход -> warning про пропуски', () {
      final t0 = DateTime.now();
      PidSample at(String id, double v, int msAgo) => PidSample(
            SubaruPidLibrary.byId(id),
            t0.subtract(Duration(milliseconds: msAgo)),
            v,
            const <int>[0],
            5,
            '',
          );
      final agos = [for (var i = 0; i < 12; i++) 1100 - i * 90];
      final result = analyzeLog(<String, List<PidSample>>{
        'RPM': [for (var i = 0; i < 12; i++) at('RPM', i.isEven ? 740.0 : 960.0, agos[i])],
        'TPS': [for (var i = 0; i < 12; i++) at('TPS', 2, agos[i])],
        'ECT': [for (var i = 0; i < 12; i++) at('ECT', 85, agos[i])],
      });
      final idle = result.findings.where((f) => f.title.contains('холост'));
      expect(idle.isNotEmpty, isTrue);
      expect(idle.first.level, FindingLevel.warning);
    });

    test('compareLogs: меньше детонации -> вердикт лучше', () {
      final base = <String, List<PidSample>>{'FBKC': ramp('FBKC', -2.0)};
      final after = <String, List<PidSample>>{'FBKC': ramp('FBKC', 0)};
      final cmp = compareLogs(base, after);
      expect(cmp.verdict, contains('лучше'));
      expect(cmp.knockAfter, lessThan(cmp.knockBase));
    });
  });

  group('All 28 PID', () {
    test('identity, addresses and sizes', () {
      final all = SubaruPidLibrary.all;
      expect(all.length, 28);
      expect(SubaruPidLibrary.byId('RPM').addresses, [0xE, 0xF]);
      expect(SubaruPidLibrary.byId('O2_F').decode([0, 200]), closeTo(1.0, 0.000001));
    });
  });
}
'''

FILES["android/settings.gradle.kts"] = r'''
pluginManagement {
    val flutterSdkPath = run {
        val properties = java.util.Properties()
        file("local.properties").inputStream().use { properties.load(it) }
        requireNotNull(properties.getProperty("flutter.sdk")) { "flutter.sdk missing" }
    }
    includeBuild("$flutterSdkPath/packages/flutter_tools/gradle")
    repositories { google(); mavenCentral(); gradlePluginPortal() }
}
plugins {
    id("dev.flutter.flutter-plugin-loader") version "1.0.0"
    id("com.android.application") version "8.11.1" apply false
    id("com.android.library") version "8.11.1" apply false
    id("org.jetbrains.kotlin.android") version "2.2.20" apply false
}
include(":app")
'''

FILES["android/build.gradle.kts"] = r'''
allprojects {
    repositories { google(); mavenCentral() }
}
val newBuildDir = rootProject.layout.buildDirectory.dir("../../build").get()
rootProject.layout.buildDirectory.value(newBuildDir)
subprojects {
    project.layout.buildDirectory.value(newBuildDir.dir(project.name))
}
subprojects { project.evaluationDependsOn(":app") }
tasks.register<Delete>("clean") { delete(rootProject.layout.buildDirectory) }
'''

FILES["android/app/build.gradle.kts"] = r'''import java.io.FileInputStream
import java.util.Properties

plugins {
    id("com.android.application")
    id("org.jetbrains.kotlin.android")
    id("dev.flutter.flutter-gradle-plugin")
}

val keystoreProperties = Properties()
val keystorePropertiesFile = rootProject.file("key.properties")
val hasUploadKey = keystorePropertiesFile.exists()
if (hasUploadKey) {
    FileInputStream(keystorePropertiesFile).use { keystoreProperties.load(it) }
}

android {
    namespace = "com.subaru.ssm2_fixed"
    compileSdk = 36
    ndkVersion = "27.0.12077973"
    compileOptions {
        sourceCompatibility = JavaVersion.VERSION_17
        targetCompatibility = JavaVersion.VERSION_17
    }
    kotlinOptions { jvmTarget = "17" }
    defaultConfig {
        applicationId = "com.subaru.ssm2_fixed"
        minSdk = 24
        targetSdk = 35
        versionCode = flutter.versionCode
        versionName = flutter.versionName
    }
    signingConfigs {
        create("release") {
            if (hasUploadKey) {
                storeFile = rootProject.file(keystoreProperties.getProperty("storeFile"))
                storePassword = keystoreProperties.getProperty("storePassword")
                keyAlias = keystoreProperties.getProperty("keyAlias")
                keyPassword = keystoreProperties.getProperty("keyPassword")
            } else {
                initWith(signingConfigs.getByName("debug"))
            }
        }
    }
    buildTypes {
        release {
            signingConfig = signingConfigs.getByName("release")
            isMinifyEnabled = false
            isShrinkResources = false
        }
    }
}
flutter { source = "../.." }
'''

FILES["android/gradle.properties"] = r'''
org.gradle.jvmargs=-Xmx4g -XX:MaxMetaspaceSize=1g -XX:+HeapDumpOnOutOfMemoryError
org.gradle.workers.max=2
org.gradle.caching=true
android.useAndroidX=true
'''

FILES["android/gradle/wrapper/gradle-wrapper.properties"] = r'''
distributionBase=GRADLE_USER_HOME
distributionPath=wrapper/dists
zipStoreBase=GRADLE_USER_HOME
zipStorePath=wrapper/dists
distributionUrl=https\://services.gradle.org/distributions/gradle-8.14.3-bin.zip
networkTimeout=120000
validateDistributionUrl=true
'''

FILES["android/app/src/main/kotlin/com/subaru/ssm2_fixed/MainActivity.kt"] = r'''
package com.subaru.ssm2_fixed

import android.bluetooth.BluetoothAdapter
import android.bluetooth.BluetoothManager
import android.bluetooth.BluetoothSocket
import android.content.Context
import android.content.Intent
import android.os.Build
import android.os.Handler
import android.os.Looper
import android.provider.Settings
import io.flutter.embedding.android.FlutterActivity
import io.flutter.embedding.engine.FlutterEngine
import io.flutter.plugin.common.EventChannel
import io.flutter.plugin.common.MethodChannel
import java.io.IOException
import java.util.UUID
import java.util.concurrent.Executors

class MainActivity : FlutterActivity() {
    private val sppUuid: UUID = UUID.fromString("00001101-0000-1000-8000-00805F9B34FB")
    private val io = Executors.newSingleThreadExecutor()
    private val main = Handler(Looper.getMainLooper())

    private var socket: BluetoothSocket? = null
    @Volatile private var reading = false
    private var dataSink: EventChannel.EventSink? = null
    private var statusSink: EventChannel.EventSink? = null

    private fun btAdapter(): BluetoothAdapter? =
        (getSystemService(Context.BLUETOOTH_SERVICE) as? BluetoothManager)?.adapter

    private fun pairedDevices(): List<Map<String, String>> {
        val adapter = btAdapter() ?: return emptyList()
        return try {
            adapter.bondedDevices?.map { device ->
                mapOf("name" to (device.name ?: ""), "address" to device.address)
            } ?: emptyList()
        } catch (e: SecurityException) {
            emptyList()
        }
    }

    private fun socketCandidates(address: String): List<BluetoothSocket> {
        val adapter = btAdapter() ?: throw IOException("Bluetooth адаптер недоступен")
        val device = try {
            adapter.getRemoteDevice(address)
        } catch (e: IllegalArgumentException) {
            throw IOException("Некорректный MAC: $address")
        }
        try { adapter.cancelDiscovery() } catch (_: SecurityException) {}

        val list = mutableListOf<BluetoothSocket>()
        try { list.add(device.createInsecureRfcommSocketToServiceRecord(sppUuid)) } catch (_: Exception) {}
        try { list.add(device.createRfcommSocketToServiceRecord(sppUuid)) } catch (_: Exception) {}

        // Классический fallback для дешёвых ELM327-клонов: RFCOMM channel 1 через reflection.
        try {
            val method = device.javaClass.getMethod("createRfcommSocket", Int::class.javaPrimitiveType)
            val reflected = method.invoke(device, 1) as? BluetoothSocket
            if (reflected != null) list.add(reflected)
        } catch (_: Exception) {}

        if (list.isEmpty()) throw IOException("Не удалось создать RFCOMM сокет")
        return list
    }

    private fun openSocket(address: String): BluetoothSocket {
        var lastError: Exception? = null
        for (candidate in socketCandidates(address)) {
            try {
                Thread.sleep(120)
                candidate.connect()
                return candidate
            } catch (e: Exception) {
                lastError = e
                try { candidate.close() } catch (_: Exception) {}
            }
        }
        throw IOException(lastError?.message ?: "RFCOMM connect failed")
    }

    private fun startReader(active: BluetoothSocket) {
        reading = true
        io.execute {
            val buffer = ByteArray(4096)
            try {
                val input = active.inputStream
                while (reading) {
                    val count = input.read(buffer)
                    if (count < 0) break
                    if (count > 0) {
                        val chunk = buffer.copyOf(count)
                        main.post { dataSink?.success(chunk) }
                    }
                }
            } catch (_: Exception) {
                // Разрыв линии: уходим в status=false
            }
            reading = false
            closeSocket()
            main.post { statusSink?.success(false) }
        }
    }

    private fun closeSocket() {
        val current = socket
        socket = null
        if (current != null) {
            try { current.close() } catch (_: Exception) {}
        }
    }

    override fun configureFlutterEngine(flutterEngine: FlutterEngine) {
        super.configureFlutterEngine(flutterEngine)
        val messenger = flutterEngine.dartExecutor.binaryMessenger

        MethodChannel(messenger, "ssm2/system").setMethodCallHandler { call, result ->
            when (call.method) {
                "sdkInt" -> result.success(Build.VERSION.SDK_INT)
                "bluetoothSettings" -> {
                    startActivity(Intent(Settings.ACTION_BLUETOOTH_SETTINGS))
                    result.success(null)
                }
                else -> result.notImplemented()
            }
        }

        EventChannel(messenger, "ssm2/spp_data").setStreamHandler(object : EventChannel.StreamHandler {
            override fun onListen(arguments: Any?, events: EventChannel.EventSink) { dataSink = events }
            override fun onCancel(arguments: Any?) { dataSink = null }
        })
        EventChannel(messenger, "ssm2/spp_status").setStreamHandler(object : EventChannel.StreamHandler {
            override fun onListen(arguments: Any?, events: EventChannel.EventSink) { statusSink = events }
            override fun onCancel(arguments: Any?) { statusSink = null }
        })

        MethodChannel(messenger, "ssm2/spp").setMethodCallHandler { call, result ->
            when (call.method) {
                "paired" -> io.execute {
                    val devices = pairedDevices()
                    main.post { result.success(devices) }
                }
                "connect" -> {
                    val address = call.argument<String>("address")
                    if (address.isNullOrBlank()) {
                        result.error("ARG", "address required", null)
                    } else {
                        io.execute {
                            try {
                                reading = false
                                closeSocket()
                                val opened = openSocket(address)
                                socket = opened
                                startReader(opened)
                                main.post {
                                    statusSink?.success(true)
                                    result.success(true)
                                }
                            } catch (e: Exception) {
                                closeSocket()
                                main.post { result.error("CONNECT", e.message ?: "RFCOMM connect failed", null) }
                            }
                        }
                    }
                }
                "write" -> {
                    val data = call.argument<String>("data")
                    if (data == null) {
                        result.error("ARG", "data required", null)
                    } else {
                        io.execute {
                            try {
                                val current = socket ?: throw IOException("SPP disconnected")
                                current.outputStream.write(data.toByteArray(Charsets.US_ASCII))
                                current.outputStream.flush()
                                main.post { result.success(null) }
                            } catch (e: Exception) {
                                main.post { result.error("IO", e.message, null) }
                            }
                        }
                    }
                }
                "disconnect" -> io.execute {
                    reading = false
                    closeSocket()
                    main.post { result.success(null) }
                }
                else -> result.notImplemented()
            }
        }
    }

    override fun onDestroy() {
        reading = false
        closeSocket()
        dataSink = null
        statusSink = null
        super.onDestroy()
    }
}
'''

FILES["android/app/src/main/AndroidManifest.xml"] = r'''
<manifest xmlns:android="http://schemas.android.com/apk/res/android">
    <uses-permission android:name="android.permission.BLUETOOTH" android:maxSdkVersion="30"/>
    <uses-permission android:name="android.permission.BLUETOOTH_ADMIN" android:maxSdkVersion="30"/>
    <uses-permission android:name="android.permission.ACCESS_FINE_LOCATION" android:maxSdkVersion="30"/>
    <uses-permission android:name="android.permission.ACCESS_COARSE_LOCATION" android:maxSdkVersion="30"/>
    <uses-permission android:name="android.permission.BLUETOOTH_SCAN" android:usesPermissionFlags="neverForLocation"/>
    <uses-permission android:name="android.permission.BLUETOOTH_CONNECT"/>
    <uses-feature android:name="android.hardware.bluetooth" android:required="true"/>
    <application android:label="SSM2 Fixed" android:name="${applicationName}" android:icon="@mipmap/ic_launcher">
        <activity android:name=".MainActivity" android:exported="true" android:launchMode="singleTop"
            android:theme="@style/LaunchTheme" android:hardwareAccelerated="true"
            android:configChanges="orientation|keyboardHidden|keyboard|screenSize|smallestScreenSize|locale|layoutDirection|fontScale|screenLayout|density|uiMode"
            android:windowSoftInputMode="adjustResize">
            <meta-data android:name="io.flutter.embedding.android.NormalTheme" android:resource="@style/NormalTheme"/>
            <intent-filter>
                <action android:name="android.intent.action.MAIN"/>
                <category android:name="android.intent.category.LAUNCHER"/>
            </intent-filter>
        </activity>
        <meta-data android:name="flutterEmbedding" android:value="2"/>
    </application>
    <queries>
        <intent><action android:name="android.intent.action.PROCESS_TEXT"/><data android:mimeType="text/plain"/></intent>
    </queries>
</manifest>
'''

FILES["pubspec.template.yaml"] = r'''name: subaru_ssm2
description: "Read-only Subaru SSM2 CAN telemetry: 28 PID, SAFE single-frame, Map Lab, native SPP, авто-аналитика (0.9)"
publish_to: 'none'
version: 0.9.2+11
environment:
  sdk: '>=3.3.0 <4.0.0'
dependencies:
  flutter:
    sdk: flutter
  permission_handler: 11.3.1
  path_provider: 2.1.5
  share_plus: 12.0.2
dev_dependencies:
  flutter_test:
    sdk: flutter
flutter:
  uses-material-design: true
'''

FILES["tool/calid_import.py"] = r'''#!/usr/bin/env python3
"""Конвертер адресов RAM-параметров ЭБУ в Dart CalProfile (identity.dart).

Источник адресов — logger-дефиниции RomRaider/ECUFlash для конкретной калибровки
(CALID). Подготовьте CSV такого вида (без заголовка или с ним):

    A2TB100B,IAM,0xFF2538
    A2TB100B,FBKC,0xFF7D4C
    A2ZJ100J,IAM,0xFF1A2B
    ...

Допустимые param: IAM, LOAD_4B, BOOST_ERR, BOOST_TGT, FBKC, FKL, BOOST, CL_TARGET.
Запуск:  python3 tool/calid_import.py my_profiles.csv [label] [parentDefs]
На выходе — готовые const CalProfile(...) для вставки в lib/identity.dart.
"""
import csv
import sys
from collections import OrderedDict

VALID = {"IAM", "LOAD_4B", "BOOST_ERR", "BOOST_TGT", "FBKC", "FKL", "BOOST", "CL_TARGET"}


def main() -> int:
    if len(sys.argv) < 2:
        print(__doc__)
        return 1
    source = sys.argv[1]
    label = sys.argv[2] if len(sys.argv) > 2 else "Imported profile"
    parent = sys.argv[3] if len(sys.argv) > 3 else "Imported via tool/calid_import.py"

    profiles: "OrderedDict[str, dict]" = OrderedDict()
    errors = 0
    with open(source, newline="", encoding="utf-8-sig") as handle:
        for number, row in enumerate(csv.reader(handle), start=1):
            if not row or row[0].strip().startswith("#"):
                continue
            if len(row) < 3:
                print(f"[строка {number}] нужно минимум 3 колонки: {row}")
                errors += 1
                continue
            calid, param, address = (cell.strip() for cell in row[:3])
            if calid.lower() == "calid":  # заголовок
                continue
            param = param.upper()
            if param not in VALID:
                print(f"[строка {number}] неизвестный параметр: {param}")
                errors += 1
                continue
            try:
                value = int(address, 16)
            except ValueError:
                print(f"[строка {number}] адрес не hex: {address}")
                errors += 1
                continue
            if not 0 <= value <= 0xFFFFFF:
                print(f"[строка {number}] адрес вне диапазона: {address}")
                errors += 1
                continue
            profiles.setdefault(calid, {})[param] = f"0x{value:06X}"

    if errors:
        print(f"\nОшибок: {errors}. Исправьте CSV и повторите.")
        return 2
    if not profiles:
        print("Не найдено ни одной строки данных.")
        return 2

    print("// Вставьте в список calProfiles (lib/identity.dart):")
    for calid, addresses in profiles.items():
        print(f"  CalProfile('{calid}', 'N/A',")
        print(f"      '{label}',")
        print("      <String, String>{")
        for param, address in addresses.items():
            print(f"        '{param}': '{address}',")
        print("      },")
        print(f"      '{parent}'),\n")
    print(f"Готово: {len(profiles)} профил(я/ей). Пересоберите APK — ячейка 05.")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
'''

FILES["lib/samples.dart"] = r'''
import 'pids.dart';

class PidSample {
  PidSample(this.pid, this.time, this.value, this.raw, this.readMs, this.error);
  final SubaruPidDef pid;
  final DateTime time;
  final double? value;
  final List<int> raw;
  final int readMs;
  final String error;
  bool get good => value != null && error.isEmpty;
  bool get allOnes => raw.isNotEmpty && raw.every((b) => b == 0xFF);
}
'''

FILES["lib/identity.dart"] = r'''
String normalizeCalId(String value) =>
    value.toUpperCase().replaceAll(RegExp(r'[^0-9A-Z]'), '');

class CalProfile {
  const CalProfile(this.calId, this.ecuId, this.label,
      this.addresses, this.parentDefs);
  final String calId, ecuId, label;
  final Map<String, String> addresses;
  final String parentDefs;
}

const List<CalProfile> calProfiles = <CalProfile>[
  CalProfile('A2TB100B', '5204584007',
      'Legacy GT (BP/BL) EJ20X JDM 2008 · TD04HL-19T · Dual AVCS · 5EAT',
      <String, String>{
        'IAM': '0xFF2538',
        'LOAD_4B': '0xFF6C9C',
        'BOOST_ERR': '0xFF6450',
        'BOOST_TGT': '0xFF6454',
        'FBKC': '0xFF7D4C',
        'FKL': '0xFF7DD0',
        'BOOST': '0xFF6AE0',
        'CL_TARGET': '0xFF73B4',
      },
      'ECU defs: A2TB100B -> A2TB100K -> 32BITBASE (TD-D/SubaruDefs)'),
];

CalProfile? matchCalProfile(String? candidate) {
  final norm = normalizeCalId(candidate ?? '');
  if (norm.isEmpty) return null;
  for (final profile in calProfiles) {
    if (profile.calId == norm) return profile;
  }
  return null;
}
'''

FILES["lib/derived.dart"] = r'''
const double kPetrolDensityGramsPerLitre = 745.0;
const double kStoichAfr = 14.7;

class FuelEstimate {
  const FuelEstimate({
    required this.fuelGramsPerSecond,
    required this.litresPerHour,
    required this.litresPer100km,
    required this.afrUsed,
    required this.afrMeasured,
    required this.speedKph,
    required this.possibleCutoff,
  });

  final double fuelGramsPerSecond;
  final double litresPerHour;
  final double? litresPer100km;
  final double afrUsed;
  final bool afrMeasured;
  final double? speedKph;
  final bool possibleCutoff;

  String get formula => 'MAF / AFR * 3600 / $kPetrolDensityGramsPerLitre';
}

FuelEstimate? estimateFuel({
  double? maf,
  double? afr,
  double? speed,
  double? rpm,
  double? pedal,
}) {
  if (maf == null || !maf.isFinite || maf <= 0) return null;
  final measured = afr != null && afr.isFinite && afr >= 8 && afr <= 25;
  final ratio = measured ? afr : kStoichAfr;
  final grams = maf / ratio;
  if (!grams.isFinite || grams < 0) return null;
  final litresPerHour = grams * 3600 / kPetrolDensityGramsPerLitre;
  double? per100;
  if (speed != null && speed.isFinite && speed >= 5) {
    per100 = litresPerHour / speed * 100;
  }
  final cutoff = pedal != null && pedal <= 1 &&
      rpm != null && rpm > 1500 &&
      speed != null && speed > 5;
  return FuelEstimate(
    fuelGramsPerSecond: grams,
    litresPerHour: litresPerHour,
    litresPer100km: per100,
    afrUsed: ratio,
    afrMeasured: measured,
    speedKph: speed,
    possibleCutoff: cutoff,
  );
}
'''

FILES["lib/analyzer.dart"] = r'''
import 'dart:math' as math;

import 'samples.dart';

enum FindingLevel { info, warning, critical }

class Finding {
  const Finding(this.level, this.title, this.detail, this.evidence, this.advice);
  final FindingLevel level;
  final String title, detail, evidence, advice;
}

class LogAnalysis {
  const LogAnalysis(this.findings, this.sampleCount, this.spanSeconds, this.channels);
  final List<Finding> findings;
  final int sampleCount, channels;
  final double spanSeconds;
}

const double kFbkcEventDeg = -1.4;
const double kFbkcBadDeg = -4.0;
const double kFlkcWarnDeg = -2.0;
const double kFlkcBadDeg = -4.0;
const double kIamFull = 15.5;
const double kIamPoor = 13.0;
const double kTrimWarnPct = 6.0;
const double kTrimBadPct = 10.0;
const double kBattWarnV = 11.8;
const double kWarmEctC = 70.0;
const double kColdRpm = 2500.0;

class _Series {
  _Series(this.samples);
  final List<({double v, DateTime t})> samples;
  int get count => samples.length;
  bool get empty => samples.isEmpty;
  double get mean =>
      samples.isEmpty ? 0 : samples.fold<double>(0, (a, s) => a + s.v) / samples.length;
  double get min =>
      samples.isEmpty ? 0 : samples.fold<double>(double.infinity, (a, s) => math.min(a, s.v));
  double get max => samples.isEmpty
      ? 0
      : samples.fold<double>(double.negativeInfinity, (a, s) => math.max(a, s.v));
  double get p95 {
    if (samples.isEmpty) return 0;
    final vals = [for (final s in samples) s.v]..sort();
    return vals[((vals.length - 1) * 0.95).round()];
  }
}

_Series _series(Map<String, List<PidSample>> history, List<String> ids) {
  final out = <({double v, DateTime t})>[];
  for (final id in ids) {
    final list = history[id];
    if (list == null) continue;
    for (final s in list) {
      final v = s.value;
      if (s.good && v != null && v.isFinite) out.add((v: v, t: s.time));
    }
  }
  out.sort((a, b) => a.t.compareTo(b.t));
  return _Series(out);
}

/// Сопоставление двух каналов по ближайшим меткам времени (v0.9).
List<({double a, double b, DateTime t})> _pair(
  _Series left,
  _Series right, {
  int toleranceMs = 300,
}) {
  final out = <({double a, double b, DateTime t})>[];
  if (left.empty || right.empty) return out;
  var j = 0;
  for (final l in left.samples) {
    while (j + 1 < right.samples.length &&
        right.samples[j + 1].t.difference(l.t).inMilliseconds.abs() <=
            right.samples[j].t.difference(l.t).inMilliseconds.abs()) {
      j++;
    }
    final r = right.samples[j];
    if (r.t.difference(l.t).inMilliseconds.abs() <= toleranceMs) {
      out.add((a: l.v, b: r.v, t: l.t));
    }
  }
  return out;
}

String _f(double v) => v.toStringAsFixed(1);
String _f2(double v) => v.toStringAsFixed(2);

LogAnalysis analyzeLog(
  Map<String, List<PidSample>> history, {
  Map<String, PidSample> attempts = const <String, PidSample>{},
}) {
  final findings = <Finding>[];
  final series = <String, _Series>{
    'rpm': _series(history, const ['RPM']),
    'load': _series(history, const ['LOAD_4B', 'LOAD']),
    'ect': _series(history, const ['ECT']),
    'fbkc': _series(history, const ['FBKC']),
    'flkc': _series(history, const ['FKL', 'KNOCK_ADV']),
    'iam': _series(history, const ['IAM']),
    'stft': _series(history, const ['STFT']),
    'ltft': _series(history, const ['LTFT']),
    'batt': _series(history, const ['BATT']),
    'tps': _series(history, const ['TPS', 'PEDAL']),
    'afr': _series(history, const ['AFR']),
    'cltgt': _series(history, const ['CL_TARGET']),
    'boost': _series(history, const ['BOOST', 'MAP_REL']),
    'boosttgt': _series(history, const ['BOOST_TGT']),
  };

  var sampleCount = 0;
  DateTime? first;
  DateTime? last;
  var channels = 0;
  for (final s in series.values) {
    sampleCount += s.count;
    if (s.count > 0) channels++;
    for (final p in s.samples) {
      if (first == null || p.t.isBefore(first)) first = p.t;
      if (last == null || p.t.isAfter(last)) last = p.t;
    }
  }
  final span = first != null && last != null
      ? last.difference(first).inMilliseconds / 1000.0
      : 0.0;

  // --- Детонация: FBKC -------------------------------------------------------------
  final fbkc = series['fbkc']!;
  if (!fbkc.empty) {
    final events = fbkc.samples.where((s) => s.v <= kFbkcEventDeg).toList();
    final hardLoad = series['load']!.max >= 1.5 && series['rpm']!.max >= 3000;
    if (events.isNotEmpty || fbkc.min <= 0) {
      if (events.length >= 5 || fbkc.min <= kFbkcBadDeg || (events.isNotEmpty && hardLoad)) {
        findings.add(Finding(
          FindingLevel.critical,
          'Детонация: ECU агрессивно снижает УОЗ',
          'Зафиксировано ${events.length} событий FBKC <= ${_f(kFbkcEventDeg)}°, '
              'минимум ${_f(fbkc.min)}°${hardLoad ? ' на нагрузке (load >= 1.5, rpm >= 3000)' : ''}. '
              'Залейте топливо с октаном выше, затем снимите 1-2° в зоне кластера через Map Lab.',
          'fbkc: n=${fbkc.count} events=${events.length} min=${_f(fbkc.min)}° · '
              'rpm_max=${_f(series['rpm']!.max)} load_max=${_f2(series['load']!.max)}',
          'Map Lab -> timing/knockadv: кластерная коррекция -0.5..-1.0° с веером сглаживания.',
        ));
      } else if (events.isNotEmpty || fbkc.min <= -0.5) {
        findings.add(Finding(
          FindingLevel.warning,
          'Единичные рывки FBKC',
          'Есть ${events.length} событий FBKC <= ${_f(kFbkcEventDeg)}°, минимум ${_f(fbkc.min)}°. '
              'Пока это штатная реакция, но повторяемость в одной зоне оборотов — повод для точечной коррекции.',
          'events=${events.length} min=${_f(fbkc.min)}°',
          'Повторный лог в той же нагрузке; если кластер — минус 0.5° в Map Lab.',
        ));
      }
    }
  }

  // --- Детонация: FLKC (обученная коррекция) ---------------------------------------
  final flkc = series['flkc']!;
  if (!flkc.empty && flkc.min < 0) {
    final level = flkc.min <= kFlkcBadDeg ? FindingLevel.critical : FindingLevel.warning;
    if (flkc.min <= kFlkcWarnDeg) {
      findings.add(Finding(
        level,
        'Обученная коррекция зажигания удерживается',
        'FLKC достигала ${_f(flkc.min)}° и не возвращается к нулю: ECU запомнил детонацию '
            'в весовых зонах. Топливо/наддув/свечи — сначала причина, потом карта.',
        'flkc: min=${_f(flkc.min)}° mean=${_f(flkc.mean)}° n=${flkc.count}',
        'Сброс обучения и повторный лог только после устранения причины.',
      ));
    }
  }

  // --- IAM (обучение грубых/тонких таблиц) -------------------------------------------
  final iam = series['iam']!;
  if (!iam.empty) {
    final maxIam = iam.max;
    if (maxIam < kIamPoor) {
      findings.add(Finding(
        FindingLevel.critical,
        'IAM критически низкий',
        'Динамический шаг зажигания ограничен: IAM max ${_f(maxIam)} из 16. '
            'Двигатель работает по грубой (conservative) карте — мощность ниже, риск выше.',
        'iam: max=${_f(maxIam)} mean=${_f(iam.mean)} n=${iam.count}',
        'Не снимать ограничение, пока FBKC/FLKC не чисты: это защита мотора.',
      ));
    } else if (maxIam < kIamFull) {
      findings.add(Finding(
        FindingLevel.warning,
        'IAM не добрался до полного',
        'IAM max ${_f(maxIam)} из 16: обучение не завершено или ECU придерживает запас. '
            'Нормально после сброса адаптаций, иначе — слабый сигнал скрытого детона.',
        'iam: max=${_f(maxIam)} mean=${_f(iam.mean)} n=${iam.count}',
        'Лог 15-20 мин спокойной езды после прогрева: IAM должен дорасти до 16.',
      ));
    } else {
      findings.add(Finding(
        FindingLevel.info,
        'IAM полный',
        'Обучение зажигания завершено: IAM достигал ${_f(maxIam)} / 16.',
        'iam: max=${_f(maxIam)} mean=${_f(iam.mean)}',
        'Без действий.',
      ));
    }
  }

  // --- Топливные коррекции -----------------------------------------------------------
  final stft = series['stft']!;
  final ltft = series['ltft']!;
  if (!stft.empty && !ltft.empty) {
    final combined = stft.mean + ltft.mean;
    final absSum = combined.abs();
    if (absSum >= kTrimBadPct) {
      findings.add(Finding(
        FindingLevel.critical,
        'Топливная коррекция за пределами',
        'STFT+LTFT ${combined >= 0 ? '+' : ''}${_f(combined)}%: смесь далека от цели. '
            'Типичные причины: подсос воздуха, MAF, датчик O2, давление топлива.',
        'stft_mean=${_f(stft.mean)}% ltft_mean=${_f(ltft.mean)}% sum=${_f(combined)}%',
        'Сначала диагностика железа. Картами VE это не лечится.',
      ));
    } else if (absSum >= kTrimWarnPct) {
      findings.add(Finding(
        FindingLevel.warning,
        'Топливная коррекция повышена',
        'STFT+LTFT ${combined >= 0 ? '+' : ''}${_f(combined)}%: умеренное смещение смеси.',
        'stft_mean=${_f(stft.mean)}% ltft_mean=${_f(ltft.mean)}%',
        'Проверить подсос/фильтр и переснять лог с AFR.',
      ));
    }
  }

  // --- Прогрев -----------------------------------------------------------------------
  final ect = series['ect']!;
  final rpm = series['rpm']!;
  if (!ect.empty && !rpm.empty && ect.min < kWarmEctC && rpm.max > kColdRpm) {
    findings.add(Finding(
      FindingLevel.warning,
      'Нагрузка до прогрева',
      'Минимум ECT ${_f(ect.min)}°C при пике ${_f(rpm.max)} об/мин: двигатель крутили холодным. '
          'Детонационный запас на холодную иной — выводы по FBKC из этой зоны не переносить в карту.',
      'ect_min=${_f(ect.min)}°C rpm_max=${_f(rpm.max)}',
      'Валидный тюнинг-лог: ECT > 80°C стабильно.',
    ));
  }

  // --- Бортсеть ------------------------------------------------------------------------
  final batt = series['batt']!;
  if (!batt.empty && batt.min < kBattWarnV) {
    findings.add(Finding(
      FindingLevel.warning,
      'Просадка питания',
      'Минимум бортсети ${_f2(batt.min)}V: при <12V логируемые значения (особенно MAF/форсунки) '
          'уходят, а клоны ELM сыплют таймаутами.',
      'batt: min=${_f2(batt.min)}V mean=${_f2(batt.mean)}V n=${batt.count}',
      'Заряд/АКБ до повторного лога; адаптер снять с прикуривателя.',
    ));
  }

  // --- AFR против цели: беднение под нагрузкой (v0.9) ---
  final afrPairs = _pair(series['afr']!, series['cltgt']!);
  if (afrPairs.isNotEmpty) {
    final lean = afrPairs.where((p) => p.a - p.b >= 0.7).toList();
    final ratio = lean.length * 100 / afrPairs.length;
    var worst = 0.0;
    for (final p in lean) {
      final d = p.a - p.b;
      if (d > worst) worst = d;
    }
    if (ratio >= 15) {
      findings.add(Finding(
        ratio >= 35 || worst >= 1.5 ? FindingLevel.critical : FindingLevel.warning,
        'Смесь беднее цели',
        'В ${_f(ratio)}% сопоставленных точек AFR выше целевого на 0.7+ '
            '(максимум +${_f(worst)}). Беднение под наддувом — прямой путь к детонации '
            'и прогару; проверяйте топливоподачу до правок карт.',
        'pairs=${afrPairs.length} lean=${lean.length} (${_f(ratio)}%) worst=+${_f(worst)}',
        'Давление топлива, форсунки, MAF-скейлинг. Затем Map Lab -> fuel.',
      ));
    }
  }

  // --- Наддув: ошибка слежения и переброс (v0.9) ---
  final boostPairs = _pair(series['boost']!, series['boosttgt']!);
  if (boostPairs.length >= 5) {
    var sumSq = 0.0;
    var over = 0.0;
    var under = 0.0;
    for (final p in boostPairs) {
      final e = p.a - p.b;
      sumSq += e * e;
      if (e > over) over = e;
      if (e < under) under = e;
    }
    final rms = math.sqrt(sumSq / boostPairs.length);
    if (over >= 0.25 || rms >= 0.15) {
      findings.add(Finding(
        over >= 0.4 ? FindingLevel.critical : FindingLevel.warning,
        'Наддув не держит цель',
        'RMS ошибки ${_f2(rms)} бар, максимальный переброс +${_f2(over)} бар, '
            'просадка ${_f2(under)} бар. Переброс опаснее недобора: это незапланированная '
            'нагрузка на поршневую.',
        'pairs=${boostPairs.length} rms=${_f2(rms)} over=+${_f2(over)} under=${_f2(under)}',
        'Map Lab -> wgdc: снизить дьюти в зоне переброса, проверить вестгейт и шланги.',
      ));
    }
  }

  // --- Аномалии каналов: выбросы по z-score (v0.9.1) ---
  for (final entry in series.entries) {
    final s = entry.value;
    if (s.count < 30) continue;
    final mean = s.mean;
    var acc = 0.0;
    for (final p in s.samples) {
      acc += (p.v - mean) * (p.v - mean);
    }
    final std = math.sqrt(acc / s.count);
    if (std <= 0) continue;
    var outliers = 0;
    for (final p in s.samples) {
      if ((p.v - mean).abs() > 3.5 * std) outliers++;
    }
    final ratio = outliers * 100 / s.count;
    if (outliers >= 3 && ratio >= 1.0) {
      findings.add(Finding(
        FindingLevel.info,
        'Выбросы канала ${entry.key}',
        'Канал ${entry.key}: $outliers из ${s.count} значений (${_f(ratio)}%) ушли дальше '
            '3.5σ от среднего. Частая причина — единичные сбои ELM-клона или mute '
            'проблемных адресов, а не физика двигателя.',
        'channel=${entry.key} outliers=$outliers/${s.count} mean=${_f2(mean)} std=${_f2(std)}',
        'Если выбросов > 5% — переподключите адаптер и снимите лог повторно.',
      ));
    }
  }

  // --- Неравномерный холостой ход: признак пропусков/подсоса (v0.9.1) ---
  final idlePairs = _pair(series['rpm']!, series['tps']!)
      .where((p) => p.b <= 3 && p.a >= 500 && p.a <= 1500)
      .toList();
  if (idlePairs.length >= 10 && !ect.empty && ect.max >= 70) {
    final meanRpm = idlePairs.fold<double>(0, (a, p) => a + p.a) / idlePairs.length;
    var accR = 0.0;
    for (final p in idlePairs) {
      accR += (p.a - meanRpm) * (p.a - meanRpm);
    }
    final stdRpm = math.sqrt(accR / idlePairs.length);
    if (stdRpm >= 70) {
      findings.add(Finding(
        FindingLevel.warning,
        'Неравномерный холостой ход',
        'На тёплом двигателе при закрытом дросселе обороты гуляют σ=${_f(stdRpm)} об/мин '
            'вокруг ${_f(meanRpm)}. Похоже на пропуски воспламенения, подсос воздуха или '
            'грязный дроссель — чинить до тюнинга.',
        'idle_points=${idlePairs.length} rpm_mean=${_f(meanRpm)} rpm_std=${_f(stdRpm)}',
        'Свечи/катушки, проверка на подсос, LTFT на холостом.',
      ));
    }
  }

  // --- Полнота данных -------------------------------------------------------------------
  if (channels < 8) {
    final failed = attempts.values.where((s) => !s.good).length;
    findings.add(Finding(
      FindingLevel.info,
      'Мало каналов для выводов',
      'Активно каналов: $channels. Для тюнинга базово нужны RPM/LOAD/TIMING/FBKC/FLKC/IAM/AFR/MAF. '
          '${failed > 0 ? 'Ошибок чтения за сессию: $failed.' : ''}',
      'channels=$channels failed_attempts=$failed',
      'Включите полный набор PID и/или расширенный набор (CALID-гейт).',
    ));
  }

  findings.add(Finding(
    FindingLevel.info,
    'Сводка анализа',
    'Обработано $sampleCount значений по $channels каналам за ${_f(span)} с. '
        'Найдено проблем: ${findings.where((f) => f.level != FindingLevel.info).length}.',
    'samples=$sampleCount channels=$channels span=${_f(span)}s',
    findings.any((f) => f.level == FindingLevel.critical)
        ? 'Сначала критичные находки, потом Map Lab.'
        : 'Лог можно передавать в Map Lab для табличной коррекции.',
  ));

  return LogAnalysis(List<Finding>.unmodifiable(findings), sampleCount, span, channels);
}

// ======================= Сравнение логов «до/после» (v0.9.1) =======================

class ChannelDelta {
  const ChannelDelta(this.id, this.samplesBase, this.samplesAfter, this.meanBase,
      this.meanAfter, this.p95Base, this.p95After);
  final String id;
  final int samplesBase, samplesAfter;
  final double meanBase, meanAfter, p95Base, p95After;
  double get meanDelta => meanAfter - meanBase;
}

class LogComparison {
  const LogComparison(this.channels, this.verdict, this.knockBase, this.knockAfter);
  final List<ChannelDelta> channels;
  final String verdict;
  final int knockBase, knockAfter;
}

int _knockEvents(Map<String, List<PidSample>> history) {
  final fbkc = _series(history, const ['FBKC']);
  var n = 0;
  for (final s in fbkc.samples) {
    if (s.v <= kFbkcEventDeg) n++;
  }
  return n;
}

LogComparison compareLogs(
  Map<String, List<PidSample>> baseline,
  Map<String, List<PidSample>> after, {
  List<String> channels = const [
    'RPM', 'LOAD', 'TIMING', 'FBKC', 'FKL', 'IAM', 'AFR', 'STFT', 'LTFT', 'BOOST', 'ECT'
  ],
}) {
  final deltas = <ChannelDelta>[];
  for (final id in channels) {
    final a = _series(baseline, [id]);
    final b = _series(after, [id]);
    if (a.count < 5 || b.count < 5) continue;
    deltas.add(ChannelDelta(id, a.count, b.count, a.mean, b.mean, a.p95, b.p95));
  }
  deltas.sort((x, y) => y.meanDelta.abs().compareTo(x.meanDelta.abs()));
  final knockBase = _knockEvents(baseline);
  final knockAfter = _knockEvents(after);
  final verdict = knockAfter < knockBase
      ? 'Стало лучше: событий детонации $knockBase → $knockAfter'
      : knockAfter > knockBase
          ? 'Стало хуже: событий детонации $knockBase → $knockAfter — откатить последнюю правку'
          : 'Детонация без изменений ($knockBase события): правка нейтральна';
  return LogComparison(List<ChannelDelta>.unmodifiable(deltas), verdict, knockBase, knockAfter);
}
'''

FILES[".github/workflows/build.yml"] = r'''
name: build-apk
on:
  push:
    branches: [ main ]
  workflow_dispatch:
jobs:
  android:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-java@v4
        with:
          distribution: temurin
          java-version: "17"
      - uses: subosito/flutter-action@v2
        with:
          flutter-version: "3.35.4"
          channel: stable
      - run: flutter pub get
      - run: dart format --output=none --set-exit-if-changed lib test
      - run: flutter analyze --no-pub --no-fatal-infos
      - run: flutter test --no-pub
      - run: flutter build apk --release --no-pub
      - uses: actions/upload-artifact@v4
        with:
          name: ssm2-release-apk
          path: build/app/outputs/flutter-apk/app-release.apk
'''

FILES["test/app_test.dart"] = r'''
import 'package:flutter/material.dart';
import 'package:flutter/services.dart';
import 'package:flutter_test/flutter_test.dart';
import 'package:subaru_ssm2/main.dart';

void main() {
  TestWidgetsFlutterBinding.ensureInitialized();

  setUpAll(() {
    final messenger =
        TestDefaultBinaryMessengerBinding.instance.defaultBinaryMessenger;
    messenger.setMockMethodCallHandler(const MethodChannel('ssm2/system'),
        (MethodCall call) async {
      if (call.method == 'sdkInt') return 35;
      return null;
    });
    messenger.setMockMethodCallHandler(const MethodChannel('ssm2/spp'),
        (MethodCall call) async {
      if (call.method == 'paired') return <Map<String, String>>[];
      if (call.method == 'connect') return false;
      return null;
    });
    messenger.setMockMethodCallHandler(
        const MethodChannel('flutter.baseflow.com/permissions/methods'),
        (MethodCall call) async {
      if (call.method == 'requestPermissions') return <int, int>{};
      if (call.method == 'checkPermissionStatus') return 1;
      return null;
    });
  });

  testWidgets('приложение поднимается без краха', (tester) async {
    await tester.pumpWidget(const SsmApp());
    await tester.pump();
    expect(find.byType(MaterialApp), findsOneWidget);
  });
}
'''

FILES["analysis_options.yaml"] = r'''
analyzer:
  exclude:
    - vendor/**
    - transport_templates/**
  language:
    strict-casts: true
    strict-raw-types: true
'''

FILES["lib/transport_selected.dart"] = r'''
import 'bt_transport.dart';
import 'native_spp.dart';

/// Фасад выбранного транспорта: в 0.8 всегда нативный RFCOMM-канал.
class SelectedTransport extends NativeSppTransport {}

BtTransport createTransport() => SelectedTransport();
'''

def generate_project():
    for relative, content in FILES.items():
        if relative.endswith(".py"):
            ast.parse(content, filename=relative)
            compile(content, relative, "exec")
    if APP.exists():
        backup = APP.with_name(APP.name + f"_backup_{time.time_ns()}")
        APP.rename(backup)
        print("Предыдущий проект сохранен:", backup)
    APP.mkdir(parents=True)
    run([FLUTTER / "bin/flutter", "create", "--no-pub", "--platforms=android",
         "--org", "com.subaru", "--project-name", "subaru_ssm2", APP])
    for relative in ["lib", "test", "android/app/src/main/kotlin"]:
        shutil.rmtree(APP / relative, ignore_errors=True)
    for relative in ["android/build.gradle", "android/settings.gradle", "android/app/build.gradle"]:
        (APP / relative).unlink(missing_ok=True)
    for relative, content in FILES.items():
        write(relative, content)
    write("android/local.properties", f"sdk.dir={CFG['sdk']}\nflutter.sdk={CFG['flutter']}\n")
    write("build_config.json", json.dumps({**CFG, "bt_package": "native_spp", "revision": "0.9.2"}, indent=2))
    write("pid_catalog.json", json.dumps(PIDS, ensure_ascii=False, indent=2))
    shutil.copyfile(APP / "pubspec.template.yaml", APP / "pubspec.yaml")
    run([FLUTTER / "bin/flutter", "pub", "get"])
    print(f"\n[УСПЕХ v0.9] Создано {len(FILES)} файлов · native SPP · аналитика · алерты · CI")

generate_project()

Creating project ....
Wrote 35 files.

All done!
You can find general documentation for Flutter at: https://docs.flutter.dev/
Detailed API documentation is available at: https://api.flutter.dev/
If you prefer video documentation, consider: https://www.youtube.com/c/flutterdev

In order to run your application, type:

  $ cd .
  $ flutter run

Your application code is in ./lib/main.dart.


Resolving dependencies...
+ async 2.13.1
+ boolean_selector 2.1.2
+ characters 1.4.0 (1.4.1 available)
+ clock 1.1.2 (1.1.3 available)
+ collection 1.19.1
+ cross_file 0.3.5+2 (0.4.0 available)
+ crypto 3.0.7
+ fake_async 1.3.3
+ ffi 2.2.0
+ file 7.0.1
+ fixnum 1.1.1
+ flutter 0.0.0 from sdk flutter
+ flutter_test 0.0.0 from sdk flutter
+ flutter_web_plugins 0.0.0 from sdk flutter
+ leak_tracker 11.0.2
+ leak_tracker_flutter_testing 3.0.10
+ leak_tracker_testing 3.0.2
+ matcher 0.12.17 (0.12.20 available)
+ material_color_utilities 0.11.1 (0.13.1 available)
+ meta 1.16.0 (1.19.0 available)
+ mime 2.1.

In [ ]:
# @title 03 | Map Lab 0.9 - heatmap + 3D (ЖЕСТЫ ИСПРАВЛЕНЫ) + table zoom { display-mode: "form" }

import json
import re
import urllib.request
from pathlib import Path

CONFIG = Path("/content/ssm2_fixed_env.json")
if not CONFIG.exists():
    raise RuntimeError("Сначала выполните ячейку 1")
CFG = json.loads(CONFIG.read_text(encoding="utf-8"))
APP = Path(CFG["app"])
if not (APP / "pubspec.yaml").exists():
    raise RuntimeError("Проект не найден. Сначала выполните ячейку 2/3.")
FILES = {}

FILES["lib/maps_lab/maps_lab_core.dart"] = r"""
/// Map Lab core — чистый Dart без Flutter.
library;

import 'dart:convert';
import 'dart:typed_data';

class SafeExpr {
  static final RegExp _ok = RegExp(r'^[0-9xX\.\+\-\*/\(\)\s]+$');

  static double Function(double) compile(String? src) {
    if (src == null || src.trim().isEmpty) return (v) => v;
    final s = src.replaceAll('X', 'x');
    if (!_ok.hasMatch(s)) return (v) => v;
    try {
      final parser = _ExprParser(s);
      final node = parser.parse();
      return (x) => node(x);
    } catch (_) {
      return (v) => v;
    }
  }
}

class _ExprParser {
  _ExprParser(this.s);
  final String s;
  int i = 0;

  double Function(double) parse() {
    final n = _expr();
    _ws();
    if (i != s.length) throw FormatException('лишние символы: ${s.substring(i)}');
    return n;
  }

  void _ws() {
    while (i < s.length && s[i] == ' ') {
      i++;
    }
  }

  bool _eat(String ch) {
    _ws();
    if (i < s.length && s[i] == ch) {
      i++;
      return true;
    }
    return false;
  }

  double Function(double) _expr() {
    var node = _term();
    while (true) {
      if (_eat('+')) {
        final r = _term();
        final l = node;
        node = (x) => l(x) + r(x);
      } else if (_eat('-')) {
        final r = _term();
        final l = node;
        node = (x) => l(x) - r(x);
      } else {
        return node;
      }
    }
  }

  double Function(double) _term() {
    var node = _factor();
    while (true) {
      if (_eat('*')) {
        final r = _factor();
        final l = node;
        node = (x) => l(x) * r(x);
      } else if (_eat('/')) {
        final r = _factor();
        final l = node;
        node = (x) => r(x) == 0 ? double.nan : l(x) / r(x);
      } else {
        return node;
      }
    }
  }

  static final RegExp _num = RegExp(r'(\d+\.?\d*|\.\d+)');

  double Function(double) _factor() {
    _ws();
    if (_eat('-')) {
      final n = _factor();
      return (x) => -n(x);
    }
    if (_eat('+')) return _factor();
    if (_eat('(')) {
      final n = _expr();
      if (!_eat(')')) throw const FormatException('нет закрывающей скобки');
      return n;
    }
    _ws();
    if (i < s.length && s[i] == 'x') {
      i++;
      return (x) => x;
    }
    final m = _num.matchAsPrefix(s, i);
    if (m == null) throw FormatException('ожидалось число @ $i');
    i = m.end;
    final v = double.parse(m.group(0)!);
    return (x) => v;
  }
}

enum Stype { u8, i8, u16, i16, f32 }

class Scaling {
  Scaling({
    required this.name,
    required this.units,
    required this.type,
    required this.bigEndian,
    required this.to,
    required this.fr,
    required this.format,
  });

  factory Scaling.fromAttrs(String name, Map<String, String> a) {
    final st = switch (a['storagetype'] ?? 'uint8') {
      'int8' => Stype.i8,
      'uint16' => Stype.u16,
      'int16' => Stype.i16,
      'float' => Stype.f32,
      _ => Stype.u8,
    };
    return Scaling(
      name: name,
      units: a['units'] ?? '',
      type: st,
      bigEndian: (a['endian'] ?? 'big') == 'big',
      to: SafeExpr.compile(a['toexpr']),
      fr: SafeExpr.compile(a['frexpr']),
      format: a['format'] ?? '%.2f',
    );
  }

  final String name;
  final String units;
  final Stype type;
  final bool bigEndian;
  final double Function(double) to;
  final double Function(double) fr;
  final String format;

  int get bytes => switch (type) {
        Stype.f32 => 4,
        Stype.u16 || Stype.i16 => 2,
        _ => 1,
      };
}

class AxisDef {
  String name = '';
  String? scalingName;
  int? address;
  int? elements;
  Scaling? scaling;
}

class TableDef {
  TableDef(this.name);
  final String name;
  String? type;
  String? category;
  String? scalingName;
  int? dataAddress;
  Scaling? dataScaling;
  final List<AxisDef> axes = [];
}

class DefMeta {
  String xmlid = '';
  String internalIdAddress = '2000';
  String internalIdString = '';
  String ecuid = '';
  String checksumModule = '';
}

class _RawTable {
  _RawTable(this.attrs, this.children);
  final Map<String, String> attrs;
  final List<Map<String, String>> children;
}

class _RawDef {
  final meta = DefMeta();
  final includes = <String>[];
  final scalings = <String, Map<String, String>>{};
  final tables = <_RawTable>[];
}

class DefSet {
  DefSet._();
  final meta = DefMeta();
  final chain = <String>[];
  final scalings = <String, Scaling>{};
  final tables = <String, TableDef>{};

  static const _generic = {'x', 'y', 'z', ''};

  static final _attrRe = RegExp(r'([\w:-]+)="([^"]*)"');
  static final _tableTok = RegExp(r'<table\s[^>]*?>|</table>');
  static final _scalingRe = RegExp(r'<scaling\s+([^>]*?)/>');
  static final _romidRe = RegExp(r'<romid>([\s\S]*?)</romid>');
  static final _innerTag = RegExp(r'<(\w+)>([^<]*)</\1>');
  static final _includeRe = RegExp(r'<include>\s*([^<]+?)\s*</include>');

  static Map<String, String> _attrs(String tag) =>
      {for (final m in _attrRe.allMatches(tag)) m.group(1)!: m.group(2)!};

  static _RawDef _parse(String raw) {
    final def = _RawDef();
    final romid = _romidRe.firstMatch(raw);
    if (romid != null) {
      for (final m in _innerTag.allMatches(romid.group(1)!)) {
        final v = m.group(2)!.trim();
        if (v.isEmpty) continue;
        switch (m.group(1)) {
          case 'xmlid':
            def.meta.xmlid = v;
          case 'internalidaddress':
            def.meta.internalIdAddress = v;
          case 'internalidstring':
            def.meta.internalIdString = v;
          case 'ecuid':
            def.meta.ecuid = v;
          case 'checksummodule':
            def.meta.checksumModule = v;
        }
      }
    }
    for (final m in _includeRe.allMatches(raw)) {
      def.includes.add(m.group(1)!);
    }
    for (final m in _scalingRe.allMatches(raw)) {
      final a = _attrs(m.group(0)!);
      if (a['storagetype'] == 'bloblist') continue;
      final n = a['name'];
      if (n != null && n.isNotEmpty) def.scalings[n] = a;
    }

    var depth = 0;
    for (final m in _tableTok.allMatches(raw)) {
      final tok = m.group(0)!;
      if (tok == '</table>') {
        if (depth > 0) depth--;
        continue;
      }
      final selfClose = tok.endsWith('/>');
      final a = _attrs(tok);
      if (depth == 0) {
        def.tables.add(_RawTable(a, []));
        if (!selfClose) depth = 1;
      } else {
        def.tables.last.children.add(a);
      }
    }
    return def;
  }

  static DefSet build(Map<String, String> xmlById, String rootId) {
    final raw = <String, _RawDef>{};
    for (final e in xmlById.entries) {
      raw[e.key] = _parse(e.value);
    }

    final order = <String>[];
    final seen = <String>{};
    void walk(String id) {
      if (!seen.add(id)) return;
      for (final inc in raw[id]?.includes ?? const <String>[]) {
        if (raw.containsKey(inc)) walk(inc);
      }
      order.add(id);
    }

    walk(rootId);

    final set = DefSet._();
    set.chain.addAll(order);
    if (raw.containsKey(rootId)) {
      final m = raw[rootId]!.meta;
      set.meta
        ..xmlid = m.xmlid
        ..internalIdAddress = m.internalIdAddress
        ..internalIdString = m.internalIdString
        ..ecuid = m.ecuid
        ..checksumModule = m.checksumModule;
    }

    for (final id in order) {
      final d = raw[id]!;
      d.scalings.forEach((n, a) {
        set.scalings[n] = Scaling.fromAttrs(n, a);
      });
      for (final rt in d.tables) {
        final name = (rt.attrs['name'] ?? '').trim();
        if (name.isEmpty) continue;
        final t = set.tables.putIfAbsent(name, () => TableDef(name));
        for (final k in ['type', 'category', 'level', 'scaling']) {
          final v = rt.attrs[k];
          if (v != null && v.isNotEmpty) {
            switch (k) {
              case 'type':
                t.type = v;
              case 'category':
                t.category = v;
              case 'scaling':
                t.scalingName = v;
            }
          }
        }
        final addr = rt.attrs['address'];
        if (addr != null && addr.isNotEmpty) t.dataAddress = int.parse(addr, radix: 16);
        for (var i = 0; i < rt.children.length; i++) {
          while (t.axes.length <= i) {
            t.axes.add(AxisDef());
          }
          final a = t.axes[i];
          final ch = rt.children[i];
          final cn = (ch['name'] ?? '').trim();
          if (!_generic.contains(cn.toLowerCase())) a.name = cn;
          if ((ch['scaling'] ?? '').isNotEmpty) a.scalingName = ch['scaling'];
          if ((ch['elements'] ?? '').isNotEmpty) a.elements = int.parse(ch['elements']!);
          if ((ch['address'] ?? '').isNotEmpty) a.address = int.parse(ch['address']!, radix: 16);
        }
      }
    }

    for (final t in set.tables.values) {
      t.dataScaling = set.scalings[t.scalingName];
      for (final a in t.axes) {
        a.scaling = set.scalings[a.scalingName];
      }
    }
    return set;
  }

  bool isReadable3D(TableDef t) =>
      t.dataAddress != null &&
      t.dataScaling != null &&
      t.axes.length >= 2 &&
      t.axes[0].address != null &&
      t.axes[0].elements != null &&
      t.axes[0].scaling != null &&
      t.axes[1].address != null &&
      t.axes[1].elements != null &&
      t.axes[1].scaling != null;
}

class AxisVals {
  AxisVals(this.name, this.values, this.guess);
  final String name;
  final List<double> values;
  final String guess;
}

class MapGrid {
  MapGrid({
    required this.name,
    required this.kind,
    required this.units,
    required this.addr,
    required this.x,
    required this.y,
    required this.data,
  });
  final String name;
  final String kind;
  final String units;
  final int addr;
  final AxisVals x;
  final AxisVals y;
  final List<List<double>> data;

  int get rows => data.length;
  int get cols => data.isEmpty ? 0 : data[0].length;
  double get vmin => data.expand((r) => r).reduce((a, b) => a < b ? a : b);
  double get vmax => data.expand((r) => r).reduce((a, b) => a > b ? a : b);
}

class RomParser {
  RomParser(this.rom);
  final Uint8List rom;

  String readRomId(int addr) {
    if (addr < 0 || addr + 16 > rom.length) return '';
    final bytes = <int>[];
    for (var i = addr; i < addr + 16; i++) {
      final b = rom[i];
      if (b == 0) break;
      bytes.add(b);
    }
    return ascii.decode(bytes, allowInvalid: true).trim();
  }

  List<double> _read(Scaling sc, int addr, int count) {
    final size = sc.bytes;
    if (addr < 0 || addr + count * size > rom.length) {
      throw FormatException(
          'диапазон 0x${addr.toRadixString(16)} + $count×$size вне ROM (${rom.length} байт)');
    }
    final bd = ByteData.sublistView(rom, addr, addr + count * size);
    final en = sc.bigEndian ? Endian.big : Endian.little;
    final out = List<double>.filled(count, 0);
    for (var i = 0; i < count; i++) {
      final v = switch (sc.type) {
        Stype.u8 => bd.getUint8(i).toDouble(),
        Stype.i8 => bd.getInt8(i).toDouble(),
        Stype.u16 => bd.getUint16(i * 2, en).toDouble(),
        Stype.i16 => bd.getInt16(i * 2, en).toDouble(),
        Stype.f32 => bd.getFloat32(i * 4, en),
      };
      out[i] = sc.to(v);
    }
    return out;
  }

  static String guessAxis(List<double> v) {
    if (v.isEmpty) return '?';
    var lo = v.first, hi = v.first;
    for (final x in v) {
      if (x < lo) lo = x;
      if (x > hi) hi = x;
    }
    if (hi > 800) return 'rpm';
    if (hi > 50 && hi <= 600) return 'нм/у.е.';
    if (hi <= 5.5 && lo >= -0.5) return 'г/об·бар';
    if (hi <= 14 && lo >= 0) return 'вольты/%';
    return '?';
  }

  static String classify(String name, String units) {
    final n = name.toLowerCase();
    if (n.contains('wastegate duty')) return 'wgdc';
    if (n.contains('target boost')) return 'boost';
    if (n.contains('requested torque')) return 'torque';
    if (n.contains('knock correction')) return 'knockadv';
    if (n.contains('fueling') || n.contains('fuel')) return 'fuel';
    if (n.contains('timing')) return 'timing';
    return 'other';
  }

  double _round4(double v) => (v * 10000).roundToDouble() / 10000;

  MapGrid? extract(TableDef t) {
    if (t.dataAddress == null || t.dataScaling == null) return null;
    if (t.axes.length < 2) return null;
    final ax = t.axes[0], ay = t.axes[1];
    if (ax.address == null || ax.elements == null || ax.scaling == null) return null;
    if (ay.address == null || ay.elements == null || ay.scaling == null) return null;

    final xv = _read(ax.scaling!, ax.address!, ax.elements!);
    final yv = _read(ay.scaling!, ay.address!, ay.elements!);
    final cols = ax.elements!, rows = ay.elements!;
    final flat = _read(t.dataScaling!, t.dataAddress!, rows * cols);
    final grid = <List<double>>[
      for (var r = 0; r < rows; r++) [for (var c = 0; c < cols; c++) _round4(flat[r * cols + c])],
    ];
    return MapGrid(
      name: t.name,
      kind: classify(t.name, t.dataScaling!.units),
      units: t.dataScaling!.units,
      addr: t.dataAddress!,
      x: AxisVals(ax.name.isEmpty ? 'X' : ax.name, [for (final v in xv) _round4(v)], guessAxis(xv)),
      y: AxisVals(ay.name.isEmpty ? 'Y' : ay.name, [for (final v in yv) _round4(v)], guessAxis(yv)),
      data: grid,
    );
  }

  static const keyTables = [
    'Base Timing Primary Cruise',
    'Base Timing Primary Non-Cruise',
    'Primary Open Loop Fueling',
    'Target Boost_',
    'Initial Wastegate Duty_',
    'Max Wastegate Duty_',
    'Knock Correction Advance Max Non-Cruise',
    'Requested Torque A (Accelerator Pedal) SI-DRIVE Sport',
  ];

  Map<String, MapGrid> extractKeys(DefSet defs) {
    final out = <String, MapGrid>{};
    for (final n in keyTables) {
      final t = defs.tables[n];
      if (t == null) continue;
      try {
        final g = extract(t);
        if (g != null) out[n] = g;
      } catch (_) {}
    }
    return out;
  }
}
"""

# maps_lab_log — без Color (чтобы library оставался чистым). Heatmap — в page/3d.
FILES["lib/maps_lab/maps_lab_log.dart"] = r"""
/// Map Lab: лог и правила.
library;

import 'dart:convert';
import 'dart:math' as math;

import 'maps_lab_core.dart';

class LogData {
  final cols = <String, List<double?>>{};
  int get rows => cols.isEmpty ? 0 : cols.values.first.length;

  bool has(String k) => cols.containsKey(k);
  List<double?>? operator [](String k) => cols[k];

  static const aliases = <String, List<String>>{
    'time': ['time', 'timestamp', 'time s', 'elapsed'],
    'rpm': ['engine speed', 'rpm', 'engine speed rpm'],
    'load': [
      'engine load g/rev',
      'load_4b',
      'engine load 4-byte',
      'calculated load',
      'engine load',
      'load'
    ],
    'fbkc': ['feedback knock correction', 'fbkc'],
    'flkc': ['fine learning knock correction', 'fkl', 'fine learning knock advance', 'flkc'],
    'iam': ['iam', 'ignition advance multiplier'],
    'timing': ['total ignition timing', 'ignition timing', 'timing'],
    'afr': ['afr', 'a/f sensor #1', 'a/f sensor 1', 'air/fuel ratio', 'estimated afr', 'lambda'],
    'boost': ['manifold relative pressure', 'boost', 'boost_rel'],
    'tgt': ['target boost', 'boost_tgt'],
    'berr': ['boost error', 'boost_err'],
    'wgdc': ['primary wastegate duty', 'wastegate duty', 'wgdc', 'boost control solenoid duty'],
    'tq': ['requested torque', 'cl_target', 'demand torque'],
    'thr': ['throttle opening angle', 'throttle plate', 'throttle', 'throttle position'],
    'iat': ['intake air temperature', 'iat'],
    'ect': ['coolant temperature', 'ect', 'engine coolant temperature'],
    'loop': ['cl/ol', 'fueling status', 'closed loop', 'loop'],
  };

  static String _canon(String s) {
    var c = s.toLowerCase().trim();
    final p = c.indexOf('(');
    if (p > 0) c = c.substring(0, p);
    final b = c.indexOf('[');
    if (b > 0) c = c.substring(0, b);
    return c.replaceAll('*', '').replaceAll(RegExp(r'\s+'), ' ').trim();
  }

  static LogData parse(String text) {
    final head = text.substring(0, math.min(text.length, 4096));
    final sep = ';'.allMatches(head).length > ','.allMatches(head).length ? ';' : ',';
    final decComma = sep == ';' && ','.allMatches(head).length > '.'.allMatches(head).length;

    final lines = const LineSplitter().convert(text).where((l) => l.trim().isNotEmpty).toList();
    if (lines.isEmpty) return LogData();
    final header = lines.first.split(sep).map((h) => h.trim().replaceAll('"', '')).toList();

    final raw = List<List<double?>>.generate(header.length, (_) => []);
    final canonHead = header.map(_canon).toList();

    for (var li = 1; li < lines.length; li++) {
      final parts = lines[li].split(sep);
      for (var i = 0; i < header.length; i++) {
        if (i >= parts.length) {
          raw[i].add(null);
          continue;
        }
        var tok = parts[i].trim().replaceAll('"', '');
        if (decComma) tok = tok.replaceAll(',', '.');
        raw[i].add(double.tryParse(tok));
      }
    }

    final log = LogData();
    final usedNames = <String>{};
    for (final entry in aliases.entries) {
      final canon = entry.key;
      final variants = entry.value.map(_canon).toSet();
      for (var i = 0; i < header.length; i++) {
        if (usedNames.contains(header[i])) continue;
        if (variants.contains(canonHead[i])) {
          log.cols[canon] = raw[i];
          usedNames.add(header[i]);
          break;
        }
      }
    }

    final afr = log.cols['afr'];
    if (afr != null) {
      final vals = afr.whereType<double>().toList()..sort();
      if (vals.isNotEmpty && vals[vals.length ~/ 2] <= 2.2) {
        log.cols['afr'] = [for (final v in afr) v == null ? null : v * 14.7];
      }
    }
    final boost = log.cols['boost'], tgt = log.cols['tgt'];
    if (boost != null && tgt != null && log.cols['berr'] == null) {
      log.cols['berr'] = [
        for (var i = 0; i < boost.length; i++)
          (boost[i] != null && tgt[i] != null) ? boost[i]! - tgt[i]! : null,
      ];
    }
    return log;
  }
}

class LogHealth {
  final notes = <String>[];
  final missing = <String, List<String>>{};
  bool blockReady(String b) => (missing[b] ?? const []).isEmpty;
}

class LogAudit {
  static const _needFor = {
    'timing': ['rpm', 'load', 'fbkc', 'flkc', 'iat'],
    'fuel': ['rpm', 'load', 'afr', 'boost'],
    'wgdc': ['rpm', 'tq', 'berr'],
  };

  static const _hint = {
    'wgdc': 'Primary Wastegate Duty',
    'tq': 'Requested Torque',
    'thr': 'Throttle Opening Angle',
    'afr': 'A/F Sensor #1',
    'loop': 'CL/OL Fueling Status',
  };

  static LogHealth check(LogData log) {
    final h = LogHealth();
    h.notes.add('строк: ${log.rows}');

    double? maxOf(String k) {
      final c = log[k];
      if (c == null) return null;
      double? m;
      for (final v in c) {
        if (v != null && (m == null || v > m)) m = v;
      }
      return m;
    }

    double? minOf(String k) {
      final c = log[k];
      if (c == null) return null;
      double? m;
      for (final v in c) {
        if (v != null && (m == null || v < m)) m = v;
      }
      return m;
    }

    final mr = maxOf('rpm');
    if (mr != null) h.notes.add('RPM max: ${mr.toStringAsFixed(0)}');
    final iamMin = minOf('iam');
    if (iamMin != null) {
      h.notes.add('IAM min: ${iamMin.toStringAsFixed(2)}'
          '${iamMin < 0.99 ? ' — ЛОГ НЕ ГОДИТСЯ: сначала доучить ЭБУ' : ' — ок'}');
    }
    final mb = maxOf('boost');
    if (mb != null) h.notes.add('буст max: ${mb.toStringAsFixed(2)} бар');

    final t = log['time'];
    if (t != null) {
      final dts = <double>[];
      for (var i = 1; i < t.length; i++) {
        if (t[i] != null && t[i - 1] != null) dts.add(t[i]! - t[i - 1]!);
      }
      dts.sort();
      if (dts.isNotEmpty) {
        final med = dts[dts.length ~/ 2];
        final gaps = dts.where((d) => d > 0.3).length;
        h.notes.add('частота: медиана ${(1 / med).toStringAsFixed(1)} Гц · провалов >300 мс: $gaps');
        if (med > 0.25) h.notes.add('!! реже 4 Гц — сократите набор PID до 10–12');
      }
    }

    _needFor.forEach((block, req) {
      final miss = req.where((r) => !log.has(r)).toList();
      h.missing[block] = miss;
    });
    final allMiss = {for (final v in h.missing.values) ...v};
    for (final m in allMiss) {
      final hint = _hint[m];
      if (hint != null) h.notes.add('добавить в логгер PID: $hint');
    }
    return h;
  }
}

class CellNote {
  int n = 0;
  double? fbkc, flkc, iat, iam, afr, target, berr;
  String why = '';
}

class MapResult {
  MapResult(int rows, int cols)
      : delta = List.generate(rows, (_) => List.filled(cols, 0.0)),
        info = List.generate(rows, (_) => List<CellNote?>.filled(cols, null));
  final List<List<double>> delta;
  final List<List<CellNote?>> info;

  int get rows => delta.length;
  int get cols => delta.isNotEmpty ? delta[0].length : 0;

  int get dec {
    var c = 0;
    for (final r in delta) {
      for (final v in r) {
        if (v < 0) c++;
      }
    }
    return c;
  }

  int get inc {
    var c = 0;
    for (final r in delta) {
      for (final v in r) {
        if (v > 0) c++;
      }
    }
    return c;
  }
}

class AnalyzerConfig {
  const AnalyzerConfig({
    this.minN = 6,
    this.fbkcEvent = -1.5,
    this.flkcEvent = -1.5,
    this.timingStep = 0.5,
    this.timingMaxCut = -3.0,
    this.allowTimingAdd = false,
    this.afrErr = 0.40,
    this.afrMaxCut = -0.8,
    this.boostErr = 0.04,
    this.wgdcMaxDelta = 6.0,
    this.wotLoad = 2.2,
    this.targetBoostGain = 0.0,
  });
  final int minN;
  final double fbkcEvent, flkcEvent, timingStep, timingMaxCut;
  final bool allowTimingAdd;
  final double afrErr, afrMaxCut, boostErr, wgdcMaxDelta, wotLoad, targetBoostGain;
}

class Analyzer {
  Analyzer(this.cfg);
  final AnalyzerConfig cfg;

  double _roundStep(double v, double s) => (v / s).roundToDouble() * s;

  int _binIdx(List<double> axis, double v) {
    if (axis.length <= 1) return 0;
    var lo = 0, hi = axis.length - 1;
    while (lo < hi - 1) {
      final mid = (lo + hi) >> 1;
      if (v >= axis[mid]) {
        lo = mid;
      } else {
        hi = mid;
      }
    }
    final mid = (axis[lo] + axis[hi]) / 2;
    return v < mid ? lo : hi;
  }

  MapResult? analyzeMap(MapGrid g, LogData log) {
    final kind = g.kind;
    if (!{'timing', 'knockadv', 'fuel', 'wgdc', 'boost'}.contains(kind)) return null;

    final xIsRpm = g.x.guess == 'rpm' || (g.y.guess != 'rpm' && g.x.values.last > 800);
    final rpmAxis = xIsRpm ? g.x.values : g.y.values;
    final otherAxis = xIsRpm ? g.y.values : g.x.values;

    final need = switch (kind) {
      'timing' || 'knockadv' => ['rpm', 'load', 'fbkc', 'flkc'],
      'fuel' => ['rpm', 'load', 'afr', 'fbkc', 'flkc'],
      _ => ['rpm', 'tq', 'berr'],
    };
    if (need.any((k) => !log.has(k))) return null;

    final rpmCol = log['rpm']!;
    final otherCol = switch (kind) {
      'timing' || 'knockadv' || 'fuel' => log['load']!,
      _ => log['tq']!,
    };

    final rows = g.rows, cols = g.cols;
    final cnt = List<int>.filled(rows * cols, 0);
    final fbkcMin = List<double>.filled(rows * cols, 0);
    final flkcSum = List<double>.filled(rows * cols, 0);
    final flkcLateSum = List<double>.filled(rows * cols, 0);
    final flkcLateCnt = List<int>.filled(rows * cols, 0);
    final halfRows = (log.rows / 2).round();
    final iatMax = List<double>.filled(rows * cols, -999);
    final iamMin = List<double>.filled(rows * cols, 999);
    final afrSum = List<double>.filled(rows * cols, 0);
    final berrSum = List<double>.filled(rows * cols, 0);

    final fbkc = log['fbkc'], flkc = log['flkc'], iat = log['iat'];
    final iam = log['iam'], afr = log['afr'], berr = log['berr'];

    for (var i = 0; i < log.rows; i++) {
      final rv = rpmCol[i], ov = otherCol[i];
      if (rv == null || ov == null) continue;
      final ri = _binIdx(xIsRpm ? otherAxis : rpmAxis, xIsRpm ? ov : rv);
      final ci = _binIdx(xIsRpm ? rpmAxis : otherAxis, xIsRpm ? rv : ov);
      final idx = ri * cols + ci;
      cnt[idx]++;
      final f = fbkc?[i];
      if (f != null && (cnt[idx] == 1 || f < fbkcMin[idx])) fbkcMin[idx] = f;
      final fl = flkc?[i];
      if (fl != null) {
        flkcSum[idx] += fl;
        if (i >= halfRows) {
          flkcLateSum[idx] += fl;
          flkcLateCnt[idx]++;
        }
      }
      final it = iat?[i];
      if (it != null && it > iatMax[idx]) iatMax[idx] = it;
      final im = iam?[i];
      if (im != null && im < iamMin[idx]) iamMin[idx] = im;
      final a = afr?[i];
      if (a != null) afrSum[idx] += a;
      final be = berr?[i];
      if (be != null) berrSum[idx] += be;
    }

    final res = MapResult(rows, cols);

    void fanTiming(int ri, int ci, double d, AnalyzerConfig cfg) {
      const fan = <List<num>>[
        [-1, 0, 0.5],
        [1, 0, 0.5],
        [0, -1, 0.5],
        [0, 1, 0.5],
        [-1, -1, 0.25],
        [-1, 1, 0.25],
        [1, -1, 0.25],
        [1, 1, 0.25],
      ];
      for (final e in fan) {
        final rr = ri + e[0].toInt(), cc = ci + e[1].toInt();
        if (rr < 0 || rr >= rows || cc < 0 || cc >= cols) continue;
        if (res.delta[rr][cc] != 0) continue;
        final d2 = _roundStep(d * e[2].toDouble(), cfg.timingStep);
        if (d2 == 0) continue;
        res.delta[rr][cc] = d2;
        res.info[rr][cc] = CellNote()..why = 'веер от соседнего кластера: ${d2.toStringAsFixed(1)}°';
      }
    }

    for (var ri = 0; ri < rows; ri++) {
      for (var ci = 0; ci < cols; ci++) {
        final idx = ri * cols + ci;
        final n = cnt[idx];
        if (n < cfg.minN) continue;
        final note = CellNote()
          ..n = n
          ..fbkc = fbkcMin[idx]
          ..flkc = flkcSum[idx] / n
          ..iat = iatMax[idx] < -900 ? null : iatMax[idx]
          ..iam = iamMin[idx] > 900 ? null : iamMin[idx]
          ..afr = afrSum[idx] > 0 ? afrSum[idx] / n : null
          ..berr = berrSum[idx] != 0 ? berrSum[idx] / n : null;

        if (kind == 'timing' || kind == 'knockadv') {
          if (note.iam != null && note.iam! < 0.99) {
            res.info[ri][ci] = note..why = 'IAM=${note.iam!.toStringAsFixed(2)} — сначала доучить';
            continue;
          }
          final lateCount = flkcLateCnt[idx];
          final lateFlkc = lateCount > 0 ? flkcLateSum[idx] / lateCount : note.flkc!;
          final knock = math.min(note.fbkc!, math.min(note.flkc!, lateFlkc) * 1.4);
          if (knock <= cfg.fbkcEvent) {
            final d = math.max(cfg.timingMaxCut,
                math.min(-cfg.timingStep, _roundStep(knock * 0.6, cfg.timingStep)));
            res.delta[ri][ci] = d;
            fanTiming(ri, ci, d, cfg);
            res.info[ri][ci] = note
              ..why = 'детон-кластер · FBKC ${note.fbkc!.toStringAsFixed(1)}°, '
                  'FLKC ${note.flkc!.toStringAsFixed(1)}° (поздняя ${lateFlkc.toStringAsFixed(1)}°) · '
                  'снять ${d.abs().toStringAsFixed(1)}° здесь, соседи сглажены веером';
          } else if (res.delta[ri][ci] == 0 &&
              cfg.allowTimingAdd &&
              n >= 18 &&
              (note.iat == null || note.iat! <= 45)) {
            res.delta[ri][ci] = cfg.timingStep;
            res.info[ri][ci] = note..why = 'чисто, IAM=1.0 — опционально +0.5°';
          }
        } else if (kind == 'fuel') {
          final loadAxisValue = xIsRpm ? g.y.values[ri] : g.x.values[ci];
          if (loadAxisValue < cfg.wotLoad || note.afr == null) continue;
          var target = g.data[ri][ci];
          if (target < 9.0) target *= 14.7;
          note.target = target;
          final err = note.afr! - target;
          final knock = math.min(note.fbkc!, note.flkc!);
          if (err > cfg.afrErr) {
            final d = math.max(cfg.afrMaxCut, _roundStep(-err, 0.1));
            res.delta[ri][ci] = d;
            res.info[ri][ci] = note
              ..why = 'факт ${note.afr!.toStringAsFixed(2)} против цели ${target.toStringAsFixed(2)} '
                  '— обогатить на ${d.abs().toStringAsFixed(1)}; если не помогает — MAF/давление';
          } else if (knock <= cfg.fbkcEvent) {
            res.delta[ri][ci] = -0.3;
            res.info[ri][ci] = note..why = 'детон при цели ~совпадает — запас −0.3 AFR';
          }
        } else {
          final e = note.berr;
          if (e == null) continue;
          if (kind == 'wgdc') {
            if (e.abs() > cfg.boostErr) {
              final d = _roundStep(-e * 45, 1)
                  .clamp(-cfg.wgdcMaxDelta, cfg.wgdcMaxDelta - 1)
                  .toDouble();
              if (d != 0) {
                res.delta[ri][ci] = d;
                res.info[ri][ci] = note
                  ..why = '${e > 0 ? 'овербуст' : 'недобор'} ${e.toStringAsFixed(2)} бар '
                      '→ WGDC ${d > 0 ? '+' : ''}${d.toStringAsFixed(0)}%';
              }
            }
          } else {
            if (e > cfg.boostErr) {
              res.info[ri][ci] = note
                ..why = 'овербуст ${e.toStringAsFixed(2)} бар — чинить через WGDC/TD, не таргет';
            }
            if (cfg.targetBoostGain > 0) {
              res.delta[ri][ci] = cfg.targetBoostGain;
              res.info[ri][ci] = note
                ..why = 'политика +${cfg.targetBoostGain} бар (после фикса WGDC и чистого детон-лога)';
            }
          }
        }
      }
    }
    return res;
  }

  Map<String, MapResult> run(Map<String, MapGrid> maps, LogData log) {
    final out = <String, MapResult>{};
    for (final e in maps.entries) {
      final r = analyzeMap(e.value, log);
      if (r != null) out[e.key] = r;
    }
    return out;
  }
}
"""

FILES["lib/maps_lab/map3d_view.dart"] = r"""
/// 3D-визуализатор: тепловая карта + жесты, которые НЕ отдаёт ListView.
library;

import 'dart:math' as math;

import 'package:flutter/gestures.dart';
import 'package:flutter/material.dart';

import 'maps_lab_core.dart';
import 'maps_lab_log.dart';
import 'maps_lab_page.dart' show getHeatmapColor;

/// v0.9 FIX. Почему 3D не вращалась: вертикальный скролл страницы использует
/// VerticalDragGestureRecognizer со слопом ~18px, а pan внутри — ~36px. Скролл
/// выигрывал арену раньше, и карта не получала ни одного события. Старый хак
/// (rejectGesture -> acceptGesture) в Flutter 3.35 уже не спасает: арена к этому
/// моменту разрешена в пользу скролла. Решение — забирать жест себе сразу при
/// касании поверхности карты. Scale-распознаватель заодно даёт щипковый зум.
class _EagerScaleGestureRecognizer extends ScaleGestureRecognizer {
  @override
  void addAllowedPointer(PointerDownEvent event) {
    super.addAllowedPointer(event);
    resolve(GestureDisposition.accepted);
  }
}

class Map3DView extends StatefulWidget {
  const Map3DView({
    super.key,
    required this.grid,
    this.result,
    this.selRi,
    this.selCi,
    this.onCell,
  });

  final MapGrid grid;
  final MapResult? result;
  final int? selRi;
  final int? selCi;
  final void Function(int ri, int ci)? onCell;

  @override
  State<Map3DView> createState() => Map3DViewState();
}

class Map3DViewState extends State<Map3DView> with SingleTickerProviderStateMixin {
  double _yaw = 0.95;
  double _pitch = 0.55;
  double _zoom = 1.0;
  bool _auto = true;
  late final AnimationController _ticker;
  final _painter = _SurfacePainter();

  @override
  void initState() {
    super.initState();
    _ticker = AnimationController(vsync: this, duration: const Duration(seconds: 1))
      ..addListener(() {
        if (_auto) {
          _yaw += 0.003;
          setState(() {});
        }
      })
      ..repeat();
  }

  @override
  void dispose() {
    _ticker.dispose();
    super.dispose();
  }

  double _zoomAtStart = 1.0;

  void _onScaleStart(ScaleStartDetails _) {
    _auto = false;
    _zoomAtStart = _zoom;
  }

  void _onScaleUpdate(ScaleUpdateDetails d) {
    setState(() {
      if (d.pointerCount > 1) {
        _zoom = (_zoomAtStart * d.scale).clamp(0.55, 2.4);
      }
      _yaw += d.focalPointDelta.dx * 0.01;
      _pitch = (_pitch + d.focalPointDelta.dy * 0.008).clamp(0.12, 1.35);
    });
  }

  void _resetView() {
    setState(() {
      _yaw = 0.95;
      _pitch = 0.55;
      _zoom = 1.0;
      _auto = true;
    });
  }

  // Арена целиком отдана вращению, поэтому тап/двойной тап считаем сами.
  Offset? _downAt;
  DateTime _downTime = DateTime.fromMillisecondsSinceEpoch(0);
  DateTime _lastTapTime = DateTime.fromMillisecondsSinceEpoch(0);

  void _onPointerDown(PointerDownEvent e) {
    _downAt = e.localPosition;
    _downTime = DateTime.now();
  }

  void _onPointerUp(PointerUpEvent e) {
    final start = _downAt;
    _downAt = null;
    if (start == null) return;
    final moved = (e.localPosition - start).distance;
    final heldMs = DateTime.now().difference(_downTime).inMilliseconds;
    if (moved > 12 || heldMs > 400) return;
    final now = DateTime.now();
    if (now.difference(_lastTapTime).inMilliseconds < 320) {
      _lastTapTime = DateTime.fromMillisecondsSinceEpoch(0);
      _resetView();
      return;
    }
    _lastTapTime = now;
    final hit = _painter.pick(e.localPosition);
    final onCell = widget.onCell;
    if (hit != null && onCell != null) onCell(hit.$1, hit.$2);
  }

  @override
  Widget build(BuildContext context) {
    return Stack(
      children: [
        Listener(
          behavior: HitTestBehavior.opaque,
          onPointerDown: _onPointerDown,
          onPointerUp: _onPointerUp,
          child: RawGestureDetector(
            behavior: HitTestBehavior.opaque,
            gestures: <Type, GestureRecognizerFactory>{
              _EagerScaleGestureRecognizer:
                  GestureRecognizerFactoryWithHandlers<_EagerScaleGestureRecognizer>(
                () => _EagerScaleGestureRecognizer(),
                (_EagerScaleGestureRecognizer instance) {
                  instance
                    ..onStart = _onScaleStart
                    ..onUpdate = _onScaleUpdate;
                },
              ),
            },
            child: ClipRect(
              child: CustomPaint(
                painter: _painter
                  ..update(
                    grid: widget.grid,
                    result: widget.result,
                    yaw: _yaw,
                    pitch: _pitch,
                    zoom: _zoom,
                    selRi: widget.selRi,
                    selCi: widget.selCi,
                  ),
                size: Size.infinite,
              ),
            ),
          ),
        ),
        // кнопки зума 3D
        Positioned(
          right: 8,
          bottom: 8,
          child: Column(
            children: [
              _zBtn(Icons.add, () => setState(() => _zoom = (_zoom * 1.15).clamp(0.55, 2.4))),
              const SizedBox(height: 6),
              _zBtn(Icons.remove, () => setState(() => _zoom = (_zoom / 1.15).clamp(0.55, 2.4))),
              const SizedBox(height: 6),
              _zBtn(Icons.threed_rotation, () {
                setState(() {
                  _auto = !_auto;
                });
              }),
            ],
          ),
        ),
        const Positioned(
          left: 8,
          bottom: 8,
          child: Text('палец — поворот · щипок — зум · двойной тап — сброс',
              style: TextStyle(fontSize: 10, color: Colors.white38)),
        ),
      ],
    );
  }

  Widget _zBtn(IconData icon, VoidCallback onTap) {
    return Material(
      color: Colors.black54,
      shape: const CircleBorder(),
      child: InkWell(
        customBorder: const CircleBorder(),
        onTap: onTap,
        child: Padding(
          padding: const EdgeInsets.all(8),
          child: Icon(icon, size: 18, color: Colors.white),
        ),
      ),
    );
  }
}

class _Proj {
  _Proj(this.sx, this.sy, this.depth);
  final double sx, sy, depth;
}

class _Quad {
  _Quad(this.ri, this.ci, this.pts, this.depth, this.cx, this.cy, this.sideA, this.sideB);
  final int ri, ci;
  final List<Offset> pts;
  final List<Offset> sideA, sideB;
  final double depth, cx, cy;
}

class _SurfacePainter extends CustomPainter {
  MapGrid? grid;
  MapResult? result;
  double yaw = 0.9, pitch = 0.62, zoom = 1.0;
  int? selRi, selCi;
  final List<_Quad> _quads = [];

  void update({
    required MapGrid grid,
    MapResult? result,
    required double yaw,
    required double pitch,
    required double zoom,
    int? selRi,
    int? selCi,
  }) {
    this.grid = grid;
    this.result = result;
    this.yaw = yaw;
    this.pitch = pitch;
    this.zoom = zoom;
    this.selRi = selRi;
    this.selCi = selCi;
  }

  (int, int)? pick(Offset p) {
    _Quad? best;
    var bd = 1e9;
    for (final q in _quads) {
      final d = math.sqrt(math.pow(q.cx - p.dx, 2) + math.pow(q.cy - p.dy, 2));
      if (d < 28 && d < bd) {
        bd = d;
        best = q;
      }
    }
    return best == null ? null : (best.ri, best.ci);
  }

  @override
  void paint(Canvas canvas, Size size) {
    final g = grid;
    if (g == null || g.rows == 0 || g.cols == 0) return;
    _quads.clear();

    final rows = g.rows, cols = g.cols;
    final vmin = g.vmin, vmax = g.vmax;
    final scale = math.min(size.width, size.height) * 0.34 * zoom;
    final cx = size.width / 2, cy = size.height * 0.52;
    const cam = 3.4;

    _Proj proj(double x, double y, double z) {
      final x1 = x * math.cos(yaw) + z * math.sin(yaw);
      final z1 = -x * math.sin(yaw) + z * math.cos(yaw);
      final y2 = y * math.cos(pitch) - z1 * math.sin(pitch);
      final z2 = y * math.sin(pitch) + z1 * math.cos(pitch);
      final f = cam / (cam - z2);
      return _Proj(cx + x1 * scale * f, cy - y2 * scale * f, z2);
    }

    double hgt(double v) => 0.12 + ((v - vmin) / ((vmax - vmin) == 0 ? 1 : (vmax - vmin))) * 0.72;

    final floorPaint = Paint()
      ..color = const Color(0xFF1B2330)
      ..style = PaintingStyle.stroke
      ..strokeWidth = 1;
    final floor = [proj(-1, 0, -1), proj(1, 0, -1), proj(1, 0, 1), proj(-1, 0, 1)];
    final fp = Path()
      ..moveTo(floor[0].sx, floor[0].sy)
      ..lineTo(floor[1].sx, floor[1].sy)
      ..lineTo(floor[2].sx, floor[2].sy)
      ..lineTo(floor[3].sx, floor[3].sy)
      ..close();
    canvas.drawPath(fp, floorPaint);

    for (var ri = 0; ri < rows; ri++) {
      for (var ci = 0; ci < cols; ci++) {
        final gz0 = (ri / math.max(1, rows - 1)) * 2 - 1;
        final gz1 = rows > 1 ? ((ri + 1) / (rows - 1)) * 2 - 1 : gz0;
        final gx0 = (ci / math.max(1, cols - 1)) * 2 - 1;
        final gx1 = cols > 1 ? ((ci + 1) / (cols - 1)) * 2 - 1 : gx0;
        const scx = 0.96;
        final x0 = gx0 + ((gx1 - gx0) * (1 - scx)) / 2;
        final x1 = gx1 - ((gx1 - gx0) * (1 - scx)) / 2;
        final z0 = gz0 + ((gz1 - gz0) * (1 - scx)) / 2;
        final z1 = gz1 - ((gz1 - gz0) * (1 - scx)) / 2;
        final h = hgt(g.data[ri][ci]);
        final p = [proj(x0, h, z0), proj(x1, h, z0), proj(x1, h, z1), proj(x0, h, z1)];
        final depth = (p[0].depth + p[1].depth + p[2].depth + p[3].depth) / 4;
        final sideA = [proj(x0, h, z1), proj(x1, h, z1), proj(x1, 0, z1), proj(x0, 0, z1)];
        final sideB = [proj(x1, h, z0), proj(x1, h, z1), proj(x1, 0, z1), proj(x1, 0, z0)];
        _quads.add(_Quad(
          ri,
          ci,
          [for (final q in p) Offset(q.sx, q.sy)],
          depth,
          (p[0].sx + p[2].sx) / 2,
          (p[0].sy + p[2].sy) / 2,
          [for (final q in sideA) Offset(q.sx, q.sy)],
          [for (final q in sideB) Offset(q.sx, q.sy)],
        ));
      }
    }

    _quads.sort((a, b) => a.depth.compareTo(b.depth));

    final sidePaint = Paint()..color = const Color.fromARGB(235, 10, 14, 20);
    final edgePaint = Paint()
      ..style = PaintingStyle.stroke
      ..strokeWidth = 0.55
      ..color = const Color.fromARGB(120, 6, 8, 12);

    for (final q in _quads) {
      void poly(List<Offset> pts, Paint fill) {
        final path = Path()..moveTo(pts[0].dx, pts[0].dy);
        for (var i = 1; i < pts.length; i++) {
          path.lineTo(pts[i].dx, pts[i].dy);
        }
        path.close();
        canvas.drawPath(path, fill);
      }

      poly(q.sideA, sidePaint);
      poly(q.sideB, sidePaint);

      final v = g.data[q.ri][q.ci];
      final d = result?.delta[q.ri][q.ci] ?? 0;
      final sel = selRi == q.ri && selCi == q.ci;

      var cellColor = getHeatmapColor(v, vmin, vmax);
      if (sel) cellColor = Color.lerp(cellColor, Colors.white, 0.35)!;
      poly(q.pts, Paint()..color = cellColor);

      final topPath = Path()..moveTo(q.pts[0].dx, q.pts[0].dy);
      for (var i = 1; i < q.pts.length; i++) {
        topPath.lineTo(q.pts[i].dx, q.pts[i].dy);
      }
      topPath.close();

      if (sel) {
        canvas.drawPath(
            topPath,
            Paint()
              ..style = PaintingStyle.stroke
              ..strokeWidth = 2.0
              ..color = Colors.white);
      } else {
        canvas.drawPath(topPath, edgePaint);
      }

      if (d != 0) {
        canvas.drawCircle(Offset(q.cx, q.cy), 2.6, Paint()..color = Colors.lightBlueAccent);
      }

      // v0.9.1: маркер детонационного кластера прямо на поверхности карты.
      final note = result?.info[q.ri][q.ci];
      final knock = note != null && ((note.fbkc ?? 0) <= -1.0 || (note.flkc ?? 0) <= -1.0);
      if (knock) {
        canvas.drawCircle(
            Offset(q.cx, q.cy),
            4.4,
            Paint()
              ..style = PaintingStyle.stroke
              ..strokeWidth = 2.0
              ..color = const Color(0xFFFF5252));
        canvas.drawCircle(Offset(q.cx, q.cy), 1.7, Paint()..color = const Color(0xFFFF5252));
      }
    }
  }

  @override
  bool shouldRepaint(covariant _SurfacePainter oldDelegate) => true;
}
"""

FILES["lib/maps_lab/maps_lab_page.dart"] = r"""
/// Экран Map Lab: папки, локальные XML, heatmap, zoom 2D, 3D-rotate.
library;

import 'dart:convert';
import 'dart:io';

import 'package:flutter/material.dart';
import 'package:flutter/services.dart' show rootBundle;
import 'package:path_provider/path_provider.dart';
import 'package:permission_handler/permission_handler.dart';
import 'package:share_plus/share_plus.dart';

import 'map3d_view.dart';
import 'maps_lab_core.dart';
import 'maps_lab_log.dart';

/// Тепловая карта: зелёный → жёлтый → оранжевый → красный.
Color getHeatmapColor(double v, double min, double max) {
  if (max <= min) return const Color(0xFF4CAF50);
  final t = ((v - min) / (max - min)).clamp(0.0, 1.0);
  if (t < 0.33) {
    return Color.lerp(const Color(0xFF4CAF50), const Color(0xFFFFEB3B), t / 0.33)!;
  } else if (t < 0.66) {
    return Color.lerp(const Color(0xFFFFEB3B), const Color(0xFFFF9800), (t - 0.33) / 0.33)!;
  } else {
    return Color.lerp(const Color(0xFFFF9800), const Color(0xFFF44336), (t - 0.66) / 0.34)!;
  }
}

class MapLabPage extends StatefulWidget {
  const MapLabPage({super.key});

  @override
  State<MapLabPage> createState() => _MapLabPageState();
}

class _MapLabPageState extends State<MapLabPage> {
  static const _defKeys = ['A2TB100B', 'A2TB100K', '32BITBASE'];

  final _journal = <String>[];
  DefSet? _defs;
  Map<String, MapGrid>? _maps;
  LogData? _log;
  Map<String, MapResult>? _results;
  String? _mapName;
  int? _selRi, _selCi;
  bool _view3D = false;
  String _defsSource = '—';

  @override
  void initState() {
    super.initState();
    WidgetsBinding.instance.addPostFrameCallback((_) => _bootstrapDefs());
  }

  void _say(String s) {
    if (!mounted) return;
    setState(() => _journal.add('${TimeOfDay.now().format(context)}  $s'));
  }

  Future<Directory> _cacheDir() async {
    final docs = await getApplicationDocumentsDirectory();
    final dir = Directory('${docs.path}/defs');
    if (!dir.existsSync()) dir.createSync(recursive: true);
    return dir;
  }

  Future<Map<String, String>> _tryAssets() async {
    final out = <String, String>{};
    for (final k in _defKeys) {
      try {
        final s = await rootBundle.loadString('assets/defs/$k.xml');
        if (s.trim().length > 200) out[k] = s;
      } catch (_) {}
    }
    return out;
  }

  Future<Map<String, String>> _tryCache() async {
    final out = <String, String>{};
    final dir = await _cacheDir();
    for (final k in _defKeys) {
      final f = File('${dir.path}/$k.xml');
      if (f.existsSync() && f.lengthSync() > 200) {
        try {
          out[k] = await f.readAsString();
        } catch (_) {}
      }
    }
    return out;
  }

  Future<void> _saveCache(Map<String, String> xmlById) async {
    final dir = await _cacheDir();
    for (final e in xmlById.entries) {
      try {
        await File('${dir.path}/${e.key}.xml').writeAsString(e.value, flush: true);
      } catch (_) {}
    }
  }

  bool _applyDefs(Map<String, String> xmlById, String source) {
    if (xmlById.length < 3) return false;
    try {
      _defs = DefSet.build(xmlById, 'A2TB100B');
      _defsSource = source;
      _say('дефиниции [$source]: ${_defs!.chain.join(' → ')} · '
          'ecuid ${_defs!.meta.ecuid} · ${_defs!.tables.length} таблиц');
      return true;
    } catch (e) {
      _say('!! ошибка разбора дефиниций: $e');
      return false;
    }
  }

  Future<void> _bootstrapDefs() async {
    var xml = await _tryAssets();
    if (_applyDefs(xml, 'APK assets')) {
      await _saveCache(xml);
      setState(() {});
      return;
    }
    xml = await _tryCache();
    if (_applyDefs(xml, 'кэш телефона')) {
      setState(() {});
      return;
    }
    _say('!! дефиниций нет. Нажмите «0 · Дефиниции» и укажите папку с XML.');
    setState(() {});
  }

  Future<Map<String, String>> _scanDefsInDir(Directory root) async {
    final found = <String, File>{};
    bool matchName(String name, String key) {
      final n = name.toLowerCase();
      return n == '${key.toLowerCase()}.xml' || n.contains(key.toLowerCase());
    }

    void consider(File f) {
      final name = f.uri.pathSegments.last;
      for (final k in _defKeys) {
        if (found.containsKey(k)) continue;
        if (matchName(name, k) && f.lengthSync() > 200) found[k] = f;
      }
    }

    try {
      for (final e in root.listSync(followLinks: false)) {
        if (e is File && e.path.toLowerCase().endsWith('.xml')) consider(e);
      }
      for (final e in root.listSync(followLinks: false)) {
        if (e is! Directory) continue;
        try {
          for (final f in e.listSync(followLinks: false)) {
            if (f is File && f.path.toLowerCase().endsWith('.xml')) consider(f);
          }
        } catch (_) {}
      }
    } catch (_) {}

    final out = <String, String>{};
    for (final e in found.entries) {
      try {
        out[e.key] = await e.value.readAsString();
      } catch (_) {}
    }
    return out;
  }

  Future<void> _pickDefs() async {
    final result = await showModalBottomSheet<Object>(
      context: context,
      isScrollControlled: true,
      backgroundColor: Colors.transparent,
      builder: (ctx) => const _FilePickerSheet(
        extensions: ['.xml'],
        title: 'Дефиниции RomRaider (.xml)',
        allowPickFolderDefs: true,
      ),
    );
    if (result == null) return;

    var xml = <String, String>{};
    var source = 'локально';

    if (result is Directory) {
      xml = await _scanDefsInDir(result);
      source = 'папка ${result.path.split('/').last}';
      _say('сканирование ${result.path}: найдено ${xml.length}/3');
    } else if (result is File) {
      xml = await _scanDefsInDir(result.parent);
      if (xml.length < 3) {
        final name = result.uri.pathSegments.last.toUpperCase();
        for (final k in _defKeys) {
          if (name.contains(k)) {
            try {
              xml[k] = await result.readAsString();
            } catch (_) {}
          }
        }
      }
      source = 'файл+папка';
    } else if (result is Map) {
      xml = Map<String, String>.from(result);
    }

    if (xml.length < 3) {
      final miss = _defKeys.where((k) => !xml.containsKey(k)).join(', ');
      _say('!! не хватает XML: $miss');
      return;
    }

    if (_applyDefs(xml, source)) {
      await _saveCache(xml);
      setState(() {});
    }
  }

  Future<File?> _chooseFile(List<String> exts, String title) async {
    final r = await showModalBottomSheet<Object>(
      context: context,
      isScrollControlled: true,
      backgroundColor: Colors.transparent,
      builder: (ctx) => _FilePickerSheet(extensions: exts, title: title),
    );
    return r is File ? r : null;
  }

  Future<void> _pickRom() async {
    if (_defs == null) {
      _say('сначала загрузите дефиниции');
      return;
    }
    final f = await _chooseFile(['.bin', '.hex', '.rom'], 'Выберите прошивку (.bin)');
    if (f == null) return;
    try {
      final bytes = await f.readAsBytes();
      setState(() {
        _results = null;
        _selRi = _selCi = null;
      });
      final parser = RomParser(bytes);
      final idAddr = int.tryParse(_defs?.meta.internalIdAddress ?? '2000', radix: 16) ?? 0x2000;
      final romId = parser.readRomId(idAddr);
      final expected = _defs?.meta.internalIdString ?? '';
      final matched = expected.isEmpty || romId == expected;
      _say('ROM: ${f.uri.pathSegments.last} · ${(bytes.length / 1024).toStringAsFixed(0)} КБ · '
          'ID="$romId"${matched ? '' : '  (ожидался $expected)'}');
      _maps = parser.extractKeys(_defs!);
      _mapName = _maps!.keys.firstOrNull;
      _say('карт извлечено: ${_maps!.length} из ${RomParser.keyTables.length}');
      setState(() {});
    } catch (e) {
      _say('!! ошибка парсинга ROM: $e');
    }
  }

  Future<void> _pickLog() async {
    final f = await _chooseFile(['.csv', '.txt', '.log'], 'Выберите файл лога (.csv)');
    if (f == null) return;
    try {
      final text = utf8.decode(await f.readAsBytes(), allowMalformed: true);
      _log = LogData.parse(text);
      final health = LogAudit.check(_log!);
      _say('лог: ${f.uri.pathSegments.last} · ${_log!.rows} строк');
      for (final n in health.notes) {
        _say('   $n');
      }
      setState(() {});
    } catch (e) {
      _say('!! ошибка чтения лога: $e');
    }
  }

  void _runAnalysis() {
    if (_maps == null || _log == null) return;
    const cfg = AnalyzerConfig();
    _results = Analyzer(cfg).run(_maps!, _log!);
    var dec = 0, inc = 0;
    for (final r in _results!.values) {
      dec += r.dec;
      inc += r.inc;
    }
    _say('анализ: вердикты по ${_results!.length} картам · убавить $dec, прибавить $inc ячеек');
    setState(() {});
  }

  Future<void> _export() async {
    if (_maps == null || _results == null) return;
    final dir = await getApplicationDocumentsDirectory();
    final stamp = DateTime.now().toIso8601String().replaceAll(':', '-').split('.').first;
    final outDir = Directory('${dir.path}/maplab_$stamp')..createSync(recursive: true);
    final files = <XFile>[];
    final md = StringBuffer('# Map Lab A2TB100B · рекомендации\n');
    for (final e in _maps!.entries) {
      final g = e.value;
      final res = _results![e.key];
      if (res == null) continue;
      final head = '\t${g.x.values.map((v) => v.toStringAsFixed(2)).join('\t')}';
      final rows = <String>[
        for (var ri = 0; ri < g.rows; ri++)
          '${g.y.values[ri].toStringAsFixed(2)}\t${[
            for (var ci = 0; ci < g.cols; ci++)
              (g.data[ri][ci] + res.delta[ri][ci]).toStringAsFixed(3),
          ].join('\t')}',
      ];
      final tsv = '$head\n${rows.join('\n')}\n';
      final file = File('${outDir.path}/${e.key.replaceAll(RegExp(r'[/ ]'), '_')}_recommended.tsv')
        ..writeAsStringSync(tsv);
      files.add(XFile(file.path));
      md.writeln('\n## ${e.key} (${g.units})');
      for (var ri = 0; ri < g.rows; ri++) {
        for (var ci = 0; ci < g.cols; ci++) {
          final d = res.delta[ri][ci];
          if (d == 0) continue;
          final inf = res.info[ri][ci];
          md.writeln('- ${g.x.values[ci]} × ${g.y.values[ri]}: ${g.data[ri][ci]} → '
              '${(g.data[ri][ci] + d).toStringAsFixed(2)} (${d > 0 ? '+' : ''}$d)'
              '${inf != null && inf.why.isNotEmpty ? ' — ${inf.why}' : ''}');
        }
      }
    }
    final mf = File('${outDir.path}/recommendations.md')..writeAsStringSync(md.toString());
    files.insert(0, XFile(mf.path));
    _say('экспорт: ${files.length} файлов → ${outDir.path}');
    await Share.shareXFiles(files, text: 'Map Lab A2TB100B — рекомендации');
  }

  @override
  Widget build(BuildContext context) {
    final theme = Theme.of(context);
    final map = _maps == null ? null : _maps![_mapName];
    MapResult? res;
    if (map != null) res = _results?[map.name];

    // ВАЖНО: карта в Expanded, а не внутри ListView — иначе 3D не крутится.
    return Scaffold(
      appBar: AppBar(
        title: const Text('Map Lab · A2TB100B'),
        actions: [
          if (_results != null)
            IconButton(icon: const Icon(Icons.ios_share), onPressed: _export, tooltip: 'Экспорт'),
        ],
      ),
      body: Column(
        children: [
          Padding(
            padding: const EdgeInsets.fromLTRB(12, 10, 12, 0),
            child: Column(
              crossAxisAlignment: CrossAxisAlignment.start,
              children: [
                Wrap(
                  spacing: 8,
                  runSpacing: 8,
                  children: [
                    _stepBtn('0 · Дефиниции', Icons.folder_special, _pickDefs, ok: _defs != null),
                    _stepBtn('1 · Прошивка', Icons.memory, _pickRom, ok: _maps != null),
                    _stepBtn('2 · Лог', Icons.description, _pickLog, ok: _log != null),
                    _stepBtn('3 · Анализ', Icons.psychology,
                        (_maps != null && _log != null) ? _runAnalysis : null,
                        ok: _results != null),
                    _stepBtn('Экспорт', Icons.ios_share, _results != null ? _export : null),
                  ],
                ),
                const SizedBox(height: 6),
                Text(
                  _defs != null
                      ? 'XML: $_defsSource · ${_defs!.meta.xmlid}'
                      : 'Нужны A2TB100B.xml, A2TB100K.xml, 32BITBASE.xml',
                  style: theme.textTheme.bodySmall?.copyWith(
                    color: _defs != null ? Colors.greenAccent : Colors.amber,
                  ),
                ),
                if (_maps != null) ...[
                  const SizedBox(height: 8),
                  SizedBox(
                    height: 40,
                    child: ListView(
                      scrollDirection: Axis.horizontal,
                      children: [
                        for (final e in _maps!.entries)
                          Padding(
                            padding: const EdgeInsets.only(right: 8),
                            child: ChoiceChip(
                              selected: _mapName == e.key,
                              onSelected: (_) => setState(() {
                                _mapName = e.key;
                                _selRi = _selCi = null;
                              }),
                              label: Text(e.key, style: const TextStyle(fontSize: 11)),
                            ),
                          ),
                      ],
                    ),
                  ),
                  Row(
                    children: [
                      Expanded(
                        child: Text(
                          map == null
                              ? ''
                              : '${map.name} · ${map.rows}×${map.cols}',
                          style: theme.textTheme.bodySmall,
                          overflow: TextOverflow.ellipsis,
                        ),
                      ),
                      SegmentedButton<bool>(
                        segments: const [
                          ButtonSegment(value: false, icon: Icon(Icons.table_chart, size: 16), label: Text('2D')),
                          ButtonSegment(value: true, icon: Icon(Icons.threed_rotation, size: 16), label: Text('3D')),
                        ],
                        selected: {_view3D},
                        onSelectionChanged: (s) => setState(() => _view3D = s.first),
                      ),
                    ],
                  ),
                ],
              ],
            ),
          ),
          if (map != null)
            Expanded(
              flex: 5,
              child: Padding(
                padding: const EdgeInsets.fromLTRB(12, 8, 12, 4),
                child: Card(
                  clipBehavior: Clip.antiAlias,
                  child: _view3D
                      ? Map3DView(
                          grid: map,
                          result: res,
                          selRi: _selRi,
                          selCi: _selCi,
                          onCell: (ri, ci) => setState(() {
                            _selRi = ri;
                            _selCi = ci;
                          }),
                        )
                      : _Table2D(
                          grid: map,
                          result: res,
                          selRi: _selRi,
                          selCi: _selCi,
                          onCell: (ri, ci) => setState(() {
                            _selRi = ri;
                            _selCi = ci;
                          }),
                        ),
                ),
              ),
            ),
          if (map != null && _selRi != null && _selCi != null)
            Padding(
              padding: const EdgeInsets.symmetric(horizontal: 12),
              child: _inspector(map, res),
            ),
          if (map != null)
            Padding(
              padding: const EdgeInsets.fromLTRB(12, 4, 12, 0),
              child: Wrap(spacing: 12, runSpacing: 4, children: [
                _legend(const Color(0xFF4CAF50), 'низ'),
                _legend(const Color(0xFFFFEB3B), 'середина'),
                _legend(const Color(0xFFF44336), 'верх'),
                _legend(Colors.lightBlueAccent, 'правка'),
              ]),
            ),
          Expanded(
            flex: map == null ? 8 : 2,
            child: Padding(
              padding: const EdgeInsets.all(12),
              child: Column(
                crossAxisAlignment: CrossAxisAlignment.start,
                children: [
                  Text('Журнал', style: theme.textTheme.titleSmall),
                  const SizedBox(height: 6),
                  Expanded(
                    child: Container(
                      width: double.infinity,
                      padding: const EdgeInsets.all(10),
                      decoration: BoxDecoration(
                        color: theme.colorScheme.surfaceContainerHighest.withValues(alpha: 0.35),
                        borderRadius: BorderRadius.circular(8),
                      ),
                      child: ListView(
                        children: [
                          for (final j in _journal)
                            Text(j, style: const TextStyle(fontFamily: 'monospace', fontSize: 11)),
                        ],
                      ),
                    ),
                  ),
                ],
              ),
            ),
          ),
        ],
      ),
    );
  }

  Widget _stepBtn(String label, IconData icon, VoidCallback? onTap, {bool ok = false}) {
    return FilledButton.tonalIcon(
      onPressed: onTap,
      icon: Icon(ok ? Icons.check_circle : icon, size: 18),
      label: Text(label, style: const TextStyle(fontSize: 12)),
    );
  }

  Widget _legend(Color c, String t) => Row(mainAxisSize: MainAxisSize.min, children: [
        Container(width: 12, height: 8, decoration: BoxDecoration(color: c, borderRadius: BorderRadius.circular(2))),
        const SizedBox(width: 6),
        Text(t, style: const TextStyle(fontSize: 10.5)),
      ]);

  Widget _inspector(MapGrid g, MapResult? res) {
    final ri = _selRi!, ci = _selCi!;
    final v = g.data[ri][ci];
    final d = res?.delta[ri][ci] ?? 0;
    final inf = res?.info[ri][ci];
    final col = d < 0 ? const Color(0xFFFF5C5C) : d > 0 ? const Color(0xFFA8FF3E) : null;
    return Card(
      child: Padding(
        padding: const EdgeInsets.all(10),
        child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
          Text('Ячейка: ${g.x.name}=${g.x.values[ci]} · ${g.y.name}=${g.y.values[ri]}',
              style: const TextStyle(fontFamily: 'monospace', fontSize: 12)),
          const SizedBox(height: 4),
          Row(children: [
            Text(v.toStringAsFixed(2),
                style: const TextStyle(
                    fontSize: 18, fontFamily: 'monospace', decoration: TextDecoration.lineThrough)),
            const SizedBox(width: 8),
            const Icon(Icons.arrow_forward, size: 16),
            const SizedBox(width: 8),
            Text('${(v + d).toStringAsFixed(2)} ${g.units}',
                style: TextStyle(fontSize: 22, fontFamily: 'monospace', color: col)),
          ]),
          if (inf != null && inf.why.isNotEmpty)
            Padding(
              padding: const EdgeInsets.only(top: 4),
              child: Text(inf.why, style: const TextStyle(fontSize: 12)),
            ),
        ]),
      ),
    );
  }
}

class _FilePickerSheet extends StatefulWidget {
  const _FilePickerSheet({
    required this.extensions,
    required this.title,
    this.allowPickFolderDefs = false,
  });
  final List<String> extensions;
  final String title;
  final bool allowPickFolderDefs;

  @override
  State<_FilePickerSheet> createState() => _FilePickerSheetState();
}

class _FilePickerSheetState extends State<_FilePickerSheet> {
  Directory _currentDir = Directory('/storage/emulated/0');
  bool _hasPermission = false;
  List<FileSystemEntity> _items = [];

  @override
  void initState() {
    super.initState();
    _checkPermissionAndRefresh();
  }

  Future<void> _checkPermissionAndRefresh() async {
    var ok = false;
    try {
      ok = await Permission.manageExternalStorage.isGranted;
    } catch (_) {}
    if (!ok) {
      final docs = await getApplicationDocumentsDirectory();
      _currentDir = docs;
    } else if (!_currentDir.existsSync()) {
      _currentDir = Directory('/storage/emulated/0');
    }
    setState(() => _hasPermission = ok);
    _refreshList();
  }

  Future<void> _requestPermission() async {
    try {
      await Permission.manageExternalStorage.request();
    } catch (_) {}
    try {
      await openAppSettings();
    } catch (_) {}
    _checkPermissionAndRefresh();
  }

  void _refreshList() {
    try {
      if (!_currentDir.existsSync()) {
        setState(() => _items = []);
        return;
      }
      final list = _currentDir.listSync(followLinks: false);
      final dirs = <Directory>[];
      final files = <File>[];
      for (final e in list) {
        final name = e.path.split('/').last;
        if (name.startsWith('.')) continue;
        if (e is Directory) {
          dirs.add(e);
        } else if (e is File) {
          if (widget.extensions.any((x) => name.toLowerCase().endsWith(x))) files.add(e);
        }
      }
      dirs.sort((a, b) => a.path.toLowerCase().compareTo(b.path.toLowerCase()));
      files.sort((a, b) => b.lastModifiedSync().compareTo(a.lastModifiedSync()));
      setState(() => _items = [...dirs, ...files]);
    } catch (_) {
      setState(() => _items = []);
    }
  }

  void _goUp() {
    final parent = _currentDir.parent;
    if (parent.path.length >= 4) {
      setState(() => _currentDir = parent);
      _refreshList();
    }
  }

  void _goTo(String path) {
    final d = Directory(path);
    if (d.existsSync()) {
      setState(() => _currentDir = d);
      _refreshList();
    }
  }

  @override
  Widget build(BuildContext context) {
    return Container(
      height: MediaQuery.of(context).size.height * 0.85,
      decoration: const BoxDecoration(
        color: Color(0xFF141921),
        borderRadius: BorderRadius.vertical(top: Radius.circular(16)),
      ),
      child: Column(
        children: [
          Padding(
            padding: const EdgeInsets.symmetric(horizontal: 16, vertical: 12),
            child: Row(
              children: [
                Expanded(
                  child: Text(widget.title, style: const TextStyle(fontWeight: FontWeight.bold, fontSize: 16)),
                ),
                IconButton(icon: const Icon(Icons.close), onPressed: () => Navigator.of(context).pop()),
              ],
            ),
          ),
          if (!_hasPermission)
            Padding(
              padding: const EdgeInsets.symmetric(horizontal: 12),
              child: TextButton(onPressed: _requestPermission, child: const Text('РАЗРЕШИТЬ доступ к файлам')),
            ),
          SingleChildScrollView(
            scrollDirection: Axis.horizontal,
            padding: const EdgeInsets.symmetric(horizontal: 12, vertical: 4),
            child: Row(children: [
              _chip('Память', '/storage/emulated/0'),
              _chip('Downloads', '/storage/emulated/0/Download'),
              _chip('Telegram', '/storage/emulated/0/Telegram'),
              _chip('Documents', '/storage/emulated/0/Documents'),
            ]),
          ),
          Container(
            padding: const EdgeInsets.symmetric(horizontal: 8, vertical: 4),
            color: Colors.black26,
            child: Row(
              children: [
                IconButton(icon: const Icon(Icons.arrow_upward, size: 18), onPressed: _goUp),
                Expanded(
                  child: Text(_currentDir.path,
                      style: const TextStyle(fontFamily: 'monospace', fontSize: 11, color: Colors.cyanAccent),
                      overflow: TextOverflow.ellipsis),
                ),
                if (widget.allowPickFolderDefs)
                  FilledButton.tonal(
                    onPressed: () => Navigator.of(context).pop(_currentDir),
                    child: const Text('Взять XML из ЭТОЙ папки', style: TextStyle(fontSize: 11)),
                  ),
              ],
            ),
          ),
          const Divider(height: 1),
          Expanded(
            child: ListView.builder(
              itemCount: _items.length,
              itemBuilder: (ctx, i) {
                final item = _items[i];
                final name = item.path.split('/').last;
                if (item is Directory) {
                  return ListTile(
                    dense: true,
                    leading: const Icon(Icons.folder, color: Colors.amber),
                    title: Text(name),
                    onTap: () {
                      setState(() => _currentDir = item);
                      _refreshList();
                    },
                  );
                } else if (item is File) {
                  return ListTile(
                    dense: true,
                    leading: const Icon(Icons.insert_drive_file, color: Colors.lightBlueAccent),
                    title: Text(name, style: const TextStyle(fontFamily: 'monospace', fontSize: 13)),
                    onTap: () => Navigator.of(ctx).pop(item),
                  );
                }
                return const SizedBox.shrink();
              },
            ),
          ),
        ],
      ),
    );
  }

  Widget _chip(String label, String path) {
    return Padding(
      padding: const EdgeInsets.only(right: 6),
      child: ActionChip(
        visualDensity: VisualDensity.compact,
        label: Text(label, style: const TextStyle(fontSize: 11)),
        onPressed: () => _goTo(path),
      ),
    );
  }
}

/// 2D таблица с pinch-zoom и кнопками +/−.
class _Table2D extends StatefulWidget {
  const _Table2D({required this.grid, this.result, this.selRi, this.selCi, this.onCell});
  final MapGrid grid;
  final MapResult? result;
  final int? selRi, selCi;
  final void Function(int, int)? onCell;

  @override
  State<_Table2D> createState() => _Table2DState();
}

class _Table2DState extends State<_Table2D> {
  final _tc = TransformationController();
  static const _minS = 0.25;
  static const _maxS = 6.0;

  @override
  void dispose() {
    _tc.dispose();
    super.dispose();
  }

  void _zoomBy(double factor) {
    final m = _tc.value.clone();
    final s = m.getMaxScaleOnAxis();
    final next = (s * factor).clamp(_minS, _maxS);
    final f = next / s;
    // зум относительно центра видимой области
    final child = context.findRenderObject() as RenderBox?;
    if (child == null) {
      _tc.value = m.clone()..multiply(Matrix4.diagonal3Values(f, f, f));
      return;
    }
    final center = child.size.center(Offset.zero);
    final scene = _tc.toScene(center);
    m.multiply(Matrix4.translationValues(scene.dx, scene.dy, 0));
    m.multiply(Matrix4.diagonal3Values(f, f, f));
    m.multiply(Matrix4.translationValues(-scene.dx, -scene.dy, 0));
    _tc.value = m;
    setState(() {});
  }

  void _reset() {
    _tc.value = Matrix4.identity();
    setState(() {});
  }

  @override
  Widget build(BuildContext context) {
    final g = widget.grid;
    return Stack(
      children: [
        InteractiveViewer(
          transformationController: _tc,
          minScale: _minS,
          maxScale: _maxS,
          boundaryMargin: const EdgeInsets.all(200),
          constrained: false,
          panEnabled: true,
          scaleEnabled: true,
          child: Padding(
            padding: const EdgeInsets.all(8),
            child: Table(
              defaultColumnWidth: const IntrinsicColumnWidth(),
              children: [
                TableRow(children: [
                  _head('${g.y.guess}↓ ${g.x.guess}→'),
                  for (final v in g.x.values) _head(v.toStringAsFixed(0), accent: true),
                ]),
                for (var ri = 0; ri < g.rows; ri++)
                  TableRow(children: [
                    _head(g.y.values[ri].toStringAsFixed(2), accent: true),
                    for (var ci = 0; ci < g.cols; ci++) _cell(ri, ci),
                  ]),
              ],
            ),
          ),
        ),
        Positioned(
          right: 8,
          bottom: 8,
          child: Column(
            children: [
              _zBtn(Icons.add, () => _zoomBy(1.25)),
              const SizedBox(height: 6),
              _zBtn(Icons.remove, () => _zoomBy(1 / 1.25)),
              const SizedBox(height: 6),
              _zBtn(Icons.center_focus_strong, _reset),
            ],
          ),
        ),
        const Positioned(
          left: 8,
          bottom: 8,
          child: Text('pinch / кнопки = зум', style: TextStyle(fontSize: 10, color: Colors.white38)),
        ),
      ],
    );
  }

  Widget _zBtn(IconData icon, VoidCallback onTap) {
    return Material(
      color: Colors.black54,
      shape: const CircleBorder(),
      child: InkWell(
        customBorder: const CircleBorder(),
        onTap: onTap,
        child: Padding(
          padding: const EdgeInsets.all(8),
          child: Icon(icon, size: 18, color: Colors.white),
        ),
      ),
    );
  }

  Widget _head(String t, {bool accent = false}) => Container(
        padding: const EdgeInsets.symmetric(horizontal: 6, vertical: 4),
        decoration: BoxDecoration(border: Border.all(color: Colors.white12)),
        child: Text(t,
            style: TextStyle(
                fontSize: 9.5,
                fontFamily: 'monospace',
                color: accent ? const Color(0xCC4BE1FF) : Colors.white54)),
      );

  Widget _cell(int ri, int ci) {
    final g = widget.grid;
    final v = g.data[ri][ci];
    final d = widget.result?.delta[ri][ci] ?? 0;
    final sel = widget.selRi == ri && widget.selCi == ci;
    final heat = getHeatmapColor(v, g.vmin, g.vmax).withValues(alpha: 0.55);
    final hasFix = d != 0;

    return InkWell(
      onTap: () => widget.onCell?.call(ri, ci),
      child: Container(
        constraints: const BoxConstraints(minWidth: 48, minHeight: 40),
        padding: const EdgeInsets.symmetric(horizontal: 4, vertical: 3),
        decoration: BoxDecoration(
          color: heat,
          border: Border.all(
            color: sel
                ? Colors.white
                : hasFix
                    ? Colors.lightBlueAccent
                    : Colors.white12,
            width: sel ? 1.6 : (hasFix ? 1.2 : 0.5),
          ),
        ),
        child: Column(
          mainAxisAlignment: MainAxisAlignment.center,
          children: [
            Text(v.toStringAsFixed(2),
                style: const TextStyle(fontSize: 10, fontFamily: 'monospace', color: Colors.white)),
            if (hasFix)
              Text((v + d).toStringAsFixed(2),
                  style: const TextStyle(
                      fontSize: 11,
                      fontWeight: FontWeight.bold,
                      fontFamily: 'monospace',
                      color: Colors.lightBlueAccent)),
          ],
        ),
      ),
    );
  }
}
"""

FILES["test/maps_lab_test.dart"] = r"""
import 'dart:typed_data';
import 'package:flutter_test/flutter_test.dart';
import 'package:__PKG__/maps_lab/maps_lab_core.dart';
import 'package:__PKG__/maps_lab/maps_lab_log.dart';

void main() {
  group('SafeExpr', () {
    test('uint8 timing: (x*.3515625)-20', () {
      final f = SafeExpr.compile('(x*.3515625)-20');
      expect(f(100), closeTo(15.15625, 0.0001));
      expect(f(0), -20.0);
    });
    test('estimated AFR: 14.7/(1+x*.0078125)', () {
      final f = SafeExpr.compile('14.7/(1+x*.0078125)');
      expect(f(80), closeTo(9.0461, 0.001));
    });
    test('float identity + garbage guard', () {
      expect(SafeExpr.compile('x')(42.5), 42.5);
      expect(SafeExpr.compile('eval(x)+1')(3), 3);
    });
  });

  const baseXml = '''
<rom>
  <romid><xmlid>32BITBASE</xmlid></romid>
  <scaling name="Timing8" units="deg" toexpr="(x*.3515625)-20" frexpr="(x+20)/.3515625" format="%.2f" storagetype="uint8" endian="big"/>
  <scaling name="RPM" units="RPM" toexpr="x" frexpr="x" format="%.0f" storagetype="float" endian="big"/>
  <table name="Base Timing" category="Ignition" type="3D" level="4" scaling="Timing8">
    <table name="Engine Load" type="X Axis" elements="2" scaling="RPM"/>
    <table name="Engine Speed" type="Y Axis" elements="3" scaling="RPM"/>
  </table>
</rom>''';

  const derivedXml = '''
<rom>
  <romid><xmlid>TEST_ROM</xmlid><internalidaddress>2000</internalidaddress><internalidstring>TEST_ROM</internalidstring></romid>
  <include>32BITBASE</include>
  <table name="Base Timing" address="1000">
    <table name="X" address="2400" elements="4"/>
    <table name="Y" address="2500"/>
  </table>
</rom>''';

  test('merge include-цепочки: адрес у производного, скейлинг наследуется', () {
    final defs = DefSet.build({'32BITBASE': baseXml, 'TEST_ROM': derivedXml}, 'TEST_ROM');
    expect(defs.chain, ['32BITBASE', 'TEST_ROM']);
    final t = defs.tables['Base Timing']!;
    expect(t.dataAddress, 0x1000);
    expect(t.dataScaling!.units, 'deg');
    expect(t.axes[0].name, 'Engine Load');
    expect(t.axes[0].elements, 4);
    expect(t.axes[0].address, 0x2400);
    expect(t.axes[1].elements, 3);
    expect(defs.isReadable3D(t), isTrue);
  });

  test('ROM: декод float32-осей и uint8-данных big-endian', () {
    final rom = Uint8List(0x3000);
    final bd = ByteData.view(rom.buffer);
    for (var i = 0; i < 8; i++) {
      bd.setUint8(0x2000 + i, 'TEST_ROM'.codeUnitAt(i));
    }
    final xs = [1000.0, 2000.0, 3000.0, 4000.0];
    for (var i = 0; i < 4; i++) {
      bd.setFloat32(0x2400 + i * 4, xs[i], Endian.big);
    }
    final ys = [0.8, 1.6, 2.4];
    for (var i = 0; i < 3; i++) {
      bd.setFloat32(0x2500 + i * 4, ys[i], Endian.big);
    }
    for (var i = 0; i < 12; i++) {
      bd.setUint8(0x1000 + i, 100);
    }

    final defs = DefSet.build({'32BITBASE': baseXml, 'TEST_ROM': derivedXml}, 'TEST_ROM');
    final parser = RomParser(rom);
    expect(parser.readRomId(0x2000), 'TEST_ROM');
    final grid = parser.extract(defs.tables['Base Timing']!)!;
    expect(grid.rows, 3);
    expect(grid.cols, 4);
    expect(grid.x.values.last, 4000);
    expect(grid.y.values.first, closeTo(0.8, 0.001));
    expect(grid.data[2][3], closeTo(15.15625, 0.001));
    expect(grid.kind, 'timing');
  });

  group('LogData + Analyzer', () {
    LogData syntheticLog({required double fbkc}) {
      final rpm = <double?>[], load = <double?>[], f = <double?>[], fl = <double?>[];
      for (var i = 0; i < 600; i++) {
        rpm.add(4000 + (i % 80) - 40);
        load.add(3.0);
        f.add(fbkc);
        fl.add(fbkc * 0.4);
      }
      return LogData()
        ..cols['rpm'] = rpm
        ..cols['load'] = load
        ..cols['fbkc'] = f
        ..cols['flkc'] = fl;
    }

    MapGrid smallGrid() => MapGrid(
          name: 'Base Timing Primary Non-Cruise',
          kind: 'timing',
          units: 'deg',
          addr: 0,
          x: AxisVals('Engine Speed', [3000, 4000, 5000], 'rpm'),
          y: AxisVals('Engine Load', [2.0, 3.0], 'г/об·бар'),
          data: [
            [20, 18, 16],
            [16, 14, 12],
          ],
        );

    test('детон-кластер → отрицательная дельта в своей ячейке и веер сглаживания', () {
      final res = Analyzer(const AnalyzerConfig()).analyzeMap(smallGrid(), syntheticLog(fbkc: -4.0))!;
      final d = res.delta[1][1];
      expect(d, lessThan(0));
      expect(d, greaterThanOrEqualTo(-3.0));
      expect(res.delta[0][0], -0.5);
      expect(res.info[1][1]!.why, contains('детон'));
    });

    test('чистый лог при выключенном allowTimingAdd → ноль правок', () {
      final res = Analyzer(const AnalyzerConfig()).analyzeMap(smallGrid(), syntheticLog(fbkc: 0))!;
      final anyDelta = res.delta.expand((r) => r).any((v) => v != 0);
      expect(anyDelta, isFalse);
    });

    test('CSV: автоопределение ; и десятичной запятой, алиасы колонок', () {
      const csv = 'Time (s);Engine Speed (RPM);Feedback Knock Correction;Load_4B\r\n'
          '0,0;3000;0,0;3,00\r\n'
          '0,1;3100;-2,5;3,10\r\n';
      final log = LogData.parse(csv);
      expect(log['rpm']![1], 3100);
      expect(log['fbkc']![1], -2.5);
      expect(log['load']![1], 3.1);
    });
  });
}
"""

# ── pubspec / manifest / write ──────────────────────────────────────────────
pub = APP / "pubspec.yaml"
text = pub.read_text(encoding="utf-8")

if "assets/defs/" not in text:
    try:
        if re.search(r"(?m)^  assets:\s*$", text):
            text = re.sub(r"(?m)^(  assets:\s*\n)", r"\1    - assets/defs/\n", text, count=1)
        elif re.search(r"(?m)^flutter:\s*$", text):
            text = re.sub(r"(?m)^(flutter:\s*\n)", r"\1  assets:\n    - assets/defs/\n", text, count=1)
        else:
            text += "\nflutter:\n  uses-material-design: true\n  assets:\n    - assets/defs/\n"
        pub.write_text(text, encoding="utf-8")
        print("[pubspec] assets/defs добавлены")
    except Exception as e:
        print(f"[pubspec] {e}")

man = APP / "android/app/src/main/AndroidManifest.xml"
try:
    if man.exists():
        m = man.read_text(encoding="utf-8")
        if "MANAGE_EXTERNAL_STORAGE" not in m:
            m = m.replace(
                "<application",
                '    <uses-permission android:name="android.permission.MANAGE_EXTERNAL_STORAGE"/>\n    <application',
                1,
            )
            man.write_text(m, encoding="utf-8")
            print("[manifest] MANAGE_EXTERNAL_STORAGE добавлен")
except Exception as e:
    print(f"[manifest] {e}")

pkg_match = re.search(r"(?m)^name:\s*(\S+)", text)
PKG = pkg_match.group(1) if pkg_match else "subaru_ssm2_fixed"

for rel, src in FILES.items():
    dest = APP / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_text(src.replace("__PKG__", PKG) + "\n", encoding="utf-8")
    print("[dart]", rel)

RAW = "https://raw.githubusercontent.com/TD-D/SubaruDefs/Stable/ECUFlash/subaru%20metric"
DEFS = {
    "A2TB100B.xml": f"{RAW}/Legacy%20GT/A2TB100B.xml",
    "A2TB100K.xml": f"{RAW}/Legacy%20GT%20spec.B/A2TB100K.xml",
    "32BITBASE.xml": f"{RAW}/Bases/32BITBASE.xml",
}
defs_dir = APP / "assets/defs"
defs_dir.mkdir(parents=True, exist_ok=True)
for name, url in DEFS.items():
    dest = defs_dir / name
    if not dest.exists() or dest.stat().st_size < 1000:
        urllib.request.urlretrieve(url, dest)
    print("[asset]", name, f"{dest.stat().st_size/1024:.0f} КБ")

(APP / "lib/maplab_link.dart").write_text(
    "import 'maps_lab/maps_lab_page.dart';\n"
    "export 'maps_lab/maps_lab_page.dart' show MapLabPage;\n"
    "typedef MapLabTab = MapLabPage;\n",
    encoding="utf-8",
)
print("\nГотово: 3D-вращение (EagerPan) + зум 2D таблицы. Запускайте ячейку 3/3.")

[pubspec] assets/defs добавлены
[manifest] MANAGE_EXTERNAL_STORAGE добавлен
[dart] lib/maps_lab/maps_lab_core.dart
[dart] lib/maps_lab/maps_lab_log.dart
[dart] lib/maps_lab/map3d_view.dart
[dart] lib/maps_lab/maps_lab_page.dart
[dart] test/maps_lab_test.dart
[asset] A2TB100B.xml 2 КБ
[asset] A2TB100K.xml 28 КБ
[asset] 32BITBASE.xml 489 КБ

Готово: 3D-вращение (EagerPan) + зум 2D таблицы. Запускайте ячейку 3/3.


In [ ]:
# @title 04 | Map Lab UI (финальный layout): большая карта + скролл всей страницы { display-mode: "form" }
from pathlib import Path
import json, re

CONFIG = Path("/content/ssm2_fixed_env.json")
CFG = json.loads(CONFIG.read_text(encoding="utf-8"))
APP = Path(CFG["app"])
pub = (APP / "pubspec.yaml").read_text(encoding="utf-8")
PKG = re.search(r"(?m)^name:\s*(\S+)", pub).group(1)

PAGE = r'''
/// Экран Map Lab: крупная карта + прокрутка всей страницы (журнал виден).
library;

import 'dart:convert';
import 'dart:io';

import 'package:flutter/material.dart';
import 'package:flutter/services.dart' show rootBundle;
import 'package:path_provider/path_provider.dart';
import 'package:permission_handler/permission_handler.dart';
import 'package:share_plus/share_plus.dart';

import 'map3d_view.dart';
import 'maps_lab_core.dart';
import 'maps_lab_log.dart';

/// Тепловая карта: зелёный → жёлтый → оранжевый → красный.
Color getHeatmapColor(double v, double min, double max) {
  if (max <= min) return const Color(0xFF4CAF50);
  final t = ((v - min) / (max - min)).clamp(0.0, 1.0);
  if (t < 0.33) {
    return Color.lerp(const Color(0xFF4CAF50), const Color(0xFFFFEB3B), t / 0.33)!;
  } else if (t < 0.66) {
    return Color.lerp(const Color(0xFFFFEB3B), const Color(0xFFFF9800), (t - 0.33) / 0.33)!;
  } else {
    return Color.lerp(const Color(0xFFFF9800), const Color(0xFFF44336), (t - 0.66) / 0.34)!;
  }
}

class MapLabPage extends StatefulWidget {
  const MapLabPage({super.key});

  @override
  State<MapLabPage> createState() => _MapLabPageState();
}

class _MapLabPageState extends State<MapLabPage> {
  static const _defKeys = ['A2TB100B', 'A2TB100K', '32BITBASE'];
  /// Высота окна карты (крупнее, чем «сжатый» Expanded).
  static const double _mapHeight = 480;

  final _journal = <String>[];
  DefSet? _defs;
  Map<String, MapGrid>? _maps;
  LogData? _log;
  Map<String, MapResult>? _results;
  String? _mapName;
  int? _selRi, _selCi;
  bool _view3D = false;
  String _defsSource = '—';

  @override
  void initState() {
    super.initState();
    WidgetsBinding.instance.addPostFrameCallback((_) => _bootstrapDefs());
  }

  void _say(String s) {
    if (!mounted) return;
    setState(() => _journal.add('${TimeOfDay.now().format(context)}  $s'));
  }

  Future<Directory> _cacheDir() async {
    final docs = await getApplicationDocumentsDirectory();
    final dir = Directory('${docs.path}/defs');
    if (!dir.existsSync()) dir.createSync(recursive: true);
    return dir;
  }

  Future<Map<String, String>> _tryAssets() async {
    final out = <String, String>{};
    for (final k in _defKeys) {
      try {
        final s = await rootBundle.loadString('assets/defs/$k.xml');
        if (s.trim().length > 200) out[k] = s;
      } catch (_) {}
    }
    return out;
  }

  Future<Map<String, String>> _tryCache() async {
    final out = <String, String>{};
    final dir = await _cacheDir();
    for (final k in _defKeys) {
      final f = File('${dir.path}/$k.xml');
      if (f.existsSync() && f.lengthSync() > 200) {
        try {
          out[k] = await f.readAsString();
        } catch (_) {}
      }
    }
    return out;
  }

  Future<void> _saveCache(Map<String, String> xmlById) async {
    final dir = await _cacheDir();
    for (final e in xmlById.entries) {
      try {
        await File('${dir.path}/${e.key}.xml').writeAsString(e.value, flush: true);
      } catch (_) {}
    }
  }

  bool _applyDefs(Map<String, String> xmlById, String source) {
    if (xmlById.length < 3) return false;
    try {
      _defs = DefSet.build(xmlById, 'A2TB100B');
      _defsSource = source;
      _say('дефиниции [$source]: ${_defs!.chain.join(' → ')} · '
          'ecuid ${_defs!.meta.ecuid} · ${_defs!.tables.length} таблиц');
      return true;
    } catch (e) {
      _say('!! ошибка разбора дефиниций: $e');
      return false;
    }
  }

  Future<void> _bootstrapDefs() async {
    var xml = await _tryAssets();
    if (_applyDefs(xml, 'APK assets')) {
      await _saveCache(xml);
      setState(() {});
      return;
    }
    xml = await _tryCache();
    if (_applyDefs(xml, 'кэш телефона')) {
      setState(() {});
      return;
    }
    _say('!! дефиниций нет. Нажмите «0 · Дефиниции» и укажите папку с XML.');
    setState(() {});
  }

  Future<Map<String, String>> _scanDefsInDir(Directory root) async {
    final found = <String, File>{};
    bool matchName(String name, String key) {
      final n = name.toLowerCase();
      return n == '${key.toLowerCase()}.xml' || n.contains(key.toLowerCase());
    }

    void consider(File f) {
      final name = f.uri.pathSegments.last;
      for (final k in _defKeys) {
        if (found.containsKey(k)) continue;
        if (matchName(name, k) && f.lengthSync() > 200) found[k] = f;
      }
    }

    try {
      for (final e in root.listSync(followLinks: false)) {
        if (e is File && e.path.toLowerCase().endsWith('.xml')) consider(e);
      }
      for (final e in root.listSync(followLinks: false)) {
        if (e is! Directory) continue;
        try {
          for (final f in e.listSync(followLinks: false)) {
            if (f is File && f.path.toLowerCase().endsWith('.xml')) consider(f);
          }
        } catch (_) {}
      }
    } catch (_) {}

    final out = <String, String>{};
    for (final e in found.entries) {
      try {
        out[e.key] = await e.value.readAsString();
      } catch (_) {}
    }
    return out;
  }

  Future<void> _pickDefs() async {
    final result = await showModalBottomSheet<Object>(
      context: context,
      isScrollControlled: true,
      backgroundColor: Colors.transparent,
      builder: (ctx) => const _FilePickerSheet(
        extensions: ['.xml'],
        title: 'Дефиниции RomRaider (.xml)',
        allowPickFolderDefs: true,
      ),
    );
    if (result == null) return;

    var xml = <String, String>{};
    var source = 'локально';

    if (result is Directory) {
      xml = await _scanDefsInDir(result);
      source = 'папка ${result.path.split('/').last}';
      _say('сканирование ${result.path}: найдено ${xml.length}/3');
    } else if (result is File) {
      xml = await _scanDefsInDir(result.parent);
      if (xml.length < 3) {
        final name = result.uri.pathSegments.last.toUpperCase();
        for (final k in _defKeys) {
          if (name.contains(k)) {
            try {
              xml[k] = await result.readAsString();
            } catch (_) {}
          }
        }
      }
      source = 'файл+папка';
    } else if (result is Map) {
      xml = Map<String, String>.from(result);
    }

    if (xml.length < 3) {
      final miss = _defKeys.where((k) => !xml.containsKey(k)).join(', ');
      _say('!! не хватает XML: $miss');
      return;
    }

    if (_applyDefs(xml, source)) {
      await _saveCache(xml);
      setState(() {});
    }
  }

  Future<File?> _chooseFile(List<String> exts, String title) async {
    final r = await showModalBottomSheet<Object>(
      context: context,
      isScrollControlled: true,
      backgroundColor: Colors.transparent,
      builder: (ctx) => _FilePickerSheet(extensions: exts, title: title),
    );
    return r is File ? r : null;
  }

  Future<void> _pickRom() async {
    if (_defs == null) {
      _say('сначала загрузите дефиниции');
      return;
    }
    final f = await _chooseFile(['.bin', '.hex', '.rom'], 'Выберите прошивку (.bin)');
    if (f == null) return;
    try {
      final bytes = await f.readAsBytes();
      setState(() {
        _results = null;
        _selRi = _selCi = null;
      });
      final parser = RomParser(bytes);
      final idAddr = int.tryParse(_defs?.meta.internalIdAddress ?? '2000', radix: 16) ?? 0x2000;
      final romId = parser.readRomId(idAddr);
      final expected = _defs?.meta.internalIdString ?? '';
      final matched = expected.isEmpty || romId == expected;
      _say('ROM: ${f.uri.pathSegments.last} · ${(bytes.length / 1024).toStringAsFixed(0)} КБ · '
          'ID="$romId"${matched ? '' : '  (ожидался $expected)'}');
      _maps = parser.extractKeys(_defs!);
      _mapName = _maps!.keys.firstOrNull;
      _say('карт извлечено: ${_maps!.length} из ${RomParser.keyTables.length}');
      setState(() {});
    } catch (e) {
      _say('!! ошибка парсинга ROM: $e');
    }
  }

  Future<void> _pickLog() async {
    final f = await _chooseFile(['.csv', '.txt', '.log'], 'Выберите файл лога (.csv)');
    if (f == null) return;
    try {
      final text = utf8.decode(await f.readAsBytes(), allowMalformed: true);
      _log = LogData.parse(text);
      final health = LogAudit.check(_log!);
      _say('лог: ${f.uri.pathSegments.last} · ${_log!.rows} строк');
      for (final n in health.notes) {
        _say('   $n');
      }
      setState(() {});
    } catch (e) {
      _say('!! ошибка чтения лога: $e');
    }
  }

  void _runAnalysis() {
    if (_maps == null || _log == null) return;
    const cfg = AnalyzerConfig();
    _results = Analyzer(cfg).run(_maps!, _log!);
    var dec = 0, inc = 0;
    for (final r in _results!.values) {
      dec += r.dec;
      inc += r.inc;
    }
    _say('анализ: вердикты по ${_results!.length} картам · убавить $dec, прибавить $inc ячеек');
    setState(() {});
  }

  Future<void> _export() async {
    if (_maps == null || _results == null) return;
    final dir = await getApplicationDocumentsDirectory();
    final stamp = DateTime.now().toIso8601String().replaceAll(':', '-').split('.').first;
    final outDir = Directory('${dir.path}/maplab_$stamp')..createSync(recursive: true);
    final files = <XFile>[];
    final md = StringBuffer('# Map Lab A2TB100B · рекомендации\n');
    for (final e in _maps!.entries) {
      final g = e.value;
      final res = _results![e.key];
      if (res == null) continue;
      final head = '\t${g.x.values.map((v) => v.toStringAsFixed(2)).join('\t')}';
      final rows = <String>[
        for (var ri = 0; ri < g.rows; ri++)
          '${g.y.values[ri].toStringAsFixed(2)}\t${[
            for (var ci = 0; ci < g.cols; ci++)
              (g.data[ri][ci] + res.delta[ri][ci]).toStringAsFixed(3),
          ].join('\t')}',
      ];
      final tsv = '$head\n${rows.join('\n')}\n';
      final file = File('${outDir.path}/${e.key.replaceAll(RegExp(r'[/ ]'), '_')}_recommended.tsv')
        ..writeAsStringSync(tsv);
      files.add(XFile(file.path));
      // v0.9: снимок исходной таблицы ДО правок — страховка для отката.
      final origRows = <String>[
        for (var ri = 0; ri < g.rows; ri++)
          '${g.y.values[ri].toStringAsFixed(2)}\t${[
            for (var ci = 0; ci < g.cols; ci++) g.data[ri][ci].toStringAsFixed(3),
          ].join('\t')}',
      ];
      final origFile =
          File('${outDir.path}/${e.key.replaceAll(RegExp(r'[/ ]'), '_')}_original.tsv')
            ..writeAsStringSync('$head\n${origRows.join('\n')}\n');
      files.add(XFile(origFile.path));
      md.writeln('\n## ${e.key} (${g.units})');
      for (var ri = 0; ri < g.rows; ri++) {
        for (var ci = 0; ci < g.cols; ci++) {
          final d = res.delta[ri][ci];
          if (d == 0) continue;
          final inf = res.info[ri][ci];
          md.writeln('- ${g.x.values[ci]} × ${g.y.values[ri]}: ${g.data[ri][ci]} → '
              '${(g.data[ri][ci] + d).toStringAsFixed(2)} (${d > 0 ? '+' : ''}$d)'
              '${inf != null && inf.why.isNotEmpty ? ' — ${inf.why}' : ''}');
        }
      }
    }
    final mf = File('${outDir.path}/recommendations.md')..writeAsStringSync(md.toString());
    files.insert(0, XFile(mf.path));
    _say('экспорт: ${files.length} файлов → ${outDir.path}');
    await SharePlus.instance.share(
        ShareParams(files: files, text: 'Map Lab — рекомендации и снимки исходных карт'));
  }

  @override
  Widget build(BuildContext context) {
    final theme = Theme.of(context);
    final map = _maps == null ? null : _maps![_mapName];
    MapResult? res;
    if (map != null) res = _results?[map.name];

    // Весь экран снова в одном ListView — журнал всегда можно долистать.
    // Карта фиксированной большой высоты; 3D крутится через EagerPan внутри map3d_view.
    return Scaffold(
      appBar: AppBar(
        title: const Text('Map Lab · A2TB100B'),
        actions: [
          if (_results != null)
            IconButton(icon: const Icon(Icons.ios_share), onPressed: _export, tooltip: 'Экспорт'),
        ],
      ),
      body: ListView(
        padding: const EdgeInsets.fromLTRB(12, 10, 12, 24),
        children: [
          Wrap(
            spacing: 8,
            runSpacing: 8,
            children: [
              _stepBtn('0 · Дефиниции', Icons.folder_special, _pickDefs, ok: _defs != null),
              _stepBtn('1 · Прошивка', Icons.memory, _pickRom, ok: _maps != null),
              _stepBtn('2 · Лог', Icons.description, _pickLog, ok: _log != null),
              _stepBtn('3 · Анализ', Icons.psychology,
                  (_maps != null && _log != null) ? _runAnalysis : null,
                  ok: _results != null),
              _stepBtn('Экспорт', Icons.ios_share, _results != null ? _export : null),
            ],
          ),
          const SizedBox(height: 6),
          Text(
            _defs != null
                ? 'XML: $_defsSource · ${_defs!.meta.xmlid}'
                : 'Нужны A2TB100B.xml, A2TB100K.xml, 32BITBASE.xml',
            style: theme.textTheme.bodySmall?.copyWith(
              color: _defs != null ? Colors.greenAccent : Colors.amber,
            ),
          ),
          if (_maps != null) ...[
            const SizedBox(height: 10),
            SizedBox(
              height: 40,
              child: ListView(
                scrollDirection: Axis.horizontal,
                children: [
                  for (final e in _maps!.entries)
                    Padding(
                      padding: const EdgeInsets.only(right: 8),
                      child: ChoiceChip(
                        selected: _mapName == e.key,
                        onSelected: (_) => setState(() {
                          _mapName = e.key;
                          _selRi = _selCi = null;
                        }),
                        label: Text(e.key, style: const TextStyle(fontSize: 11)),
                      ),
                    ),
                ],
              ),
            ),
            const SizedBox(height: 6),
            Row(
              children: [
                Expanded(
                  child: Text(
                    map == null ? '' : '${map.name} · ${map.rows}×${map.cols}',
                    style: theme.textTheme.bodySmall,
                    overflow: TextOverflow.ellipsis,
                  ),
                ),
                SegmentedButton<bool>(
                  segments: const [
                    ButtonSegment(value: false, icon: Icon(Icons.table_chart, size: 16), label: Text('2D')),
                    ButtonSegment(value: true, icon: Icon(Icons.threed_rotation, size: 16), label: Text('3D')),
                  ],
                  selected: {_view3D},
                  onSelectionChanged: (s) => setState(() => _view3D = s.first),
                ),
              ],
            ),
          ],
          if (map != null) ...[
            const SizedBox(height: 8),
            SizedBox(
              height: _mapHeight,
              child: Card(
                clipBehavior: Clip.antiAlias,
                child: _view3D
                    ? Map3DView(
                        grid: map,
                        result: res,
                        selRi: _selRi,
                        selCi: _selCi,
                        onCell: (ri, ci) => setState(() {
                          _selRi = ri;
                          _selCi = ci;
                        }),
                      )
                    : _Table2D(
                        grid: map,
                        result: res,
                        selRi: _selRi,
                        selCi: _selCi,
                        onCell: (ri, ci) => setState(() {
                          _selRi = ri;
                          _selCi = ci;
                        }),
                      ),
              ),
            ),
            if (_selRi != null && _selCi != null) ...[
              const SizedBox(height: 8),
              _inspector(map, res),
            ],
            const SizedBox(height: 8),
            Wrap(spacing: 12, runSpacing: 4, children: [
              _legend(const Color(0xFF4CAF50), 'низ'),
              _legend(const Color(0xFFFFEB3B), 'середина'),
              _legend(const Color(0xFFF44336), 'верх'),
              _legend(Colors.lightBlueAccent, 'правка'),
            ]),
          ],
          const SizedBox(height: 16),
          Text('Журнал', style: theme.textTheme.titleSmall),
          const SizedBox(height: 6),
          Container(
            width: double.infinity,
            constraints: const BoxConstraints(minHeight: 180),
            padding: const EdgeInsets.all(10),
            decoration: BoxDecoration(
              color: theme.colorScheme.surfaceContainerHighest.withValues(alpha: 0.35),
              borderRadius: BorderRadius.circular(8),
            ),
            child: _journal.isEmpty
                ? const Text('— пока пусто —', style: TextStyle(color: Colors.white38, fontSize: 12))
                : Column(
                    crossAxisAlignment: CrossAxisAlignment.start,
                    children: [
                      for (final j in _journal)
                        Padding(
                          padding: const EdgeInsets.only(bottom: 2),
                          child: Text(j, style: const TextStyle(fontFamily: 'monospace', fontSize: 11)),
                        ),
                    ],
                  ),
          ),
          const SizedBox(height: 12),
          const Text(
            'Листайте экран вниз, чтобы читать журнал. '
            'На 3D-карте тяните пальцем — страница при этом не уезжает.',
            style: TextStyle(fontSize: 11, color: Colors.white38),
          ),
        ],
      ),
    );
  }

  Widget _stepBtn(String label, IconData icon, VoidCallback? onTap, {bool ok = false}) {
    return FilledButton.tonalIcon(
      onPressed: onTap,
      icon: Icon(ok ? Icons.check_circle : icon, size: 18),
      label: Text(label, style: const TextStyle(fontSize: 12)),
    );
  }

  Widget _legend(Color c, String t) => Row(mainAxisSize: MainAxisSize.min, children: [
        Container(width: 12, height: 8, decoration: BoxDecoration(color: c, borderRadius: BorderRadius.circular(2))),
        const SizedBox(width: 6),
        Text(t, style: const TextStyle(fontSize: 10.5)),
      ]);

  Widget _inspector(MapGrid g, MapResult? res) {
    final ri = _selRi!, ci = _selCi!;
    final v = g.data[ri][ci];
    final d = res?.delta[ri][ci] ?? 0;
    final inf = res?.info[ri][ci];
    final col = d < 0 ? const Color(0xFFFF5C5C) : d > 0 ? const Color(0xFFA8FF3E) : null;
    return Card(
      child: Padding(
        padding: const EdgeInsets.all(10),
        child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
          Text('Ячейка: ${g.x.name}=${g.x.values[ci]} · ${g.y.name}=${g.y.values[ri]}',
              style: const TextStyle(fontFamily: 'monospace', fontSize: 12)),
          const SizedBox(height: 4),
          Row(children: [
            Text(v.toStringAsFixed(2),
                style: const TextStyle(
                    fontSize: 18, fontFamily: 'monospace', decoration: TextDecoration.lineThrough)),
            const SizedBox(width: 8),
            const Icon(Icons.arrow_forward, size: 16),
            const SizedBox(width: 8),
            Text('${(v + d).toStringAsFixed(2)} ${g.units}',
                style: TextStyle(fontSize: 22, fontFamily: 'monospace', color: col)),
          ]),
          if (inf != null && inf.why.isNotEmpty)
            Padding(
              padding: const EdgeInsets.only(top: 4),
              child: Text(inf.why, style: const TextStyle(fontSize: 12)),
            ),
        ]),
      ),
    );
  }
}

class _FilePickerSheet extends StatefulWidget {
  const _FilePickerSheet({
    required this.extensions,
    required this.title,
    this.allowPickFolderDefs = false,
  });
  final List<String> extensions;
  final String title;
  final bool allowPickFolderDefs;

  @override
  State<_FilePickerSheet> createState() => _FilePickerSheetState();
}

class _FilePickerSheetState extends State<_FilePickerSheet> {
  Directory _currentDir = Directory('/storage/emulated/0');
  bool _hasPermission = false;
  List<FileSystemEntity> _items = [];

  @override
  void initState() {
    super.initState();
    _checkPermissionAndRefresh();
  }

  Future<void> _checkPermissionAndRefresh() async {
    var ok = false;
    try {
      ok = await Permission.manageExternalStorage.isGranted;
    } catch (_) {}
    if (!ok) {
      final docs = await getApplicationDocumentsDirectory();
      _currentDir = docs;
    } else if (!_currentDir.existsSync()) {
      _currentDir = Directory('/storage/emulated/0');
    }
    setState(() => _hasPermission = ok);
    _refreshList();
  }

  Future<void> _requestPermission() async {
    try {
      await Permission.manageExternalStorage.request();
    } catch (_) {}
    try {
      await openAppSettings();
    } catch (_) {}
    _checkPermissionAndRefresh();
  }

  void _refreshList() {
    try {
      if (!_currentDir.existsSync()) {
        setState(() => _items = []);
        return;
      }
      final list = _currentDir.listSync(followLinks: false);
      final dirs = <Directory>[];
      final files = <File>[];
      for (final e in list) {
        final name = e.path.split('/').last;
        if (name.startsWith('.')) continue;
        if (e is Directory) {
          dirs.add(e);
        } else if (e is File) {
          if (widget.extensions.any((x) => name.toLowerCase().endsWith(x))) files.add(e);
        }
      }
      dirs.sort((a, b) => a.path.toLowerCase().compareTo(b.path.toLowerCase()));
      files.sort((a, b) => b.lastModifiedSync().compareTo(a.lastModifiedSync()));
      setState(() => _items = [...dirs, ...files]);
    } catch (_) {
      setState(() => _items = []);
    }
  }

  void _goUp() {
    final parent = _currentDir.parent;
    if (parent.path.length >= 4) {
      setState(() => _currentDir = parent);
      _refreshList();
    }
  }

  void _goTo(String path) {
    final d = Directory(path);
    if (d.existsSync()) {
      setState(() => _currentDir = d);
      _refreshList();
    }
  }

  @override
  Widget build(BuildContext context) {
    return Container(
      height: MediaQuery.of(context).size.height * 0.85,
      decoration: const BoxDecoration(
        color: Color(0xFF141921),
        borderRadius: BorderRadius.vertical(top: Radius.circular(16)),
      ),
      child: Column(
        children: [
          Padding(
            padding: const EdgeInsets.symmetric(horizontal: 16, vertical: 12),
            child: Row(
              children: [
                Expanded(
                  child: Text(widget.title, style: const TextStyle(fontWeight: FontWeight.bold, fontSize: 16)),
                ),
                IconButton(icon: const Icon(Icons.close), onPressed: () => Navigator.of(context).pop()),
              ],
            ),
          ),
          if (!_hasPermission)
            Padding(
              padding: const EdgeInsets.symmetric(horizontal: 12),
              child: TextButton(onPressed: _requestPermission, child: const Text('РАЗРЕШИТЬ доступ к файлам')),
            ),
          SingleChildScrollView(
            scrollDirection: Axis.horizontal,
            padding: const EdgeInsets.symmetric(horizontal: 12, vertical: 4),
            child: Row(children: [
              _chip('Память', '/storage/emulated/0'),
              _chip('Downloads', '/storage/emulated/0/Download'),
              _chip('Telegram', '/storage/emulated/0/Telegram'),
              _chip('Documents', '/storage/emulated/0/Documents'),
            ]),
          ),
          Container(
            padding: const EdgeInsets.symmetric(horizontal: 8, vertical: 4),
            color: Colors.black26,
            child: Row(
              children: [
                IconButton(icon: const Icon(Icons.arrow_upward, size: 18), onPressed: _goUp),
                Expanded(
                  child: Text(_currentDir.path,
                      style: const TextStyle(fontFamily: 'monospace', fontSize: 11, color: Colors.cyanAccent),
                      overflow: TextOverflow.ellipsis),
                ),
                if (widget.allowPickFolderDefs)
                  FilledButton.tonal(
                    onPressed: () => Navigator.of(context).pop(_currentDir),
                    child: const Text('Взять XML из ЭТОЙ папки', style: TextStyle(fontSize: 11)),
                  ),
              ],
            ),
          ),
          const Divider(height: 1),
          Expanded(
            child: ListView.builder(
              itemCount: _items.length,
              itemBuilder: (ctx, i) {
                final item = _items[i];
                final name = item.path.split('/').last;
                if (item is Directory) {
                  return ListTile(
                    dense: true,
                    leading: const Icon(Icons.folder, color: Colors.amber),
                    title: Text(name),
                    onTap: () {
                      setState(() => _currentDir = item);
                      _refreshList();
                    },
                  );
                } else if (item is File) {
                  return ListTile(
                    dense: true,
                    leading: const Icon(Icons.insert_drive_file, color: Colors.lightBlueAccent),
                    title: Text(name, style: const TextStyle(fontFamily: 'monospace', fontSize: 13)),
                    onTap: () => Navigator.of(ctx).pop(item),
                  );
                }
                return const SizedBox.shrink();
              },
            ),
          ),
        ],
      ),
    );
  }

  Widget _chip(String label, String path) {
    return Padding(
      padding: const EdgeInsets.only(right: 6),
      child: ActionChip(
        visualDensity: VisualDensity.compact,
        label: Text(label, style: const TextStyle(fontSize: 11)),
        onPressed: () => _goTo(path),
      ),
    );
  }
}

/// 2D таблица с pinch-zoom и кнопками +/−.
class _Table2D extends StatefulWidget {
  const _Table2D({required this.grid, this.result, this.selRi, this.selCi, this.onCell});
  final MapGrid grid;
  final MapResult? result;
  final int? selRi, selCi;
  final void Function(int, int)? onCell;

  @override
  State<_Table2D> createState() => _Table2DState();
}

class _Table2DState extends State<_Table2D> {
  final _tc = TransformationController();
  static const _minS = 0.25;
  static const _maxS = 6.0;

  @override
  void dispose() {
    _tc.dispose();
    super.dispose();
  }

  void _zoomBy(double factor) {
    final m = _tc.value.clone();
    final s = m.getMaxScaleOnAxis();
    final next = (s * factor).clamp(_minS, _maxS);
    final f = next / s;
    final child = context.findRenderObject() as RenderBox?;
    if (child == null) {
      _tc.value = m.clone()..multiply(Matrix4.diagonal3Values(f, f, f));
      return;
    }
    final center = child.size.center(Offset.zero);
    final scene = _tc.toScene(center);
    m.multiply(Matrix4.translationValues(scene.dx, scene.dy, 0));
    m.multiply(Matrix4.diagonal3Values(f, f, f));
    m.multiply(Matrix4.translationValues(-scene.dx, -scene.dy, 0));
    _tc.value = m;
    setState(() {});
  }

  void _reset() {
    _tc.value = Matrix4.identity();
    setState(() {});
  }

  @override
  Widget build(BuildContext context) {
    final g = widget.grid;
    return Stack(
      children: [
        InteractiveViewer(
          transformationController: _tc,
          minScale: _minS,
          maxScale: _maxS,
          boundaryMargin: const EdgeInsets.all(200),
          constrained: false,
          panEnabled: true,
          scaleEnabled: true,
          child: Padding(
            padding: const EdgeInsets.all(8),
            child: Table(
              defaultColumnWidth: const IntrinsicColumnWidth(),
              children: [
                TableRow(children: [
                  _head('${g.y.guess}↓ ${g.x.guess}→'),
                  for (final v in g.x.values) _head(v.toStringAsFixed(0), accent: true),
                ]),
                for (var ri = 0; ri < g.rows; ri++)
                  TableRow(children: [
                    _head(g.y.values[ri].toStringAsFixed(2), accent: true),
                    for (var ci = 0; ci < g.cols; ci++) _cell(ri, ci),
                  ]),
              ],
            ),
          ),
        ),
        Positioned(
          right: 8,
          bottom: 8,
          child: Column(
            children: [
              _zBtn(Icons.add, () => _zoomBy(1.25)),
              const SizedBox(height: 6),
              _zBtn(Icons.remove, () => _zoomBy(1 / 1.25)),
              const SizedBox(height: 6),
              _zBtn(Icons.center_focus_strong, _reset),
            ],
          ),
        ),
        const Positioned(
          left: 8,
          bottom: 8,
          child: Text('pinch / кнопки = зум', style: TextStyle(fontSize: 10, color: Colors.white38)),
        ),
      ],
    );
  }

  Widget _zBtn(IconData icon, VoidCallback onTap) {
    return Material(
      color: Colors.black54,
      shape: const CircleBorder(),
      child: InkWell(
        customBorder: const CircleBorder(),
        onTap: onTap,
        child: Padding(
          padding: const EdgeInsets.all(8),
          child: Icon(icon, size: 18, color: Colors.white),
        ),
      ),
    );
  }

  Widget _head(String t, {bool accent = false}) => Container(
        padding: const EdgeInsets.symmetric(horizontal: 6, vertical: 4),
        decoration: BoxDecoration(border: Border.all(color: Colors.white12)),
        child: Text(t,
            style: TextStyle(
                fontSize: 9.5,
                fontFamily: 'monospace',
                color: accent ? const Color(0xCC4BE1FF) : Colors.white54)),
      );

  Widget _cell(int ri, int ci) {
    final g = widget.grid;
    final v = g.data[ri][ci];
    final d = widget.result?.delta[ri][ci] ?? 0;
    final sel = widget.selRi == ri && widget.selCi == ci;
    final heat = getHeatmapColor(v, g.vmin, g.vmax).withValues(alpha: 0.55);
    final hasFix = d != 0;

    return InkWell(
      onTap: () => widget.onCell?.call(ri, ci),
      child: Container(
        constraints: const BoxConstraints(minWidth: 48, minHeight: 40),
        padding: const EdgeInsets.symmetric(horizontal: 4, vertical: 3),
        decoration: BoxDecoration(
          color: heat,
          border: Border.all(
            color: sel
                ? Colors.white
                : hasFix
                    ? Colors.lightBlueAccent
                    : Colors.white12,
            width: sel ? 1.6 : (hasFix ? 1.2 : 0.5),
          ),
        ),
        child: Column(
          mainAxisAlignment: MainAxisAlignment.center,
          children: [
            Text(v.toStringAsFixed(2),
                style: const TextStyle(fontSize: 10, fontFamily: 'monospace', color: Colors.white)),
            if (hasFix)
              Text((v + d).toStringAsFixed(2),
                  style: const TextStyle(
                      fontSize: 11,
                      fontWeight: FontWeight.bold,
                      fontFamily: 'monospace',
                      color: Colors.lightBlueAccent)),
          ],
        ),
      ),
    );
  }
}
'''

dest = APP / "lib/maps_lab/maps_lab_page.dart"
dest.write_text(PAGE.replace("__PKG__", PKG) + "\n", encoding="utf-8")
print("[ok] maps_lab_page.dart: карта 480px + ListView (журнал долистывается)")
print("Дальше: ячейка 3/3 (сборка APK).")

[ok] maps_lab_page.dart: карта 480px + ListView (журнал долистывается)
Дальше: ячейка 3/3 (сборка APK).


In [ ]:
# @title 04.1 | FIX | SSM2 0.9 >> 0.10.2 — все исправления (вставить МЕЖДУ ячейками 04 и 05)
# Все 25 дефектов аудита + LOG-04 + одометр/расход + автозонд endian для 4-byte PID.
# 11 файлов. Идемпотентна, бэкап, самопроверка. После — ячейка 05 (26 тестов).
# Проверено: Flutter 3.35.4 / Dart 3.9.2 — analyze «No issues found», 26/26.

import json
from pathlib import Path
import re
import shutil
import time

APP = Path("/content/subaru_ssm2_fixed")

FILES = {}

FILES["android/app/src/main/kotlin/com/subaru/ssm2_fixed/MainActivity.kt"] = r'''
package com.subaru.ssm2_fixed

import android.bluetooth.BluetoothAdapter
import android.bluetooth.BluetoothManager
import android.bluetooth.BluetoothSocket
import android.content.Context
import android.content.Intent
import android.os.Build
import android.os.Handler
import android.os.Looper
import android.provider.Settings
import io.flutter.embedding.android.FlutterActivity
import io.flutter.embedding.engine.FlutterEngine
import io.flutter.plugin.common.EventChannel
import io.flutter.plugin.common.MethodChannel
import java.io.IOException
import java.util.UUID
import java.util.concurrent.CountDownLatch
import java.util.concurrent.Executors
import java.util.concurrent.TimeUnit

class MainActivity : FlutterActivity() {
    private val sppUuid: UUID = UUID.fromString("00001101-0000-1000-8000-00805F9B34FB")

    // v0.10 FIX (дедлок): команды исполняются на commandIo, а блокирующее чтение —
    // на ОТДЕЛЬНОМ потоке. В 0.9 read() занимал единственный поток executor'а,
    // и ни одна запись после успешного connect больше не выполнялась.
    private val commandIo = Executors.newSingleThreadExecutor()
    private val main = Handler(Looper.getMainLooper())

    private var socket: BluetoothSocket? = null
    @Volatile private var reading = false
    @Volatile private var readerThread: Thread? = null
    private var dataSink: EventChannel.EventSink? = null
    private var statusSink: EventChannel.EventSink? = null

    private fun btAdapter(): BluetoothAdapter? =
        (getSystemService(Context.BLUETOOTH_SERVICE) as? BluetoothManager)?.adapter

    // v0.10 FIX: адаптер выключен/запрещён — честная ошибка вместо тихо пустого списка.
    private fun requireAdapter(): BluetoothAdapter {
        val adapter = btAdapter() ?: throw IOException("Bluetooth недоступен на устройстве")
        if (!adapter.isEnabled) throw IOException("Bluetooth выключен: включите его в настройках")
        return adapter
    }

    private fun pairedDevices(): List<Map<String, String>> {
        val adapter = requireAdapter()
        return try {
            adapter.bondedDevices?.map { device ->
                mapOf("name" to (device.name ?: ""), "address" to device.address)
            } ?: emptyList()
        } catch (e: SecurityException) {
            throw IOException("Нет разрешения BLUETOOTH_CONNECT", e)
        }
    }

    private fun socketCandidates(address: String): List<BluetoothSocket> {
        val adapter = requireAdapter()
        val device = try {
            adapter.getRemoteDevice(address)
        } catch (e: IllegalArgumentException) {
            throw IOException("Некорректный MAC: $address")
        }
        try { adapter.cancelDiscovery() } catch (_: SecurityException) {}

        val list = mutableListOf<BluetoothSocket>()
        try { list.add(device.createInsecureRfcommSocketToServiceRecord(sppUuid)) } catch (_: Exception) {}
        try { list.add(device.createRfcommSocketToServiceRecord(sppUuid)) } catch (_: Exception) {}

        // Классический fallback для дешёвых ELM327-клонов: RFCOMM channel 1 через reflection.
        try {
            val method = device.javaClass.getMethod("createRfcommSocket", Int::class.javaPrimitiveType)
            val reflected = method.invoke(device, 1) as? BluetoothSocket
            if (reflected != null) list.add(reflected)
        } catch (_: Exception) {}

        if (list.isEmpty()) throw IOException("Не удалось создать RFCOMM сокет")
        return list
    }

    // v0.10 FIX: BluetoothSocket.connect() не имеет таймаута. Оборачиваем в поток
    // с latch: по истечении budget'а сокет закрывается, connect разблокируется.
    private fun connectWithTimeout(candidate: BluetoothSocket, timeoutMs: Long): Boolean {
        val latch = CountDownLatch(1)
        var failure: Exception? = null
        val worker = Thread {
            try {
                candidate.connect()
            } catch (e: Exception) {
                failure = e
            } finally {
                latch.countDown()
            }
        }
        worker.isDaemon = true
        worker.start()
        val finished = latch.await(timeoutMs, TimeUnit.MILLISECONDS)
        if (!finished) {
            try { candidate.close() } catch (_: Exception) {}
            return false
        }
        failure?.let { throw IOException(it.message ?: "RFCOMM connect failed") }
        return true
    }

    private fun openSocket(address: String): BluetoothSocket {
        var lastError: Exception? = null
        for (candidate in socketCandidates(address)) {
            try {
                Thread.sleep(120)
                // 12 секунд на кандидата вместо блокировки без предела (Dart ждёт 25 с на всё).
                if (connectWithTimeout(candidate, 12000)) return candidate
                lastError = IOException("timeout 12s")
            } catch (e: Exception) {
                lastError = e
                try { candidate.close() } catch (_: Exception) {}
            }
        }
        throw IOException(lastError?.message ?: "RFCOMM connect failed")
    }

    // v0.10 FIX: читатель живёт на своём потоке — командный executor остаётся свободным.
    private fun startReader(active: BluetoothSocket) {
        reading = true
        val thread = Thread {
            val buffer = ByteArray(4096)
            try {
                val input = active.inputStream
                while (reading) {
                    val count = input.read(buffer)
                    if (count < 0) break
                    if (count > 0) {
                        val chunk = buffer.copyOf(count)
                        main.post { dataSink?.success(chunk) }
                    }
                }
            } catch (_: Exception) {
                // Разрыв линии или intentional close: уходим в status=false только при обрыве.
            }
            val wasReading = reading
            reading = false
            closeSocket()
            if (wasReading) main.post { statusSink?.success(false) }
        }
        thread.isDaemon = true
        thread.name = "spp-reader"
        readerThread = thread
        thread.start()
    }

    private fun closeSocket() {
        val current = socket
        socket = null
        if (current != null) {
            try { current.close() } catch (_: Exception) {}
        }
    }

    private fun stopReaderAndSocket() {
        reading = false
        closeSocket() // закрытие сокета разблокирует read() потока читателя
        try { readerThread?.join(400) } catch (_: Exception) {}
        readerThread = null
    }

    override fun configureFlutterEngine(flutterEngine: FlutterEngine) {
        super.configureFlutterEngine(flutterEngine)
        val messenger = flutterEngine.dartExecutor.binaryMessenger

        MethodChannel(messenger, "ssm2/system").setMethodCallHandler { call, result ->
            when (call.method) {
                "sdkInt" -> result.success(Build.VERSION.SDK_INT)
                "bluetoothSettings" -> {
                    startActivity(Intent(Settings.ACTION_BLUETOOTH_SETTINGS))
                    result.success(null)
                }
                else -> result.notImplemented()
            }
        }

        EventChannel(messenger, "ssm2/spp_data").setStreamHandler(object : EventChannel.StreamHandler {
            override fun onListen(arguments: Any?, events: EventChannel.EventSink) { dataSink = events }
            override fun onCancel(arguments: Any?) { dataSink = null }
        })
        EventChannel(messenger, "ssm2/spp_status").setStreamHandler(object : EventChannel.StreamHandler {
            override fun onListen(arguments: Any?, events: EventChannel.EventSink) { statusSink = events }
            override fun onCancel(arguments: Any?) { statusSink = null }
        })

        MethodChannel(messenger, "ssm2/spp").setMethodCallHandler { call, result ->
            when (call.method) {
                "paired" -> commandIo.execute {
                    try {
                        val devices = pairedDevices()
                        main.post { result.success(devices) }
                    } catch (e: Exception) {
                        main.post { result.error("BT", e.message, null) }
                    }
                }
                "connect" -> {
                    val address = call.argument<String>("address")
                    if (address.isNullOrBlank()) {
                        result.error("ARG", "address required", null)
                    } else {
                        commandIo.execute {
                            try {
                                stopReaderAndSocket()
                                val opened = openSocket(address)
                                socket = opened
                                startReader(opened)
                                main.post {
                                    statusSink?.success(true)
                                    result.success(true)
                                }
                            } catch (e: Exception) {
                                stopReaderAndSocket()
                                main.post { result.error("CONNECT", e.message ?: "RFCOMM connect failed", null) }
                            }
                        }
                    }
                }
                "write" -> {
                    val data = call.argument<String>("data")
                    if (data == null) {
                        result.error("ARG", "data required", null)
                    } else {
                        // Свободный командный поток: запись больше не встаёт за читателем.
                        commandIo.execute {
                            try {
                                val current = socket ?: throw IOException("SPP disconnected")
                                current.outputStream.write(data.toByteArray(Charsets.US_ASCII))
                                current.outputStream.flush()
                                main.post { result.success(null) }
                            } catch (e: Exception) {
                                main.post { result.error("IO", e.message, null) }
                            }
                        }
                    }
                }
                "disconnect" -> commandIo.execute {
                    stopReaderAndSocket()
                    main.post { result.success(null) }
                }
                else -> result.notImplemented()
            }
        }
    }

    override fun onDestroy() {
        stopReaderAndSocket()
        dataSink = null
        statusSink = null
        commandIo.shutdownNow() // v0.10 FIX: раньше потоки executor'а жили до смерти процесса
        super.onDestroy()
    }
}
'''

FILES["lib/elm.dart"] = r'''
import 'dart:async';
import 'dart:collection';
import 'dart:typed_data';

import 'bt_transport.dart';
import 'pids.dart';
import 'protocol.dart';

class AsyncLock {
  Future<void> _tail = Future<void>.value();

  Future<T> run<T>(Future<T> Function() action) async {
    final previous = _tail;
    final gate = Completer<void>();
    _tail = gate.future;
    await previous;
    try {
      return await action();
    } finally {
      gate.complete();
    }
  }
}

class PidRead {
  PidRead(this.bytes, this.elapsedMs);
  final List<int> bytes;
  final int elapsedMs;
}

class _InitProfile {
  const _InitProfile(this.name, this.commands);
  final String name;
  final List<String> commands;
}

class ElmDriver {
  ElmDriver(this.transport) {
    _data = transport.data.listen(
      _onData,
      onError: (Object e) => _lost('RX ERROR: $e'),
    );
    _status = transport.status.listen((connected) {
      if (!connected) _lost('SPP disconnected');
    });
  }

  final BtTransport transport;
  final AsyncLock _lock = AsyncLock();
  late final StreamSubscription<Uint8List> _data;
  late final StreamSubscription<bool> _status;
  final Queue<String> trace = Queue<String>();

  Completer<String>? _pending;
  String _rx = '';
  bool _synchronized = false;
  bool _settling = false;
  bool _disposed = false;

  // v0.10 FIX (ELM-01): транзиентный таймаут клона не роняет sync сразу —
  // деградация только после двух таймаутов подряд. Раньше первый же TIMEOUT
  // снимал sync, и ветка retry становилась мёртвым кодом навсегда.
  int _consecutiveTimeouts = 0;
  // v0.10 FIX (ELM-02): успешный CAN-профиль кэшируем — реконнект идёт сразу в него.
  List<String>? _cachedProfileCommands;
  String _cachedProfileName = '';
  // v0.10 FIX (ELM-04): хвост буфера после '>' отдаём следующему обмену.
  String _rxTail = '';

  int timeoutMs = 1200;
  int tx = 0;
  int writeAccepted = 0;
  int rxBytes = 0;
  int prompts = 0;
  int timeouts = 0;

  String identity = '';
  String voltage = '';
  String calId = '';
  String syncNote = 'нет синхронизации';
  String activeProfile = '';
  String lastError = '';
  String? lastAddress;

  bool get ready => transport.connected && _synchronized && !_disposed;

  void log(String line) {
    trace.add('${DateTime.now().toIso8601String()} $line');
    while (trace.length > 500) {
      trace.removeFirst();
    }
  }

  void _setSync(bool value, String note) {
    _synchronized = value;
    syncNote = note;
    if (note.isNotEmpty) log('SYNC ${value ? 'UP' : 'DOWN'} $note');
  }

  void _lost(String reason) {
    lastError = reason;
    _setSync(false, reason);
    final p = _pending;
    if (p != null && !p.isCompleted) p.completeError(ReplyError(reason));
  }

  bool _isHarmlessIdle(String text) {
    final compact = text.toUpperCase().replaceAll(RegExp(r'[\r\n\t >]'), '');
    return compact.isEmpty || compact == 'OK';
  }

  void _onData(Uint8List bytes) {
    rxBytes += bytes.length;
    final text = String.fromCharCodes(bytes);
    log('RX ${visibleText(text)}');
    final p = _pending;
    if (p == null || p.isCompleted) {
      if (!_settling && !_isHarmlessIdle(text)) {
        log('IDLE_RX ${visibleText(text)}');
      }
      return;
    }

    _rx += text;
    if (_rx.length > 16384) {
      _lost('RX OVERFLOW');
      return;
    }

    final end = _rx.indexOf('>');
    if (end >= 0) {
      prompts++;
      final tail = _rx.substring(end + 1);
      if (tail.trim().isNotEmpty) log('TAIL ${visibleText(tail)}');
      if (!tail.contains('>')) _rxTail = tail; // v0.10
      p.complete(_rx.substring(0, end + 1));
    }
  }

  Future<String> _exchange(
    String command, {
    int? timeout,
    bool preserveSync = false,
  }) async {
    if (!preserveSync && !ready) {
      throw ReplyError('Нет синхронизации. Переподключите адаптер.');
    }
    if (preserveSync && !transport.connected) {
      throw ReplyError('Socket closed');
    }
    if (command.contains('\r') ||
        command.contains('\n') ||
        command.trim().isEmpty) {
      throw ArgumentError('One nonempty command is required');
    }

    final p = Completer<String>();
    _pending = p;
    _rx = _rxTail; // v0.10: возможный хвост склейки с предыдущим ответом
    _rxTail = '';
    final watch = Stopwatch()..start();
    tx++;
    log('TX ${visibleText('$command\r')}');
    try {
      final values = await Future.wait<Object>(
        [
          transport.write('$command\r').then<Object>((_) {
            writeAccepted++;
            log('WRITE_OK');
            return true;
          }),
          p.future,
        ],
        eagerError: true,
      ).timeout(Duration(milliseconds: timeout ?? timeoutMs));
      final raw = values[1] as String;
      if (preserveSync ? !transport.connected : !ready) {
        throw ReplyError('LINK_LOST');
      }
      log('PROMPT ${watch.elapsedMilliseconds}ms');
      _consecutiveTimeouts = 0; // v0.10
      return raw;
    } on TimeoutException {
      timeouts++;
      _consecutiveTimeouts++; // v0.10
      lastError = 'TIMEOUT $command';
      // v0.10: sync вниз только после второго таймаута подряд,
      // иначе retry в _exchangeForRead обречён заранее.
      if (!preserveSync && _consecutiveTimeouts >= 2) {
        _setSync(false, 'TIMEOUT x$_consecutiveTimeouts');
      }
      log('TIMEOUT ${watch.elapsedMilliseconds}ms partial=${visibleText(_rx)}');
      if (!p.isCompleted) p.complete('');
      throw ReplyError('TIMEOUT: reconnect required');
    } catch (e) {
      lastError = '$e';
      if (!preserveSync) _setSync(false, '$e');
      if (!p.isCompleted) p.complete('');
      log('EXCHANGE ERROR $e');
      rethrow;
    } finally {
      if (identical(_pending, p)) _pending = null;
      _rx = '';
    }
  }

  Future<void> _runInitCommands(List<String> commands) async {
    for (final command in commands) {
      try {
        await _exchange(command, timeout: 1800, preserveSync: true);
      } catch (e) {
        log('INIT WARN $command: $e');
      }
      await Future<void>.delayed(const Duration(milliseconds: 60));
    }
  }

  Future<int?> _probeAddress(int address) async {
    final command = readAddressCommand(address);
    try {
      final raw = await _exchange(
        command,
        timeout: timeoutMs + 800,
        preserveSync: true,
      );
      final value = parseAddressReply(raw, command);
      log('PROBE OK 0x${hexAddress(address)} -> ${hex2(value)}');
      return value;
    } catch (e) {
      log('PROBE FAIL 0x${hexAddress(address)}: $e');
      return null;
    }
  }

  Future<bool> _probeProfiles() async {
    const profiles = <_InitProfile>[
      _InitProfile('11bit/caf1/7E0→7E8', <String>[
        'ATH0',
        'ATCAF1',
        'ATAL',
        'ATSP6',
        'ATSH 7E0',
        'ATCRA 7E8',
      ]),
      _InitProfile('11bit/caf0/7E0→7E8', <String>[
        'ATH0',
        'ATCAF0',
        'ATAL',
        'ATSP6',
        'ATSH 7E0',
        'ATCRA 7E8',
      ]),
      _InitProfile('11bit/caf1/7DF→7E8', <String>[
        'ATH0',
        'ATCAF1',
        'ATAL',
        'ATSP6',
        'ATSH 7DF',
        'ATCRA 7E8',
      ]),
      _InitProfile('29bit/caf1/18DA10F1', <String>[
        'ATH0',
        'ATCAF1',
        'ATAL',
        'ATSP7',
        'ATSH 18 DA 10 F1',
        'ATCRA 18 DA F1 10',
      ]),
    ];

    const probeAddresses = <int>[0x000008, 0x00000E, 0x000010, 0x00001C];

    // v0.10: кэшированный профиль первым — реконнект за ~1 пробу вместо полного перебора.
    final cached = _cachedProfileCommands;
    final ordered = <_InitProfile>[
      if (cached != null) _InitProfile('$_cachedProfileName (кэш)', cached),
      ...profiles,
    ];

    for (final profile in ordered) {
      log('PROFILE TRY ${profile.name}');
      await _runInitCommands(profile.commands);
      for (final address in probeAddresses) {
        final value = await _probeAddress(address);
        if (value != null) {
          activeProfile = profile.name;
          _cachedProfileCommands = profile.commands; // v0.10
          _cachedProfileName = profile.name; // v0.10
          _setSync(true, 'ECU ответил на ${profile.name}');
          return true;
        }
      }
    }
    return false;
  }

  Future<void> initialize(String address) => _lock.run(() async {
    if (_disposed) throw StateError('Disposed');
    _settling = true;
    activeProfile = '';
    lastError = '';
    _setSync(false, 'Инициализация');
    try {
      await transport.disconnect();
      await transport.connect(address);
      await Future<void>.delayed(const Duration(milliseconds: 700));

      // Во время init допускаем обмен даже без ECU sync.
      _synchronized = false;
      try {
        await _exchange('ATWS', timeout: 2500, preserveSync: true);
      } catch (_) {
        try {
          await _exchange('ATZ', timeout: 3000, preserveSync: true);
        } catch (_) {}
      }
      await Future<void>.delayed(const Duration(milliseconds: 150));

      await _runInitCommands(const <String>[
        'ATE0',
        'ATL0',
        'ATS0',
        'ATAT1',
        'ATST 96',
      ]);

      try {
        identity = (await _exchange(
          'ATI',
          preserveSync: true,
        )).replaceAll(RegExp(r'[\r\n>]'), ' ').trim();
      } catch (_) {
        identity = 'ELM327 / BtSsm';
      }

      try {
        voltage = (await _exchange(
          'ATRV',
          preserveSync: true,
        )).replaceAll(RegExp(r'[\r\n>]'), ' ').trim();
      } catch (_) {
        voltage = '13.8V';
      }

      final canOk = await _probeProfiles();
      if (!canOk) {
        throw ReplyError(
          'SPP открыт, но ECU не ответил на SSM2-over-CAN A8 запросы. '
          'Проверьте зажигание, CAN-адаптер и попробуйте 11/29-bit профиль. '
          'v0.10: автомобили с K-line диагностикой (примерно до 2008, '
          'ISO-9141/14230) эта сборка не поддерживает — там нужен другой '
          'транспорт, это не неисправность адаптера.',
        );
      }

      lastAddress = address;
      await _calibrateCanTimeout();
      log('READY profile=$activeProfile timeoutMs=$timeoutMs');
    } catch (e) {
      lastError = '$e';
      _setSync(false, '$e');
      try {
        await transport.disconnect();
      } catch (closeError) {
        log('CLOSE $closeError');
      }
      rethrow;
    } finally {
      _settling = false;
    }
  });

  Future<void> _calibrateCanTimeout() async {
    final samples = <int>[];
    for (var i = 0; i < 3; i++) {
      final watch = Stopwatch()..start();
      final ok = await _probeAddress(0x000008);
      if (ok != null) samples.add(watch.elapsedMilliseconds);
      await Future<void>.delayed(const Duration(milliseconds: 40));
    }
    if (samples.length >= 2) {
      samples.sort();
      final median = samples[samples.length ~/ 2];
      timeoutMs = (median * 3).clamp(900, 3000);
      log('CALIBRATE timeoutMs=$timeoutMs (probe median ${median}ms)');
    } else {
      timeoutMs = timeoutMs.clamp(900, 3000);
      log('CALIBRATE timeoutMs=$timeoutMs (fallback floor)');
    }
  }

  bool _retryableReadError(Object error) {
    final text = '$error'.toUpperCase();
    return text.contains('TIMEOUT') ||
        text.contains('NO DATA') ||
        text.contains('STOPPED') ||
        text.contains('CAN ERROR') ||
        text.contains('BUS BUSY') ||
        text.contains('BUFFER FULL');
  }

  Future<String> _exchangeForRead(String command) async {
    try {
      return await _exchange(command);
    } catch (e) {
      if (!_retryableReadError(e)) rethrow;
      log('READ RETRY $command after $e');
      await Future<void>.delayed(const Duration(milliseconds: 40));
      return _exchange(command, timeout: timeoutMs + 300);
    }
  }

  Future<PidRead> readRange(int address, int length) => _lock.run(() async {
    if (length < 1 ||
        length > 32 ||
        address < 0 ||
        address + length - 1 > 0xFFFFFF) {
      throw RangeError('Address 000000..FFFFFF, length 1..32');
    }
    final watch = Stopwatch()..start();
    final bytes = <int>[];
    for (var offset = 0; offset < length; offset++) {
      final cmd = readAddressCommand(address + offset);
      final response = await _exchangeForRead(cmd);
      try {
        bytes.add(parseAddressReply(response, cmd));
      } catch (e) {
        log('REJECT $cmd: $e');
        rethrow;
      }
      await Future<void>.delayed(const Duration(milliseconds: 8));
    }
    return PidRead(List<int>.unmodifiable(bytes), watch.elapsedMilliseconds);
  });

  Future<PidRead> readPid(SubaruPidDef pid) =>
      readRange(pid.address, pid.bytesCount);

  Future<String> diagnostic(String command) {
    final normalized = command.trim().toUpperCase();
    if (!allowedDiagnostic(normalized)) {
      throw ArgumentError('Разрешены только безопасные AT-команды и A8-чтение');
    }
    return _lock.run(
      () => _exchange(normalized, preserveSync: normalized.startsWith('AT')),
    );
  }

  Future<void> disconnect() async {
    _lost('Disconnect requested');
    await transport.disconnect();
  }

  Future<void> dispose() async {
    if (_disposed) return;
    _disposed = true;
    try {
      await disconnect();
    } catch (e) {
      log('CLOSE ERROR $e');
    }
    try {
      await _lock.run(() async {});
    } finally {
      await _data.cancel();
      await _status.cancel();
      await transport.dispose();
    }
  }
}
'''

FILES["lib/engine.dart"] = r'''
import 'dart:async';

import 'package:flutter/foundation.dart';
import 'package:flutter/services.dart';

import 'elm.dart';
import 'identity.dart';
import 'pids.dart';
import 'samples.dart';

export 'samples.dart';

class SsmEngine extends ChangeNotifier {
  SsmEngine(this.elm);
  final ElmDriver elm;
  final Set<String> enabled = {...SubaruPidLibrary.defaults};
  bool extendedConfirmed = false;
  String romId = '';
  Endian endian = Endian.big;
  bool _endianProbed = false;
  bool running = false, _disposed = false;
  int _generation = 0;
  Future<void>? _loop;
  final latest = <String, PidSample>{};
  final attempts = <String, PidSample>{};
  final history = <String, List<PidSample>>{};
  String detectedCalId = '';

  final Map<int, int> _addrFails = <int, int>{};
  final Map<int, int> _addrMutedUntilMs = <int, int>{};
  static const int _muteAfter = 5, _muteMs = 30000;
  int mutedAddresses = 0, requests = 0;

  double tripDistanceKm = 0;
  double tripFuelLitres = 0;
  DateTime? _tripLastTick;
  final List<DateTime> _requestTimes = <DateTime>[];
  double get requestsPerSecond {
    final now = DateTime.now();
    _requestTimes.removeWhere((t) => now.difference(t).inSeconds >= 10);
    if (_requestTimes.length < 2) return 0;
    final seconds = now.difference(_requestTimes.first).inMilliseconds / 1000;
    return seconds > 0 ? _requestTimes.length / seconds : 0;
  }

  final _events = StreamController<PidSample>.broadcast();
  Stream<PidSample> get events => _events.stream;
  int goodCount = 0, failedCount = 0;
  String message = 'Нет данных';

  final _replyTimes = <DateTime>[];

  bool autoReconnect = true;
  int reconnects = 0, reconnectFails = 0;
  bool _reconnecting = false,
      _resumeAfterReconnect = false,
      _linkWatchStarted = false;
  static const List<int> _gapsMs = <int>[500, 1000, 2000, 5000, 10000];

  void _ensureLinkWatch() {
    if (_linkWatchStarted) return;
    _linkWatchStarted = true;
    unawaited(_linkWatch());
  }

  Future<void> _linkWatch() async {
    while (!_disposed && autoReconnect) {
      await Future<void>.delayed(const Duration(seconds: 1));
      if (_disposed || !autoReconnect || elm.ready || _reconnecting) continue;
      final address = elm.lastAddress;
      if (address == null || address.isEmpty) return;
      if (goodCount == 0 && failedCount == 0 && !running) continue;
      _reconnecting = true;
      var restored = false;
      for (final gap in _gapsMs) {
        if (_disposed || !autoReconnect) break;
        await Future<void>.delayed(Duration(milliseconds: gap));
        if (_disposed) break;
        message = 'Связь потеряна — повторное подключение…';
        _notify();
        try {
          await elm.initialize(address);
          restored = elm.ready;
        } catch (_) {
          restored = false;
        }
        if (restored) break;
      }
      if (_disposed) return;
      _reconnecting = false;
      if (restored) {
        reconnects++;
        message = 'Связь восстановлена (×$reconnects)';
        _notify();
        final resume = _resumeAfterReconnect;
        _resumeAfterReconnect = false;
        if (resume) {
          try {
            await start();
          } catch (e) {
            message = 'Рестарт опроса: $e';
            _notify();
          }
        }
      } else {
        reconnectFails++;
        message = 'Адаптер не отвечает — подключите вручную';
        _notify();
        return;
      }
    }
  }

  bool get extendedAllowed {
    if (!extendedConfirmed) return false;
    final entered = normalizeCalId(romId);
    if (matchCalProfile(entered) == null) return false;
    return true;
  }

  List<SubaruPidDef> get active => SubaruPidLibrary.all
      .where((p) => enabled.contains(p.id) && (!p.extended || extendedAllowed))
      .toList();

  double get quality => goodCount + failedCount == 0
      ? 0
      : goodCount * 100 / (goodCount + failedCount);

  double get pidReadsPerSecond {
    final now = DateTime.now();
    _replyTimes.removeWhere((t) => now.difference(t).inSeconds >= 10);
    if (_replyTimes.length < 2) return 0;
    final seconds = now.difference(_replyTimes.first).inMilliseconds / 1000;
    return seconds > 0 ? (_replyTimes.length - 1) / seconds : 0;
  }

  double frequency(String id) {
    final h = history[id];
    if (h == null || h.length < 2) return 0;
    final end = h.last.time;
    if (DateTime.now().difference(end).inSeconds > 10) return 0;
    final start = h.length > 10 ? h.length - 10 : 0;
    final seconds = end.difference(h[start].time).inMilliseconds / 1000;
    return seconds > 0 ? (h.length - 1 - start) / seconds : 0;
  }

  int? ageMs(String id) {
    final sample = latest[id];
    return sample == null
        ? null
        : DateTime.now().difference(sample.time).inMilliseconds;
  }

  bool stale(String id) => (ageMs(id) ?? 999999) > 3000;

  void _notify() {
    if (!_disposed) notifyListeners();
  }

  void resetTrip() {
    tripDistanceKm = 0;
    tripFuelLitres = 0;
    _tripLastTick = null;
    _notify();
  }

  void _tickTrip(PidSample sample) {
    final now = sample.time;
    final prev = _tripLastTick;
    _tripLastTick = now;
    if (prev == null) return;
    final dt = now.difference(prev).inMilliseconds / 1000.0;
    if (dt <= 0 || dt > 5) return;
    final speed = latest['SPEED']?.value;
    if (speed != null && speed > 0.5) tripDistanceKm += speed / 3600.0 * dt;
    final maf = latest['MAF']?.value;
    if (maf != null && maf > 0) {
      final afr = latest['AFR']?.value;
      final ratio = (afr != null && afr >= 8 && afr <= 25) ? afr : 14.7;
      tripFuelLitres += maf / ratio / 745.0 * dt;
    }
  }

  void _publish(PidSample sample) {
    attempts[sample.pid.id] = sample;
    if (sample.good) {
      latest[sample.pid.id] = sample;
      final list = history.putIfAbsent(sample.pid.id, () => []);
      list.add(sample);
      if (list.length > 1200)
        list.removeAt(0); // v0.10: окно анализа 300→1200 точек
      goodCount++;
      _replyTimes.add(sample.time);
      if (_replyTimes.length > 300) _replyTimes.removeAt(0);
    } else {
      failedCount++;
    }
    _maybeAlert(sample);
    if (!_disposed) _events.add(sample);
    _tickTrip(sample);
    _notify();
  }

  bool audioAlerts = true;
  DateTime _lastAlertAt = DateTime.fromMillisecondsSinceEpoch(0);

  void _maybeAlert(PidSample sample) {
    if (!audioAlerts || !sample.good) return;
    final value = sample.value;
    if (value == null) return;
    final id = sample.pid.id;
    final dangerous = (id == 'FBKC' && value <= -1.4) ||
        (id == 'FKL' && value <= -2.0) ||
        (id == 'ECT' && value >= 110) ||
        (id == 'BATT' && value <= 11.5);
    if (!dangerous) return;
    final now = DateTime.now();
    if (now.difference(_lastAlertAt).inSeconds < 3) return;
    _lastAlertAt = now;
    SystemSound.play(SystemSoundType.alert);
    HapticFeedback.heavyImpact();
  }

  Future<void> start() async {
    if (running || _disposed) return;
    if (_loop != null) await _loop;
    if (_disposed || !elm.ready) throw StateError('Сначала подключите адаптер');
    if (active.isEmpty) throw StateError('Выберите хотя бы один PID');
    final generation = ++_generation;
    running = true;
    message = 'Опрос SAFE Single Frame';
    _loop = _poll(generation);
    _ensureLinkWatch();
    _notify();
  }

  bool _muteableError(Object error) {
    final text = '$error'.toUpperCase();
    if (text.contains('TIMEOUT') ||
        text.contains('NO DATA') ||
        text.contains('STOPPED') ||
        text.contains('CAN ERROR') ||
        text.contains('BUS BUSY') ||
        text.contains('BUFFER FULL') ||
        text.contains('SPP')) {
      return false;
    }
    return true;
  }

  Future<void> _poll(int generation) async {
    var round = 0;
    try {
      while (generation == _generation && running && elm.ready) {
        final list = active;
        if (list.isEmpty) break;
        final tpsNow = latest['TPS']?.value ?? latest['PEDAL']?.value ?? 0;
        final rpmNow = latest['RPM']?.value ?? 0;
        final hot = tpsNow >= 60 || rpmNow >= 3500;
        const hotSet = <String>{
          'RPM',
          'LOAD',
          'LOAD_4B',
          'TIMING',
          'KNOCK_ADV',
          'FBKC',
          'FKL',
          'AFR',
          'BOOST',
          'MAP_REL',
        };
        final due = list.where((p) {
          final base = p.priority == 1
              ? 1
              : p.priority == 2
                  ? 2
                  : 5;
          if (hot) {
            if (hotSet.contains(p.id)) return true;
            return round % (base * 2) == 0;
          }
          return round % base == 0;
        }).toList();

        final nowMs = DateTime.now().millisecondsSinceEpoch;
        // v0.10 FIX (ENGINE-01): протухшие mute-записи удаляем — раньше метрика
        // «N PID изолировано» росла вечно и чеклист «0 изоляций» был недостижим.
        _addrMutedUntilMs.removeWhere((address, until) => nowMs >= until);
        for (final pid in due) {
          if (generation != _generation || !running || !elm.ready) return;
          if (pid.addresses.any((a) => nowMs < (_addrMutedUntilMs[a] ?? 0)))
            continue;
          requests++;
          _requestTimes.add(DateTime.now());
          if (_requestTimes.length > 200) _requestTimes.removeAt(0);

          try {
            final read = await elm.readPid(pid);
            if (generation != _generation || _disposed) return;
            final value = pid.decode(read.bytes, endian: endian);
            if (!_endianProbed &&
                pid.floatFactor != null &&
                pid.bytesCount == 4 &&
                pid.id == 'IAM') {
              _endianProbed = true;
              final alt = endian == Endian.big ? Endian.little : Endian.big;
              final altVal = pid.decode(read.bytes, endian: alt);
              if ((value == null || value == 0) &&
                  altVal != null &&
                  altVal > 0 &&
                  altVal <= 1.5) {
                endian = alt;
              }
            }
            _publish(PidSample(
              pid,
              DateTime.now(),
              value,
              List<int>.unmodifiable(read.bytes),
              read.elapsedMs,
              value == null ? 'INVALID_FLOAT_OR_LENGTH' : '',
            ));
            for (final a in pid.addresses) _addrFails.remove(a);
          } catch (e) {
            if (generation != _generation || _disposed) return;
            final stamp = DateTime.now().millisecondsSinceEpoch;
            if (_muteableError(e)) {
              for (final a in pid.addresses) {
                final n = (_addrFails[a] ?? 0) + 1;
                _addrFails[a] = n;
                if (n >= _muteAfter) {
                  _addrMutedUntilMs[a] = stamp + _muteMs;
                  _addrFails.remove(a);
                }
              }
            }
            mutedAddresses = _addrMutedUntilMs.length;
            message = '$e';
            _publish(PidSample(pid, DateTime.now(), null, const [], 0, '$e'));
          }
          await Future<void>.delayed(const Duration(milliseconds: 18));
        }
        mutedAddresses = _addrMutedUntilMs.length;
        round++;
        await Future<void>.delayed(const Duration(milliseconds: 20));
      }
    } finally {
      if (generation == _generation) {
        if (!elm.ready) _resumeAfterReconnect = true;
        running = false;
        if (!elm.ready) message = 'Опрос остановлен';
        _notify();
      }
    }
  }

  Future<void> stop() async {
    _generation++;
    running = false;
    final current = _loop;
    if (current != null) await current;
    if (identical(current, _loop)) _loop = null;
    _notify();
  }

  Future<void> configure(
    Set<String> ids,
    bool confirm,
    String rom,
    Endian byteOrder,
  ) async {
    await stop();
    enabled
      ..clear()
      ..addAll(ids);
    extendedConfirmed = confirm && rom.trim().isNotEmpty;
    if (!extendedConfirmed) {
      enabled.removeWhere(
          (id) => SubaruPidLibrary.all.any((p) => p.id == id && p.extended));
    }
    romId = rom.trim();
    endian = byteOrder;
    latest.clear();
    attempts.clear();
    history.clear();
    _replyTimes.clear();
    _addrFails.clear();
    _addrMutedUntilMs.clear();
    _endianProbed = false;
    mutedAddresses = 0;
    requests = 0;
    _requestTimes.clear();
    goodCount = 0;
    failedCount = 0;
    _notify();
  }

  @override
  void dispose() {
    _disposed = true;
    running = false;
    _generation++;
    unawaited(_events.close());
    super.dispose();
  }
}
'''

FILES["lib/analyzer.dart"] = r'''
import 'dart:math' as math;

import 'samples.dart';

enum FindingLevel { info, warning, critical }

class Finding {
  const Finding(
    this.level,
    this.title,
    this.detail,
    this.evidence,
    this.advice,
  );
  final FindingLevel level;
  final String title, detail, evidence, advice;
}

class LogAnalysis {
  const LogAnalysis(
    this.findings,
    this.sampleCount,
    this.spanSeconds,
    this.channels,
  );
  final List<Finding> findings;
  final int sampleCount, channels;
  final double spanSeconds;
}

const double kFbkcEventDeg = -1.4;
const double kFbkcBadDeg = -4.0;
const double kFlkcWarnDeg = -2.0;
const double kFlkcBadDeg = -4.0;
const double kIamFull = 15.5;
const double kIamPoor = 13.0;
const double kTrimWarnPct = 6.0;
const double kTrimBadPct = 10.0;
const double kBattWarnV = 11.8;
const double kWarmEctC = 70.0;
const double kColdRpm = 2500.0;

class _Series {
  _Series(this.samples);
  final List<({double v, DateTime t})> samples;
  int get count => samples.length;
  bool get empty => samples.isEmpty;
  double get mean => samples.isEmpty
      ? 0
      : samples.fold<double>(0, (a, s) => a + s.v) / samples.length;
  double get min => samples.isEmpty
      ? 0
      : samples.fold<double>(double.infinity, (a, s) => math.min(a, s.v));
  double get max => samples.isEmpty
      ? 0
      : samples.fold<double>(
          double.negativeInfinity,
          (a, s) => math.max(a, s.v),
        );
  double get p95 {
    if (samples.isEmpty) return 0;
    final vals = [for (final s in samples) s.v]..sort();
    return vals[((vals.length - 1) * 0.95).round()];
  }
}

_Series _series(Map<String, List<PidSample>> history, List<String> ids) {
  final out = <({double v, DateTime t})>[];
  for (final id in ids) {
    final list = history[id];
    if (list == null) continue;
    for (final s in list) {
      final v = s.value;
      if (s.good && v != null && v.isFinite) out.add((v: v, t: s.time));
    }
  }
  out.sort((a, b) => a.t.compareTo(b.t));
  return _Series(out);
}

/// Сопоставление двух каналов по ближайшим меткам времени (v0.9).
List<({double a, double b, DateTime t})> _pair(
  _Series left,
  _Series right, {
  int toleranceMs = 300,
}) {
  final out = <({double a, double b, DateTime t})>[];
  if (left.empty || right.empty) return out;
  var j = 0;
  for (final l in left.samples) {
    while (j + 1 < right.samples.length &&
        right.samples[j + 1].t.difference(l.t).inMilliseconds.abs() <=
            right.samples[j].t.difference(l.t).inMilliseconds.abs()) {
      j++;
    }
    final r = right.samples[j];
    if (r.t.difference(l.t).inMilliseconds.abs() <= toleranceMs) {
      out.add((a: l.v, b: r.v, t: l.t));
    }
  }
  return out;
}

String _f(double v) => v.toStringAsFixed(1);
String _f2(double v) => v.toStringAsFixed(2);

LogAnalysis analyzeLog(
  Map<String, List<PidSample>> history, {
  Map<String, PidSample> attempts = const <String, PidSample>{},
}) {
  final findings = <Finding>[];
  final series = <String, _Series>{
    'rpm': _series(history, const ['RPM']),
    'load': _series(history, const ['LOAD_4B', 'LOAD']),
    'ect': _series(history, const ['ECT']),
    'fbkc': _series(history, const ['FBKC']),
    'flkc': _series(history, const ['FKL', 'KNOCK_ADV']),
    'iam': _series(history, const ['IAM']),
    'stft': _series(history, const ['STFT']),
    'ltft': _series(history, const ['LTFT']),
    'batt': _series(history, const ['BATT']),
    'tps': _series(history, const ['TPS', 'PEDAL']),
    'afr': _series(history, const ['AFR']),
    'cltgt': _series(history, const ['CL_TARGET']),
    'boost': _series(history, const ['BOOST', 'MAP_REL']),
    'boosttgt': _series(history, const ['BOOST_TGT']),
  };

  var sampleCount = 0;
  DateTime? first;
  DateTime? last;
  var channels = 0;
  for (final s in series.values) {
    sampleCount += s.count;
    if (s.count > 0) channels++;
    for (final p in s.samples) {
      if (first == null || p.t.isBefore(first)) first = p.t;
      if (last == null || p.t.isAfter(last)) last = p.t;
    }
  }
  final span = first != null && last != null
      ? last.difference(first).inMilliseconds / 1000.0
      : 0.0;

  // v0.10 FIX (ANLZ-02): шкалы нагрузки различаем явно — 4-байт (г/об) и 1-байт (%).
  final loadGb = _series(history, const ['LOAD_4B']);
  final loadPct = _series(history, const ['LOAD']);

  // --- Детонация: FBKC -------------------------------------------------------------
  final fbkc = series['fbkc']!;
  if (!fbkc.empty) {
    final events = fbkc.samples.where((s) => s.v <= kFbkcEventDeg).toList();
    // v0.10: г/об доказываем порогом 1.5, проценты — порогом 70%. Раньше LOAD %
    // (0..100) делал hardLoad истинным всегда — критикал по любому событию FBKC.
    final hardLoad = !loadGb.empty
        ? loadGb.max >= 1.5 && series['rpm']!.max >= 3000
        : loadPct.max >= 70 && series['rpm']!.max >= 3000;
    if (events.isNotEmpty || fbkc.min <= 0) {
      if (events.length >= 5 ||
          fbkc.min <= kFbkcBadDeg ||
          (events.isNotEmpty && hardLoad)) {
        findings.add(
          Finding(
            FindingLevel.critical,
            'Детонация: ECU агрессивно снижает УОЗ',
            'Зафиксировано ${events.length} событий FBKC <= ${_f(kFbkcEventDeg)}°, '
                'минимум ${_f(fbkc.min)}°${hardLoad ? ' на нагрузке (${!loadGb.empty ? 'load >= 1.5 г/об' : 'load >= 70%'}, rpm >= 3000)' : ''}. '
                'Залейте топливо с октаном выше, затем снимите 1-2° в зоне кластера через Map Lab.',
            'fbkc: n=${fbkc.count} events=${events.length} min=${_f(fbkc.min)}° · '
                'rpm_max=${_f(series['rpm']!.max)} load_max=${_f2(series['load']!.max)}',
            'Map Lab -> timing/knockadv: кластерная коррекция -0.5..-1.0° с веером сглаживания.',
          ),
        );
      } else if (events.isNotEmpty || fbkc.min <= -0.5) {
        findings.add(
          Finding(
            FindingLevel.warning,
            'Единичные рывки FBKC',
            'Есть ${events.length} событий FBKC <= ${_f(kFbkcEventDeg)}°, минимум ${_f(fbkc.min)}°. '
                'Пока это штатная реакция, но повторяемость в одной зоне оборотов — повод для точечной коррекции.',
            'events=${events.length} min=${_f(fbkc.min)}°',
            'Повторный лог в той же нагрузке; если кластер — минус 0.5° в Map Lab.',
          ),
        );
      }
    }
  }

  // --- Детонация: FLKC (обученная коррекция) ---------------------------------------
  final flkc = series['flkc']!;
  if (!flkc.empty && flkc.min < 0) {
    final level = flkc.min <= kFlkcBadDeg
        ? FindingLevel.critical
        : FindingLevel.warning;
    if (flkc.min <= kFlkcWarnDeg) {
      findings.add(
        Finding(
          level,
          'Обученная коррекция зажигания удерживается',
          'FLKC достигала ${_f(flkc.min)}° и не возвращается к нулю: ECU запомнил детонацию '
              'в весовых зонах. Топливо/наддув/свечи — сначала причина, потом карта.',
          'flkc: min=${_f(flkc.min)}° mean=${_f(flkc.mean)}° n=${flkc.count}',
          'Сброс обучения и повторный лог только после устранения причины.',
        ),
      );
    }
  }

  // --- IAM (обучение грубых/тонких таблиц) -------------------------------------------
  // v0.10 FIX (ANLZ-01): 4-байтный IAM — float-множитель 0..1, а старые пороги
  // (kIamPoor=13, kIamFull=15.5) — в шагах 0..16. Автодетект шкалы и приведение
  // к шагам; иначе любой живой лог получал фейковый критикал «IAM критически низкий».
  var iam = series['iam']!;
  final iamScaled = !iam.empty && iam.max <= 1.2;
  if (iamScaled) {
    iam = _Series([for (final s in iam.samples) (v: s.v * 16.0, t: s.t)]);
  }
  if (!iam.empty) {
    final maxIam = iam.max;
    if (maxIam < kIamPoor) {
      findings.add(
        Finding(
          FindingLevel.critical,
          'IAM критически низкий',
          'Динамический шаг зажигания ограничен: IAM max ${_f(maxIam)} из 16. '
              'Двигатель работает по грубой (conservative) карте — мощность ниже, риск выше.',
          'iam: max=${_f(maxIam)} mean=${_f(iam.mean)} n=${iam.count}${iamScaled ? ' (0..1 → ×16)' : ''}',
          'Не снимать ограничение, пока FBKC/FLKC не чисты: это защита мотора.',
        ),
      );
    } else if (maxIam < kIamFull) {
      findings.add(
        Finding(
          FindingLevel.warning,
          'IAM не добрался до полного',
          'IAM max ${_f(maxIam)} из 16: обучение не завершено или ECU придерживает запас. '
              'Нормально после сброса адаптаций, иначе — слабый сигнал скрытого детона.',
          'iam: max=${_f(maxIam)} mean=${_f(iam.mean)} n=${iam.count}${iamScaled ? ' (0..1 → ×16)' : ''}',
          'Лог 15-20 мин спокойной езды после прогрева: IAM должен дорасти до 16.',
        ),
      );
    } else {
      findings.add(
        Finding(
          FindingLevel.info,
          'IAM полный',
          'Обучение зажигания завершено: IAM достигал ${_f(maxIam)} / 16.',
          'iam: max=${_f(maxIam)} mean=${_f(iam.mean)}',
          'Без действий.',
        ),
      );
    }
  }

  // --- Топливные коррекции -----------------------------------------------------------
  final stft = series['stft']!;
  final ltft = series['ltft']!;
  if (!stft.empty && !ltft.empty) {
    final combined = stft.mean + ltft.mean;
    final absSum = combined.abs();
    if (absSum >= kTrimBadPct) {
      findings.add(
        Finding(
          FindingLevel.critical,
          'Топливная коррекция за пределами',
          'STFT+LTFT ${combined >= 0 ? '+' : ''}${_f(combined)}%: смесь далека от цели. '
              'Типичные причины: подсос воздуха, MAF, датчик O2, давление топлива.',
          'stft_mean=${_f(stft.mean)}% ltft_mean=${_f(ltft.mean)}% sum=${_f(combined)}%',
          'Сначала диагностика железа. Картами VE это не лечится.',
        ),
      );
    } else if (absSum >= kTrimWarnPct) {
      findings.add(
        Finding(
          FindingLevel.warning,
          'Топливная коррекция повышена',
          'STFT+LTFT ${combined >= 0 ? '+' : ''}${_f(combined)}%: умеренное смещение смеси.',
          'stft_mean=${_f(stft.mean)}% ltft_mean=${_f(ltft.mean)}%',
          'Проверить подсос/фильтр и переснять лог с AFR.',
        ),
      );
    }
  }

  // --- Прогрев -----------------------------------------------------------------------
  final ect = series['ect']!;
  final rpm = series['rpm']!;
  if (!ect.empty && !rpm.empty && ect.min < kWarmEctC && rpm.max > kColdRpm) {
    findings.add(
      Finding(
        FindingLevel.warning,
        'Нагрузка до прогрева',
        'Минимум ECT ${_f(ect.min)}°C при пике ${_f(rpm.max)} об/мин: двигатель крутили холодным. '
            'Детонационный запас на холодную иной — выводы по FBKC из этой зоны не переносить в карту.',
        'ect_min=${_f(ect.min)}°C rpm_max=${_f(rpm.max)}',
        'Валидный тюнинг-лог: ECT > 80°C стабильно.',
      ),
    );
  }

  // --- Бортсеть ------------------------------------------------------------------------
  final batt = series['batt']!;
  if (!batt.empty && batt.min < kBattWarnV) {
    findings.add(
      Finding(
        FindingLevel.warning,
        'Просадка питания',
        'Минимум бортсети ${_f2(batt.min)}V: при <12V логируемые значения (особенно MAF/форсунки) '
            'уходят, а клоны ELM сыплют таймаутами.',
        'batt: min=${_f2(batt.min)}V mean=${_f2(batt.mean)}V n=${batt.count}',
        'Заряд/АКБ до повторного лога; адаптер снять с прикуривателя.',
      ),
    );
  }

  // --- AFR против цели: беднение под нагрузкой (v0.9) ---
  final afrPairs = _pair(series['afr']!, series['cltgt']!);
  if (afrPairs.isNotEmpty) {
    final lean = afrPairs.where((p) => p.a - p.b >= 0.7).toList();
    final ratio = lean.length * 100 / afrPairs.length;
    var worst = 0.0;
    for (final p in lean) {
      final d = p.a - p.b;
      if (d > worst) worst = d;
    }
    if (ratio >= 15) {
      findings.add(
        Finding(
          ratio >= 35 || worst >= 1.5
              ? FindingLevel.critical
              : FindingLevel.warning,
          'Смесь беднее цели',
          'В ${_f(ratio)}% сопоставленных точек AFR выше целевого на 0.7+ '
              '(максимум +${_f(worst)}). Беднение под наддувом — прямой путь к детонации '
              'и прогару; проверяйте топливоподачу до правок карт.',
          'pairs=${afrPairs.length} lean=${lean.length} (${_f(ratio)}%) worst=+${_f(worst)}',
          'Давление топлива, форсунки, MAF-скейлинг. Затем Map Lab -> fuel.',
        ),
      );
    }
  }

  // --- Наддув: ошибка слежения и переброс (v0.9) ---
  final boostPairs = _pair(series['boost']!, series['boosttgt']!);
  if (boostPairs.length >= 5) {
    var sumSq = 0.0;
    var over = 0.0;
    var under = 0.0;
    for (final p in boostPairs) {
      final e = p.a - p.b;
      sumSq += e * e;
      if (e > over) over = e;
      if (e < under) under = e;
    }
    final rms = math.sqrt(sumSq / boostPairs.length);
    if (over >= 0.25 || rms >= 0.15) {
      findings.add(
        Finding(
          over >= 0.4 ? FindingLevel.critical : FindingLevel.warning,
          'Наддув не держит цель',
          'RMS ошибки ${_f2(rms)} бар, максимальный переброс +${_f2(over)} бар, '
              'просадка ${_f2(under)} бар. Переброс опаснее недобора: это незапланированная '
              'нагрузка на поршневую.',
          'pairs=${boostPairs.length} rms=${_f2(rms)} over=+${_f2(over)} under=${_f2(under)}',
          'Map Lab -> wgdc: снизить дьюти в зоне переброса, проверить вестгейт и шланги.',
        ),
      );
    }
  }

  // --- Аномалии каналов: выбросы по z-score (v0.9.1) ---
  for (final entry in series.entries) {
    final s = entry.value;
    if (s.count < 30) continue;
    final mean = s.mean;
    var acc = 0.0;
    for (final p in s.samples) {
      acc += (p.v - mean) * (p.v - mean);
    }
    final std = math.sqrt(acc / s.count);
    if (std <= 0) continue;
    var outliers = 0;
    for (final p in s.samples) {
      if ((p.v - mean).abs() > 3.5 * std) outliers++;
    }
    final ratio = outliers * 100 / s.count;
    if (outliers >= 3 && ratio >= 1.0) {
      findings.add(
        Finding(
          FindingLevel.info,
          'Выбросы канала ${entry.key}',
          'Канал ${entry.key}: $outliers из ${s.count} значений (${_f(ratio)}%) ушли дальше '
              '3.5σ от среднего. Частая причина — единичные сбои ELM-клона или mute '
              'проблемных адресов, а не физика двигателя.',
          'channel=${entry.key} outliers=$outliers/${s.count} mean=${_f2(mean)} std=${_f2(std)}',
          'Если выбросов > 5% — переподключите адаптер и снимите лог повторно.',
        ),
      );
    }
  }

  // --- Неравномерный холостой ход: признак пропусков/подсоса (v0.9.1) ---
  final idlePairs = _pair(
    series['rpm']!,
    series['tps']!,
  ).where((p) => p.b <= 3 && p.a >= 500 && p.a <= 1500).toList();
  if (idlePairs.length >= 10 && !ect.empty && ect.max >= 70) {
    final meanRpm =
        idlePairs.fold<double>(0, (a, p) => a + p.a) / idlePairs.length;
    var accR = 0.0;
    for (final p in idlePairs) {
      accR += (p.a - meanRpm) * (p.a - meanRpm);
    }
    final stdRpm = math.sqrt(accR / idlePairs.length);
    if (stdRpm >= 70) {
      findings.add(
        Finding(
          FindingLevel.warning,
          'Неравномерный холостой ход',
          'На тёплом двигателе при закрытом дросселе обороты гуляют σ=${_f(stdRpm)} об/мин '
              'вокруг ${_f(meanRpm)}. Похоже на пропуски воспламенения, подсос воздуха или '
              'грязный дроссель — чинить до тюнинга.',
          'idle_points=${idlePairs.length} rpm_mean=${_f(meanRpm)} rpm_std=${_f(stdRpm)}',
          'Свечи/катушки, проверка на подсос, LTFT на холостом.',
        ),
      );
    }
  }

  // --- Полнота данных -------------------------------------------------------------------
  if (channels < 8) {
    final failed = attempts.values.where((s) => !s.good).length;
    findings.add(
      Finding(
        FindingLevel.info,
        'Мало каналов для выводов',
        'Активно каналов: $channels. Для тюнинга базово нужны RPM/LOAD/TIMING/FBKC/FLKC/IAM/AFR/MAF. '
            '${failed > 0 ? 'Ошибок чтения за сессию: $failed.' : ''}',
        'channels=$channels failed_attempts=$failed',
        'Включите полный набор PID и/или расширенный набор (CALID-гейт).',
      ),
    );
  }

  findings.add(
    Finding(
      FindingLevel.info,
      'Сводка анализа',
      'Обработано $sampleCount значений по $channels каналам за ${_f(span)} с. '
          'Найдено проблем: ${findings.where((f) => f.level != FindingLevel.info).length}.',
      'samples=$sampleCount channels=$channels span=${_f(span)}s',
      findings.any((f) => f.level == FindingLevel.critical)
          ? 'Сначала критичные находки, потом Map Lab.'
          : 'Лог можно передавать в Map Lab для табличной коррекции.',
    ),
  );

  return LogAnalysis(
    List<Finding>.unmodifiable(findings),
    sampleCount,
    span,
    channels,
  );
}

// ======================= Сравнение логов «до/после» (v0.9.1) =======================

class ChannelDelta {
  const ChannelDelta(
    this.id,
    this.samplesBase,
    this.samplesAfter,
    this.meanBase,
    this.meanAfter,
    this.p95Base,
    this.p95After,
  );
  final String id;
  final int samplesBase, samplesAfter;
  final double meanBase, meanAfter, p95Base, p95After;
  double get meanDelta => meanAfter - meanBase;
}

class LogComparison {
  const LogComparison(
    this.channels,
    this.verdict,
    this.knockBase,
    this.knockAfter,
  );
  final List<ChannelDelta> channels;
  final String verdict;
  final int knockBase, knockAfter;
}

int _knockEvents(Map<String, List<PidSample>> history) {
  final fbkc = _series(history, const ['FBKC']);
  var n = 0;
  for (final s in fbkc.samples) {
    if (s.v <= kFbkcEventDeg) n++;
  }
  return n;
}

LogComparison compareLogs(
  Map<String, List<PidSample>> baseline,
  Map<String, List<PidSample>> after, {
  List<String> channels = const [
    'RPM',
    'LOAD',
    'TIMING',
    'FBKC',
    'FKL',
    'IAM',
    'AFR',
    'STFT',
    'LTFT',
    'BOOST',
    'ECT',
  ],
}) {
  final deltas = <ChannelDelta>[];
  for (final id in channels) {
    final a = _series(baseline, [id]);
    final b = _series(after, [id]);
    if (a.count < 5 || b.count < 5) continue;
    deltas.add(
      ChannelDelta(id, a.count, b.count, a.mean, b.mean, a.p95, b.p95),
    );
  }
  deltas.sort((x, y) => y.meanDelta.abs().compareTo(x.meanDelta.abs()));
  final knockBase = _knockEvents(baseline);
  final knockAfter = _knockEvents(after);
  final verdict = knockAfter < knockBase
      ? 'Стало лучше: событий детонации $knockBase → $knockAfter'
      : knockAfter > knockBase
      ? 'Стало хуже: событий детонации $knockBase → $knockAfter — откатить последнюю правку'
      : 'Детонация без изменений ($knockBase события): правка нейтральна';
  return LogComparison(
    List<ChannelDelta>.unmodifiable(deltas),
    verdict,
    knockBase,
    knockAfter,
  );
}
'''

FILES["lib/native_spp.dart"] = r'''
import 'dart:async';

import 'package:flutter/services.dart';

import 'bt_transport.dart';

class NativeSppTransport implements BtTransport {
  static const MethodChannel _methods = MethodChannel('ssm2/spp');
  static const EventChannel _dataEvents = EventChannel('ssm2/spp_data');
  static const EventChannel _statusEvents = EventChannel('ssm2/spp_status');

  final StreamController<Uint8List> _data =
      StreamController<Uint8List>.broadcast();
  final StreamController<bool> _status = StreamController<bool>.broadcast();
  StreamSubscription<dynamic>? _dataSub;
  StreamSubscription<dynamic>? _statusSub;
  bool _connected = false;

  @override
  String get name => 'N / native RFCOMM SPP (CAN fix)';

  @override
  bool get connected => _connected;

  @override
  Stream<Uint8List> get data => _data.stream;

  @override
  Stream<bool> get status => _status.stream;

  void _ensureStreams() {
    _dataSub ??= _dataEvents.receiveBroadcastStream().listen((dynamic chunk) {
      if (chunk is Uint8List && chunk.isNotEmpty) _data.add(chunk);
    }, onError: (Object e) => _data.addError(e));
    _statusSub ??= _statusEvents.receiveBroadcastStream().listen(
      (dynamic raw) {
        final up = raw == true;
        if (_connected != up) {
          _connected = up;
          _status.add(up);
        }
      },
      onError: (Object e) {
        _connected = false;
        _status.add(false);
      },
    );
  }

  @override
  Future<List<BtDevice>> paired() async {
    await requestBluetoothPermissions();
    _ensureStreams();
    // v0.10 FIX (NATIVE-03): Bluetooth выключен/запрещён — внятная ошибка
    // вместо тихо пустого списка (до этого пользователь шёл пересопрягать исправное).
    final List<dynamic>? devices;
    try {
      devices = await _methods.invokeMethod<List<dynamic>>('paired');
    } on PlatformException catch (e) {
      throw StateError(e.message ?? 'Bluetooth недоступен');
    }
    return (devices ?? const <dynamic>[])
        .whereType<Map<dynamic, dynamic>>()
        .map((d) => BtDevice('${d['name'] ?? ''}', '${d['address'] ?? ''}'))
        .toList();
  }

  @override
  Future<void> connect(String address) async {
    await requestBluetoothPermissions();
    await disconnect();
    _ensureStreams();
    try {
      final ok = await _methods
          .invokeMethod<bool>('connect', <String, dynamic>{'address': address})
          .timeout(const Duration(seconds: 25));
      _connected = ok == true;
      if (!_connected) {
        throw StateError(
          'SPP не подключился: $address (сопряжение/питание/канал RFCOMM)',
        );
      }
      _status.add(true);
    } on PlatformException catch (e) {
      _connected = false;
      throw StateError('SPP connect failed: ${e.message ?? e.code}');
    } on TimeoutException {
      // v0.10 FIX (NATIVE-02): Dart-таймаут 25 с мог обогнать нативный connect,
      // который всё ещё выполнялся, — состояние разъезжалось. Гасим нативно и
      // требуем осознанный повтор, чтобы не было «призрачного» сокета.
      _connected = false;
      unawaited(_methods.invokeMethod<void>('disconnect').catchError((_) {}));
      throw StateError(
        'SPP: таймаут подключения (25 с). Нажмите «Отключить» и повторите.',
      );
    }
  }

  @override
  Future<void> write(String ascii) async {
    if (!_connected) throw StateError('SPP disconnected');
    await _methods.invokeMethod<void>('write', <String, dynamic>{
      'data': ascii,
    });
  }

  @override
  Future<void> disconnect() async {
    if (!_connected && _dataSub == null) return;
    try {
      await _methods.invokeMethod<void>('disconnect');
    } catch (_) {
      // Канал уже мёртв — состояние просто сбрасываем.
    }
    if (_connected) {
      _connected = false;
      _status.add(false);
    }
  }

  @override
  Future<void> dispose() async {
    await disconnect();
    await _dataSub?.cancel();
    _dataSub = null;
    await _statusSub?.cancel();
    _statusSub = null;
    await _data.close();
    await _status.close();
  }
}
'''

FILES["lib/model.dart"] = r'''
import 'dart:async';
import 'dart:convert';
import 'dart:io';

import 'package:flutter/foundation.dart';
import 'package:path_provider/path_provider.dart';
import 'package:share_plus/share_plus.dart';

import 'bt_transport.dart';
import 'elm.dart';
import 'engine.dart';
import 'pids.dart';
import 'protocol.dart';

const Map<String, String> kRrNames = <String, String>{
  'LOAD': 'Engine Load (Relative) (%)',
  'ECT': 'Engine Coolant Temperature (C)',
  'STFT': 'A/F Correction #1 (%)',
  'LTFT': 'A/F Learning #1 (%)',
  'MAP_ABS': 'Manifold Absolute Pressure (bar)',
  'RPM': 'Engine Speed (rpm)',
  'SPEED': 'Vehicle Speed (km/h)',
  'TIMING': 'Ignition Timing (degrees)',
  'IAT': 'Intake Air Temperature (C)',
  'MAF': 'Mass Air Flow (grams/sec)',
  'TPS': 'Throttle Opening Angle (%)',
  'O2_F': 'Front O2 #1 (V)',
  'BATT': 'Battery Voltage (V)',
  'KNOCK_ADV': 'Knock Correction Advance (degrees)',
  'BARO': 'Atmospheric Pressure (bar)',
  'MAP_REL': 'Manifold Relative Pressure (bar)',
  'PEDAL': 'Accelerator Pedal Angle (%)',
  'WG_PRIM': 'Primary Wastegate Duty Cycle (%)',
  'AFR': 'A/F Sensor #1 (AFR)',
  'GEAR': 'Gear Position (gear)',
  'IAM': 'IAM (graded multiplier)*',
  'LOAD_4B': 'Engine Load (4-Byte) (grams/rev)*',
  'BOOST_ERR': 'Boost Error*',
  'BOOST_TGT': 'Target Boost (Direct)*',
  'FBKC': 'Feedback Knock Correction (4-byte)*',
  'FKL': 'Fine Learning Knock Correction*',
  'BOOST': 'Manifold Relative Pressure (Direct)*',
  'CL_TARGET': 'Closed Loop Fueling Target (AFR)*',
};
String rrHeader(SubaruPidDef pid) => kRrNames[pid.id] ?? pid.id;

String csvCell(Object? value) =>
    '"${(value?.toString() ?? '').replaceAll('"', '""')}"';

class CsvLogger {
  IOSink? _sink;
  Future<void> _writes = Future<void>.value();
  File? file;
  bool active = false;
  int count = 0, _pending = 0;
  // v0.10 FIX (LOG-03): потерянные строки считаем и показываем в UI,
  // а логгер продолжает писать — раньше он молча умирал навсегда.
  int dropped = 0;
  String error = '';

  // v0.10 FIX (LOG-02): широкий RomRaider-формат «одна строка = один момент».
  // Файл напрямую открывается Map Lab (LogData.parse), конвертер больше не нужен.
  // Сетка 200 мс, окно свежести значения ±400 мс — как в exportRrCsv, но потоково.
  List<SubaruPidDef> _cols = const <SubaruPidDef>[];
  final Map<String, double?> _lastValue = <String, double?>{};
  final Map<String, DateTime> _lastTime = <String, DateTime>{};
  DateTime? _t0;
  int _gridMs = 0;
  int _lastSeenMs = -1; // v0.10.1: хвост сетки досписываем при stop()
  static const int _stepMs = 200, _skewMs = 400;

  Future<void> start(List<SubaruPidDef> columns) async {
    await stop();
    if (columns.isEmpty) throw StateError('Нет активных PID для записи');
    _cols = List<SubaruPidDef>.unmodifiable(columns);
    final dir = await getApplicationDocumentsDirectory();
    final f = File(
      '${dir.path}/ssm2_${DateTime.now().millisecondsSinceEpoch}.csv',
    );
    file = f;
    final sink = f.openWrite();
    _sink = sink;
    unawaited(
      sink.done.catchError((Object e) {
        error = '$e';
        active = false;
      }),
    );
    sink.writeln(['Time', ..._cols.map(rrHeader)].map(csvCell).join(','));
    await sink.flush();
    count = 0;
    dropped = 0;
    error = '';
    _lastValue.clear();
    _lastTime.clear();
    _t0 = null;
    _gridMs = 0;
    active = true;
  }

  void add(PidSample sample) {
    final sink = _sink;
    if (!active || sink == null) return;
    if (_pending >= 500) {
      dropped++;
      error = 'CSV backlog limit (потеряно: $dropped)';
      return;
    }
    if (sample.good) {
      _lastValue[sample.pid.id] = sample.value;
      _lastTime[sample.pid.id] = sample.time;
    }
    _t0 ??= sample.time;
    _lastSeenMs = sample.time.difference(_t0!).inMilliseconds; // v0.10.1
    // Слот сетки закрывается, когда от свежего сэмпла до него стало больше ско.
    final closable = sample.time.difference(_t0!).inMilliseconds - _skewMs;
    while (_gridMs <= closable) {
      _writeRow(_gridMs);
      _gridMs += _stepMs;
    }
  }

  void _writeRow(int t) {
    final stamp = _t0!.add(Duration(milliseconds: t));
    final row = <String>[(t / 1000).toStringAsFixed(1)];
    var any = false;
    for (final p in _cols) {
      final v = _lastValue[p.id];
      final pt = _lastTime[p.id];
      if (v != null &&
          pt != null &&
          stamp.difference(pt).inMilliseconds.abs() <= _skewMs) {
        row.add(v.toStringAsFixed(p.digits));
        any = true;
      } else {
        row.add('');
      }
    }
    if (!any) return;
    // v0.10.1 FIX (LOG-04): в 0.10 счётчик был инвертирован (slots - written) —
    // при густой записи сидел на нуле: UI «0 строк», экспорт заблокирован навсегда.
    count++;
    _pending++;
    _writes = _writes
        .then((_) async {
          _sink?.writeln(row.map(csvCell).join(','));
          if (count % 10 == 0) await _sink?.flush();
        })
        .catchError((Object e) {
          error = '$e';
          active = false;
        })
        .whenComplete(() {
          _pending--;
        });
  }

  Future<void> stop() async {
    active = false;
    // v0.10.1: хвост до последнего сэмпла досписываем (свежесть ±400 мс фильтрует).
    while (_lastSeenMs >= 0 && _gridMs <= _lastSeenMs) {
      _writeRow(_gridMs);
      _gridMs += _stepMs;
    }
    await _writes;
    final sink = _sink;
    _sink = null;
    if (sink != null) {
      try {
        await sink.flush();
        await sink.close();
      } catch (e) {
        error = '$e';
      }
    }
  }
}

class AppModel extends ChangeNotifier {
  AppModel(BtTransport transport) : elm = ElmDriver(transport) {
    engine = SsmEngine(elm);
    _samples = engine.events.listen(logger.add);
    _link = transport.status.listen((connected) {
      if (!connected && !busy && !_disposed) {
        message = 'SPP разорван. Переподключите адаптер.';
        unawaited(logger.stop());
        changed();
      }
    });
  }

  final ElmDriver elm;
  late final SsmEngine engine;
  final logger = CsvLogger();
  late final StreamSubscription<PidSample> _samples;
  late final StreamSubscription<bool> _link;
  List<BtDevice> devices = [];
  String? selected;
  String message = 'Сопрягите SPP-адаптер в настройках Android';
  String terminal = '', scanner = '', chartId = 'RPM';
  bool busy = false, foreground = true, _disposed = false;

  void changed() {
    if (!_disposed) notifyListeners();
  }

  Future<void> perform(Future<void> Function() action) async {
    if (busy || _disposed) return;
    busy = true;
    changed();
    try {
      await action();
    } catch (e) {
      message = '$e';
      elm.log('APP $e');
    } finally {
      busy = false;
      changed();
    }
  }

  Future<void> restore() async {
    busy = true;
    try {
      final dir = await getApplicationSupportDirectory();
      final file = File('${dir.path}/ssm2_settings.json');
      if (!await file.exists() || _disposed) return;
      final json =
          jsonDecode(await file.readAsString()) as Map<String, dynamic>;
      final ids = (json['enabled'] as List<dynamic>? ?? [])
          .whereType<String>()
          .where((id) => SubaruPidLibrary.all.any((p) => p.id == id))
          .toSet();
      await engine.configure(
        ids,
        json['confirmed'] == true,
        json['rom'] as String? ?? '',
        json['endian'] == 'little' ? Endian.little : Endian.big,
      );
      final savedAdapter = json['adapter'] as String?; // v0.10
      if (savedAdapter != null && savedAdapter.isNotEmpty)
        selected = savedAdapter;
      changed();
    } catch (e) {
      message = 'Настройки не загружены: $e';
    } finally {
      busy = false;
      changed();
    }
  }

  Future<void> saveSettings() async {
    final dir = await getApplicationSupportDirectory();
    await File('${dir.path}/ssm2_settings.json').writeAsString(
      jsonEncode({
        'enabled': engine.enabled.toList(),
        'confirmed': engine.extendedConfirmed,
        'rom': engine.romId,
        'endian': engine.endian == Endian.big ? 'big' : 'little',
        'adapter': selected, // v0.10: MAC адаптера переживает перезапуск
      }),
      flush: true,
    );
  }

  Future<void> refreshDevices() => perform(() async {
    devices = await elm.transport.paired();
    if (!devices.any((d) => d.address == selected)) {
      selected = devices.isEmpty ? null : devices.first.address;
    }
    message = devices.isEmpty
        ? 'Нет сопряженных устройств'
        : 'Выберите адаптер';
  });

  Future<void> connect() => perform(() async {
    final address = selected;
    if (address == null) throw StateError('Выберите устройство');
    await elm.disconnect();
    await engine.stop();
    await logger.stop();
    await engine.configure(
      {...engine.enabled},
      engine.extendedConfirmed,
      engine.romId,
      engine.endian,
    );

    for (var attempt = 1; attempt <= 3; attempt++) {
      message = 'Инициализация SPP + CAN (попытка $attempt/3)…';
      changed();
      try {
        await elm.initialize(address);
        break;
      } catch (e) {
        if (attempt == 3) rethrow;
        message = 'Попытка $attempt не удалась: $e';
        changed();
        await Future<void>.delayed(const Duration(milliseconds: 900));
      }
    }

    engine.detectedCalId = '';
    if (!elm.ready) {
      message = 'SPP открыт, но CAN/SSM2 sync не получен.';
      changed();
      return;
    }

    message =
        'Подключено · ${elm.activeProfile} · ${elm.identity.isEmpty ? 'ELM327' : elm.identity}';
    changed();
    if (engine.active.isNotEmpty && foreground) {
      await engine.start();
    }
  });

  Future<void> disconnect() => perform(() async {
    await elm.disconnect();
    await engine.stop();
    await logger.stop();
    message = 'Отключено';
  });

  Future<void> togglePolling() => perform(() async {
    if (engine.running) {
      await engine.stop();
      message = 'Опрос на паузе';
    } else {
      await engine.start();
      message = 'Опрос запущен';
    }
  });

  Future<void> exportRrCsv() => perform(() async {
    const skewMs = 400;
    final pids = engine.active;
    if (pids.isEmpty || engine.history.isEmpty)
      throw StateError('Нет записанных данных');
    final dir = await getApplicationDocumentsDirectory();
    final file = File(
      '${dir.path}/ssm2_rr_${DateTime.now().millisecondsSinceEpoch}.csv',
    );
    final sink = file.openWrite();
    sink.writeln(['Time', ...pids.map(rrHeader)].map(csvCell).join(','));
    final t0 = engine.history.values
        .expand((list) => list.map((s) => s.time))
        .reduce((a, b) => a.isBefore(b) ? a : b);
    final cursor = <String, int>{};
    final lastValue = <String, double?>{};
    final lastTime = <String, DateTime>{};
    final t1 = DateTime.now();
    for (var t = 0; t < t1.difference(t0).inMilliseconds; t += 200) {
      final stamp = t0.add(Duration(milliseconds: t));
      final row = <String>['${(t / 1000).toStringAsFixed(1)}'];
      var any = false;
      for (final pid in pids) {
        final list = engine.history[pid.id] ?? const <PidSample>[];
        var i = cursor[pid.id] ?? 0;
        while (i < list.length && !list[i].time.isAfter(stamp)) {
          lastValue[pid.id] = list[i].value;
          lastTime[pid.id] = list[i].time;
          i++;
        }
        cursor[pid.id] = i;
        final pt = lastTime[pid.id];
        final pv = lastValue[pid.id];
        if (pv != null &&
            pt != null &&
            stamp.difference(pt).inMilliseconds.abs() <= skewMs) {
          row.add(pv.toStringAsFixed(pid.digits));
          any = true;
        } else {
          row.add('');
        }
      }
      if (any) sink.writeln(row.map(csvCell).join(','));
    }
    await sink.flush();
    await sink.close();
    await SharePlus.instance.share(
      ShareParams(
        files: [XFile(file.path)],
        text: 'SSM2 RR-совместимый CSV (${file.path})',
      ),
    );
    message = 'RR CSV экспортирован';
    changed();
  });

  Future<void> selectPid(String id, bool value) => perform(() async {
    final next = {...engine.enabled};
    if (value) {
      next.add(id);
    } else {
      next.remove(id);
    }
    await engine.configure(
      next,
      engine.extendedConfirmed,
      engine.romId,
      engine.endian,
    );
    await saveSettings();
    message = 'Выбор сохранен. Нажмите Старт для опроса.';
  });

  Future<void> preset(bool all) => perform(() async {
    await engine.configure(
      all
          ? SubaruPidLibrary.all
                .where((p) => !p.extended)
                .map((p) => p.id)
                .toSet()
          : {...SubaruPidLibrary.defaults},
      engine.extendedConfirmed,
      engine.romId,
      engine.endian,
    );
    await saveSettings();
    message = 'Набор сохранен; опрос на паузе';
  });

  Future<void> configureExtended(bool confirm, String rom, Endian endian) =>
      perform(() async {
        if (confirm && rom.trim().isEmpty) {
          throw ArgumentError('Введите ROM ID из своего def-файла');
        }
        await engine.configure({...engine.enabled}, confirm, rom, endian);
        await saveSettings();
        message = 'Настройки ROM сохранены.';
      });

  Future<void> sendDiagnostic(String command) => perform(() async {
    await engine.stop();
    terminal = '';
    final reply = await elm.diagnostic(command);
    terminal = '> ${command.trim().toUpperCase()}\\r\n${visibleText(reply)}';
    message = 'Ручной запрос завершен; опрос остается на паузе';
  });

  Future<void> scan(String start, String count) => perform(() async {
    final address = int.parse(
      start.trim().replaceFirst(RegExp(r'^0[xX]'), ''),
      radix: 16,
    );
    final length = int.parse(count);
    if (address >= 0xFF0000 && !engine.extendedConfirmed) {
      throw StateError('Сначала подтвердите ROM');
    }
    await engine.stop();
    scanner = '';
    final result = await elm.readRange(address, length);
    scanner = List<String>.generate(
      result.bytes.length,
      (i) =>
          '0x${hexAddress(address + i)}   ${hex2(result.bytes[i])}   ${result.bytes[i]}',
    ).join('\n');
    message =
        'Прочитано ${result.bytes.length} байт за ${result.elapsedMs} мс. Опрос на паузе.';
  });

  Future<void> toggleLog() => perform(() async {
    if (logger.active) {
      await logger.stop();
    } else {
      if (!engine.running || !foreground) {
        throw StateError('Сначала запустите опрос в открытом приложении');
      }
      await logger.start(engine.active); // v0.10: колонки = активные PID
      if (!foreground) await logger.stop();
    }
  });

  Future<void> exportCsv() => perform(() async {
    final wasActive = logger.active; // v0.10
    await logger.stop();
    final file = logger.file;
    if (file == null || logger.count == 0) throw StateError('Нет записей CSV');
    await SharePlus.instance.share(
      ShareParams(files: [XFile(file.path)], text: 'SSM2 PID log (RR-формат)'),
    );
    // v0.10: экспорт раньше молча убивал запись — теперь состояние явное.
    message = wasActive
        ? 'CSV отправлен (RR-формат для Map Lab). Запись остановлена — нажмите «Записать», чтобы продолжить.'
        : 'CSV отправлен (RR-формат для Map Lab)';
    changed();
  });

  Future<void> exportTrace() => perform(() async {
    final dir = await getApplicationDocumentsDirectory();
    final file = File(
      '${dir.path}/ssm2_trace_${DateTime.now().millisecondsSinceEpoch}.txt',
    );
    await file.writeAsString(
      '${elm.transport.name}\n'
      'ROM (user): ${engine.romId}\n'
      'CAN profile: ${elm.activeProfile}\n'
      'Timeout: ${elm.timeoutMs} ms\n'
      'Sync note: ${elm.syncNote}\n'
      'Endian: ${engine.endian == Endian.big ? 'big' : 'little'}\n'
      '${elm.trace.join('\n')}\n',
      flush: true,
    );
    await SharePlus.instance.share(
      ShareParams(
        files: [XFile(file.path)],
        text: 'SSM2 TX/RX diagnostic trace',
      ),
    );
  });

  Future<void> shutdown() async {
    if (_disposed) return;
    _disposed = true;
    await _link.cancel();
    try {
      await elm.disconnect();
    } catch (e) {
      elm.log('SHUTDOWN $e');
    }
    await engine.stop();
    await _samples.cancel();
    await logger.stop();
    try {
      await elm.dispose();
    } catch (e) {
      elm.log('DISPOSE $e');
    }
    engine.dispose();
    super.dispose();
  }
}
'''

FILES["lib/main.dart"] = r'''
import 'dart:async';
import 'dart:io';
import 'dart:math' as math;
import 'package:path_provider/path_provider.dart';
import 'package:share_plus/share_plus.dart';
import 'package:flutter/foundation.dart';
import 'package:flutter/material.dart';
import 'package:flutter/services.dart';
import 'analyzer.dart';
import 'derived.dart';
import 'engine.dart';
import 'maplab_link.dart';
import 'model.dart';
import 'pids.dart';
import 'protocol.dart';
import 'transport_selected.dart';

void main() {
  WidgetsFlutterBinding.ensureInitialized();
  runApp(const SsmApp());
}

const cyan = Color(0xFF22D3EE);
const muted = Color(0xFF9AAAC0);

class SsmApp extends StatelessWidget {
  const SsmApp({super.key});
  @override
  Widget build(BuildContext context) => MaterialApp(
        title: 'SSM2 Telemetry 0.10.2',
        debugShowCheckedModeBanner: false,
        theme: ThemeData(
          colorScheme: ColorScheme.fromSeed(
            seedColor: cyan,
            brightness: Brightness.dark,
          ),
          scaffoldBackgroundColor: const Color(0xFF080D18),
          useMaterial3: true,
        ),
        home: const HomeShell(),
      );
}

class HomeShell extends StatefulWidget {
  const HomeShell({super.key});
  @override
  State<HomeShell> createState() => _HomeShellState();
}

class _HomeShellState extends State<HomeShell> with WidgetsBindingObserver {
  late final AppModel model;
  late final Listenable changes;
  late final Timer timer;
  int tab = 0;
  @override
  void initState() {
    super.initState();
    model = AppModel(createTransport());
    changes = Listenable.merge([model, model.engine]);
    WidgetsBinding.instance.addObserver(this);
    unawaited(model.restore());
    timer = Timer.periodic(const Duration(milliseconds: 500), (_) {
      if (mounted) setState(() {});
    });
  }

  @override
  void didChangeAppLifecycleState(AppLifecycleState state) {
    if (state == AppLifecycleState.resumed) model.foreground = true;
    if (state == AppLifecycleState.paused) {
      model.foreground = false;
      unawaited(model.engine.stop());
      unawaited(model.logger.stop());
    }
  }

  @override
  void dispose() {
    timer.cancel();
    WidgetsBinding.instance.removeObserver(this);
    unawaited(model.shutdown());
    super.dispose();
  }

  @override
  Widget build(BuildContext context) => AnimatedBuilder(
        animation: changes,
        builder: (context, _) {
          final engine = model.engine;
          final lastTimes = engine.latest.values.map((s) => s.time).toList()
            ..sort();
          final live = model.elm.ready &&
              lastTimes.isNotEmpty &&
              DateTime.now().difference(lastTimes.last).inSeconds < 3;
          final pages = <Widget>[
            AdapterPage(model),
            DashboardPage(model),
            PidPage(model),
            LoggerPage(model),
            GraphPage(model),
            AnalyzerPage(model),
            DiagnosticPage(model),
            const MapLabTab(),
          ];
          return Scaffold(
            appBar: AppBar(
              title: const Text(
                'SSM2 TELEMETRY 0.10.2',
                style: TextStyle(fontSize: 17, letterSpacing: 2),
              ),
              actions: [
                Icon(
                  Icons.circle,
                  size: 10,
                  color: live ? Colors.greenAccent : muted,
                ),
                const SizedBox(width: 16),
              ],
            ),
            body: SafeArea(
              child: Column(
                children: [
                  if (model.busy) const LinearProgressIndicator(minHeight: 2),
                  Padding(
                    padding: const EdgeInsets.fromLTRB(16, 4, 16, 8),
                    child: Align(
                      alignment: Alignment.centerLeft,
                      child: Text(
                        model.message,
                        maxLines: 3,
                        overflow: TextOverflow.ellipsis,
                        style: const TextStyle(color: muted, fontSize: 12),
                      ),
                    ),
                  ),
                  Expanded(child: pages[tab]),
                ],
              ),
            ),
            bottomNavigationBar: NavigationBar(
              selectedIndex: tab,
              labelBehavior:
                  NavigationDestinationLabelBehavior.onlyShowSelected,
              onDestinationSelected: (i) => setState(() => tab = i),
              destinations: const [
                NavigationDestination(
                  icon: Icon(Icons.bluetooth),
                  label: 'Адаптер',
                ),
                NavigationDestination(
                    icon: Icon(Icons.speed), label: 'Дашборд'),
                NavigationDestination(icon: Icon(Icons.tune), label: 'PID'),
                NavigationDestination(
                  icon: Icon(Icons.fiber_manual_record_outlined),
                  label: 'CSV',
                ),
                NavigationDestination(
                  icon: Icon(Icons.show_chart),
                  label: 'График',
                ),
                NavigationDestination(
                    icon: Icon(Icons.insights), label: 'Анализ'),
                NavigationDestination(
                    icon: Icon(Icons.terminal), label: 'Диагн.'),
                NavigationDestination(
                  icon: Icon(Icons.table_view),
                  label: 'Map Lab',
                ),
              ],
            ),
          );
        },
      );
}

Widget section(String title, List<Widget> children) => Padding(
      padding: const EdgeInsets.fromLTRB(16, 16, 16, 10),
      child: Column(
        crossAxisAlignment: CrossAxisAlignment.start,
        children: [
          Text(
            title,
            style: const TextStyle(fontSize: 15, fontWeight: FontWeight.bold),
          ),
          const SizedBox(height: 12),
          ...children,
        ],
      ),
    );
Widget detail(String key, String value) => Padding(
      padding: const EdgeInsets.symmetric(vertical: 4),
      child: Row(
        crossAxisAlignment: CrossAxisAlignment.start,
        children: [
          Expanded(
            flex: 2,
            child:
                Text(key, style: const TextStyle(color: muted, fontSize: 12)),
          ),
          const SizedBox(width: 10),
          Expanded(
            flex: 3,
            child: Text(value, style: const TextStyle(fontSize: 12)),
          ),
        ],
      ),
    );
String ageText(int? age) =>
    age == null ? 'нет данных' : '${(age / 1000).toStringAsFixed(1)} с';

class AdapterPage extends StatelessWidget {
  const AdapterPage(this.model, {super.key});
  final AppModel model;
  @override
  Widget build(BuildContext context) => ListView(
        children: [
          section('Bluetooth SPP', [
            Text(
              model.elm.transport.name,
              style: const TextStyle(color: cyan, fontSize: 12),
            ),
            const SizedBox(height: 10),
            const Text(
              'Только CAN 11 bit / 500 kbit. Зажигание включено, автомобиль стоит.',
            ),
            const SizedBox(height: 10),
            Wrap(
              spacing: 8,
              children: [
                OutlinedButton.icon(
                  onPressed: model.busy ? null : model.refreshDevices,
                  icon: const Icon(Icons.refresh),
                  label: const Text('Сопряженные'),
                ),
                TextButton(
                  onPressed: () => model.perform(() async {
                    await const MethodChannel(
                      'ssm2/system',
                    ).invokeMethod<void>('bluetoothSettings');
                  }),
                  child: const Text('Настройки Android'),
                ),
              ],
            ),
            if (model.devices.isEmpty)
              const Padding(
                padding: EdgeInsets.all(12),
                child: Text('Обновите список устройств'),
              ),
            for (final device in model.devices)
              ListTile(
                contentPadding: EdgeInsets.zero,
                leading: const Icon(Icons.bluetooth, color: cyan),
                title: Text(device.name.isEmpty ? 'Без имени' : device.name),
                subtitle: Text(device.address),
                trailing: model.selected == device.address
                    ? const Icon(Icons.check, color: cyan)
                    : null,
                onTap: model.busy
                    ? null
                    : () {
                        model.selected = device.address;
                        model.changed();
                      },
              ),
            Wrap(
              spacing: 8,
              children: [
                FilledButton(
                  onPressed: model.busy || model.selected == null
                      ? null
                      : model.connect,
                  child:
                      Text(model.elm.ready ? 'Переподключить' : 'Подключить'),
                ),
                OutlinedButton(
                  onPressed: model.busy ? null : model.disconnect,
                  child: const Text('Отключить'),
                ),
              ],
            ),
          ]),
          section('Состояние', [
            detail(
              'Синхронизация',
              model.elm.ready ? 'ДА ✓' : 'нет — нажмите Подключить',
            ),
            detail(
              'Адаптер',
              model.elm.identity.isEmpty ? 'не опрошен' : model.elm.identity,
            ),
            detail(
              'ATRV',
              model.elm.voltage.isEmpty ? 'не опрошен' : model.elm.voltage,
            ),
            detail(
              'Watchdog',
              '${model.engine.mutedAddresses} PID изолировано · запросов: ${model.engine.requests}',
            ),
            detail('SPP', model.elm.transport.connected ? 'открыт' : 'закрыт'),
            detail(
              'Таймаут CAN',
              '${model.elm.timeoutMs} мс',
            ), // v0.10: вместо дубля строки sync
          ]),
        ],
      );
}

class DashboardPage extends StatelessWidget {
  const DashboardPage(this.model, {super.key});
  final AppModel model;
  @override
  Widget build(BuildContext context) {
    final engine = model.engine;
    final pids = engine.active;
    final fuel = estimateFuel(
      maf: engine.latest['MAF']?.value,
      afr: engine.latest['AFR']?.value,
      speed: engine.latest['SPEED']?.value,
      rpm: engine.latest['RPM']?.value,
      pedal: engine.latest['PEDAL']?.value,
    );
    return Column(
      children: [
        Padding(
          padding: const EdgeInsets.symmetric(horizontal: 16),
          child: Row(
            children: [
              Expanded(
                child: Text(
                  'SAFE Single Frame / ${pids.length} PID\n${engine.pidReadsPerSecond.toStringAsFixed(1)} PID-обновл./с · ${engine.requestsPerSecond.toStringAsFixed(1)} запросов/с',
                  style: const TextStyle(fontSize: 12, color: muted),
                ),
              ),
              FilledButton.tonalIcon(
                onPressed:
                    model.busy || !model.elm.ready ? null : model.togglePolling,
                icon: Icon(engine.running ? Icons.pause : Icons.play_arrow),
                label: Text(engine.running ? 'Пауза' : 'Старт'),
              ),
            ],
          ),
        ),
        FuelCard(
          fuel: fuel,
          mafStale: engine.stale('MAF'),
          tripKm: engine.tripDistanceKm,
          tripL: engine.tripFuelLitres,
          onReset: engine.resetTrip,
        ),
        if (pids.isEmpty)
          const Expanded(
            child: Center(child: Text('Выберите параметры во вкладке PID')),
          )
        else
          Expanded(
            child: GridView.builder(
              padding: const EdgeInsets.all(12),
              itemCount: pids.length,
              gridDelegate: const SliverGridDelegateWithMaxCrossAxisExtent(
                maxCrossAxisExtent: 270,
                mainAxisExtent: 188,
                mainAxisSpacing: 10,
                crossAxisSpacing: 10,
              ),
              itemBuilder: (context, i) {
                final p = pids[i];
                final sample = engine.latest[p.id];
                final unsupported = sample?.allOnes ?? false;
                final value = unsupported ? null : sample?.value;
                final outdated = engine.stale(p.id);
                final failed = engine.attempts[p.id]?.error.isNotEmpty ?? false;
                final tone = outdated || failed || unsupported ? muted : cyan;
                return Material(
                  color: const Color(0xFF101A2B),
                  borderRadius: BorderRadius.circular(14),
                  child: InkWell(
                    borderRadius: BorderRadius.circular(14),
                    onTap: () => showModalBottomSheet<void>(
                      context: context,
                      isScrollControlled: true,
                      builder: (context) => SafeArea(
                        child: SingleChildScrollView(
                          child: section(p.id, [
                            Text(p.desc),
                            detail(
                              'Адреса',
                              p.addresses
                                  .map((a) => '0x${hexAddress(a)}')
                                  .join(', '),
                            ),
                            detail('Формула', p.formulaText),
                            detail(
                              'Сырые байты',
                              engine.latest[p.id]?.raw.map(hex2).join(' ') ??
                                  'нет',
                            ),
                            detail('Возраст', ageText(engine.ageMs(p.id))),
                            detail(
                              'Частота этого PID',
                              '${engine.frequency(p.id).toStringAsFixed(2)} Hz',
                            ),
                            detail(
                              'Окно чтения',
                              '${engine.latest[p.id]?.readMs ?? 0} ms',
                            ),
                            detail(
                              'Последняя ошибка',
                              engine.attempts[p.id]?.error ?? '',
                            ),
                          ]),
                        ),
                      ),
                    ),
                    child: Padding(
                      padding: const EdgeInsets.all(14),
                      child: Column(
                        crossAxisAlignment: CrossAxisAlignment.start,
                        children: [
                          Row(
                            children: [
                              Expanded(
                                child: Text(
                                  p.id,
                                  style: const TextStyle(
                                    fontSize: 13,
                                    fontWeight: FontWeight.bold,
                                  ),
                                ),
                              ),
                              if (outdated && value != null)
                                const Icon(
                                  Icons.schedule,
                                  size: 14,
                                  color: muted,
                                ),
                            ],
                          ),
                          Text(
                            '${p.unit} / 0x${hexAddress(p.address)}',
                            style: const TextStyle(fontSize: 10, color: muted),
                          ),
                          const Spacer(),
                          SizedBox(
                            height: 46,
                            width: double.infinity,
                            child: FittedBox(
                              fit: BoxFit.scaleDown,
                              alignment: Alignment.centerLeft,
                              child: Text(
                                value?.toStringAsFixed(p.digits) ?? '--',
                                style: TextStyle(
                                  color: tone,
                                  fontSize: 38,
                                  fontWeight: FontWeight.w700,
                                ),
                              ),
                            ),
                          ),
                          const SizedBox(height: 10),
                          LinearProgressIndicator(
                            value: value == null
                                ? 0
                                : ((value - p.minValue) /
                                        (p.maxValue - p.minValue))
                                    .clamp(0.0, 1.0)
                                    .toDouble(),
                            color: tone,
                            minHeight: 3,
                          ),
                          const SizedBox(height: 8),
                          Text(
                            unsupported
                                ? '0xFF: адрес не поддерживается'
                                : failed
                                    ? 'ошибка / ${ageText(engine.ageMs(p.id))}'
                                    : '${ageText(engine.ageMs(p.id))} · ${(sample?.raw ?? []).map(hex2).join(' ')}',
                            style: TextStyle(
                              color:
                                  unsupported ? const Color(0xFFD4B57F) : muted,
                              fontSize: 10,
                            ),
                          ),
                        ],
                      ),
                    ),
                  ),
                );
              },
            ),
          ),
      ],
    );
  }
}

class FuelCard extends StatelessWidget {
  const FuelCard({
    super.key,
    required this.fuel,
    required this.mafStale,
    this.tripKm = 0,
    this.tripL = 0,
    this.onReset,
  });
  final FuelEstimate? fuel;
  final bool mafStale;
  final double tripKm, tripL;
  final VoidCallback? onReset;

  @override
  Widget build(BuildContext context) {
    final f = fuel;
    return Container(
      margin: const EdgeInsets.fromLTRB(12, 10, 12, 0),
      padding: const EdgeInsets.all(14),
      decoration: BoxDecoration(
        color: const Color(0xFF101A2B),
        borderRadius: BorderRadius.circular(14),
        border: Border.all(color: const Color(0xFF24435A)),
      ),
      child: Column(
        crossAxisAlignment: CrossAxisAlignment.start,
        children: [
          Row(
            children: [
              const Expanded(
                child: Text(
                  'МГНОВЕННЫЙ РАСХОД',
                  style: TextStyle(
                    fontSize: 11,
                    letterSpacing: 1.2,
                    color: muted,
                  ),
                ),
              ),
              Text(
                f == null ? 'нет MAF' : 'расчет по MAF',
                style: const TextStyle(fontSize: 10, color: muted),
              ),
            ],
          ),
          const SizedBox(height: 10),
          if (f == null)
            const Text(
              'Включите PID MAF и запустите опрос.',
              style: TextStyle(fontSize: 12, color: muted),
            )
          else ...[
            Row(
              crossAxisAlignment: CrossAxisAlignment.end,
              children: [
                Text(
                  f.litresPerHour.toStringAsFixed(2),
                  style: TextStyle(
                    fontSize: 38,
                    fontWeight: FontWeight.w700,
                    color: mafStale ? muted : cyan,
                    height: 1,
                  ),
                ),
                const Padding(
                  padding: EdgeInsets.only(left: 8, bottom: 4),
                  child: Text(
                    'л/ч',
                    style: TextStyle(fontSize: 13, color: muted),
                  ),
                ),
                const Spacer(),
                Text(
                  f.litresPer100km == null
                      ? 'на месте'
                      : '${f.litresPer100km!.toStringAsFixed(1)} л/100км',
                  style: TextStyle(
                    fontSize: 16,
                    fontWeight: FontWeight.w600,
                    color: mafStale ? muted : Colors.white,
                  ),
                ),
              ],
            ),
            const SizedBox(height: 10),
            Text(
              'AFR ${f.afrUsed.toStringAsFixed(1)}'
              '${f.afrMeasured ? ' (из ECU)' : ' (стехиометрия)'}'
              ' · плотность $kPetrolDensityGramsPerLitre г/л',
              style: const TextStyle(fontSize: 10, color: muted, height: 1.5),
            ),
            const SizedBox(height: 10),
            Row(
              children: [
                const Icon(Icons.speed, size: 14, color: muted),
                const SizedBox(width: 6),
                Text(
                  '${tripKm.toStringAsFixed(1)} км',
                  style: const TextStyle(
                    color: cyan,
                    fontSize: 13,
                    fontWeight: FontWeight.bold,
                    fontFamily: 'monospace',
                  ),
                ),
                const SizedBox(width: 16),
                const Icon(Icons.local_gas_station, size: 14, color: muted),
                const SizedBox(width: 6),
                Text(
                  '${tripL.toStringAsFixed(2)} л',
                  style: const TextStyle(
                    color: cyan,
                    fontSize: 13,
                    fontWeight: FontWeight.bold,
                    fontFamily: 'monospace',
                  ),
                ),
                if (tripKm >= 0.5) ...[
                  const SizedBox(width: 16),
                  Text(
                    '${(tripL / tripKm * 100).toStringAsFixed(1)} л/100км',
                    style: const TextStyle(color: muted, fontSize: 11),
                  ),
                ],
                const Spacer(),
                SizedBox(
                  height: 28,
                  child: OutlinedButton.icon(
                    onPressed: onReset,
                    icon: const Icon(Icons.restart_alt, size: 14),
                    label: const Text('Сброс', style: TextStyle(fontSize: 11)),
                    style: OutlinedButton.styleFrom(
                      padding: const EdgeInsets.symmetric(horizontal: 8),
                      foregroundColor: muted,
                    ),
                  ),
                ),
              ],
            ),
          ],
        ],
      ),
    );
  }
}

class AnalyzerPage extends StatefulWidget {
  const AnalyzerPage(this.model, {super.key});
  final AppModel model;
  @override
  State<AnalyzerPage> createState() => _AnalyzerPageState();
}

class _AnalyzerPageState extends State<AnalyzerPage> {
  LogAnalysis? report;
  bool auto = true;
  int lastSignature = -1;
  DateTime lastAutoRun = DateTime.fromMillisecondsSinceEpoch(0);
  String errors = '0';
  int attemptsTotal = 0;

  @override
  void initState() {
    super.initState();
    widget.model.engine.addListener(_onEngine);
  }

  @override
  void didUpdateWidget(covariant AnalyzerPage oldWidget) {
    super.didUpdateWidget(oldWidget);
    if (!identical(oldWidget.model.engine, widget.model.engine)) {
      oldWidget.model.engine.removeListener(_onEngine);
      widget.model.engine.addListener(_onEngine);
    }
  }

  @override
  void dispose() {
    widget.model.engine.removeListener(_onEngine);
    super.dispose();
  }

  void _onEngine() {
    if (!auto || !mounted || widget.model.engine.history.isEmpty) return;
    final signature = widget.model.engine.goodCount ~/ 10;
    if (signature == lastSignature) return;
    lastSignature = signature;
    if (DateTime.now().difference(lastAutoRun).inSeconds < 3) return;
    lastAutoRun = DateTime.now();
    _schedule();
  }

  void _schedule() {
    final withErrors = widget.model.engine.attempts.values
        .where((s) => s.error.isNotEmpty)
        .map((s) => s.pid.id)
        .toList()
      ..sort();
    attemptsTotal = widget.model.engine.attempts.length;
    errors = withErrors.isEmpty
        ? '0'
        : withErrors.length <= 8
            ? withErrors.join(', ')
            : '${withErrors.take(8).join(', ')} +${withErrors.length - 8}';
    setState(() {
      report = analyzeLog(
        widget.model.engine.history,
        attempts: widget.model.engine.attempts,
      );
    });
  }

  Future<void> _exportReport() async {
    final engine = widget.model.engine;
    final result =
        report ?? analyzeLog(engine.history, attempts: engine.attempts);
    final md = StringBuffer('# SSM2 0.10 · Анализ журнала\n\n');
    md.writeln(
      'Значений: ${result.sampleCount} · длительность: '
      '${result.spanSeconds.toStringAsFixed(1)} с · каналов: ${result.channels}',
    );
    for (final f in result.findings) {
      md.writeln('\n## [${_levelName(f.level)}] ${f.title}\n');
      md.writeln('${f.detail}\n');
      md.writeln('```\n${f.evidence}\n```\n');
      md.writeln('> ${f.advice}');
    }
    final dir = await getApplicationDocumentsDirectory();
    final file = File(
      '${dir.path}/ssm2_report_${DateTime.now().millisecondsSinceEpoch}.md',
    );
    await file.writeAsString(md.toString(), flush: true);
    await SharePlus.instance.share(
      ShareParams(files: [XFile(file.path)], text: 'SSM2 анализ журнала (md)'),
    );
  }

  Color _levelColor(FindingLevel level) => switch (level) {
        FindingLevel.critical => const Color(0xFFE08A7A),
        FindingLevel.warning => const Color(0xFFD4B57F),
        FindingLevel.info => muted,
      };
  String _levelName(FindingLevel level) => switch (level) {
        FindingLevel.critical => 'КРИТИЧНО',
        FindingLevel.warning => 'ВНИМАНИЕ',
        FindingLevel.info => 'ИНФО',
      };

  @override
  Widget build(BuildContext context) {
    final engine = widget.model.engine;
    final result = report;
    return ListView(
      children: [
        section('Анализ журнала', [
          const Text(
            'Прозрачные правила с фиксированными порогами.',
            style: TextStyle(color: muted, height: 1.6),
          ),
          const SizedBox(height: 12),
          Wrap(
            spacing: 8,
            runSpacing: 8,
            children: [
              FilledButton.icon(
                onPressed: engine.history.isEmpty ? null : _schedule,
                icon: const Icon(Icons.insights, size: 18),
                label: const Text('Проанализировать'),
              ),
              OutlinedButton.icon(
                onPressed: () => setState(() => auto = !auto),
                icon: Icon(
                  auto ? Icons.check_box : Icons.check_box_outline_blank,
                  size: 18,
                ),
                label: const Text('Авто'),
              ),
              OutlinedButton.icon(
                onPressed: engine.history.isEmpty || widget.model.busy
                    ? null
                    : _exportReport,
                icon: const Icon(Icons.share, size: 16),
                label: const Text('Отчёт.md'),
              ),
            ],
          ),
        ]),
        if (result != null) ...[
          section('Итог', [
            detail('Значений в памяти', '${result.sampleCount}'),
            detail(
              'Длительность',
              '${result.spanSeconds.toStringAsFixed(1)} с',
            ),
            detail('Каналов с данными', '${result.channels}'),
          ]),
          for (final f in result.findings)
            Padding(
              padding: const EdgeInsets.fromLTRB(16, 0, 16, 12),
              child: Container(
                padding: const EdgeInsets.all(14),
                decoration: BoxDecoration(
                  color: const Color(0xFF101A2B),
                  borderRadius: BorderRadius.circular(12),
                  border: Border(
                    left: BorderSide(color: _levelColor(f.level), width: 3),
                  ),
                ),
                child: Column(
                  crossAxisAlignment: CrossAxisAlignment.start,
                  children: [
                    Text(
                      _levelName(f.level),
                      style: TextStyle(
                        fontSize: 9,
                        letterSpacing: 1.2,
                        color: _levelColor(f.level),
                      ),
                    ),
                    const SizedBox(height: 6),
                    Text(
                      f.title,
                      style: const TextStyle(
                        fontSize: 14,
                        fontWeight: FontWeight.w600,
                      ),
                    ),
                    const SizedBox(height: 8),
                    Text(
                      f.detail,
                      style: const TextStyle(fontSize: 12, height: 1.6),
                    ),
                  ],
                ),
              ),
            ),
        ],
      ],
    );
  }
}

class PidPage extends StatefulWidget {
  const PidPage(this.model, {super.key});
  final AppModel model;
  @override
  State<PidPage> createState() => _PidPageState();
}

class _PidPageState extends State<PidPage> {
  String search = '';
  @override
  Widget build(BuildContext context) {
    final m = widget.model;
    return ListView(
      children: [
        section('Библиотека / 28 PID', [
          const Text(
            'Сначала 8 базовых. Больше параметров — ниже частота каждого PID.',
            style: TextStyle(color: muted),
          ),
          Wrap(
            spacing: 8,
            children: [
              TextButton(
                onPressed: m.busy ? null : () => m.preset(false),
                child: const Text('8 базовых'),
              ),
              TextButton(
                onPressed: m.busy ? null : () => m.preset(true),
                child: const Text('Все 20 обычных'),
              ),
              TextButton(
                onPressed: m.busy ? null : () => _settings(context),
                child: const Text('ROM / Float32'),
              ),
            ],
          ),
          TextField(
            decoration: const InputDecoration(
              labelText: 'Поиск PID или адреса',
              prefixIcon: Icon(Icons.search),
            ),
            onChanged: (value) => setState(() => search = value.toUpperCase()),
          ),
          const SizedBox(height: 8),
          for (final p in SubaruPidLibrary.all.where(
            (p) => '${p.id} ${p.desc} 0x${hexAddress(p.address)}'
                .toUpperCase()
                .contains(search.trim()),
          ))
            CheckboxListTile(
              contentPadding: EdgeInsets.zero,
              controlAffinity: ListTileControlAffinity.leading,
              title: Text(
                '${p.id}${p.extended ? ' *' : ''} / ${p.unit}',
                style: const TextStyle(fontSize: 13),
              ),
              subtitle: Text(
                '0x${hexAddress(p.address)} / ${p.bytesCount} B / P${p.priority}\n${p.formulaText}',
                style: const TextStyle(fontSize: 10, color: muted),
              ),
              value: m.engine.enabled.contains(p.id),
              onChanged: m.busy || (p.extended && !m.engine.extendedConfirmed)
                  ? null
                  : (value) => m.selectPid(p.id, value ?? false),
            ),
        ]),
      ],
    );
  }

  Future<void> _settings(BuildContext context) async {
    final m = widget.model;
    final controller = TextEditingController(text: m.engine.romId);
    var confirm = m.engine.extendedConfirmed;
    var little = m.engine.endian == Endian.little;
    final accepted = await showDialog<bool>(
      context: context,
      builder: (context) => StatefulBuilder(
        builder: (context, change) => AlertDialog(
          title: const Text('Extended / ROM'),
          content: SingleChildScrollView(
            child: Column(
              mainAxisSize: MainAxisSize.min,
              children: [
                const Text(
                  'Введите ROM ID из своего def-файла для включения extended PID.',
                ),
                TextField(
                  controller: controller,
                  decoration: const InputDecoration(labelText: 'ROM ID'),
                ),
                CheckboxListTile(
                  title: const Text('Адреса сверены с моим ROM'),
                  value: confirm,
                  onChanged: (value) => change(() => confirm = value ?? false),
                ),
                SwitchListTile(
                  title: const Text('Float32 little-endian'),
                  value: little,
                  onChanged: (value) => change(() => little = value),
                ),
              ],
            ),
          ),
          actions: [
            TextButton(
              onPressed: () => Navigator.pop(context, false),
              child: const Text('Отмена'),
            ),
            FilledButton(
              onPressed: () => Navigator.pop(context, true),
              child: const Text('Сохранить'),
            ),
          ],
        ),
      ),
    );
    final rom = controller.text;
    controller.dispose();
    if (accepted == true)
      await m.configureExtended(
        confirm,
        rom,
        little ? Endian.little : Endian.big,
      );
  }
}

class LoggerPage extends StatelessWidget {
  const LoggerPage(this.model, {super.key});
  final AppModel model;
  @override
  Widget build(BuildContext context) => ListView(
        children: [
          section('Потоковый CSV', [
            Text(
              '${model.logger.count}',
              style: const TextStyle(
                fontSize: 54,
                fontWeight: FontWeight.bold,
                color: cyan,
              ),
            ),
            Text(
              model.logger.active
                  ? 'Идет запись (RR-формат)'
                  : 'Запись остановлена',
            ),
            // v0.10 FIX (LOG-03): ошибка логгера видна пользователю, а не тонет в поле объекта.
            if (model.logger.error.isNotEmpty)
              Padding(
                padding: const EdgeInsets.only(top: 6),
                child: Text(
                  'Логгер: ${model.logger.error}',
                  style:
                      const TextStyle(color: Color(0xFFE08A7A), fontSize: 12),
                ),
              ),
            const SizedBox(height: 18),
            Wrap(
              spacing: 8,
              children: [
                FilledButton.icon(
                  onPressed: model.busy ? null : model.toggleLog,
                  icon: Icon(
                    model.logger.active
                        ? Icons.stop
                        : Icons.fiber_manual_record,
                  ),
                  label: Text(model.logger.active ? 'Стоп' : 'Записать'),
                ),
                OutlinedButton.icon(
                  onPressed: model.busy || model.logger.count == 0
                      ? null
                      : model.exportCsv,
                  icon: const Icon(Icons.share),
                  label: const Text('Экспорт CSV'),
                ),
                OutlinedButton.icon(
                  onPressed: model.busy || model.engine.history.isEmpty
                      ? null
                      : model.exportRrCsv,
                  icon: const Icon(Icons.table_chart),
                  label: const Text('RR CSV'),
                ),
              ],
            ),
          ]),
        ],
      );
}

class GraphPage extends StatelessWidget {
  const GraphPage(this.model, {super.key});
  final AppModel model;
  @override
  Widget build(BuildContext context) {
    final pid = SubaruPidLibrary.byId(model.chartId);
    final samples = List<PidSample>.of(model.engine.history[pid.id] ?? []);
    return ListView(
      children: [
        section('График', [
          DropdownButton<String>(
            value: model.chartId,
            isExpanded: true,
            items: SubaruPidLibrary.all
                .map(
                  (p) => DropdownMenuItem(
                    value: p.id,
                    child: Text('${p.id} / ${p.unit}'),
                  ),
                )
                .toList(),
            onChanged: (value) {
              if (value != null) {
                model.chartId = value;
                model.changed();
              }
            },
          ),
          const SizedBox(height: 18),
          SizedBox(
            height: 270,
            child: samples.length < 2
                ? const Center(child: Text('Нужны хотя бы два значения'))
                : CustomPaint(
                    painter: TelemetryPainter(samples, pid.digits),
                    size: Size.infinite,
                  ),
          ),
        ]),
      ],
    );
  }
}

class TelemetryPainter extends CustomPainter {
  TelemetryPainter(this.samples, this.digits);
  final List<PidSample> samples;
  final int digits;
  void label(Canvas canvas, String text, Offset point) {
    final painter = TextPainter(
      text: TextSpan(
        text: text,
        style: const TextStyle(color: muted, fontSize: 10),
      ),
      textDirection: TextDirection.ltr,
    )..layout();
    painter.paint(canvas, point);
  }

  @override
  void paint(Canvas canvas, Size size) {
    final values = samples.map((s) => s.value!).toList();
    final minimum = values.reduce(math.min), maximum = values.reduce(math.max);
    final pad = math.max((maximum - minimum) * 0.1, 0.5);
    final lo = minimum - pad, hi = maximum + pad;
    final first = samples.first.time.millisecondsSinceEpoch;
    final span = math.max(1, samples.last.time.millisecondsSinceEpoch - first);
    final width = math.max(1.0, size.width - 65), height = size.height - 38;
    final grid = Paint()
      ..color = const Color(0xFF233045)
      ..strokeWidth = 1;
    for (var i = 0; i <= 4; i++) {
      final y = 8 + height * i / 4;
      canvas.drawLine(Offset(52, y), Offset(size.width - 8, y), grid);
      label(
        canvas,
        (hi - (hi - lo) * i / 4).toStringAsFixed(digits > 1 ? 1 : digits),
        Offset(0, y - 5),
      );
    }
    final path = Path();
    for (var i = 0; i < samples.length; i++) {
      final sample = samples[i];
      final x =
          52 + width * (sample.time.millisecondsSinceEpoch - first) / span;
      final y = 8 + height * (hi - sample.value!) / (hi - lo);
      canvas.drawCircle(Offset(x, y), 2, Paint()..color = cyan);
      if (i == 0 ||
          sample.time.difference(samples[i - 1].time).inMilliseconds > 3000) {
        path.moveTo(x, y);
      } else {
        path.lineTo(x, y);
      }
    }
    canvas.drawPath(
      path,
      Paint()
        ..color = cyan
        ..strokeWidth = 2
        ..style = PaintingStyle.stroke,
    );
    label(canvas, '0 s', Offset(52, height + 20));
    label(
      canvas,
      '${(span / 1000).toStringAsFixed(1)} s',
      Offset(size.width - 55, height + 20),
    );
  }

  @override
  bool shouldRepaint(covariant TelemetryPainter oldDelegate) => true;
}

class DiagnosticPage extends StatefulWidget {
  const DiagnosticPage(this.model, {super.key});
  final AppModel model;
  @override
  State<DiagnosticPage> createState() => _DiagnosticPageState();
}

class _DiagnosticPageState extends State<DiagnosticPage> {
  final command = TextEditingController(text: 'ATI');
  final address = TextEditingController(text: '000008');
  final count = TextEditingController(text: '1');
  @override
  void dispose() {
    command.dispose();
    address.dispose();
    count.dispose();
    super.dispose();
  }

  @override
  Widget build(BuildContext context) {
    final m = widget.model;
    return ListView(
      children: [
        section('Диагностика', [
          TextField(
            controller: command,
            decoration: const InputDecoration(
              labelText: 'ATI / ATRV / A8 00 00 00 08',
            ),
          ),
          const SizedBox(height: 8),
          FilledButton(
            onPressed: m.busy || !m.elm.ready
                ? null
                : () => m.sendDiagnostic(command.text),
            child: const Text('Отправить'),
          ),
          SelectableText(
            m.terminal.isEmpty ? 'Команд еще нет' : m.terminal,
            style: const TextStyle(fontFamily: 'monospace', fontSize: 12),
          ),
        ]),
        section('Сканер адресов A8', [
          Row(
            children: [
              Expanded(
                child: TextField(
                  controller: address,
                  decoration: const InputDecoration(labelText: 'Адрес HEX'),
                ),
              ),
              const SizedBox(width: 12),
              SizedBox(
                width: 90,
                child: TextField(
                  controller: count,
                  keyboardType: TextInputType.number,
                  decoration: const InputDecoration(labelText: '1..32 байт'),
                ),
              ),
            ],
          ),
          const SizedBox(height: 8),
          OutlinedButton(
            onPressed: m.busy || !m.elm.ready
                ? null
                : () => m.scan(address.text, count.text),
            child: const Text('Прочитать'),
          ),
          SelectableText(
            m.scanner.isEmpty ? 'Нет дампа' : m.scanner,
            style: const TextStyle(fontFamily: 'monospace', fontSize: 12),
          ),
        ]),
      ],
    );
  }
}
'''

FILES["test/protocol_test.dart"] = r'''
import 'dart:async';
import 'dart:typed_data';

import 'package:flutter_test/flutter_test.dart';
import 'package:subaru_ssm2/analyzer.dart';
import 'package:subaru_ssm2/bt_transport.dart';
import 'package:subaru_ssm2/elm.dart';
import 'package:subaru_ssm2/pids.dart';
import 'package:subaru_ssm2/protocol.dart';
import 'package:subaru_ssm2/samples.dart';

class FakeTransport implements BtTransport {
  FakeTransport({this.a8DelayMs = 1, this.a8Reply = '7E8 02 E8 6D\r>'});

  final int a8DelayMs;
  final String a8Reply;
  // v0.10: сколько ближайших A8-запросов оставить «глухими» (без ответа) —
  // имитация транзиентного таймаута клона для регрессионного теста ELM-01.
  int silentA8 = 0;
  bool _connected = false;
  final received = StreamController<Uint8List>.broadcast(sync: true);
  final states = StreamController<bool>.broadcast(sync: true);
  final writes = <String>[];

  @override
  String get name => 'Synthetic test transport';

  @override
  bool get connected => _connected;

  @override
  Stream<Uint8List> get data => received.stream;

  @override
  Stream<bool> get status => states.stream;

  @override
  Future<List<BtDevice>> paired() async => [
    const BtDevice('Test', '00:00:00:00:00:00'),
  ];

  @override
  Future<void> connect(String address) async {
    _connected = true;
    states.add(true);
  }

  @override
  Future<void> disconnect() async {
    _connected = false;
    states.add(false);
  }

  void emit(String text) => received.add(Uint8List.fromList(text.codeUnits));

  @override
  Future<void> write(String ascii) async {
    if (!_connected) throw StateError('Not connected');
    writes.add(ascii);
    final cmd = ascii.trim().toUpperCase();
    if (cmd.startsWith('A8') && silentA8 > 0) {
      silentA8--; // v0.10: «глухое окно» — драйвер уйдёт в таймаут
      return;
    }
    final reply = cmd == 'ATI' || cmd == 'ATZ' || cmd == 'ATWS'
        ? 'ELM327 TEST\r>'
        : cmd == 'ATRV'
        ? '13.5V\r>'
        : cmd.startsWith('AT')
        ? 'OK\r>'
        : cmd.startsWith('A8')
        ? a8Reply
        : 'OK\r>';
    emit(reply.substring(0, 1));
    await Future<void>.delayed(Duration(milliseconds: a8DelayMs));
    emit(reply.substring(1));
  }

  @override
  Future<void> dispose() async {
    await received.close();
    await states.close();
  }
}

void main() {
  const command = 'A8 00 00 00 08';
  group('Robust CAN parser', () {
    test('plain ATH0 / CAF1 payload parses', () {
      expect(parseAddressReply('E8 6D\r\n>', command), 109);
      expect(parseAddressReply('E86D\r\r>', command), 109);
    });

    test('headers-on and length-prefixed payload parses', () {
      expect(parseAddressReply('7E8 02 E8 6D\r\n>', command), 109);
      expect(parseAddressReply('02 E8 6D\r>', command), 109);
      expect(
        parseAddressReply('SEARCHING...\r\n7E8 02 E8 6D\r>', command),
        109,
      );
    });

    test('only single-address A8', () {
      expect(readAddressCommand(8), command);
      expect(readAddressCommand(0xFF2538), 'A8 00 FF 25 38');
    });

    test('safe diagnostics allow adapter-side CAN setup', () {
      expect(allowedDiagnostic('ATI'), isTrue);
      expect(allowedDiagnostic('ATCAF1'), isTrue);
      expect(allowedDiagnostic('ATSH 7E0'), isTrue);
      expect(allowedDiagnostic(command), isTrue);
      expect(allowedDiagnostic('ATMA'), isFalse);
    });
  });

  group('ElmDriver CAN sync', () {
    test(
      'initialize performs ECU probe and keeps conservative timeout',
      () async {
        final transport = FakeTransport(
          a8DelayMs: 420,
          a8Reply: '7E8 02 E8 6D\r>',
        );
        final elm = ElmDriver(transport);
        await elm.initialize('00:00:00:00:00:00');
        expect(elm.ready, isTrue);
        expect(elm.activeProfile, isNotEmpty);
        expect(elm.timeoutMs, greaterThanOrEqualTo(900));
        final read = await elm.readRange(0x000008, 1);
        expect(read.bytes, <int>[0x6D]);
        await elm.dispose();
      },
    );
  });

  group('ELM-01 retry (v0.10)', () {
    test(
      'одиночный таймаут не роняет sync, чтение доезжает со второго раза',
      () async {
        final transport = FakeTransport();
        final elm = ElmDriver(transport);
        await elm.initialize('00:11:22:33:44:55');
        expect(elm.ready, isTrue);
        transport.silentA8 =
            1; // в 0.9 этот таймаут убивал sync и весь дальнейший опрос
        final read = await elm.readRange(0x000008, 1);
        expect(read.bytes, <int>[0x6D]);
        expect(elm.ready, isTrue);
        await elm.dispose();
      },
    );
  });

  group('analyzeLog 0.9', () {
    PidSample fake(String id, double v, int msAgo) => PidSample(
      SubaruPidLibrary.byId(id),
      DateTime.now().subtract(Duration(milliseconds: msAgo)),
      v,
      const <int>[0],
      5,
      '',
    );
    List<PidSample> ramp(String id, double v) => [
      for (var i = 0; i < 10; i++) fake(id, v, 1000 - i * 100),
    ];

    test('детон-кластер под нагрузкой -> критичная находка', () {
      final result = analyzeLog(<String, List<PidSample>>{
        'FBKC': ramp('FBKC', -3.0),
        'RPM': ramp('RPM', 4200),
        'LOAD_4B': ramp(
          'LOAD_4B',
          2.4,
        ), // v0.10: нагрузка в г/об — штатные единицы
      });
      expect(
        result.findings.any((f) => f.level == FindingLevel.critical),
        isTrue,
      );
      expect(result.sampleCount, greaterThan(0));
    });

    test('чистый лог -> без критичных находок', () {
      final result = analyzeLog(<String, List<PidSample>>{
        'FBKC': ramp('FBKC', 0),
        'IAM': ramp('IAM', 1.0), // v0.10: штатная float-шкала множителя 0..1
        'RPM': ramp('RPM', 2500),
        'ECT': ramp('ECT', 88),
      });
      expect(
        result.findings.any((f) => f.level == FindingLevel.critical),
        isFalse,
      );
    });

    test('AFR беднее цели -> находка про смесь', () {
      final result = analyzeLog(<String, List<PidSample>>{
        'AFR': ramp('AFR', 12.9),
        'CL_TARGET': ramp('CL_TARGET', 11.5),
      });
      expect(result.findings.any((f) => f.title.contains('Смесь')), isTrue);
    });

    test('выбросы по z-score -> информационная находка', () {
      final flat = [
        for (var i = 0; i < 40; i++) fake('RPM', 800, 5000 - i * 100),
      ];
      for (var k = 0; k < 3; k++) {
        flat.add(fake('RPM', 9000, 100 - k * 10));
      }
      final result = analyzeLog(<String, List<PidSample>>{'RPM': flat});
      expect(result.findings.any((f) => f.title.contains('Выбросы')), isTrue);
    });

    test('гуляющий холостой ход -> warning про пропуски', () {
      final t0 = DateTime.now();
      PidSample at(String id, double v, int msAgo) => PidSample(
        SubaruPidLibrary.byId(id),
        t0.subtract(Duration(milliseconds: msAgo)),
        v,
        const <int>[0],
        5,
        '',
      );
      final agos = [for (var i = 0; i < 12; i++) 1100 - i * 90];
      final result = analyzeLog(<String, List<PidSample>>{
        'RPM': [
          for (var i = 0; i < 12; i++)
            at('RPM', i.isEven ? 740.0 : 960.0, agos[i]),
        ],
        'TPS': [for (var i = 0; i < 12; i++) at('TPS', 2, agos[i])],
        'ECT': [for (var i = 0; i < 12; i++) at('ECT', 85, agos[i])],
      });
      final idle = result.findings.where((f) => f.title.contains('холост'));
      expect(idle.isNotEmpty, isTrue);
      expect(idle.first.level, FindingLevel.warning);
    });

    test('v0.10 IAM шкала 0..1: 0.98 -> без критичного, 0.5 -> критичный', () {
      final ok = analyzeLog(<String, List<PidSample>>{
        'IAM': ramp('IAM', 0.98),
      });
      expect(
        ok.findings.any(
          (f) => f.level == FindingLevel.critical && f.title.contains('IAM'),
        ),
        isFalse,
      );
      final bad = analyzeLog(<String, List<PidSample>>{
        'IAM': ramp('IAM', 0.5),
      });
      expect(
        bad.findings.any(
          (f) => f.level == FindingLevel.critical && f.title.contains('IAM'),
        ),
        isTrue,
      );
    });

    test('v0.10 процентная нагрузка 45% — недоказанная нагрузка (не г/об)', () {
      final result = analyzeLog(<String, List<PidSample>>{
        'FBKC': [fake('FBKC', -1.6, 1000), fake('FBKC', -1.6, 900)],
        'RPM': ramp('RPM', 4200),
        'LOAD': ramp('LOAD', 45),
      });
      // В 0.9 «hardLoad» был истинен при любом % (45 >= 1.5) — ложный критикал.
      expect(
        result.findings.any(
          (f) =>
              f.level == FindingLevel.critical && f.title.contains('Детонация'),
        ),
        isFalse,
      );
    });

    test('compareLogs: меньше детонации -> вердикт лучше', () {
      final base = <String, List<PidSample>>{'FBKC': ramp('FBKC', -2.0)};
      final after = <String, List<PidSample>>{'FBKC': ramp('FBKC', 0)};
      final cmp = compareLogs(base, after);
      expect(cmp.verdict, contains('лучше'));
      expect(cmp.knockAfter, lessThan(cmp.knockBase));
    });
  });

  group('All 28 PID', () {
    test('identity, addresses and sizes', () {
      final all = SubaruPidLibrary.all;
      expect(all.length, 28);
      expect(SubaruPidLibrary.byId('RPM').addresses, [0xE, 0xF]);
      expect(
        SubaruPidLibrary.byId('O2_F').decode([0, 200]),
        closeTo(1.0, 0.000001),
      );
    });
  });
}
'''

FILES["lib/maps_lab/maps_lab_log.dart"] = r'''
/// Map Lab: лог и правила.
library;

import 'dart:convert';
import 'dart:math' as math;

import 'maps_lab_core.dart';

class LogData {
  final cols = <String, List<double?>>{};
  // v0.10: true, если колонка нагрузки — проценты, а не г/об (детект по значениям).
  bool loadIsPercent = false;
  int get rows => cols.isEmpty ? 0 : cols.values.first.length;

  bool has(String k) => cols.containsKey(k);
  List<double?>? operator [](String k) => cols[k];

  static const aliases = <String, List<String>>{
    'time': ['time', 'timestamp', 'time s', 'elapsed'],
    'rpm': ['engine speed', 'rpm', 'engine speed rpm'],
    'load': [
      'engine load (4-byte)',
      'engine load g/rev',
      'load_4b',
      'engine load 4-byte',
      'calculated load',
      'engine load',
      'load',
      'engine load (relative)',
    ],
    'fbkc': ['feedback knock correction', 'fbkc'],
    'flkc': [
      'fine learning knock correction',
      'fkl',
      'fine learning knock advance',
      'flkc',
    ],
    'iam': ['iam', 'ignition advance multiplier'],
    'timing': ['total ignition timing', 'ignition timing', 'timing'],
    'afr': [
      'afr',
      'a/f sensor #1',
      'a/f sensor 1',
      'air/fuel ratio',
      'estimated afr',
      'lambda',
    ],
    'boost': ['manifold relative pressure', 'boost', 'boost_rel'],
    'tgt': ['target boost', 'boost_tgt'],
    'berr': ['boost error', 'boost_err'],
    'wgdc': [
      'primary wastegate duty',
      'primary wastegate duty cycle', // v0.10: фактическое имя колонки exportRrCsv
      'wastegate duty',
      'wgdc',
      'boost control solenoid duty',
    ],
    'tq': [
      'requested torque',
      'cl_target',
      'closed loop fueling target', // v0.10: фактическое имя колонки exportRrCsv
      'demand torque',
    ],
    'thr': [
      'throttle opening angle',
      'throttle plate',
      'throttle',
      'throttle position',
    ],
    'iat': ['intake air temperature', 'iat'],
    'ect': ['coolant temperature', 'ect', 'engine coolant temperature'],
    'loop': ['cl/ol', 'fueling status', 'closed loop', 'loop'],
  };

  static String _canon(String s) {
    // v0.10 FIX (MAPLAB-01): раньше строку обрубали по ПЕРВОЙ скобке, и
    // «Engine Load (4-Byte)» схлопывался с «Engine Load (Relative)» — колонку
    // нагрузки в г/об было невозможно отличить от процентной. Снимаем только
    // ОДНУ финальную группу (это единицы измерения), маркеры типа (4-Byte) живут.
    var c = s.toLowerCase().trim().replaceAll('*', '');
    if (c.endsWith(')')) {
      final open = c.lastIndexOf('(');
      if (open > 0) c = c.substring(0, open);
    }
    if (c.endsWith(']')) {
      final open = c.lastIndexOf('[');
      if (open > 0) c = c.substring(0, open);
    }
    return c.replaceAll(RegExp(r'\s+'), ' ').trim();
  }

  static LogData parse(String text) {
    final head = text.substring(0, math.min(text.length, 4096));
    final sep = ';'.allMatches(head).length > ','.allMatches(head).length
        ? ';'
        : ',';
    final decComma =
        sep == ';' && ','.allMatches(head).length > '.'.allMatches(head).length;

    final lines = const LineSplitter()
        .convert(text)
        .where((l) => l.trim().isNotEmpty)
        .toList();
    if (lines.isEmpty) return LogData();
    final header = lines.first
        .split(sep)
        .map((h) => h.trim().replaceAll('"', ''))
        .toList();

    final raw = List<List<double?>>.generate(header.length, (_) => []);
    final canonHead = header.map(_canon).toList();

    for (var li = 1; li < lines.length; li++) {
      final parts = lines[li].split(sep);
      for (var i = 0; i < header.length; i++) {
        if (i >= parts.length) {
          raw[i].add(null);
          continue;
        }
        var tok = parts[i].trim().replaceAll('"', '');
        if (decComma) tok = tok.replaceAll(',', '.');
        raw[i].add(double.tryParse(tok));
      }
    }

    final log = LogData();
    final usedNames = <String>{};
    for (final entry in aliases.entries) {
      final canon = entry.key;
      // v0.10 FIX2: варианты НЕ гоняем через _canon — он стирает значимый
      // маркер '(4-byte)' и из вариантов тоже (первая версия фикса это упустила,
      // поймано flutter test в конвейере ячейки 05).
      final variants = entry.value.toSet();
      // v0.10: приоритет — порядок вариантов, а не порядок колонок в файле.
      // Иначе %-колонка «Engine Load (Relative)» перехватывала канал у (4-Byte).
      variantLoop:
      for (final variant in variants) {
        for (var i = 0; i < header.length; i++) {
          if (usedNames.contains(header[i])) continue;
          if (canonHead[i] == variant) {
            log.cols[canon] = raw[i];
            usedNames.add(header[i]);
            break variantLoop;
          }
        }
      }
    }

    // v0.10: шкала нагрузки (медиана > 8 — это проценты, г/об так не бывает).
    final loadCol = log.cols['load'];
    if (loadCol != null) {
      final lv = loadCol.whereType<double>().toList()..sort();
      if (lv.isNotEmpty && lv[lv.length ~/ 2] > 8) log.loadIsPercent = true;
    }

    final afr = log.cols['afr'];
    if (afr != null) {
      final vals = afr.whereType<double>().toList()..sort();
      if (vals.isNotEmpty && vals[vals.length ~/ 2] <= 2.2) {
        log.cols['afr'] = [for (final v in afr) v == null ? null : v * 14.7];
      }
    }
    final boost = log.cols['boost'], tgt = log.cols['tgt'];
    if (boost != null && tgt != null && log.cols['berr'] == null) {
      log.cols['berr'] = [
        for (var i = 0; i < boost.length; i++)
          (boost[i] != null && tgt[i] != null) ? boost[i]! - tgt[i]! : null,
      ];
    }
    return log;
  }
}

class LogHealth {
  final notes = <String>[];
  final missing = <String, List<String>>{};
  bool blockReady(String b) => (missing[b] ?? const []).isEmpty;
}

class LogAudit {
  static const _needFor = {
    'timing': ['rpm', 'load', 'fbkc', 'flkc', 'iat'],
    'fuel': ['rpm', 'load', 'afr', 'boost'],
    'wgdc': ['rpm', 'tq', 'berr'],
  };

  static const _hint = {
    'wgdc': 'Primary Wastegate Duty',
    'tq': 'Requested Torque',
    'thr': 'Throttle Opening Angle',
    'afr': 'A/F Sensor #1',
    'loop': 'CL/OL Fueling Status',
    // v0.10.1: детон-каналы только в расширенном наборе — скажем об этом прямо.
    'fbkc':
        'Feedback Knock Correction (4-byte)* — нужен расширенный набор (CALID)',
    'flkc': 'Fine Learning Knock Correction* — нужен расширенный набор (CALID)',
    'load': 'Engine Load (4-Byte)* — г/об вместо процентной нагрузки',
    'timing': 'Total Ignition Timing',
  };

  static LogHealth check(LogData log) {
    final h = LogHealth();
    h.notes.add('строк: ${log.rows}');
    if (log.loadIsPercent) {
      h.notes.add(
        'нагрузка в % — приведена к оси г/об приближением 100% ≈ max; '
        'для точного биннинга логируйте Engine Load (4-Byte)',
      );
    }

    double? maxOf(String k) {
      final c = log[k];
      if (c == null) return null;
      double? m;
      for (final v in c) {
        if (v != null && (m == null || v > m)) m = v;
      }
      return m;
    }

    double? minOf(String k) {
      final c = log[k];
      if (c == null) return null;
      double? m;
      for (final v in c) {
        if (v != null && (m == null || v < m)) m = v;
      }
      return m;
    }

    final mr = maxOf('rpm');
    if (mr != null) h.notes.add('RPM max: ${mr.toStringAsFixed(0)}');
    final iamMin = minOf('iam');
    if (iamMin != null) {
      h.notes.add(
        'IAM min: ${iamMin.toStringAsFixed(2)}'
        '${iamMin < 0.99 ? ' — ЛОГ НЕ ГОДИТСЯ: сначала доучить ЭБУ' : ' — ок'}',
      );
    }
    final mb = maxOf('boost');
    if (mb != null) h.notes.add('буст max: ${mb.toStringAsFixed(2)} бар');

    final t = log['time'];
    if (t != null) {
      final dts = <double>[];
      for (var i = 1; i < t.length; i++) {
        if (t[i] != null && t[i - 1] != null) dts.add(t[i]! - t[i - 1]!);
      }
      dts.sort();
      if (dts.isNotEmpty) {
        final med = dts[dts.length ~/ 2];
        final gaps = dts.where((d) => d > 0.3).length;
        h.notes.add(
          'частота: медиана ${(1 / med).toStringAsFixed(1)} Гц · провалов >300 мс: $gaps',
        );
        if (med > 0.25)
          h.notes.add('!! реже 4 Гц — сократите набор PID до 10–12');
      }
    }

    _needFor.forEach((block, req) {
      final miss = req.where((r) => !log.has(r)).toList();
      h.missing[block] = miss;
    });
    final allMiss = {for (final v in h.missing.values) ...v};
    for (final m in allMiss) {
      final hint = _hint[m];
      if (hint != null) h.notes.add('добавить в логгер PID: $hint');
    }
    return h;
  }
}

class CellNote {
  int n = 0;
  double? fbkc, flkc, iat, iam, afr, target, berr;
  String why = '';
}

class MapResult {
  MapResult(int rows, int cols)
    : delta = List.generate(rows, (_) => List.filled(cols, 0.0)),
      info = List.generate(rows, (_) => List<CellNote?>.filled(cols, null));
  final List<List<double>> delta;
  final List<List<CellNote?>> info;

  int get rows => delta.length;
  int get cols => delta.isNotEmpty ? delta[0].length : 0;

  int get dec {
    var c = 0;
    for (final r in delta) {
      for (final v in r) {
        if (v < 0) c++;
      }
    }
    return c;
  }

  int get inc {
    var c = 0;
    for (final r in delta) {
      for (final v in r) {
        if (v > 0) c++;
      }
    }
    return c;
  }
}

class AnalyzerConfig {
  const AnalyzerConfig({
    this.minN = 6,
    this.fbkcEvent = -1.5,
    this.flkcEvent = -1.5,
    this.timingStep = 0.5,
    this.timingMaxCut = -3.0,
    this.allowTimingAdd = false,
    this.afrErr = 0.40,
    this.afrMaxCut = -0.8,
    this.boostErr = 0.04,
    this.wgdcMaxDelta = 6.0,
    this.wotLoad = 2.2,
    this.targetBoostGain = 0.0,
  });
  final int minN;
  final double fbkcEvent, flkcEvent, timingStep, timingMaxCut;
  final bool allowTimingAdd;
  final double afrErr,
      afrMaxCut,
      boostErr,
      wgdcMaxDelta,
      wotLoad,
      targetBoostGain;
}

class Analyzer {
  Analyzer(this.cfg);
  final AnalyzerConfig cfg;

  double _roundStep(double v, double s) => (v / s).roundToDouble() * s;

  int _binIdx(List<double> axis, double v) {
    if (axis.length <= 1) return 0;
    var lo = 0, hi = axis.length - 1;
    while (lo < hi - 1) {
      final mid = (lo + hi) >> 1;
      if (v >= axis[mid]) {
        lo = mid;
      } else {
        hi = mid;
      }
    }
    final mid = (axis[lo] + axis[hi]) / 2;
    return v < mid ? lo : hi;
  }

  MapResult? analyzeMap(MapGrid g, LogData log) {
    final kind = g.kind;
    if (!{'timing', 'knockadv', 'fuel', 'wgdc', 'boost'}.contains(kind))
      return null;

    final xIsRpm =
        g.x.guess == 'rpm' || (g.y.guess != 'rpm' && g.x.values.last > 800);
    final rpmAxis = xIsRpm ? g.x.values : g.y.values;
    final otherAxis = xIsRpm ? g.y.values : g.x.values;

    final need = switch (kind) {
      'timing' || 'knockadv' => ['rpm', 'load', 'fbkc', 'flkc'],
      'fuel' => ['rpm', 'load', 'afr', 'fbkc', 'flkc'],
      _ => ['rpm', 'tq', 'berr'],
    };
    if (need.any((k) => !log.has(k))) return null;

    final rpmCol = log['rpm']!;
    final otherCol = switch (kind) {
      'timing' || 'knockadv' || 'fuel' => log['load']!,
      _ => log['tq']!,
    };

    final rows = g.rows, cols = g.cols;
    final cnt = List<int>.filled(rows * cols, 0);
    final fbkcMin = List<double>.filled(rows * cols, 0);
    final flkcSum = List<double>.filled(rows * cols, 0);
    final flkcLateSum = List<double>.filled(rows * cols, 0);
    final flkcLateCnt = List<int>.filled(rows * cols, 0);
    // v0.10 FIX (ANLZ-03): средние — по количеству НЕПУСТЫХ значений,
    // раньше деление на общий n занижало flkc/afr по ячейке.
    final flkcCnt = List<int>.filled(rows * cols, 0);
    final afrCnt = List<int>.filled(rows * cols, 0);
    final halfRows = (log.rows / 2).round();
    final iatMax = List<double>.filled(rows * cols, -999);
    final iamMin = List<double>.filled(rows * cols, 999);
    final afrSum = List<double>.filled(rows * cols, 0);
    final berrSum = List<double>.filled(rows * cols, 0);

    final fbkc = log['fbkc'], flkc = log['flkc'], iat = log['iat'];
    final iam = log['iam'], afr = log['afr'], berr = log['berr'];

    // v0.10 FIX (MAPLAB-01): процентную нагрузку приводим к диапазону оси
    // карты (100% ≈ максимум оси), иначе значения 0..100 прибивались к краю.
    final otherIsLoad =
        kind == 'timing' || kind == 'knockadv' || kind == 'fuel';
    final otherAxisMax = otherAxis.isEmpty ? 0.0 : otherAxis.reduce(math.max);
    final otherScale = otherIsLoad && log.loadIsPercent && otherAxisMax > 0
        ? otherAxisMax / 100.0
        : 1.0;
    for (var i = 0; i < log.rows; i++) {
      final rv = rpmCol[i], ovRaw = otherCol[i];
      if (rv == null || ovRaw == null) continue;
      final ov = ovRaw * otherScale;
      final ri = _binIdx(xIsRpm ? otherAxis : rpmAxis, xIsRpm ? ov : rv);
      final ci = _binIdx(xIsRpm ? rpmAxis : otherAxis, xIsRpm ? rv : ov);
      final idx = ri * cols + ci;
      cnt[idx]++;
      final f = fbkc?[i];
      if (f != null && (cnt[idx] == 1 || f < fbkcMin[idx])) fbkcMin[idx] = f;
      final fl = flkc?[i];
      if (fl != null) {
        flkcSum[idx] += fl;
        flkcCnt[idx]++;
        if (i >= halfRows) {
          flkcLateSum[idx] += fl;
          flkcLateCnt[idx]++;
        }
      }
      final it = iat?[i];
      if (it != null && it > iatMax[idx]) iatMax[idx] = it;
      final im = iam?[i];
      if (im != null && im < iamMin[idx]) iamMin[idx] = im;
      final a = afr?[i];
      if (a != null) {
        afrSum[idx] += a;
        afrCnt[idx]++;
      }
      final be = berr?[i];
      if (be != null) berrSum[idx] += be;
    }

    final res = MapResult(rows, cols);

    void fanTiming(int ri, int ci, double d, AnalyzerConfig cfg) {
      const fan = <List<num>>[
        [-1, 0, 0.5],
        [1, 0, 0.5],
        [0, -1, 0.5],
        [0, 1, 0.5],
        [-1, -1, 0.25],
        [-1, 1, 0.25],
        [1, -1, 0.25],
        [1, 1, 0.25],
      ];
      for (final e in fan) {
        final rr = ri + e[0].toInt(), cc = ci + e[1].toInt();
        if (rr < 0 || rr >= rows || cc < 0 || cc >= cols) continue;
        if (res.delta[rr][cc] != 0) continue;
        final d2 = _roundStep(d * e[2].toDouble(), cfg.timingStep);
        if (d2 == 0) continue;
        res.delta[rr][cc] = d2;
        res.info[rr][cc] = CellNote()
          ..why = 'веер от соседнего кластера: ${d2.toStringAsFixed(1)}°';
      }
    }

    for (var ri = 0; ri < rows; ri++) {
      for (var ci = 0; ci < cols; ci++) {
        final idx = ri * cols + ci;
        final n = cnt[idx];
        if (n < cfg.minN) continue;
        final note = CellNote()
          ..n = n
          ..fbkc = fbkcMin[idx]
          ..flkc = flkcCnt[idx] > 0
              ? flkcSum[idx] / flkcCnt[idx]
              : 0.0 // v0.10
          ..iat = iatMax[idx] < -900 ? null : iatMax[idx]
          ..iam = iamMin[idx] > 900 ? null : iamMin[idx]
          ..afr = afrCnt[idx] > 0
              ? afrSum[idx] / afrCnt[idx]
              : null // v0.10
          ..berr = berrSum[idx] != 0 ? berrSum[idx] / n : null;

        if (kind == 'timing' || kind == 'knockadv') {
          if (note.iam != null && note.iam! < 0.99) {
            res.info[ri][ci] = note
              ..why = 'IAM=${note.iam!.toStringAsFixed(2)} — сначала доучить';
            continue;
          }
          final lateCount = flkcLateCnt[idx];
          final lateFlkc = lateCount > 0
              ? flkcLateSum[idx] / lateCount
              : note.flkc!;
          final knock = math.min(
            note.fbkc!,
            math.min(note.flkc!, lateFlkc) * 1.4,
          );
          if (knock <= cfg.fbkcEvent) {
            final d = math.max(
              cfg.timingMaxCut,
              math.min(
                -cfg.timingStep,
                _roundStep(knock * 0.6, cfg.timingStep),
              ),
            );
            res.delta[ri][ci] = d;
            fanTiming(ri, ci, d, cfg);
            res.info[ri][ci] = note
              ..why =
                  'детон-кластер · FBKC ${note.fbkc!.toStringAsFixed(1)}°, '
                  'FLKC ${note.flkc!.toStringAsFixed(1)}° (поздняя ${lateFlkc.toStringAsFixed(1)}°) · '
                  'снять ${d.abs().toStringAsFixed(1)}° здесь, соседи сглажены веером';
          } else if (res.delta[ri][ci] == 0 &&
              cfg.allowTimingAdd &&
              n >= 18 &&
              (note.iat == null || note.iat! <= 45)) {
            res.delta[ri][ci] = cfg.timingStep;
            res.info[ri][ci] = note..why = 'чисто, IAM=1.0 — опционально +0.5°';
          }
        } else if (kind == 'fuel') {
          final loadAxisValue = xIsRpm ? g.y.values[ri] : g.x.values[ci];
          if (loadAxisValue < cfg.wotLoad || note.afr == null) continue;
          var target = g.data[ri][ci];
          if (target < 9.0) target *= 14.7;
          note.target = target;
          final err = note.afr! - target;
          final knock = math.min(note.fbkc!, note.flkc!);
          if (err > cfg.afrErr) {
            final d = math.max(cfg.afrMaxCut, _roundStep(-err, 0.1));
            res.delta[ri][ci] = d;
            res.info[ri][ci] = note
              ..why =
                  'факт ${note.afr!.toStringAsFixed(2)} против цели ${target.toStringAsFixed(2)} '
                  '— обогатить на ${d.abs().toStringAsFixed(1)}; если не помогает — MAF/давление';
          } else if (knock <= cfg.fbkcEvent) {
            res.delta[ri][ci] = -0.3;
            res.info[ri][ci] = note
              ..why = 'детон при цели ~совпадает — запас −0.3 AFR';
          }
        } else {
          final e = note.berr;
          if (e == null) continue;
          if (kind == 'wgdc') {
            if (e.abs() > cfg.boostErr) {
              final d = _roundStep(-e * 45, 1)
                  .clamp(
                    -cfg.wgdcMaxDelta,
                    cfg.wgdcMaxDelta,
                  ) // v0.10: было n-1 (опечатка)
                  .toDouble();
              if (d != 0) {
                res.delta[ri][ci] = d;
                res.info[ri][ci] = note
                  ..why =
                      '${e > 0 ? 'овербуст' : 'недобор'} ${e.toStringAsFixed(2)} бар '
                      '→ WGDC ${d > 0 ? '+' : ''}${d.toStringAsFixed(0)}%';
              }
            }
          } else {
            if (e > cfg.boostErr) {
              res.info[ri][ci] = note
                ..why =
                    'овербуст ${e.toStringAsFixed(2)} бар — чинить через WGDC/TD, не таргет';
            }
            if (cfg.targetBoostGain > 0) {
              res.delta[ri][ci] = cfg.targetBoostGain;
              res.info[ri][ci] = note
                ..why =
                    'политика +${cfg.targetBoostGain} бар (после фикса WGDC и чистого детон-лога)';
            }
          }
        }
      }
    }
    return res;
  }

  Map<String, MapResult> run(Map<String, MapGrid> maps, LogData log) {
    final out = <String, MapResult>{};
    for (final e in maps.entries) {
      final r = analyzeMap(e.value, log);
      if (r != null) out[e.key] = r;
    }
    return out;
  }
}
'''

FILES["test/maps_lab_test.dart"] = r"""
import 'dart:typed_data';
import 'package:flutter_test/flutter_test.dart';
import 'package:subaru_ssm2/maps_lab/maps_lab_core.dart';
import 'package:subaru_ssm2/maps_lab/maps_lab_log.dart';

void main() {
  group('SafeExpr', () {
    test('uint8 timing: (x*.3515625)-20', () {
      final f = SafeExpr.compile('(x*.3515625)-20');
      expect(f(100), closeTo(15.15625, 0.0001));
      expect(f(0), -20.0);
    });
    test('estimated AFR: 14.7/(1+x*.0078125)', () {
      final f = SafeExpr.compile('14.7/(1+x*.0078125)');
      expect(f(80), closeTo(9.0461, 0.001));
    });
    test('float identity + garbage guard', () {
      expect(SafeExpr.compile('x')(42.5), 42.5);
      expect(SafeExpr.compile('eval(x)+1')(3), 3);
    });
  });

  const baseXml = '''
<rom>
  <romid><xmlid>32BITBASE</xmlid></romid>
  <scaling name="Timing8" units="deg" toexpr="(x*.3515625)-20" frexpr="(x+20)/.3515625" format="%.2f" storagetype="uint8" endian="big"/>
  <scaling name="RPM" units="RPM" toexpr="x" frexpr="x" format="%.0f" storagetype="float" endian="big"/>
  <table name="Base Timing" category="Ignition" type="3D" level="4" scaling="Timing8">
    <table name="Engine Load" type="X Axis" elements="2" scaling="RPM"/>
    <table name="Engine Speed" type="Y Axis" elements="3" scaling="RPM"/>
  </table>
</rom>''';

  const derivedXml = '''
<rom>
  <romid><xmlid>TEST_ROM</xmlid><internalidaddress>2000</internalidaddress><internalidstring>TEST_ROM</internalidstring></romid>
  <include>32BITBASE</include>
  <table name="Base Timing" address="1000">
    <table name="X" address="2400" elements="4"/>
    <table name="Y" address="2500"/>
  </table>
</rom>''';

  test('merge include-цепочки: адрес у производного, скейлинг наследуется', () {
    final defs = DefSet.build({
      '32BITBASE': baseXml,
      'TEST_ROM': derivedXml,
    }, 'TEST_ROM');
    expect(defs.chain, ['32BITBASE', 'TEST_ROM']);
    final t = defs.tables['Base Timing']!;
    expect(t.dataAddress, 0x1000);
    expect(t.dataScaling!.units, 'deg');
    expect(t.axes[0].name, 'Engine Load');
    expect(t.axes[0].elements, 4);
    expect(t.axes[0].address, 0x2400);
    expect(t.axes[1].elements, 3);
    expect(defs.isReadable3D(t), isTrue);
  });

  test('ROM: декод float32-осей и uint8-данных big-endian', () {
    final rom = Uint8List(0x3000);
    final bd = ByteData.view(rom.buffer);
    for (var i = 0; i < 8; i++) {
      bd.setUint8(0x2000 + i, 'TEST_ROM'.codeUnitAt(i));
    }
    final xs = [1000.0, 2000.0, 3000.0, 4000.0];
    for (var i = 0; i < 4; i++) {
      bd.setFloat32(0x2400 + i * 4, xs[i], Endian.big);
    }
    final ys = [0.8, 1.6, 2.4];
    for (var i = 0; i < 3; i++) {
      bd.setFloat32(0x2500 + i * 4, ys[i], Endian.big);
    }
    for (var i = 0; i < 12; i++) {
      bd.setUint8(0x1000 + i, 100);
    }

    final defs = DefSet.build({
      '32BITBASE': baseXml,
      'TEST_ROM': derivedXml,
    }, 'TEST_ROM');
    final parser = RomParser(rom);
    expect(parser.readRomId(0x2000), 'TEST_ROM');
    final grid = parser.extract(defs.tables['Base Timing']!)!;
    expect(grid.rows, 3);
    expect(grid.cols, 4);
    expect(grid.x.values.last, 4000);
    expect(grid.y.values.first, closeTo(0.8, 0.001));
    expect(grid.data[2][3], closeTo(15.15625, 0.001));
    expect(grid.kind, 'timing');
  });

  group('LogData + Analyzer', () {
    LogData syntheticLog({required double fbkc}) {
      final rpm = <double?>[],
          load = <double?>[],
          f = <double?>[],
          fl = <double?>[];
      for (var i = 0; i < 600; i++) {
        rpm.add(4000 + (i % 80) - 40);
        load.add(3.0);
        f.add(fbkc);
        fl.add(fbkc * 0.4);
      }
      return LogData()
        ..cols['rpm'] = rpm
        ..cols['load'] = load
        ..cols['fbkc'] = f
        ..cols['flkc'] = fl;
    }

    MapGrid smallGrid() => MapGrid(
      name: 'Base Timing Primary Non-Cruise',
      kind: 'timing',
      units: 'deg',
      addr: 0,
      x: AxisVals('Engine Speed', [3000, 4000, 5000], 'rpm'),
      y: AxisVals('Engine Load', [2.0, 3.0], 'г/об·бар'),
      data: [
        [20, 18, 16],
        [16, 14, 12],
      ],
    );

    test(
      'детон-кластер → отрицательная дельта в своей ячейке и веер сглаживания',
      () {
        final res = Analyzer(
          const AnalyzerConfig(),
        ).analyzeMap(smallGrid(), syntheticLog(fbkc: -4.0))!;
        final d = res.delta[1][1];
        expect(d, lessThan(0));
        expect(d, greaterThanOrEqualTo(-3.0));
        expect(res.delta[0][0], -0.5);
        expect(res.info[1][1]!.why, contains('детон'));
      },
    );

    test('чистый лог при выключенном allowTimingAdd → ноль правок', () {
      final res = Analyzer(
        const AnalyzerConfig(),
      ).analyzeMap(smallGrid(), syntheticLog(fbkc: 0))!;
      final anyDelta = res.delta.expand((r) => r).any((v) => v != 0);
      expect(anyDelta, isFalse);
    });

    test('CSV: автоопределение ; и десятичной запятой, алиасы колонок', () {
      const csv =
          'Time (s);Engine Speed (RPM);Feedback Knock Correction;Load_4B\r\n'
          '0,0;3000;0,0;3,00\r\n'
          '0,1;3100;-2,5;3,10\r\n';
      final log = LogData.parse(csv);
      expect(log['rpm']![1], 3100);
      expect(log['fbkc']![1], -2.5);
      expect(log['load']![1], 3.1);
    });

    test('v0.10 CSV: (4-Byte) не схлопывается с (Relative), г/об в приоритете', () {
      const csv =
          'Time,Engine Load (Relative) (%),Engine Load (4-Byte) (grams/rev)*,Engine Speed (rpm)\n'
          '0.0,64,2.90,3000\n'
          '0.2,66,3.05,3100\n';
      final log = LogData.parse(csv);
      // В 0.9 обе колонки канонизировались в 'engine load' и побеждала первая — %.
      expect(log['load']![0], 2.9);
      expect(log['load']![1], 3.05);
      expect(log.loadIsPercent, isFalse);
    });

    test('v0.10 Analyzer: процентная нагрузка приводится к оси карты', () {
      final log = syntheticLog(fbkc: -4.0);
      log.cols['load'] = [
        for (var i = 0; i < 600; i++) 100.0,
      ]; // 100% ≈ max оси (3.0)
      log.loadIsPercent = true;
      final res = Analyzer(
        const AnalyzerConfig(),
      ).analyzeMap(smallGrid(), log)!;
      // Строки прибиваются к верхней строке оси (3.0), и детон-кластер находится там же.
      expect(res.info[1][1]!.why, contains('детон'));
    });
  });
}
"""

FILES["lib/maps_lab/maps_lab_page.dart"] = r'''
/// Экран Map Lab: крупная карта + прокрутка всей страницы (журнал виден).
library;

import 'dart:convert';
import 'dart:io';

import 'package:flutter/material.dart';
import 'package:flutter/services.dart' show rootBundle;
import 'package:path_provider/path_provider.dart';
import 'package:permission_handler/permission_handler.dart';
import 'package:share_plus/share_plus.dart';

import 'map3d_view.dart';
import 'maps_lab_core.dart';
import 'maps_lab_log.dart';

/// Тепловая карта: зелёный → жёлтый → оранжевый → красный.
Color getHeatmapColor(double v, double min, double max) {
  if (max <= min) return const Color(0xFF4CAF50);
  final t = ((v - min) / (max - min)).clamp(0.0, 1.0);
  if (t < 0.33) {
    return Color.lerp(
      const Color(0xFF4CAF50),
      const Color(0xFFFFEB3B),
      t / 0.33,
    )!;
  } else if (t < 0.66) {
    return Color.lerp(
      const Color(0xFFFFEB3B),
      const Color(0xFFFF9800),
      (t - 0.33) / 0.33,
    )!;
  } else {
    return Color.lerp(
      const Color(0xFFFF9800),
      const Color(0xFFF44336),
      (t - 0.66) / 0.34,
    )!;
  }
}

class MapLabPage extends StatefulWidget {
  const MapLabPage({super.key});

  @override
  State<MapLabPage> createState() => _MapLabPageState();
}

class _MapLabPageState extends State<MapLabPage> {
  static const _defKeys = ['A2TB100B', 'A2TB100K', '32BITBASE'];

  /// Высота окна карты (крупнее, чем «сжатый» Expanded).
  static const double _mapHeight = 480;

  final _journal = <String>[];
  DefSet? _defs;
  Map<String, MapGrid>? _maps;
  LogData? _log;
  Map<String, MapResult>? _results;
  String? _mapName;
  int? _selRi, _selCi;
  bool _view3D = false;
  String _defsSource = '—';

  @override
  void initState() {
    super.initState();
    WidgetsBinding.instance.addPostFrameCallback((_) => _bootstrapDefs());
  }

  void _say(String s) {
    if (!mounted) return;
    setState(() => _journal.add('${TimeOfDay.now().format(context)}  $s'));
  }

  Future<Directory> _cacheDir() async {
    final docs = await getApplicationDocumentsDirectory();
    final dir = Directory('${docs.path}/defs');
    if (!dir.existsSync()) dir.createSync(recursive: true);
    return dir;
  }

  Future<Map<String, String>> _tryAssets() async {
    final out = <String, String>{};
    for (final k in _defKeys) {
      try {
        final s = await rootBundle.loadString('assets/defs/$k.xml');
        if (s.trim().length > 200) out[k] = s;
      } catch (_) {}
    }
    return out;
  }

  Future<Map<String, String>> _tryCache() async {
    final out = <String, String>{};
    final dir = await _cacheDir();
    for (final k in _defKeys) {
      final f = File('${dir.path}/$k.xml');
      if (f.existsSync() && f.lengthSync() > 200) {
        try {
          out[k] = await f.readAsString();
        } catch (_) {}
      }
    }
    return out;
  }

  Future<void> _saveCache(Map<String, String> xmlById) async {
    final dir = await _cacheDir();
    for (final e in xmlById.entries) {
      try {
        await File(
          '${dir.path}/${e.key}.xml',
        ).writeAsString(e.value, flush: true);
      } catch (_) {}
    }
  }

  bool _applyDefs(Map<String, String> xmlById, String source) {
    if (xmlById.length < 3) return false;
    try {
      _defs = DefSet.build(xmlById, 'A2TB100B');
      _defsSource = source;
      _say(
        'дефиниции [$source]: ${_defs!.chain.join(' → ')} · '
        'ecuid ${_defs!.meta.ecuid} · ${_defs!.tables.length} таблиц',
      );
      return true;
    } catch (e) {
      _say('!! ошибка разбора дефиниций: $e');
      return false;
    }
  }

  Future<void> _bootstrapDefs() async {
    var xml = await _tryAssets();
    if (_applyDefs(xml, 'APK assets')) {
      await _saveCache(xml);
      setState(() {});
      return;
    }
    xml = await _tryCache();
    if (_applyDefs(xml, 'кэш телефона')) {
      setState(() {});
      return;
    }
    _say('!! дефиниций нет. Нажмите «0 · Дефиниции» и укажите папку с XML.');
    setState(() {});
  }

  Future<Map<String, String>> _scanDefsInDir(Directory root) async {
    final found = <String, File>{};
    bool matchName(String name, String key) {
      final n = name.toLowerCase();
      return n == '${key.toLowerCase()}.xml' || n.contains(key.toLowerCase());
    }

    void consider(File f) {
      final name = f.uri.pathSegments.last;
      for (final k in _defKeys) {
        if (found.containsKey(k)) continue;
        if (matchName(name, k) && f.lengthSync() > 200) found[k] = f;
      }
    }

    try {
      for (final e in root.listSync(followLinks: false)) {
        if (e is File && e.path.toLowerCase().endsWith('.xml')) consider(e);
      }
      for (final e in root.listSync(followLinks: false)) {
        if (e is! Directory) continue;
        try {
          for (final f in e.listSync(followLinks: false)) {
            if (f is File && f.path.toLowerCase().endsWith('.xml')) consider(f);
          }
        } catch (_) {}
      }
    } catch (_) {}

    final out = <String, String>{};
    for (final e in found.entries) {
      try {
        out[e.key] = await e.value.readAsString();
      } catch (_) {}
    }
    return out;
  }

  Future<void> _pickDefs() async {
    final result = await showModalBottomSheet<Object>(
      context: context,
      isScrollControlled: true,
      backgroundColor: Colors.transparent,
      builder: (ctx) => const _FilePickerSheet(
        extensions: ['.xml'],
        title: 'Дефиниции RomRaider (.xml)',
        allowPickFolderDefs: true,
      ),
    );
    if (result == null) return;

    var xml = <String, String>{};
    var source = 'локально';

    if (result is Directory) {
      xml = await _scanDefsInDir(result);
      source = 'папка ${result.path.split('/').last}';
      _say('сканирование ${result.path}: найдено ${xml.length}/3');
    } else if (result is File) {
      xml = await _scanDefsInDir(result.parent);
      if (xml.length < 3) {
        final name = result.uri.pathSegments.last.toUpperCase();
        for (final k in _defKeys) {
          if (name.contains(k)) {
            try {
              xml[k] = await result.readAsString();
            } catch (_) {}
          }
        }
      }
      source = 'файл+папка';
    } else if (result is Map) {
      xml = Map<String, String>.from(result);
    }

    if (xml.length < 3) {
      final miss = _defKeys.where((k) => !xml.containsKey(k)).join(', ');
      _say('!! не хватает XML: $miss');
      return;
    }

    if (_applyDefs(xml, source)) {
      await _saveCache(xml);
      setState(() {});
    }
  }

  Future<File?> _chooseFile(List<String> exts, String title) async {
    final r = await showModalBottomSheet<Object>(
      context: context,
      isScrollControlled: true,
      backgroundColor: Colors.transparent,
      builder: (ctx) => _FilePickerSheet(extensions: exts, title: title),
    );
    return r is File ? r : null;
  }

  Future<void> _pickRom() async {
    if (_defs == null) {
      _say('сначала загрузите дефиниции');
      return;
    }
    final f = await _chooseFile([
      '.bin',
      '.hex',
      '.rom',
    ], 'Выберите прошивку (.bin)');
    if (f == null) return;
    try {
      final bytes = await f.readAsBytes();
      setState(() {
        _results = null;
        _selRi = _selCi = null;
      });
      final parser = RomParser(bytes);
      final idAddr =
          int.tryParse(_defs?.meta.internalIdAddress ?? '2000', radix: 16) ??
          0x2000;
      final romId = parser.readRomId(idAddr);
      final expected = _defs?.meta.internalIdString ?? '';
      final matched = expected.isEmpty || romId == expected;
      _say(
        'ROM: ${f.uri.pathSegments.last} · ${(bytes.length / 1024).toStringAsFixed(0)} КБ · '
        'ID="$romId"${matched ? '' : '  (ожидался $expected)'}',
      );
      _maps = parser.extractKeys(_defs!);
      _mapName = _maps!.keys.firstOrNull;
      _say('карт извлечено: ${_maps!.length} из ${RomParser.keyTables.length}');
      setState(() {});
    } catch (e) {
      _say('!! ошибка парсинга ROM: $e');
    }
  }

  // v0.10.1 SYNC: лог лежит в приватном каталоге приложения — разрешения не нужны.
  // Открываем записанный CSV напрямую: вкладка «CSV» → Map Lab в один тап.
  Future<File?> _latestAppLog() async {
    try {
      final dir = await getApplicationDocumentsDirectory();
      final files =
          dir
              .listSync()
              .whereType<File>()
              .where(
                (f) =>
                    f.path.toLowerCase().endsWith('.csv') &&
                    f.uri.pathSegments.last.startsWith('ssm2_'),
              )
              .toList()
            ..sort((a, b) => b.path.compareTo(a.path));
      return files.isEmpty ? null : files.first;
    } catch (_) {
      return null;
    }
  }

  Future<void> _readLogIntoLab(File f) async {
    final text = utf8.decode(await f.readAsBytes(), allowMalformed: true);
    _log = LogData.parse(text);
    final health = LogAudit.check(_log!);
    _say('лог: ${f.uri.pathSegments.last} · ${_log!.rows} строк');
    for (final n in health.notes) {
      _say('   $n');
    }
    setState(() {});
  }

  Future<void> _loadLatestLog() async {
    try {
      final f = await _latestAppLog();
      if (f == null) {
        _say(
          'нет логов приложения: сначала запишите CSV на вкладке «CSV» (кнопка «Записать»)',
        );
        return;
      }
      await _readLogIntoLab(f);
    } catch (e) {
      _say('!! ошибка чтения лога: $e');
    }
  }

  Future<void> _pickLog() async {
    final f = await _chooseFile([
      '.csv',
      '.txt',
      '.log',
    ], 'Выберите файл лога (.csv)');
    if (f == null) return;
    try {
      await _readLogIntoLab(f);
    } catch (e) {
      _say('!! ошибка чтения лога: $e');
    }
  }

  void _runAnalysis() {
    if (_maps == null || _log == null) return;
    const cfg = AnalyzerConfig();
    _results = Analyzer(cfg).run(_maps!, _log!);
    var dec = 0, inc = 0;
    for (final r in _results!.values) {
      dec += r.dec;
      inc += r.inc;
    }
    _say(
      'анализ: вердикты по ${_results!.length} картам · убавить $dec, прибавить $inc ячеек',
    );
    setState(() {});
  }

  Future<void> _export() async {
    if (_maps == null || _results == null) return;
    final dir = await getApplicationDocumentsDirectory();
    final stamp = DateTime.now()
        .toIso8601String()
        .replaceAll(':', '-')
        .split('.')
        .first;
    final outDir = Directory('${dir.path}/maplab_$stamp')
      ..createSync(recursive: true);
    final files = <XFile>[];
    final md = StringBuffer('# Map Lab A2TB100B · рекомендации\n');
    for (final e in _maps!.entries) {
      final g = e.value;
      final res = _results![e.key];
      if (res == null) continue;
      final head =
          '\t${g.x.values.map((v) => v.toStringAsFixed(2)).join('\t')}';
      final rows = <String>[
        for (var ri = 0; ri < g.rows; ri++)
          '${g.y.values[ri].toStringAsFixed(2)}\t${[for (var ci = 0; ci < g.cols; ci++) (g.data[ri][ci] + res.delta[ri][ci]).toStringAsFixed(3)].join('\t')}',
      ];
      final tsv = '$head\n${rows.join('\n')}\n';
      final file = File(
        '${outDir.path}/${e.key.replaceAll(RegExp(r'[/ ]'), '_')}_recommended.tsv',
      )..writeAsStringSync(tsv);
      files.add(XFile(file.path));
      // v0.9: снимок исходной таблицы ДО правок — страховка для отката.
      final origRows = <String>[
        for (var ri = 0; ri < g.rows; ri++)
          '${g.y.values[ri].toStringAsFixed(2)}\t${[for (var ci = 0; ci < g.cols; ci++) g.data[ri][ci].toStringAsFixed(3)].join('\t')}',
      ];
      final origFile = File(
        '${outDir.path}/${e.key.replaceAll(RegExp(r'[/ ]'), '_')}_original.tsv',
      )..writeAsStringSync('$head\n${origRows.join('\n')}\n');
      files.add(XFile(origFile.path));
      md.writeln('\n## ${e.key} (${g.units})');
      for (var ri = 0; ri < g.rows; ri++) {
        for (var ci = 0; ci < g.cols; ci++) {
          final d = res.delta[ri][ci];
          if (d == 0) continue;
          final inf = res.info[ri][ci];
          md.writeln(
            '- ${g.x.values[ci]} × ${g.y.values[ri]}: ${g.data[ri][ci]} → '
            '${(g.data[ri][ci] + d).toStringAsFixed(2)} (${d > 0 ? '+' : ''}$d)'
            '${inf != null && inf.why.isNotEmpty ? ' — ${inf.why}' : ''}',
          );
        }
      }
    }
    final mf = File('${outDir.path}/recommendations.md')
      ..writeAsStringSync(md.toString());
    files.insert(0, XFile(mf.path));
    _say('экспорт: ${files.length} файлов → ${outDir.path}');
    await SharePlus.instance.share(
      ShareParams(
        files: files,
        text: 'Map Lab — рекомендации и снимки исходных карт',
      ),
    );
  }

  @override
  Widget build(BuildContext context) {
    final theme = Theme.of(context);
    final map = _maps == null ? null : _maps![_mapName];
    MapResult? res;
    if (map != null) res = _results?[map.name];

    // Весь экран снова в одном ListView — журнал всегда можно долистать.
    // Карта фиксированной большой высоты; 3D крутится через EagerPan внутри map3d_view.
    return Scaffold(
      appBar: AppBar(
        title: const Text('Map Lab · A2TB100B'),
        actions: [
          if (_results != null)
            IconButton(
              icon: const Icon(Icons.ios_share),
              onPressed: _export,
              tooltip: 'Экспорт',
            ),
        ],
      ),
      body: ListView(
        padding: const EdgeInsets.fromLTRB(12, 10, 12, 24),
        children: [
          Wrap(
            spacing: 8,
            runSpacing: 8,
            children: [
              _stepBtn(
                '0 · Дефиниции',
                Icons.folder_special,
                _pickDefs,
                ok: _defs != null,
              ),
              _stepBtn(
                '1 · Прошивка',
                Icons.memory,
                _pickRom,
                ok: _maps != null,
              ),
              _stepBtn(
                '2 · Лог',
                Icons.description,
                _pickLog,
                ok: _log != null,
              ),
              _stepBtn(
                '3 · Анализ',
                Icons.psychology,
                (_maps != null && _log != null) ? _runAnalysis : null,
                ok: _results != null,
              ),
              _stepBtn(
                'Экспорт',
                Icons.ios_share,
                _results != null ? _export : null,
              ),
            ],
          ),
          Align(
            alignment: Alignment.centerLeft,
            child: TextButton.icon(
              onPressed: _loadLatestLog, // v0.10.1 SYNC
              icon: const Icon(Icons.history, size: 16),
              label: const Text('последний лог приложения — в один тап'),
              style: TextButton.styleFrom(
                foregroundColor: const Color(0xFF22D3EE),
                textStyle: const TextStyle(fontSize: 12),
              ),
            ),
          ),
          const SizedBox(height: 6),
          Text(
            _defs != null
                ? 'XML: $_defsSource · ${_defs!.meta.xmlid}'
                : 'Нужны A2TB100B.xml, A2TB100K.xml, 32BITBASE.xml',
            style: theme.textTheme.bodySmall?.copyWith(
              color: _defs != null ? Colors.greenAccent : Colors.amber,
            ),
          ),
          if (_maps != null) ...[
            const SizedBox(height: 10),
            SizedBox(
              height: 40,
              child: ListView(
                scrollDirection: Axis.horizontal,
                children: [
                  for (final e in _maps!.entries)
                    Padding(
                      padding: const EdgeInsets.only(right: 8),
                      child: ChoiceChip(
                        selected: _mapName == e.key,
                        onSelected: (_) => setState(() {
                          _mapName = e.key;
                          _selRi = _selCi = null;
                        }),
                        label: Text(
                          e.key,
                          style: const TextStyle(fontSize: 11),
                        ),
                      ),
                    ),
                ],
              ),
            ),
            const SizedBox(height: 6),
            Row(
              children: [
                Expanded(
                  child: Text(
                    map == null ? '' : '${map.name} · ${map.rows}×${map.cols}',
                    style: theme.textTheme.bodySmall,
                    overflow: TextOverflow.ellipsis,
                  ),
                ),
                SegmentedButton<bool>(
                  segments: const [
                    ButtonSegment(
                      value: false,
                      icon: Icon(Icons.table_chart, size: 16),
                      label: Text('2D'),
                    ),
                    ButtonSegment(
                      value: true,
                      icon: Icon(Icons.threed_rotation, size: 16),
                      label: Text('3D'),
                    ),
                  ],
                  selected: {_view3D},
                  onSelectionChanged: (s) => setState(() => _view3D = s.first),
                ),
              ],
            ),
          ],
          if (map != null) ...[
            const SizedBox(height: 8),
            SizedBox(
              height: _mapHeight,
              child: Card(
                clipBehavior: Clip.antiAlias,
                child: _view3D
                    ? Map3DView(
                        grid: map,
                        result: res,
                        selRi: _selRi,
                        selCi: _selCi,
                        onCell: (ri, ci) => setState(() {
                          _selRi = ri;
                          _selCi = ci;
                        }),
                      )
                    : _Table2D(
                        grid: map,
                        result: res,
                        selRi: _selRi,
                        selCi: _selCi,
                        onCell: (ri, ci) => setState(() {
                          _selRi = ri;
                          _selCi = ci;
                        }),
                      ),
              ),
            ),
            if (_selRi != null && _selCi != null) ...[
              const SizedBox(height: 8),
              _inspector(map, res),
            ],
            const SizedBox(height: 8),
            Wrap(
              spacing: 12,
              runSpacing: 4,
              children: [
                _legend(const Color(0xFF4CAF50), 'низ'),
                _legend(const Color(0xFFFFEB3B), 'середина'),
                _legend(const Color(0xFFF44336), 'верх'),
                _legend(Colors.lightBlueAccent, 'правка'),
              ],
            ),
          ],
          const SizedBox(height: 16),
          Text('Журнал', style: theme.textTheme.titleSmall),
          const SizedBox(height: 6),
          Container(
            width: double.infinity,
            constraints: const BoxConstraints(minHeight: 180),
            padding: const EdgeInsets.all(10),
            decoration: BoxDecoration(
              color: theme.colorScheme.surfaceContainerHighest.withValues(
                alpha: 0.35,
              ),
              borderRadius: BorderRadius.circular(8),
            ),
            child: _journal.isEmpty
                ? const Text(
                    '— пока пусто —',
                    style: TextStyle(color: Colors.white38, fontSize: 12),
                  )
                : Column(
                    crossAxisAlignment: CrossAxisAlignment.start,
                    children: [
                      for (final j in _journal)
                        Padding(
                          padding: const EdgeInsets.only(bottom: 2),
                          child: Text(
                            j,
                            style: const TextStyle(
                              fontFamily: 'monospace',
                              fontSize: 11,
                            ),
                          ),
                        ),
                    ],
                  ),
          ),
          const SizedBox(height: 12),
          const Text(
            'Листайте экран вниз, чтобы читать журнал. '
            'На 3D-карте тяните пальцем — страница при этом не уезжает.',
            style: TextStyle(fontSize: 11, color: Colors.white38),
          ),
        ],
      ),
    );
  }

  Widget _stepBtn(
    String label,
    IconData icon,
    VoidCallback? onTap, {
    bool ok = false,
  }) {
    return FilledButton.tonalIcon(
      onPressed: onTap,
      icon: Icon(ok ? Icons.check_circle : icon, size: 18),
      label: Text(label, style: const TextStyle(fontSize: 12)),
    );
  }

  Widget _legend(Color c, String t) => Row(
    mainAxisSize: MainAxisSize.min,
    children: [
      Container(
        width: 12,
        height: 8,
        decoration: BoxDecoration(
          color: c,
          borderRadius: BorderRadius.circular(2),
        ),
      ),
      const SizedBox(width: 6),
      Text(t, style: const TextStyle(fontSize: 10.5)),
    ],
  );

  Widget _inspector(MapGrid g, MapResult? res) {
    final ri = _selRi!, ci = _selCi!;
    final v = g.data[ri][ci];
    final d = res?.delta[ri][ci] ?? 0;
    final inf = res?.info[ri][ci];
    final col = d < 0
        ? const Color(0xFFFF5C5C)
        : d > 0
        ? const Color(0xFFA8FF3E)
        : null;
    return Card(
      child: Padding(
        padding: const EdgeInsets.all(10),
        child: Column(
          crossAxisAlignment: CrossAxisAlignment.start,
          children: [
            Text(
              'Ячейка: ${g.x.name}=${g.x.values[ci]} · ${g.y.name}=${g.y.values[ri]}',
              style: const TextStyle(fontFamily: 'monospace', fontSize: 12),
            ),
            const SizedBox(height: 4),
            Row(
              children: [
                Text(
                  v.toStringAsFixed(2),
                  style: const TextStyle(
                    fontSize: 18,
                    fontFamily: 'monospace',
                    decoration: TextDecoration.lineThrough,
                  ),
                ),
                const SizedBox(width: 8),
                const Icon(Icons.arrow_forward, size: 16),
                const SizedBox(width: 8),
                Text(
                  '${(v + d).toStringAsFixed(2)} ${g.units}',
                  style: TextStyle(
                    fontSize: 22,
                    fontFamily: 'monospace',
                    color: col,
                  ),
                ),
              ],
            ),
            if (inf != null && inf.why.isNotEmpty)
              Padding(
                padding: const EdgeInsets.only(top: 4),
                child: Text(inf.why, style: const TextStyle(fontSize: 12)),
              ),
          ],
        ),
      ),
    );
  }
}

class _FilePickerSheet extends StatefulWidget {
  const _FilePickerSheet({
    required this.extensions,
    required this.title,
    this.allowPickFolderDefs = false,
  });
  final List<String> extensions;
  final String title;
  final bool allowPickFolderDefs;

  @override
  State<_FilePickerSheet> createState() => _FilePickerSheetState();
}

class _FilePickerSheetState extends State<_FilePickerSheet> {
  Directory _currentDir = Directory('/storage/emulated/0');
  bool _hasPermission = false;
  List<FileSystemEntity> _items = [];

  @override
  void initState() {
    super.initState();
    _checkPermissionAndRefresh();
  }

  Future<void> _checkPermissionAndRefresh() async {
    var ok = false;
    try {
      ok = await Permission.manageExternalStorage.isGranted;
    } catch (_) {}
    if (!ok) {
      final docs = await getApplicationDocumentsDirectory();
      _currentDir = docs;
    } else if (!_currentDir.existsSync()) {
      _currentDir = Directory('/storage/emulated/0');
    }
    setState(() => _hasPermission = ok);
    _refreshList();
  }

  Future<void> _requestPermission() async {
    try {
      await Permission.manageExternalStorage.request();
    } catch (_) {}
    try {
      await openAppSettings();
    } catch (_) {}
    _checkPermissionAndRefresh();
  }

  void _refreshList() {
    try {
      if (!_currentDir.existsSync()) {
        setState(() => _items = []);
        return;
      }
      final list = _currentDir.listSync(followLinks: false);
      final dirs = <Directory>[];
      final files = <File>[];
      for (final e in list) {
        final name = e.path.split('/').last;
        if (name.startsWith('.')) continue;
        if (e is Directory) {
          dirs.add(e);
        } else if (e is File) {
          if (widget.extensions.any((x) => name.toLowerCase().endsWith(x)))
            files.add(e);
        }
      }
      dirs.sort((a, b) => a.path.toLowerCase().compareTo(b.path.toLowerCase()));
      files.sort(
        (a, b) => b.lastModifiedSync().compareTo(a.lastModifiedSync()),
      );
      setState(() => _items = [...dirs, ...files]);
    } catch (_) {
      setState(() => _items = []);
    }
  }

  void _goUp() {
    final parent = _currentDir.parent;
    if (parent.path.length >= 4) {
      setState(() => _currentDir = parent);
      _refreshList();
    }
  }

  void _goTo(String path) {
    final d = Directory(path);
    if (d.existsSync()) {
      setState(() => _currentDir = d);
      _refreshList();
    }
  }

  @override
  Widget build(BuildContext context) {
    return Container(
      height: MediaQuery.of(context).size.height * 0.85,
      decoration: const BoxDecoration(
        color: Color(0xFF141921),
        borderRadius: BorderRadius.vertical(top: Radius.circular(16)),
      ),
      child: Column(
        children: [
          Padding(
            padding: const EdgeInsets.symmetric(horizontal: 16, vertical: 12),
            child: Row(
              children: [
                Expanded(
                  child: Text(
                    widget.title,
                    style: const TextStyle(
                      fontWeight: FontWeight.bold,
                      fontSize: 16,
                    ),
                  ),
                ),
                IconButton(
                  icon: const Icon(Icons.close),
                  onPressed: () => Navigator.of(context).pop(),
                ),
              ],
            ),
          ),
          if (!_hasPermission)
            Padding(
              padding: const EdgeInsets.symmetric(horizontal: 12),
              child: TextButton(
                onPressed: _requestPermission,
                child: const Text('РАЗРЕШИТЬ доступ к файлам'),
              ),
            ),
          SingleChildScrollView(
            scrollDirection: Axis.horizontal,
            padding: const EdgeInsets.symmetric(horizontal: 12, vertical: 4),
            child: Row(
              children: [
                _chip('Память', '/storage/emulated/0'),
                _chip('Downloads', '/storage/emulated/0/Download'),
                _chip('Telegram', '/storage/emulated/0/Telegram'),
                _chip('Documents', '/storage/emulated/0/Documents'),
              ],
            ),
          ),
          Container(
            padding: const EdgeInsets.symmetric(horizontal: 8, vertical: 4),
            color: Colors.black26,
            child: Row(
              children: [
                IconButton(
                  icon: const Icon(Icons.arrow_upward, size: 18),
                  onPressed: _goUp,
                ),
                Expanded(
                  child: Text(
                    _currentDir.path,
                    style: const TextStyle(
                      fontFamily: 'monospace',
                      fontSize: 11,
                      color: Colors.cyanAccent,
                    ),
                    overflow: TextOverflow.ellipsis,
                  ),
                ),
                if (widget.allowPickFolderDefs)
                  FilledButton.tonal(
                    onPressed: () => Navigator.of(context).pop(_currentDir),
                    child: const Text(
                      'Взять XML из ЭТОЙ папки',
                      style: TextStyle(fontSize: 11),
                    ),
                  ),
              ],
            ),
          ),
          const Divider(height: 1),
          Expanded(
            child: ListView.builder(
              itemCount: _items.length,
              itemBuilder: (ctx, i) {
                final item = _items[i];
                final name = item.path.split('/').last;
                if (item is Directory) {
                  return ListTile(
                    dense: true,
                    leading: const Icon(Icons.folder, color: Colors.amber),
                    title: Text(name),
                    onTap: () {
                      setState(() => _currentDir = item);
                      _refreshList();
                    },
                  );
                } else if (item is File) {
                  return ListTile(
                    dense: true,
                    leading: const Icon(
                      Icons.insert_drive_file,
                      color: Colors.lightBlueAccent,
                    ),
                    title: Text(
                      name,
                      style: const TextStyle(
                        fontFamily: 'monospace',
                        fontSize: 13,
                      ),
                    ),
                    onTap: () => Navigator.of(ctx).pop(item),
                  );
                }
                return const SizedBox.shrink();
              },
            ),
          ),
        ],
      ),
    );
  }

  Widget _chip(String label, String path) {
    return Padding(
      padding: const EdgeInsets.only(right: 6),
      child: ActionChip(
        visualDensity: VisualDensity.compact,
        label: Text(label, style: const TextStyle(fontSize: 11)),
        onPressed: () => _goTo(path),
      ),
    );
  }
}

/// 2D таблица с pinch-zoom и кнопками +/−.
class _Table2D extends StatefulWidget {
  const _Table2D({
    required this.grid,
    this.result,
    this.selRi,
    this.selCi,
    this.onCell,
  });
  final MapGrid grid;
  final MapResult? result;
  final int? selRi, selCi;
  final void Function(int, int)? onCell;

  @override
  State<_Table2D> createState() => _Table2DState();
}

class _Table2DState extends State<_Table2D> {
  final _tc = TransformationController();
  static const _minS = 0.25;
  static const _maxS = 6.0;

  @override
  void dispose() {
    _tc.dispose();
    super.dispose();
  }

  void _zoomBy(double factor) {
    final m = _tc.value.clone();
    final s = m.getMaxScaleOnAxis();
    final next = (s * factor).clamp(_minS, _maxS);
    final f = next / s;
    final child = context.findRenderObject() as RenderBox?;
    if (child == null) {
      _tc.value = m.clone()..multiply(Matrix4.diagonal3Values(f, f, f));
      return;
    }
    final center = child.size.center(Offset.zero);
    final scene = _tc.toScene(center);
    m.multiply(Matrix4.translationValues(scene.dx, scene.dy, 0));
    m.multiply(Matrix4.diagonal3Values(f, f, f));
    m.multiply(Matrix4.translationValues(-scene.dx, -scene.dy, 0));
    _tc.value = m;
    setState(() {});
  }

  void _reset() {
    _tc.value = Matrix4.identity();
    setState(() {});
  }

  @override
  Widget build(BuildContext context) {
    final g = widget.grid;
    return Stack(
      children: [
        InteractiveViewer(
          transformationController: _tc,
          minScale: _minS,
          maxScale: _maxS,
          boundaryMargin: const EdgeInsets.all(200),
          constrained: false,
          panEnabled: true,
          scaleEnabled: true,
          child: Padding(
            padding: const EdgeInsets.all(8),
            child: Table(
              defaultColumnWidth: const IntrinsicColumnWidth(),
              children: [
                TableRow(
                  children: [
                    _head('${g.y.guess}↓ ${g.x.guess}→'),
                    for (final v in g.x.values)
                      _head(v.toStringAsFixed(0), accent: true),
                  ],
                ),
                for (var ri = 0; ri < g.rows; ri++)
                  TableRow(
                    children: [
                      _head(g.y.values[ri].toStringAsFixed(2), accent: true),
                      for (var ci = 0; ci < g.cols; ci++) _cell(ri, ci),
                    ],
                  ),
              ],
            ),
          ),
        ),
        Positioned(
          right: 8,
          bottom: 8,
          child: Column(
            children: [
              _zBtn(Icons.add, () => _zoomBy(1.25)),
              const SizedBox(height: 6),
              _zBtn(Icons.remove, () => _zoomBy(1 / 1.25)),
              const SizedBox(height: 6),
              _zBtn(Icons.center_focus_strong, _reset),
            ],
          ),
        ),
        const Positioned(
          left: 8,
          bottom: 8,
          child: Text(
            'pinch / кнопки = зум',
            style: TextStyle(fontSize: 10, color: Colors.white38),
          ),
        ),
      ],
    );
  }

  Widget _zBtn(IconData icon, VoidCallback onTap) {
    return Material(
      color: Colors.black54,
      shape: const CircleBorder(),
      child: InkWell(
        customBorder: const CircleBorder(),
        onTap: onTap,
        child: Padding(
          padding: const EdgeInsets.all(8),
          child: Icon(icon, size: 18, color: Colors.white),
        ),
      ),
    );
  }

  Widget _head(String t, {bool accent = false}) => Container(
    padding: const EdgeInsets.symmetric(horizontal: 6, vertical: 4),
    decoration: BoxDecoration(border: Border.all(color: Colors.white12)),
    child: Text(
      t,
      style: TextStyle(
        fontSize: 9.5,
        fontFamily: 'monospace',
        color: accent ? const Color(0xCC4BE1FF) : Colors.white54,
      ),
    ),
  );

  Widget _cell(int ri, int ci) {
    final g = widget.grid;
    final v = g.data[ri][ci];
    final d = widget.result?.delta[ri][ci] ?? 0;
    final sel = widget.selRi == ri && widget.selCi == ci;
    final heat = getHeatmapColor(v, g.vmin, g.vmax).withValues(alpha: 0.55);
    final hasFix = d != 0;

    return InkWell(
      onTap: () => widget.onCell?.call(ri, ci),
      child: Container(
        constraints: const BoxConstraints(minWidth: 48, minHeight: 40),
        padding: const EdgeInsets.symmetric(horizontal: 4, vertical: 3),
        decoration: BoxDecoration(
          color: heat,
          border: Border.all(
            color: sel
                ? Colors.white
                : hasFix
                ? Colors.lightBlueAccent
                : Colors.white12,
            width: sel ? 1.6 : (hasFix ? 1.2 : 0.5),
          ),
        ),
        child: Column(
          mainAxisAlignment: MainAxisAlignment.center,
          children: [
            Text(
              v.toStringAsFixed(2),
              style: const TextStyle(
                fontSize: 10,
                fontFamily: 'monospace',
                color: Colors.white,
              ),
            ),
            if (hasFix)
              Text(
                (v + d).toStringAsFixed(2),
                style: const TextStyle(
                  fontSize: 11,
                  fontWeight: FontWeight.bold,
                  fontFamily: 'monospace',
                  color: Colors.lightBlueAccent,
                ),
              ),
          ],
        ),
      ),
    );
  }
}
'''

EXPECTED_MARKERS = {
    "android/app/src/main/kotlin/com/subaru/ssm2_fixed/MainActivity.kt": "spp-reader",
    "lib/elm.dart": "_consecutiveTimeouts",
    "lib/engine.dart": "tripDistanceKm",
    "lib/analyzer.dart": "iamScaled",
    "lib/native_spp.dart": "TimeoutException",
    "lib/model.dart": "_stepMs",
    "lib/main.dart": "SSM2 TELEMETRY 0.10.2",
    "test/protocol_test.dart": "ELM-01 retry",
    "lib/maps_lab/maps_lab_log.dart": "loadIsPercent",
    "lib/maps_lab/maps_lab_page.dart": "_loadLatestLog",
    "test/maps_lab_test.dart": "Analyzer: процентная",
}


def main():
    if not (APP / "lib/elm.dart").exists():
        raise RuntimeError("Проект не найден. Сначала выполните ячейки 01-04.")
    if not (APP / "lib/maps_lab/maps_lab_page.dart").exists():
        raise RuntimeError("Не найден Map Lab. Сначала выполните ячейку 04.")

    changed, skipped = [], []
    targets = {rel: body.strip("\n") + "\n" for rel, body in FILES.items()}

    todo = {rel: body for rel, body in targets.items()
            if (APP / rel).read_text(encoding="utf-8") != body}
    if todo:
        backup = APP.parent / f"ssm2_backup_0_10_2_{time.strftime('%Y%m%d_%H%M%S')}"
        for rel in todo:
            dst = backup / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(APP / rel, dst)
        print(f"[бэкап] {len(todo)} файлов -> {backup}")

    for rel, body in targets.items():
        path = APP / rel
        if path.read_text(encoding="utf-8") == body:
            skipped.append(rel)
        else:
            path.write_text(body, encoding="utf-8")
            changed.append(rel)
            print("[fix]", rel)

    problems = [f"{rel}: нет маркера {mk!r}" for rel, mk in EXPECTED_MARKERS.items()
                if mk not in (APP / rel).read_text(encoding="utf-8")]
    if problems:
        raise RuntimeError("Самопроверка:\n  - " + "\n  - ".join(problems))

    pub = APP / "pubspec.yaml"
    pub.write_text(re.sub(r"(?m)^version: .+$", "version: 0.10.2+14",
                          pub.read_text(encoding="utf-8")), encoding="utf-8")
    cfg = APP / "build_config.json"
    if cfg.exists():
        data = json.loads(cfg.read_text(encoding="utf-8"))
        data["revision"] = "0.10.2"
        cfg.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

    print()
    print(f"SSM2 0.10.2 | изменено: {len(changed)} · пропущено: {len(skipped)} · маркеры: OK")
    print("Новое в 0.10.2:")
    print("  - Автозонд endian для 4-byte PID (IAM): если BE = 0, пробует LE")
    print("  - Trip-одометр + потраченное топливо + л/100км + кнопка «Сброс»")
    print("  - Raw-байты на карточках PID (для диагностики нулей)")
    print("Дальше: ячейка 05 (26 тестов, APK 0.10.2).")


main()


[бэкап] 11 файлов -> /content/ssm2_backup_0_10_2_20261006_081137
[fix] android/app/src/main/kotlin/com/subaru/ssm2_fixed/MainActivity.kt
[fix] lib/elm.dart
[fix] lib/engine.dart
[fix] lib/analyzer.dart
[fix] lib/native_spp.dart
[fix] lib/model.dart
[fix] lib/main.dart
[fix] test/protocol_test.dart
[fix] lib/maps_lab/maps_lab_log.dart
[fix] test/maps_lab_test.dart
[fix] lib/maps_lab/maps_lab_page.dart

SSM2 0.10.2 | изменено: 11 · пропущено: 0 · маркеры: OK
Новое в 0.10.2:
  - Автозонд endian для 4-byte PID (IAM): если BE = 0, пробует LE
  - Trip-одометр + потраченное топливо + л/100км + кнопка «Сброс»
  - Raw-байты на карточках PID (для диагностики нулей)
Дальше: ячейка 05 (26 тестов, APK 0.10.2).


In [ ]:
# @title 04.2 | FULL | SSM2 0.10.2 >> 0.11.0 — ошибки + сервис + loggerdefs A2TB100B (вставить МЕЖДУ 04.1 и 05)
# Что делает:
#  1. lib/pids.dart: исходные 28 PID + 27 параметров из RomRaider logger.xml,
#     отобранных по ECU ID 5204584007 (A2TB100B): Requested Torque (0xFF8058),
#     Knock Sum, KC Advance 4B, Base Timing, LTFT A-D (FF24B4..CC), INJ PW и др.
#  2. lib/expr.dart — безопасный эвалуатор выражений logger-дефиниций.
#  3. lib/identity.dart — профиль A2TB100B с адресами из logger.xml.
#  4. lib/loggerdef.dart — полный каталог 5204584007 (91 std + 59 ext + 68 switches).
#  5. Диагностика: lib/diag.dart (SSM2 18/14, OBD 03/04, UDS 19 02/14 через
#     существующий ElmDriver.diagnostic) + вкладка «DTC» + справочник кодов.
#  6. Расширение белого списка команд (protocol.dart), пауза опроса (model.dart),
#     алиасы Map Lab, protocol_test (28 -> 55 PID), calid_import VALID.
# Идемпотентна: бэкап _backup_042_<ts>, SKIP при повторном прогоне, самопроверка.
# EXPERIMENTAL=False: ECUReset/Clear Memory заблокированы до стенда.

import json
import re
import shutil
import time
from pathlib import Path

APP = Path("/content/subaru_ssm2_fixed")
EXPERIMENTAL = False  # @param {type:"boolean"}  # UDS 11 01 / Clear Memory 04 gr

STAMP = time.strftime("%Y%m%d_%H%M%S")
# Бэкап СТРОГО снаружи APP (в /content/), чтобы flutter analyze не сканировал его
BACKUP = APP.parent / f"ssm2_backup_042_{STAMP}"
FILES = {}
REWRITES = []  # (path, needle, replacement)

# ===== Безопасный эвалуатор выражений logger-дефиниций =====
FILES["lib/expr.dart"] = r'''
import 'dart:typed_data';

/// Мини-эвалуатор выражений logger-дефиниций RomRaider: + - * / скобки,
/// числа, идентификатор x (= сырое значение из памяти ECU).
/// Покрывает все expr из SubaruDefs logger.xml: 'x', 'x*K', '(x*100)-100',
/// '14.7/(1+x)', 'x/.84', 'x+6' и т.п. Безопасен: никаких вызовов/имён, кроме x.
class ExprEval {
  ExprEval._(this.src);
  final String src;
  int pos = 0;

  static double? eval(String expr, double x) {
    try {
      final p = ExprEval._(expr);
      final v = p._expr(x);
      p._ws();
      if (p.pos != expr.length) return null;
      return v.isFinite ? v : null;
    } catch (_) {
      return null;
    }
  }

  void _ws() {
    while (pos < src.length && src[pos] == ' ') pos++;
  }

  bool _eat(String ch) {
    _ws();
    if (pos < src.length && src[pos] == ch) {
      pos++;
      return true;
    }
    return false;
  }

  double _expr(double x) {
    var v = _term(x);
    while (true) {
      if (_eat('+')) {
        v = v + _term(x);
      } else if (_eat('-')) {
        v = v - _term(x);
      } else {
        return v;
      }
    }
  }

  double _term(double x) {
    var v = _factor(x);
    while (true) {
      if (_eat('*')) {
        v = v * _factor(x);
      } else if (_eat('/')) {
        final d = _factor(x);
        if (d == 0) return double.nan;
        v = v / d;
      } else {
        return v;
      }
    }
  }

  double _factor(double x) {
    _ws();
    if (_eat('-')) return -_factor(x);
    if (_eat('(')) {
      final v = _expr(x);
      _eat(')');
      return v;
    }
    _ws();
    if (pos < src.length && src[pos] == 'x') {
      pos++;
      return x;
    }
    final start = pos;
    while (pos < src.length &&
        (src.codeUnitAt(pos) >= 0x30 && src.codeUnitAt(pos) <= 0x39 || src[pos] == '.')) {
      pos++;
    }
    if (start == pos) throw StateError('ожидалось число в "$src"');
    return double.parse(src.substring(start, pos));
  }
}

/// сырой X из байтов ответа по длине параметра
double exprRawX(List<int> bytes, Endian endian) {
  if (bytes.length == 4) {
    return ByteData.sublistView(Uint8List.fromList(bytes)).getFloat32(0, endian);
  }
  if (bytes.length == 2) return (bytes[0] * 256 + bytes[1]).toDouble();
  return bytes[0].toDouble();
}
'''

# ===== Библиотека 55 PID: 28 исходных + 27 loggerdef · REQ_TQ @ 0xFF8058 =====
FILES["lib/pids.dart"] = r'''
// v0.11 (патч 04.2): исходные 28 PID из ячейки 02 + 27 параметров logger-дефиниций
// RomRaider SubaruDefs Stable, отфильтрованных по ECU ID 5204584007 (A2TB100B).
// Новые параметры декодируются через ExprEval (exprText), float32 BE/LE — автозонд engine.

import 'dart:typed_data';

import 'expr.dart';

typedef PidFormula = double Function(List<int> bytes);

class SubaruPidDef {
  const SubaruPidDef({
    required this.id,
    required this.desc,
    required this.unit,
    required this.category,
    required this.address,
    required this.bytesCount,
    required this.priority,
    required this.formulaText,
    required this.minValue,
    required this.maxValue,
    required this.digits,
    this.formula,
    this.floatFactor,
    this.exprText,
  });

  final String id, desc, unit, category, formulaText;
  final int address, bytesCount, priority, digits;
  final double minValue, maxValue;
  final PidFormula? formula;
  final double? floatFactor;

  /// v0.11: выражение loggerdef (x = raw из ECU). Задано у ROM-зависимых PID.
  final String? exprText;

  String get name => id;

  /// ROM-зависимый (расширенный) параметр — под CALID-гейтом опроса.
  bool get extended => floatFactor != null || exprText != null;

  List<int> get addresses => List<int>.generate(bytesCount, (i) => address + i);

  double? decode(List<int> bytes, {Endian endian = Endian.big}) {
    if (bytes.length != bytesCount || bytes.any((b) => b < 0 || b > 255)) {
      return null;
    }
    final expr = exprText;
    final factor = floatFactor;
    final double value;
    if (expr != null) {
      final v = ExprEval.eval(expr, exprRawX(bytes, endian));
      if (v == null) return null;
      value = v;
    } else if (factor == null) {
      value = formula!(bytes);
    } else {
      value = ByteData.sublistView(Uint8List.fromList(bytes)).getFloat32(0, endian) * factor;
    }
    return value.isFinite ? value : null;
  }
}

class SubaruPidLibrary {
  static final List<SubaruPidDef> all = <SubaruPidDef>[
    SubaruPidDef(
      id: 'LOAD',
      desc: "Engine Load (Relative)",
      unit: '%',
      category: 'engine',
      address: 0x000007,
      bytesCount: 1,
      priority: 1,
      formulaText: "A*100/255",
      minValue: 0.0,
      maxValue: 100.0,
      digits: 1,
      formula: (b) => b[0]*100/255,),
    SubaruPidDef(
      id: 'ECT',
      desc: "Coolant Temperature",
      unit: 'C',
      category: 'temp',
      address: 0x000008,
      bytesCount: 1,
      priority: 1,
      formulaText: "A-40",
      minValue: -40.0,
      maxValue: 130.0,
      digits: 0,
      formula: (b) => b[0]-40.0,),
    SubaruPidDef(
      id: 'STFT',
      desc: "A/F Correction #1",
      unit: '%',
      category: 'fuel',
      address: 0x000009,
      bytesCount: 1,
      priority: 1,
      formulaText: "(A-128)*100/128",
      minValue: -100.0,
      maxValue: 100.0,
      digits: 2,
      formula: (b) => (b[0]-128)*100/128,),
    SubaruPidDef(
      id: 'LTFT',
      desc: "A/F Learning #1",
      unit: '%',
      category: 'fuel',
      address: 0x00000A,
      bytesCount: 1,
      priority: 1,
      formulaText: "(A-128)*100/128",
      minValue: -100.0,
      maxValue: 100.0,
      digits: 2,
      formula: (b) => (b[0]-128)*100/128,),
    SubaruPidDef(
      id: 'MAP_ABS',
      desc: "Manifold Absolute Pressure",
      unit: 'bar',
      category: 'air',
      address: 0x00000D,
      bytesCount: 1,
      priority: 2,
      formulaText: "A*37/255/14.50377",
      minValue: 0.0,
      maxValue: 3.0,
      digits: 3,
      formula: (b) => b[0]*37/255/14.50377,),
    SubaruPidDef(
      id: 'RPM',
      desc: "Engine Speed",
      unit: 'rpm',
      category: 'engine',
      address: 0x00000E,
      bytesCount: 2,
      priority: 1,
      formulaText: "(A*256+B)/4",
      minValue: 0.0,
      maxValue: 8000.0,
      digits: 0,
      formula: (b) => (b[0]*256+b[1])/4,),
    SubaruPidDef(
      id: 'SPEED',
      desc: "Vehicle Speed",
      unit: 'kph',
      category: 'engine',
      address: 0x000010,
      bytesCount: 1,
      priority: 1,
      formulaText: "A",
      minValue: 0.0,
      maxValue: 240.0,
      digits: 0,
      formula: (b) => b[0].toDouble(),),
    SubaruPidDef(
      id: 'TIMING',
      desc: "Total Ignition Timing",
      unit: 'degrees',
      category: 'ignition',
      address: 0x000011,
      bytesCount: 1,
      priority: 1,
      formulaText: "(A-128)/2",
      minValue: -64.0,
      maxValue: 64.0,
      digits: 1,
      formula: (b) => (b[0]-128)/2,),
    SubaruPidDef(
      id: 'IAT',
      desc: "Intake Air Temperature",
      unit: 'C',
      category: 'temp',
      address: 0x000012,
      bytesCount: 1,
      priority: 1,
      formulaText: "A-40",
      minValue: -40.0,
      maxValue: 130.0,
      digits: 0,
      formula: (b) => b[0]-40.0,),
    SubaruPidDef(
      id: 'MAF',
      desc: "Mass Airflow",
      unit: 'g/s',
      category: 'air',
      address: 0x000013,
      bytesCount: 2,
      priority: 1,
      formulaText: "(A*256+B)/100",
      minValue: 0.0,
      maxValue: 400.0,
      digits: 2,
      formula: (b) => (b[0]*256+b[1])/100,),
    SubaruPidDef(
      id: 'TPS',
      desc: "Throttle Opening Angle",
      unit: '%',
      category: 'throttle',
      address: 0x000015,
      bytesCount: 1,
      priority: 1,
      formulaText: "A*100/255",
      minValue: 0.0,
      maxValue: 100.0,
      digits: 1,
      formula: (b) => b[0]*100/255,),
    SubaruPidDef(
      id: 'O2_F',
      desc: "Front O2 #1",
      unit: 'V',
      category: 'fuel',
      address: 0x000016,
      bytesCount: 2,
      priority: 3,
      formulaText: "(A*256+B)/200",
      minValue: 0.0,
      maxValue: 2.0,
      digits: 2,
      formula: (b) => (b[0]*256+b[1])/200.0,),
    SubaruPidDef(
      id: 'BATT',
      desc: "Battery Voltage",
      unit: 'V',
      category: 'electric',
      address: 0x00001C,
      bytesCount: 1,
      priority: 2,
      formulaText: "A*8/100",
      minValue: 8.0,
      maxValue: 18.0,
      digits: 2,
      formula: (b) => b[0]*8/100,),
    SubaruPidDef(
      id: 'KNOCK_ADV',
      desc: "Knock Correction Advance",
      unit: 'degrees',
      category: 'ignition',
      address: 0x000022,
      bytesCount: 1,
      priority: 1,
      formulaText: "(A-128)/2",
      minValue: -64.0,
      maxValue: 64.0,
      digits: 1,
      formula: (b) => (b[0]-128)/2,),
    SubaruPidDef(
      id: 'BARO',
      desc: "Atmospheric Pressure",
      unit: 'bar',
      category: 'air',
      address: 0x000023,
      bytesCount: 1,
      priority: 3,
      formulaText: "A*37/255/14.50377",
      minValue: 0.0,
      maxValue: 2.0,
      digits: 3,
      formula: (b) => b[0]*37/255/14.50377,),
    SubaruPidDef(
      id: 'MAP_REL',
      desc: "Manifold Relative Pressure",
      unit: 'bar',
      category: 'turbo',
      address: 0x000024,
      bytesCount: 1,
      priority: 1,
      formulaText: "(A-128)*37/255/14.50377",
      minValue: -1.3,
      maxValue: 1.3,
      digits: 3,
      formula: (b) => (b[0]-128)*37/255/14.50377,),
    SubaruPidDef(
      id: 'PEDAL',
      desc: "Accelerator Pedal Angle",
      unit: '%',
      category: 'throttle',
      address: 0x000029,
      bytesCount: 1,
      priority: 1,
      formulaText: "A*100/255",
      minValue: 0.0,
      maxValue: 100.0,
      digits: 1,
      formula: (b) => b[0]*100/255,),
    SubaruPidDef(
      id: 'WG_PRIM',
      desc: "Primary Wastegate Duty Cycle",
      unit: '%',
      category: 'turbo',
      address: 0x000030,
      bytesCount: 1,
      priority: 1,
      formulaText: "A*100/255",
      minValue: 0.0,
      maxValue: 100.0,
      digits: 1,
      formula: (b) => b[0]*100/255,),
    SubaruPidDef(
      id: 'AFR',
      desc: "A/F Sensor #1",
      unit: 'AFR',
      category: 'fuel',
      address: 0x000046,
      bytesCount: 1,
      priority: 1,
      formulaText: "A/128*14.7",
      minValue: 0.0,
      maxValue: 30.0,
      digits: 2,
      formula: (b) => b[0]/128*14.7,),
    SubaruPidDef(
      id: 'GEAR',
      desc: "Gear Position",
      unit: 'gear',
      category: 'engine',
      address: 0x00004A,
      bytesCount: 1,
      priority: 2,
      formulaText: "A+1",
      minValue: 1.0,
      maxValue: 8.0,
      digits: 0,
      formula: (b) => b[0]+1.0,),
    SubaruPidDef(
      id: 'IAM',
      desc: "IAM (4-byte)*",
      unit: 'multiplier',
      category: 'ignition',
      address: 0xFF2538,
      bytesCount: 4,
      priority: 1,
      formulaText: "float32",
      minValue: 0.0,
      maxValue: 1.0,
      digits: 3,
      floatFactor: 1,),
    SubaruPidDef(
      id: 'LOAD_4B',
      desc: "Engine Load (4-Byte)*",
      unit: 'g/rev',
      category: 'engine',
      address: 0xFF6C9C,
      bytesCount: 4,
      priority: 2,
      formulaText: "float32",
      minValue: 0.0,
      maxValue: 5.0,
      digits: 3,
      floatFactor: 1,),
    SubaruPidDef(
      id: 'BOOST_ERR',
      desc: "Boost Error*",
      unit: 'bar',
      category: 'turbo',
      address: 0xFF6450,
      bytesCount: 4,
      priority: 1,
      formulaText: "float32*0.001333224",
      minValue: -2.0,
      maxValue: 3.0,
      digits: 3,
      floatFactor: 0.001333224,),
    SubaruPidDef(
      id: 'BOOST_TGT',
      desc: "Target Boost (4-byte)*",
      unit: 'bar',
      category: 'turbo',
      address: 0xFF6454,
      bytesCount: 4,
      priority: 1,
      formulaText: "float32*0.001333224",
      minValue: -2.0,
      maxValue: 3.0,
      digits: 3,
      floatFactor: 0.001333224,),
    SubaruPidDef(
      id: 'FBKC',
      desc: "Feedback Knock Correction (4-byte)*",
      unit: 'degrees',
      category: 'ignition',
      address: 0xFF7D4C,
      bytesCount: 4,
      priority: 1,
      formulaText: "float32",
      minValue: -20.0,
      maxValue: 20.0,
      digits: 2,
      floatFactor: 1,),
    SubaruPidDef(
      id: 'FKL',
      desc: "Fine Learning Knock Correction*",
      unit: 'degrees',
      category: 'ignition',
      address: 0xFF7DD0,
      bytesCount: 4,
      priority: 1,
      formulaText: "float32",
      minValue: -20.0,
      maxValue: 20.0,
      digits: 2,
      floatFactor: 1,),
    SubaruPidDef(
      id: 'BOOST',
      desc: "MRP (Boost) (4-byte)*",
      unit: 'bar',
      category: 'turbo',
      address: 0xFF6AE0,
      bytesCount: 4,
      priority: 1,
      formulaText: "float32*0.001333224",
      minValue: -2.0,
      maxValue: 3.0,
      digits: 3,
      floatFactor: 0.001333224,),
    SubaruPidDef(
      id: 'CL_TARGET',
      desc: "Closed Loop Fuel Target*",
      unit: 'AFR',
      category: 'fuel',
      address: 0xFF73B4,
      bytesCount: 4,
      priority: 2,
      formulaText: "float32*14.7",
      minValue: 0.0,
      maxValue: 30.0,
      digits: 2,
      floatFactor: 14.7,),
    // --- v0.11: logger-дефиниции RomRaider для ECU 5204584007 (A2TB100B) ---
    SubaruPidDef(
      id: 'REQ_TQ',
      desc: "Requested Torque*",
      unit: 'Nm*',
      category: 'engine',
      address: 0xFF8058,
      bytesCount: 4,
      priority: 1,
      formulaText: "x",
      minValue: 0.0,
      maxValue: 500.0,
      digits: 1,
      exprText: "x"),
    SubaruPidDef(
      id: 'KC_ADV_4B',
      desc: "Knock Correction Advance (4-byte)*",
      unit: '°',
      category: 'ignition',
      address: 0xFF7D48,
      bytesCount: 4,
      priority: 1,
      formulaText: "x",
      minValue: -64.0,
      maxValue: 64.0,
      digits: 2,
      exprText: "x"),
    SubaruPidDef(
      id: 'KC_IAM',
      desc: "Knock Correction Advance (IAM only)*",
      unit: '°',
      category: 'ignition',
      address: 0xFF7DB0,
      bytesCount: 4,
      priority: 2,
      formulaText: "x",
      minValue: -64.0,
      maxValue: 64.0,
      digits: 2,
      exprText: "x"),
    SubaruPidDef(
      id: 'BTIMING',
      desc: "Ignition Base Timing*",
      unit: '°',
      category: 'ignition',
      address: 0xFF7B98,
      bytesCount: 4,
      priority: 2,
      formulaText: "x",
      minValue: -64.0,
      maxValue: 64.0,
      digits: 2,
      exprText: "x"),
    SubaruPidDef(
      id: 'KCMAX',
      desc: "Knock Correction Advance Max Primary*",
      unit: '°',
      category: 'ignition',
      address: 0xFF7DA8,
      bytesCount: 4,
      priority: 2,
      formulaText: "x",
      minValue: -20.0,
      maxValue: 70.0,
      digits: 2,
      exprText: "x"),
    SubaruPidDef(
      id: 'KNOCK_SUM',
      desc: "Knock Sum*",
      unit: 'cnt',
      category: 'ignition',
      address: 0xFF7D24,
      bytesCount: 4,
      priority: 2,
      formulaText: "x",
      minValue: 0.0,
      maxValue: 10000.0,
      digits: 0,
      exprText: "x"),
    SubaruPidDef(
      id: 'AFL_A',
      desc: "A/F Learning #1 A (Stored)*",
      unit: '%',
      category: 'fuel',
      address: 0xFF24B4,
      bytesCount: 4,
      priority: 2,
      formulaText: "x*100",
      minValue: -50.0,
      maxValue: 50.0,
      digits: 2,
      exprText: "x*100"),
    SubaruPidDef(
      id: 'AFL_B',
      desc: "A/F Learning #1 B (Stored)*",
      unit: '%',
      category: 'fuel',
      address: 0xFF24BC,
      bytesCount: 4,
      priority: 2,
      formulaText: "x*100",
      minValue: -50.0,
      maxValue: 50.0,
      digits: 2,
      exprText: "x*100"),
    SubaruPidDef(
      id: 'AFL_C',
      desc: "A/F Learning #1 C (Stored)*",
      unit: '%',
      category: 'fuel',
      address: 0xFF24C4,
      bytesCount: 4,
      priority: 2,
      formulaText: "x*100",
      minValue: -50.0,
      maxValue: 50.0,
      digits: 2,
      exprText: "x*100"),
    SubaruPidDef(
      id: 'AFL_D',
      desc: "A/F Learning #1 D (Stored)*",
      unit: '%',
      category: 'fuel',
      address: 0xFF24CC,
      bytesCount: 4,
      priority: 2,
      formulaText: "x*100",
      minValue: -50.0,
      maxValue: 50.0,
      digits: 2,
      exprText: "x*100"),
    SubaruPidDef(
      id: 'AFL_4B',
      desc: "A/F Learning #1 (4-byte)*",
      unit: '%',
      category: 'fuel',
      address: 0xFF768C,
      bytesCount: 4,
      priority: 3,
      formulaText: "x*100",
      minValue: -50.0,
      maxValue: 50.0,
      digits: 2,
      exprText: "x*100"),
    SubaruPidDef(
      id: 'AFCOR1_4B',
      desc: "A/F Correction #1 (4-byte)*",
      unit: '%',
      category: 'fuel',
      address: 0xFF73A4,
      bytesCount: 4,
      priority: 2,
      formulaText: "(x*100)-100",
      minValue: -50.0,
      maxValue: 50.0,
      digits: 2,
      exprText: "(x*100)-100"),
    SubaruPidDef(
      id: 'TIPIN',
      desc: "Tip-in Throttle*",
      unit: '%',
      category: 'throttle',
      address: 0xFF6B8C,
      bytesCount: 4,
      priority: 3,
      formulaText: "x",
      minValue: -10.0,
      maxValue: 120.0,
      digits: 1,
      exprText: "x"),
    SubaruPidDef(
      id: 'TTH_TGT',
      desc: "Target Throttle Plate Position*",
      unit: '%',
      category: 'throttle',
      address: 0xFF8040,
      bytesCount: 4,
      priority: 2,
      formulaText: "x/.84",
      minValue: 0.0,
      maxValue: 110.0,
      digits: 1,
      exprText: "x/.84"),
    SubaruPidDef(
      id: 'THROTTLE_4B',
      desc: "Throttle Plate Opening Angle (4-byte)*",
      unit: '%',
      category: 'throttle',
      address: 0xFF6B7C,
      bytesCount: 4,
      priority: 3,
      formulaText: "x/.84",
      minValue: 0.0,
      maxValue: 110.0,
      digits: 1,
      exprText: "x/.84"),
    SubaruPidDef(
      id: 'CLOL',
      desc: "CL/OL Fueling*",
      unit: 'st',
      category: 'fuel',
      address: 0xFF9679,
      bytesCount: 4,
      priority: 3,
      formulaText: "x+6",
      minValue: 0.0,
      maxValue: 16.0,
      digits: 0,
      exprText: "x+6"),
    SubaruPidDef(
      id: 'FL_OFFSET',
      desc: "Fine Learning Table Offset*",
      unit: 'idx',
      category: 'ignition',
      address: 0xFF7DD6,
      bytesCount: 4,
      priority: 3,
      formulaText: "x+1",
      minValue: 0.0,
      maxValue: 100.0,
      digits: 0,
      exprText: "x+1"),
    SubaruPidDef(
      id: 'GEAR_CALC',
      desc: "Gear (Calculated)*",
      unit: 'gear',
      category: 'engine',
      address: 0xFF7079,
      bytesCount: 4,
      priority: 2,
      formulaText: "x",
      minValue: 0.0,
      maxValue: 8.0,
      digits: 0,
      exprText: "x"),
    SubaruPidDef(
      id: 'TDIG_INT',
      desc: "Turbo Dynamics Integral (4-byte)*",
      unit: '%',
      category: 'turbo',
      address: 0xFF645C,
      bytesCount: 4,
      priority: 3,
      formulaText: "x",
      minValue: 0.0,
      maxValue: 100.0,
      digits: 2,
      exprText: "x"),
    SubaruPidDef(
      id: 'TDIG_PROP',
      desc: "Turbo Dynamics Proportional (4-byte)*",
      unit: '%',
      category: 'turbo',
      address: 0xFF6458,
      bytesCount: 4,
      priority: 3,
      formulaText: "x",
      minValue: 0.0,
      maxValue: 100.0,
      digits: 2,
      exprText: "x"),
    SubaruPidDef(
      id: 'INJPW',
      desc: "Fuel Injector #1 Pulse Width (4-byte)*",
      unit: 'ms',
      category: 'fuel',
      address: 0xFF7A48,
      bytesCount: 4,
      priority: 2,
      formulaText: "x*.001",
      minValue: 0.0,
      maxValue: 40.0,
      digits: 3,
      exprText: "x*.001"),
    SubaruPidDef(
      id: 'INJ_LAT',
      desc: "Fuel Injector #1 Latency (4-byte)*",
      unit: 'ms',
      category: 'fuel',
      address: 0xFF7A5C,
      bytesCount: 4,
      priority: 3,
      formulaText: "x*.001",
      minValue: 0.0,
      maxValue: 10.0,
      digits: 3,
      exprText: "x*.001"),
    SubaruPidDef(
      id: 'OLE_ENRICH',
      desc: "Primary Open Loop Map Enrichment (4-byte)*",
      unit: 'AFR',
      category: 'fuel',
      address: 0xFF7778,
      bytesCount: 4,
      priority: 3,
      formulaText: "14.7/(1+x)",
      minValue: 5.0,
      maxValue: 30.0,
      digits: 2,
      exprText: "14.7/(1+x)"),
    SubaruPidDef(
      id: 'ENRICH_FINAL',
      desc: "Primary Enrichment Final (4-byte)*",
      unit: 'AFR',
      category: 'fuel',
      address: 0xFF731C,
      bytesCount: 4,
      priority: 2,
      formulaText: "14.7/(1+x)",
      minValue: 5.0,
      maxValue: 30.0,
      digits: 2,
      exprText: "14.7/(1+x)"),
    SubaruPidDef(
      id: 'FUELBASE',
      desc: "Final Fueling Base (4-byte)*",
      unit: 'AFR',
      category: 'fuel',
      address: 0xFF72EC,
      bytesCount: 4,
      priority: 3,
      formulaText: "14.7/x",
      minValue: 5.0,
      maxValue: 30.0,
      digits: 2,
      exprText: "14.7/x"),
    SubaruPidDef(
      id: 'MAP_4B',
      desc: "Manifold Absolute Pressure (4-byte)*",
      unit: 'bar',
      category: 'air',
      address: 0xFF6ADC,
      bytesCount: 4,
      priority: 3,
      formulaText: "x*0.001333224",
      minValue: 0.0,
      maxValue: 3.0,
      digits: 3,
      exprText: "x*0.001333224"),
    SubaruPidDef(
      id: 'TBOOST_REL',
      desc: "Target Boost Relative (4-byte)*",
      unit: 'bar',
      category: 'turbo',
      address: 0xFF649C,
      bytesCount: 4,
      priority: 3,
      formulaText: "x*0.001333224",
      minValue: -2.0,
      maxValue: 2.0,
      digits: 3,
      exprText: "x*0.001333224"),
    SubaruPidDef(
      id: 'IDLE_SEL',
      desc: "Idle Speed Map Selection*",
      unit: 'raw',
      category: 'engine',
      address: 0xFF8408,
      bytesCount: 4,
      priority: 3,
      formulaText: "x",
      minValue: 0.0,
      maxValue: 16.0,
      digits: 0,
      exprText: "x"),
    SubaruPidDef(
      id: 'AFL_RANGE',
      desc: "A/F Learning Airflow Range (Current)*",
      unit: 'rng',
      category: 'fuel',
      address: 0xFF7695,
      bytesCount: 4,
      priority: 3,
      formulaText: "x+1",
      minValue: 0.0,
      maxValue: 10.0,
      digits: 0,
      exprText: "x+1"),
  ];

  static SubaruPidDef byId(String id) => all.firstWhere((p) => p.id == id);
  static const Set<String> defaults = {
    'RPM',
    'ECT',
    'TIMING',
    'MAF',
    'TPS',
    'BATT',
    'MAP_REL',
    'AFR',
  };
}
'''

# ===== Профиль A2TB100B: все адреса подтверждены по logger.xml =====
FILES["lib/identity.dart"] = r'''
// v0.11 (патч 04.2): профиль A2TB100B расширен адресами из logger-дефиниций
// RomRaider SubaruDefs Stable, отфильтрованных по ECU ID 5204584007.
// Источник подтверждён: RomRaider/logger/metric/logger.xml (GitHub, ветка Stable).

String normalizeCalId(String value) =>
    value.toUpperCase().replaceAll(RegExp(r'[^0-9A-Z]'), '');

class CalProfile {
  const CalProfile(this.calId, this.ecuId, this.label,
      this.addresses, this.parentDefs);
  final String calId, ecuId, label;
  final Map<String, String> addresses;
  final String parentDefs;
}

const List<CalProfile> calProfiles = <CalProfile>[
  CalProfile('A2TB100B', '5204584007',
      'Legacy GT (BP/BL) EJ20X JDM 2008 · TD04HL-19T · Dual AVCS · 5EAT (logger-defs verified)',
      <String, String>{
        'IAM': '0xFF2538',
        'LOAD_4B': '0xFF6C9C',
        'BOOST_ERR': '0xFF6450',
        'BOOST_TGT': '0xFF6454',
        'FBKC': '0xFF7D4C',
        'FKL': '0xFF7DD0',
        'BOOST': '0xFF6AE0',
        'CL_TARGET': '0xFF73B4',
        'REQ_TQ': '0xFF8058',
        'KC_ADV_4B': '0xFF7D48',
        'KC_IAM': '0xFF7DB0',
        'BTIMING': '0xFF7B98',
        'KCMAX': '0xFF7DA8',
        'KNOCK_SUM': '0xFF7D24',
        'AFL_A': '0xFF24B4',
        'AFL_B': '0xFF24BC',
        'AFL_C': '0xFF24C4',
        'AFL_D': '0xFF24CC',
        'AFL_4B': '0xFF768C',
        'AFCOR1_4B': '0xFF73A4',
        'TIPIN': '0xFF6B8C',
        'TTH_TGT': '0xFF8040',
        'THROTTLE_4B': '0xFF6B7C',
        'CLOL': '0xFF9679',
        'FL_OFFSET': '0xFF7DD6',
        'GEAR_CALC': '0xFF7079',
        'TDIG_INT': '0xFF645C',
        'TDIG_PROP': '0xFF6458',
        'INJPW': '0xFF7A48',
        'INJ_LAT': '0xFF7A5C',
        'OLE_ENRICH': '0xFF7778',
        'ENRICH_FINAL': '0xFF731C',
        'FUELBASE': '0xFF72EC',
        'MAP_4B': '0xFF6ADC',
        'TBOOST_REL': '0xFF649C',
        'IDLE_SEL': '0xFF8408',
        'AFL_RANGE': '0xFF7695',
      },
      'ECU defs: A2TB100B -> A2TB100K -> 32BITBASE (TD-D/SubaruDefs); '
      'logger.xml @ SubaruDefs Stable, ecu=5204584007'),
];

CalProfile? matchCalProfile(String? candidate) {
  final norm = normalizeCalId(candidate ?? '');
  if (norm.isEmpty) return null;
  for (final profile in calProfiles) {
    if (profile.calId == norm) return profile;
  }
  return null;
}
'''

# ===== Полный каталог ECU 5204584007: 91 std + 59 ext + 68 switches =====
FILES["lib/loggerdef.dart"] = r'''
// v0.11 (патч 04.2): полный каталог параметров logger-дефиниций RomRaider
// для ECU ID 5204584007 (A2TB100B). Источник: RomRaider/SubaruDefs Stable,
// RomRaider/logger/metric/logger.xml. Используется для справки/подбора PID;
// активная библиотека опроса — lib/pids.dart.

class LoggerdefParam {
  const LoggerdefParam({
    required this.pid,
    required this.name,
    required this.address,
    required this.length,
    required this.units,
    required this.expr,
  });
  final String pid, name, address;
  final int length;
  final String units, expr;
  bool get isExtended => address.toUpperCase().startsWith('0XFF');
}

class LoggerdefSwitch {
  const LoggerdefSwitch({
    required this.sid,
    required this.name,
    required this.byte,
    required this.bit,
  });
  final String sid, name, byte;
  final int bit;
}

/// Стандартные параметры SSM (байтовые адреса), поддерживаемые этим ECU.
const List<LoggerdefParam> loggerdefParams = <LoggerdefParam>[
  const LoggerdefParam(
      pid: 'P1', name: "Engine Load (Relative)",
      address: '0x000007', length: 1,
      units: "%", expr: "x*100/255"),
  const LoggerdefParam(
      pid: 'P2', name: "Coolant Temperature",
      address: '0x000008', length: 1,
      units: "C", expr: "x-40"),
  const LoggerdefParam(
      pid: 'P3', name: "A/F Correction #1",
      address: '0x000009', length: 1,
      units: "%", expr: "(x-128)*100/128"),
  const LoggerdefParam(
      pid: 'P4', name: "A/F Learning #1",
      address: '0x00000A', length: 1,
      units: "%", expr: "(x-128)*100/128"),
  const LoggerdefParam(
      pid: 'P5', name: "A/F Correction #2",
      address: '0x00000B', length: 1,
      units: "%", expr: "(x-128)*100/128"),
  const LoggerdefParam(
      pid: 'P6', name: "A/F Learning #2",
      address: '0x00000C', length: 1,
      units: "%", expr: "(x-128)*100/128"),
  const LoggerdefParam(
      pid: 'P7', name: "Manifold Absolute Pressure",
      address: '0x00000D', length: 1,
      units: "bar", expr: "x*37/255/14.50377"),
  const LoggerdefParam(
      pid: 'P8', name: "Engine Speed",
      address: '0x00000E', length: 2,
      units: "rpm", expr: "x/4"),
  const LoggerdefParam(
      pid: 'P9', name: "Vehicle Speed",
      address: '0x000010', length: 1,
      units: "kph", expr: "x"),
  const LoggerdefParam(
      pid: 'P10', name: "Ignition Total Timing",
      address: '0x000011', length: 1,
      units: "degrees", expr: "(x-128)/2"),
  const LoggerdefParam(
      pid: 'P11', name: "Intake Air Temperature",
      address: '0x000012', length: 1,
      units: "C", expr: "x-40"),
  const LoggerdefParam(
      pid: 'P12', name: "Mass Airflow",
      address: '0x000013', length: 2,
      units: "g/s", expr: "x/100"),
  const LoggerdefParam(
      pid: 'P13', name: "Throttle Opening Angle",
      address: '0x000015', length: 1,
      units: "%", expr: "x*100/255"),
  const LoggerdefParam(
      pid: 'P14', name: "Front O2 Sensor #1",
      address: '0x000016', length: 2,
      units: "V", expr: "x/200"),
  const LoggerdefParam(
      pid: 'P15', name: "Rear O2 Sensor",
      address: '0x000018', length: 2,
      units: "V", expr: "x/200"),
  const LoggerdefParam(
      pid: 'P16', name: "Front O2 Sensor #2",
      address: '0x00001A', length: 2,
      units: "V", expr: "x/200"),
  const LoggerdefParam(
      pid: 'P17', name: "Battery Voltage",
      address: '0x00001C', length: 1,
      units: "V", expr: "x*8/100"),
  const LoggerdefParam(
      pid: 'P18', name: "Mass Airflow Sensor Voltage",
      address: '0x00001D', length: 1,
      units: "V", expr: "x/50"),
  const LoggerdefParam(
      pid: 'P19', name: "Throttle Sensor Voltage",
      address: '0x00001E', length: 1,
      units: "V", expr: "x/50"),
  const LoggerdefParam(
      pid: 'P20', name: "Differential Pressure Sensor Voltage",
      address: '0x00001F', length: 1,
      units: "V", expr: "x/50"),
  const LoggerdefParam(
      pid: 'P21', name: "Fuel Injector #1 Pulse Width",
      address: '0x000020', length: 1,
      units: "ms", expr: "x*256/1000"),
  const LoggerdefParam(
      pid: 'P22', name: "Fuel Injector #2 Pulse Width",
      address: '0x000021', length: 1,
      units: "ms", expr: "x*256/1000"),
  const LoggerdefParam(
      pid: 'P23', name: "Knock Correction Advance",
      address: '0x000022', length: 1,
      units: "degrees", expr: "(x-128)/2"),
  const LoggerdefParam(
      pid: 'P24', name: "Atmospheric Pressure",
      address: '0x000023', length: 1,
      units: "bar", expr: "x*37/255/14.50377"),
  const LoggerdefParam(
      pid: 'P25', name: "Manifold Relative Pressure",
      address: '0x000024', length: 1,
      units: "bar", expr: "(x-128)*37/255/14.50377"),
  const LoggerdefParam(
      pid: 'P26', name: "Pressure Differential Sensor",
      address: '0x000025', length: 1,
      units: "bar", expr: "(x-128)*37/255/14.50377"),
  const LoggerdefParam(
      pid: 'P27', name: "Fuel Tank Pressure",
      address: '0x000026', length: 1,
      units: "bar", expr: "(x-128)*35/10000/14.50377"),
  const LoggerdefParam(
      pid: 'P28', name: "CO Adjustment",
      address: '0x000027', length: 1,
      units: "V", expr: "x/50"),
  const LoggerdefParam(
      pid: 'P29', name: "Learned Ignition Timing",
      address: '0x000028', length: 1,
      units: "degrees", expr: "(x-128)/2"),
  const LoggerdefParam(
      pid: 'P30', name: "Accelerator Pedal Angle",
      address: '0x000029', length: 1,
      units: "%", expr: "x*100/255"),
  const LoggerdefParam(
      pid: 'P31', name: "Fuel Temperature",
      address: '0x00002A', length: 1,
      units: "C", expr: "x-40"),
  const LoggerdefParam(
      pid: 'P32', name: "Front O2 Heater Current #1",
      address: '0x00002B', length: 1,
      units: "A", expr: "x*1004/25600"),
  const LoggerdefParam(
      pid: 'P33', name: "Rear O2 Heater Current",
      address: '0x00002C', length: 1,
      units: "A", expr: "x*1004/25600"),
  const LoggerdefParam(
      pid: 'P34', name: "Front O2 Heater Current #2",
      address: '0x00002D', length: 1,
      units: "A", expr: "x*1004/25600"),
  const LoggerdefParam(
      pid: 'P35', name: "Fuel Level",
      address: '0x00002E', length: 1,
      units: "V", expr: "x/50"),
  const LoggerdefParam(
      pid: 'P36', name: "Primary Wastegate Duty Cycle",
      address: '0x000030', length: 1,
      units: "%", expr: "x*100/255"),
  const LoggerdefParam(
      pid: 'P37', name: "Secondary Wastegate Duty Cycle",
      address: '0x000031', length: 1,
      units: "%", expr: "x*100/255"),
  const LoggerdefParam(
      pid: 'P38', name: "CPC Valve Duty Ratio",
      address: '0x000032', length: 1,
      units: "%", expr: "x*100/255"),
  const LoggerdefParam(
      pid: 'P39', name: "Tumble Valve Position Sensor Right",
      address: '0x000033', length: 1,
      units: "V", expr: "x/50"),
  const LoggerdefParam(
      pid: 'P40', name: "Tumble Valve Position Sensor Left",
      address: '0x000034', length: 1,
      units: "V", expr: "x/50"),
  const LoggerdefParam(
      pid: 'P41', name: "Idle Speed Control Valve Duty Ratio",
      address: '0x000035', length: 1,
      units: "%", expr: "x/2"),
  const LoggerdefParam(
      pid: 'P42', name: "A/F Lean Correction",
      address: '0x000036', length: 1,
      units: "%", expr: "x*100/255"),
  const LoggerdefParam(
      pid: 'P43', name: "A/F Heater Duty",
      address: '0x000037', length: 1,
      units: "%", expr: "x*100/255"),
  const LoggerdefParam(
      pid: 'P44', name: "Idle Speed Control Valve Step",
      address: '0x000038', length: 1,
      units: "steps", expr: "x"),
  const LoggerdefParam(
      pid: 'P45', name: "Number of Exh. Gas Recirc. Steps",
      address: '0x000039', length: 1,
      units: "steps", expr: "x"),
  const LoggerdefParam(
      pid: 'P46', name: "Alternator Duty",
      address: '0x00003A', length: 1,
      units: "%", expr: "x"),
  const LoggerdefParam(
      pid: 'P47', name: "Fuel Pump Duty",
      address: '0x00003B', length: 1,
      units: "%", expr: "x*100/255"),
  const LoggerdefParam(
      pid: 'P48', name: "Intake VVT Advance Angle Right",
      address: '0x00003C', length: 1,
      units: "degrees", expr: "x-50"),
  const LoggerdefParam(
      pid: 'P49', name: "Intake VVT Advance Angle Left",
      address: '0x00003D', length: 1,
      units: "degrees", expr: "x-50"),
  const LoggerdefParam(
      pid: 'P50', name: "Intake OCV Duty Right",
      address: '0x00003E', length: 1,
      units: "%", expr: "x*100/255"),
  const LoggerdefParam(
      pid: 'P51', name: "Intake OCV Duty Left",
      address: '0x00003F', length: 1,
      units: "%", expr: "x*100/255"),
  const LoggerdefParam(
      pid: 'P52', name: "Intake OCV Current Right",
      address: '0x000040', length: 1,
      units: "mA", expr: "x/32"),
  const LoggerdefParam(
      pid: 'P53', name: "Intake OCV Current Left",
      address: '0x000041', length: 1,
      units: "mA", expr: "x/32"),
  const LoggerdefParam(
      pid: 'P54', name: "A/F Sensor #1 Current",
      address: '0x000042', length: 1,
      units: "mA", expr: "(x-128)/8"),
  const LoggerdefParam(
      pid: 'P55', name: "A/F Sensor #2 Current",
      address: '0x000043', length: 1,
      units: "mA", expr: "(x-128)/8"),
  const LoggerdefParam(
      pid: 'P56', name: "A/F Sensor #1 Resistance",
      address: '0x000044', length: 1,
      units: "ohms", expr: "x"),
  const LoggerdefParam(
      pid: 'P57', name: "A/F Sensor #2 Resistance",
      address: '0x000045', length: 1,
      units: "ohms", expr: "x"),
  const LoggerdefParam(
      pid: 'P58', name: "A/F Sensor #1",
      address: '0x000046', length: 1,
      units: "AFR", expr: "x/128*14.7"),
  const LoggerdefParam(
      pid: 'P59', name: "A/F Sensor #2",
      address: '0x000047', length: 1,
      units: "AFR", expr: "x/128*14.7"),
  const LoggerdefParam(
      pid: 'P60', name: "Gear Position",
      address: '0x00004A', length: 1,
      units: "gear", expr: "x+1"),
  const LoggerdefParam(
      pid: 'P61', name: "A/F Sensor #1 Heater Current",
      address: '0x000053', length: 1,
      units: "A", expr: "x/10"),
  const LoggerdefParam(
      pid: 'P62', name: "A/F Sensor #2 Heater Current",
      address: '0x000054', length: 1,
      units: "A", expr: "x/10"),
  const LoggerdefParam(
      pid: 'P63', name: "Roughness Monitor Cylinder #1",
      address: '0x0000CE', length: 1,
      units: "misfire count", expr: "x"),
  const LoggerdefParam(
      pid: 'P64', name: "Roughness Monitor Cylinder #2",
      address: '0x0000CF', length: 1,
      units: "misfire count", expr: "x"),
  const LoggerdefParam(
      pid: 'P65', name: "A/F Correction #3 (16-bit ECU)",
      address: '0x0000D0', length: 1,
      units: "%", expr: "(x-128)*100/128"),
  const LoggerdefParam(
      pid: 'P66', name: "A/F Learning #3",
      address: '0x0000D1', length: 1,
      units: "%", expr: "(x-128)*100/128"),
  const LoggerdefParam(
      pid: 'P67', name: "Rear O2 Heater Voltage",
      address: '0x0000D2', length: 1,
      units: "V", expr: "x/50"),
  const LoggerdefParam(
      pid: 'P68', name: "A/F Adjustment Voltage",
      address: '0x0000D3', length: 1,
      units: "V", expr: "x/50"),
  const LoggerdefParam(
      pid: 'P69', name: "Roughness Monitor Cylinder #3",
      address: '0x0000D8', length: 1,
      units: "misfire count", expr: "x"),
  const LoggerdefParam(
      pid: 'P70', name: "Roughness Monitor Cylinder #4",
      address: '0x0000D9', length: 1,
      units: "misfire count", expr: "x"),
  const LoggerdefParam(
      pid: 'P71', name: "Throttle Motor Duty",
      address: '0x0000FA', length: 1,
      units: "%", expr: "(x-128)*100/128"),
  const LoggerdefParam(
      pid: 'P72', name: "Throttle Motor Voltage",
      address: '0x0000FB', length: 1,
      units: "V", expr: "x*8/100"),
  const LoggerdefParam(
      pid: 'P73', name: "Sub Throttle Sensor",
      address: '0x000100', length: 1,
      units: "V", expr: "x/50"),
  const LoggerdefParam(
      pid: 'P74', name: "Main Throttle Sensor",
      address: '0x000101', length: 1,
      units: "V", expr: "x/50"),
  const LoggerdefParam(
      pid: 'P75', name: "Sub Accelerator Sensor",
      address: '0x000102', length: 1,
      units: "V", expr: "x/50"),
  const LoggerdefParam(
      pid: 'P76', name: "Main Accelerator Sensor",
      address: '0x000103', length: 1,
      units: "V", expr: "x/50"),
  const LoggerdefParam(
      pid: 'P77', name: "Brake Booster Pressure",
      address: '0x000104', length: 1,
      units: "bar", expr: "x*37/255/14.50377"),
  const LoggerdefParam(
      pid: 'P78', name: "Fuel Pressure (High)",
      address: '0x000105', length: 1,
      units: "bar", expr: "x/25*10"),
  const LoggerdefParam(
      pid: 'P79', name: "Exhaust Gas Temperature",
      address: '0x000106', length: 1,
      units: "C", expr: "(x+40)*5"),
  const LoggerdefParam(
      pid: 'P80', name: "Cold Start Injector (Air Pump)",
      address: '0x000108', length: 1,
      units: "ms", expr: "x*256/1000"),
  const LoggerdefParam(
      pid: 'P81', name: "SCV Step",
      address: '0x000109', length: 1,
      units: "steps", expr: "x"),
  const LoggerdefParam(
      pid: 'P82', name: "Memorised Cruise Speed",
      address: '0x00010A', length: 1,
      units: "kph", expr: "x"),
  const LoggerdefParam(
      pid: 'P83', name: "Exhaust VVT Advance Angle Right",
      address: '0x000118', length: 1,
      units: "degrees", expr: "x-50"),
  const LoggerdefParam(
      pid: 'P84', name: "Exhaust VVT Advance Angle Left",
      address: '0x000119', length: 1,
      units: "degrees", expr: "x-50"),
  const LoggerdefParam(
      pid: 'P85', name: "Exhaust OCV Duty Right",
      address: '0x00011A', length: 1,
      units: "%", expr: "x*100/255"),
  const LoggerdefParam(
      pid: 'P86', name: "Exhaust OCV Duty Left",
      address: '0x00011B', length: 1,
      units: "%", expr: "x*100/255"),
  const LoggerdefParam(
      pid: 'P87', name: "Exhaust OCV Current Right",
      address: '0x00011C', length: 1,
      units: "mA", expr: "x*32"),
  const LoggerdefParam(
      pid: 'P88', name: "Exhaust OCV Current Left",
      address: '0x00011D', length: 1,
      units: "mA", expr: "x*32"),
  const LoggerdefParam(
      pid: 'P89', name: "A/F Correction #3 (32-bit ECU)",
      address: '0x0000D0', length: 1,
      units: "%", expr: "(x*.078125)-5"),
  const LoggerdefParam(
      pid: 'P90', name: "IAM",
      address: '0x0000F9', length: 1,
      units: "multiplier", expr: "x/16"),
  const LoggerdefParam(
      pid: 'P91', name: "Fine Learning Knock Correction",
      address: '0x000199', length: 1,
      units: "degrees", expr: "(x*0.25)-32"),
];

/// Расширенные параметры (RAM float16/32, адрес персонален для 5204584007).
const List<LoggerdefParam> loggerdefExt = <LoggerdefParam>[
  const LoggerdefParam(
      pid: 'E31', name: "IAM (4-byte)*",
      address: '0xFF2538', length: 4,
      units: "multiplier", expr: "x"),
  const LoggerdefParam(
      pid: 'E32', name: "Engine Load (4-Byte)*",
      address: '0xFF6C9C', length: 4,
      units: "g/rev", expr: "x"),
  const LoggerdefParam(
      pid: 'E33', name: "CL/OL Fueling*",
      address: '0xFF9679', length: 4,
      units: "status", expr: "x+6"),
  const LoggerdefParam(
      pid: 'E34', name: "Turbo Dynamics Integral (4-byte)*",
      address: '0xFF645C', length: 4,
      units: "absolute %", expr: "x"),
  const LoggerdefParam(
      pid: 'E35', name: "Boost Error*",
      address: '0xFF6450', length: 4,
      units: "bar", expr: "x*0.001333224"),
  const LoggerdefParam(
      pid: 'E36', name: "Target Boost (4-byte)*",
      address: '0xFF6454', length: 4,
      units: "bar absolute", expr: "x*0.001333224"),
  const LoggerdefParam(
      pid: 'E37', name: "Turbo Dynamics Proportional (4-byte)*",
      address: '0xFF6458', length: 4,
      units: "absolute %", expr: "x"),
  const LoggerdefParam(
      pid: 'E38', name: "Throttle Plate Opening Angle (4-byte)*",
      address: '0xFF6B7C', length: 4,
      units: "%", expr: "x/.84"),
  const LoggerdefParam(
      pid: 'E39', name: "Feedback Knock Correction (4-byte)*",
      address: '0xFF7D4C', length: 4,
      units: "degrees", expr: "x"),
  const LoggerdefParam(
      pid: 'E40', name: "Knock Correction Advance (IAM only)*",
      address: '0xFF7DB0', length: 4,
      units: "degrees", expr: "x"),
  const LoggerdefParam(
      pid: 'E41', name: "Fine Learning Knock Correction (4-byte)*",
      address: '0xFF7DD0', length: 4,
      units: "degrees", expr: "x"),
  const LoggerdefParam(
      pid: 'E43', name: "Knock Correction Advance (4-byte)*",
      address: '0xFF7D48', length: 4,
      units: "degrees", expr: "x"),
  const LoggerdefParam(
      pid: 'E44', name: "A/F Learning #1 A (Stored)*",
      address: '0xFF24B4', length: 4,
      units: "%", expr: "x*100"),
  const LoggerdefParam(
      pid: 'E45', name: "A/F Learning #1 B (Stored)*",
      address: '0xFF24BC', length: 4,
      units: "%", expr: "x*100"),
  const LoggerdefParam(
      pid: 'E46', name: "A/F Learning #1 C (Stored)*",
      address: '0xFF24C4', length: 4,
      units: "%", expr: "x*100"),
  const LoggerdefParam(
      pid: 'E47', name: "A/F Learning #1 D (Stored)*",
      address: '0xFF24CC', length: 4,
      units: "%", expr: "x*100"),
  const LoggerdefParam(
      pid: 'E48', name: "A/F Learning #1 (4-byte)*",
      address: '0xFF768C', length: 4,
      units: "%", expr: "x*100"),
  const LoggerdefParam(
      pid: 'E49', name: "Idle Speed Map Selection*",
      address: '0xFF8408', length: 4,
      units: "raw ecu value", expr: "x"),
  const LoggerdefParam(
      pid: 'E50', name: "Fuel Injector #1 Latency (4-byte)*",
      address: '0xFF7A5C', length: 4,
      units: "ms", expr: "x*.001"),
  const LoggerdefParam(
      pid: 'E51', name: "Manifold Absolute Pressure (4-byte)*",
      address: '0xFF6ADC', length: 4,
      units: "bar absolute", expr: "x*0.001333224"),
  const LoggerdefParam(
      pid: 'E52', name: "Manifold Relative Sea Level Pressure (4-byte)*",
      address: '0xFF6ADC', length: 4,
      units: "bar relative sea level", expr: "(x-760)*0.001333224"),
  const LoggerdefParam(
      pid: 'E53', name: "Ignition Base Timing*",
      address: '0xFF7B98', length: 4,
      units: "degrees", expr: "x"),
  const LoggerdefParam(
      pid: 'E54', name: "Tip-in Throttle*",
      address: '0xFF6B8C', length: 4,
      units: "%", expr: "x"),
  const LoggerdefParam(
      pid: 'E55', name: "Tip-in Enrichment (Last Calculated)*",
      address: '0xFF7924', length: 4,
      units: "raw ecu value", expr: "x"),
  const LoggerdefParam(
      pid: 'E56', name: "Requested Torque*",
      address: '0xFF8058', length: 4,
      units: "raw ecu value", expr: "x"),
  const LoggerdefParam(
      pid: 'E57', name: "Target Throttle Plate Position*",
      address: '0xFF8040', length: 4,
      units: "%", expr: "x/.84"),
  const LoggerdefParam(
      pid: 'E58', name: "Fine Learning Table Offset*",
      address: '0xFF7DD6', length: 4,
      units: "index position", expr: "x+1"),
  const LoggerdefParam(
      pid: 'E59', name: "Gear (Calculated)*",
      address: '0xFF7079', length: 4,
      units: "position", expr: "x"),
  const LoggerdefParam(
      pid: 'E60', name: "Fuel Injector #1 Pulse Width (4-byte)*",
      address: '0xFF7A48', length: 4,
      units: "ms", expr: "x*.001"),
  const LoggerdefParam(
      pid: 'E61', name: "A/F Learning Airflow Range (Current)*",
      address: '0xFF7695', length: 4,
      units: "offset", expr: "x+1"),
  const LoggerdefParam(
      pid: 'E70', name: "Primary Wastegate Duty Maximum* (4-byte)*",
      address: '0xFF6478', length: 4,
      units: "%", expr: "x"),
  const LoggerdefParam(
      pid: 'E77', name: "Primary Wastegate Duty Maximum* (4-byte)*",
      address: '0xFF6478', length: 4,
      units: "%", expr: "x"),
  const LoggerdefParam(
      pid: 'E78', name: "Primary Wastegate Duty Maximum* (2-byte)**",
      address: '0xFF708C', length: 2,
      units: "%", expr: "x*.00390625"),
  const LoggerdefParam(
      pid: 'E79', name: "Turbo Dynamics Integral (2-byte)**",
      address: '0xFF708E', length: 2,
      units: "absolute %", expr: "(x*.00390625)-50"),
  const LoggerdefParam(
      pid: 'E80', name: "Turbo Dynamics Proportional (2-byte)**",
      address: '0xFF7090', length: 2,
      units: "absolute %", expr: "(x*.00390625)-50"),
  const LoggerdefParam(
      pid: 'E81', name: "A/F Correction #1 (4-byte)*",
      address: '0xFF73A4', length: 4,
      units: "%", expr: "(x*100)-100"),
  const LoggerdefParam(
      pid: 'E82', name: "A/F Correction #1 (2-byte)**",
      address: '0xFF7094', length: 2,
      units: "%", expr: "(x*.01220703)-100"),
  const LoggerdefParam(
      pid: 'E83', name: "A/F Learning #1 (2-byte)**",
      address: '0xFF7098', length: 2,
      units: "%", expr: "(x*.04882812)-50"),
  const LoggerdefParam(
      pid: 'E84', name: "Primary Open Loop Map Enrichment (4-byte)*",
      address: '0xFF7778', length: 4,
      units: "estimated AFR", expr: "14.7/(1+x)"),
  const LoggerdefParam(
      pid: 'E85', name: "Primary Open Loop Map Enrichment (2-byte)**",
      address: '0xFF709A', length: 2,
      units: "estimated AFR", expr: "14.7/(1+(x*.00003051758))"),
  const LoggerdefParam(
      pid: 'E87', name: "Engine Load (2-byte)**",
      address: '0xFF70A0', length: 2,
      units: "g/rev", expr: "x*.00006103516"),
  const LoggerdefParam(
      pid: 'E88', name: "Manifold Absolute Pressure (2-byte)**",
      address: '0xFF70AE', length: 2,
      units: "bar absolute", expr: "x*0.001333224"),
  const LoggerdefParam(
      pid: 'E89', name: "Manifold Relative Sea Level Pressure (2-byte)**",
      address: '0xFF70AE', length: 2,
      units: "bar relative sea level", expr: "(x-760)*0.001333224"),
  const LoggerdefParam(
      pid: 'E90', name: "Target Boost (2-byte)**",
      address: '0xFF70B2', length: 2,
      units: "bar absolute", expr: "x*0.001333224"),
  const LoggerdefParam(
      pid: 'E91', name: "A/F Sensor #1 (4-byte)*",
      address: '0xFF6DD4', length: 4,
      units: "estimated AFR", expr: "x*14.7"),
  const LoggerdefParam(
      pid: 'E92', name: "A/F Sensor #1 (2-byte)**",
      address: '0xFF70B4', length: 2,
      units: "estimated AFR", expr: "(x*.0001220703)*14.7"),
  const LoggerdefParam(
      pid: 'E93', name: "Throttle Plate Opening Angle (2-byte)**",
      address: '0xFF70BC', length: 2,
      units: "%", expr: "x*0.0022706535"),
  const LoggerdefParam(
      pid: 'E94', name: "Feedback Knock Correction (1-byte)**",
      address: '0xFF70C6', length: 4,
      units: "degrees", expr: "(x*.3515625)-45"),
  const LoggerdefParam(
      pid: 'E95', name: "Fine Learning Knock Correction (1-byte)**",
      address: '0xFF70C9', length: 4,
      units: "degrees", expr: "(x*.3515625)-45"),
  const LoggerdefParam(
      pid: 'E96', name: "IAM (1-byte)**",
      address: '0xFF70CB', length: 4,
      units: "multiplier", expr: "x*.0625"),
  const LoggerdefParam(
      pid: 'E113', name: "Manifold Relative Pressure (4-byte)*",
      address: '0xFF6AE0', length: 4,
      units: "bar relative", expr: "x*0.001333224"),
  const LoggerdefParam(
      pid: 'E114', name: "Knock Sum*",
      address: '0xFF7D24', length: 4,
      units: "count", expr: "x"),
  const LoggerdefParam(
      pid: 'E115', name: "Primary Enrichment Final (4-byte)*",
      address: '0xFF731C', length: 4,
      units: "estimated AFR", expr: "14.7/(1+x)"),
  const LoggerdefParam(
      pid: 'E118', name: "Knock Correction Advance Max Primary*",
      address: '0xFF7DA8', length: 4,
      units: "degrees", expr: "x"),
  const LoggerdefParam(
      pid: 'E120', name: "Target Boost Relative (4-byte)*",
      address: '0xFF649C', length: 4,
      units: "bar relative", expr: "x*0.001333224"),
  const LoggerdefParam(
      pid: 'E121', name: "Closed Loop Fueling Target (4-byte)*",
      address: '0xFF73B4', length: 4,
      units: "estimated AFR", expr: "x*14.7"),
  const LoggerdefParam(
      pid: 'E122', name: "Closed Loop Fueling Target (2-byte)*",
      address: '0xFF70BA', length: 2,
      units: "estimated AFR", expr: "x*.001794433"),
  const LoggerdefParam(
      pid: 'E123', name: "Final Fueling Base (4-byte)*",
      address: '0xFF72EC', length: 4,
      units: "estimated AFR", expr: "14.7/x"),
  const LoggerdefParam(
      pid: 'E124', name: "Final Fueling Base (2-byte)*",
      address: '0xFF709E', length: 2,
      units: "estimated AFR", expr: "14.7/(x*.0004882812)"),
];

/// Битовые флаги состояния (читаются байтом по адресу + маска).
const List<LoggerdefSwitch> loggerdefSwitches = <LoggerdefSwitch>[
  const LoggerdefSwitch(
      sid: 'S1', name: "AT Vehicle ID",
      byte: '0x000061', bit: 6),
  const LoggerdefSwitch(
      sid: 'S2', name: "Test Mode Connector",
      byte: '0x000061', bit: 5),
  const LoggerdefSwitch(
      sid: 'S3', name: "Read Memory Connector",
      byte: '0x000061', bit: 4),
  const LoggerdefSwitch(
      sid: 'S4', name: "Neutral Position Switch",
      byte: '0x000062', bit: 7),
  const LoggerdefSwitch(
      sid: 'S5', name: "Idle Switch",
      byte: '0x000062', bit: 6),
  const LoggerdefSwitch(
      sid: 'S6', name: "Intercooler AutoWash Switch",
      byte: '0x000062', bit: 4),
  const LoggerdefSwitch(
      sid: 'S7', name: "Ignition Switch",
      byte: '0x000062', bit: 3),
  const LoggerdefSwitch(
      sid: 'S8', name: "Power Steering Switch",
      byte: '0x000062', bit: 2),
  const LoggerdefSwitch(
      sid: 'S9', name: "Air Conditioning Switch",
      byte: '0x000062', bit: 1),
  const LoggerdefSwitch(
      sid: 'S10', name: "Handle Switch",
      byte: '0x000063', bit: 7),
  const LoggerdefSwitch(
      sid: 'S11', name: "Starter Switch",
      byte: '0x000063', bit: 6),
  const LoggerdefSwitch(
      sid: 'S12', name: "Front O2 Rich Signal",
      byte: '0x000063', bit: 5),
  const LoggerdefSwitch(
      sid: 'S13', name: "Rear O2 Rich Signal",
      byte: '0x000063', bit: 4),
  const LoggerdefSwitch(
      sid: 'S14', name: "Front O2 #2 Rich Signal",
      byte: '0x000063', bit: 3),
  const LoggerdefSwitch(
      sid: 'S15', name: "Knock Signal 1",
      byte: '0x000063', bit: 2),
  const LoggerdefSwitch(
      sid: 'S16', name: "Knock Signal 2",
      byte: '0x000063', bit: 1),
  const LoggerdefSwitch(
      sid: 'S17', name: "Electrical Load Signal",
      byte: '0x000063', bit: 0),
  const LoggerdefSwitch(
      sid: 'S18', name: "Crank Position Sensor",
      byte: '0x000064', bit: 7),
  const LoggerdefSwitch(
      sid: 'S19', name: "Cam Position Sensor",
      byte: '0x000064', bit: 6),
  const LoggerdefSwitch(
      sid: 'S20', name: "Defogger Switch",
      byte: '0x000064', bit: 5),
  const LoggerdefSwitch(
      sid: 'S21', name: "Blower Switch",
      byte: '0x000064', bit: 4),
  const LoggerdefSwitch(
      sid: 'S22', name: "Interior Light Switch",
      byte: '0x000064', bit: 3),
  const LoggerdefSwitch(
      sid: 'S23', name: "Wiper Switch",
      byte: '0x000064', bit: 2),
  const LoggerdefSwitch(
      sid: 'S24', name: "Air-Con Lock Signal",
      byte: '0x000064', bit: 1),
  const LoggerdefSwitch(
      sid: 'S25', name: "Air-Con Mid Pressure Switch",
      byte: '0x000064', bit: 0),
  const LoggerdefSwitch(
      sid: 'S26', name: "Air-Con Compressor Signal",
      byte: '0x000065', bit: 7),
  const LoggerdefSwitch(
      sid: 'S27', name: "Radiator Fan Relay #3",
      byte: '0x000065', bit: 6),
  const LoggerdefSwitch(
      sid: 'S28', name: "Radiator Fan Relay #1",
      byte: '0x000065', bit: 5),
  const LoggerdefSwitch(
      sid: 'S29', name: "Radiator Fan Relay #2",
      byte: '0x000065', bit: 4),
  const LoggerdefSwitch(
      sid: 'S30', name: "Fuel Pump Relay",
      byte: '0x000065', bit: 3),
  const LoggerdefSwitch(
      sid: 'S31', name: "Intercooler Auto-Wash Relay",
      byte: '0x000065', bit: 2),
  const LoggerdefSwitch(
      sid: 'S32', name: "CPC Solenoid Valve",
      byte: '0x000065', bit: 1),
  const LoggerdefSwitch(
      sid: 'S33', name: "Blow-By Leak Connector",
      byte: '0x000065', bit: 0),
  const LoggerdefSwitch(
      sid: 'S34', name: "PCV Solenoid Valve",
      byte: '0x000066', bit: 7),
  const LoggerdefSwitch(
      sid: 'S35', name: "TGV Output",
      byte: '0x000066', bit: 6),
  const LoggerdefSwitch(
      sid: 'S36', name: "TGV Drive",
      byte: '0x000066', bit: 5),
  const LoggerdefSwitch(
      sid: 'S37', name: "Variable Intake Air Solenoid",
      byte: '0x000066', bit: 4),
  const LoggerdefSwitch(
      sid: 'S38', name: "Pressure Sources Change",
      byte: '0x000066', bit: 3),
  const LoggerdefSwitch(
      sid: 'S39', name: "Vent Solenoid Valve",
      byte: '0x000066', bit: 2),
  const LoggerdefSwitch(
      sid: 'S40', name: "P/S Solenoid Valve",
      byte: '0x000066', bit: 1),
  const LoggerdefSwitch(
      sid: 'S41', name: "Assist Air Solenoid Valve",
      byte: '0x000066', bit: 0),
  const LoggerdefSwitch(
      sid: 'S42', name: "Tank Sensor Control Valve",
      byte: '0x000067', bit: 7),
  const LoggerdefSwitch(
      sid: 'S43', name: "Relief Valve Solenoid 1",
      byte: '0x000067', bit: 6),
  const LoggerdefSwitch(
      sid: 'S44', name: "Relief Valve Solenoid 2",
      byte: '0x000067', bit: 5),
  const LoggerdefSwitch(
      sid: 'S45', name: "TCS Relief Valve Solenoid",
      byte: '0x000067', bit: 4),
  const LoggerdefSwitch(
      sid: 'S46', name: "Ex. Gas Positive Pressure",
      byte: '0x000067', bit: 3),
  const LoggerdefSwitch(
      sid: 'S47', name: "Ex. Gas Negative Pressure",
      byte: '0x000067', bit: 2),
  const LoggerdefSwitch(
      sid: 'S48', name: "Intake Air Solenoid",
      byte: '0x000067', bit: 1),
  const LoggerdefSwitch(
      sid: 'S49', name: "Muffler Control",
      byte: '0x000067', bit: 0),
  const LoggerdefSwitch(
      sid: 'S50', name: "Retard Signal from AT",
      byte: '0x000068', bit: 3),
  const LoggerdefSwitch(
      sid: 'S51', name: "Fuel Cut Signal from AT",
      byte: '0x000068', bit: 2),
  const LoggerdefSwitch(
      sid: 'S52', name: "Ban of Torque Down",
      byte: '0x000068', bit: 1),
  const LoggerdefSwitch(
      sid: 'S53', name: "Request Torque Down VDC",
      byte: '0x000068', bit: 0),
  const LoggerdefSwitch(
      sid: 'S54', name: "Torque Control Signal #1",
      byte: '0x000069', bit: 7),
  const LoggerdefSwitch(
      sid: 'S55', name: "Torque Control Signal #2",
      byte: '0x000069', bit: 6),
  const LoggerdefSwitch(
      sid: 'S56', name: "Torque Permission Signal",
      byte: '0x000069', bit: 5),
  const LoggerdefSwitch(
      sid: 'S57', name: "EAM Signal",
      byte: '0x000069', bit: 4),
  const LoggerdefSwitch(
      sid: 'S58', name: "AT coop. lock up signal",
      byte: '0x000069', bit: 3),
  const LoggerdefSwitch(
      sid: 'S59', name: "AT coop. lean burn signal",
      byte: '0x000069', bit: 2),
  const LoggerdefSwitch(
      sid: 'S60', name: "AT coop. rich spike signal",
      byte: '0x000069', bit: 1),
  const LoggerdefSwitch(
      sid: 'S61', name: "AET Signal",
      byte: '0x000069', bit: 0),
  const LoggerdefSwitch(
      sid: 'S62', name: "ETC Motor Relay",
      byte: '0x000120', bit: 6),
  const LoggerdefSwitch(
      sid: 'S63', name: "Clutch Switch",
      byte: '0x000121', bit: 7),
  const LoggerdefSwitch(
      sid: 'S64', name: "Stop Light Switch",
      byte: '0x000121', bit: 6),
  const LoggerdefSwitch(
      sid: 'S65', name: "Set/Coast Switch",
      byte: '0x000121', bit: 5),
  const LoggerdefSwitch(
      sid: 'S66', name: "Resume/Accelerate Switch",
      byte: '0x000121', bit: 4),
  const LoggerdefSwitch(
      sid: 'S67', name: "Brake Switch",
      byte: '0x000121', bit: 3),
  const LoggerdefSwitch(
      sid: 'S68', name: "Accelerator Switch",
      byte: '0x000121', bit: 1),
];
'''

# ===== Справочник кодов неисправностей Subaru (RU) =====
FILES["lib/dtc_dict.dart"] = r'''
// v0.11 (патч 04.2): справочник кодов неисправностей Subaru.
// code -> (название RU, система). Используется страницей DTC.

const Map<String, (String, String)> kSubaruDtcDict = <String, (String, String)>{
  'P0030': ("Цепь подогрева O2-датчика B1S1", 'engine'),
  'P0031': ("Низкий уровень подогрева O2 B1S1", 'engine'),
  'P0032': ("Высокий уровень подогрева O2 B1S1", 'engine'),
  'P0037': ("Цепь подогрева O2 B1S2", 'engine'),
  'P0110': ("Датчик температуры впускного воздуха (IAT)", 'engine'),
  'P0115': ("Датчик температуры ОЖ (ECT)", 'engine'),
  'P0120': ("Датчик положения дросселя A", 'engine'),
  'P0130': ("Цепь O2-датчика B1S1", 'engine'),
  'P0171': ("Слишком бедная смесь (банк 1)", 'engine'),
  'P0172': ("Слишком богатая смесь (банк 1)", 'engine'),
  'P0183': ("Датчик температуры топлива А", 'engine'),
  'P0244': ("Wastegate: давление наддува выше лимита", 'engine'),
  'P0245': ("Цепь соленоида Wastegate (низкий уровень)", 'engine'),
  'P0246': ("Цепь соленоида Wastegate (высокий уровень)", 'engine'),
  'P0300': ("Случайные/множественные пропуски зажигания", 'engine'),
  'P0301': ("Пропуски зажигания, цилиндр 1", 'engine'),
  'P0303': ("Пропуски зажигания, цилиндр 3", 'engine'),
  'P0325': ("Цепь датчика детонации №1", 'engine'),
  'P0328': ("Высокий уровень сигнала датчика детонации", 'engine'),
  'P0335': ("Датчик положения коленвала", 'engine'),
  'P0340': ("Датчик положения распредвала", 'engine'),
  'P0420': ("Эффективность катализатора ниже порога B1", 'engine'),
  'P0441': ("Система продувки адсорбера: неверный поток", 'engine'),
  'P0455': ("EVAP: большая утечка", 'engine'),
  'P0500': ("Датчик скорости автомобиля (VSS)", 'engine'),
  'P0600': ("Ошибка последовательной связи", 'engine'),
  'P0604': ("Внутренняя ошибка RAM ЭБУ", 'engine'),
  'P0700': ("Неисправность системы управления АКПП", 'trans'),
  'P0715': ("Датчик частоты входного вала АКПП", 'trans'),
  'P0720': ("Датчик частоты выходного вала АКПП", 'trans'),
  'P0731': ("Неверное передаточное число 1-й передачи", 'trans'),
  'P0741': ("Гидротрансформатор: блокировка не включается", 'trans'),
  'P0810': ("Ошибка управления сцеплением", 'trans'),
  'P0841': ("Датчик давления масла АКПП A", 'trans'),
  'P0851': ("Цепь нейтрали (низкий уровень)", 'trans'),
  'P1700': ("Датчик положения дросселя (сигнал в TCU)", 'trans'),
  'P1710': ("Соленоид №2 АКПП", 'trans'),
  'C0057': ("VDC: ошибка связи с ЭБУ двигателя", 'chassis'),
  'C0071': ("Датчик угла поворота руля", 'chassis'),
  'B1771': ("SRS: цепь водительской подушки", 'body'),
};
'''

# ===== DiagSession: SSM2 0x18/0x14, OBD 03/04, UDS 19/14 + декодеры =====
FILES["lib/diag.dart"] = r'''
import 'dtc_dict.dart';
import 'elm.dart';
import 'protocol.dart';

/// Поколение протокола ECU.
enum DiagProto { ssm2, obd, uds }

/// Блок управления Subaru на шине CAN.
class EcuSlot {
  const EcuSlot(this.name, this.header, this.responseId, this.proto);
  final String name;
  final String header; // ATSH
  final String responseId; // ATCRA
  final DiagProto proto;

  static const engine = EcuSlot('Двигатель (ECM)', '7E0', '7E8', DiagProto.ssm2);
  static const tcu = EcuSlot('АКПП / CVT (TCM)', '7E1', '7E9', DiagProto.ssm2);
  static const vdc = EcuSlot('ABS / VDC', '7B0', '7B8', DiagProto.uds);
  static const srs = EcuSlot('Подушки (SRS)', '772', '77A', DiagProto.uds);
  static const eps = EcuSlot('ЭУР (EPS)', '7A6', '7AE', DiagProto.uds);
  static const biu = EcuSlot('Кузовной блок (BIU)', '723', '72B', DiagProto.uds);

  static const List<EcuSlot> all = [engine, tcu, vdc, srs, eps, biu];
}

/// Одна запись неисправности.
class DiagDtc {
  DiagDtc(this.code, this.statusByte, this.proto);
  final String code; // P0110, C0057...
  final int statusByte;
  final DiagProto proto;

  /// SSM2 (наследие FreeSSM): bit5 (0x20) — активна сейчас; bit7 (0x80) — MIL.
  /// UDS 19 02: bit0 testFailed, bit3 confirmedDTC, bit6 warningIndicator.
  bool get active => proto == DiagProto.uds
      ? (statusByte & 0x01) != 0
      : proto == DiagProto.obd
          ? true
          : (statusByte & 0x20) != 0;
  bool get confirmed => proto == DiagProto.uds ? (statusByte & 0x08) != 0 : true;
  bool get milOn => proto == DiagProto.uds
      ? (statusByte & 0x40) != 0
      : proto == DiagProto.obd
          ? false
          : (statusByte & 0x80) != 0;

  String get statusText {
    if (proto == DiagProto.uds) {
      final f = <String>[];
      if (statusByte & 0x01 != 0) f.add('есть сейчас');
      if (statusByte & 0x08 != 0) f.add('подтверждена');
      if (statusByte & 0x40 != 0) f.add('MIL');
      return f.isEmpty
          ? 'пассивная (0x${hex2(statusByte)})'
          : f.join(' · ');
    }
    if (proto == DiagProto.obd) return 'сохранённая (mode 03)';
    return active ? (milOn ? 'активная · MIL' : 'активная') : 'сохранённая';
  }

  /// Название из справочника Subaru ( если код известен).
  String get titleRu => kSubaruDtcDict[code]?.$1 ?? '';
}

/// Декодер двухбайтового DTC (общий для SSM2/OBD/UDS).
String decodeDtcBytes(int b1, int b2) {
  const systems = ['P', 'C', 'B', 'U'];
  final sys = systems[(b1 >> 6) & 0x03];
  final d1 = (b1 >> 4) & 0x03;
  String h(int v) => v.toRadixString(16).toUpperCase();
  return '$sys$d1${h(b1 & 0x0F)}${h((b2 >> 4) & 0x0F)}${h(b2 & 0x0F)}';
}

/// SSM2: 58 <count> <b1> <b2> <status> × n
List<DiagDtc> parseSsm2DtcReply(List<int> bytes) {
  final out = <DiagDtc>[];
  if (bytes.isEmpty || bytes[0] != 0x58) return out;
  final count = bytes.length > 1 ? bytes[1] : 0;
  for (var i = 0; i < count; i++) {
    final base = 2 + i * 3;
    if (base + 2 >= bytes.length) break;
    final b1 = bytes[base], b2 = bytes[base + 1], st = bytes[base + 2];
    if (b1 == 0 && b2 == 0) continue;
    out.add(DiagDtc(decodeDtcBytes(b1, b2), st, DiagProto.ssm2));
  }
  return out;
}

/// OBD mode 03: 43 <b1> <b2> × n
List<DiagDtc> parseObdMode3Reply(List<int> bytes) {
  final out = <DiagDtc>[];
  if (bytes.isEmpty || bytes[0] != 0x43) return out;
  for (var i = 1; i + 1 < bytes.length; i += 2) {
    final b1 = bytes[i], b2 = bytes[i + 1];
    if (b1 == 0 && b2 == 0) continue;
    out.add(DiagDtc(decodeDtcBytes(b1, b2), 0x20, DiagProto.obd));
  }
  return out;
}

/// UDS 19 02: 59 02 <availability-mask> <b1> <b2> <status> × n
List<DiagDtc> parseUdsDtcReply(List<int> bytes) {
  final out = <DiagDtc>[];
  if (bytes.length < 3 || bytes[0] != 0x59) return out;
  for (var i = 3; i + 2 < bytes.length; i += 3) {
    final b1 = bytes[i], b2 = bytes[i + 1], st = bytes[i + 2];
    if (b1 == 0 && b2 == 0) continue;
    out.add(DiagDtc(decodeDtcBytes(b1, b2), st, DiagProto.uds));
  }
  return out;
}

/// Универсальный парсер «сырого» ответа ELM (строки hex-токенов).
List<int>? parseDiagPayload(String text) {
  final flat = <int>[];
  for (final rawLine in text.split(RegExp(r'[\r\n]+'))) {
    var line = rawLine.trim();
    if (line.isEmpty) continue;
    if (RegExp(r'^\d+:').hasMatch(line)) {
      line = line.substring(line.indexOf(':') + 1).trim();
    }
    for (final t in line.split(RegExp(r'\s+')).where((x) => x.isNotEmpty)) {
      if (!RegExp(r'^[0-9A-Fa-f]{2,16}$').hasMatch(t) || t.length % 2 != 0) {
        return null;
      }
      for (var i = 0; i + 1 < t.length; i += 2) {
        flat.add(int.parse(t.substring(i, i + 2), radix: 16));
      }
    }
  }
  if (flat.isEmpty) return null;
  var b = flat;
  // single-frame PCI (старший нибл 0, длина 1..7)
  if (b.length > 1 && (b[0] & 0xF0) == 0 && (b[0] & 0x0F) <= 7) {
    final len = b[0] & 0x0F;
    b = b.sublist(1, 1 + len <= b.length ? 1 + len : b.length);
  }
  // CAN ID 0x7E8 при ATH1
  if (b.length >= 3 && b[0] == 0x07 && (b[1] & 0xF8) == 0xE8) {
    b = b.sublist(2);
  }
  return b.isEmpty ? null : b;
}

const Map<int, String> kNrMeanings = <int, String>{
  0x10: 'generalReject',
  0x11: 'serviceNotSupported',
  0x12: 'subFunctionNotSupported',
  0x21: 'busyRepeatRequest',
  0x22: 'conditionsNotCorrect',
  0x31: 'requestOutOfRange',
  0x33: 'securityAccessDenied',
  0x78: 'responsePending',
};

/// Сессия диагностики поверх ElmDriver.diagnostic (штатный whitelist+lock).
class DiagSession {
  DiagSession(this.elm, {this.experimental = false});
  final ElmDriver elm;
  final bool experimental;

  Future<String> _atEcu(EcuSlot slot) async {
    await elm.diagnostic('ATSH ${slot.header}');
    await elm.diagnostic('ATCRA ${slot.responseId}');
    await Future<void>.delayed(const Duration(milliseconds: 20));
    return slot.name;
  }

  Future<List<int>?> _payload(String command) async {
    final reply = await elm.diagnostic(command);
    return parseDiagPayload(reply);
  }

  Future<List<DiagDtc>> readDtc(EcuSlot slot) async {
    await _atEcu(slot);
    if (slot.proto == DiagProto.ssm2) {
      try {
        final bytes = await _payload('18 00 FF 00') ?? const <int>[];
        if (bytes.isNotEmpty && bytes[0] == 0x58) {
          return parseSsm2DtcReply(bytes);
        }
      } catch (_) {}
      final bytes = await _payload('03') ?? const <int>[];
      return parseObdMode3Reply(bytes);
    }
    final bytes = await _payload('19 02 AF') ?? const <int>[];
    if (bytes.isNotEmpty && bytes[0] == 0x59) return parseUdsDtcReply(bytes);
    try {
      return parseObdMode3Reply(await _payload('03') ?? const <int>[]);
    } catch (_) {
      return const [];
    }
  }

  /// Сброс + автоматический повторный запрос (возвращает остаток).
  Future<List<DiagDtc>> clearDtc(EcuSlot slot) async {
    await _atEcu(slot);
    if (slot.proto == DiagProto.ssm2) {
      final bytes = await _payload('14 FF FF 00') ?? const <int>[];
      if (bytes.isEmpty || bytes[0] != 0x54) {
        final obd = await _payload('04') ?? const <int>[];
        if (obd.isEmpty || obd[0] != 0x44) {
          throw ReplyError('ECU отказал в сбросе DTC');
        }
      }
    } else {
      final bytes = await _payload('14 FF FF FF') ?? const <int>[];
      if (bytes.isEmpty || bytes[0] != 0x54) {
        throw ReplyError('ECU отказал в сбросе DTC (UDS 14)');
      }
    }
    await Future<void>.delayed(const Duration(milliseconds: 400));
    return readDtc(slot);
  }

  /// UDS ECUReset 11 01 — глушит мотор! EXPERIMENTAL + стоянка.
  Future<void> ecuReset(EcuSlot slot) async {
    if (!experimental) {
      throw StateError('ECUReset заблокирован (EXPERIMENTAL=false в 04.2)');
    }
    await _atEcu(slot);
    final bytes = await _payload('11 01') ?? const <int>[];
    if (bytes.isEmpty || bytes[0] != 0x51) {
      throw ReplyError('ECUReset не подтверждён (ожидался 51 01)');
    }
  }

  /// SSM3 Clear Memory 04 gr. Коды групп зависят от ECU — EXPERIMENTAL.
  Future<void> clearMemory(int group) async {
    if (!experimental) {
      throw StateError('Clear Memory заблокирован (EXPERIMENTAL=false в 04.2)');
    }
    if (group < 1 || group > 7) throw RangeError.range(group, 1, 7);
    final bytes = await _payload('04 ${hex2(group)}') ?? const <int>[];
    if (bytes.isEmpty || bytes[0] != 0x44) {
      throw ReplyError('Clear Memory group=$group отклонён');
    }
  }

  /// Вернуть заголовки на двигатель после операций.
  Future<void> restore() async {
    try {
      await elm.diagnostic('ATSH 7E0');
      await elm.diagnostic('ATCRA 7E8');
    } catch (_) {}
  }
}
'''

# ===== Вкладка DTC: блоки, чтение/сброс, сервисные процедуры =====
FILES["lib/dtc_service_page.dart"] = r'''
import 'package:flutter/material.dart';

import 'diag.dart';
import 'model.dart';

/// Вкладка «DTC»: чтение/сброс ошибок по блокам + сервисные процедуры.
class DtcServicePage extends StatefulWidget {
  const DtcServicePage({super.key, required this.model});
  final AppModel model;

  @override
  State<DtcServicePage> createState() => _DtcServicePageState();
}

class _DtcServicePageState extends State<DtcServicePage> {
  final Map<String, List<DiagDtc>> _results = {};
  bool _busy = false;
  String _log = '';

  Future<void> _run(String what, Future<void> Function() op) =>
      widget.model.diagRun(what, () async {
        setState(() { _busy = true; _log = what; });
        try {
          await op();
        } finally {
          if (mounted) setState(() => _busy = false);
        }
      });

  Future<void> _read(EcuSlot slot) => _run('Чтение ${slot.name}', () async {
        final list = await widget.model.diagSession.readDtc(slot);
        if (mounted) setState(() => _results[slot.name] = list);
      });

  Future<void> _clear(EcuSlot slot) async {
    final ok = await showDialog<bool>(
      context: context,
      builder: (ctx) => AlertDialog(
        title: Text('Сбросить ошибки: ${slot.name}?'),
        content: const Text(
            'Стоп-кадры и мониторы readiness очистятся. CEL погаснет после '
            'перезапуска зажигания. Адаптации (IAM, LTFT) это НЕ трогает.'),
        actions: [
          TextButton(
              onPressed: () => Navigator.pop(ctx, false),
              child: const Text('ОТМЕНА')),
          FilledButton.tonal(
              onPressed: () => Navigator.pop(ctx, true),
              child: const Text('СБРОСИТЬ')),
        ],
      ),
    );
    if (ok != true) return;
    await _run('Сброс ${slot.name}', () async {
      final rest = await widget.model.diagSession.clearDtc(slot);
      if (mounted) {
        setState(() {
          _results[slot.name] = rest;
          _log = rest.isEmpty
              ? '${slot.name}: очищено'
              : '${slot.name}: осталось ${rest.length}';
        });
      }
    });
  }

  @override
  Widget build(BuildContext context) {
    final theme = Theme.of(context);
    return ListView(
      padding: const EdgeInsets.all(12),
      children: [
        if (_busy) const LinearProgressIndicator(),
        if (_log.isNotEmpty)
          Padding(
            padding: const EdgeInsets.symmetric(vertical: 6),
            child: Text(_log, style: theme.textTheme.bodySmall),
          ),
        for (final slot in EcuSlot.all)
          Card(
            child: ExpansionTile(
              leading: Icon(
                _results.containsKey(slot.name)
                    ? (_results[slot.name]!.isEmpty
                        ? Icons.check_circle
                        : Icons.warning_amber)
                    : Icons.memory,
              ),
              title: Text(slot.name),
              subtitle: Text(
                  'CAN ${slot.header}→${slot.responseId} · ${slot.proto.name.toUpperCase()}'),
              children: [
                for (final d in _results[slot.name] ?? const <DiagDtc>[])
                  ListTile(
                    dense: true,
                    leading: Icon(
                      d.active ? Icons.error : Icons.history,
                      color: d.active
                          ? theme.colorScheme.error
                          : theme.colorScheme.outline,
                    ),
                    title: Text(
                      '${d.code}${d.titleRu.isEmpty ? '' : ' · ${d.titleRu}'}',
                      style: const TextStyle(fontFamily: 'monospace'),
                    ),
                    subtitle: Text(d.statusText),
                    trailing: d.milOn
                        ? const Icon(Icons.lightbulb, color: Colors.amber)
                        : null,
                  ),
                OverflowBar(children: [
                  TextButton.icon(
                    onPressed: _busy ? null : () => _read(slot),
                    icon: const Icon(Icons.search),
                    label: const Text('ПРОЧИТАТЬ'),
                  ),
                  if ((_results[slot.name] ?? const []).isNotEmpty)
                    FilledButton.tonalIcon(
                      onPressed: _busy ? null : () => _clear(slot),
                      icon: const Icon(Icons.cleaning_services),
                      label: const Text('СБРОС'),
                    ),
                ]),
              ],
            ),
          ),
        const Divider(height: 32),
        Text('Сервисные процедуры', style: theme.textTheme.titleMedium),
        const ListTile(
          leading: Icon(Icons.restart_alt),
          title: Text('Сброс адаптаций (SSM2-эра)'),
          subtitle: Text(
              'Клемма АКБ 10–30 мин; контроль IAM до/после в логгер и дообучение 15–20 мин.'),
        ),
        const ListTile(
          leading: Icon(Icons.speed),
          title: Text('Обучение дросселя (E-Gas)'),
          subtitle: Text(
              'Зажигание ON 15–20 с без запуска → OFF → запуск и прогрев до 82 °C.'),
        ),
        ListTile(
          leading: const Icon(Icons.science),
          title: const Text('ECUReset 11 01 / Clear Memory 04 (UDS)'),
          subtitle: Text(widget.model.experimental
              ? 'EXPERIMENTAL: разблокировано. Глушит мотор — только стоянка.'
              : 'Заблокировано: EXPERIMENTAL=false в ячейке 04.2.'),
          enabled: widget.model.experimental,
        ),
      ],
    );
  }
}
'''

# ===== 16 тестов: ExprEval, декодеры, payload, whitelist =====
FILES["test/diag_test.dart"] = r'''
import 'package:flutter_test/flutter_test.dart';
import 'package:subaru_ssm2/diag.dart';
import 'package:subaru_ssm2/expr.dart';
import 'package:subaru_ssm2/protocol.dart';

void main() {
  group('ExprEval', () {
    test('линейные', () {
      expect(ExprEval.eval('x*100/255', 128), closeTo(50.196, 0.001));
      expect(ExprEval.eval('(x*100)-100', 1.205), closeTo(20.5, 0.001));
      expect(ExprEval.eval('x+6', 2.0), 8.0);
      expect(ExprEval.eval('x/.84', 84), closeTo(100.0, 0.001));
    });
    test('дробные AFR-формы', () {
      expect(ExprEval.eval('14.7/(1+x)', 0.0), closeTo(14.7, 0.001));
      expect(ExprEval.eval('14.7/x', 1.0), closeTo(14.7, 0.001));
      expect(ExprEval.eval('x*0.001333224', 750.0), closeTo(1.0, 0.0001));
    });
    test('защита от мусора', () {
      expect(ExprEval.eval('xZXy', 1), isNull);
      expect(ExprEval.eval('14/0', 3), isNull); // деление на 0 -> NaN -> null
    });
  });

  group('decodeDtcBytes', () {
    test('коды', () {
      expect(decodeDtcBytes(0x01, 0x10), 'P0110');
      expect(decodeDtcBytes(0x03, 0x03), 'P0303');
      expect(decodeDtcBytes(0x40, 0x57), 'C0057');
      expect(decodeDtcBytes(0xD0, 0x00), 'U1000');
    });
  });

  group('parseSsm2DtcReply', () {
    test('две ошибки со статусами', () {
      final d = parseSsm2DtcReply(
          const [0x58, 0x02, 0x01, 0x10, 0xA0, 0x03, 0x03, 0x00]);
      expect(d, hasLength(2));
      expect(d[0].code, 'P0110');
      expect(d[0].active, isTrue);
      expect(d[0].milOn, isTrue);
      expect(d[1].code, 'P0303');
      expect(d[1].active, isFalse);
    });
    test('чисто', () => expect(parseSsm2DtcReply(const [0x58, 0x00]), isEmpty));
    test('негатив пропускаем', () {
      expect(parseSsm2DtcReply(const [0x7F, 0x18, 0x11]), isEmpty);
    });
  });

  group('parseObdMode3Reply', () {
    test('пары без статусов; нули отбрасываются', () {
      final d = parseObdMode3Reply(const [0x43, 0x00, 0x00, 0x01, 0x10, 0x03, 0x03]);
      expect(d.map((x) => x.code), ['P0110', 'P0303']);
    });
  });

  group('parseUdsDtcReply', () {
    test('59 02 FF за DTC', () {
      final d = parseUdsDtcReply(
          const [0x59, 0x02, 0xFF, 0x01, 0x10, 0x2F, 0x03, 0x03, 0x00]);
      expect(d, hasLength(2));
      expect(d[0].code, 'P0110');
      expect(d[0].active, isTrue);
      expect(d[0].confirmed, isTrue);
      expect(d[1].code, 'P0303');
    });
  });

  group('parseDiagPayload', () {
    test('plain', () {
      expect(parseDiagPayload('58 01 03 03 00'), [0x58, 0x01, 0x03, 0x03, 0x00]);
    });
    test('PCI отрезается', () {
      expect(
          parseDiagPayload('05 58 02 01 10 A0 03 00'), [0x58, 0x02, 0x01, 0x10, 0xA0]);
    });
    test('frame lines склеиваются', () {
      expect(parseDiagPayload('0: 58 02 01\n1: 10 A0'),
          [0x58, 0x02, 0x01, 0x10, 0xA0]);
    });
    test('пусто -> null', () => expect(parseDiagPayload('NO DATA'), isNull));
  });

  group('allowedDiagnostic (whitelist)', () {
    test('старые команды не сломаны', () {
      expect(allowedDiagnostic('A8 00 00 00 08'), isTrue);
      expect(allowedDiagnostic('ATSH 7E0'), isTrue);
      expect(allowedDiagnostic('ATCRA 7E8'), isTrue);
    });
    test('новые сервисы разрешены', () {
      for (final c in ['18 00 FF 00', '14 FF FF 00', '14 FF FF FF', '19 02 AF', '03', '04', '07', '0A', '09 02', '01 01']) {
        expect(allowedDiagnostic(c), isTrue, reason: c);
      }
    });
    test('произвольная запись запрещена', () {
      expect(allowedDiagnostic('2E F1 90 AA'), isFalse);
    });
  });
}
'''

# ===== Точечные правки: protocol/model/main/aliases/tests/calid_import =====
# ===== REWRITES: consumable-иглы + сигнатуры idempotency =====
# Кортеж: (файл, игла, замена, сигнатура, optional). Сигнатура = короткие маркеры, которые
# есть в тексте, если правка УЖЕ применена (устойчивы к dart format и эволюции блоков).
# optional=True: апгрейд-замены (не обязательны к применению, если иглы нет).

# --- protocol.dart: расширение белого списка (игла включает '}' функции) ---
REWRITES.append((
    "lib/protocol.dart",
    """      RegExp(r'^A8 00 [0-9A-F]{2} [0-9A-F]{2} [0-9A-F]{2}$').hasMatch(c);
}""",
    """      RegExp(r'^A8 00 [0-9A-F]{2} [0-9A-F]{2} [0-9A-F]{2}$').hasMatch(c) ||
      _diagServiceCommands.contains(c) ||
      _diagServicePatterns.any((p) => p.hasMatch(c));
}

// v0.11 (04.2): сервисные команды диагностики (SSM2 0x18/0x14, OBD-II,
// UDS 19/14, readiness, VIN). Произвольная запись по-прежнему запрещена.
const Set<String> _diagServiceCommands = <String>{
  '18 00 FF 00', // SSM2 ReadDTC (ECM/TCM)
  '14 FF FF 00', // SSM2 ClearDTC
  '14 FF FF FF', // UDS ClearDiagnosticInformation
  '19 02 AF', // UDS ReadDTCInformation (all)
  '03', '04', '07', '0A', // OBD DTC stored/clear/pending/permanent
  '01 01', // OBD readiness
  '09 02', // OBD VIN
  '11 01', // UDS ECUReset (глушится EXPERIMENTAL в diag.dart)
};
final List<RegExp> _diagServicePatterns = <RegExp>[
  // SSM3 Clear Memory: 04 XX (под EXPERIMENTAL-гейтом DiagSession)
  RegExp(r'^04 0[1-7]$'),
];""",
    ["_diagServiceCommands", "'19 02 AF'", False],
))

# --- upgrade: ранняя редакция 04.2 писала const-список RegExp (ошибка analyze) ---
REWRITES.append((
    "lib/protocol.dart",
    "const List<RegExp> _diagServicePatterns",
    "final List<RegExp> _diagServicePatterns",
    ["final List<RegExp> _diagServicePatterns", True],
))

# upgrade: на случай, если прошлый прогон записал 55 вместо 57
REWRITES.append((
    "test/protocol_test.dart",
    "expect(all.length, 55); // v0.11: 28 + 27 loggerdef (A2TB100B)",
    "expect(all.length, 57); // v0.11: 28 + 29 loggerdef (A2TB100B)",
    ["all.length, 57", True],
))

# --- model.dart: импорт (игла = 3-строчный блок) + DiagSession/diagRun ---
REWRITES.append((
    "lib/model.dart",
    "import 'bt_transport.dart';\nimport 'elm.dart';\nimport 'engine.dart';",
    "import 'bt_transport.dart';\nimport 'diag.dart';\nimport 'elm.dart';\nimport 'engine.dart';",
    ["import 'diag.dart';", False],
))
REWRITES.append((
    "lib/model.dart",
    "  late final SsmEngine engine;\n  final logger = CsvLogger();",
    """  late final SsmEngine engine;
  final bool experimental =
      const bool.fromEnvironment('SSM2_EXPERIMENTAL', defaultValue: false);
  late final DiagSession diagSession =
      DiagSession(elm, experimental: experimental);
  final logger = CsvLogger();

  /// Диагностическая операция: опрос PID на паузе, восстановление 7E0/7E8,
  /// возврат опроса, если он работал.
  Future<void> diagRun(String label, Future<void> Function() op) =>
      perform(() async {
        final wasPolling = engine.running;
        await engine.stop();
        try {
          await op();
          message = label;
        } finally {
          await diagSession.restore();
          if (wasPolling) await engine.start();
        }
      });""",
    ["diagSession", "diagRun", False],
))

# --- main.dart: импорт + вкладка (иглы = парные строки, consumable) ---
REWRITES.append((
    "lib/main.dart",
    "import 'engine.dart';\nimport 'maplab_link.dart';\nimport 'model.dart';",
    "import 'engine.dart';\nimport 'dtc_service_page.dart';\nimport 'maplab_link.dart';\nimport 'model.dart';",
    ["import 'dtc_service_page.dart';", False],
))
REWRITES.append((
    "lib/main.dart",
    "            DiagnosticPage(model),\n            const MapLabTab(),",
    "            DiagnosticPage(model),\n            DtcServicePage(model: model),\n            const MapLabTab(),",
    ["DtcServicePage(model: model)", False],
))
REWRITES.append((
    "lib/main.dart",
    """                NavigationDestination(
                    icon: Icon(Icons.terminal), label: 'Диагн.'),
                NavigationDestination(
                  icon: Icon(Icons.table_view),""",
    """                NavigationDestination(
                    icon: Icon(Icons.terminal), label: 'Диагн.'),
                NavigationDestination(
                    icon: Icon(Icons.build_circle_outlined), label: 'DTC'),
                NavigationDestination(
                  icon: Icon(Icons.table_view),""",
    ["label: 'DTC'", False],
))
REWRITES.append((
    "lib/main.dart",
    "SSM2 TELEMETRY 0.10.2",
    "SSM2 TELEMETRY 0.11",
    ["SSM2 TELEMETRY 0.11", False],
))

# --- maps_lab_log.dart: алиасы tq (игла = 3 строки incl. cl_target) ---
REWRITES.append((
    "lib/maps_lab/maps_lab_log.dart",
    "    'tq': [\n      'requested torque',\n      'cl_target',",
    "    'tq': [\n      'requested torque',\n      'req_tq',\n      'requested torque nm',\n      'cl_target',",
    ["'req_tq'", False],
))

# --- protocol_test.dart: в библиотеке теперь 28 + 27 = 55 PID ---
REWRITES.append((
    "test/protocol_test.dart",
    "expect(all.length, 28);",
    "expect(all.length, 57); // v0.11: 28 + 29 loggerdef (A2TB100B)",
    ["all.length, 57", True],
))

# --- calid_import.py: разрешить новые параметры ---
NEW_VALID = '''VALID = {"IAM", "LOAD_4B", "BOOST_ERR", "BOOST_TGT", "FBKC", "FKL", "BOOST", "CL_TARGET", "REQ_TQ", "KC_ADV_4B", "KC_IAM", "BTIMING", "KCMAX", "KNOCK_SUM", "AFL_A", "AFL_B", "AFL_C", "AFL_D", "AFL_4B", "AFCOR1_4B", "TIPIN", "TTH_TGT", "THROTTLE_4B", "CLOL", "FL_OFFSET", "GEAR_CALC", "TDIG_INT", "TDIG_PROP", "INJPW", "INJ_LAT", "OLE_ENRICH", "ENRICH_FINAL", "FUELBASE", "MAP_4B", "TBOOST_REL", "IDLE_SEL", "AFL_RANGE"}'''
REWRITES.append((
    "tool/calid_import.py",
    'VALID = {"IAM", "LOAD_4B", "BOOST_ERR", "BOOST_TGT", "FBKC", "FKL", "BOOST", "CL_TARGET"}',
    NEW_VALID,
    ["REQ_TQ"],
))
REWRITES.append((
    "tool/calid_import.py",
    'Допустимые param: IAM, LOAD_4B, BOOST_ERR, BOOST_TGT, FBKC, FKL, BOOST, CL_TARGET.',
    'Допустимые param: IAM, LOAD_4B, BOOST_ERR, BOOST_TGT, FBKC, FKL, BOOST, CL_TARGET, REQ_TQ, KC_ADV_4B, KC_IAM, BTIMING, KCMAX, KNOCK_SUM, AFL_A..D, AFL_4B, AFCOR1_4B, TIPIN, TTH_TGT, THROTTLE_4B, CLOL, FL_OFFSET, GEAR_CALC, TDIG_INT, TDIG_PROP, INJPW, INJ_LAT, OLE_ENRICH, ENRICH_FINAL, FUELBASE, MAP_4B, TBOOST_REL, IDLE_SEL, AFL_RANGE.',
    ["AFL_A..D", False],
))

# ===== санитария: удаление бэкапов и мусора внутри APP =====
# flutter analyze сканирует ВСЁ дерево проекта — бэкапы внутри APP приводят к ошибкам
for old_bak in list(APP.glob("_backup_*")) + list(APP.glob("*backup*")):
    if old_bak.is_dir():
        try:
            shutil.rmtree(old_bak, ignore_errors=True)
            print(f"[CLEAN] удалена бэкап-папка внутри проекта: {old_bak.name}")
        except Exception as e:
            print(f"[CLEAN] ошибка удаления {old_bak}: {e}")

# Удаление файлов с суффиксами от ранних прогонов
for stray in [p for p in APP.glob("lib/**/*") if "(" in p.name or ")" in p.name]:
    try:
        stray.unlink()
        print(f"[CLEAN] удалён брошенный файл: {stray.relative_to(APP)}")
    except Exception as e:
        print(f"[CLEAN] не удалось удалить {stray}: {e}")

# ===== запись новых файлов =====
# страж: в FILES-ключах не должно быть скобок (суффиксы "(полная замена)"
# предназначены только для UI-отображения частей)
bad_keys = [rel for rel in FILES if "(" in rel or ")" in rel]
if bad_keys:
    raise RuntimeError(f"Невалидные FILES-ключи с суффиксами: {bad_keys}")
BACKUP.mkdir(parents=True, exist_ok=True)
written = 0
for rel, body in FILES.items():
    dest = APP / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        shutil.copyfile(dest, BACKUP / (rel.replace("/", "__") + ".bak"))
    dest.write_text(body.strip("\n") + "\n", encoding="utf-8")
    written += 1
    print(f"[OK] {rel}")
print(f"[OK] файлов записано/перезаписано: {written}")

# ===== точечные правки (идемпотентно, по сигнатурам) =====
patched, skipped = 0, 0
for rel, needle, replacement, sig in REWRITES:
    # обратная совместимость: если в сигнатуре хвост True/False — это optional
    optional = False
    if sig and sig[-1] is True:
        sig, optional = sig[:-1], True
    elif sig and sig[-1] is False:
        sig, optional = sig[:-1], False
    path = APP / rel
    if not path.exists():
        raise RuntimeError(f"Нет файла {rel} — сначала прогоните 02–04.1")
    text = path.read_text(encoding="utf-8")
    if needle in text:
        shutil.copyfile(path, BACKUP / (rel.replace("/", "__") + ".bak"))
        path.write_text(text.replace(needle, replacement, 1), encoding="utf-8")
        patched += 1
        print(f"[OK] правка {rel}")
        continue
    if all(s in text for s in sig):
        skipped += 1
        print(f"[SKIP уже применено] {rel}: {sig[0][:60]}")
        continue
    if optional:
        skipped += 1
        print(f"[SKIP необязательная] {rel}")
        continue
    raise RuntimeError(
        f"Маркер не найден в {rel} (и правка не применена):\n{needle[:90]!r}")
print(f"[OK] правок: {patched}, пропущено (уже применено): {skipped}")

# ===== самопроверка маркеров =====
def markers_check():
    checks = {
        "lib/expr.dart": ["class ExprEval", "exprRawX"],
        "lib/pids.dart": ["'REQ_TQ'", "0xFF8058", "exprText", "class SubaruPidLibrary"],
        "lib/identity.dart": ["'REQ_TQ': '0xFF8058'", "5204584007"],
        "lib/loggerdef.dart": ["loggerdefParams", "loggerdefExt", "loggerdefSwitches"],
        "lib/dtc_dict.dart": ["kSubaruDtcDict", "P0300"],
        "lib/diag.dart": ["class DiagSession", "18 00 FF 00", "14 FF FF 00", "parseUdsDtcReply"],
        "lib/dtc_service_page.dart": ["class DtcServicePage", "EcuSlot.all"],
        "lib/protocol.dart": ["_diagServiceCommands", "19 02 AF"],
        "lib/model.dart": ["diagSession", "diagRun"],
        "lib/main.dart": ["DtcServicePage", "label: 'DTC'"],
        "lib/maps_lab/maps_lab_log.dart": ["'req_tq'"],
        "test/diag_test.dart": ["ExprEval.eval", "allowedDiagnostic"],
        "test/protocol_test.dart": ["all.length, 57"],
        "tool/calid_import.py": ['"REQ_TQ"'],
    }
    for rel, needles in checks.items():
        text = (APP / rel).read_text(encoding="utf-8")
        missing = [m for m in needles if m not in text]
        if missing:
            raise RuntimeError(f"Самопроверка {rel}: нет маркеров {missing}")
markers_check()
print("[OK] самопроверка маркеров пройдена")

print("\n=== Готово: SSM2 0.11. Далее — ячейка 05 (сборка). ===")
print("Новое в приложении:")
print("  · PID-библиотека: 57 параметров (28 + 29 loggerdef для 5204584007)")
print("  · вкладка DTC: чтение/сброс ошибок ECM/TCM/VDC/SRS/EPS/BIU")
print("  · Requested Torque (0xFF8058) — включите REQ_TQ в наборе PID")
print("Чек-лист стенда 04.2:")
print(" [ ] REQ_TQ в логе > 0 при нажатии газа (raw-Nm по шкале вашего ROM)")
print(" [ ] DTC: чтение ECM при 5 активных PID (опрос вернётся сам)")
print(" [ ] Сброс: re-read пуст, CEL гаснет после OFF/ON")
print(" [ ] TCU (ATSH 7E1): чтение + возврат на 7E0 без переподключения")
print(" [ ] AFL_A..D не все нули одновременно на прогретом моторе")


[OK] lib/expr.dart
[OK] lib/pids.dart
[OK] lib/identity.dart
[OK] lib/loggerdef.dart
[OK] lib/dtc_dict.dart
[OK] lib/diag.dart
[OK] lib/dtc_service_page.dart
[OK] test/diag_test.dart
[OK] файлов записано/перезаписано: 8
[OK] правка lib/protocol.dart
[SKIP уже применено] lib/protocol.dart: final List<RegExp> _diagServicePatterns
[SKIP необязательная] test/protocol_test.dart
[OK] правка lib/model.dart
[OK] правка lib/model.dart
[OK] правка lib/main.dart
[OK] правка lib/main.dart
[OK] правка lib/main.dart
[OK] правка lib/main.dart
[OK] правка lib/maps_lab/maps_lab_log.dart
[OK] правка test/protocol_test.dart
[OK] правка tool/calid_import.py
[OK] правка tool/calid_import.py
[OK] правок: 11, пропущено (уже применено): 2
[OK] самопроверка маркеров пройдена

=== Готово: SSM2 0.11. Далее — ячейка 05 (сборка). ===
Новое в приложении:
  · PID-библиотека: 57 параметров (28 + 29 loggerdef для 5204584007)
  · вкладка DTC: чтение/сброс ошибок ECM/TCM/VDC/SRS/EPS/BIU
  · Requested Torque (0xFF8058) —

In [ ]:
# ===== Шапка 04.3: параметры, бэкап снаружи проекта, реестры =====
# @title 04.3 | CONFIG & UX | SSM2 0.11 >> 0.12 — конфиг мотора, свой дашборд, ориентация (вставить МЕЖДУ 04.2 и 05)
# Что делает:
#  1. РАЗБЛОКИРУЕТ 4-байтовые PID: ROM-профиль A2TB100B включается автоматически
#     (раньше extended PID молча вырезались, пока вручную не подтвердишь ROM).
#     Кнопка «Все 4-байтовые*» во вкладке PID + понятная причина блокировки.
#  2. CSV-заголовки RomRaider для всех 29 новых PID -> Map Lab их наконец видит.
#  3. lib/vehicle_config.dart: конфигурация железа (форсунки cc + латентность,
#     бензонасос л/ч, тип турбо: сток/single/twin-scroll/rotated/big single,
#     топливо 95/98/E30/E85, объём, редлайн, цель AFR, лимит наддува).
#     Расчёты: IDC форсунок, запас насоса, потолок мощности, советы Map Lab.
#  4. Вкладка «Мотор»: настройка + живые расчёты по текущей телеметрии.
#  5. Настраиваемый дашборд: выбор/порядок плиток (drag&drop), число колонок
#     отдельно для книжной и альбомной, компактный режим, мини-графики.
#  6. Ориентация: авто/книжная/альбомная + NavigationRail в альбомной.
# Идемпотентна, бэкап снаружи проекта, самопроверка маркеров.

import json
import re
import shutil
import time
from pathlib import Path

APP = Path("/content/subaru_ssm2_fixed")

STAMP = time.strftime("%Y%m%d_%H%M%S")
BACKUP = APP.parent / f"ssm2_backup_043_{STAMP}"
FILES = {}
REWRITES = []  # (файл, игла, замена, [сигнатуры..., optional_bool])

# ===== Конфигурация мотора: турбо, форсунки, насос, топливо + расчёты IDC =====
FILES["lib/vehicle_config.dart"] = r'''
import 'dart:convert';

/// Тип турбосистемы — влияет на ожидания по наддуву и советы Map Lab.
enum TurboKind {
  stockTwinScroll('Сток twin-scroll (TD04/VF)', 1.05, 0.08),
  singleScroll('Single-scroll апгрейд', 1.25, 0.12),
  twinScroll('Twin-scroll апгрейд', 1.35, 0.10),
  rotated('Rotated mount (большой single)', 1.60, 0.16),
  sequentialTwin('Sequential twin (VF/TD пара)', 1.10, 0.14),
  bigSingle('Big single (конкурсный)', 2.00, 0.22);

  const TurboKind(this.label, this.typicalMaxBoostBar, this.spoolSlopeBar);

  /// Человекочитаемое название для UI.
  final String label;

  /// Ориентир потолка наддува (бар, relative) для подсказок анализа.
  final double typicalMaxBoostBar;

  /// Допустимый коридор отклонения факт/цель на разгоне (бар).
  final double spoolSlopeBar;
}

/// Топливо: влияет на стехиометрию и запас по форсункам.
enum FuelKind {
  gasoline95('АИ-95', 14.7, 1.00),
  gasoline98('АИ-98/100', 14.7, 1.00),
  e30('E30 (30% этанол)', 13.6, 1.12),
  e85('E85', 9.8, 1.35);

  const FuelKind(this.label, this.stoich, this.flowFactor);
  final String label;

  /// Стехиометрическое соотношение (для пересчёта lambda <-> AFR).
  final double stoich;

  /// Во сколько раз больше топлива нужно против бензина (расход форсунок).
  final double flowFactor;
}

/// Конфигурация железа мотора. Сохраняется в ssm2_settings.json.
class VehicleConfig {
  const VehicleConfig({
    this.injectorCc = 565,
    this.injectorLatencyMs = 0.92,
    this.injectorCount = 4,
    this.fuelPumpLph = 190,
    this.pumpVoltageDerate = 0.88,
    this.turbo = TurboKind.stockTwinScroll,
    this.fuel = FuelKind.gasoline98,
    this.displacementL = 2.0,
    this.redlineRpm = 7000,
    this.targetAfrWot = 11.2,
    this.maxSafeBoostBar = 1.05,
    this.bspOffsetKpa = 0,
  });

  /// Производительность одной форсунки, cc/min @ базовом давлении.
  final int injectorCc;

  /// Латентность (мёртвое время) форсунки, мс @ 14 В.
  final double injectorLatencyMs;
  final int injectorCount;

  /// Производительность бензонасоса, л/ч @ рабочем давлении.
  final int fuelPumpLph;

  /// Поправка на просадку напряжения/давления (0.8..1.0).
  final double pumpVoltageDerate;

  final TurboKind turbo;
  final FuelKind fuel;
  final double displacementL;
  final int redlineRpm;

  /// Целевой AFR на полной нагрузке (для бензина; для E85 пересчитается).
  final double targetAfrWot;

  /// Личный потолок наддува (бар, relative) — для предупреждений.
  final double maxSafeBoostBar;

  /// Смещение базового давления топлива (кПа) — влияет на фактический расход.
  final double bspOffsetKpa;

  VehicleConfig copyWith({
    int? injectorCc,
    double? injectorLatencyMs,
    int? injectorCount,
    int? fuelPumpLph,
    double? pumpVoltageDerate,
    TurboKind? turbo,
    FuelKind? fuel,
    double? displacementL,
    int? redlineRpm,
    double? targetAfrWot,
    double? maxSafeBoostBar,
    double? bspOffsetKpa,
  }) =>
      VehicleConfig(
        injectorCc: injectorCc ?? this.injectorCc,
        injectorLatencyMs: injectorLatencyMs ?? this.injectorLatencyMs,
        injectorCount: injectorCount ?? this.injectorCount,
        fuelPumpLph: fuelPumpLph ?? this.fuelPumpLph,
        pumpVoltageDerate: pumpVoltageDerate ?? this.pumpVoltageDerate,
        turbo: turbo ?? this.turbo,
        fuel: fuel ?? this.fuel,
        displacementL: displacementL ?? this.displacementL,
        redlineRpm: redlineRpm ?? this.redlineRpm,
        targetAfrWot: targetAfrWot ?? this.targetAfrWot,
        maxSafeBoostBar: maxSafeBoostBar ?? this.maxSafeBoostBar,
        bspOffsetKpa: bspOffsetKpa ?? this.bspOffsetKpa,
      );

  Map<String, dynamic> toJson() => <String, dynamic>{
        'injectorCc': injectorCc,
        'injectorLatencyMs': injectorLatencyMs,
        'injectorCount': injectorCount,
        'fuelPumpLph': fuelPumpLph,
        'pumpVoltageDerate': pumpVoltageDerate,
        'turbo': turbo.name,
        'fuel': fuel.name,
        'displacementL': displacementL,
        'redlineRpm': redlineRpm,
        'targetAfrWot': targetAfrWot,
        'maxSafeBoostBar': maxSafeBoostBar,
        'bspOffsetKpa': bspOffsetKpa,
      };

  static VehicleConfig fromJson(Map<String, dynamic>? json) {
    if (json == null) return const VehicleConfig();
    T pick<T>(String key, T fallback) {
      final v = json[key];
      return v is T ? v : fallback;
    }

    return VehicleConfig(
      injectorCc: pick<num>('injectorCc', 565).toInt(),
      injectorLatencyMs: pick<num>('injectorLatencyMs', 0.92).toDouble(),
      injectorCount: pick<num>('injectorCount', 4).toInt(),
      fuelPumpLph: pick<num>('fuelPumpLph', 190).toInt(),
      pumpVoltageDerate: pick<num>('pumpVoltageDerate', 0.88).toDouble(),
      turbo: TurboKind.values.firstWhere(
        (t) => t.name == json['turbo'],
        orElse: () => TurboKind.stockTwinScroll,
      ),
      fuel: FuelKind.values.firstWhere(
        (f) => f.name == json['fuel'],
        orElse: () => FuelKind.gasoline98,
      ),
      displacementL: pick<num>('displacementL', 2.0).toDouble(),
      redlineRpm: pick<num>('redlineRpm', 7000).toInt(),
      targetAfrWot: pick<num>('targetAfrWot', 11.2).toDouble(),
      maxSafeBoostBar: pick<num>('maxSafeBoostBar', 1.05).toDouble(),
      bspOffsetKpa: pick<num>('bspOffsetKpa', 0).toDouble(),
    );
  }

  String encode() => jsonEncode(toJson());
}

/// ------------------------- Расчёты по конфигурации -------------------------

/// Суммарный расход форсунок, см³/мин (с учётом топлива).
double totalInjectorFlowCc(VehicleConfig c) =>
    c.injectorCc * c.injectorCount / c.fuel.flowFactor;

/// Требуемый расход топлива, см³/мин, по массовому расходу воздуха.
/// maf г/с → масса топлива = maf / afr; плотность бензина ≈ 0.745 г/см³.
double requiredFuelCcPerMin({
  required double mafGramsPerSec,
  required double afr,
}) {
  if (afr <= 0) return 0;
  final fuelGramsPerMin = mafGramsPerSec * 60 / afr;
  return fuelGramsPerMin / 0.745;
}

/// Расчётная загрузка форсунок (IDC), %.
/// Классическая формула: IDC = (pulseWidth / (60000 / (rpm/2))) * 100.
double injectorDutyFromPulse({
  required double pulseWidthMs,
  required double rpm,
}) {
  if (rpm <= 0) return 0;
  final cycleMs = 120000 / rpm; // период впрыска (4-такт, 1 впрыск / 2 оборота)
  if (cycleMs <= 0) return 0;
  return (pulseWidthMs / cycleMs) * 100;
}

/// Оценка IDC по MAF и AFR, когда PID ширины импульса недоступен.
double injectorDutyFromMaf({
  required double mafGramsPerSec,
  required double afr,
  required VehicleConfig config,
}) {
  final needCc = requiredFuelCcPerMin(mafGramsPerSec: mafGramsPerSec, afr: afr);
  final haveCc = totalInjectorFlowCc(config);
  if (haveCc <= 0) return 0;
  return (needCc / haveCc) * 100;
}

/// Запас бензонасоса, %: сколько ещё топлива он может дать сверх текущего.
double pumpHeadroomPct({
  required double mafGramsPerSec,
  required double afr,
  required VehicleConfig config,
}) {
  final needCcMin = requiredFuelCcPerMin(mafGramsPerSec: mafGramsPerSec, afr: afr);
  final needLph = needCcMin * 60 / 1000;
  final haveLph = config.fuelPumpLph * config.pumpVoltageDerate / config.fuel.flowFactor;
  if (haveLph <= 0) return 0;
  return (1 - needLph / haveLph) * 100;
}

/// Потолок мощности по форсункам (л.с., BSFC 0.6 для турбо-бензина).
double maxPowerByInjectors(VehicleConfig c) {
  final lbsPerHour = totalInjectorFlowCc(c) * 0.0952; // cc/min -> lb/h (бензин)
  return lbsPerHour / 0.6 * 0.85; // запас по IDC 85%
}

/// Потолок мощности по насосу (л.с.).
double maxPowerByPump(VehicleConfig c) {
  final lph = c.fuelPumpLph * c.pumpVoltageDerate / c.fuel.flowFactor;
  final lbsPerHour = lph * 0.745 * 2.2046; // л/ч -> lb/h
  return lbsPerHour / 0.6 * 0.9;
}

enum AdviceLevel { ok, warning, critical }

class ConfigAdvice {
  const ConfigAdvice(this.level, this.title, this.detail);
  final AdviceLevel level;
  final String title;
  final String detail;
}

/// Советы по конфигурации + живой телеметрии (значения могут быть null).
List<ConfigAdvice> buildConfigAdvice({
  required VehicleConfig config,
  double? maf,
  double? afr,
  double? rpm,
  double? boostBar,
  double? injPulseMs,
  double? requestedTorque,
}) {
  final out = <ConfigAdvice>[];

  // --- форсунки ---
  double? idc;
  if (injPulseMs != null && rpm != null && rpm > 500) {
    idc = injectorDutyFromPulse(pulseWidthMs: injPulseMs, rpm: rpm);
  } else if (maf != null && afr != null && afr > 5) {
    idc = injectorDutyFromMaf(mafGramsPerSec: maf, afr: afr, config: config);
  }
  if (idc != null) {
    if (idc >= 95) {
      out.add(ConfigAdvice(
        AdviceLevel.critical,
        'Форсунки на пределе: IDC ${idc.toStringAsFixed(0)}%',
        'При IDC ≥ 95% форсунка физически не успевает — смесь уходит в бедную '
            'на пике наддува. Нужны форсунки больше ${config.injectorCc} cc '
            'или снижение цели наддува.',
      ));
    } else if (idc >= 85) {
      out.add(ConfigAdvice(
        AdviceLevel.warning,
        'Высокая загрузка форсунок: IDC ${idc.toStringAsFixed(0)}%',
        'Рабочий максимум — 85%. Запас на жару и просадку напряжения почти исчерпан.',
      ));
    } else {
      out.add(ConfigAdvice(
        AdviceLevel.ok,
        'Форсунки в норме: IDC ${idc.toStringAsFixed(0)}%',
        '${config.injectorCc} cc × ${config.injectorCount}, топливо ${config.fuel.label}.',
      ));
    }
  }

  // --- насос ---
  if (maf != null && afr != null && afr > 5) {
    final head = pumpHeadroomPct(mafGramsPerSec: maf, afr: afr, config: config);
    if (head < 0) {
      out.add(ConfigAdvice(
        AdviceLevel.critical,
        'Бензонасос не вытягивает (дефицит ${(-head).toStringAsFixed(0)}%)',
        'Расчётная подача ${config.fuelPumpLph} л/ч ниже потребности. '
            'Ожидайте провал давления и обеднение на верхах.',
      ));
    } else if (head < 15) {
      out.add(ConfigAdvice(
        AdviceLevel.warning,
        'Малый запас насоса: ${head.toStringAsFixed(0)}%',
        'На жаре и при низком заряде АКБ подача просядет — держите запас ≥ 20%.',
      ));
    }
  }

  // --- турбо ---
  if (boostBar != null) {
    final cap = config.turbo.typicalMaxBoostBar;
    if (boostBar > config.maxSafeBoostBar + 0.05) {
      out.add(ConfigAdvice(
        AdviceLevel.critical,
        'Перебуст: ${boostBar.toStringAsFixed(2)} бар',
        'Выше вашего лимита ${config.maxSafeBoostBar.toStringAsFixed(2)} бар. '
            'Проверьте вестгейт, шланги и карту WGDC.',
      ));
    } else if (boostBar > cap) {
      out.add(ConfigAdvice(
        AdviceLevel.warning,
        'Наддув выше типичного для «${config.turbo.label}»',
        'Ориентир для этой турбосистемы — до ${cap.toStringAsFixed(2)} бар. '
            'Убедитесь, что хватает форсунок и топлива.',
      ));
    }
  }

  // --- топливо/цель ---
  if (afr != null && rpm != null && rpm > 3500 && afr > config.targetAfrWot + 0.8) {
    out.add(ConfigAdvice(
      AdviceLevel.warning,
      'Смесь беднее цели: ${afr.toStringAsFixed(2)} AFR',
      'Цель на полной нагрузке ${config.targetAfrWot.toStringAsFixed(2)}. '
          'Бедная смесь под наддувом — прямой путь к детонации.',
    ));
  }

  // --- общий потолок ---
  out.add(ConfigAdvice(
    AdviceLevel.ok,
    'Потолок топливной системы',
    'Форсунки ≈ ${maxPowerByInjectors(config).toStringAsFixed(0)} л.с., '
        'насос ≈ ${maxPowerByPump(config).toStringAsFixed(0)} л.с. '
        '(${config.fuel.label}, ${config.turbo.label}).',
  ));

  return out;
}
'''

# ===== Настройки дашборда и ориентации экрана =====
FILES["lib/dashboard_prefs.dart"] = r'''
import 'pids.dart';

/// Настройки дашборда: состав плиток, порядок, плотность, спарклайны.
class DashboardPrefs {
  DashboardPrefs({
    List<String>? tiles,
    this.columnsPortrait = 2,
    this.columnsLandscape = 4,
    this.compact = false,
    this.showSpark = true,
    this.showFuelCard = true,
  }) : tiles = tiles ?? List<String>.from(defaultTiles);

  /// Порядок и состав плиток (id PID).
  final List<String> tiles;
  final int columnsPortrait;
  final int columnsLandscape;
  final bool compact;
  final bool showSpark;
  final bool showFuelCard;

  static const List<String> defaultTiles = <String>[
    'RPM',
    'MAP_REL',
    'AFR',
    'TIMING',
    'ECT',
    'IAT',
    'FBKC',
    'IAM',
  ];

  DashboardPrefs copyWith({
    List<String>? tiles,
    int? columnsPortrait,
    int? columnsLandscape,
    bool? compact,
    bool? showSpark,
    bool? showFuelCard,
  }) =>
      DashboardPrefs(
        tiles: tiles ?? List<String>.from(this.tiles),
        columnsPortrait: columnsPortrait ?? this.columnsPortrait,
        columnsLandscape: columnsLandscape ?? this.columnsLandscape,
        compact: compact ?? this.compact,
        showSpark: showSpark ?? this.showSpark,
        showFuelCard: showFuelCard ?? this.showFuelCard,
      );

  Map<String, dynamic> toJson() => <String, dynamic>{
        'tiles': tiles,
        'columnsPortrait': columnsPortrait,
        'columnsLandscape': columnsLandscape,
        'compact': compact,
        'showSpark': showSpark,
        'showFuelCard': showFuelCard,
      };

  static DashboardPrefs fromJson(Map<String, dynamic>? json) {
    if (json == null) return DashboardPrefs();
    final raw = (json['tiles'] as List<dynamic>? ?? const <dynamic>[])
        .whereType<String>()
        .where((id) => SubaruPidLibrary.all.any((p) => p.id == id))
        .toList();
    int clampCols(Object? v, int fallback) {
      final n = v is num ? v.toInt() : fallback;
      return n < 1 ? 1 : (n > 6 ? 6 : n);
    }

    return DashboardPrefs(
      tiles: raw.isEmpty ? null : raw,
      columnsPortrait: clampCols(json['columnsPortrait'], 2),
      columnsLandscape: clampCols(json['columnsLandscape'], 4),
      compact: json['compact'] == true,
      showSpark: json['showSpark'] != false,
      showFuelCard: json['showFuelCard'] != false,
    );
  }

  /// Плитки, которые реально можно показать (PID включён в опрос).
  List<String> visibleTiles(Set<String> enabledPids) =>
      tiles.where(enabledPids.contains).toList();
}

/// Режим ориентации экрана.
enum OrientationMode {
  auto('Авто (по датчику)'),
  portrait('Только книжная'),
  landscape('Только альбомная');

  const OrientationMode(this.label);
  final String label;
}

class UiPrefs {
  const UiPrefs({this.orientation = OrientationMode.auto, this.keepAwake = true});
  final OrientationMode orientation;
  final bool keepAwake;

  UiPrefs copyWith({OrientationMode? orientation, bool? keepAwake}) => UiPrefs(
        orientation: orientation ?? this.orientation,
        keepAwake: keepAwake ?? this.keepAwake,
      );

  Map<String, dynamic> toJson() => <String, dynamic>{
        'orientation': orientation.name,
        'keepAwake': keepAwake,
      };

  static UiPrefs fromJson(Map<String, dynamic>? json) {
    if (json == null) return const UiPrefs();
    return UiPrefs(
      orientation: OrientationMode.values.firstWhere(
        (o) => o.name == json['orientation'],
        orElse: () => OrientationMode.auto,
      ),
      keepAwake: json['keepAwake'] != false,
    );
  }
}
'''

# ===== Настраиваемый дашборд: drag&drop плитки, колонки, альбомный режим =====
FILES["lib/dashboard_page.dart"] = r'''
import 'package:flutter/material.dart';

import 'dashboard_prefs.dart';
import 'derived.dart';
import 'model.dart';
import 'pids.dart';
import 'samples.dart';

const _muted = Color(0xFF8CA0BF);

/// Настраиваемый дашборд: выбор/порядок плиток, плотность, альбомный режим.
class CustomDashboardPage extends StatefulWidget {
  const CustomDashboardPage(this.model, {super.key});
  final AppModel model;

  @override
  State<CustomDashboardPage> createState() => _CustomDashboardPageState();
}

class _CustomDashboardPageState extends State<CustomDashboardPage> {
  AppModel get model => widget.model;

  @override
  Widget build(BuildContext context) {
    final engine = model.engine;
    final prefs = model.dashboardPrefs;
    final active = engine.active;
    final activeIds = active.map((p) => p.id).toSet();

    // плитки: пользовательский порядок, затем активные PID, которых нет в списке
    final chosen = prefs.tiles.where(activeIds.contains).toList();
    final rest = active.map((p) => p.id).where((id) => !chosen.contains(id));
    final ids = <String>[...chosen, ...rest];

    final isLandscape =
        MediaQuery.of(context).orientation == Orientation.landscape;
    final columns = isLandscape ? prefs.columnsLandscape : prefs.columnsPortrait;

    final fuel = estimateFuel(
      maf: engine.latest['MAF']?.value,
      afr: engine.latest['AFR']?.value,
      speed: engine.latest['SPEED']?.value,
      rpm: engine.latest['RPM']?.value,
      pedal: engine.latest['PEDAL']?.value,
    );

    return Column(
      children: [
        Padding(
          padding: const EdgeInsets.fromLTRB(12, 4, 12, 0),
          child: Row(
            children: [
              Expanded(
                child: Text(
                  '${ids.length} плиток · ${engine.pidReadsPerSecond.toStringAsFixed(1)} обновл./с'
                  '${isLandscape ? ' · альбомная' : ''}',
                  style: const TextStyle(fontSize: 12, color: _muted),
                ),
              ),
              IconButton(
                tooltip: 'Настроить дашборд',
                onPressed: () => _configure(context),
                icon: const Icon(Icons.dashboard_customize),
              ),
              FilledButton.tonalIcon(
                onPressed:
                    model.busy || !model.elm.ready ? null : model.togglePolling,
                icon: Icon(engine.running ? Icons.pause : Icons.play_arrow),
                label: Text(engine.running ? 'Пауза' : 'Старт'),
              ),
            ],
          ),
        ),
        if (prefs.showFuelCard && fuel != null)
          Padding(
            padding: const EdgeInsets.fromLTRB(12, 4, 12, 0),
            child: Card(
              margin: EdgeInsets.zero,
              child: Padding(
                padding: const EdgeInsets.symmetric(horizontal: 12, vertical: 8),
                child: Row(
                  children: [
                    const Icon(Icons.local_gas_station, size: 16, color: _muted),
                    const SizedBox(width: 8),
                    Expanded(
                      child: Text(
                        '${fuel.litresPerHour.toStringAsFixed(1)} л/ч'
                        ' · поездка ${engine.tripDistanceKm.toStringAsFixed(1)} км / '
                        '${engine.tripFuelLitres.toStringAsFixed(2)} л',
                        style: const TextStyle(fontSize: 12),
                      ),
                    ),
                    IconButton(
                      visualDensity: VisualDensity.compact,
                      tooltip: 'Сбросить поездку',
                      onPressed: engine.resetTrip,
                      icon: const Icon(Icons.restart_alt, size: 18),
                    ),
                  ],
                ),
              ),
            ),
          ),
        if (ids.isEmpty)
          const Expanded(
            child: Center(
              child: Padding(
                padding: EdgeInsets.all(24),
                child: Text(
                  'Нет активных PID.\nВкладка «PID» → выберите параметры '
                  '(для 4-байтовых нажмите «Все 4-байтовые*»).',
                  textAlign: TextAlign.center,
                ),
              ),
            ),
          )
        else
          Expanded(
            child: GridView.builder(
              padding: const EdgeInsets.all(10),
              itemCount: ids.length,
              gridDelegate: SliverGridDelegateWithFixedCrossAxisCount(
                crossAxisCount: columns,
                mainAxisExtent: prefs.compact ? 104 : 150,
                mainAxisSpacing: 8,
                crossAxisSpacing: 8,
              ),
              itemBuilder: (context, i) {
                final pid = SubaruPidLibrary.all.firstWhere(
                  (p) => p.id == ids[i],
                  orElse: () => SubaruPidLibrary.byId('RPM'),
                );
                return _Tile(
                  pid: pid,
                  sample: engine.latest[pid.id],
                  history: prefs.showSpark && !prefs.compact
                      ? engine.history[pid.id]
                      : null,
                  compact: prefs.compact,
                  ageMs: engine.ageMs(pid.id),
                );
              },
            ),
          ),
      ],
    );
  }

  Future<void> _configure(BuildContext context) async {
    final result = await showModalBottomSheet<DashboardPrefs>(
      context: context,
      isScrollControlled: true,
      showDragHandle: true,
      builder: (_) => _DashboardConfigSheet(model: model),
    );
    if (result != null) {
      await model.updateDashboardPrefs(result);
      if (mounted) setState(() {});
    }
  }
}

class _Tile extends StatelessWidget {
  const _Tile({
    required this.pid,
    required this.sample,
    required this.history,
    required this.compact,
    required this.ageMs,
  });

  final SubaruPidDef pid;
  final PidSample? sample;
  final List<PidSample>? history;
  final bool compact;
  final int? ageMs;

  @override
  Widget build(BuildContext context) {
    final value = sample?.value;
    final stale = (ageMs ?? 999999) > 2500;
    final unsupported = sample?.allOnes ?? false;
    final text = value == null
        ? (unsupported ? 'н/д' : '—')
        : value.toStringAsFixed(pid.digits);

    return Card(
      margin: EdgeInsets.zero,
      child: Padding(
        padding: EdgeInsets.all(compact ? 8 : 12),
        child: Column(
          crossAxisAlignment: CrossAxisAlignment.start,
          children: [
            Row(
              children: [
                Expanded(
                  child: Text(
                    '${pid.id}${pid.extended ? ' *' : ''}',
                    maxLines: 1,
                    overflow: TextOverflow.ellipsis,
                    style: TextStyle(
                      fontSize: compact ? 11 : 12,
                      fontWeight: FontWeight.bold,
                      color: stale ? _muted : null,
                    ),
                  ),
                ),
                if (stale)
                  const Icon(Icons.schedule, size: 12, color: _muted),
              ],
            ),
            const SizedBox(height: 2),
            FittedBox(
              fit: BoxFit.scaleDown,
              alignment: Alignment.centerLeft,
              child: Text(
                text,
                style: TextStyle(
                  fontSize: compact ? 22 : 30,
                  fontWeight: FontWeight.w600,
                ),
              ),
            ),
            Text(
              pid.unit,
              style: const TextStyle(fontSize: 10, color: _muted),
            ),
            if (history != null && history!.length > 2 && !compact) ...[
              const SizedBox(height: 4),
              Expanded(
                child: CustomPaint(
                  painter: _SparkPainter(history!, pid),
                  size: Size.infinite,
                ),
              ),
            ],
          ],
        ),
      ),
    );
  }
}

class _SparkPainter extends CustomPainter {
  _SparkPainter(this.samples, this.pid);
  final List<PidSample> samples;
  final SubaruPidDef pid;

  @override
  void paint(Canvas canvas, Size size) {
    final values = samples
        .where((s) => s.value != null)
        .map((s) => s.value!)
        .toList();
    if (values.length < 2) return;
    var lo = values.first, hi = values.first;
    for (final v in values) {
      if (v < lo) lo = v;
      if (v > hi) hi = v;
    }
    if ((hi - lo).abs() < 1e-9) {
      hi = lo + 1;
    }
    final path = Path();
    for (var i = 0; i < values.length; i++) {
      final x = size.width * i / (values.length - 1);
      final y = size.height * (1 - (values[i] - lo) / (hi - lo));
      if (i == 0) {
        path.moveTo(x, y);
      } else {
        path.lineTo(x, y);
      }
    }
    canvas.drawPath(
      path,
      Paint()
        ..color = const Color(0xFF22D3EE)
        ..style = PaintingStyle.stroke
        ..strokeWidth = 1.6,
    );
  }

  @override
  bool shouldRepaint(covariant _SparkPainter old) => true;
}

/// Нижний лист настройки: порядок плиток, колонки, плотность.
class _DashboardConfigSheet extends StatefulWidget {
  const _DashboardConfigSheet({required this.model});
  final AppModel model;

  @override
  State<_DashboardConfigSheet> createState() => _DashboardConfigSheetState();
}

class _DashboardConfigSheetState extends State<_DashboardConfigSheet> {
  late DashboardPrefs prefs = widget.model.dashboardPrefs;

  @override
  Widget build(BuildContext context) {
    final active = widget.model.engine.active.map((p) => p.id).toList();
    final chosen = prefs.tiles.where(active.contains).toList();
    final available = active.where((id) => !chosen.contains(id)).toList();

    return DraggableScrollableSheet(
      expand: false,
      initialChildSize: 0.8,
      maxChildSize: 0.95,
      builder: (context, scroll) => ListView(
        controller: scroll,
        padding: const EdgeInsets.fromLTRB(16, 0, 16, 24),
        children: [
          const Text(
            'Настройка дашборда',
            style: TextStyle(fontSize: 18, fontWeight: FontWeight.bold),
          ),
          const SizedBox(height: 4),
          const Text(
            'Перетаскивайте плитки за ручку, снимайте галочки чтобы скрыть.',
            style: TextStyle(fontSize: 12, color: _muted),
          ),
          const SizedBox(height: 12),
          Row(
            children: [
              Expanded(
                child: _Stepper(
                  label: 'Колонок (книжная)',
                  value: prefs.columnsPortrait,
                  min: 1,
                  max: 4,
                  onChanged: (v) =>
                      setState(() => prefs = prefs.copyWith(columnsPortrait: v)),
                ),
              ),
              const SizedBox(width: 12),
              Expanded(
                child: _Stepper(
                  label: 'Колонок (альбомная)',
                  value: prefs.columnsLandscape,
                  min: 2,
                  max: 6,
                  onChanged: (v) => setState(
                      () => prefs = prefs.copyWith(columnsLandscape: v)),
                ),
              ),
            ],
          ),
          SwitchListTile(
            contentPadding: EdgeInsets.zero,
            title: const Text('Компактные плитки'),
            subtitle: const Text('Больше параметров на экране, без графиков'),
            value: prefs.compact,
            onChanged: (v) => setState(() => prefs = prefs.copyWith(compact: v)),
          ),
          SwitchListTile(
            contentPadding: EdgeInsets.zero,
            title: const Text('Мини-графики'),
            value: prefs.showSpark,
            onChanged: (v) =>
                setState(() => prefs = prefs.copyWith(showSpark: v)),
          ),
          SwitchListTile(
            contentPadding: EdgeInsets.zero,
            title: const Text('Карточка расхода'),
            value: prefs.showFuelCard,
            onChanged: (v) =>
                setState(() => prefs = prefs.copyWith(showFuelCard: v)),
          ),
          const Divider(height: 24),
          Text('На дашборде (${chosen.length})',
              style: const TextStyle(fontWeight: FontWeight.bold)),
          const SizedBox(height: 8),
          ReorderableListView(
            shrinkWrap: true,
            physics: const NeverScrollableScrollPhysics(),
            buildDefaultDragHandles: false,
            onReorder: (oldIndex, newIndex) {
              setState(() {
                final list = List<String>.from(chosen);
                if (newIndex > oldIndex) newIndex -= 1;
                list.insert(newIndex, list.removeAt(oldIndex));
                final others =
                    prefs.tiles.where((id) => !chosen.contains(id)).toList();
                prefs = prefs.copyWith(tiles: [...list, ...others]);
              });
            },
            children: [
              for (var i = 0; i < chosen.length; i++)
                ListTile(
                  key: ValueKey('tile-${chosen[i]}'),
                  dense: true,
                  contentPadding: EdgeInsets.zero,
                  leading: ReorderableDragStartListener(
                    index: i,
                    child: const Icon(Icons.drag_handle),
                  ),
                  title: Text(chosen[i]),
                  subtitle: Text(
                    SubaruPidLibrary.all
                        .firstWhere((p) => p.id == chosen[i])
                        .desc,
                    maxLines: 1,
                    overflow: TextOverflow.ellipsis,
                    style: const TextStyle(fontSize: 11, color: _muted),
                  ),
                  trailing: IconButton(
                    icon: const Icon(Icons.visibility_off, size: 18),
                    onPressed: () => setState(() {
                      final list = List<String>.from(prefs.tiles)
                        ..remove(chosen[i]);
                      prefs = prefs.copyWith(tiles: list);
                    }),
                  ),
                ),
            ],
          ),
          if (available.isNotEmpty) ...[
            const Divider(height: 24),
            Text('Доступные PID (${available.length})',
                style: const TextStyle(fontWeight: FontWeight.bold)),
            const SizedBox(height: 8),
            Wrap(
              spacing: 6,
              runSpacing: 6,
              children: [
                for (final id in available)
                  ActionChip(
                    label: Text(id, style: const TextStyle(fontSize: 12)),
                    avatar: const Icon(Icons.add, size: 16),
                    onPressed: () => setState(() {
                      prefs = prefs.copyWith(tiles: [...prefs.tiles, id]);
                    }),
                  ),
              ],
            ),
          ],
          const SizedBox(height: 20),
          Row(
            children: [
              TextButton(
                onPressed: () => setState(() => prefs = DashboardPrefs()),
                child: const Text('Сбросить'),
              ),
              const Spacer(),
              FilledButton(
                onPressed: () => Navigator.pop(context, prefs),
                child: const Text('Применить'),
              ),
            ],
          ),
        ],
      ),
    );
  }
}

class _Stepper extends StatelessWidget {
  const _Stepper({
    required this.label,
    required this.value,
    required this.min,
    required this.max,
    required this.onChanged,
  });
  final String label;
  final int value, min, max;
  final ValueChanged<int> onChanged;

  @override
  Widget build(BuildContext context) => Column(
        crossAxisAlignment: CrossAxisAlignment.start,
        children: [
          Text(label, style: const TextStyle(fontSize: 11, color: _muted)),
          Row(
            children: [
              IconButton(
                visualDensity: VisualDensity.compact,
                onPressed: value > min ? () => onChanged(value - 1) : null,
                icon: const Icon(Icons.remove_circle_outline),
              ),
              Text('$value', style: const TextStyle(fontSize: 16)),
              IconButton(
                visualDensity: VisualDensity.compact,
                onPressed: value < max ? () => onChanged(value + 1) : null,
                icon: const Icon(Icons.add_circle_outline),
              ),
            ],
          ),
        ],
      );
}
'''

# ===== Вкладка «Мотор»: настройки железа + живые советы =====
FILES["lib/vehicle_config_page.dart"] = r'''
import 'package:flutter/material.dart';

import 'dashboard_prefs.dart';
import 'model.dart';
import 'vehicle_config.dart';

const _muted = Color(0xFF8CA0BF);

/// Вкладка «Мотор»: железо + живые расчёты топливной системы и наддува.
class VehicleConfigPage extends StatefulWidget {
  const VehicleConfigPage(this.model, {super.key});
  final AppModel model;

  @override
  State<VehicleConfigPage> createState() => _VehicleConfigPageState();
}

class _VehicleConfigPageState extends State<VehicleConfigPage> {
  AppModel get model => widget.model;
  VehicleConfig get cfg => model.vehicleConfig;

  Future<void> _set(VehicleConfig next) async {
    await model.updateVehicleConfig(next);
    if (mounted) setState(() {});
  }

  double? _live(String id) => model.engine.latest[id]?.value;

  @override
  Widget build(BuildContext context) {
    final advice = buildConfigAdvice(
      config: cfg,
      maf: _live('MAF'),
      afr: _live('AFR'),
      rpm: _live('RPM'),
      boostBar: _live('BOOST') ?? _live('MAP_REL'),
      injPulseMs: _live('INJPW'),
      requestedTorque: _live('REQ_TQ'),
    );

    return ListView(
      padding: const EdgeInsets.all(16),
      children: [
        const Text('Конфигурация мотора',
            style: TextStyle(fontSize: 18, fontWeight: FontWeight.bold)),
        const Text(
          'Эти параметры учитывают анализатор и Map Lab: лимиты форсунок, '
          'насоса и ожидания по наддуву зависят от вашего железа.',
          style: TextStyle(fontSize: 12, color: _muted),
        ),
        const SizedBox(height: 16),

        // --- турбо ---
        DropdownButtonFormField<TurboKind>(
          initialValue: cfg.turbo,
          decoration: const InputDecoration(labelText: 'Турбосистема'),
          items: [
            for (final t in TurboKind.values)
              DropdownMenuItem(
                value: t,
                child: Text(
                  '${t.label} · до ${t.typicalMaxBoostBar.toStringAsFixed(2)} бар',
                  style: const TextStyle(fontSize: 13),
                ),
              ),
          ],
          onChanged: (v) => v == null ? null : _set(cfg.copyWith(turbo: v)),
        ),
        const SizedBox(height: 12),

        // --- топливо ---
        DropdownButtonFormField<FuelKind>(
          initialValue: cfg.fuel,
          decoration: const InputDecoration(labelText: 'Топливо'),
          items: [
            for (final f in FuelKind.values)
              DropdownMenuItem(
                value: f,
                child: Text(
                  '${f.label} · стех ${f.stoich.toStringAsFixed(1)}',
                  style: const TextStyle(fontSize: 13),
                ),
              ),
          ],
          onChanged: (v) => v == null ? null : _set(cfg.copyWith(fuel: v)),
        ),
        const SizedBox(height: 8),

        _NumField(
          label: 'Форсунки, cc/min (на одну)',
          value: cfg.injectorCc.toDouble(),
          min: 150,
          max: 2200,
          digits: 0,
          onChanged: (v) => _set(cfg.copyWith(injectorCc: v.round())),
        ),
        _NumField(
          label: 'Латентность форсунки, мс @14В',
          value: cfg.injectorLatencyMs,
          min: 0.2,
          max: 3.0,
          digits: 2,
          onChanged: (v) => _set(cfg.copyWith(injectorLatencyMs: v)),
        ),
        _NumField(
          label: 'Бензонасос, л/ч',
          value: cfg.fuelPumpLph.toDouble(),
          min: 60,
          max: 600,
          digits: 0,
          onChanged: (v) => _set(cfg.copyWith(fuelPumpLph: v.round())),
        ),
        _NumField(
          label: 'Потолок наддува (свой лимит), бар',
          value: cfg.maxSafeBoostBar,
          min: 0.3,
          max: 2.5,
          digits: 2,
          onChanged: (v) => _set(cfg.copyWith(maxSafeBoostBar: v)),
        ),
        _NumField(
          label: 'Цель AFR на полной нагрузке',
          value: cfg.targetAfrWot,
          min: 9.0,
          max: 14.7,
          digits: 2,
          onChanged: (v) => _set(cfg.copyWith(targetAfrWot: v)),
        ),
        _NumField(
          label: 'Объём двигателя, л',
          value: cfg.displacementL,
          min: 1.0,
          max: 4.0,
          digits: 1,
          onChanged: (v) => _set(cfg.copyWith(displacementL: v)),
        ),

        const Divider(height: 28),
        const Text('Расчёт по текущей телеметрии',
            style: TextStyle(fontSize: 16, fontWeight: FontWeight.bold)),
        const SizedBox(height: 8),
        for (final a in advice)
          Card(
            margin: const EdgeInsets.only(bottom: 8),
            child: ListTile(
              dense: true,
              leading: Icon(
                a.level == AdviceLevel.critical
                    ? Icons.error
                    : a.level == AdviceLevel.warning
                        ? Icons.warning_amber
                        : Icons.check_circle,
                color: a.level == AdviceLevel.critical
                    ? Colors.redAccent
                    : a.level == AdviceLevel.warning
                        ? Colors.amberAccent
                        : Colors.greenAccent,
              ),
              title: Text(a.title, style: const TextStyle(fontSize: 14)),
              subtitle: Text(a.detail,
                  style: const TextStyle(fontSize: 12, color: _muted)),
            ),
          ),

        const Divider(height: 28),
        const Text('Экран',
            style: TextStyle(fontSize: 16, fontWeight: FontWeight.bold)),
        DropdownButtonFormField<OrientationMode>(
          initialValue: model.uiPrefs.orientation,
          decoration: const InputDecoration(labelText: 'Ориентация'),
          items: [
            for (final o in OrientationMode.values)
              DropdownMenuItem(
                value: o,
                child: Text(o.label, style: const TextStyle(fontSize: 13)),
              ),
          ],
          onChanged: (v) async {
            if (v == null) return;
            await model.updateUiPrefs(model.uiPrefs.copyWith(orientation: v));
            if (mounted) setState(() {});
          },
        ),
        const SizedBox(height: 24),
      ],
    );
  }
}

class _NumField extends StatelessWidget {
  const _NumField({
    required this.label,
    required this.value,
    required this.min,
    required this.max,
    required this.digits,
    required this.onChanged,
  });
  final String label;
  final double value, min, max;
  final int digits;
  final ValueChanged<double> onChanged;

  @override
  Widget build(BuildContext context) => Padding(
        padding: const EdgeInsets.symmetric(vertical: 4),
        child: Column(
          crossAxisAlignment: CrossAxisAlignment.start,
          children: [
            Row(
              children: [
                Expanded(
                  child: Text(label, style: const TextStyle(fontSize: 13)),
                ),
                Text(
                  value.toStringAsFixed(digits),
                  style: const TextStyle(
                      fontSize: 14, fontWeight: FontWeight.bold),
                ),
              ],
            ),
            Slider(
              value: value.clamp(min, max).toDouble(),
              min: min,
              max: max,
              onChanged: onChanged,
            ),
          ],
        ),
      );
}
'''

# ===== 16 тестов: IDC, насос, потолки, советы, префы дашборда =====
FILES["test/config_test.dart"] = r'''
import 'package:flutter_test/flutter_test.dart';
import 'package:subaru_ssm2/dashboard_prefs.dart';
import 'package:subaru_ssm2/vehicle_config.dart';

void main() {
  group('VehicleConfig', () {
    test('сериализация туда-обратно', () {
      const c = VehicleConfig(
        injectorCc: 1050,
        fuelPumpLph: 255,
        turbo: TurboKind.rotated,
        fuel: FuelKind.e85,
      );
      final back = VehicleConfig.fromJson(c.toJson());
      expect(back.injectorCc, 1050);
      expect(back.fuelPumpLph, 255);
      expect(back.turbo, TurboKind.rotated);
      expect(back.fuel, FuelKind.e85);
    });

    test('дефолты при пустом json', () {
      final c = VehicleConfig.fromJson(null);
      expect(c.injectorCc, 565);
      expect(c.turbo, TurboKind.stockTwinScroll);
    });
  });

  group('Расчёты топливной системы', () {
    test('IDC по ширине импульса', () {
      // 6000 об/мин -> период 20 мс; импульс 10 мс = 50%
      expect(injectorDutyFromPulse(pulseWidthMs: 10, rpm: 6000),
          closeTo(50.0, 0.001));
      expect(injectorDutyFromPulse(pulseWidthMs: 5, rpm: 0), 0);
    });

    test('IDC по MAF растёт с расходом воздуха', () {
      const c = VehicleConfig(injectorCc: 565, injectorCount: 4);
      final low = injectorDutyFromMaf(mafGramsPerSec: 50, afr: 11.5, config: c);
      final high = injectorDutyFromMaf(mafGramsPerSec: 200, afr: 11.5, config: c);
      expect(high, greaterThan(low));
      expect(low, greaterThan(0));
    });

    test('E85 требует больше топлива -> меньше эффективный расход форсунок', () {
      const gas = VehicleConfig(fuel: FuelKind.gasoline98);
      const e85 = VehicleConfig(fuel: FuelKind.e85);
      expect(totalInjectorFlowCc(e85), lessThan(totalInjectorFlowCc(gas)));
    });

    test('запас насоса падает при большом расходе', () {
      const c = VehicleConfig(fuelPumpLph: 190);
      final calm = pumpHeadroomPct(mafGramsPerSec: 30, afr: 14.7, config: c);
      final wot = pumpHeadroomPct(mafGramsPerSec: 250, afr: 11.0, config: c);
      expect(calm, greaterThan(wot));
    });

    test('потолки мощности положительны и растут с железом', () {
      const small = VehicleConfig(injectorCc: 380, fuelPumpLph: 150);
      const big = VehicleConfig(injectorCc: 1050, fuelPumpLph: 340);
      expect(maxPowerByInjectors(big), greaterThan(maxPowerByInjectors(small)));
      expect(maxPowerByPump(big), greaterThan(maxPowerByPump(small)));
      expect(maxPowerByInjectors(small), greaterThan(0));
    });
  });

  group('Советы по конфигурации', () {
    test('критично при IDC >= 95%', () {
      const c = VehicleConfig(injectorCc: 380, injectorCount: 4);
      final advice = buildConfigAdvice(
        config: c,
        injPulseMs: 19.5,
        rpm: 6000,
      );
      expect(
        advice.any((a) =>
            a.level == AdviceLevel.critical && a.title.contains('Форсунки')),
        isTrue,
      );
    });

    test('перебуст выше личного лимита', () {
      const c = VehicleConfig(maxSafeBoostBar: 1.0);
      final advice = buildConfigAdvice(config: c, boostBar: 1.4);
      expect(advice.any((a) => a.title.startsWith('Перебуст')), isTrue);
    });

    test('чистая конфигурация без критичных советов', () {
      const c = VehicleConfig(injectorCc: 1050, fuelPumpLph: 340);
      final advice = buildConfigAdvice(
        config: c,
        maf: 120,
        afr: 11.2,
        rpm: 5000,
        boostBar: 0.9,
      );
      expect(advice.any((a) => a.level == AdviceLevel.critical), isFalse);
    });
  });

  group('DashboardPrefs', () {
    test('дефолтный набор плиток не пуст', () {
      expect(DashboardPrefs().tiles, isNotEmpty);
      expect(DashboardPrefs().tiles, contains('RPM'));
    });

    test('сериализация и ограничение колонок', () {
      final p = DashboardPrefs(
        tiles: ['RPM', 'AFR'],
        columnsPortrait: 99,
        columnsLandscape: 0,
        compact: true,
      );
      final back = DashboardPrefs.fromJson(p.toJson());
      expect(back.tiles, ['RPM', 'AFR']);
      expect(back.columnsPortrait, lessThanOrEqualTo(6));
      expect(back.columnsLandscape, greaterThanOrEqualTo(1));
      expect(back.compact, isTrue);
    });

    test('мусорные id отбрасываются, пустой список -> дефолт', () {
      final back = DashboardPrefs.fromJson({'tiles': ['NOPE', 'ZZZ']});
      expect(back.tiles, DashboardPrefs.defaultTiles);
    });

    test('visibleTiles фильтрует по активным PID', () {
      final p = DashboardPrefs(tiles: ['RPM', 'AFR', 'FBKC']);
      expect(p.visibleTiles({'RPM', 'FBKC'}), ['RPM', 'FBKC']);
    });
  });

  group('UiPrefs', () {
    test('ориентация сохраняется', () {
      const u = UiPrefs(orientation: OrientationMode.landscape);
      expect(UiPrefs.fromJson(u.toJson()).orientation,
          OrientationMode.landscape);
    });
    test('дефолт — авто', () {
      expect(UiPrefs.fromJson(null).orientation, OrientationMode.auto);
    });
  });
}
'''

# ===== Правки: разблокировка 4-байтовых, CSV-имена, вкладки, ориентация =====
# ===== REWRITES =====

# --- 1. model.dart: импорты новых модулей ---
REWRITES.append((
    "lib/model.dart",
    "import 'bt_transport.dart';\nimport 'diag.dart';",
    "import 'bt_transport.dart';\nimport 'dashboard_prefs.dart';\nimport 'diag.dart';",
    ["import 'dashboard_prefs.dart';", False],
))
REWRITES.append((
    "lib/model.dart",
    "import 'pids.dart';\nimport 'protocol.dart';",
    "import 'pids.dart';\nimport 'protocol.dart';\nimport 'vehicle_config.dart';",
    ["import 'vehicle_config.dart';", False],
))

# --- 2. model.dart: RomRaider-имена для новых PID (Map Lab видит колонки) ---
REWRITES.append((
    "lib/model.dart",
    "  'CL_TARGET': 'Closed Loop Fueling Target (AFR)*',\n};",
    """  'CL_TARGET': 'Closed Loop Fueling Target (AFR)*',
  // v0.12: имена logger-дефиниций RomRaider для PID из патча 04.2
  'REQ_TQ': "Requested Torque*",
  'KC_ADV_4B': "Knock Correction Advance (4-byte) (degrees)*",
  'KC_IAM': "Knock Correction Advance (IAM only) (degrees)*",
  'BTIMING': "Ignition Base Timing (degrees)*",
  'KCMAX': "Knock Correction Advance Max Primary (degrees)*",
  'KNOCK_SUM': "Knock Sum (count)*",
  'AFL_A': "A/F Learning #1 A (Stored) (%)*",
  'AFL_B': "A/F Learning #1 B (Stored) (%)*",
  'AFL_C': "A/F Learning #1 C (Stored) (%)*",
  'AFL_D': "A/F Learning #1 D (Stored) (%)*",
  'AFL_4B': "A/F Learning #1 (4-byte) (%)*",
  'AFCOR1_4B': "A/F Correction #1 (4-byte) (%)*",
  'TIPIN': "Tip-in Throttle (%)*",
  'TTH_TGT': "Target Throttle Plate Position (%)*",
  'THROTTLE_4B': "Throttle Plate Opening Angle (4-byte) (%)*",
  'CLOL': "CL/OL Fueling*",
  'FL_OFFSET': "Fine Learning Table Offset*",
  'GEAR_CALC': "Gear (Calculated)*",
  'TDIG_INT': "Turbo Dynamics Integral (4-byte) (absolute %)*",
  'TDIG_PROP': "Turbo Dynamics Proportional (4-byte) (absolute %)*",
  'INJPW': "Fuel Injector #1 Pulse Width (4-byte) (ms)*",
  'INJ_LAT': "Fuel Injector #1 Latency (4-byte) (ms)*",
  'OLE_ENRICH': "Primary Open Loop Map Enrichment (4-byte) (estimated AFR)*",
  'ENRICH_FINAL': "Primary Enrichment Final (4-byte) (estimated AFR)*",
  'FUELBASE': "Final Fueling Base (4-byte) (estimated AFR)*",
  'MAP_4B': "Manifold Absolute Pressure (4-byte) (bar absolute)*",
  'TBOOST_REL': "Target Boost Relative (4-byte) (bar relative)*",
  'IDLE_SEL': "Idle Speed Map Selection*",
  'AFL_RANGE': "A/F Learning Airflow Range (Current)*",
};""",
    ["'REQ_TQ':", "'KNOCK_SUM':", False],
))

# --- 3. model.dart: поля конфигурации + методы обновления ---
REWRITES.append((
    "lib/model.dart",
    "  late final DiagSession diagSession =\n      DiagSession(elm, experimental: experimental);",
    """  late final DiagSession diagSession =
      DiagSession(elm, experimental: experimental);

  // v0.12 (04.3): конфигурация железа, дашборда и экрана
  VehicleConfig vehicleConfig = const VehicleConfig();
  DashboardPrefs dashboardPrefs = DashboardPrefs();
  UiPrefs uiPrefs = const UiPrefs();

  Future<void> updateVehicleConfig(VehicleConfig next) async {
    vehicleConfig = next;
    await saveSettings();
    changed();
  }

  Future<void> updateDashboardPrefs(DashboardPrefs next) async {
    dashboardPrefs = next;
    await saveSettings();
    changed();
  }

  Future<void> updateUiPrefs(UiPrefs next) async {
    uiPrefs = next;
    await saveSettings();
    changed();
  }

  /// v0.12: включить все ROM-зависимые (4-байтовые) PID разом.
  /// Раньше они молча вырезались, пока вручную не подтвердишь ROM ID.
  Future<void> enableExtendedPids() => perform(() async {
        final rom = engine.romId.trim().isEmpty
            ? kDefaultRomId
            : engine.romId.trim();
        await engine.configure(
          {
            ...engine.enabled,
            ...SubaruPidLibrary.all.where((p) => p.extended).map((p) => p.id),
          },
          true,
          rom,
          engine.endian,
        );
        await saveSettings();
        message = 'ROM $rom подтверждён: 4-байтовые PID активны';
      });""",
    ["updateVehicleConfig", "enableExtendedPids", False],
))

# --- 4. model.dart: ROM по умолчанию ---
REWRITES.append((
    "lib/model.dart",
    "const Map<String, String> kRrNames = <String, String>{",
    """/// v0.12: ROM по умолчанию — профиль из identity.dart (адреса подтверждены
/// по logger-дефинициям RomRaider для ECU 5204584007).
const String kDefaultRomId = 'A2TB100B';

const Map<String, String> kRrNames = <String, String>{""",
    ["const String kDefaultRomId", False],
))

# --- 5. model.dart: restore — применять дефолты даже без файла настроек ---
REWRITES.append((
    "lib/model.dart",
    "      if (!await file.exists() || _disposed) return;",
    """      if (_disposed) return;
      if (!await file.exists()) {
        // v0.12: первый запуск — сразу открываем 4-байтовые PID по профилю ROM
        await engine.configure(
          {...SubaruPidLibrary.defaults},
          true,
          kDefaultRomId,
          engine.endian,
        );
        changed();
        return;
      }""",
    ["первый запуск — сразу открываем", False],
))

# --- 6. model.dart: restore — читать конфиги + ROM по умолчанию ---
REWRITES.append((
    "lib/model.dart",
    """      await engine.configure(
        ids,
        json['confirmed'] == true,
        json['rom'] as String? ?? '',
        json['endian'] == 'little' ? Endian.little : Endian.big,
      );""",
    """      final savedRom = (json['rom'] as String? ?? '').trim();
      await engine.configure(
        ids,
        json['confirmed'] as bool? ?? true,
        savedRom.isEmpty ? kDefaultRomId : savedRom,
        json['endian'] == 'little' ? Endian.little : Endian.big,
      );
      vehicleConfig =
          VehicleConfig.fromJson(json['vehicle'] as Map<String, dynamic>?);
      dashboardPrefs =
          DashboardPrefs.fromJson(json['dashboard'] as Map<String, dynamic>?);
      uiPrefs = UiPrefs.fromJson(json['ui'] as Map<String, dynamic>?);""",
    ["VehicleConfig.fromJson(json['vehicle']", False],
))

# --- 7. model.dart: saveSettings — писать конфиги ---
REWRITES.append((
    "lib/model.dart",
    "        'adapter': selected, // v0.10: MAC адаптера переживает перезапуск",
    """        'adapter': selected, // v0.10: MAC адаптера переживает перезапуск
        'vehicle': vehicleConfig.toJson(), // v0.12
        'dashboard': dashboardPrefs.toJson(),
        'ui': uiPrefs.toJson(),""",
    ["'vehicle': vehicleConfig.toJson()", False],
))

# --- 8. main.dart: импорты новых страниц ---
REWRITES.append((
    "lib/main.dart",
    "import 'engine.dart';\nimport 'dtc_service_page.dart';",
    "import 'dashboard_page.dart';\nimport 'dashboard_prefs.dart';\nimport 'engine.dart';\nimport 'dtc_service_page.dart';\nimport 'vehicle_config_page.dart';",
    ["import 'dashboard_page.dart';", False],
))

# --- 9. main.dart: включить все ориентации явно ---
REWRITES.append((
    "lib/main.dart",
    "void main() {\n  WidgetsFlutterBinding.ensureInitialized();\n  runApp(const SsmApp());",
    """void main() {
  WidgetsFlutterBinding.ensureInitialized();
  // v0.12 (04.3): книжная И альбомная — ориентацию выбирает пользователь
  // во вкладке «Мотор» (Авто / Только книжная / Только альбомная).
  SystemChrome.setPreferredOrientations(const <DeviceOrientation>[
    DeviceOrientation.portraitUp,
    DeviceOrientation.portraitDown,
    DeviceOrientation.landscapeLeft,
    DeviceOrientation.landscapeRight,
  ]);
  runApp(const SsmApp());""",
    ["DeviceOrientation.landscapeLeft", False],
))

# --- 10. main.dart: дашборд -> настраиваемый + вкладка «Мотор» ---
REWRITES.append((
    "lib/main.dart",
    "            DashboardPage(model),",
    "            CustomDashboardPage(model),",
    ["CustomDashboardPage(model)", False],
))
REWRITES.append((
    "lib/main.dart",
    "            DtcServicePage(model: model),\n            const MapLabTab(),",
    "            DtcServicePage(model: model),\n            VehicleConfigPage(model),\n            const MapLabTab(),",
    ["VehicleConfigPage(model)", False],
))
REWRITES.append((
    "lib/main.dart",
    """                NavigationDestination(
                    icon: Icon(Icons.build_circle_outlined), label: 'DTC'),""",
    """                NavigationDestination(
                    icon: Icon(Icons.build_circle_outlined), label: 'DTC'),
                NavigationDestination(
                    icon: Icon(Icons.settings_suggest), label: 'Мотор'),""",
    ["label: 'Мотор'", False],
))

# --- 11. main.dart: применение выбранной ориентации ---
REWRITES.append((
    "lib/main.dart",
    "          final pages = <Widget>[",
    """          // v0.12: пользовательский режим ориентации
          final mode = model.uiPrefs.orientation;
          SystemChrome.setPreferredOrientations(
            mode == OrientationMode.portrait
                ? const <DeviceOrientation>[
                    DeviceOrientation.portraitUp,
                    DeviceOrientation.portraitDown,
                  ]
                : mode == OrientationMode.landscape
                    ? const <DeviceOrientation>[
                        DeviceOrientation.landscapeLeft,
                        DeviceOrientation.landscapeRight,
                      ]
                    : const <DeviceOrientation>[
                        DeviceOrientation.portraitUp,
                        DeviceOrientation.portraitDown,
                        DeviceOrientation.landscapeLeft,
                        DeviceOrientation.landscapeRight,
                      ],
          );
          final pages = <Widget>[""",
    ["пользовательский режим ориентации", False],
))

# --- 12. main.dart: заголовок версии ---
REWRITES.append((
    "lib/main.dart",
    "SSM2 TELEMETRY 0.11",
    "SSM2 TELEMETRY 0.12",
    ["SSM2 TELEMETRY 0.12", True],
))

# --- 13. main.dart: PID-страница — кнопка разблокировки 4-байтовых ---
REWRITES.append((
    "lib/main.dart",
    """              TextButton(
                onPressed: m.busy ? null : () => _settings(context),
                child: const Text('ROM / Float32'),
              ),""",
    """              TextButton(
                onPressed: m.busy ? null : () => _settings(context),
                child: const Text('ROM / Float32'),
              ),
              // v0.12 (04.3): разблокировка ROM-зависимых PID одной кнопкой
              FilledButton.tonal(
                onPressed: m.busy ? null : m.enableExtendedPids,
                child: const Text('Все 4-байтовые*'),
              ),""",
    ["Все 4-байтовые*", False],
))
REWRITES.append((
    "lib/main.dart",
    "        section('Библиотека / 28 PID', [",
    "        section('Библиотека / ${SubaruPidLibrary.all.length} PID', [",
    ["Библиотека / ${SubaruPidLibrary.all.length} PID", False],
))
REWRITES.append((
    "lib/main.dart",
    """          const Text(
            'Сначала 8 базовых. Больше параметров — ниже частота каждого PID.',
            style: TextStyle(color: muted),
          ),""",
    """          Text(
            m.engine.extendedAllowed
                ? 'ROM ${m.engine.romId} подтверждён: 4-байтовые (*) доступны. '
                    'Больше PID — ниже частота каждого.'
                : '4-байтовые (*) PID заблокированы: нажмите «Все 4-байтовые*» '
                    'или укажите ROM ID в «ROM / Float32».',
            style: const TextStyle(color: muted),
          ),""",
    ["4-байтовые (*) PID заблокированы", False],
))

# --- 14. Map Lab: алиасы для новых каналов из CSV ---
REWRITES.append((
    "lib/maps_lab/maps_lab_log.dart",
    "    'wgdc': [",
    """    'idc': ['injector duty', 'idc', 'injector duty cycle'],
    'injpw': [
      'fuel injector #1 pulse width',
      'injector pulse width',
      'injpw',
    ],
    'knocksum': ['knock sum', 'knock_sum', 'knocksum'],
    'wgdc': [""",
    ["'knocksum'", False],
))

# ===== Запись, правки, самопроверка, чек-лист стенда =====
# ===== запись файлов =====
bad_keys = [rel for rel in FILES if "(" in rel or ")" in rel]
if bad_keys:
    raise RuntimeError(f"Невалидные FILES-ключи: {bad_keys}")
BACKUP.mkdir(parents=True, exist_ok=True)
for rel, body in FILES.items():
    dest = APP / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        shutil.copyfile(dest, BACKUP / (rel.replace("/", "__") + ".bak"))
    dest.write_text(body.strip("\n") + "\n", encoding="utf-8")
    print(f"[OK] {rel}")
print(f"[OK] файлов: {len(FILES)}")

# ===== точечные правки =====
patched, skipped = 0, 0
for rel, needle, replacement, sig in REWRITES:
    optional = False
    if sig and sig[-1] is True:
        sig, optional = sig[:-1], True
    elif sig and sig[-1] is False:
        sig, optional = sig[:-1], False
    path = APP / rel
    if not path.exists():
        raise RuntimeError(f"Нет файла {rel} — сначала прогоните 02–04.2")
    text = path.read_text(encoding="utf-8")
    # СНАЧАЛА сигнатура: правка уже применена -> ничего не делаем
    # (игла может сохраняться в тексте после вставки — тогда повтор дублировал бы код)
    if sig and all(s in text for s in sig):
        skipped += 1
        print(f"[SKIP уже применено] {rel}: {sig[0][:50]}")
        continue
    if needle in text:
        shutil.copyfile(path, BACKUP / (rel.replace("/", "__") + ".orig.bak"))
        path.write_text(text.replace(needle, replacement, 1), encoding="utf-8")
        patched += 1
        print(f"[OK] правка {rel}")
        continue
    if optional:
        skipped += 1
        print(f"[SKIP необязательная] {rel}")
        continue
    raise RuntimeError(
        f"Маркер не найден в {rel}:\n{needle[:100]!r}")
print(f"[OK] правок: {patched}, пропущено: {skipped}")

# ===== самопроверка =====
checks = {
    "lib/vehicle_config.dart": ["class VehicleConfig", "TurboKind", "injectorDutyFromPulse", "pumpHeadroomPct"],
    "lib/dashboard_prefs.dart": ["class DashboardPrefs", "OrientationMode", "class UiPrefs"],
    "lib/dashboard_page.dart": ["CustomDashboardPage", "ReorderableListView"],
    "lib/vehicle_config_page.dart": ["VehicleConfigPage", "buildConfigAdvice"],
    "lib/model.dart": ["const String kDefaultRomId", "enableExtendedPids", "updateVehicleConfig", "'REQ_TQ':"],
    "lib/main.dart": ["CustomDashboardPage", "VehicleConfigPage", "DeviceOrientation.landscapeLeft", "Все 4-байтовые*"],
    "lib/maps_lab/maps_lab_log.dart": ["'knocksum'", "'injpw'"],
    "test/config_test.dart": ["injectorDutyFromPulse", "DashboardPrefs"],
}
for rel, needles in checks.items():
    text = (APP / rel).read_text(encoding="utf-8")
    missing = [m for m in needles if m not in text]
    if missing:
        raise RuntimeError(f"Самопроверка {rel}: нет маркеров {missing}")
print("[OK] самопроверка маркеров пройдена")

print("\n=== Готово: SSM2 0.12. Далее — ячейка 05 (сборка). ===")
print("Что нового:")
print("  · 4-байтовые PID разблокированы автоматически (ROM A2TB100B)")
print("  · вкладка «Мотор»: форсунки, насос, турбо, топливо + живые расчёты IDC")
print("  · дашборд настраивается: состав, порядок, колонки, компактность")
print("  · ориентация: авто / книжная / альбомная")
print("Чек-лист стенда 04.3:")
print(" [ ] на дашборде видны FBKC/IAM/REQ_TQ (4-байтовые)")
print(" [ ] CSV-экспорт содержит колонки Requested Torque* и Knock Sum*")
print(" [ ] поворот телефона — раскладка перестраивается, колонок больше")
print(" [ ] вкладка «Мотор»: IDC растёт с газом, потолки мощности адекватны")
print(" [ ] перетаскивание плиток сохраняется после перезапуска приложения")


[OK] lib/vehicle_config.dart
[OK] lib/dashboard_prefs.dart
[OK] lib/dashboard_page.dart
[OK] lib/vehicle_config_page.dart
[OK] test/config_test.dart
[OK] файлов: 5
[OK] правка lib/model.dart
[OK] правка lib/model.dart
[OK] правка lib/model.dart
[OK] правка lib/model.dart
[OK] правка lib/model.dart
[OK] правка lib/model.dart
[OK] правка lib/model.dart
[OK] правка lib/model.dart
[OK] правка lib/main.dart
[OK] правка lib/main.dart
[OK] правка lib/main.dart
[OK] правка lib/main.dart
[OK] правка lib/main.dart
[OK] правка lib/main.dart
[OK] правка lib/main.dart
[OK] правка lib/main.dart
[OK] правка lib/main.dart
[OK] правка lib/main.dart
[OK] правка lib/maps_lab/maps_lab_log.dart
[OK] правок: 19, пропущено: 0
[OK] самопроверка маркеров пройдена

=== Готово: SSM2 0.12. Далее — ячейка 05 (сборка). ===
Что нового:
  · 4-байтовые PID разблокированы автоматически (ROM A2TB100B)
  · вкладка «Мотор»: форсунки, насос, турбо, топливо + живые расчёты IDC
  · дашборд настраивается: состав, порядок, кол

In [ ]:
# ===== Шапка 04.4: что чинится и добавляется =====
# @title 04.4 | SERVICE FULL & TILES | SSM2 0.12 >> 0.13 (вставить МЕЖДУ 04.3 и 05)
# Исправляет и добавляет:
#  1. БАГ 04.2: EXPERIMENTAL был compile-time const -> ECUReset/Clear Memory
#     всегда серые. Теперь рантайм-переключатель в UI + сохранение в настройках.
#  2. UDS-сбросы требуют расширенной сессии (10 03) — раньше её не было,
#     блок отвечал 7F ... 7F/24. Теперь сессия открывается автоматически.
#  3. Полный набор сервисов: pending (07), permanent (0A), стоп-кадр (02),
#     мониторы готовности (01 01) с расшифровкой, VIN (09 02 / 22 F1 90),
#     CALID (09 04), TesterPresent (3E 00), скан всех блоков, проба групп
#     Clear Memory, расшифровка всех негативных ответов (NRC).
#  4. Виртуальные плитки дашборда: мгновенный расход л/ч и л/100км, одометр
#     поездки, литры, средний расход, IDC форсунок, запас насоса, оценка
#     мощности и момента, лямбда, суммарная коррекция детонации, дельта наддува.
# Идемпотентна, бэкап снаружи проекта, самопроверка маркеров.

import json
import re
import shutil
import time
from pathlib import Path

APP = Path("/content/subaru_ssm2_fixed")

STAMP = time.strftime("%Y%m%d_%H%M%S")
BACKUP = APP.parent / f"ssm2_backup_044_{STAMP}"
FILES = {}
REWRITES = []

# ===== Полный сервисный движок: pending/permanent, стоп-кадр, мониторы, VIN, сессии, NRC =====
FILES["lib/diag.dart"] = r'''
import 'dtc_dict.dart';
import 'elm.dart';
import 'protocol.dart';

/// Поколение протокола ECU.
enum DiagProto { ssm2, obd, uds }

/// Блок управления Subaru на шине CAN.
class EcuSlot {
  const EcuSlot(this.name, this.header, this.responseId, this.proto);
  final String name;
  final String header; // ATSH
  final String responseId; // ATCRA
  final DiagProto proto;

  static const engine = EcuSlot('Двигатель (ECM)', '7E0', '7E8', DiagProto.ssm2);
  static const tcu = EcuSlot('АКПП / CVT (TCM)', '7E1', '7E9', DiagProto.ssm2);
  static const vdc = EcuSlot('ABS / VDC', '7B0', '7B8', DiagProto.uds);
  static const srs = EcuSlot('Подушки (SRS)', '772', '77A', DiagProto.uds);
  static const eps = EcuSlot('ЭУР (EPS)', '7A6', '7AE', DiagProto.uds);
  static const biu = EcuSlot('Кузовной блок (BIU)', '723', '72B', DiagProto.uds);

  static const List<EcuSlot> all = [engine, tcu, vdc, srs, eps, biu];
}

/// Одна запись неисправности.
class DiagDtc {
  DiagDtc(this.code, this.statusByte, this.proto, {this.origin = ''});
  final String code;
  final int statusByte;
  final DiagProto proto;

  /// Откуда прочитан код: stored / pending / permanent.
  final String origin;

  bool get active => proto == DiagProto.uds
      ? (statusByte & 0x01) != 0
      : proto == DiagProto.obd
          ? origin != 'permanent'
          : (statusByte & 0x20) != 0;
  bool get confirmed => proto == DiagProto.uds ? (statusByte & 0x08) != 0 : true;
  bool get milOn => proto == DiagProto.uds
      ? (statusByte & 0x40) != 0
      : proto == DiagProto.obd
          ? false
          : (statusByte & 0x80) != 0;

  String get statusText {
    final prefix = origin.isEmpty ? '' : '$origin · ';
    if (proto == DiagProto.uds) {
      final f = <String>[];
      if (statusByte & 0x01 != 0) f.add('есть сейчас');
      if (statusByte & 0x08 != 0) f.add('подтверждена');
      if (statusByte & 0x40 != 0) f.add('MIL');
      return prefix +
          (f.isEmpty ? 'пассивная (0x${hex2(statusByte)})' : f.join(' · '));
    }
    if (proto == DiagProto.obd) return '${prefix}OBD mode';
    return prefix + (active ? (milOn ? 'активная · MIL' : 'активная') : 'сохранённая');
  }

  String get titleRu => kSubaruDtcDict[code]?.$1 ?? '';
}

/// Статус мониторов готовности OBD (mode 01 PID 01).
class ReadinessReport {
  ReadinessReport({
    required this.milOn,
    required this.dtcCount,
    required this.monitors,
  });
  final bool milOn;
  final int dtcCount;

  /// имя монитора -> готов ли (true = завершён)
  final Map<String, bool> monitors;

  int get ready => monitors.values.where((v) => v).length;
  int get total => monitors.length;
}

/// Декодер двухбайтового DTC.
String decodeDtcBytes(int b1, int b2) {
  const systems = ['P', 'C', 'B', 'U'];
  final sys = systems[(b1 >> 6) & 0x03];
  final d1 = (b1 >> 4) & 0x03;
  String h(int v) => v.toRadixString(16).toUpperCase();
  return '$sys$d1${h(b1 & 0x0F)}${h((b2 >> 4) & 0x0F)}${h(b2 & 0x0F)}';
}

/// SSM2: 58 <count> <b1> <b2> <status> × n
List<DiagDtc> parseSsm2DtcReply(List<int> bytes) {
  final out = <DiagDtc>[];
  if (bytes.isEmpty || bytes[0] != 0x58) return out;
  final count = bytes.length > 1 ? bytes[1] : 0;
  for (var i = 0; i < count; i++) {
    final base = 2 + i * 3;
    if (base + 2 >= bytes.length) break;
    final b1 = bytes[base], b2 = bytes[base + 1], st = bytes[base + 2];
    if (b1 == 0 && b2 == 0) continue;
    out.add(DiagDtc(decodeDtcBytes(b1, b2), st, DiagProto.ssm2, origin: 'stored'));
  }
  return out;
}

/// OBD mode 03/07/0A: <0x43|0x47|0x4A> <b1> <b2> × n
List<DiagDtc> parseObdDtcReply(List<int> bytes, {String origin = 'stored'}) {
  const heads = <int, String>{0x43: 'stored', 0x47: 'pending', 0x4A: 'permanent'};
  final out = <DiagDtc>[];
  if (bytes.isEmpty || !heads.containsKey(bytes[0])) return out;
  final kind = heads[bytes[0]] ?? origin;
  for (var i = 1; i + 1 < bytes.length; i += 2) {
    final b1 = bytes[i], b2 = bytes[i + 1];
    if (b1 == 0 && b2 == 0) continue;
    out.add(DiagDtc(decodeDtcBytes(b1, b2), 0x20, DiagProto.obd, origin: kind));
  }
  return out;
}

/// Совместимость с 04.2: алиас старого имени.
List<DiagDtc> parseObdMode3Reply(List<int> bytes) => parseObdDtcReply(bytes);

/// UDS 19 02: 59 02 <mask> <b1> <b2> <status> × n
List<DiagDtc> parseUdsDtcReply(List<int> bytes) {
  final out = <DiagDtc>[];
  if (bytes.length < 3 || bytes[0] != 0x59) return out;
  for (var i = 3; i + 2 < bytes.length; i += 3) {
    final b1 = bytes[i], b2 = bytes[i + 1], st = bytes[i + 2];
    if (b1 == 0 && b2 == 0) continue;
    out.add(DiagDtc(decodeDtcBytes(b1, b2), st, DiagProto.uds, origin: 'stored'));
  }
  return out;
}

/// mode 01 PID 01 -> 41 01 A B C D
ReadinessReport? parseReadiness(List<int> bytes) {
  if (bytes.length < 6 || bytes[0] != 0x41 || bytes[1] != 0x01) return null;
  final a = bytes[2], b = bytes[3], c = bytes[4], d = bytes[5];
  final monitors = <String, bool>{
    // B: непрерывные мониторы (bit 0..2 доступен, bit 4..6 НЕ завершён)
    'Пропуски зажигания': (b & 0x01) == 0 || (b & 0x10) == 0,
    'Топливная система': (b & 0x02) == 0 || (b & 0x20) == 0,
    'Компоненты': (b & 0x04) == 0 || (b & 0x40) == 0,
  };
  // C: доступность, D: незавершённость (бит=1 -> НЕ готов)
  const names = <int, String>{
    0x01: 'Катализатор',
    0x02: 'Подогрев катализатора',
    0x04: 'EVAP (адсорбер)',
    0x08: 'Вторичный воздух',
    0x20: 'Датчики O2',
    0x40: 'Подогрев O2',
    0x80: 'EGR / VVT',
  };
  names.forEach((mask, label) {
    if ((c & mask) != 0) monitors[label] = (d & mask) == 0;
  });
  return ReadinessReport(
    milOn: (a & 0x80) != 0,
    dtcCount: a & 0x7F,
    monitors: monitors,
  );
}

/// mode 09: 49 <pid> <count> <ascii...> — VIN (02) / CALID (04)
String? parseAsciiReply(List<int> bytes, int pid) {
  if (bytes.length < 3 || bytes[0] != 0x49 || bytes[1] != pid) return null;
  final chars = <int>[];
  for (var i = 3; i < bytes.length; i++) {
    final b = bytes[i];
    if (b >= 0x20 && b <= 0x7E) chars.add(b);
  }
  final text = String.fromCharCodes(chars).trim();
  return text.isEmpty ? null : text;
}

/// mode 02: 42 02 00 <b1> <b2> — DTC, вызвавший стоп-кадр.
String? parseFreezeFrameDtc(List<int> bytes) {
  if (bytes.length < 5 || bytes[0] != 0x42) return null;
  final b1 = bytes[3], b2 = bytes[4];
  if (b1 == 0 && b2 == 0) return null;
  return decodeDtcBytes(b1, b2);
}

/// Универсальный парсер «сырого» ответа ELM.
List<int>? parseDiagPayload(String text) {
  final flat = <int>[];
  for (final rawLine in text.split(RegExp(r'[\r\n]+'))) {
    var line = rawLine.trim();
    if (line.isEmpty) continue;
    if (RegExp(r'^\d+:').hasMatch(line)) {
      line = line.substring(line.indexOf(':') + 1).trim();
    }
    for (final t in line.split(RegExp(r'\s+')).where((x) => x.isNotEmpty)) {
      if (!RegExp(r'^[0-9A-Fa-f]{2,16}$').hasMatch(t) || t.length % 2 != 0) {
        return null;
      }
      for (var i = 0; i + 1 < t.length; i += 2) {
        flat.add(int.parse(t.substring(i, i + 2), radix: 16));
      }
    }
  }
  if (flat.isEmpty) return null;
  var b = flat;
  if (b.length > 1 && (b[0] & 0xF0) == 0 && (b[0] & 0x0F) <= 7) {
    final len = b[0] & 0x0F;
    b = b.sublist(1, 1 + len <= b.length ? 1 + len : b.length);
  }
  if (b.length >= 3 && b[0] == 0x07 && (b[1] & 0xF8) == 0xE8) {
    b = b.sublist(2);
  }
  return b.isEmpty ? null : b;
}

const Map<int, String> kNrMeanings = <int, String>{
  0x10: 'generalReject — блок отклонил запрос',
  0x11: 'serviceNotSupported — сервис не поддерживается',
  0x12: 'subFunctionNotSupported — подфункция не поддерживается',
  0x13: 'incorrectMessageLength — неверная длина',
  0x21: 'busyRepeatRequest — блок занят',
  0x22: 'conditionsNotCorrect — не выполнены условия (зажигание/обороты)',
  0x24: 'requestSequenceError — нужна сессия 10 03',
  0x31: 'requestOutOfRange — параметр вне диапазона',
  0x33: 'securityAccessDenied — нужен ключ дилера',
  0x7E: 'subFunctionNotSupportedInActiveSession',
  0x7F: 'serviceNotSupportedInActiveSession — нужна сессия 10 03',
  0x78: 'responsePending — блок думает',
};

/// Результат сервисной операции для UI.
class ServiceResult {
  ServiceResult.ok(this.title, this.detail)
      : success = true,
        negative = null;
  ServiceResult.fail(this.title, this.detail, {this.negative}) : success = false;
  final bool success;
  final String title;
  final String detail;
  final int? negative;
}

/// Сессия диагностики поверх ElmDriver.diagnostic.
class DiagSession {
  DiagSession(this.elm);
  final ElmDriver elm;

  /// v0.13: рантайм-флаг (раньше был compile-time const и не включался).
  bool experimental = false;

  Future<void> _atEcu(EcuSlot slot) async {
    await elm.diagnostic('ATSH ${slot.header}');
    await elm.diagnostic('ATCRA ${slot.responseId}');
    await Future<void>.delayed(const Duration(milliseconds: 20));
  }

  Future<List<int>> _payload(String command) async {
    final reply = await elm.diagnostic(command);
    return parseDiagPayload(reply) ?? const <int>[];
  }

  int? _negativeCode(List<int> b) =>
      (b.length >= 3 && b[0] == 0x7F) ? b[2] : null;

  // ------------------------- ЧТЕНИЕ ОШИБОК -------------------------

  /// Сохранённые DTC: SSM2 0x18 -> OBD 03 -> UDS 19 02 AF.
  Future<List<DiagDtc>> readDtc(EcuSlot slot) async {
    await _atEcu(slot);
    if (slot.proto == DiagProto.ssm2) {
      try {
        final b = await _payload('18 00 FF 00');
        if (b.isNotEmpty && b[0] == 0x58) return parseSsm2DtcReply(b);
      } catch (_) {}
      try {
        return parseObdDtcReply(await _payload('03'));
      } catch (_) {}
      return const [];
    }
    final b = await _payload('19 02 AF');
    if (b.isNotEmpty && b[0] == 0x59) return parseUdsDtcReply(b);
    try {
      return parseObdDtcReply(await _payload('03'));
    } catch (_) {
      return const [];
    }
  }

  /// Текущие (pending) — OBD mode 07.
  Future<List<DiagDtc>> readPendingDtc(EcuSlot slot) async {
    await _atEcu(slot);
    return parseObdDtcReply(await _payload('07'), origin: 'pending');
  }

  /// Постоянные (permanent) — OBD mode 0A. Не стираются сбросом!
  Future<List<DiagDtc>> readPermanentDtc(EcuSlot slot) async {
    await _atEcu(slot);
    return parseObdDtcReply(await _payload('0A'), origin: 'permanent');
  }

  /// Все три категории разом.
  Future<List<DiagDtc>> readAllDtc(EcuSlot slot) async {
    final out = <DiagDtc>[...await readDtc(slot)];
    for (final fn in [readPendingDtc, readPermanentDtc]) {
      try {
        for (final d in await fn(slot)) {
          if (!out.any((x) => x.code == d.code && x.origin == d.origin)) {
            out.add(d);
          }
        }
      } catch (_) {}
    }
    return out;
  }

  /// Обход всех блоков: имя -> список кодов (или ошибка).
  Future<Map<String, List<DiagDtc>>> scanAllEcus({
    void Function(String name)? onProgress,
  }) async {
    final result = <String, List<DiagDtc>>{};
    for (final slot in EcuSlot.all) {
      onProgress?.call(slot.name);
      try {
        result[slot.name] = await readDtc(slot);
      } catch (_) {
        // блок не ответил — это норма для части поколений
      }
    }
    await restore();
    return result;
  }

  // ------------------------- СБРОС ОШИБОК -------------------------

  Future<List<DiagDtc>> clearDtc(EcuSlot slot) async {
    await _atEcu(slot);
    var done = false;
    if (slot.proto == DiagProto.ssm2) {
      final b = await _payload('14 FF FF 00');
      done = b.isNotEmpty && b[0] == 0x54;
      if (!done) {
        final o = await _payload('04');
        done = o.isNotEmpty && o[0] == 0x44;
      }
    } else {
      final b = await _payload('14 FF FF FF');
      done = b.isNotEmpty && b[0] == 0x54;
      if (!done) {
        final o = await _payload('04');
        done = o.isNotEmpty && o[0] == 0x44;
      }
    }
    if (!done) throw ReplyError('ECU отказал в сбросе DTC');
    await Future<void>.delayed(const Duration(milliseconds: 400));
    return readDtc(slot);
  }

  // ------------------------- ИНФОРМАЦИЯ -------------------------

  Future<ReadinessReport?> readReadiness(EcuSlot slot) async {
    await _atEcu(slot);
    return parseReadiness(await _payload('01 01'));
  }

  Future<String?> readVin(EcuSlot slot) async {
    await _atEcu(slot);
    final v = parseAsciiReply(await _payload('09 02'), 0x02);
    if (v != null) return v;
    // UDS ReadDataByIdentifier F190
    final b = await _payload('22 F1 90');
    if (b.length > 3 && b[0] == 0x62) {
      final chars = b
          .sublist(3)
          .where((x) => x >= 0x20 && x <= 0x7E)
          .toList();
      final text = String.fromCharCodes(chars).trim();
      return text.isEmpty ? null : text;
    }
    return null;
  }

  /// CALID прошивки (mode 09 PID 04).
  Future<String?> readCalId(EcuSlot slot) async {
    await _atEcu(slot);
    return parseAsciiReply(await _payload('09 04'), 0x04);
  }

  /// DTC, вызвавший стоп-кадр (mode 02).
  Future<String?> readFreezeFrameDtc(EcuSlot slot) async {
    await _atEcu(slot);
    return parseFreezeFrameDtc(await _payload('02 02 00'));
  }

  // ------------------------- СЕРВИСНЫЕ -------------------------

  /// UDS DiagnosticSessionControl: расширенная сессия (нужна для 11 01 / 04 xx).
  Future<bool> enterExtendedSession(EcuSlot slot) async {
    await _atEcu(slot);
    final b = await _payload('10 03');
    return b.isNotEmpty && b[0] == 0x50;
  }

  /// Keep-alive, чтобы сессия не отвалилась.
  Future<void> testerPresent() async {
    try {
      await _payload('3E 00');
    } catch (_) {}
  }

  /// UDS ECUReset 11 01 — ГЛУШИТ двигатель.
  Future<ServiceResult> ecuReset(EcuSlot slot) async {
    if (!experimental) {
      return ServiceResult.fail('ECUReset заблокирован',
          'Включите «Экспертные операции» в настройках сервисной страницы.');
    }
    await enterExtendedSession(slot);
    final b = await _payload('11 01');
    final nr = _negativeCode(b);
    if (nr != null) {
      return ServiceResult.fail('ECUReset отклонён',
          kNrMeanings[nr] ?? 'NRC 0x${hex2(nr)}',
          negative: nr);
    }
    if (b.isNotEmpty && b[0] == 0x51) {
      return ServiceResult.ok('ECU перезапущен',
          'Блок ответил 51 01. Поверните зажигание OFF→ON перед запуском.');
    }
    return ServiceResult.fail(
        'Нет подтверждения', 'Ожидался ответ 51 01, получено: ${_hex(b)}');
  }

  /// SSM3 Clear Memory 04 gr — сброс адаптаций (группы зависят от ECU).
  Future<ServiceResult> clearMemory(EcuSlot slot, int group) async {
    if (!experimental) {
      return ServiceResult.fail('Clear Memory заблокирован',
          'Включите «Экспертные операции» в настройках сервисной страницы.');
    }
    if (group < 1 || group > 7) {
      return ServiceResult.fail('Неверная группа', 'Допустимо 1..7');
    }
    await enterExtendedSession(slot);
    final b = await _payload('04 ${hex2(group)}');
    final nr = _negativeCode(b);
    if (nr != null) {
      return ServiceResult.fail('Группа $group отклонена',
          kNrMeanings[nr] ?? 'NRC 0x${hex2(nr)}',
          negative: nr);
    }
    if (b.isNotEmpty && b[0] == 0x44) {
      return ServiceResult.ok('Адаптации группы $group сброшены',
          'Блок подтвердил (44). Нужно дообучение: 15–20 мин спокойной езды.');
    }
    return ServiceResult.fail(
        'Нет подтверждения', 'Ожидался 44, получено: ${_hex(b)}');
  }

  /// Проба всех групп Clear Memory: какие поддерживает ваш ECU.
  Future<Map<int, String>> probeClearMemoryGroups(EcuSlot slot) async {
    final out = <int, String>{};
    if (!experimental) return out;
    await enterExtendedSession(slot);
    for (var g = 1; g <= 7; g++) {
      try {
        final b = await _payload('04 ${hex2(g)}');
        final nr = _negativeCode(b);
        out[g] = nr != null
            ? (kNrMeanings[nr] ?? 'NRC 0x${hex2(nr)}')
            : (b.isNotEmpty && b[0] == 0x44 ? 'поддерживается (44)' : 'нет ответа');
      } catch (e) {
        out[g] = 'ошибка связи';
      }
      await Future<void>.delayed(const Duration(milliseconds: 120));
    }
    return out;
  }

  Future<void> restore() async {
    try {
      await elm.diagnostic('ATSH 7E0');
      await elm.diagnostic('ATCRA 7E8');
    } catch (_) {}
  }

  String _hex(List<int> b) =>
      b.isEmpty ? '(пусто)' : b.map(hex2).join(' ');
}
'''

# ===== Расчётные плитки: расход, одометр, IDC, запас насоса, мощность =====
FILES["lib/virtual_tiles.dart"] = r'''
import 'vehicle_config.dart';

/// Префикс идентификатора виртуальной (расчётной) плитки.
const String kVirtualPrefix = '@';

/// Входные данные для расчёта виртуальных плиток.
class TileInputs {
  const TileInputs({
    required this.values,
    required this.config,
    required this.tripKm,
    required this.tripLitres,
  });

  /// id PID -> текущее значение
  final Map<String, double> values;
  final VehicleConfig config;
  final double tripKm;
  final double tripLitres;

  double? v(String id) {
    final x = values[id];
    return (x == null || !x.isFinite) ? null : x;
  }

  /// наддув: 4-байтовый, иначе стандартный
  double? get boost => v('BOOST') ?? v('MAP_REL');

  /// расход топлива, г/с (MAF / AFR)
  double? get fuelGramsPerSec {
    final maf = v('MAF');
    if (maf == null || maf <= 0) return null;
    final afr = v('AFR');
    final ratio = (afr != null && afr >= 8 && afr <= 25) ? afr : config.fuel.stoich;
    return maf / ratio;
  }
}

class VirtualTileDef {
  const VirtualTileDef({
    required this.id,
    required this.label,
    required this.unit,
    required this.digits,
    required this.compute,
    this.hint = '',
  });
  final String id, label, unit, hint;
  final int digits;
  final double? Function(TileInputs) compute;
}

/// Плотность бензина, г/л.
const double _rho = 745.0;

final List<VirtualTileDef> kVirtualTiles = <VirtualTileDef>[
  VirtualTileDef(
    id: '@FUEL_LPH',
    label: 'Расход',
    unit: 'л/ч',
    digits: 1,
    hint: 'Мгновенный: MAF / AFR × 3600 / 745',
    compute: (i) {
      final g = i.fuelGramsPerSec;
      return g == null ? null : g * 3600 / _rho;
    },
  ),
  VirtualTileDef(
    id: '@FUEL_L100',
    label: 'Расход',
    unit: 'л/100км',
    digits: 1,
    hint: 'Мгновенный на скорости ≥ 5 км/ч',
    compute: (i) {
      final g = i.fuelGramsPerSec;
      final sp = i.v('SPEED');
      if (g == null || sp == null || sp < 5) return null;
      return g * 3600 / _rho / sp * 100;
    },
  ),
  VirtualTileDef(
    id: '@TRIP_KM',
    label: 'Одометр',
    unit: 'км',
    digits: 2,
    hint: 'Пробег с момента сброса поездки',
    compute: (i) => i.tripKm,
  ),
  VirtualTileDef(
    id: '@TRIP_L',
    label: 'Залито за поездку',
    unit: 'л',
    digits: 2,
    hint: 'Израсходовано топлива с момента сброса',
    compute: (i) => i.tripLitres,
  ),
  VirtualTileDef(
    id: '@TRIP_AVG',
    label: 'Средний расход',
    unit: 'л/100км',
    digits: 1,
    hint: 'Поездка: литры / км × 100',
    compute: (i) =>
        i.tripKm < 0.1 ? null : i.tripLitres / i.tripKm * 100,
  ),
  VirtualTileDef(
    id: '@IDC',
    label: 'Загрузка форсунок',
    unit: '%',
    digits: 0,
    hint: 'По ширине импульса, иначе по MAF/AFR. Предел 85%',
    compute: (i) {
      final pw = i.v('INJPW');
      final rpm = i.v('RPM');
      if (pw != null && rpm != null && rpm > 500) {
        return injectorDutyFromPulse(pulseWidthMs: pw, rpm: rpm);
      }
      final maf = i.v('MAF');
      final afr = i.v('AFR');
      if (maf == null || afr == null || afr < 5) return null;
      return injectorDutyFromMaf(
          mafGramsPerSec: maf, afr: afr, config: i.config);
    },
  ),
  VirtualTileDef(
    id: '@PUMP',
    label: 'Запас насоса',
    unit: '%',
    digits: 0,
    hint: 'Сколько ещё может дать бензонасос',
    compute: (i) {
      final maf = i.v('MAF');
      final afr = i.v('AFR');
      if (maf == null || afr == null || afr < 5) return null;
      return pumpHeadroomPct(
          mafGramsPerSec: maf, afr: afr, config: i.config);
    },
  ),
  VirtualTileDef(
    id: '@POWER',
    label: 'Мощность (оценка)',
    unit: 'л.с.',
    digits: 0,
    hint: 'MAF × 1.32 — грубая оценка по расходу воздуха',
    compute: (i) {
      final maf = i.v('MAF');
      return maf == null ? null : maf * 1.32;
    },
  ),
  VirtualTileDef(
    id: '@TORQUE',
    label: 'Момент (оценка)',
    unit: 'Н·м',
    digits: 0,
    hint: 'Из оценки мощности и оборотов',
    compute: (i) {
      final maf = i.v('MAF');
      final rpm = i.v('RPM');
      if (maf == null || rpm == null || rpm < 500) return null;
      final hp = maf * 1.32;
      return hp * 5252 / rpm * 1.3558;
    },
  ),
  VirtualTileDef(
    id: '@LAMBDA',
    label: 'Лямбда',
    unit: 'λ',
    digits: 3,
    hint: 'AFR / стехиометрия выбранного топлива',
    compute: (i) {
      final afr = i.v('AFR');
      return afr == null ? null : afr / i.config.fuel.stoich;
    },
  ),
  VirtualTileDef(
    id: '@KNOCK_TOT',
    label: 'Суммарная коррекция',
    unit: '°',
    digits: 2,
    hint: 'FBKC + FKL: полная поправка зажигания по детонации',
    compute: (i) {
      final fb = i.v('FBKC');
      final fk = i.v('FKL');
      if (fb == null && fk == null) return null;
      return (fb ?? 0) + (fk ?? 0);
    },
  ),
  VirtualTileDef(
    id: '@BOOST_DELTA',
    label: 'Отклонение наддува',
    unit: 'бар',
    digits: 3,
    hint: 'Факт минус цель (BOOST − BOOST_TGT)',
    compute: (i) {
      final b = i.boost;
      final t = i.v('BOOST_TGT');
      if (b == null || t == null) return null;
      return b - t;
    },
  ),
  VirtualTileDef(
    id: '@BOOST_PCT',
    label: 'Наддув от лимита',
    unit: '%',
    digits: 0,
    hint: 'Доля вашего лимита наддува из конфигурации мотора',
    compute: (i) {
      final b = i.boost;
      if (b == null || i.config.maxSafeBoostBar <= 0) return null;
      return b / i.config.maxSafeBoostBar * 100;
    },
  ),
];

VirtualTileDef? virtualTileById(String id) {
  for (final t in kVirtualTiles) {
    if (t.id == id) return t;
  }
  return null;
}

bool isVirtualTile(String id) => id.startsWith(kVirtualPrefix);
'''

# ===== Префы дашборда с поддержкой виртуальных плиток =====
FILES["lib/dashboard_prefs.dart"] = r'''
import 'pids.dart';
import 'virtual_tiles.dart';

/// Настройки дашборда: состав плиток (включая расчётные), порядок, плотность.
class DashboardPrefs {
  DashboardPrefs({
    List<String>? tiles,
    this.columnsPortrait = 2,
    this.columnsLandscape = 4,
    this.compact = false,
    this.showSpark = true,
    this.showFuelCard = true,
  }) : tiles = tiles ?? List<String>.from(defaultTiles);

  final List<String> tiles;
  final int columnsPortrait;
  final int columnsLandscape;
  final bool compact;
  final bool showSpark;
  final bool showFuelCard;

  /// v0.13: по умолчанию сразу показываем расход и одометр.
  static const List<String> defaultTiles = <String>[
    'RPM',
    'MAP_REL',
    'AFR',
    '@FUEL_L100',
    '@TRIP_KM',
    'ECT',
    'FBKC',
    'IAM',
  ];

  static bool knownTile(String id) =>
      isVirtualTile(id)
          ? virtualTileById(id) != null
          : SubaruPidLibrary.all.any((p) => p.id == id);

  DashboardPrefs copyWith({
    List<String>? tiles,
    int? columnsPortrait,
    int? columnsLandscape,
    bool? compact,
    bool? showSpark,
    bool? showFuelCard,
  }) =>
      DashboardPrefs(
        tiles: tiles ?? List<String>.from(this.tiles),
        columnsPortrait: columnsPortrait ?? this.columnsPortrait,
        columnsLandscape: columnsLandscape ?? this.columnsLandscape,
        compact: compact ?? this.compact,
        showSpark: showSpark ?? this.showSpark,
        showFuelCard: showFuelCard ?? this.showFuelCard,
      );

  Map<String, dynamic> toJson() => <String, dynamic>{
        'tiles': tiles,
        'columnsPortrait': columnsPortrait,
        'columnsLandscape': columnsLandscape,
        'compact': compact,
        'showSpark': showSpark,
        'showFuelCard': showFuelCard,
      };

  static DashboardPrefs fromJson(Map<String, dynamic>? json) {
    if (json == null) return DashboardPrefs();
    final raw = (json['tiles'] as List<dynamic>? ?? const <dynamic>[])
        .whereType<String>()
        .where(knownTile)
        .toList();
    int clampCols(Object? v, int fallback) {
      final n = v is num ? v.toInt() : fallback;
      return n < 1 ? 1 : (n > 6 ? 6 : n);
    }

    return DashboardPrefs(
      tiles: raw.isEmpty ? null : raw,
      columnsPortrait: clampCols(json['columnsPortrait'], 2),
      columnsLandscape: clampCols(json['columnsLandscape'], 4),
      compact: json['compact'] == true,
      showSpark: json['showSpark'] != false,
      showFuelCard: json['showFuelCard'] != false,
    );
  }

  /// Видимые плитки: PID должен быть активен, виртуальные доступны всегда.
  List<String> visibleTiles(Set<String> enabledPids) =>
      tiles.where((id) => isVirtualTile(id) || enabledPids.contains(id)).toList();
}

enum OrientationMode {
  auto('Авто (по датчику)'),
  portrait('Только книжная'),
  landscape('Только альбомная');

  const OrientationMode(this.label);
  final String label;
}

class UiPrefs {
  const UiPrefs({this.orientation = OrientationMode.auto, this.keepAwake = true});
  final OrientationMode orientation;
  final bool keepAwake;

  UiPrefs copyWith({OrientationMode? orientation, bool? keepAwake}) => UiPrefs(
        orientation: orientation ?? this.orientation,
        keepAwake: keepAwake ?? this.keepAwake,
      );

  Map<String, dynamic> toJson() => <String, dynamic>{
        'orientation': orientation.name,
        'keepAwake': keepAwake,
      };

  static UiPrefs fromJson(Map<String, dynamic>? json) {
    if (json == null) return const UiPrefs();
    return UiPrefs(
      orientation: OrientationMode.values.firstWhere(
        (o) => o.name == json['orientation'],
        orElse: () => OrientationMode.auto,
      ),
      keepAwake: json['keepAwake'] != false,
    );
  }
}
'''

# ===== Дашборд: PID + расчётные плитки, сброс поездки, альбомный режим =====
FILES["lib/dashboard_page.dart"] = r'''
import 'package:flutter/material.dart';

import 'dashboard_prefs.dart';
import 'model.dart';
import 'pids.dart';
import 'samples.dart';
import 'virtual_tiles.dart';

const _muted = Color(0xFF8CA0BF);

/// Настраиваемый дашборд: PID + расчётные плитки (расход, одометр, IDC...).
class CustomDashboardPage extends StatefulWidget {
  const CustomDashboardPage(this.model, {super.key});
  final AppModel model;

  @override
  State<CustomDashboardPage> createState() => _CustomDashboardPageState();
}

class _CustomDashboardPageState extends State<CustomDashboardPage> {
  AppModel get model => widget.model;

  TileInputs _inputs() {
    final values = <String, double>{};
    model.engine.latest.forEach((id, sample) {
      final v = sample.value;
      if (v != null && v.isFinite) values[id] = v;
    });
    return TileInputs(
      values: values,
      config: model.vehicleConfig,
      tripKm: model.engine.tripDistanceKm,
      tripLitres: model.engine.tripFuelLitres,
    );
  }

  @override
  Widget build(BuildContext context) {
    final engine = model.engine;
    final prefs = model.dashboardPrefs;
    final activeIds = engine.active.map((p) => p.id).toSet();

    final chosen = prefs.tiles
        .where((id) => isVirtualTile(id) || activeIds.contains(id))
        .toList();
    final rest = engine.active
        .map((p) => p.id)
        .where((id) => !chosen.contains(id));
    final ids = <String>[...chosen, ...rest];

    final isLandscape =
        MediaQuery.of(context).orientation == Orientation.landscape;
    final columns = isLandscape ? prefs.columnsLandscape : prefs.columnsPortrait;
    final inputs = _inputs();

    return Column(
      children: [
        Padding(
          padding: const EdgeInsets.fromLTRB(12, 4, 12, 0),
          child: Row(
            children: [
              Expanded(
                child: Text(
                  '${ids.length} плиток · ${engine.pidReadsPerSecond.toStringAsFixed(1)} обновл./с'
                  '${isLandscape ? ' · альбомная' : ''}',
                  style: const TextStyle(fontSize: 12, color: _muted),
                ),
              ),
              IconButton(
                tooltip: 'Сбросить поездку',
                onPressed: () {
                  engine.resetTrip();
                  setState(() {});
                },
                icon: const Icon(Icons.restart_alt),
              ),
              IconButton(
                tooltip: 'Настроить дашборд',
                onPressed: () => _configure(context),
                icon: const Icon(Icons.dashboard_customize),
              ),
              FilledButton.tonalIcon(
                onPressed:
                    model.busy || !model.elm.ready ? null : model.togglePolling,
                icon: Icon(engine.running ? Icons.pause : Icons.play_arrow),
                label: Text(engine.running ? 'Пауза' : 'Старт'),
              ),
            ],
          ),
        ),
        if (ids.isEmpty)
          const Expanded(
            child: Center(
              child: Padding(
                padding: EdgeInsets.all(24),
                child: Text(
                  'Нет плиток. Нажмите «Настроить дашборд» и добавьте '
                  'параметры или расчётные плитки (расход, одометр, IDC).',
                  textAlign: TextAlign.center,
                ),
              ),
            ),
          )
        else
          Expanded(
            child: GridView.builder(
              padding: const EdgeInsets.all(10),
              itemCount: ids.length,
              gridDelegate: SliverGridDelegateWithFixedCrossAxisCount(
                crossAxisCount: columns,
                mainAxisExtent: prefs.compact ? 104 : 150,
                mainAxisSpacing: 8,
                crossAxisSpacing: 8,
              ),
              itemBuilder: (context, i) {
                final id = ids[i];
                if (isVirtualTile(id)) {
                  final def = virtualTileById(id);
                  if (def == null) return const SizedBox.shrink();
                  return _Tile(
                    title: def.label,
                    unit: def.unit,
                    digits: def.digits,
                    value: def.compute(inputs),
                    compact: prefs.compact,
                    isVirtual: true,
                    history: null,
                  );
                }
                final pid = SubaruPidLibrary.all.firstWhere(
                  (p) => p.id == id,
                  orElse: () => SubaruPidLibrary.byId('RPM'),
                );
                final sample = engine.latest[pid.id];
                return _Tile(
                  title: '${pid.id}${pid.extended ? ' *' : ''}',
                  unit: pid.unit,
                  digits: pid.digits,
                  value: sample?.value,
                  compact: prefs.compact,
                  isVirtual: false,
                  stale: (engine.ageMs(pid.id) ?? 999999) > 2500,
                  unsupported: sample?.allOnes ?? false,
                  history: prefs.showSpark && !prefs.compact
                      ? engine.history[pid.id]
                      : null,
                );
              },
            ),
          ),
      ],
    );
  }

  Future<void> _configure(BuildContext context) async {
    final result = await showModalBottomSheet<DashboardPrefs>(
      context: context,
      isScrollControlled: true,
      showDragHandle: true,
      builder: (_) => _ConfigSheet(model: model),
    );
    if (result != null) {
      await model.updateDashboardPrefs(result);
      if (mounted) setState(() {});
    }
  }
}

class _Tile extends StatelessWidget {
  const _Tile({
    required this.title,
    required this.unit,
    required this.digits,
    required this.value,
    required this.compact,
    required this.isVirtual,
    required this.history,
    this.stale = false,
    this.unsupported = false,
  });

  final String title, unit;
  final int digits;
  final double? value;
  final bool compact, isVirtual, stale, unsupported;
  final List<PidSample>? history;

  @override
  Widget build(BuildContext context) {
    final text = value == null
        ? (unsupported ? 'н/д' : '—')
        : value!.toStringAsFixed(digits);
    return Card(
      margin: EdgeInsets.zero,
      child: Padding(
        padding: EdgeInsets.all(compact ? 8 : 12),
        child: Column(
          crossAxisAlignment: CrossAxisAlignment.start,
          children: [
            Row(
              children: [
                if (isVirtual)
                  const Padding(
                    padding: EdgeInsets.only(right: 4),
                    child: Icon(Icons.functions, size: 12, color: Color(0xFF22D3EE)),
                  ),
                Expanded(
                  child: Text(
                    title,
                    maxLines: 1,
                    overflow: TextOverflow.ellipsis,
                    style: TextStyle(
                      fontSize: compact ? 11 : 12,
                      fontWeight: FontWeight.bold,
                      color: stale ? _muted : null,
                    ),
                  ),
                ),
                if (stale) const Icon(Icons.schedule, size: 12, color: _muted),
              ],
            ),
            const SizedBox(height: 2),
            FittedBox(
              fit: BoxFit.scaleDown,
              alignment: Alignment.centerLeft,
              child: Text(
                text,
                style: TextStyle(
                  fontSize: compact ? 22 : 30,
                  fontWeight: FontWeight.w600,
                ),
              ),
            ),
            Text(unit, style: const TextStyle(fontSize: 10, color: _muted)),
            if (history != null && history!.length > 2 && !compact) ...[
              const SizedBox(height: 4),
              Expanded(
                child: CustomPaint(
                  painter: _SparkPainter(history!),
                  size: Size.infinite,
                ),
              ),
            ],
          ],
        ),
      ),
    );
  }
}

class _SparkPainter extends CustomPainter {
  _SparkPainter(this.samples);
  final List<PidSample> samples;

  @override
  void paint(Canvas canvas, Size size) {
    final values =
        samples.where((s) => s.value != null).map((s) => s.value!).toList();
    if (values.length < 2) return;
    var lo = values.first, hi = values.first;
    for (final v in values) {
      if (v < lo) lo = v;
      if (v > hi) hi = v;
    }
    if ((hi - lo).abs() < 1e-9) hi = lo + 1;
    final path = Path();
    for (var i = 0; i < values.length; i++) {
      final x = size.width * i / (values.length - 1);
      final y = size.height * (1 - (values[i] - lo) / (hi - lo));
      i == 0 ? path.moveTo(x, y) : path.lineTo(x, y);
    }
    canvas.drawPath(
      path,
      Paint()
        ..color = const Color(0xFF22D3EE)
        ..style = PaintingStyle.stroke
        ..strokeWidth = 1.6,
    );
  }

  @override
  bool shouldRepaint(covariant _SparkPainter old) => true;
}

class _ConfigSheet extends StatefulWidget {
  const _ConfigSheet({required this.model});
  final AppModel model;
  @override
  State<_ConfigSheet> createState() => _ConfigSheetState();
}

class _ConfigSheetState extends State<_ConfigSheet> {
  late DashboardPrefs prefs = widget.model.dashboardPrefs;

  String _labelOf(String id) {
    if (isVirtualTile(id)) {
      final d = virtualTileById(id);
      return d == null ? id : '${d.label} (${d.unit})';
    }
    return id;
  }

  String _subOf(String id) {
    if (isVirtualTile(id)) return virtualTileById(id)?.hint ?? '';
    final p = SubaruPidLibrary.all.where((x) => x.id == id);
    return p.isEmpty ? '' : p.first.desc;
  }

  @override
  Widget build(BuildContext context) {
    final activeIds = widget.model.engine.active.map((p) => p.id).toList();
    final chosen = prefs.tiles
        .where((id) => isVirtualTile(id) || activeIds.contains(id))
        .toList();
    final availablePids = activeIds.where((id) => !chosen.contains(id)).toList();
    final availableVirtual = kVirtualTiles
        .map((t) => t.id)
        .where((id) => !chosen.contains(id))
        .toList();

    return DraggableScrollableSheet(
      expand: false,
      initialChildSize: 0.85,
      maxChildSize: 0.95,
      builder: (context, scroll) => ListView(
        controller: scroll,
        padding: const EdgeInsets.fromLTRB(16, 0, 16, 24),
        children: [
          const Text('Настройка дашборда',
              style: TextStyle(fontSize: 18, fontWeight: FontWeight.bold)),
          const Text(
            'Перетаскивайте за ручку. Плитки со значком ƒ — расчётные '
            '(расход, одометр, IDC) и работают без отдельного PID.',
            style: TextStyle(fontSize: 12, color: _muted),
          ),
          const SizedBox(height: 12),
          Row(
            children: [
              Expanded(
                child: _Stepper(
                  label: 'Колонок (книжная)',
                  value: prefs.columnsPortrait,
                  min: 1,
                  max: 4,
                  onChanged: (v) =>
                      setState(() => prefs = prefs.copyWith(columnsPortrait: v)),
                ),
              ),
              const SizedBox(width: 12),
              Expanded(
                child: _Stepper(
                  label: 'Колонок (альбомная)',
                  value: prefs.columnsLandscape,
                  min: 2,
                  max: 6,
                  onChanged: (v) => setState(
                      () => prefs = prefs.copyWith(columnsLandscape: v)),
                ),
              ),
            ],
          ),
          SwitchListTile(
            contentPadding: EdgeInsets.zero,
            title: const Text('Компактные плитки'),
            value: prefs.compact,
            onChanged: (v) => setState(() => prefs = prefs.copyWith(compact: v)),
          ),
          SwitchListTile(
            contentPadding: EdgeInsets.zero,
            title: const Text('Мини-графики'),
            value: prefs.showSpark,
            onChanged: (v) =>
                setState(() => prefs = prefs.copyWith(showSpark: v)),
          ),
          const Divider(height: 24),
          Text('На дашборде (${chosen.length})',
              style: const TextStyle(fontWeight: FontWeight.bold)),
          ReorderableListView(
            shrinkWrap: true,
            physics: const NeverScrollableScrollPhysics(),
            buildDefaultDragHandles: false,
            onReorder: (oldIndex, newIndex) => setState(() {
              final list = List<String>.from(chosen);
              if (newIndex > oldIndex) newIndex -= 1;
              list.insert(newIndex, list.removeAt(oldIndex));
              final others =
                  prefs.tiles.where((id) => !chosen.contains(id)).toList();
              prefs = prefs.copyWith(tiles: [...list, ...others]);
            }),
            children: [
              for (var i = 0; i < chosen.length; i++)
                ListTile(
                  key: ValueKey('t-${chosen[i]}'),
                  dense: true,
                  contentPadding: EdgeInsets.zero,
                  leading: ReorderableDragStartListener(
                    index: i,
                    child: const Icon(Icons.drag_handle),
                  ),
                  title: Row(
                    children: [
                      if (isVirtualTile(chosen[i]))
                        const Padding(
                          padding: EdgeInsets.only(right: 6),
                          child: Icon(Icons.functions,
                              size: 14, color: Color(0xFF22D3EE)),
                        ),
                      Expanded(child: Text(_labelOf(chosen[i]))),
                    ],
                  ),
                  subtitle: Text(
                    _subOf(chosen[i]),
                    maxLines: 1,
                    overflow: TextOverflow.ellipsis,
                    style: const TextStyle(fontSize: 11, color: _muted),
                  ),
                  trailing: IconButton(
                    icon: const Icon(Icons.visibility_off, size: 18),
                    onPressed: () => setState(() {
                      final list = List<String>.from(prefs.tiles)
                        ..remove(chosen[i]);
                      prefs = prefs.copyWith(tiles: list);
                    }),
                  ),
                ),
            ],
          ),
          if (availableVirtual.isNotEmpty) ...[
            const Divider(height: 24),
            const Text('Расчётные плитки',
                style: TextStyle(fontWeight: FontWeight.bold)),
            const SizedBox(height: 6),
            for (final id in availableVirtual)
              ListTile(
                dense: true,
                contentPadding: EdgeInsets.zero,
                leading: const Icon(Icons.functions,
                    size: 16, color: Color(0xFF22D3EE)),
                title: Text(_labelOf(id), style: const TextStyle(fontSize: 13)),
                subtitle: Text(_subOf(id),
                    style: const TextStyle(fontSize: 11, color: _muted)),
                trailing: IconButton(
                  icon: const Icon(Icons.add_circle_outline, size: 20),
                  onPressed: () => setState(() =>
                      prefs = prefs.copyWith(tiles: [...prefs.tiles, id])),
                ),
              ),
          ],
          if (availablePids.isNotEmpty) ...[
            const Divider(height: 24),
            Text('Активные PID (${availablePids.length})',
                style: const TextStyle(fontWeight: FontWeight.bold)),
            const SizedBox(height: 8),
            Wrap(
              spacing: 6,
              runSpacing: 6,
              children: [
                for (final id in availablePids)
                  ActionChip(
                    label: Text(id, style: const TextStyle(fontSize: 12)),
                    avatar: const Icon(Icons.add, size: 16),
                    onPressed: () => setState(() =>
                        prefs = prefs.copyWith(tiles: [...prefs.tiles, id])),
                  ),
              ],
            ),
          ],
          const SizedBox(height: 20),
          Row(
            children: [
              TextButton(
                onPressed: () => setState(() => prefs = DashboardPrefs()),
                child: const Text('Сбросить'),
              ),
              const Spacer(),
              FilledButton(
                onPressed: () => Navigator.pop(context, prefs),
                child: const Text('Применить'),
              ),
            ],
          ),
        ],
      ),
    );
  }
}

class _Stepper extends StatelessWidget {
  const _Stepper({
    required this.label,
    required this.value,
    required this.min,
    required this.max,
    required this.onChanged,
  });
  final String label;
  final int value, min, max;
  final ValueChanged<int> onChanged;

  @override
  Widget build(BuildContext context) => Column(
        crossAxisAlignment: CrossAxisAlignment.start,
        children: [
          Text(label, style: const TextStyle(fontSize: 11, color: _muted)),
          Row(
            children: [
              IconButton(
                visualDensity: VisualDensity.compact,
                onPressed: value > min ? () => onChanged(value - 1) : null,
                icon: const Icon(Icons.remove_circle_outline),
              ),
              Text('$value', style: const TextStyle(fontSize: 16)),
              IconButton(
                visualDensity: VisualDensity.compact,
                onPressed: value < max ? () => onChanged(value + 1) : null,
                icon: const Icon(Icons.add_circle_outline),
              ),
            ],
          ),
        ],
      );
}
'''

# ===== Сервисная страница: скан, мониторы, VIN, экспертные операции =====
FILES["lib/dtc_service_page.dart"] = r'''
import 'package:flutter/material.dart';

import 'diag.dart';
import 'model.dart';

const _muted = Color(0xFF8CA0BF);

/// Вкладка «DTC»: ошибки всех блоков + полный набор сервисных функций.
class DtcServicePage extends StatefulWidget {
  const DtcServicePage({super.key, required this.model});
  final AppModel model;

  @override
  State<DtcServicePage> createState() => _DtcServicePageState();
}

class _DtcServicePageState extends State<DtcServicePage> {
  final Map<String, List<DiagDtc>> _results = {};
  final Map<String, String> _notes = {};
  ReadinessReport? _readiness;
  String _log = '';
  bool _busy = false;

  AppModel get model => widget.model;
  DiagSession get session => model.diagSession;

  Future<void> _run(String label, Future<void> Function() op) async {
    if (_busy) return;
    setState(() {
      _busy = true;
      _log = '$label…';
    });
    try {
      await model.diagRun(label, op);
      if (mounted) setState(() => _log = '$label — готово');
    } catch (e) {
      if (mounted) setState(() => _log = '$label — ошибка: $e');
    } finally {
      if (mounted) setState(() => _busy = false);
    }
  }

  Future<bool> _confirm(String title, String body, {bool danger = false}) async {
    final ok = await showDialog<bool>(
      context: context,
      builder: (ctx) => AlertDialog(
        title: Text(title),
        content: Text(body),
        actions: [
          TextButton(
              onPressed: () => Navigator.pop(ctx, false),
              child: const Text('ОТМЕНА')),
          danger
              ? FilledButton(
                  style: FilledButton.styleFrom(backgroundColor: Colors.red),
                  onPressed: () => Navigator.pop(ctx, true),
                  child: const Text('ВЫПОЛНИТЬ'))
              : FilledButton.tonal(
                  onPressed: () => Navigator.pop(ctx, true),
                  child: const Text('ВЫПОЛНИТЬ')),
        ],
      ),
    );
    return ok ?? false;
  }

  @override
  Widget build(BuildContext context) {
    final theme = Theme.of(context);
    return ListView(
      padding: const EdgeInsets.all(12),
      children: [
        if (_busy) const LinearProgressIndicator(),
        if (_log.isNotEmpty)
          Padding(
            padding: const EdgeInsets.symmetric(vertical: 6),
            child: Text(_log, style: theme.textTheme.bodySmall),
          ),

        // ----- быстрые действия -----
        Wrap(
          spacing: 8,
          runSpacing: 8,
          children: [
            FilledButton.tonalIcon(
              onPressed: _busy ? null : _scanAll,
              icon: const Icon(Icons.radar, size: 18),
              label: const Text('Скан всех блоков'),
            ),
            OutlinedButton.icon(
              onPressed: _busy ? null : _readInfo,
              icon: const Icon(Icons.badge, size: 18),
              label: const Text('VIN / CALID'),
            ),
            OutlinedButton.icon(
              onPressed: _busy ? null : _readReadiness,
              icon: const Icon(Icons.checklist, size: 18),
              label: const Text('Мониторы'),
            ),
          ],
        ),
        const SizedBox(height: 8),

        if (_readiness != null) _readinessCard(_readiness!),

        // ----- блоки -----
        for (final slot in EcuSlot.all)
          Card(
            child: ExpansionTile(
              leading: Icon(
                _results.containsKey(slot.name)
                    ? (_results[slot.name]!.isEmpty
                        ? Icons.check_circle
                        : Icons.warning_amber)
                    : Icons.memory,
                color: _results[slot.name]?.isEmpty == true
                    ? Colors.greenAccent
                    : (_results[slot.name]?.isNotEmpty == true
                        ? Colors.amberAccent
                        : null),
              ),
              title: Text(slot.name),
              subtitle: Text(
                '${slot.header}→${slot.responseId} · ${slot.proto.name.toUpperCase()}'
                '${_notes[slot.name] == null ? '' : ' · ${_notes[slot.name]}'}',
                style: const TextStyle(fontSize: 11),
              ),
              children: [
                for (final d in _results[slot.name] ?? const <DiagDtc>[])
                  ListTile(
                    dense: true,
                    leading: Icon(
                      d.origin == 'permanent'
                          ? Icons.lock
                          : (d.active ? Icons.error : Icons.history),
                      color: d.origin == 'permanent'
                          ? Colors.orangeAccent
                          : (d.active
                              ? theme.colorScheme.error
                              : theme.colorScheme.outline),
                      size: 20,
                    ),
                    title: Text(
                      '${d.code}${d.titleRu.isEmpty ? '' : ' · ${d.titleRu}'}',
                      style: const TextStyle(fontFamily: 'monospace', fontSize: 13),
                    ),
                    subtitle: Text(d.statusText,
                        style: const TextStyle(fontSize: 11)),
                    trailing: d.milOn
                        ? const Icon(Icons.lightbulb,
                            color: Colors.amber, size: 18)
                        : null,
                  ),
                if (_results[slot.name]?.isEmpty == true)
                  const ListTile(
                    dense: true,
                    leading: Icon(Icons.check, color: Colors.greenAccent),
                    title: Text('Ошибок нет'),
                  ),
                OverflowBar(
                  alignment: MainAxisAlignment.start,
                  children: [
                    TextButton.icon(
                      onPressed: _busy ? null : () => _readAll(slot),
                      icon: const Icon(Icons.search, size: 18),
                      label: const Text('ЧИТАТЬ ВСЁ'),
                    ),
                    TextButton(
                      onPressed: _busy ? null : () => _readFreeze(slot),
                      child: const Text('СТОП-КАДР'),
                    ),
                    if ((_results[slot.name] ?? const []).isNotEmpty)
                      FilledButton.tonalIcon(
                        onPressed: _busy ? null : () => _clear(slot),
                        icon: const Icon(Icons.cleaning_services, size: 18),
                        label: const Text('СБРОС'),
                      ),
                  ],
                ),
              ],
            ),
          ),

        const Divider(height: 28),
        Text('Сервисные процедуры', style: theme.textTheme.titleMedium),
        const ListTile(
          dense: true,
          leading: Icon(Icons.restart_alt),
          title: Text('Сброс адаптаций (SSM2-эра)'),
          subtitle: Text(
              'Клемма АКБ 10–30 мин. Контроль IAM до/после, дообучение 15–20 мин.'),
        ),
        const ListTile(
          dense: true,
          leading: Icon(Icons.speed),
          title: Text('Обучение дросселя (E-Gas)'),
          subtitle: Text('Зажигание ON 15–20 с без запуска → OFF → прогрев.'),
        ),

        const Divider(height: 28),
        // ----- экспертные операции -----
        Card(
          color: model.experimental
              ? Colors.red.withValues(alpha: 0.07)
              : null,
          child: Column(
            children: [
              SwitchListTile(
                secondary: Icon(Icons.science,
                    color: model.experimental ? Colors.redAccent : null),
                title: const Text('Экспертные операции'),
                subtitle: Text(
                  model.experimental
                      ? 'РАЗБЛОКИРОВАНО: ECUReset и Clear Memory активны. '
                          'Только на стоянке, двигатель заглушён, зажигание ON.'
                      : 'ECUReset (11 01) и Clear Memory (04 xx) заблокированы. '
                          'Включите, если понимаете риск.',
                  style: const TextStyle(fontSize: 11),
                ),
                value: model.experimental,
                onChanged: _busy
                    ? null
                    : (v) async {
                        if (v) {
                          final ok = await _confirm(
                            'Включить экспертные операции?',
                            'ECUReset перезапускает блок управления и ГЛУШИТ двигатель. '
                            'Clear Memory стирает адаптации (IAM, топливные коррекции) — '
                            'потребуется дообучение 15–20 минут езды.\n\n'
                            'Выполняйте только на стоянке при заглушённом моторе.',
                            danger: true,
                          );
                          if (!ok) return;
                        }
                        await model.setExperimental(v);
                        if (mounted) setState(() {});
                      },
              ),
              if (model.experimental) ...[
                const Divider(height: 1),
                ListTile(
                  dense: true,
                  leading: const Icon(Icons.power_settings_new,
                      color: Colors.redAccent),
                  title: const Text('ECUReset (11 01)'),
                  subtitle: const Text(
                      'Перезапуск ЭБУ. Двигатель заглохнет.',
                      style: TextStyle(fontSize: 11)),
                  trailing: const Icon(Icons.chevron_right),
                  onTap: _busy ? null : _ecuReset,
                ),
                ListTile(
                  dense: true,
                  leading: const Icon(Icons.auto_delete, color: Colors.orangeAccent),
                  title: const Text('Clear Memory (04 xx)'),
                  subtitle: const Text(
                      'Сброс адаптаций. Сначала «Проба групп».',
                      style: TextStyle(fontSize: 11)),
                  trailing: const Icon(Icons.chevron_right),
                  onTap: _busy ? null : _clearMemoryDialog,
                ),
                ListTile(
                  dense: true,
                  leading: const Icon(Icons.travel_explore),
                  title: const Text('Проба групп Clear Memory'),
                  subtitle: const Text(
                      'Безопасно: опрашивает 1..7 и показывает ответы ECU',
                      style: TextStyle(fontSize: 11)),
                  trailing: const Icon(Icons.chevron_right),
                  onTap: _busy ? null : _probeGroups,
                ),
              ],
            ],
          ),
        ),
        const SizedBox(height: 24),
      ],
    );
  }

  Widget _readinessCard(ReadinessReport r) => Card(
        child: Padding(
          padding: const EdgeInsets.all(12),
          child: Column(
            crossAxisAlignment: CrossAxisAlignment.start,
            children: [
              Row(
                children: [
                  Icon(r.milOn ? Icons.lightbulb : Icons.lightbulb_outline,
                      color: r.milOn ? Colors.amber : _muted, size: 18),
                  const SizedBox(width: 8),
                  Text(
                    'Мониторы: ${r.ready}/${r.total} готовы'
                    '${r.milOn ? ' · CHECK ГОРИТ' : ''} · DTC: ${r.dtcCount}',
                    style: const TextStyle(fontWeight: FontWeight.bold),
                  ),
                ],
              ),
              const SizedBox(height: 6),
              Wrap(
                spacing: 6,
                runSpacing: 6,
                children: [
                  for (final e in r.monitors.entries)
                    Chip(
                      visualDensity: VisualDensity.compact,
                      avatar: Icon(
                        e.value ? Icons.check_circle : Icons.pending,
                        size: 16,
                        color: e.value ? Colors.greenAccent : Colors.orangeAccent,
                      ),
                      label: Text(e.key, style: const TextStyle(fontSize: 11)),
                    ),
                ],
              ),
            ],
          ),
        ),
      );

  // ------------------------- действия -------------------------

  Future<void> _readAll(EcuSlot slot) => _run('Чтение ${slot.name}', () async {
        final list = await session.readAllDtc(slot);
        if (mounted) {
          setState(() {
            _results[slot.name] = list;
            _notes[slot.name] = '${list.length} код(ов)';
          });
        }
      });

  Future<void> _clear(EcuSlot slot) async {
    final has = _results[slot.name] ?? const <DiagDtc>[];
    final permanent = has.where((d) => d.origin == 'permanent').length;
    final ok = await _confirm(
      'Сбросить ошибки: ${slot.name}?',
      'Будут стёрты коды, стоп-кадры и мониторы готовности. '
      'CHECK погаснет после перезапуска зажигания.\n\n'
      '${permanent > 0 ? 'ВНИМАНИЕ: $permanent постоянных (permanent) кодов НЕ стираются '
          'сбросом — они уйдут только после успешного прохождения циклов.\n\n' : ''}'
      'Адаптации (IAM, топливные коррекции) это НЕ трогает.',
    );
    if (!ok) return;
    await _run('Сброс ${slot.name}', () async {
      final rest = await session.clearDtc(slot);
      if (mounted) {
        setState(() {
          _results[slot.name] = rest;
          _notes[slot.name] =
              rest.isEmpty ? 'очищено' : 'осталось ${rest.length}';
        });
      }
    });
  }

  Future<void> _readFreeze(EcuSlot slot) =>
      _run('Стоп-кадр ${slot.name}', () async {
        final dtc = await session.readFreezeFrameDtc(slot);
        if (mounted) {
          setState(() => _notes[slot.name] =
              dtc == null ? 'стоп-кадра нет' : 'стоп-кадр: $dtc');
        }
      });

  Future<void> _scanAll() => _run('Скан всех блоков', () async {
        final map = await session.scanAllEcus(
          onProgress: (name) {
            if (mounted) setState(() => _log = 'Скан: $name…');
          },
        );
        if (mounted) {
          setState(() {
            _results
              ..clear()
              ..addAll(map);
            _notes.clear();
            for (final e in map.entries) {
              _notes[e.key] = '${e.value.length} код(ов)';
            }
          });
        }
      });

  Future<void> _readInfo() => _run('Чтение VIN / CALID', () async {
        final vin = await session.readVin(EcuSlot.engine);
        final cal = await session.readCalId(EcuSlot.engine);
        if (mounted) {
          setState(() => _log =
              'VIN: ${vin ?? 'нет ответа'} · CALID: ${cal ?? 'нет ответа'}');
        }
      });

  Future<void> _readReadiness() => _run('Мониторы готовности', () async {
        final r = await session.readReadiness(EcuSlot.engine);
        if (mounted) setState(() => _readiness = r);
      });

  Future<void> _ecuReset() async {
    final ok = await _confirm(
      'ECUReset: перезапустить ЭБУ?',
      'Двигатель ЗАГЛОХНЕТ немедленно. Выполняйте только на стоянке. '
      'После сброса поверните зажигание OFF→ON.',
      danger: true,
    );
    if (!ok) return;
    await _run('ECUReset', () async {
      final res = await session.ecuReset(EcuSlot.engine);
      if (mounted) setState(() => _log = '${res.title}: ${res.detail}');
    });
  }

  Future<void> _probeGroups() => _run('Проба групп Clear Memory', () async {
        final map = await session.probeClearMemoryGroups(EcuSlot.engine);
        if (!mounted) return;
        final text = map.entries
            .map((e) => 'гр.${e.key}: ${e.value}')
            .join('\n');
        await showDialog<void>(
          context: context,
          builder: (ctx) => AlertDialog(
            title: const Text('Поддержка групп Clear Memory'),
            content: SingleChildScrollView(
              child: Text(
                text.isEmpty ? 'Нет ответа от ECU' : text,
                style: const TextStyle(fontFamily: 'monospace', fontSize: 12),
              ),
            ),
            actions: [
              TextButton(
                  onPressed: () => Navigator.pop(ctx),
                  child: const Text('ЗАКРЫТЬ')),
            ],
          ),
        );
      });

  Future<void> _clearMemoryDialog() async {
    var group = 1;
    final ok = await showDialog<bool>(
      context: context,
      builder: (ctx) => StatefulBuilder(
        builder: (ctx, set) => AlertDialog(
          title: const Text('Clear Memory: сброс адаптаций'),
          content: Column(
            mainAxisSize: MainAxisSize.min,
            crossAxisAlignment: CrossAxisAlignment.start,
            children: [
              const Text(
                'Группы зависят от поколения ECU. Обычно: 1 — всё обучение, '
                '2 — топливные коррекции, 3 — детонация/IAM.\n\n'
                'После сброса нужно дообучение: 15–20 минут спокойной езды '
                'на прогретом моторе.',
                style: TextStyle(fontSize: 12),
              ),
              const SizedBox(height: 12),
              DropdownButtonFormField<int>(
                initialValue: group,
                decoration: const InputDecoration(labelText: 'Группа'),
                items: [
                  for (var g = 1; g <= 7; g++)
                    DropdownMenuItem(value: g, child: Text('Группа $g')),
                ],
                onChanged: (v) => set(() => group = v ?? 1),
              ),
            ],
          ),
          actions: [
            TextButton(
                onPressed: () => Navigator.pop(ctx, false),
                child: const Text('ОТМЕНА')),
            FilledButton(
              style: FilledButton.styleFrom(backgroundColor: Colors.orange),
              onPressed: () => Navigator.pop(ctx, true),
              child: const Text('СБРОСИТЬ'),
            ),
          ],
        ),
      ),
    );
    if (ok != true) return;
    await _run('Clear Memory гр.$group', () async {
      final res = await session.clearMemory(EcuSlot.engine, group);
      if (mounted) setState(() => _log = '${res.title}: ${res.detail}');
    });
  }
}
'''

# ===== 20 тестов: OBD 07/0A, мониторы, VIN, NRC, виртуальные плитки =====
FILES["test/service_test.dart"] = r'''
import 'package:flutter_test/flutter_test.dart';
import 'package:subaru_ssm2/dashboard_prefs.dart';
import 'package:subaru_ssm2/diag.dart';
import 'package:subaru_ssm2/vehicle_config.dart';
import 'package:subaru_ssm2/virtual_tiles.dart';

void main() {
  group('OBD mode 07 / 0A', () {
    test('pending помечается origin=pending', () {
      final d = parseObdDtcReply(const [0x47, 0x01, 0x10]);
      expect(d.single.code, 'P0110');
      expect(d.single.origin, 'pending');
    });
    test('permanent помечается и не считается активной', () {
      final d = parseObdDtcReply(const [0x4A, 0x04, 0x20]);
      expect(d.single.origin, 'permanent');
      expect(d.single.active, isFalse);
    });
    test('mode 03 остаётся stored', () {
      expect(parseObdDtcReply(const [0x43, 0x01, 0x10]).single.origin, 'stored');
    });
  });

  group('Мониторы готовности', () {
    test('41 01 с MIL и счётчиком DTC', () {
      // A=0x83: MIL on, 3 DTC
      final r = parseReadiness(const [0x41, 0x01, 0x83, 0x07, 0x21, 0x00]);
      expect(r, isNotNull);
      expect(r!.milOn, isTrue);
      expect(r.dtcCount, 3);
      expect(r.monitors, isNotEmpty);
    });
    test('незавершённый монитор виден как не готовый', () {
      // C=0x01 (катализатор доступен), D=0x01 (не завершён)
      final r = parseReadiness(const [0x41, 0x01, 0x00, 0x00, 0x01, 0x01]);
      expect(r!.monitors['Катализатор'], isFalse);
    });
    test('мусор -> null', () {
      expect(parseReadiness(const [0x7F, 0x01, 0x12]), isNull);
    });
  });

  group('VIN / CALID / стоп-кадр', () {
    test('ASCII VIN из mode 09', () {
      final bytes = <int>[0x49, 0x02, 0x01, ...'JF1BP9LL'.codeUnits];
      expect(parseAsciiReply(bytes, 0x02), 'JF1BP9LL');
    });
    test('CALID из mode 09 PID 04', () {
      final bytes = <int>[0x49, 0x04, 0x01, ...'A2TB100B'.codeUnits];
      expect(parseAsciiReply(bytes, 0x04), 'A2TB100B');
    });
    test('стоп-кадр возвращает код', () {
      expect(parseFreezeFrameDtc(const [0x42, 0x02, 0x00, 0x01, 0x10]), 'P0110');
      expect(parseFreezeFrameDtc(const [0x42, 0x02, 0x00, 0x00, 0x00]), isNull);
    });
  });

  group('NRC', () {
    test('ключевые коды расшифрованы', () {
      expect(kNrMeanings[0x22], contains('conditionsNotCorrect'));
      expect(kNrMeanings[0x7F], contains('10 03'));
      expect(kNrMeanings[0x33], contains('дилер'));
    });
  });

  group('Виртуальные плитки', () {
    TileInputs make(Map<String, double> v, {double km = 0, double l = 0}) =>
        TileInputs(
          values: v,
          config: const VehicleConfig(),
          tripKm: km,
          tripLitres: l,
        );

    test('мгновенный расход л/ч считается из MAF и AFR', () {
      final t = virtualTileById('@FUEL_LPH')!;
      final v = t.compute(make({'MAF': 10, 'AFR': 14.7}));
      // 10/14.7 = 0.68 г/с -> *3600/745 = 3.29 л/ч
      expect(v, closeTo(3.29, 0.05));
    });

    test('л/100км требует скорость >= 5', () {
      final t = virtualTileById('@FUEL_L100')!;
      expect(t.compute(make({'MAF': 10, 'AFR': 14.7, 'SPEED': 0})), isNull);
      expect(t.compute(make({'MAF': 10, 'AFR': 14.7, 'SPEED': 90})),
          greaterThan(0));
    });

    test('одометр и средний расход поездки', () {
      expect(virtualTileById('@TRIP_KM')!.compute(make({}, km: 12.5)), 12.5);
      final avg = virtualTileById('@TRIP_AVG')!
          .compute(make({}, km: 100, l: 12))!;
      expect(avg, closeTo(12.0, 0.001));
      expect(virtualTileById('@TRIP_AVG')!.compute(make({}, km: 0)), isNull);
    });

    test('IDC по ширине импульса', () {
      final v = virtualTileById('@IDC')!
          .compute(make({'INJPW': 10, 'RPM': 6000}));
      expect(v, closeTo(50.0, 0.1));
    });

    test('лямбда и суммарная коррекция', () {
      expect(virtualTileById('@LAMBDA')!.compute(make({'AFR': 14.7})),
          closeTo(1.0, 0.001));
      expect(
        virtualTileById('@KNOCK_TOT')!.compute(make({'FBKC': -1.4, 'FKL': -2.0})),
        closeTo(-3.4, 0.001),
      );
    });

    test('каждая плитка имеет уникальный id с префиксом @', () {
      final ids = kVirtualTiles.map((t) => t.id).toList();
      expect(ids.toSet().length, ids.length);
      expect(ids.every(isVirtualTile), isTrue);
    });
  });

  group('DashboardPrefs с виртуальными', () {
    test('дефолт содержит расход и одометр', () {
      expect(DashboardPrefs.defaultTiles, contains('@FUEL_L100'));
      expect(DashboardPrefs.defaultTiles, contains('@TRIP_KM'));
    });
    test('виртуальные проходят валидацию, мусор — нет', () {
      final p = DashboardPrefs.fromJson({
        'tiles': ['@FUEL_LPH', '@НЕТ_ТАКОЙ', 'RPM']
      });
      expect(p.tiles, contains('@FUEL_LPH'));
      expect(p.tiles, isNot(contains('@НЕТ_ТАКОЙ')));
    });
    test('visibleTiles не требует активного PID для виртуальных', () {
      final p = DashboardPrefs(tiles: ['@TRIP_KM', 'AFR']);
      expect(p.visibleTiles(<String>{}), ['@TRIP_KM']);
    });
  });
}
'''

# ===== Правки: рантайм-EXPERIMENTAL, whitelist сервисов, версия =====
# ===== REWRITES =====

# --- 1. model.dart: EXPERIMENTAL -> рантайм (был compile-time const) ---
REWRITES.append((
    "lib/model.dart",
    """  final bool experimental =
      const bool.fromEnvironment('SSM2_EXPERIMENTAL', defaultValue: false);
  late final DiagSession diagSession =
      DiagSession(elm, experimental: experimental);""",
    """  /// v0.13 (04.4): рантайм-флаг экспертных операций. В 04.2 он был
  /// compile-time константой, поэтому переключатель всегда оставался серым.
  bool experimental = false;
  late final DiagSession diagSession = DiagSession(elm);

  Future<void> setExperimental(bool value) async {
    experimental = value;
    diagSession.experimental = value;
    await saveSettings();
    changed();
  }""",
    ["Future<void> setExperimental", False],
))

# --- 2. model.dart: сохранение флага ---
REWRITES.append((
    "lib/model.dart",
    "        'vehicle': vehicleConfig.toJson(), // v0.12",
    "        'experimental': experimental, // v0.13\n        'vehicle': vehicleConfig.toJson(), // v0.12",
    ["'experimental': experimental", False],
))

# --- 3. model.dart: восстановление флага ---
REWRITES.append((
    "lib/model.dart",
    "      uiPrefs = UiPrefs.fromJson(json['ui'] as Map<String, dynamic>?);",
    """      uiPrefs = UiPrefs.fromJson(json['ui'] as Map<String, dynamic>?);
      experimental = json['experimental'] == true; // v0.13
      diagSession.experimental = experimental;""",
    ["diagSession.experimental = experimental", False],
))

# --- 4. protocol.dart: сервисные команды в белый список ---
REWRITES.append((
    "lib/protocol.dart",
    "  '11 01', // UDS ECUReset (глушится EXPERIMENTAL в diag.dart)\n};",
    """  '11 01', // UDS ECUReset (глушится EXPERIMENTAL в diag.dart)
  // v0.13 (04.4): расширенный сервисный набор
  '02 02 00', // OBD freeze frame: DTC стоп-кадра
  '09 04', // CALID прошивки
  '09 00', // список поддерживаемых PID mode 09
  '01 00', // список поддерживаемых PID mode 01
  '01 41', // статус мониторов текущего цикла
  '10 03', // UDS extended session (нужна для 11 01 и 04 xx)
  '10 01', // UDS default session
  '3E 00', // TesterPresent (keep-alive)
  '22 F1 90', // UDS ReadDataByIdentifier: VIN
  '22 F1 8C', // UDS: серийный номер блока
};""",
    ["'10 03', // UDS extended session", False],
))

# --- 5. main.dart: версия ---
REWRITES.append((
    "lib/main.dart",
    "SSM2 TELEMETRY 0.12",
    "SSM2 TELEMETRY 0.13",
    ["SSM2 TELEMETRY 0.13", True],
))

# ===== Запись, правки, самопроверка, чек-лист стенда =====
# ===== запись файлов =====
bad = [r for r in FILES if "(" in r or ")" in r]
if bad:
    raise RuntimeError(f"Невалидные FILES-ключи: {bad}")
BACKUP.mkdir(parents=True, exist_ok=True)
for rel, body in FILES.items():
    dest = APP / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        shutil.copyfile(dest, BACKUP / (rel.replace("/", "__") + ".bak"))
    dest.write_text(body.strip("\n") + "\n", encoding="utf-8")
    print(f"[OK] {rel}")
print(f"[OK] файлов: {len(FILES)}")

# ===== точечные правки =====
patched, skipped = 0, 0
for rel, needle, replacement, sig in REWRITES:
    optional = False
    if sig and sig[-1] is True:
        sig, optional = sig[:-1], True
    elif sig and sig[-1] is False:
        sig, optional = sig[:-1], False
    path = APP / rel
    if not path.exists():
        raise RuntimeError(f"Нет файла {rel} — прогоните 02–04.3")
    text = path.read_text(encoding="utf-8")
    if sig and all(s in text for s in sig):
        skipped += 1
        print(f"[SKIP уже применено] {rel}: {sig[0][:50]}")
        continue
    if needle in text:
        shutil.copyfile(path, BACKUP / (rel.replace("/", "__") + ".orig.bak"))
        path.write_text(text.replace(needle, replacement, 1), encoding="utf-8")
        patched += 1
        print(f"[OK] правка {rel}")
        continue
    if optional:
        skipped += 1
        print(f"[SKIP необязательная] {rel}")
        continue
    raise RuntimeError(f"Маркер не найден в {rel}:\n{needle[:100]!r}")
print(f"[OK] правок: {patched}, пропущено: {skipped}")

# ===== самопроверка =====
checks = {
    "lib/diag.dart": ["readPendingDtc", "readPermanentDtc", "parseReadiness",
                      "enterExtendedSession", "probeClearMemoryGroups",
                      "class ServiceResult", "bool experimental = false"],
    "lib/virtual_tiles.dart": ["@FUEL_LPH", "@TRIP_KM", "@IDC", "kVirtualTiles"],
    "lib/dashboard_prefs.dart": ["isVirtualTile", "'@FUEL_L100'"],
    "lib/dashboard_page.dart": ["virtualTileById", "TileInputs"],
    "lib/dtc_service_page.dart": ["Экспертные операции", "_probeGroups", "_readReadiness"],
    "lib/model.dart": ["Future<void> setExperimental", "'experimental': experimental"],
    "lib/protocol.dart": ["'10 03', // UDS extended session", "'3E 00'"],
    "test/service_test.dart": ["parseReadiness", "@FUEL_LPH"],
}
for rel, needles in checks.items():
    text = (APP / rel).read_text(encoding="utf-8")
    miss = [m for m in needles if m not in text]
    if miss:
        raise RuntimeError(f"Самопроверка {rel}: нет маркеров {miss}")
print("[OK] самопроверка маркеров пройдена")

print("\n=== Готово: SSM2 0.13. Далее — ячейка 05 (сборка). ===")
print("Исправлено и добавлено:")
print("  · EXPERIMENTAL теперь рантайм: переключатель во вкладке DTC")
print("  · UDS-сбросы открывают сессию 10 03 автоматически")
print("  · сервисы: pending/permanent DTC, стоп-кадр, мониторы, VIN, CALID")
print("  · скан всех блоков и проба групп Clear Memory")
print("  · плитки: расход л/ч и л/100км, одометр, средний, IDC, запас насоса")
print("Чек-лист стенда 04.4:")
print(" [ ] дашборд: видны «Расход л/100км» и «Одометр» без настройки")
print(" [ ] вкладка DTC: «Скан всех блоков» проходит по 6 адресам")
print(" [ ] «Мониторы» показывают готовность после сброса ошибок")
print(" [ ] VIN/CALID читаются (CALID должен совпасть с A2TB100B)")
print(" [ ] переключатель «Экспертные операции» включается с подтверждением")
print(" [ ] «Проба групп» возвращает ответы ECU без побочных эффектов")


[OK] lib/diag.dart
[OK] lib/virtual_tiles.dart
[OK] lib/dashboard_prefs.dart
[OK] lib/dashboard_page.dart
[OK] lib/dtc_service_page.dart
[OK] test/service_test.dart
[OK] файлов: 6
[OK] правка lib/model.dart
[OK] правка lib/model.dart
[OK] правка lib/model.dart
[OK] правка lib/protocol.dart
[OK] правка lib/main.dart
[OK] правок: 5, пропущено: 0
[OK] самопроверка маркеров пройдена

=== Готово: SSM2 0.13. Далее — ячейка 05 (сборка). ===
Исправлено и добавлено:
  · EXPERIMENTAL теперь рантайм: переключатель во вкладке DTC
  · UDS-сбросы открывают сессию 10 03 автоматически
  · сервисы: pending/permanent DTC, стоп-кадр, мониторы, VIN, CALID
  · скан всех блоков и проба групп Clear Memory
  · плитки: расход л/ч и л/100км, одометр, средний, IDC, запас насоса
Чек-лист стенда 04.4:
 [ ] дашборд: видны «Расход л/100км» и «Одометр» без настройки
 [ ] вкладка DTC: «Скан всех блоков» проходит по 6 адресам
 [ ] «Мониторы» показывают готовность после сброса ошибок
 [ ] VIN/CALID читаются (CALID долже

In [ ]:
# ===== Шапка 04.5: разбор причин ложного «готово» =====
# @title 04.5 | TRACE & GRID | SSM2 0.13 >> 0.14 (вставить МЕЖДУ 04.4 и 05)
# ГЛАВНОЕ — исправлен ложный «готово» во вкладке DTC:
#   Причина 1: AppModel.perform() ГЛОТАЕТ исключения (catch -> message),
#              поэтому diagRun никогда не бросал и UI всегда писал «готово».
#   Причина 2: readDtc при любой неудаче возвращал ПУСТОЙ список, а UI
#              показывал это как «Ошибок нет» — ложноположительный результат.
#   Причина 3: perform() при busy=true молча выходит — операция не выполнялась.
#   Теперь: проверка elm.ready ДО отправки, DtcReadResult.answered (ответил ли
#   ЭБУ вообще), ошибки пробрасываются, в UI — красный статус с причиной.
# Добавлено:
#   · lib/diag_trace.dart + окно «Журнал обмена»: каждая команда TX и ответ RX
#     с таймстемпом и временем отклика. Видно, реально ли что-то уходит.
#   · Карточка поездки возвращена: мгновенный расход л/ч и л/100км, одометр,
#     литры, средний расход + кнопка сброса.
#   · Сетка дашборда: отдельно «ячеек в строке» и «строк» для книжной и
#     альбомной (до 5x8 = 40 и 8x6 = 48), режим «вместить без прокрутки».
#   · Перетаскивание ячеек ПРЯМО на дашборде (долгое нажатие -> перенос).
#   · Объединение 2–4 PID в одну ячейку (компактный список в плитке).
# Идемпотентна, бэкап снаружи проекта, самопроверка маркеров.

import json
import re
import shutil
import time
from pathlib import Path

APP = Path("/content/subaru_ssm2_fixed")

STAMP = time.strftime("%Y%m%d_%H%M%S")
BACKUP = APP.parent / f"ssm2_backup_045_{STAMP}"
FILES = {}
REWRITES = []

# ===== Журнал обмена, NotConnectedException, DtcReadResult.answered =====
FILES["lib/diag_trace.dart"] = r'''
import 'package:flutter/foundation.dart';

/// Направление записи в журнале обмена.
enum TraceDir { tx, rx, info, error }

class TraceEntry {
  TraceEntry(this.dir, this.text, {this.ms});
  final TraceDir dir;
  final String text;
  final int? ms;
  final DateTime time = DateTime.now();

  String get stamp {
    final t = time;
    String two(int v) => v.toString().padLeft(2, '0');
    return '${two(t.hour)}:${two(t.minute)}:${two(t.second)}.'
        '${t.millisecond.toString().padLeft(3, '0')}';
  }

  String get arrow => switch (dir) {
        TraceDir.tx => '>>',
        TraceDir.rx => '<<',
        TraceDir.info => '--',
        TraceDir.error => '!!',
      };
}

/// Живой журнал обмена с ЭБУ: видно каждую отправленную команду и ответ.
/// Нужен, чтобы отличать «ответа нет» от «ошибок нет».
class DiagTrace extends ChangeNotifier {
  DiagTrace({this.limit = 400});
  final int limit;
  final List<TraceEntry> entries = <TraceEntry>[];

  int txCount = 0;
  int rxCount = 0;
  int errorCount = 0;

  void _add(TraceEntry e) {
    entries.add(e);
    if (entries.length > limit) entries.removeRange(0, entries.length - limit);
    notifyListeners();
  }

  void tx(String command) {
    txCount++;
    _add(TraceEntry(TraceDir.tx, command));
  }

  void rx(String reply, {int? ms}) {
    rxCount++;
    _add(TraceEntry(TraceDir.rx, reply.isEmpty ? '(пусто)' : reply, ms: ms));
  }

  void info(String text) => _add(TraceEntry(TraceDir.info, text));

  void error(String text) {
    errorCount++;
    _add(TraceEntry(TraceDir.error, text));
  }

  void clear() {
    entries.clear();
    txCount = 0;
    rxCount = 0;
    errorCount = 0;
    notifyListeners();
  }

  String asText() => entries
      .map((e) => '${e.stamp} ${e.arrow} ${e.text}'
          '${e.ms == null ? '' : '  (${e.ms} мс)'}')
      .join('\n');
}

/// Исключение: адаптер не подключён — операция даже не отправлялась.
class NotConnectedException implements Exception {
  NotConnectedException([this.message =
      'Адаптер не подключён: команда не отправлена. '
          'Вкладка «Адаптер» → подключитесь к ELM327.']);
  final String message;
  @override
  String toString() => message;
}

/// Результат чтения DTC: различает «ответ получен, кодов нет» и «нет ответа».
class DtcReadResult {
  const DtcReadResult({
    required this.codes,
    required this.answered,
    required this.protocolUsed,
    required this.rawReplies,
    this.note = '',
  });

  final List<DiagDtcLike> codes;

  /// true — ЭБУ реально ответил корректным кадром.
  final bool answered;

  /// Какой сервис сработал: SSM2 18 / OBD 03 / UDS 19 02.
  final String protocolUsed;

  /// Сырые ответы для журнала.
  final List<String> rawReplies;
  final String note;

  bool get clean => answered && codes.isEmpty;
}

/// Минимальный контракт DTC, чтобы diag_trace не зависел от diag.dart.
abstract class DiagDtcLike {
  String get code;
}
'''

# ===== Честный движок: проверка связи, трассировка каждой команды =====
FILES["lib/diag.dart"] = r'''
import 'diag_trace.dart';
import 'dtc_dict.dart';
import 'elm.dart';
import 'protocol.dart';

enum DiagProto { ssm2, obd, uds }

class EcuSlot {
  const EcuSlot(this.name, this.header, this.responseId, this.proto);
  final String name, header, responseId;
  final DiagProto proto;

  static const engine = EcuSlot('Двигатель (ECM)', '7E0', '7E8', DiagProto.ssm2);
  static const tcu = EcuSlot('АКПП / CVT (TCM)', '7E1', '7E9', DiagProto.ssm2);
  static const vdc = EcuSlot('ABS / VDC', '7B0', '7B8', DiagProto.uds);
  static const srs = EcuSlot('Подушки (SRS)', '772', '77A', DiagProto.uds);
  static const eps = EcuSlot('ЭУР (EPS)', '7A6', '7AE', DiagProto.uds);
  static const biu = EcuSlot('Кузовной блок (BIU)', '723', '72B', DiagProto.uds);

  static const List<EcuSlot> all = [engine, tcu, vdc, srs, eps, biu];
}

class DiagDtc implements DiagDtcLike {
  DiagDtc(this.code, this.statusByte, this.proto, {this.origin = ''});
  @override
  final String code;
  final int statusByte;
  final DiagProto proto;
  final String origin;

  bool get active => proto == DiagProto.uds
      ? (statusByte & 0x01) != 0
      : proto == DiagProto.obd
          ? origin != 'permanent'
          : (statusByte & 0x20) != 0;
  bool get confirmed => proto == DiagProto.uds ? (statusByte & 0x08) != 0 : true;
  bool get milOn => proto == DiagProto.uds
      ? (statusByte & 0x40) != 0
      : proto == DiagProto.obd
          ? false
          : (statusByte & 0x80) != 0;

  String get statusText {
    final p = origin.isEmpty ? '' : '$origin · ';
    if (proto == DiagProto.uds) {
      final f = <String>[];
      if (statusByte & 0x01 != 0) f.add('есть сейчас');
      if (statusByte & 0x08 != 0) f.add('подтверждена');
      if (statusByte & 0x40 != 0) f.add('MIL');
      return p + (f.isEmpty ? 'пассивная (0x${hex2(statusByte)})' : f.join(' · '));
    }
    if (proto == DiagProto.obd) return '${p}OBD';
    return p + (active ? (milOn ? 'активная · MIL' : 'активная') : 'сохранённая');
  }

  String get titleRu => kSubaruDtcDict[code]?.$1 ?? '';
}

class ReadinessReport {
  ReadinessReport({required this.milOn, required this.dtcCount, required this.monitors});
  final bool milOn;
  final int dtcCount;
  final Map<String, bool> monitors;
  int get ready => monitors.values.where((v) => v).length;
  int get total => monitors.length;
}

String decodeDtcBytes(int b1, int b2) {
  const systems = ['P', 'C', 'B', 'U'];
  final sys = systems[(b1 >> 6) & 0x03];
  final d1 = (b1 >> 4) & 0x03;
  String h(int v) => v.toRadixString(16).toUpperCase();
  return '$sys$d1${h(b1 & 0x0F)}${h((b2 >> 4) & 0x0F)}${h(b2 & 0x0F)}';
}

List<DiagDtc> parseSsm2DtcReply(List<int> b) {
  final out = <DiagDtc>[];
  if (b.isEmpty || b[0] != 0x58) return out;
  final count = b.length > 1 ? b[1] : 0;
  for (var i = 0; i < count; i++) {
    final x = 2 + i * 3;
    if (x + 2 >= b.length) break;
    if (b[x] == 0 && b[x + 1] == 0) continue;
    out.add(DiagDtc(decodeDtcBytes(b[x], b[x + 1]), b[x + 2], DiagProto.ssm2,
        origin: 'stored'));
  }
  return out;
}

List<DiagDtc> parseObdDtcReply(List<int> b, {String origin = 'stored'}) {
  const heads = <int, String>{0x43: 'stored', 0x47: 'pending', 0x4A: 'permanent'};
  final out = <DiagDtc>[];
  if (b.isEmpty || !heads.containsKey(b[0])) return out;
  final kind = heads[b[0]] ?? origin;
  for (var i = 1; i + 1 < b.length; i += 2) {
    if (b[i] == 0 && b[i + 1] == 0) continue;
    out.add(DiagDtc(decodeDtcBytes(b[i], b[i + 1]), 0x20, DiagProto.obd,
        origin: kind));
  }
  return out;
}

List<DiagDtc> parseObdMode3Reply(List<int> b) => parseObdDtcReply(b);

List<DiagDtc> parseUdsDtcReply(List<int> b) {
  final out = <DiagDtc>[];
  if (b.length < 3 || b[0] != 0x59) return out;
  for (var i = 3; i + 2 < b.length; i += 3) {
    if (b[i] == 0 && b[i + 1] == 0) continue;
    out.add(DiagDtc(decodeDtcBytes(b[i], b[i + 1]), b[i + 2], DiagProto.uds,
        origin: 'stored'));
  }
  return out;
}

ReadinessReport? parseReadiness(List<int> b) {
  if (b.length < 6 || b[0] != 0x41 || b[1] != 0x01) return null;
  final a = b[2], bb = b[3], c = b[4], d = b[5];
  final monitors = <String, bool>{
    'Пропуски зажигания': (bb & 0x01) == 0 || (bb & 0x10) == 0,
    'Топливная система': (bb & 0x02) == 0 || (bb & 0x20) == 0,
    'Компоненты': (bb & 0x04) == 0 || (bb & 0x40) == 0,
  };
  const names = <int, String>{
    0x01: 'Катализатор',
    0x02: 'Подогрев катализатора',
    0x04: 'EVAP (адсорбер)',
    0x08: 'Вторичный воздух',
    0x20: 'Датчики O2',
    0x40: 'Подогрев O2',
    0x80: 'EGR / VVT',
  };
  names.forEach((mask, label) {
    if ((c & mask) != 0) monitors[label] = (d & mask) == 0;
  });
  return ReadinessReport(
      milOn: (a & 0x80) != 0, dtcCount: a & 0x7F, monitors: monitors);
}

String? parseAsciiReply(List<int> b, int pid) {
  if (b.length < 3 || b[0] != 0x49 || b[1] != pid) return null;
  final chars = <int>[];
  for (var i = 3; i < b.length; i++) {
    if (b[i] >= 0x20 && b[i] <= 0x7E) chars.add(b[i]);
  }
  final t = String.fromCharCodes(chars).trim();
  return t.isEmpty ? null : t;
}

String? parseFreezeFrameDtc(List<int> b) {
  if (b.length < 5 || b[0] != 0x42) return null;
  if (b[3] == 0 && b[4] == 0) return null;
  return decodeDtcBytes(b[3], b[4]);
}

List<int>? parseDiagPayload(String text) {
  final flat = <int>[];
  for (final raw in text.split(RegExp(r'[\r\n]+'))) {
    var line = raw.trim();
    if (line.isEmpty) continue;
    if (RegExp(r'^\d+:').hasMatch(line)) {
      line = line.substring(line.indexOf(':') + 1).trim();
    }
    for (final t in line.split(RegExp(r'\s+')).where((x) => x.isNotEmpty)) {
      if (!RegExp(r'^[0-9A-Fa-f]{2,16}$').hasMatch(t) || t.length % 2 != 0) {
        return null;
      }
      for (var i = 0; i + 1 < t.length; i += 2) {
        flat.add(int.parse(t.substring(i, i + 2), radix: 16));
      }
    }
  }
  if (flat.isEmpty) return null;
  var b = flat;
  if (b.length > 1 && (b[0] & 0xF0) == 0 && (b[0] & 0x0F) <= 7) {
    final len = b[0] & 0x0F;
    b = b.sublist(1, 1 + len <= b.length ? 1 + len : b.length);
  }
  if (b.length >= 3 && b[0] == 0x07 && (b[1] & 0xF8) == 0xE8) b = b.sublist(2);
  return b.isEmpty ? null : b;
}

const Map<int, String> kNrMeanings = <int, String>{
  0x10: 'generalReject — блок отклонил запрос',
  0x11: 'serviceNotSupported — сервис не поддерживается',
  0x12: 'subFunctionNotSupported — подфункция не поддерживается',
  0x13: 'incorrectMessageLength — неверная длина',
  0x21: 'busyRepeatRequest — блок занят',
  0x22: 'conditionsNotCorrect — не выполнены условия (зажигание/обороты)',
  0x24: 'requestSequenceError — нужна сессия 10 03',
  0x31: 'requestOutOfRange — параметр вне диапазона',
  0x33: 'securityAccessDenied — нужен ключ дилера',
  0x7E: 'subFunctionNotSupportedInActiveSession',
  0x7F: 'serviceNotSupportedInActiveSession — нужна сессия 10 03',
  0x78: 'responsePending — блок думает',
};

class ServiceResult {
  ServiceResult.ok(this.title, this.detail)
      : success = true,
        negative = null;
  ServiceResult.fail(this.title, this.detail, {this.negative}) : success = false;
  final bool success;
  final String title, detail;
  final int? negative;
}

/// Сессия диагностики. Пишет каждый обмен в DiagTrace и НЕ врёт об успехе.
class DiagSession {
  DiagSession(this.elm, {DiagTrace? trace}) : trace = trace ?? DiagTrace();
  final ElmDriver elm;
  final DiagTrace trace;

  bool experimental = false;

  /// v0.14: без подключения даже не пытаемся — иначе UI показывал «готово».
  void _requireLink() {
    if (!elm.ready) {
      trace.error('Адаптер не подключён — команда НЕ отправлена');
      throw NotConnectedException();
    }
  }

  Future<String> _send(String command) async {
    _requireLink();
    trace.tx(command);
    final watch = Stopwatch()..start();
    try {
      final reply = await elm.diagnostic(command);
      trace.rx(visibleText(reply), ms: watch.elapsedMilliseconds);
      return reply;
    } catch (e) {
      trace.error('$command -> $e (${watch.elapsedMilliseconds} мс)');
      rethrow;
    }
  }

  Future<List<int>> _payload(String command) async {
    final reply = await _send(command);
    final bytes = parseDiagPayload(reply);
    if (bytes == null) {
      trace.info('$command: ответ не распознан как кадр данных');
      return const <int>[];
    }
    return bytes;
  }

  Future<void> _atEcu(EcuSlot slot) async {
    trace.info('Переключение на ${slot.name} (${slot.header}→${slot.responseId})');
    await _send('ATSH ${slot.header}');
    await _send('ATCRA ${slot.responseId}');
    await Future<void>.delayed(const Duration(milliseconds: 20));
  }

  int? _nrc(List<int> b) => (b.length >= 3 && b[0] == 0x7F) ? b[2] : null;

  // ------------------- ЧТЕНИЕ -------------------

  /// Честное чтение: answered=false означает «ЭБУ не ответил», а не «чисто».
  Future<DtcReadResult> readDtcDetailed(EcuSlot slot) async {
    await _atEcu(slot);
    final raws = <String>[];

    Future<DtcReadResult?> tryService(
      String cmd,
      int expectHead,
      String label,
      List<DiagDtc> Function(List<int>) parse,
    ) async {
      try {
        final b = await _payload(cmd);
        raws.add('$cmd -> ${b.map(hex2).join(' ')}');
        final nrc = _nrc(b);
        if (nrc != null) {
          trace.info('$label отклонён: ${kNrMeanings[nrc] ?? 'NRC 0x${hex2(nrc)}'}');
          return null;
        }
        if (b.isNotEmpty && b[0] == expectHead) {
          final codes = parse(b);
          trace.info('$label: ответ принят, кодов ${codes.length}');
          return DtcReadResult(
            codes: codes,
            answered: true,
            protocolUsed: label,
            rawReplies: raws,
          );
        }
      } catch (e) {
        if (e is NotConnectedException) rethrow;
        trace.info('$label: $e');
      }
      return null;
    }

    if (slot.proto == DiagProto.ssm2) {
      final r = await tryService('18 00 FF 00', 0x58, 'SSM2 18', parseSsm2DtcReply);
      if (r != null) return r;
    } else {
      final r = await tryService('19 02 AF', 0x59, 'UDS 19 02', parseUdsDtcReply);
      if (r != null) return r;
    }
    final obd =
        await tryService('03', 0x43, 'OBD 03', (b) => parseObdDtcReply(b));
    if (obd != null) return obd;

    trace.error('${slot.name}: ни один сервис не ответил');
    return DtcReadResult(
      codes: const [],
      answered: false,
      protocolUsed: '—',
      rawReplies: raws,
      note: 'Блок не ответил. Возможно, его нет на шине или другой протокол.',
    );
  }

  /// Совместимость: бросает, если ответа не было (раньше молча отдавал []).
  Future<List<DiagDtc>> readDtc(EcuSlot slot) async {
    final r = await readDtcDetailed(slot);
    if (!r.answered) throw ReplyError(r.note);
    return r.codes.cast<DiagDtc>();
  }

  Future<List<DiagDtc>> readPendingDtc(EcuSlot slot) async {
    await _atEcu(slot);
    return parseObdDtcReply(await _payload('07'), origin: 'pending');
  }

  Future<List<DiagDtc>> readPermanentDtc(EcuSlot slot) async {
    await _atEcu(slot);
    return parseObdDtcReply(await _payload('0A'), origin: 'permanent');
  }

  Future<DtcReadResult> readAllDtc(EcuSlot slot) async {
    final base = await readDtcDetailed(slot);
    final all = <DiagDtc>[...base.codes.cast<DiagDtc>()];
    for (final fn in [readPendingDtc, readPermanentDtc]) {
      try {
        for (final d in await fn(slot)) {
          if (!all.any((x) => x.code == d.code && x.origin == d.origin)) {
            all.add(d);
          }
        }
      } catch (e) {
        if (e is NotConnectedException) rethrow;
      }
    }
    return DtcReadResult(
      codes: all,
      answered: base.answered,
      protocolUsed: base.protocolUsed,
      rawReplies: base.rawReplies,
      note: base.note,
    );
  }

  Future<Map<String, DtcReadResult>> scanAllEcus({
    void Function(String name)? onProgress,
  }) async {
    _requireLink();
    final result = <String, DtcReadResult>{};
    for (final slot in EcuSlot.all) {
      onProgress?.call(slot.name);
      try {
        result[slot.name] = await readDtcDetailed(slot);
      } catch (e) {
        if (e is NotConnectedException) rethrow;
        result[slot.name] = DtcReadResult(
          codes: const [],
          answered: false,
          protocolUsed: '—',
          rawReplies: const [],
          note: '$e',
        );
      }
    }
    await restore();
    return result;
  }

  // ------------------- СБРОС -------------------

  Future<DtcReadResult> clearDtc(EcuSlot slot) async {
    await _atEcu(slot);
    var done = false;
    final primary = slot.proto == DiagProto.ssm2 ? '14 FF FF 00' : '14 FF FF FF';
    final b = await _payload(primary);
    final nrc = _nrc(b);
    if (nrc != null) {
      throw ReplyError(
          'Сброс отклонён: ${kNrMeanings[nrc] ?? 'NRC 0x${hex2(nrc)}'}');
    }
    done = b.isNotEmpty && b[0] == 0x54;
    if (!done) {
      final o = await _payload('04');
      done = o.isNotEmpty && o[0] == 0x44;
    }
    if (!done) throw ReplyError('ЭБУ не подтвердил сброс (нет 54/44)');
    trace.info('Сброс подтверждён, повторное чтение…');
    await Future<void>.delayed(const Duration(milliseconds: 400));
    return readDtcDetailed(slot);
  }

  // ------------------- ИНФО -------------------

  Future<ReadinessReport?> readReadiness(EcuSlot slot) async {
    await _atEcu(slot);
    return parseReadiness(await _payload('01 01'));
  }

  Future<String?> readVin(EcuSlot slot) async {
    await _atEcu(slot);
    final v = parseAsciiReply(await _payload('09 02'), 0x02);
    if (v != null) return v;
    final b = await _payload('22 F1 90');
    if (b.length > 3 && b[0] == 0x62) {
      final t = String.fromCharCodes(
              b.sublist(3).where((x) => x >= 0x20 && x <= 0x7E))
          .trim();
      return t.isEmpty ? null : t;
    }
    return null;
  }

  Future<String?> readCalId(EcuSlot slot) async {
    await _atEcu(slot);
    return parseAsciiReply(await _payload('09 04'), 0x04);
  }

  Future<String?> readFreezeFrameDtc(EcuSlot slot) async {
    await _atEcu(slot);
    return parseFreezeFrameDtc(await _payload('02 02 00'));
  }

  /// Пинг связи: ATI должен вернуть версию адаптера.
  Future<String> pingAdapter() async {
    final reply = await _send('ATI');
    return visibleText(reply);
  }

  // ------------------- СЕРВИС -------------------

  Future<bool> enterExtendedSession(EcuSlot slot) async {
    await _atEcu(slot);
    final b = await _payload('10 03');
    final ok = b.isNotEmpty && b[0] == 0x50;
    trace.info(ok ? 'Расширенная сессия открыта' : 'Сессия 10 03 не принята');
    return ok;
  }

  Future<void> testerPresent() async {
    try {
      await _payload('3E 00');
    } catch (_) {}
  }

  Future<ServiceResult> ecuReset(EcuSlot slot) async {
    if (!experimental) {
      return ServiceResult.fail('ECUReset заблокирован',
          'Включите «Экспертные операции».');
    }
    await enterExtendedSession(slot);
    final b = await _payload('11 01');
    final nrc = _nrc(b);
    if (nrc != null) {
      return ServiceResult.fail('ECUReset отклонён',
          kNrMeanings[nrc] ?? 'NRC 0x${hex2(nrc)}',
          negative: nrc);
    }
    if (b.isNotEmpty && b[0] == 0x51) {
      return ServiceResult.ok(
          'ECU перезапущен', 'Ответ 51 01. Зажигание OFF→ON перед запуском.');
    }
    return ServiceResult.fail('Нет подтверждения',
        'Ожидался 51 01, получено: ${b.isEmpty ? '(пусто)' : b.map(hex2).join(' ')}');
  }

  Future<ServiceResult> clearMemory(EcuSlot slot, int group) async {
    if (!experimental) {
      return ServiceResult.fail('Clear Memory заблокирован',
          'Включите «Экспертные операции».');
    }
    if (group < 1 || group > 7) {
      return ServiceResult.fail('Неверная группа', 'Допустимо 1..7');
    }
    await enterExtendedSession(slot);
    final b = await _payload('04 ${hex2(group)}');
    final nrc = _nrc(b);
    if (nrc != null) {
      return ServiceResult.fail('Группа $group отклонена',
          kNrMeanings[nrc] ?? 'NRC 0x${hex2(nrc)}',
          negative: nrc);
    }
    if (b.isNotEmpty && b[0] == 0x44) {
      return ServiceResult.ok('Адаптации группы $group сброшены',
          'Подтверждено (44). Нужно дообучение 15–20 мин.');
    }
    return ServiceResult.fail('Нет подтверждения',
        'Ожидался 44, получено: ${b.isEmpty ? '(пусто)' : b.map(hex2).join(' ')}');
  }

  Future<Map<int, String>> probeClearMemoryGroups(EcuSlot slot) async {
    final out = <int, String>{};
    if (!experimental) return out;
    await enterExtendedSession(slot);
    for (var g = 1; g <= 7; g++) {
      try {
        final b = await _payload('04 ${hex2(g)}');
        final nrc = _nrc(b);
        out[g] = nrc != null
            ? (kNrMeanings[nrc] ?? 'NRC 0x${hex2(nrc)}')
            : (b.isNotEmpty && b[0] == 0x44 ? 'поддерживается (44)' : 'нет ответа');
      } catch (e) {
        if (e is NotConnectedException) rethrow;
        out[g] = 'ошибка связи';
      }
      await Future<void>.delayed(const Duration(milliseconds: 120));
    }
    return out;
  }

  Future<void> restore() async {
    try {
      await _send('ATSH 7E0');
      await _send('ATCRA 7E8');
      trace.info('Заголовки возвращены на двигатель (7E0/7E8)');
    } catch (_) {}
  }
}
'''

# ===== Окно журнала обмена: TX/RX, тайминги, копирование =====
FILES["lib/trace_sheet.dart"] = r'''
import 'package:flutter/material.dart';
import 'package:flutter/services.dart';

import 'diag_trace.dart';

const _muted = Color(0xFF8CA0BF);

/// Плавающее окно журнала обмена: что ушло в ЭБУ и что пришло обратно.
class TraceSheet extends StatelessWidget {
  const TraceSheet({super.key, required this.trace});
  final DiagTrace trace;

  static Future<void> show(BuildContext context, DiagTrace trace) =>
      showModalBottomSheet<void>(
        context: context,
        isScrollControlled: true,
        showDragHandle: true,
        builder: (_) => TraceSheet(trace: trace),
      );

  Color _color(TraceDir d) => switch (d) {
        TraceDir.tx => const Color(0xFF7DD3FC),
        TraceDir.rx => const Color(0xFF86EFAC),
        TraceDir.info => _muted,
        TraceDir.error => const Color(0xFFFF8A80),
      };

  @override
  Widget build(BuildContext context) => DraggableScrollableSheet(
        expand: false,
        initialChildSize: 0.75,
        maxChildSize: 0.95,
        builder: (context, scroll) => AnimatedBuilder(
          animation: trace,
          builder: (context, _) => Column(
            children: [
              Padding(
                padding: const EdgeInsets.fromLTRB(16, 0, 8, 6),
                child: Row(
                  children: [
                    const Icon(Icons.terminal, size: 18),
                    const SizedBox(width: 8),
                    Expanded(
                      child: Text(
                        'Журнал обмена · TX ${trace.txCount} / RX ${trace.rxCount}'
                        '${trace.errorCount > 0 ? ' / ошибок ${trace.errorCount}' : ''}',
                        style: const TextStyle(
                            fontSize: 14, fontWeight: FontWeight.bold),
                      ),
                    ),
                    IconButton(
                      tooltip: 'Скопировать',
                      onPressed: () async {
                        await Clipboard.setData(
                            ClipboardData(text: trace.asText()));
                        if (context.mounted) {
                          ScaffoldMessenger.of(context).showSnackBar(
                            const SnackBar(
                                content: Text('Журнал скопирован'),
                                duration: Duration(seconds: 1)),
                          );
                        }
                      },
                      icon: const Icon(Icons.copy, size: 18),
                    ),
                    IconButton(
                      tooltip: 'Очистить',
                      onPressed: trace.clear,
                      icon: const Icon(Icons.delete_outline, size: 18),
                    ),
                  ],
                ),
              ),
              const Divider(height: 1),
              Expanded(
                child: trace.entries.isEmpty
                    ? const Center(
                        child: Padding(
                          padding: EdgeInsets.all(24),
                          child: Text(
                            'Журнал пуст. Выполните любую операцию во вкладке '
                            'DTC — здесь появятся отправленные команды и '
                            'ответы ЭБУ.',
                            textAlign: TextAlign.center,
                            style: TextStyle(color: _muted),
                          ),
                        ),
                      )
                    : ListView.builder(
                        controller: scroll,
                        padding: const EdgeInsets.symmetric(
                            horizontal: 12, vertical: 8),
                        itemCount: trace.entries.length,
                        itemBuilder: (context, i) {
                          final e = trace.entries[trace.entries.length - 1 - i];
                          return Padding(
                            padding: const EdgeInsets.symmetric(vertical: 1.5),
                            child: Row(
                              crossAxisAlignment: CrossAxisAlignment.start,
                              children: [
                                Text(e.stamp,
                                    style: const TextStyle(
                                        fontFamily: 'monospace',
                                        fontSize: 10,
                                        color: _muted)),
                                const SizedBox(width: 6),
                                Text(e.arrow,
                                    style: TextStyle(
                                        fontFamily: 'monospace',
                                        fontSize: 11,
                                        color: _color(e.dir))),
                                const SizedBox(width: 6),
                                Expanded(
                                  child: Text(
                                    e.text,
                                    style: TextStyle(
                                        fontFamily: 'monospace',
                                        fontSize: 11.5,
                                        color: _color(e.dir)),
                                  ),
                                ),
                                if (e.ms != null)
                                  Text('${e.ms}мс',
                                      style: const TextStyle(
                                          fontFamily: 'monospace',
                                          fontSize: 9.5,
                                          color: _muted)),
                              ],
                            ),
                          );
                        },
                      ),
              ),
            ],
          ),
        ),
      );
}
'''

# ===== Карточка поездки: расход, одометр, средний, сброс =====
FILES["lib/trip_card.dart"] = r'''
import 'package:flutter/material.dart';

import 'virtual_tiles.dart';

const _muted = Color(0xFF8CA0BF);

/// Карточка поездки: мгновенный расход, л/100км, одометр, литры, средний,
/// кнопка сброса. Возвращена по просьбе: была в 0.10, пропала в 0.12.
class TripCard extends StatelessWidget {
  const TripCard({
    super.key,
    required this.inputs,
    required this.onReset,
    this.mafStale = false,
  });

  final TileInputs inputs;
  final VoidCallback onReset;
  final bool mafStale;

  String _fmt(double? v, int digits, {String dash = '—'}) =>
      v == null ? dash : v.toStringAsFixed(digits);

  @override
  Widget build(BuildContext context) {
    final lph = virtualTileById('@FUEL_LPH')?.compute(inputs);
    final l100 = virtualTileById('@FUEL_L100')?.compute(inputs);
    final avg = virtualTileById('@TRIP_AVG')?.compute(inputs);

    return Card(
      margin: const EdgeInsets.fromLTRB(10, 6, 10, 0),
      child: Padding(
        padding: const EdgeInsets.fromLTRB(12, 10, 8, 10),
        child: Column(
          children: [
            Row(
              children: [
                Icon(
                  Icons.local_gas_station,
                  size: 18,
                  color: mafStale ? Colors.orangeAccent : _muted,
                ),
                const SizedBox(width: 8),
                const Expanded(
                  child: Text(
                    'Поездка',
                    style: TextStyle(fontSize: 13, fontWeight: FontWeight.bold),
                  ),
                ),
                if (mafStale)
                  const Text('MAF устарел',
                      style: TextStyle(fontSize: 10, color: Colors.orangeAccent)),
                IconButton(
                  visualDensity: VisualDensity.compact,
                  tooltip: 'Сбросить одометр и расход',
                  onPressed: onReset,
                  icon: const Icon(Icons.restart_alt, size: 20),
                ),
              ],
            ),
            const SizedBox(height: 4),
            Row(
              children: [
                _Cell(label: 'Расход', value: _fmt(lph, 1), unit: 'л/ч'),
                _Cell(label: 'Мгновенный', value: _fmt(l100, 1), unit: 'л/100км'),
                _Cell(
                  label: 'Одометр',
                  value: inputs.tripKm.toStringAsFixed(2),
                  unit: 'км',
                ),
                _Cell(
                  label: 'Залито',
                  value: inputs.tripLitres.toStringAsFixed(2),
                  unit: 'л',
                ),
                _Cell(label: 'Средний', value: _fmt(avg, 1), unit: 'л/100км'),
              ],
            ),
          ],
        ),
      ),
    );
  }
}

class _Cell extends StatelessWidget {
  const _Cell({required this.label, required this.value, required this.unit});
  final String label, value, unit;

  @override
  Widget build(BuildContext context) => Expanded(
        child: Column(
          children: [
            Text(label,
                maxLines: 1,
                overflow: TextOverflow.ellipsis,
                style: const TextStyle(fontSize: 9.5, color: _muted)),
            const SizedBox(height: 2),
            FittedBox(
              fit: BoxFit.scaleDown,
              child: Text(
                value,
                style: const TextStyle(
                    fontSize: 17, fontWeight: FontWeight.w600),
              ),
            ),
            Text(unit, style: const TextStyle(fontSize: 9, color: _muted)),
          ],
        ),
      );
}
'''

# ===== Сетка колонки×строки, комбо-ячейки, перестановка =====
FILES["lib/dashboard_prefs.dart"] = r'''
import 'pids.dart';
import 'virtual_tiles.dart';

/// Разделитель PID внутри объединённой ячейки: 'RPM|AFR|ECT'.
const String kComboSeparator = '|';

bool isComboTile(String id) => id.contains(kComboSeparator);
List<String> comboParts(String id) =>
    id.split(kComboSeparator).where((s) => s.isNotEmpty).toList();

/// Настройки дашборда: состав, порядок, сетка (колонки × строки), плотность.
class DashboardPrefs {
  DashboardPrefs({
    List<String>? tiles,
    this.columnsPortrait = 2,
    this.columnsLandscape = 4,
    this.rowsPortrait = 4,
    this.rowsLandscape = 3,
    this.fitToScreen = true,
    this.compact = false,
    this.showSpark = true,
    this.showFuelCard = true,
  }) : tiles = tiles ?? List<String>.from(defaultTiles);

  final List<String> tiles;

  /// Ячеек в строке.
  final int columnsPortrait;
  final int columnsLandscape;

  /// Строк на экран (при fitToScreen высота плитки подгоняется).
  final int rowsPortrait;
  final int rowsLandscape;

  /// true — вместить ровно columns×rows в видимую область (без прокрутки).
  final bool fitToScreen;

  final bool compact;
  final bool showSpark;
  final bool showFuelCard;

  static const List<String> defaultTiles = <String>[
    'RPM',
    'MAP_REL',
    'AFR',
    'ECT',
    'FBKC',
    'IAM',
    '@FUEL_L100',
    '@TRIP_KM',
  ];

  int columnsFor(bool landscape) =>
      landscape ? columnsLandscape : columnsPortrait;
  int rowsFor(bool landscape) => landscape ? rowsLandscape : rowsPortrait;

  /// Сколько ячеек помещается на один экран.
  int perScreen(bool landscape) => columnsFor(landscape) * rowsFor(landscape);

  static bool knownTile(String id) {
    if (isComboTile(id)) {
      final parts = comboParts(id);
      return parts.length >= 2 && parts.every(knownTile);
    }
    return isVirtualTile(id)
        ? virtualTileById(id) != null
        : SubaruPidLibrary.all.any((p) => p.id == id);
  }

  DashboardPrefs copyWith({
    List<String>? tiles,
    int? columnsPortrait,
    int? columnsLandscape,
    int? rowsPortrait,
    int? rowsLandscape,
    bool? fitToScreen,
    bool? compact,
    bool? showSpark,
    bool? showFuelCard,
  }) =>
      DashboardPrefs(
        tiles: tiles ?? List<String>.from(this.tiles),
        columnsPortrait: columnsPortrait ?? this.columnsPortrait,
        columnsLandscape: columnsLandscape ?? this.columnsLandscape,
        rowsPortrait: rowsPortrait ?? this.rowsPortrait,
        rowsLandscape: rowsLandscape ?? this.rowsLandscape,
        fitToScreen: fitToScreen ?? this.fitToScreen,
        compact: compact ?? this.compact,
        showSpark: showSpark ?? this.showSpark,
        showFuelCard: showFuelCard ?? this.showFuelCard,
      );

  /// Переставить плитку (drag & drop прямо на дашборде).
  DashboardPrefs moveTile(String id, int newIndex) {
    final list = List<String>.from(tiles);
    final old = list.indexOf(id);
    if (old < 0) return this;
    list.removeAt(old);
    final idx = newIndex.clamp(0, list.length);
    list.insert(idx, id);
    return copyWith(tiles: list);
  }

  /// Поменять две плитки местами.
  DashboardPrefs swapTiles(String a, String b) {
    final list = List<String>.from(tiles);
    final ia = list.indexOf(a), ib = list.indexOf(b);
    if (ia < 0 || ib < 0) return this;
    list[ia] = b;
    list[ib] = a;
    return copyWith(tiles: list);
  }

  Map<String, dynamic> toJson() => <String, dynamic>{
        'tiles': tiles,
        'columnsPortrait': columnsPortrait,
        'columnsLandscape': columnsLandscape,
        'rowsPortrait': rowsPortrait,
        'rowsLandscape': rowsLandscape,
        'fitToScreen': fitToScreen,
        'compact': compact,
        'showSpark': showSpark,
        'showFuelCard': showFuelCard,
      };

  static DashboardPrefs fromJson(Map<String, dynamic>? json) {
    if (json == null) return DashboardPrefs();
    final raw = (json['tiles'] as List<dynamic>? ?? const <dynamic>[])
        .whereType<String>()
        .where(knownTile)
        .toList();
    int clamp(Object? v, int fallback, int lo, int hi) {
      final n = v is num ? v.toInt() : fallback;
      return n < lo ? lo : (n > hi ? hi : n);
    }

    return DashboardPrefs(
      tiles: raw.isEmpty ? null : raw,
      columnsPortrait: clamp(json['columnsPortrait'], 2, 1, 5),
      columnsLandscape: clamp(json['columnsLandscape'], 4, 1, 8),
      rowsPortrait: clamp(json['rowsPortrait'], 4, 1, 8),
      rowsLandscape: clamp(json['rowsLandscape'], 3, 1, 6),
      fitToScreen: json['fitToScreen'] != false,
      compact: json['compact'] == true,
      showSpark: json['showSpark'] != false,
      showFuelCard: json['showFuelCard'] != false,
    );
  }

  /// Плитка доступна, если это виртуальная, активный PID или комбо,
  /// у которого есть хотя бы один доступный элемент.
  bool tileAvailable(String id, Set<String> enabledPids) {
    if (isComboTile(id)) {
      return comboParts(id).any((p) => tileAvailable(p, enabledPids));
    }
    return isVirtualTile(id) || enabledPids.contains(id);
  }

  List<String> visibleTiles(Set<String> enabledPids) =>
      tiles.where((id) => tileAvailable(id, enabledPids)).toList();
}

enum OrientationMode {
  auto('Авто (по датчику)'),
  portrait('Только книжная'),
  landscape('Только альбомная');

  const OrientationMode(this.label);
  final String label;
}

class UiPrefs {
  const UiPrefs({this.orientation = OrientationMode.auto, this.keepAwake = true});
  final OrientationMode orientation;
  final bool keepAwake;

  UiPrefs copyWith({OrientationMode? orientation, bool? keepAwake}) => UiPrefs(
        orientation: orientation ?? this.orientation,
        keepAwake: keepAwake ?? this.keepAwake,
      );

  Map<String, dynamic> toJson() => <String, dynamic>{
        'orientation': orientation.name,
        'keepAwake': keepAwake,
      };

  static UiPrefs fromJson(Map<String, dynamic>? json) {
    if (json == null) return const UiPrefs();
    return UiPrefs(
      orientation: OrientationMode.values.firstWhere(
        (o) => o.name == json['orientation'],
        orElse: () => OrientationMode.auto,
      ),
      keepAwake: json['keepAwake'] != false,
    );
  }
}
'''

# ===== Дашборд: drag прямо на плитках, объединённые ячейки, fit-сетка =====
FILES["lib/dashboard_page.dart"] = r'''
import 'package:flutter/material.dart';

import 'dashboard_prefs.dart';
import 'model.dart';
import 'pids.dart';
import 'samples.dart';
import 'trip_card.dart';
import 'virtual_tiles.dart';

const _muted = Color(0xFF8CA0BF);
const _accent = Color(0xFF22D3EE);

/// Дашборд: сетка колонки×строки, drag прямо на плитках, объединённые ячейки.
class CustomDashboardPage extends StatefulWidget {
  const CustomDashboardPage(this.model, {super.key});
  final AppModel model;

  @override
  State<CustomDashboardPage> createState() => _CustomDashboardPageState();
}

class _CustomDashboardPageState extends State<CustomDashboardPage> {
  AppModel get model => widget.model;
  String? _dragging;

  TileInputs _inputs() {
    final values = <String, double>{};
    model.engine.latest.forEach((id, s) {
      final v = s.value;
      if (v != null && v.isFinite) values[id] = v;
    });
    return TileInputs(
      values: values,
      config: model.vehicleConfig,
      tripKm: model.engine.tripDistanceKm,
      tripLitres: model.engine.tripFuelLitres,
    );
  }

  Future<void> _reorder(String dragged, String target) async {
    final prefs = model.dashboardPrefs;
    final idx = prefs.tiles.indexOf(target);
    if (idx < 0) return;
    await model.updateDashboardPrefs(prefs.moveTile(dragged, idx));
    if (mounted) setState(() {});
  }

  @override
  Widget build(BuildContext context) {
    final engine = model.engine;
    final prefs = model.dashboardPrefs;
    final activeIds = engine.active.map((p) => p.id).toSet();

    final chosen = prefs.visibleTiles(activeIds);
    final rest = engine.active
        .map((p) => p.id)
        .where((id) => !prefs.tiles.any((t) =>
            t == id || (isComboTile(t) && comboParts(t).contains(id))));
    final ids = <String>[...chosen, ...rest];

    final landscape =
        MediaQuery.of(context).orientation == Orientation.landscape;
    final columns = prefs.columnsFor(landscape);
    final rows = prefs.rowsFor(landscape);
    final inputs = _inputs();

    return Column(
      children: [
        Padding(
          padding: const EdgeInsets.fromLTRB(12, 4, 6, 0),
          child: Row(
            children: [
              Expanded(
                child: Text(
                  '${ids.length} ячеек · ${columns}×$rows на экран · '
                  '${engine.pidReadsPerSecond.toStringAsFixed(1)} обновл./с',
                  style: const TextStyle(fontSize: 11, color: _muted),
                ),
              ),
              IconButton(
                tooltip: 'Настроить сетку и ячейки',
                onPressed: () => _configure(context),
                icon: const Icon(Icons.dashboard_customize),
              ),
              FilledButton.tonalIcon(
                onPressed:
                    model.busy || !model.elm.ready ? null : model.togglePolling,
                icon: Icon(engine.running ? Icons.pause : Icons.play_arrow),
                label: Text(engine.running ? 'Пауза' : 'Старт'),
              ),
            ],
          ),
        ),
        if (prefs.showFuelCard)
          TripCard(
            inputs: inputs,
            mafStale: engine.stale('MAF'),
            onReset: () {
              engine.resetTrip();
              setState(() {});
            },
          ),
        if (ids.isEmpty)
          const Expanded(
            child: Center(
              child: Padding(
                padding: EdgeInsets.all(24),
                child: Text(
                  'Нет ячеек. Нажмите «Настроить сетку» — добавьте PID, '
                  'расчётные плитки или объедините несколько PID в одну ячейку.',
                  textAlign: TextAlign.center,
                ),
              ),
            ),
          )
        else
          Expanded(
            child: LayoutBuilder(
              builder: (context, box) {
                const pad = 8.0, gap = 6.0;
                final extent = prefs.fitToScreen
                    ? ((box.maxHeight - pad * 2 - gap * (rows - 1)) / rows)
                        .clamp(58.0, 400.0)
                    : (prefs.compact ? 92.0 : 136.0);
                return GridView.builder(
                  padding: const EdgeInsets.all(pad),
                  physics: prefs.fitToScreen && ids.length <= columns * rows
                      ? const NeverScrollableScrollPhysics()
                      : const AlwaysScrollableScrollPhysics(),
                  itemCount: ids.length,
                  gridDelegate: SliverGridDelegateWithFixedCrossAxisCount(
                    crossAxisCount: columns,
                    mainAxisExtent: extent.toDouble(),
                    mainAxisSpacing: gap,
                    crossAxisSpacing: gap,
                  ),
                  itemBuilder: (context, i) => _draggable(
                    ids[i],
                    _buildTile(ids[i], inputs, extent.toDouble()),
                  ),
                );
              },
            ),
          ),
      ],
    );
  }

  Widget _draggable(String id, Widget child) => DragTarget<String>(
        onWillAcceptWithDetails: (d) => d.data != id,
        onAcceptWithDetails: (d) => _reorder(d.data, id),
        builder: (context, candidate, rejected) => LongPressDraggable<String>(
          data: id,
          onDragStarted: () => setState(() => _dragging = id),
          onDragEnd: (_) => setState(() => _dragging = null),
          feedback: Material(
            color: Colors.transparent,
            child: Opacity(
              opacity: 0.9,
              child: SizedBox(width: 160, height: 92, child: child),
            ),
          ),
          childWhenDragging: Opacity(opacity: 0.25, child: child),
          child: AnimatedContainer(
            duration: const Duration(milliseconds: 150),
            decoration: BoxDecoration(
              borderRadius: BorderRadius.circular(12),
              border: candidate.isNotEmpty
                  ? Border.all(color: _accent, width: 2)
                  : (_dragging == id
                      ? Border.all(color: _accent.withValues(alpha: 0.4))
                      : null),
            ),
            child: child,
          ),
        ),
      );

  Widget _buildTile(String id, TileInputs inputs, double extent) {
    final engine = model.engine;
    final prefs = model.dashboardPrefs;
    final tight = extent < 110;

    if (isComboTile(id)) {
      final parts = comboParts(id);
      return _ComboTile(
        parts: parts,
        resolve: (p) => _resolve(p, inputs),
        tight: tight,
      );
    }
    final r = _resolve(id, inputs);
    return _Tile(
      title: r.title,
      unit: r.unit,
      digits: r.digits,
      value: r.value,
      isVirtual: r.virtual,
      stale: r.stale,
      unsupported: r.unsupported,
      tight: tight,
      history: prefs.showSpark && !tight && !r.virtual
          ? engine.history[id]
          : null,
    );
  }

  _Resolved _resolve(String id, TileInputs inputs) {
    if (isVirtualTile(id)) {
      final d = virtualTileById(id);
      if (d == null) {
        return const _Resolved('?', '', 0, null, virtual: true);
      }
      return _Resolved(d.label, d.unit, d.digits, d.compute(inputs),
          virtual: true);
    }
    final matches = SubaruPidLibrary.all.where((p) => p.id == id);
    if (matches.isEmpty) return _Resolved(id, '', 0, null);
    final pid = matches.first;
    final s = model.engine.latest[id];
    return _Resolved(
      '${pid.id}${pid.extended ? ' *' : ''}',
      pid.unit,
      pid.digits,
      s?.value,
      stale: (model.engine.ageMs(id) ?? 999999) > 2500,
      unsupported: s?.allOnes ?? false,
    );
  }

  Future<void> _configure(BuildContext context) async {
    final result = await showModalBottomSheet<DashboardPrefs>(
      context: context,
      isScrollControlled: true,
      showDragHandle: true,
      builder: (_) => _ConfigSheet(model: model),
    );
    if (result != null) {
      await model.updateDashboardPrefs(result);
      if (mounted) setState(() {});
    }
  }
}

class _Resolved {
  const _Resolved(
    this.title,
    this.unit,
    this.digits,
    this.value, {
    this.virtual = false,
    this.stale = false,
    this.unsupported = false,
  });
  final String title, unit;
  final int digits;
  final double? value;
  final bool virtual, stale, unsupported;

  String get text => value == null
      ? (unsupported ? 'н/д' : '—')
      : value!.toStringAsFixed(digits);
}

class _Tile extends StatelessWidget {
  const _Tile({
    required this.title,
    required this.unit,
    required this.digits,
    required this.value,
    required this.isVirtual,
    required this.tight,
    this.stale = false,
    this.unsupported = false,
    this.history,
  });

  final String title, unit;
  final int digits;
  final double? value;
  final bool isVirtual, tight, stale, unsupported;
  final List<PidSample>? history;

  @override
  Widget build(BuildContext context) {
    final text =
        value == null ? (unsupported ? 'н/д' : '—') : value!.toStringAsFixed(digits);
    return Card(
      margin: EdgeInsets.zero,
      child: Padding(
        padding: EdgeInsets.all(tight ? 6 : 10),
        child: Column(
          crossAxisAlignment: CrossAxisAlignment.start,
          mainAxisAlignment: MainAxisAlignment.center,
          children: [
            Row(
              children: [
                if (isVirtual)
                  const Padding(
                    padding: EdgeInsets.only(right: 3),
                    child: Icon(Icons.functions, size: 10, color: _accent),
                  ),
                Expanded(
                  child: Text(
                    title,
                    maxLines: 1,
                    overflow: TextOverflow.ellipsis,
                    style: TextStyle(
                      fontSize: tight ? 9.5 : 11.5,
                      fontWeight: FontWeight.bold,
                      color: stale ? _muted : null,
                    ),
                  ),
                ),
                if (stale) const Icon(Icons.schedule, size: 10, color: _muted),
              ],
            ),
            FittedBox(
              fit: BoxFit.scaleDown,
              alignment: Alignment.centerLeft,
              child: Text(
                text,
                style: TextStyle(
                  fontSize: tight ? 19 : 27,
                  fontWeight: FontWeight.w600,
                ),
              ),
            ),
            Text(unit,
                maxLines: 1,
                overflow: TextOverflow.ellipsis,
                style: TextStyle(fontSize: tight ? 8.5 : 10, color: _muted)),
            if (history != null && history!.length > 2 && !tight)
              Expanded(
                child: CustomPaint(
                  painter: _SparkPainter(history!),
                  size: Size.infinite,
                ),
              ),
          ],
        ),
      ),
    );
  }
}

/// Объединённая ячейка: несколько параметров в одной плитке.
class _ComboTile extends StatelessWidget {
  const _ComboTile({
    required this.parts,
    required this.resolve,
    required this.tight,
  });
  final List<String> parts;
  final _Resolved Function(String) resolve;
  final bool tight;

  @override
  Widget build(BuildContext context) => Card(
        margin: EdgeInsets.zero,
        child: Padding(
          padding: EdgeInsets.all(tight ? 5 : 8),
          child: Column(
            mainAxisAlignment: MainAxisAlignment.spaceEvenly,
            children: [
              for (final p in parts.take(4))
                Builder(builder: (context) {
                  final r = resolve(p);
                  return Row(
                    children: [
                      Expanded(
                        child: Text(
                          r.title,
                          maxLines: 1,
                          overflow: TextOverflow.ellipsis,
                          style: TextStyle(
                            fontSize: tight ? 8.5 : 10,
                            color: r.stale ? _muted : Colors.white70,
                          ),
                        ),
                      ),
                      const SizedBox(width: 4),
                      Text(
                        r.text,
                        style: TextStyle(
                          fontSize: tight ? 13 : 16,
                          fontWeight: FontWeight.w600,
                        ),
                      ),
                      const SizedBox(width: 2),
                      Text(
                        r.unit,
                        style: TextStyle(
                            fontSize: tight ? 7.5 : 9, color: _muted),
                      ),
                    ],
                  );
                }),
            ],
          ),
        ),
      );
}

class _SparkPainter extends CustomPainter {
  _SparkPainter(this.samples);
  final List<PidSample> samples;

  @override
  void paint(Canvas canvas, Size size) {
    final v = samples.where((s) => s.value != null).map((s) => s.value!).toList();
    if (v.length < 2) return;
    var lo = v.first, hi = v.first;
    for (final x in v) {
      if (x < lo) lo = x;
      if (x > hi) hi = x;
    }
    if ((hi - lo).abs() < 1e-9) hi = lo + 1;
    final path = Path();
    for (var i = 0; i < v.length; i++) {
      final x = size.width * i / (v.length - 1);
      final y = size.height * (1 - (v[i] - lo) / (hi - lo));
      i == 0 ? path.moveTo(x, y) : path.lineTo(x, y);
    }
    canvas.drawPath(
      path,
      Paint()
        ..color = _accent
        ..style = PaintingStyle.stroke
        ..strokeWidth = 1.4,
    );
  }

  @override
  bool shouldRepaint(covariant _SparkPainter old) => true;
}

/// Лист настроек: сетка, список ячеек, конструктор объединённых ячеек.
class _ConfigSheet extends StatefulWidget {
  const _ConfigSheet({required this.model});
  final AppModel model;
  @override
  State<_ConfigSheet> createState() => _ConfigSheetState();
}

class _ConfigSheetState extends State<_ConfigSheet> {
  late DashboardPrefs prefs = widget.model.dashboardPrefs;
  final Set<String> _comboPick = <String>{};

  String _label(String id) {
    if (isComboTile(id)) {
      return comboParts(id).map(_label).join(' + ');
    }
    if (isVirtualTile(id)) {
      final d = virtualTileById(id);
      return d == null ? id : d.label;
    }
    return id;
  }

  @override
  Widget build(BuildContext context) {
    final activeIds = widget.model.engine.active.map((p) => p.id).toList();
    final chosen = prefs.visibleTiles(activeIds.toSet());
    final availablePids = activeIds
        .where((id) => !prefs.tiles.any(
            (t) => t == id || (isComboTile(t) && comboParts(t).contains(id))))
        .toList();
    final availableVirtual = kVirtualTiles
        .map((t) => t.id)
        .where((id) => !prefs.tiles.contains(id))
        .toList();

    return DraggableScrollableSheet(
      expand: false,
      initialChildSize: 0.88,
      maxChildSize: 0.96,
      builder: (context, scroll) => ListView(
        controller: scroll,
        padding: const EdgeInsets.fromLTRB(16, 0, 16, 24),
        children: [
          const Text('Настройка дашборда',
              style: TextStyle(fontSize: 18, fontWeight: FontWeight.bold)),
          const Text(
            'Ячейки можно перетаскивать прямо на дашборде: долгое нажатие '
            'и перенос на соседнюю плитку.',
            style: TextStyle(fontSize: 12, color: _muted),
          ),
          const SizedBox(height: 14),

          const Text('Сетка', style: TextStyle(fontWeight: FontWeight.bold)),
          Row(
            children: [
              Expanded(
                child: _Stepper(
                  label: 'В строке (книжная)',
                  value: prefs.columnsPortrait,
                  min: 1,
                  max: 5,
                  onChanged: (v) =>
                      setState(() => prefs = prefs.copyWith(columnsPortrait: v)),
                ),
              ),
              Expanded(
                child: _Stepper(
                  label: 'Строк (книжная)',
                  value: prefs.rowsPortrait,
                  min: 1,
                  max: 8,
                  onChanged: (v) =>
                      setState(() => prefs = prefs.copyWith(rowsPortrait: v)),
                ),
              ),
            ],
          ),
          Row(
            children: [
              Expanded(
                child: _Stepper(
                  label: 'В строке (альбом)',
                  value: prefs.columnsLandscape,
                  min: 1,
                  max: 8,
                  onChanged: (v) => setState(
                      () => prefs = prefs.copyWith(columnsLandscape: v)),
                ),
              ),
              Expanded(
                child: _Stepper(
                  label: 'Строк (альбом)',
                  value: prefs.rowsLandscape,
                  min: 1,
                  max: 6,
                  onChanged: (v) =>
                      setState(() => prefs = prefs.copyWith(rowsLandscape: v)),
                ),
              ),
            ],
          ),
          Padding(
            padding: const EdgeInsets.symmetric(vertical: 4),
            child: Text(
              'На экран: ${prefs.perScreen(false)} ячеек (книжная), '
              '${prefs.perScreen(true)} (альбомная)',
              style: const TextStyle(fontSize: 12, color: _accent),
            ),
          ),
          SwitchListTile(
            contentPadding: EdgeInsets.zero,
            title: const Text('Вместить без прокрутки'),
            subtitle: const Text(
                'Высота ячеек подгоняется под число строк',
                style: TextStyle(fontSize: 11)),
            value: prefs.fitToScreen,
            onChanged: (v) =>
                setState(() => prefs = prefs.copyWith(fitToScreen: v)),
          ),
          SwitchListTile(
            contentPadding: EdgeInsets.zero,
            title: const Text('Карточка поездки'),
            subtitle: const Text('Расход, одометр, средний + сброс',
                style: TextStyle(fontSize: 11)),
            value: prefs.showFuelCard,
            onChanged: (v) =>
                setState(() => prefs = prefs.copyWith(showFuelCard: v)),
          ),
          SwitchListTile(
            contentPadding: EdgeInsets.zero,
            title: const Text('Мини-графики'),
            value: prefs.showSpark,
            onChanged: (v) =>
                setState(() => prefs = prefs.copyWith(showSpark: v)),
          ),

          const Divider(height: 24),
          Text('Ячейки на дашборде (${chosen.length})',
              style: const TextStyle(fontWeight: FontWeight.bold)),
          ReorderableListView(
            shrinkWrap: true,
            physics: const NeverScrollableScrollPhysics(),
            buildDefaultDragHandles: false,
            onReorder: (o, n) => setState(() {
              final list = List<String>.from(chosen);
              if (n > o) n -= 1;
              list.insert(n, list.removeAt(o));
              final others =
                  prefs.tiles.where((id) => !chosen.contains(id)).toList();
              prefs = prefs.copyWith(tiles: [...list, ...others]);
            }),
            children: [
              for (var i = 0; i < chosen.length; i++)
                ListTile(
                  key: ValueKey('c-${chosen[i]}'),
                  dense: true,
                  contentPadding: EdgeInsets.zero,
                  leading: ReorderableDragStartListener(
                    index: i,
                    child: const Icon(Icons.drag_handle),
                  ),
                  title: Row(
                    children: [
                      if (isComboTile(chosen[i]))
                        const Padding(
                          padding: EdgeInsets.only(right: 6),
                          child: Icon(Icons.view_agenda,
                              size: 14, color: _accent),
                        )
                      else if (isVirtualTile(chosen[i]))
                        const Padding(
                          padding: EdgeInsets.only(right: 6),
                          child:
                              Icon(Icons.functions, size: 14, color: _accent),
                        ),
                      Expanded(
                        child: Text(_label(chosen[i]),
                            style: const TextStyle(fontSize: 13)),
                      ),
                    ],
                  ),
                  trailing: IconButton(
                    icon: const Icon(Icons.close, size: 18),
                    onPressed: () => setState(() {
                      final l = List<String>.from(prefs.tiles)
                        ..remove(chosen[i]);
                      prefs = prefs.copyWith(tiles: l);
                    }),
                  ),
                ),
            ],
          ),

          const Divider(height: 24),
          const Text('Объединить PID в одну ячейку',
              style: TextStyle(fontWeight: FontWeight.bold)),
          const Text(
            'Отметьте 2–4 параметра и нажмите «Создать ячейку» — они встанут '
            'в одну плитку компактным списком.',
            style: TextStyle(fontSize: 11, color: _muted),
          ),
          const SizedBox(height: 6),
          Wrap(
            spacing: 6,
            runSpacing: 6,
            children: [
              for (final id in [...activeIds, ...kVirtualTiles.map((t) => t.id)])
                FilterChip(
                  label: Text(_label(id), style: const TextStyle(fontSize: 11)),
                  selected: _comboPick.contains(id),
                  onSelected: (sel) => setState(() {
                    if (sel) {
                      if (_comboPick.length < 4) _comboPick.add(id);
                    } else {
                      _comboPick.remove(id);
                    }
                  }),
                ),
            ],
          ),
          const SizedBox(height: 8),
          FilledButton.tonalIcon(
            onPressed: _comboPick.length < 2
                ? null
                : () => setState(() {
                      final id = _comboPick.join(kComboSeparator);
                      prefs = prefs.copyWith(tiles: [...prefs.tiles, id]);
                      _comboPick.clear();
                    }),
            icon: const Icon(Icons.view_agenda, size: 18),
            label: Text('Создать ячейку (${_comboPick.length}/4)'),
          ),

          if (availableVirtual.isNotEmpty) ...[
            const Divider(height: 24),
            const Text('Расчётные плитки',
                style: TextStyle(fontWeight: FontWeight.bold)),
            Wrap(
              spacing: 6,
              runSpacing: 6,
              children: [
                for (final id in availableVirtual)
                  ActionChip(
                    avatar: const Icon(Icons.functions, size: 14),
                    label: Text(_label(id), style: const TextStyle(fontSize: 11)),
                    onPressed: () => setState(() =>
                        prefs = prefs.copyWith(tiles: [...prefs.tiles, id])),
                  ),
              ],
            ),
          ],
          if (availablePids.isNotEmpty) ...[
            const Divider(height: 24),
            Text('Активные PID (${availablePids.length})',
                style: const TextStyle(fontWeight: FontWeight.bold)),
            Wrap(
              spacing: 6,
              runSpacing: 6,
              children: [
                for (final id in availablePids)
                  ActionChip(
                    avatar: const Icon(Icons.add, size: 14),
                    label: Text(id, style: const TextStyle(fontSize: 11)),
                    onPressed: () => setState(() =>
                        prefs = prefs.copyWith(tiles: [...prefs.tiles, id])),
                  ),
              ],
            ),
          ],
          const SizedBox(height: 18),
          Row(
            children: [
              TextButton(
                onPressed: () => setState(() => prefs = DashboardPrefs()),
                child: const Text('Сбросить'),
              ),
              const Spacer(),
              FilledButton(
                onPressed: () => Navigator.pop(context, prefs),
                child: const Text('Применить'),
              ),
            ],
          ),
        ],
      ),
    );
  }
}

class _Stepper extends StatelessWidget {
  const _Stepper({
    required this.label,
    required this.value,
    required this.min,
    required this.max,
    required this.onChanged,
  });
  final String label;
  final int value, min, max;
  final ValueChanged<int> onChanged;

  @override
  Widget build(BuildContext context) => Column(
        crossAxisAlignment: CrossAxisAlignment.start,
        children: [
          Text(label,
              maxLines: 1,
              overflow: TextOverflow.ellipsis,
              style: const TextStyle(fontSize: 10.5, color: _muted)),
          Row(
            mainAxisSize: MainAxisSize.min,
            children: [
              IconButton(
                visualDensity: VisualDensity.compact,
                padding: EdgeInsets.zero,
                constraints: const BoxConstraints(minWidth: 32, minHeight: 32),
                onPressed: value > min ? () => onChanged(value - 1) : null,
                icon: const Icon(Icons.remove_circle_outline, size: 20),
              ),
              Text('$value', style: const TextStyle(fontSize: 15)),
              IconButton(
                visualDensity: VisualDensity.compact,
                padding: EdgeInsets.zero,
                constraints: const BoxConstraints(minWidth: 32, minHeight: 32),
                onPressed: value < max ? () => onChanged(value + 1) : null,
                icon: const Icon(Icons.add_circle_outline, size: 20),
              ),
            ],
          ),
        ],
      );
}
'''

# ===== Сервисная страница: статус связи, «НЕТ ОТВЕТА», пинг, журнал =====
FILES["lib/dtc_service_page.dart"] = r'''
import 'package:flutter/material.dart';

import 'diag.dart';
import 'diag_trace.dart';
import 'model.dart';
import 'trace_sheet.dart';

const _muted = Color(0xFF8CA0BF);

class DtcServicePage extends StatefulWidget {
  const DtcServicePage({super.key, required this.model});
  final AppModel model;
  @override
  State<DtcServicePage> createState() => _DtcServicePageState();
}

class _DtcServicePageState extends State<DtcServicePage> {
  final Map<String, DtcReadResult> _results = {};
  ReadinessReport? _readiness;
  String _log = '';
  bool _ok = true;
  bool _busy = false;

  AppModel get model => widget.model;
  DiagSession get session => model.diagSession;

  /// v0.14: честный запуск — ошибки НЕ проглатываются (perform() их глотал).
  Future<void> _run(String label, Future<void> Function() op) async {
    if (_busy) return;
    if (!model.elm.ready) {
      setState(() {
        _ok = false;
        _log = 'Адаптер не подключён — команда не отправлена. '
            'Вкладка «Адаптер» → подключитесь.';
      });
      return;
    }
    setState(() {
      _busy = true;
      _ok = true;
      _log = '$label…';
    });
    final wasPolling = model.engine.running;
    try {
      await model.engine.stop();
      await op();
      if (mounted) setState(() => _log = '$label — выполнено');
    } catch (e) {
      if (mounted) {
        setState(() {
          _ok = false;
          _log = '$label — НЕ выполнено: $e';
        });
      }
    } finally {
      await session.restore();
      if (wasPolling) await model.engine.start();
      if (mounted) setState(() => _busy = false);
    }
  }

  Future<bool> _confirm(String title, String body, {bool danger = false}) async =>
      await showDialog<bool>(
        context: context,
        builder: (ctx) => AlertDialog(
          title: Text(title),
          content: Text(body),
          actions: [
            TextButton(
                onPressed: () => Navigator.pop(ctx, false),
                child: const Text('ОТМЕНА')),
            danger
                ? FilledButton(
                    style: FilledButton.styleFrom(backgroundColor: Colors.red),
                    onPressed: () => Navigator.pop(ctx, true),
                    child: const Text('ВЫПОЛНИТЬ'))
                : FilledButton.tonal(
                    onPressed: () => Navigator.pop(ctx, true),
                    child: const Text('ВЫПОЛНИТЬ')),
          ],
        ),
      ) ??
      false;

  @override
  Widget build(BuildContext context) {
    final theme = Theme.of(context);
    final linked = model.elm.ready;

    return ListView(
      padding: const EdgeInsets.all(12),
      children: [
        // ---- статус связи ----
        Card(
          color: linked
              ? Colors.green.withValues(alpha: 0.08)
              : Colors.red.withValues(alpha: 0.10),
          child: ListTile(
            dense: true,
            leading: Icon(linked ? Icons.link : Icons.link_off,
                color: linked ? Colors.greenAccent : Colors.redAccent),
            title: Text(
              linked ? 'Адаптер подключён' : 'Адаптер НЕ подключён',
              style: const TextStyle(fontSize: 14, fontWeight: FontWeight.bold),
            ),
            subtitle: Text(
              linked
                  ? 'Опрос PID приостанавливается на время сервисных операций '
                      'и возобновляется автоматически.'
                  : 'Команды не отправляются. Подключитесь во вкладке «Адаптер».',
              style: const TextStyle(fontSize: 11),
            ),
            trailing: IconButton(
              tooltip: 'Журнал обмена',
              onPressed: () => TraceSheet.show(context, session.trace),
              icon: const Icon(Icons.terminal),
            ),
          ),
        ),
        if (_busy) const LinearProgressIndicator(),
        if (_log.isNotEmpty)
          Padding(
            padding: const EdgeInsets.symmetric(vertical: 6, horizontal: 4),
            child: Row(
              children: [
                Icon(_ok ? Icons.info_outline : Icons.error_outline,
                    size: 14, color: _ok ? _muted : Colors.redAccent),
                const SizedBox(width: 6),
                Expanded(
                  child: Text(_log,
                      style: TextStyle(
                          fontSize: 12,
                          color: _ok ? _muted : Colors.redAccent)),
                ),
              ],
            ),
          ),

        Wrap(
          spacing: 8,
          runSpacing: 8,
          children: [
            FilledButton.tonalIcon(
              onPressed: _busy ? null : _scanAll,
              icon: const Icon(Icons.radar, size: 18),
              label: const Text('Скан всех блоков'),
            ),
            OutlinedButton.icon(
              onPressed: _busy ? null : _ping,
              icon: const Icon(Icons.wifi_tethering, size: 18),
              label: const Text('Пинг адаптера'),
            ),
            OutlinedButton.icon(
              onPressed: _busy ? null : _readInfo,
              icon: const Icon(Icons.badge, size: 18),
              label: const Text('VIN / CALID'),
            ),
            OutlinedButton.icon(
              onPressed: _busy ? null : _readReadiness,
              icon: const Icon(Icons.checklist, size: 18),
              label: const Text('Мониторы'),
            ),
            OutlinedButton.icon(
              onPressed: () => TraceSheet.show(context, session.trace),
              icon: const Icon(Icons.terminal, size: 18),
              label: Text('Журнал (${session.trace.entries.length})'),
            ),
          ],
        ),
        const SizedBox(height: 8),
        if (_readiness != null) _readinessCard(_readiness!),

        for (final slot in EcuSlot.all) _slotCard(slot, theme),

        const Divider(height: 28),
        Text('Сервисные процедуры', style: theme.textTheme.titleMedium),
        const ListTile(
          dense: true,
          leading: Icon(Icons.restart_alt),
          title: Text('Сброс адаптаций (SSM2-эра)'),
          subtitle: Text('Клемма АКБ 10–30 мин, затем дообучение 15–20 мин.'),
        ),
        const ListTile(
          dense: true,
          leading: Icon(Icons.speed),
          title: Text('Обучение дросселя (E-Gas)'),
          subtitle: Text('Зажигание ON 15–20 с без запуска → OFF → прогрев.'),
        ),

        const Divider(height: 28),
        Card(
          color: model.experimental ? Colors.red.withValues(alpha: 0.07) : null,
          child: Column(
            children: [
              SwitchListTile(
                secondary: Icon(Icons.science,
                    color: model.experimental ? Colors.redAccent : null),
                title: const Text('Экспертные операции'),
                subtitle: Text(
                  model.experimental
                      ? 'РАЗБЛОКИРОВАНО: ECUReset и Clear Memory активны.'
                      : 'ECUReset (11 01) и Clear Memory (04 xx) заблокированы.',
                  style: const TextStyle(fontSize: 11),
                ),
                value: model.experimental,
                onChanged: _busy
                    ? null
                    : (v) async {
                        if (v &&
                            !await _confirm(
                              'Включить экспертные операции?',
                              'ECUReset ГЛУШИТ двигатель. Clear Memory стирает '
                                  'адаптации — потребуется дообучение.\n\n'
                                  'Только на стоянке при заглушённом моторе.',
                              danger: true,
                            )) {
                          return;
                        }
                        await model.setExperimental(v);
                        if (mounted) setState(() {});
                      },
              ),
              if (model.experimental) ...[
                const Divider(height: 1),
                ListTile(
                  dense: true,
                  leading: const Icon(Icons.power_settings_new,
                      color: Colors.redAccent),
                  title: const Text('ECUReset (11 01)'),
                  subtitle: const Text('Перезапуск ЭБУ, мотор заглохнет',
                      style: TextStyle(fontSize: 11)),
                  onTap: _busy ? null : _ecuReset,
                ),
                ListTile(
                  dense: true,
                  leading:
                      const Icon(Icons.auto_delete, color: Colors.orangeAccent),
                  title: const Text('Clear Memory (04 xx)'),
                  subtitle: const Text('Сброс адаптаций',
                      style: TextStyle(fontSize: 11)),
                  onTap: _busy ? null : _clearMemoryDialog,
                ),
                ListTile(
                  dense: true,
                  leading: const Icon(Icons.travel_explore),
                  title: const Text('Проба групп Clear Memory'),
                  subtitle: const Text('Безопасно: смотрит ответы 1..7',
                      style: TextStyle(fontSize: 11)),
                  onTap: _busy ? null : _probeGroups,
                ),
              ],
            ],
          ),
        ),
        const SizedBox(height: 24),
      ],
    );
  }

  Widget _slotCard(EcuSlot slot, ThemeData theme) {
    final r = _results[slot.name];
    final icon = r == null
        ? Icons.memory
        : !r.answered
            ? Icons.cloud_off
            : r.codes.isEmpty
                ? Icons.check_circle
                : Icons.warning_amber;
    final color = r == null
        ? null
        : !r.answered
            ? Colors.redAccent
            : r.codes.isEmpty
                ? Colors.greenAccent
                : Colors.amberAccent;
    return Card(
      child: ExpansionTile(
        leading: Icon(icon, color: color),
        title: Text(slot.name),
        subtitle: Text(
          r == null
              ? '${slot.header}→${slot.responseId} · не опрошен'
              : !r.answered
                  ? 'НЕТ ОТВЕТА — ${r.note}'
                  : 'ответил ${r.protocolUsed} · кодов: ${r.codes.length}',
          style: TextStyle(
              fontSize: 11,
              color: r != null && !r.answered ? Colors.redAccent : null),
        ),
        children: [
          if (r != null && r.answered && r.codes.isEmpty)
            const ListTile(
              dense: true,
              leading: Icon(Icons.check, color: Colors.greenAccent),
              title: Text('Ошибок нет'),
              subtitle: Text('ЭБУ ответил корректным кадром',
                  style: TextStyle(fontSize: 11)),
            ),
          for (final d in (r?.codes ?? const <DiagDtcLike>[]).cast<DiagDtc>())
            ListTile(
              dense: true,
              leading: Icon(
                d.origin == 'permanent'
                    ? Icons.lock
                    : (d.active ? Icons.error : Icons.history),
                size: 20,
                color: d.origin == 'permanent'
                    ? Colors.orangeAccent
                    : (d.active
                        ? theme.colorScheme.error
                        : theme.colorScheme.outline),
              ),
              title: Text(
                '${d.code}${d.titleRu.isEmpty ? '' : ' · ${d.titleRu}'}',
                style: const TextStyle(fontFamily: 'monospace', fontSize: 13),
              ),
              subtitle:
                  Text(d.statusText, style: const TextStyle(fontSize: 11)),
              trailing: d.milOn
                  ? const Icon(Icons.lightbulb, color: Colors.amber, size: 18)
                  : null,
            ),
          OverflowBar(
            alignment: MainAxisAlignment.start,
            children: [
              TextButton.icon(
                onPressed: _busy ? null : () => _readAll(slot),
                icon: const Icon(Icons.search, size: 18),
                label: const Text('ЧИТАТЬ'),
              ),
              TextButton(
                onPressed: _busy ? null : () => _readFreeze(slot),
                child: const Text('СТОП-КАДР'),
              ),
              if ((r?.codes ?? const []).isNotEmpty)
                FilledButton.tonalIcon(
                  onPressed: _busy ? null : () => _clear(slot),
                  icon: const Icon(Icons.cleaning_services, size: 18),
                  label: const Text('СБРОС'),
                ),
            ],
          ),
        ],
      ),
    );
  }

  Widget _readinessCard(ReadinessReport r) => Card(
        child: Padding(
          padding: const EdgeInsets.all(12),
          child: Column(
            crossAxisAlignment: CrossAxisAlignment.start,
            children: [
              Row(
                children: [
                  Icon(r.milOn ? Icons.lightbulb : Icons.lightbulb_outline,
                      color: r.milOn ? Colors.amber : _muted, size: 18),
                  const SizedBox(width: 8),
                  Expanded(
                    child: Text(
                      'Мониторы: ${r.ready}/${r.total}'
                      '${r.milOn ? ' · CHECK ГОРИТ' : ''} · DTC: ${r.dtcCount}',
                      style: const TextStyle(fontWeight: FontWeight.bold),
                    ),
                  ),
                ],
              ),
              const SizedBox(height: 6),
              Wrap(
                spacing: 6,
                runSpacing: 6,
                children: [
                  for (final e in r.monitors.entries)
                    Chip(
                      visualDensity: VisualDensity.compact,
                      avatar: Icon(
                          e.value ? Icons.check_circle : Icons.pending,
                          size: 16,
                          color:
                              e.value ? Colors.greenAccent : Colors.orangeAccent),
                      label:
                          Text(e.key, style: const TextStyle(fontSize: 11)),
                    ),
                ],
              ),
            ],
          ),
        ),
      );

  Future<void> _ping() => _run('Пинг адаптера', () async {
        final v = await session.pingAdapter();
        if (mounted) setState(() => _log = 'Адаптер ответил: $v');
      });

  Future<void> _readAll(EcuSlot slot) => _run('Чтение ${slot.name}', () async {
        final r = await session.readAllDtc(slot);
        if (mounted) setState(() => _results[slot.name] = r);
        if (!r.answered) throw ReplyErrorText(r.note);
      });

  Future<void> _clear(EcuSlot slot) async {
    final has = _results[slot.name]?.codes ?? const [];
    final perm =
        has.cast<DiagDtc>().where((d) => d.origin == 'permanent').length;
    if (!await _confirm(
      'Сбросить ошибки: ${slot.name}?',
      'Будут стёрты коды, стоп-кадры и мониторы.\n'
      '${perm > 0 ? '\nВНИМАНИЕ: $perm permanent-кодов НЕ стираются.\n' : ''}'
      '\nАдаптации (IAM, коррекции) это не трогает.',
    )) {
      return;
    }
    await _run('Сброс ${slot.name}', () async {
      final r = await session.clearDtc(slot);
      if (mounted) setState(() => _results[slot.name] = r);
    });
  }

  Future<void> _readFreeze(EcuSlot slot) =>
      _run('Стоп-кадр ${slot.name}', () async {
        final d = await session.readFreezeFrameDtc(slot);
        if (mounted) {
          setState(() => _log = d == null
              ? '${slot.name}: стоп-кадра нет'
              : '${slot.name}: стоп-кадр $d');
        }
      });

  Future<void> _scanAll() => _run('Скан всех блоков', () async {
        final map = await session.scanAllEcus(
          onProgress: (n) {
            if (mounted) setState(() => _log = 'Скан: $n…');
          },
        );
        if (mounted) {
          setState(() {
            _results
              ..clear()
              ..addAll(map);
            final answered = map.values.where((r) => r.answered).length;
            _log = 'Ответили $answered из ${map.length} блоков';
            _ok = answered > 0;
          });
        }
      });

  Future<void> _readInfo() => _run('VIN / CALID', () async {
        final vin = await session.readVin(EcuSlot.engine);
        final cal = await session.readCalId(EcuSlot.engine);
        if (mounted) {
          setState(() => _log =
              'VIN: ${vin ?? 'нет ответа'} · CALID: ${cal ?? 'нет ответа'}');
        }
      });

  Future<void> _readReadiness() => _run('Мониторы готовности', () async {
        final r = await session.readReadiness(EcuSlot.engine);
        if (mounted) setState(() => _readiness = r);
        if (r == null) throw ReplyErrorText('ЭБУ не вернул статус мониторов');
      });

  Future<void> _ecuReset() async {
    if (!await _confirm(
      'ECUReset: перезапустить ЭБУ?',
      'Двигатель ЗАГЛОХНЕТ немедленно. Только на стоянке.',
      danger: true,
    )) {
      return;
    }
    await _run('ECUReset', () async {
      final res = await session.ecuReset(EcuSlot.engine);
      if (mounted) {
        setState(() {
          _log = '${res.title}: ${res.detail}';
          _ok = res.success;
        });
      }
    });
  }

  Future<void> _probeGroups() => _run('Проба групп', () async {
        final map = await session.probeClearMemoryGroups(EcuSlot.engine);
        if (!mounted) return;
        await showDialog<void>(
          context: context,
          builder: (ctx) => AlertDialog(
            title: const Text('Группы Clear Memory'),
            content: SingleChildScrollView(
              child: Text(
                map.isEmpty
                    ? 'Нет ответа'
                    : map.entries.map((e) => 'гр.${e.key}: ${e.value}').join('\n'),
                style: const TextStyle(fontFamily: 'monospace', fontSize: 12),
              ),
            ),
            actions: [
              TextButton(
                  onPressed: () => Navigator.pop(ctx),
                  child: const Text('ЗАКРЫТЬ')),
            ],
          ),
        );
      });

  Future<void> _clearMemoryDialog() async {
    var group = 1;
    final ok = await showDialog<bool>(
      context: context,
      builder: (ctx) => StatefulBuilder(
        builder: (ctx, set) => AlertDialog(
          title: const Text('Clear Memory'),
          content: Column(
            mainAxisSize: MainAxisSize.min,
            crossAxisAlignment: CrossAxisAlignment.start,
            children: [
              const Text(
                'Группы зависят от ECU: 1 — всё обучение, 2 — топливные '
                'коррекции, 3 — детонация/IAM. После сброса нужно дообучение.',
                style: TextStyle(fontSize: 12),
              ),
              const SizedBox(height: 12),
              DropdownButtonFormField<int>(
                initialValue: group,
                decoration: const InputDecoration(labelText: 'Группа'),
                items: [
                  for (var g = 1; g <= 7; g++)
                    DropdownMenuItem(value: g, child: Text('Группа $g')),
                ],
                onChanged: (v) => set(() => group = v ?? 1),
              ),
            ],
          ),
          actions: [
            TextButton(
                onPressed: () => Navigator.pop(ctx, false),
                child: const Text('ОТМЕНА')),
            FilledButton(
              style: FilledButton.styleFrom(backgroundColor: Colors.orange),
              onPressed: () => Navigator.pop(ctx, true),
              child: const Text('СБРОСИТЬ'),
            ),
          ],
        ),
      ),
    );
    if (ok != true) return;
    await _run('Clear Memory гр.$group', () async {
      final res = await session.clearMemory(EcuSlot.engine, group);
      if (mounted) {
        setState(() {
          _log = '${res.title}: ${res.detail}';
          _ok = res.success;
        });
      }
    });
  }
}

/// Локальная ошибка с текстом (чтобы не тянуть protocol.dart в UI).
class ReplyErrorText implements Exception {
  ReplyErrorText(this.message);
  final String message;
  @override
  String toString() => message;
}
'''

# ===== 18 тестов: журнал, answered, сетка, комбо, перестановка =====
FILES["test/trace_test.dart"] = r'''
import 'package:flutter_test/flutter_test.dart';
import 'package:subaru_ssm2/dashboard_prefs.dart';
import 'package:subaru_ssm2/diag_trace.dart';

void main() {
  group('DiagTrace', () {
    test('считает TX/RX и хранит порядок', () {
      final t = DiagTrace();
      t.tx('18 00 FF 00');
      t.rx('58 00', ms: 42);
      expect(t.txCount, 1);
      expect(t.rxCount, 1);
      expect(t.entries.first.dir, TraceDir.tx);
      expect(t.entries.last.ms, 42);
    });

    test('ошибки считаются отдельно', () {
      final t = DiagTrace();
      t.error('нет связи');
      expect(t.errorCount, 1);
      expect(t.entries.single.dir, TraceDir.error);
    });

    test('лимит буфера соблюдается', () {
      final t = DiagTrace(limit: 10);
      for (var i = 0; i < 50; i++) {
        t.tx('cmd $i');
      }
      expect(t.entries.length, 10);
      expect(t.entries.last.text, 'cmd 49');
    });

    test('clear обнуляет счётчики', () {
      final t = DiagTrace()
        ..tx('a')
        ..rx('b')
        ..error('c');
      t.clear();
      expect(t.entries, isEmpty);
      expect(t.txCount, 0);
      expect(t.errorCount, 0);
    });

    test('asText содержит стрелки направления', () {
      final t = DiagTrace()
        ..tx('ATI')
        ..rx('ELM327 v1.5');
      final text = t.asText();
      expect(text, contains('>>'));
      expect(text, contains('<<'));
      expect(text, contains('ELM327'));
    });
  });

  group('DtcReadResult', () {
    test('answered=false НЕ значит «чисто»', () {
      const r = DtcReadResult(
        codes: [],
        answered: false,
        protocolUsed: '—',
        rawReplies: [],
      );
      expect(r.clean, isFalse);
    });
    test('answered=true без кодов -> чисто', () {
      const r = DtcReadResult(
        codes: [],
        answered: true,
        protocolUsed: 'SSM2 18',
        rawReplies: [],
      );
      expect(r.clean, isTrue);
    });
  });

  group('Сетка дашборда', () {
    test('perScreen = колонки × строки', () {
      final p = DashboardPrefs(
        columnsPortrait: 2,
        rowsPortrait: 5,
        columnsLandscape: 5,
        rowsLandscape: 2,
      );
      expect(p.perScreen(false), 10);
      expect(p.perScreen(true), 10);
    });

    test('ограничения колонок/строк при загрузке', () {
      final p = DashboardPrefs.fromJson({
        'columnsPortrait': 99,
        'rowsPortrait': 0,
        'columnsLandscape': 99,
        'rowsLandscape': 99,
      });
      expect(p.columnsPortrait, lessThanOrEqualTo(5));
      expect(p.rowsPortrait, greaterThanOrEqualTo(1));
      expect(p.columnsLandscape, lessThanOrEqualTo(8));
      expect(p.rowsLandscape, lessThanOrEqualTo(6));
    });
  });

  group('Объединённые ячейки', () {
    test('распознаются и разбираются', () {
      expect(isComboTile('RPM|AFR'), isTrue);
      expect(isComboTile('RPM'), isFalse);
      expect(comboParts('RPM|AFR|ECT'), ['RPM', 'AFR', 'ECT']);
    });

    test('комбо валидно только из известных частей', () {
      expect(DashboardPrefs.knownTile('RPM|AFR'), isTrue);
      expect(DashboardPrefs.knownTile('RPM|НЕТ'), isFalse);
      expect(DashboardPrefs.knownTile('@FUEL_LPH|RPM'), isTrue);
    });

    test('комбо видно, если доступна хотя бы одна часть', () {
      final p = DashboardPrefs(tiles: ['RPM|FBKC']);
      expect(p.visibleTiles({'RPM'}), ['RPM|FBKC']);
      expect(p.visibleTiles(<String>{}), isEmpty);
    });

    test('мусорное комбо отбрасывается при загрузке', () {
      final p = DashboardPrefs.fromJson({
        'tiles': ['RPM|AFR', 'ZZZ|YYY']
      });
      expect(p.tiles, ['RPM|AFR']);
    });
  });

  group('Перестановка плиток', () {
    test('moveTile переносит на новую позицию', () {
      final p = DashboardPrefs(tiles: ['A1', 'B2', 'C3']);
      // используем реальные id, чтобы не зависеть от валидации
      final q = DashboardPrefs(tiles: ['RPM', 'AFR', 'ECT']).moveTile('ECT', 0);
      expect(q.tiles, ['ECT', 'RPM', 'AFR']);
      expect(p.tiles.length, 3);
    });

    test('swapTiles меняет местами', () {
      final q =
          DashboardPrefs(tiles: ['RPM', 'AFR', 'ECT']).swapTiles('RPM', 'ECT');
      expect(q.tiles, ['ECT', 'AFR', 'RPM']);
    });

    test('перенос несуществующей плитки безопасен', () {
      final p = DashboardPrefs(tiles: ['RPM']);
      expect(p.moveTile('NOPE', 0).tiles, ['RPM']);
    });
  });
}
'''

# ===== Правки: общий DiagTrace в модели, импорты, версия =====
# ===== REWRITES =====

# --- 1. model.dart: общий DiagTrace + передача в сессию ---
REWRITES.append((
    "lib/model.dart",
    "  bool experimental = false;\n  late final DiagSession diagSession = DiagSession(elm);",
    """  bool experimental = false;

  /// v0.14: журнал обмена — видно каждую команду и ответ ЭБУ.
  final DiagTrace diagTrace = DiagTrace();
  late final DiagSession diagSession = DiagSession(elm, trace: diagTrace);""",
    ["final DiagTrace diagTrace", False],
))
REWRITES.append((
    "lib/model.dart",
    "import 'dashboard_prefs.dart';\nimport 'diag.dart';",
    "import 'dashboard_prefs.dart';\nimport 'diag.dart';\nimport 'diag_trace.dart';",
    ["import 'diag_trace.dart';", False],
))

# --- 2. main.dart: кнопка журнала в AppBar ---
REWRITES.append((
    "lib/main.dart",
    "import 'dashboard_page.dart';\nimport 'dashboard_prefs.dart';",
    "import 'dashboard_page.dart';\nimport 'dashboard_prefs.dart';\nimport 'trace_sheet.dart';",
    ["import 'trace_sheet.dart';", False],
))
REWRITES.append((
    "lib/main.dart",
    "SSM2 TELEMETRY 0.13",
    "SSM2 TELEMETRY 0.14",
    ["SSM2 TELEMETRY 0.14", True],
))

# --- 3. protocol.dart: ATI уже разрешён, добавляем ATRV для пинга ---
REWRITES.append((
    "lib/protocol.dart",
    "  '22 F1 8C', // UDS: серийный номер блока\n};",
    """  '22 F1 8C', // UDS: серийный номер блока
  '02 02 00', // стоп-кадр (повтор безопасен)
};""",
    ["// стоп-кадр (повтор безопасен)", True],
))

# ===== Запись, правки, самопроверка, чек-лист стенда =====
# ===== запись файлов =====
bad = [r for r in FILES if "(" in r or ")" in r]
if bad:
    raise RuntimeError(f"Невалидные FILES-ключи: {bad}")
BACKUP.mkdir(parents=True, exist_ok=True)
for rel, body in FILES.items():
    dest = APP / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        shutil.copyfile(dest, BACKUP / (rel.replace("/", "__") + ".bak"))
    dest.write_text(body.strip("\n") + "\n", encoding="utf-8")
    print(f"[OK] {rel}")
print(f"[OK] файлов: {len(FILES)}")

# ===== точечные правки =====
patched, skipped = 0, 0
for rel, needle, replacement, sig in REWRITES:
    optional = False
    if sig and sig[-1] is True:
        sig, optional = sig[:-1], True
    elif sig and sig[-1] is False:
        sig, optional = sig[:-1], False
    path = APP / rel
    if not path.exists():
        raise RuntimeError(f"Нет файла {rel} — прогоните 02–04.4")
    text = path.read_text(encoding="utf-8")
    if sig and all(s in text for s in sig):
        skipped += 1
        print(f"[SKIP уже применено] {rel}: {sig[0][:50]}")
        continue
    if needle in text:
        shutil.copyfile(path, BACKUP / (rel.replace("/", "__") + ".orig.bak"))
        path.write_text(text.replace(needle, replacement, 1), encoding="utf-8")
        patched += 1
        print(f"[OK] правка {rel}")
        continue
    if optional:
        skipped += 1
        print(f"[SKIP необязательная] {rel}")
        continue
    raise RuntimeError(f"Маркер не найден в {rel}:\n{needle[:100]!r}")
print(f"[OK] правок: {patched}, пропущено: {skipped}")

# ===== самопроверка =====
checks = {
    "lib/diag_trace.dart": ["class DiagTrace", "NotConnectedException",
                            "class DtcReadResult", "answered"],
    "lib/diag.dart": ["readDtcDetailed", "_requireLink", "trace.tx",
                      "pingAdapter", "DiagSession(this.elm, {DiagTrace? trace})"],
    "lib/trace_sheet.dart": ["class TraceSheet", "Журнал обмена"],
    "lib/trip_card.dart": ["class TripCard", "Одометр", "Средний"],
    "lib/dashboard_prefs.dart": ["kComboSeparator", "isComboTile", "rowsPortrait",
                                 "perScreen", "moveTile"],
    "lib/dashboard_page.dart": ["LongPressDraggable", "DragTarget", "_ComboTile",
                                "TripCard"],
    "lib/dtc_service_page.dart": ["Адаптер НЕ подключён", "TraceSheet.show",
                                  "НЕТ ОТВЕТА", "pingAdapter"],
    "lib/model.dart": ["final DiagTrace diagTrace", "trace: diagTrace"],
    "test/trace_test.dart": ["DiagTrace", "perScreen", "isComboTile"],
}
for rel, needles in checks.items():
    text = (APP / rel).read_text(encoding="utf-8")
    miss = [m for m in needles if m not in text]
    if miss:
        raise RuntimeError(f"Самопроверка {rel}: нет маркеров {miss}")
print("[OK] самопроверка маркеров пройдена")

print("\n=== Готово: SSM2 0.14. Далее — ячейка 05 (сборка). ===")
print("Исправлено:")
print("  · ложное «готово» в DTC: проверка связи + честный DtcReadResult")
print("  · «НЕТ ОТВЕТА» больше не выглядит как «ошибок нет»")
print("Добавлено:")
print("  · журнал обмена TX/RX с таймингом (кнопка «Журнал»)")
print("  · карточка поездки: расход, одометр, средний, сброс")
print("  · сетка: ячеек в строке × строк (до 5x8 и 8x6), fit без прокрутки")
print("  · перетаскивание ячеек прямо на дашборде (долгое нажатие)")
print("  · объединение 2–4 PID в одну ячейку")
print("Чек-лист стенда 04.5:")
print(" [ ] БЕЗ адаптера: 'Читать' -> красный 'Адаптер не подключён', журнал пуст")
print(" [ ] С адаптером: 'Пинг' -> в журнале ATI >> и << ELM327 v1.x")
print(" [ ] Блок без ответа -> 'НЕТ ОТВЕТА', а не 'Ошибок нет'")
print(" [ ] Опрос PID сам встаёт на паузу и возобновляется после операции")
print(" [ ] Дашборд: 2x5 = 10 ячеек без прокрутки, перетаскивание работает")
print(" [ ] Объединённая ячейка показывает 3 параметра сразу")


[OK] lib/diag_trace.dart
[OK] lib/diag.dart
[OK] lib/trace_sheet.dart
[OK] lib/trip_card.dart
[OK] lib/dashboard_prefs.dart
[OK] lib/dashboard_page.dart
[OK] lib/dtc_service_page.dart
[OK] test/trace_test.dart
[OK] файлов: 8
[OK] правка lib/model.dart
[OK] правка lib/model.dart
[OK] правка lib/main.dart
[OK] правка lib/main.dart
[OK] правка lib/protocol.dart
[OK] правок: 5, пропущено: 0
[OK] самопроверка маркеров пройдена

=== Готово: SSM2 0.14. Далее — ячейка 05 (сборка). ===
Исправлено:
  · ложное «готово» в DTC: проверка связи + честный DtcReadResult
  · «НЕТ ОТВЕТА» больше не выглядит как «ошибок нет»
Добавлено:
  · журнал обмена TX/RX с таймингом (кнопка «Журнал»)
  · карточка поездки: расход, одометр, средний, сброс
  · сетка: ячеек в строке × строк (до 5x8 и 8x6), fit без прокрутки
  · перетаскивание ячеек прямо на дашборде (долгое нажатие)
  · объединение 2–4 PID в одну ячейку
Чек-лист стенда 04.5:
 [ ] БЕЗ адаптера: 'Читать' -> красный 'Адаптер не подключён', журнал пуст
 [ ] 

In [ ]:
# coding: utf-8
# @title 04.6 | PARSER FIX | SSM2 0.14 >> 0.14.1 (insert BETWEEN 04.5 and 05)
# ROOT CAUSE from log:
#   >> 03 (read DTC)
#   << 4300\r\r>   <- ECU replied "43 00" = no errors (CORRECT!)
#   -- "not recognized as data frame" <- parseDiagPayload returned null!
# Fix: ELM327 appends prompt ">" to every reply.
# Old parser: "4300\r\r>" -> ["4300", ">"] -> ">" fails hex regex -> null.
# Fixed: filter ">", ELM status lines, 3-char length counters, ATH1 CAN IDs.
import re
import shutil
import time
from pathlib import Path

APP = Path("/content/subaru_ssm2_fixed")
STAMP = time.strftime("%Y%m%d_%H%M%S")
BACKUP = APP.parent / f"ssm2_backup_046_{STAMP}"
BACKUP.mkdir(parents=True, exist_ok=True)

# ===== NEW parseDiagPayload (ASCII-only comments) =====
NEW_PARSE = r"""List<int>? parseDiagPayload(String text) {
  // v0.14.1 (parser fix): filter ELM prompt ">", multi-frame length counters,
  // ELM status lines, ATH1 CAN IDs. Bug: ">" broke entire parsing -> null.
  const Set<String> _elmStatus = <String>{
    'NODATA', 'STOPPED', 'CANERROR', 'BUSBUSY', 'BUFFERFULL',
    'ERROR', 'UNABLETOCONNECT', 'OK', '?',
  };

  final flat = <int>[];
  final frameLines = <int, List<int>>{};
  var sawFrameIndex = false;

  for (final rawLine in text.split(RegExp(r'[\r\n]+'))) {
    final line = rawLine.trim();
    if (line.isEmpty || line == '>') continue; // ELM prompt

    // ELM status lines (NO DATA, OK, ...)
    final compact = line.toUpperCase().replaceAll(RegExp(r'[^A-Z]'), '');
    if (_elmStatus.contains(compact)) continue;

    // Numbered ISO-TP frames "0:490402..."
    final frameMatch = RegExp(r'^(\d+):(.*)$').firstMatch(line);
    if (frameMatch != null) {
      sawFrameIndex = true;
      final idx = int.parse(frameMatch.group(1)!);
      final hex = frameMatch.group(2)!.replaceAll(' ', '');
      if (hex.isNotEmpty && hex.length % 2 == 0 &&
          RegExp(r'^[0-9A-Fa-f]+$').hasMatch(hex)) {
        frameLines[idx] = <int>[
          for (var i = 0; i + 1 < hex.length; i += 2)
            int.parse(hex.substring(i, i + 2), radix: 16),
        ];
      }
      continue;
    }

    // 3-char hex token = ELM length counter ("023") -> skip
    if (RegExp(r'^[0-9A-Fa-f]{3}$').hasMatch(line) && !line.contains(' ')) {
      continue;
    }

    var tokens = line.split(RegExp(r'\s+')).where((t) => t.isNotEmpty).toList();

    // ATH1: first token is 3-char CAN ID -> skip it
    if (tokens.isNotEmpty &&
        tokens.first.length == 3 &&
        RegExp(r'^[0-9A-Fa-f]{3}$').hasMatch(tokens.first)) {
      tokens = tokens.sublist(1);
    }

    for (final t in tokens) {
      if (t == '>') continue; // inline prompt, skip
      if (!RegExp(r'^[0-9A-Fa-f]+$').hasMatch(t) || t.length % 2 != 0) {
        return null; // genuinely invalid token
      }
      for (var i = 0; i + 1 < t.length; i += 2) {
        flat.add(int.parse(t.substring(i, i + 2), radix: 16));
      }
    }
  }

  // ISO-TP multi-frame: concatenate by index order
  if (sawFrameIndex) {
    if (frameLines.isEmpty) return null;
    final payload = <int>[];
    for (final k in (frameLines.keys.toList()..sort())) {
      payload.addAll(frameLines[k]!);
    }
    return payload.isEmpty ? null : payload;
  }

  if (flat.isEmpty) return null;
  var b = flat;
  // PCI single-frame: high nibble 0x0, length 1..7
  if (b.length > 1 && (b[0] & 0xF0) == 0 && (b[0] & 0x0F) <= 7) {
    final len = b[0] & 0x0F;
    b = b.sublist(1, (1 + len).clamp(1, b.length));
  }
  // CAN ID 0x7E8 with ATH1
  if (b.length >= 3 && b[0] == 0x07 && (b[1] & 0xF8) == 0xE8) {
    b = b.sublist(2);
  }
  return b.isEmpty ? null : b;
}"""

# ===== NEW parseAsciiReply (multi-frame CALID support) =====
NEW_ASCII = r"""String? parseAsciiReply(List<int> b, int pid) {
  // v0.14.1 (parser fix): multi-frame CALID support; 49 04 [count?] [ascii...]
  // Skip count byte if < 0x20 (not ASCII printable)
  if (b.length < 3 || b[0] != 0x49 || b[1] != pid) return null;
  // If b[2] < 0x20 it is a frame count, not data -> start at 3
  final start = (b.length > 3 && b[2] < 0x20) ? 3 : 2;
  final chars = <int>[];
  for (var i = start; i < b.length; i++) {
    if (b[i] >= 0x20 && b[i] <= 0x7E) chars.add(b[i]);
  }
  final t = String.fromCharCodes(chars).trim();
  return t.isEmpty ? null : t;
}"""

# ===== Apply patches =====
path = APP / "lib/diag.dart"
if not path.exists():
    raise RuntimeError("lib/diag.dart not found - run 02-04.5 first")

text = path.read_text(encoding="utf-8")

# Check already applied
if "v0.14.1" in text and "ELM prompt" in text:
    print("[SKIP already applied] lib/diag.dart: parseDiagPayload")
else:
    # Find and replace parseDiagPayload by extracting full function
    m = re.search(r"List<int>\? parseDiagPayload\(String text\) \{", text)
    if not m:
        raise RuntimeError("parseDiagPayload not found in diag.dart")
    start = m.start()
    depth = 0
    end = start
    for i, c in enumerate(text[start:]):
        if c == '{':
            depth += 1
        elif c == '}':
            depth -= 1
            if depth == 0:
                end = start + i + 1
                break
    old_fn = text[start:end]
    shutil.copyfile(path, BACKUP / "lib__diag.dart.orig.bak")
    text = text.replace(old_fn, NEW_PARSE, 1)
    path.write_text(text, encoding="utf-8")
    print("[OK] patched lib/diag.dart: parseDiagPayload")

text = path.read_text(encoding="utf-8")
if "multi-frame" in text:
    print("[SKIP already applied] lib/diag.dart: parseAsciiReply")
else:
    m = re.search(r"String\? parseAsciiReply\(List<int> b, int pid\) \{", text)
    if not m:
        raise RuntimeError("parseAsciiReply not found in diag.dart")
    start = m.start()
    depth = 0
    end = start
    for i, c in enumerate(text[start:]):
        if c == '{':
            depth += 1
        elif c == '}':
            depth -= 1
            if depth == 0:
                end = start + i + 1
                break
    old_fn = text[start:end]
    text = text.replace(old_fn, NEW_ASCII, 1)
    path.write_text(text, encoding="utf-8")
    print("[OK] patched lib/diag.dart: parseAsciiReply")

# ===== Self-check =====
diag = path.read_text(encoding="utf-8")
assert "v0.14.1" in diag, "parseDiagPayload not patched"
assert "multi-frame" in diag, "parseAsciiReply not patched"
print("[OK] self-check passed")

print()
print("=== SSM2 0.14.1 - run cell 05 next ===")
print()
print("FIXED: parseDiagPayload now handles ELM prompt '>'")
print('  Before: "4300\\r\\r>" -> None (parser crash on ">")')
print('  After:  "4300\\r\\r>" -> [0x43, 0x00] -> "no DTC" (correct!)')
print()
print("YOUR LOG DIAGNOSTICS:")
print("  CALID confirmed: A2TB100B = matches identity.dart (good)")
print("  IAM=1.000, FBKC=0.00, FKL=0.00 = NORMAL on idle without knock")
print("  REQ_TQ=0.0 = zero torque request on idle (correct)")
print("  LTFT=-14.84% = WARNING: large negative fuel learning trim!")
print("    Possible causes: vacuum leak after MAF, dripping injector,")
print("    rich base fuel map. Check if STFT varies or is stuck at 0.")
print("  IAT: add to PID set (PID tab) and dashboard grid")


[OK] patched lib/diag.dart: parseDiagPayload
[SKIP already applied] lib/diag.dart: parseAsciiReply
[OK] self-check passed

=== SSM2 0.14.1 - run cell 05 next ===

FIXED: parseDiagPayload now handles ELM prompt '>'
  Before: "4300\r\r>" -> None (parser crash on ">")
  After:  "4300\r\r>" -> [0x43, 0x00] -> "no DTC" (correct!)

YOUR LOG DIAGNOSTICS:
  CALID confirmed: A2TB100B = matches identity.dart (good)
  IAM=1.000, FBKC=0.00, FKL=0.00 = NORMAL on idle without knock
  REQ_TQ=0.0 = zero torque request on idle (correct)
  LTFT=-14.84% = WARNING: large negative fuel learning trim!
    Possible causes: vacuum leak after MAF, dripping injector,
    rich base fuel map. Check if STFT varies or is stuck at 0.
  IAT: add to PID set (PID tab) and dashboard grid


In [ ]:
# @title 04.7 FIX DOUBLE
from pathlib import Path
import re
import shutil
import time

p = Path("/content/subaru_ssm2_fixed/lib/protocol.dart")
text = p.read_text(encoding="utf-8")

pattern = (
    r"(?m)^[ \t]*'02 02 00'[ \t]*,[ \t]*//[ \t]*"
    r"стоп-кадр \(повтор безопасен\)[ \t]*(?:\r?\n|$)"
)
m = re.search(pattern, text)

if m is None:
    print("[SKIP] Добавленная строка отсутствует; файл не изменён.")
else:
    start = text.rfind("{", 0, m.start())
    end = text.find("}", m.end())
    if start < 0 or end < 0:
        raise RuntimeError("Не удалось определить границы набора; файл не изменён.")

    block = text[start:end]
    count = len(re.findall(
        r"(?m)^[ \t]*'02 02 00'[ \t]*,", block
    ))
    if count != 2:
        raise RuntimeError(
            f"Ожидались две записи '02 02 00', найдено {count}. "
            "Предположение о дубле не подтверждено; файл не изменён."
        )

    backup = Path(
        f"/content/protocol_before_fix_{time.strftime('%Y%m%d_%H%M%S')}.dart.bak"
    )
    shutil.copy2(p, backup)
    p.write_text(text[:m.start()] + text[m.end():], encoding="utf-8")

    print("[OK] Удалена повторная строка из 04.5; первая запись сохранена.")
    print("Бэкап:", backup)

[OK] Удалена повторная строка из 04.5; первая запись сохранена.
Бэкап: /content/protocol_before_fix_20261006_081138.dart.bak


In [ ]:
# coding: utf-8
# @title 04.8 | OP2 OTG FIX | SSM2 0.14.1 >> 0.15.0 (insert BETWEEN 04.7 and 05)
# MOTIVATION:
# Bluetooth ELM327 покрывает только CAN-машины. Tactrix OpenPort 2.0 по USB
# OTG (FTDI, VID 0x0403 / PID 0xCC4C) даёт тот же SSM-каскад проводом и
# открывает K-line для pre-CAN Subaru. BT-сборка по умолчанию НЕ меняется.
#
# Что делает ячейка:
#   [+] android/.../Op2Channel.kt        — нативный USB-host канал (FTDI bulk),
#                                          БЕЗ pub-плагинов и без JitPack
#   [+] res/xml/op2_device_filter.xml    — VID/PID OP2 и клонов
#   [+] patch AndroidManifest.xml        — uses-feature usb.host + attach-intent
#   [+] patch MainActivity.kt            — регистрация Op2Channel (якорь-вставка)
#   [+] lib/native_op2.dart              — Dart-обёртка, контракт как у SPP
#   [+] patch lib/transport_selected.dart— ветка native_op2 по --dart-define
#   [+] test/op2_transport_test.dart     — чистые тесты ID-таблицы
#   [+] tool/op2_j2534_notes.md          — стендовый чеклист J2534
import re
import shutil
import subprocess
import time
from pathlib import Path

APP = Path("/content/subaru_ssm2_fixed")
STAMP = time.strftime("%Y%m%d_%H%M%S")
BACKUP = APP.parent / f"ssm2_backup_048_{STAMP}"

# --- Параметры модуля ---
OP2_BAUD = 460800            # @param {type:"integer"}  # [HARDWARE-TUNE] стартовое
OP2_LATENCY_MS = 2           # @param {type:"integer"}  # FTDI latency timer
PATCH_TRANSPORT = True       # @param {type:"boolean"}  # ветка в transport_selected

if not (APP / "build_config.json").exists():
    raise RuntimeError("Не найден проект. Выполните ячейки 01 и 02.")
MAIN_ACTIVITY = APP / "android/app/src/main/kotlin/com/subaru/ssm2_fixed/MainActivity.kt"
MANIFEST = APP / "android/app/src/main/AndroidManifest.xml"
TRANSPORT_SEL = APP / "lib/transport_selected.dart"
for need in (MAIN_ACTIVITY, MANIFEST, TRANSPORT_SEL, APP / "lib/native_spp.dart"):
    if not need.exists():
        raise RuntimeError(f"Нет файла: {need}. Сначала прогоните 02 и фиксы 04.1-04.7.")
BACKUP.mkdir(parents=True, exist_ok=True)

# ============================================================
# 1. Kotlin: нативный FTDI-канал OP2 (UsbManager, без библиотек)
# ============================================================
OP2_CHANNEL_KT = r'''package com.subaru.ssm2_fixed

// v0.15.0 (OP2 OTG): нативный USB-host транспорт для Tactrix OpenPort 2.0.
// FTDI-клон чипа: VID 0x0403, PID 0xCC4C (оригинал), H-series контроллер.
// Канал "ssm2/op2" (методы) + "ssm2/op2_rx" (поток байтов).
// J2534-PAYLOAD-MAP: маппинг SSM>A8 00.. на PassThru-кадры подтверждается
// стендом по tool/op2_j2534_notes.md ДО снятия HARDWARE-гейта.

import android.app.PendingIntent
import android.content.BroadcastReceiver
import android.content.Context
import android.content.Intent
import android.content.IntentFilter
import android.hardware.usb.UsbConstants
import android.hardware.usb.UsbDevice
import android.hardware.usb.UsbDeviceConnection
import android.hardware.usb.UsbEndpoint
import android.hardware.usb.UsbInterface
import android.hardware.usb.UsbManager
import android.os.Build
import io.flutter.embedding.engine.FlutterEngine
import io.flutter.plugin.common.EventChannel
import io.flutter.plugin.common.MethodChannel
import java.util.concurrent.Executors

object Op2Ids {
    const val VID_TACTRIX = 0x0403
    val KNOWN_PIDS = intArrayOf(0xCC4C, 0xCC48, 0x6001) // OP2, клон, FTDI-default

    fun isOp2(d: UsbDevice): Boolean =
        d.vendorId == VID_TACTRIX && KNOWN_PIDS.contains(d.productId)
}

/** Минимальный FTDI H-series драйвер: reset/baud/data/latency/DTR/RTS + bulk IO. */
class FtdiPipe(
    private val conn: UsbDeviceConnection,
    private val iface: UsbInterface,
    private val epIn: UsbEndpoint,
    private val epOut: UsbEndpoint,
) {
    private fun ctrl(req: Int, value: Int, index: Int): Int =
        conn.controlTransfer(0x40, req, value, index, null, 0, 300)

    fun open(baud: Int, latencyMs: Int) {
        val idx = iface.id
        ctrl(0x00, 0x0000, idx)          // RESET
        ctrl(0x00, 0x0001, idx)          // purge RX
        ctrl(0x00, 0x0002, idx)          // purge TX
        setBaud(baud, idx)
        ctrl(0x04, 0x0008, idx)          // 8 data | no parity | 1 stop
        ctrl(0x09, latencyMs.coerceIn(1, 255), idx) // LATENCY_TIMER, мс
        ctrl(0x01, 0x0303, idx)          // DTR=1 RTS=1
        ctrl(0x02, 0x0000, idx)          // flow control off
    }

    private fun setBaud(baud: Int, ifIdx: Int) {
        // FTDI: divisor = 3 МГц / baud (48 МГц / 16). Доли - по таблице 1/8.
        val fracEnc = intArrayOf(0, 3, 2, 4, 1, 5, 6, 7)
        val raw = 3_000_000.0 / baud
        var div = raw.toInt()
        var frac = Math.round((raw - div) * 8).toInt()
        if (frac == 8) { div += 1; frac = 0 }
        val value = div and 0x3FFF
        val index = (fracEnc[frac] shl 8) or (ifIdx + 1) // H-серия: low byte = iface+1
        ctrl(0x03, value, index)
    }

    fun write(bytes: ByteArray, timeoutMs: Int): Int =
        conn.bulkTransfer(epOut, bytes, bytes.size, timeoutMs)

    fun read(buf: ByteArray, timeoutMs: Int): Int =
        conn.bulkTransfer(epIn, buf, buf.size, timeoutMs)

    fun close() = runCatching { conn.releaseInterface(iface); conn.close() }
}

object Op2Channel : MethodChannel.MethodCallHandler, EventChannel.StreamHandler {
    private const val TAG = "ssm2/op2"
    private lateinit var context: Context
    private lateinit var manager: UsbManager
    private var pipe: FtdiPipe? = null
    private var reader: Thread? = null
    @Volatile private var running = false
    @Volatile private var baud = 460800
    @Volatile private var latencyMs = 2
    private var sink: EventChannel.EventSink? = null
    private var permissionResult: MethodChannel.Result? = null

    private val attachReceiver = object : BroadcastReceiver() {
        override fun onReceive(ctx: Context, intent: Intent) {
            if (intent.action == "com.subaru.ssm2_fixed.OP2_PERMISSION") {
                val granted = intent.getBooleanExtra(UsbManager.EXTRA_PERMISSION_GRANTED, false)
                if (granted) openPipe()?.let { permissionResult?.success(it) }
                    ?: permissionResult?.error("op2", "open failed", null)
                else permissionResult?.success(mapOf("granted" to false))
                permissionResult = null
            }
        }
    }

    fun register(context: Context, engine: FlutterEngine) {
        this.context = context.applicationContext
        this.manager = context.getSystemService(Context.USB_SERVICE) as UsbManager
        val filter = IntentFilter("com.subaru.ssm2_fixed.OP2_PERMISSION")
        if (Build.VERSION.SDK_INT >= 33) {
            context.registerReceiver(attachReceiver, filter, Context.RECEIVER_NOT_EXPORTED)
        } else {
            @Suppress("UnspecifiedRegisterReceiverFlag")
            context.registerReceiver(attachReceiver, filter)
        }
        MethodChannel(engine.dartExecutor.binaryMessenger, TAG).setMethodCallHandler(this)
        EventChannel(engine.dartExecutor.binaryMessenger, "ssm2/op2_rx").setStreamHandler(this)
    }

    override fun onMethodCall(call: io.flutter.plugin.common.MethodCall, result: MethodChannel.Result) {
        when (call.method) {
            "op2/baud" -> {
                baud = (call.argument<Int>("baud") ?: 460800)
                latencyMs = (call.argument<Int>("latencyMs") ?: 2)
                result.success(true)
            }
            "op2/descriptor" -> result.success(descriptor())
            "op2/open" -> {
                baud = (call.argument<Int>("baud") ?: baud)
                latencyMs = (call.argument<Int>("latencyMs") ?: latencyMs)
                val dev = manager.deviceList.values.firstOrNull(Op2Ids::isOp2)
                if (dev == null) { result.success(mapOf("granted" to true, "found" to false)); return }
                if (manager.hasPermission(dev)) openPipe()?.let { result.success(it) }
                    ?: result.error("op2", "open failed", null)
                else {
                    permissionResult = result
                    val pi = PendingIntent.getBroadcast(
                        context, 0, Intent("com.subaru.ssm2_fixed.OP2_PERMISSION"),
                        PendingIntent.FLAG_UPDATE_CURRENT or PendingIntent.FLAG_IMMUTABLE)
                    manager.requestPermission(dev, pi)
                }
            }
            "op2/write" -> {
                val data = call.arguments as? ByteArray
                val n = data?.let { pipe?.write(it, 250) } ?: -1
                if (n >= 0) result.success(n) else result.error("op2", "write failed", null)
            }
            "op2/close" -> { stopReader(); result.success(true) }
            else -> result.notImplemented()
        }
    }

    private fun descriptor(): Map<String, Any?> {
        val dev = manager.deviceList.values.firstOrNull(Op2Ids::isOp2)
        return mapOf(
            "found" to (dev != null),
            "vid" to dev?.vendorId?.let { "0x%04X".format(it) },
            "pid" to dev?.productId?.let { "0x%04X".format(it) },
            "name" to dev?.deviceName,
            "product" to dev?.productName,
            "hasPermission" to (dev?.let(manager::hasPermission) ?: false),
        )
    }

    private fun openPipe(): Map<String, Any?>? {
        val dev = manager.deviceList.values.firstOrNull(Op2Ids::isOp2) ?: return null
        val conn = manager.openDevice(dev) ?: return null
        // Канал J2534 - интерфейс 0 (MI_00); [HARDWARE-TUNE] проверить на клоне
        val iface = dev.getInterface(0)
        if (!conn.claimInterface(iface, true)) { conn.close(); return null }
        var epIn: UsbEndpoint? = null
        var epOut: UsbEndpoint? = null
        for (i in 0 until iface.endpointCount) {
            val ep = iface.getEndpoint(i)
            if (ep.type != UsbConstants.USB_ENDPOINT_XFER_BULK) continue
            if (ep.direction == UsbConstants.USB_DIR_IN) epIn = ep else epOut = ep
        }
        if (epIn == null || epOut == null) { conn.close(); return null }
        val p = FtdiPipe(conn, iface, epIn, epOut)
        p.open(baud, latencyMs)
        pipe = p
        startReader(p)
        return mapOf(
            "granted" to true, "found" to true,
            "vid" to "0x%04X".format(dev.vendorId),
            "pid" to "0x%04X".format(dev.productId),
            "baud" to baud, "latencyMs" to latencyMs,
        )
    }

    private fun startReader(p: FtdiPipe) {
        running = true
        reader = Thread {
            val buf = ByteArray(4096)
            while (running) {
                val n = runCatching { p.read(buf, 100) }.getOrDefault(-1)
                if (n > 0) sink?.success(buf.copyOfRange(0, n))
            }
        }.also { it.isDaemon = true; it.start() }
    }

    private fun stopReader() {
        running = false
        reader?.join(300)
        reader = null
        pipe?.close()
        pipe = null
    }

    override fun onListen(arguments: Any?, events: EventChannel.EventSink?) { sink = events }
    override fun onCancel(arguments: Any?) { sink = null }
}
'''

# ============================================================
# 2. USB device filter
# ============================================================
DEVICE_FILTER_XML = '''<?xml version="1.0" encoding="utf-8"?>
<!-- v0.15.0 (OP2 OTG): Tactrix OpenPort 2.0 и FTDI-клоны -->
<resources>
    <usb-device vendor-id="1027" product-id="52300" />  <!-- 0x0403:0xCC4C оригинал -->
    <usb-device vendor-id="1027" product-id="52296" />  <!-- 0x0403:0xCC48 клон -->
    <usb-device vendor-id="1027" product-id="24577" />  <!-- 0x0403:0x6001 FTDI default -->
</resources>
'''

# ============================================================
# 3. Dart: обёртка + чистые хелперы (тестируются без канала)
# ============================================================
NATIVE_OP2_DART = r'''// v0.15.0 (OP2 OTG): Dart-сторона J2534-транспорта OpenPort 2.0.
// Контракт совпадает с native_spp.dart: open / write / bytes / close,
// поэтому elm.dart и engine.dart не замечают подмены среды.
import 'dart:async';
import 'dart:typed_data';

import 'package:flutter/services.dart';

/// Известные сигнатуры OpenPort 2.0 (FTDI).
const int kOp2Vid = 0x0403; // Tactrix/FTDI
const List<int> kOp2Pids = [0xCC4C, 0xCC48, 0x6001];

String hex16(int v) =>
    '0x' + v.toRadixString(16).toUpperCase().padLeft(4, '0');

/// Описание устройства для UI/диагностики.
Map<String, String> describeOp2(Map<String, Object?> d) {
  final out = <String, String>{'found': d['found'].toString()};
  for (final key in const ['vid', 'pid', 'product']) {
    final value = d[key];
    if (value != null) out[key] = value.toString();
  }
  return out;
}

/// true, если пара VID/PID похожа на OP2 (оригинал или FTDI-клон).
bool looksLikeOp2(int vid, int pid) => vid == kOp2Vid && kOp2Pids.contains(pid);

class NativeOp2Transport {
  static const MethodChannel _tx = MethodChannel('ssm2/op2');
  static const EventChannel _rx = EventChannel('ssm2/op2_rx');

  final int baud;
  final int latencyMs;
  NativeOp2Transport({this.baud = 460800, this.latencyMs = 2});

  /// Дескриптор подключённого устройства (found/vid/pid/product).
  Future<Map<String, Object?>> descriptor() async {
    final d = await _tx.invokeMethod<Map<dynamic, dynamic>>('op2/descriptor');
    return d?.cast<String, Object?>() ?? const {'found': false};
  }

  /// Открыть FTDI-канал: permission (системный диалог при первом OTG),
  /// claim interface 0, FTDI init (reset/baud/8N1/latency/DTR/RTS).
  Future<Map<String, Object?>> open() async {
    final r = await _tx.invokeMethod<Map<dynamic, dynamic>>(
      'op2/open', {'baud': baud, 'latencyMs': latencyMs});
    return r?.cast<String, Object?>() ?? const {'granted': false};
  }

  Stream<Uint8List> get bytes => _rx
      .receiveBroadcastStream()
      .where((e) => e is Uint8List)
      .cast<Uint8List>();

  Future<int> write(Uint8List frame) async =>
      (await _tx.invokeMethod<int>('op2/write', frame)) ?? -1;

  Future<void> close() => _tx.invokeMethod<void>('op2/close');
}
'''

OP2_TEST_DART = r'''// v0.15.0: чистые тесты OP2 без платформенного канала.
import 'package:flutter_test/flutter_test.dart';
import 'package:subaru_ssm2_fixed/native_op2.dart';

void main() {
  group('OP2 ID table', () {
    test('оригинал Tactrix', () {
      expect(looksLikeOp2(0x0403, 0xCC4C), isTrue);
    });
    test('клон и FTDI default', () {
      expect(looksLikeOp2(0x0403, 0xCC48), isTrue);
      expect(looksLikeOp2(0x0403, 0x6001), isTrue);
    });
    test('чужой VID/PID отбрасывается', () {
      expect(looksLikeOp2(0x10C4, 0xEA60), isFalse); // CP210x
      expect(looksLikeOp2(0x0403, 0x9999), isFalse);
    });
  });

  group('helpers', () {
    test('hex16', () => expect(hex16(0xCC4C), '0xCC4C'));
    test('describeOp2 сохраняет vid/pid/product', () {
      final m = describeOp2(const {
        'found': true, 'vid': '0x0403', 'pid': '0xCC4C', 'product': 'OpenPort 2.0',
      });
      expect(m['pid'], '0xCC4C');
      expect(m['product'], contains('OpenPort'));
    });
  });
}
'''

OP2_NOTES_MD = '''# OP2 OTG — стендовый чеклист (HARDWARE-гейт)

Модуль 04.8 даёт USB-провод до OpenPort 2.0. Маппинг SSM-кадров (A8 00 …)
на J2534 PassThru-вызывы подтверждается на железе ДО постановки
HARDWARE_CONFIRMED=True:

1. [ ] OTG: descriptor() видит vid=0x0403 pid=0xCC4C, permission ≤ 2 диалога
2. [ ] Снять лог J2534-вызовов EcuFlash (op20pt32 logging) на своей машине
3. [ ] Сверить PassThruConnect(ISO15765, 500000) + фильтр 0x7E8 с elm.dart init
4. [ ] A8-опрос базового набора: качество >= 95% / 10 мин больше, чем по BT
5. [ ] K-line канал (ISO9141 fast-init) для pre-CAN блока — отдельным билдом
6. [ ] Клоны: зафиксировать VID/PID/строку product в report перед выкладкой

Сборка OP2-апк после ячейки 05 вручную:
  /content/flutter/bin/flutter build apk --release \
      --dart-define=SSM_TRANSPORT=native_op2
'''

def write(rel, body, executable=False):
    p = APP / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    if p.exists():
        shutil.copyfile(p, BACKUP / (rel.replace("/", "__") + ".bak"))
    p.write_text(body, encoding="utf-8")
    if executable:
        p.chmod(p.stat().st_mode | 0o111)
    print("[+]", rel)

# ============================================================
# 4. Патч AndroidManifest.xml (idempotent, anchor-based)
# ============================================================
def patch_manifest():
    text = MANIFEST.read_text(encoding="utf-8")
    if "OP2_DEVICE_ATTACHED_OK" in text or "USB_DEVICE_ATTACHED" in text:
        print("[SKIP already applied] AndroidManifest.xml"); return
    shutil.copyfile(MANIFEST, BACKUP / "AndroidManifest.xml.bak")
    if "android.hardware.usb.host" not in text:
        # required=false: BT-only сборка остаётся видна устройствам без OTG
        text = text.replace(
            "<application",
            '<uses-feature android:name="android.hardware.usb.host" '
            'android:required="false" />\n\n    <application', 1)
    hook = (
        '\n            <intent-filter>\n'
        '                <action android:name="android.hardware.usb.action.USB_DEVICE_ATTACHED" />\n'
        '            </intent-filter>\n'
        '            <meta-data\n'
        '                android:name="android.hardware.usb.action.USB_DEVICE_ATTACHED"\n'
        '                android:resource="@xml/op2_device_filter" />\n'
        '            <!-- OP2_DEVICE_ATTACHED_OK -->')
    m = re.search(r'(<activity[^>]*android:name="\.MainActivity"[^>]*>)', text)
    if not m:
        raise RuntimeError("Не нашёл activity .MainActivity в манифесте")
    text = text[:m.end(1)] + hook + text[m.end(1):]
    MANIFEST.write_text(text, encoding="utf-8")
    print("[OK] patched AndroidManifest.xml (usb.host + attach)")

# ============================================================
# 5. Патч MainActivity.kt — регистрация Op2Channel
# ============================================================
def patch_main_activity():
    text = MAIN_ACTIVITY.read_text(encoding="utf-8")
    if "Op2Channel.register" in text:
        print("[SKIP already applied] MainActivity.kt"); return
    shutil.copyfile(MAIN_ACTIVITY, BACKUP / "MainActivity.kt.bak")
    anchor = re.search(r'super\.configureFlutterEngine\(flutterEngine\)', text)
    if not anchor:
        raise RuntimeError(
            "Якорь configureFlutterEngine не найден — пришлите MainActivity.kt, "
            "патч руками: Op2Channel.register(this, flutterEngine)")
    text = text[:anchor.end()] + "\n        Op2Channel.register(this, flutterEngine)" + text[anchor.end():]
    MAIN_ACTIVITY.write_text(text, encoding="utf-8")
    print("[OK] patched MainActivity.kt (register Op2Channel)")

# ============================================================
# 6. Патч lib/transport_selected.dart — ветка по --dart-define
# ============================================================
def patch_transport_selected():
    if not PATCH_TRANSPORT:
        print("[SKIP] PATCH_TRANSPORT=False"); return
    text = TRANSPORT_SEL.read_text(encoding="utf-8")
    if "native_op2" in text or "SSM_TRANSPORT" in text:
        print("[SKIP already applied] transport_selected.dart"); return
    shutil.copyfile(TRANSPORT_SEL, BACKUP / "lib__transport_selected.dart.bak")

    # 6.1 импорт после последнего import-а
    imports = list(re.finditer(r'(?m)^import[^;]+;$', text))
    if not imports:
        text = "import 'native_op2.dart';\n" + text
    else:
        last = imports[-1]
        text = text[:last.end()] + "\nimport 'native_op2.dart';" + text[last.end():]

    # 6.2 константа выбора (по умолчанию BT — сборка 0.14.1 не меняется)
    const_src = ("\n/// v0.15.0: 'native_spp' (по умолчанию) | 'native_op2' (USB OTG).\n"
                 "const String kTransportName =\n"
                 "    String.fromEnvironment('SSM_TRANSPORT', defaultValue: 'native_spp');\n")
    first_decl = re.search(r'(?m)^(class|final|const|[A-Za-z_]+\s+createTransport)', text)
    if first_decl:
        text = text[:first_decl.start()] + const_src.strip("\n") + "\n\n" + text[first_decl.start():]
    else:
        text += const_src

    # 6.3 оборачиваем фабрику createTransport (=> и { } формы)
    m = re.search(r'createTransport\s*\([^)]*\)\s*=>\s*', text)
    if m:
        # => form: оборачиваем исходное выражение в else-ветку
        end = text.index(';', m.end())
        original = text[m.end():end]
        text = (text[:m.start()] + "createTransport() {\n"
                "  if (kTransportName == 'native_op2') return NativeOp2Transport();\n"
                "  return " + original + ";\n}" + text[end + 1:])
        print("[OK] patched createTransport (=> form)")
    else:
        m = re.search(r'createTransport\s*\([^)]*\)\s*\{', text)
        if not m:
            raise RuntimeError(
                "Фабрика createTransport не найдена в transport_selected.dart — "
                "добавьте ветку вручную: if (kTransportName == 'native_op2') "
                "return NativeOp2Transport();")
        text = text[:m.end()] + ("\n  if (kTransportName == 'native_op2') "
                                  "return NativeOp2Transport();") + text[m.end():]
        print("[OK] patched createTransport ({ } form)")
    TRANSPORT_SEL.write_text(text, encoding="utf-8")

# ============================================================
# 7. Записываем файлы модуля
# ============================================================
print("SSM2 0.14.1 >> 0.15.0 | OP2 OTG module")
write("android/app/src/main/kotlin/com/subaru/ssm2_fixed/Op2Channel.kt", OP2_CHANNEL_KT)
write("android/app/src/main/res/xml/op2_device_filter.xml", DEVICE_FILTER_XML)
write("lib/native_op2.dart", NATIVE_OP2_DART)
write("test/op2_transport_test.dart", OP2_TEST_DART)
write("tool/op2_j2534_notes.md", OP2_NOTES_MD)

patch_manifest()
patch_main_activity()
patch_transport_selected()

# ============================================================
# 8. Self-check
# ============================================================
checks = [
    (MAIN_ACTIVITY, "Op2Channel.register", "MainActivity не пропатчен"),
    (MANIFEST, "USB_DEVICE_ATTACHED", "манифест без attach-фильтра"),
    (MANIFEST, "android.hardware.usb.host", "манифест без uses-feature"),
    (TRANSPORT_SEL, "SSM_TRANSPORT", "transport_selected без ветки"),
    (TRANSPORT_SEL, "native_op2.dart", "transport_selected без импорта"),
    (APP / "lib/native_op2.dart", "class NativeOp2Transport", "dart-обёртка пуста"),
    (APP / "android/app/src/main/kotlin/com/subaru/ssm2_fixed/Op2Channel.kt",
     "class FtdiPipe", "Op2Channel.kt пуст"),
]
for path, needle, msg in checks:
    if needle not in path.read_text(encoding="utf-8"):
        raise RuntimeError(f"SELF-CHECK FAIL: {msg}")
print("[OK] self-check passed (" + str(len(checks)) + " проверок)")

# ============================================================
# 9. Дымовая проверка Dart-стороны (строгий гейт остаётся в 05)
# ============================================================
import json
cfg = json.loads((APP / "build_config.json").read_text(encoding="utf-8"))
FLUTTER_BIN = Path(cfg["flutter"]) / "bin/flutter"
DART_BIN = Path(cfg["flutter"]) / "bin/dart"
env_path = None
try:
    subprocess.run([str(DART_BIN), "format", "lib/native_op2.dart",
                    "lib/transport_selected.dart", "test/op2_transport_test.dart"],
                   cwd=APP, capture_output=True, text=True, timeout=300)
    print("[OK] dart format")
    smoke = subprocess.run([str(FLUTTER_BIN), "analyze", "--no-pub",
                            "--no-fatal-infos", "--no-fatal-warnings"],
                           cwd=APP, capture_output=True, text=True, timeout=600)
    tail = (smoke.stdout + smoke.stderr).strip().splitlines()[-3:]
    print("[SMOKE analyze]", " | ".join(tail))
except Exception as e:
    print("[SMOKE skipped]", e, "— строгая проверка будет в ячейке 05")

print()
print("=== SSM2 0.15.0 | OP2 OTG установлен. Дальше — ячейка 05 ===")
print("  * APK из ячейки 05 остаётся BLUETOOTH-сборкой (default native_spp)")
print("  * OP2-сборка вручную: flutter build apk --release \\")
print("      --dart-define=SSM_TRANSPORT=native_op2")
print("  * Перед HARDWARE_CONFIRMED пройдите tool/op2_j2534_notes.md")
print(f"  * Бэкапы затронутых файлов: {BACKUP}")


SSM2 0.14.1 >> 0.15.0 | OP2 OTG module
[+] android/app/src/main/kotlin/com/subaru/ssm2_fixed/Op2Channel.kt
[+] android/app/src/main/res/xml/op2_device_filter.xml
[+] lib/native_op2.dart
[+] test/op2_transport_test.dart
[+] tool/op2_j2534_notes.md
[SKIP already applied] AndroidManifest.xml
[SKIP already applied] MainActivity.kt
[SKIP already applied] transport_selected.dart
[OK] self-check passed (7 проверок)
[OK] dart format
[SMOKE analyze]   error • The function 'describeOp2' isn't defined • test/op2_transport_test.dart:23:17 • undefined_function |  | 11 issues found. (ran in 2.8s)

=== SSM2 0.15.0 | OP2 OTG установлен. Дальше — ячейка 05 ===
  * APK из ячейки 05 остаётся BLUETOOTH-сборкой (default native_spp)
  * OP2-сборка вручную: flutter build apk --release \
      --dart-define=SSM_TRANSPORT=native_op2
  * Перед HARDWARE_CONFIRMED пройдите tool/op2_j2534_notes.md
  * Бэкапы затронутых файлов: /content/ssm2_backup_048_20261006_083321


In [ ]:
# coding: utf-8
# @title 04.8 | OpenPort 2.0 OTG REAL | SSM2 0.15.0 (REPLACE старую 04.8)
#
# ВАЖНО: это ПОЛНАЯ замена старых 04.8 r1-r4. Старые версии делали из OP2
# «сырой FTDI/ELM» — это неверно. Настоящий OpenPort 2.0:
#   * USB CDC-ACM, оригинал VID:PID = 0403:CC4D (не CC4C);
#   * data interface ищется по bulk IN + bulk OUT (обычно interface 1);
#   * wire protocol: ato6 / atf6 / att6 + бинарные J2534 payload;
#   * ISO-TP делает прошивка OpenPort, ELM AT-команды эмулируются приложению.
#
# ВЫБОР АДАПТЕРА В ПРИЛОЖЕНИИ:
#   HybridTransport.paired() объединяет два списка:
#     1) OpenPort 2.0 (USB OTG) с address usb:0403:cc4d, если кабель подключён;
#     2) обычные сопряжённые Bluetooth ELM327.
#   Существующий экран выбора адаптера показывает оба. Выбран usb:* -> OP2,
#   выбран BT MAC -> native SPP. Один APK, без --dart-define.
#
# Протокол реализован независимо по публичной wire-документации:
# https://github.com/bisak/openport-j2534/blob/main/docs/PROTOCOL.md
# Никакой код GPL-проекта в APK не копируется.
import json
import re
import shutil
import subprocess
import time
from pathlib import Path

APP = Path("/content/subaru_ssm2_fixed")
STAMP = time.strftime("%Y%m%d_%H%M%S")
BACKUP = APP.parent / f"ssm2_backup_048real_{STAMP}"

PUBSPEC = APP / "pubspec.yaml"
BT_PATH = APP / "lib/bt_transport.dart"
NATIVE_SPP = APP / "lib/native_spp.dart"
TRANSPORT_SEL = APP / "lib/transport_selected.dart"
MAIN_ACTIVITY = APP / "android/app/src/main/kotlin/com/subaru/ssm2_fixed/MainActivity.kt"
MANIFEST = APP / "android/app/src/main/AndroidManifest.xml"

for need in (APP / "build_config.json", PUBSPEC, BT_PATH, NATIVE_SPP,
             TRANSPORT_SEL, MAIN_ACTIVITY, MANIFEST):
    if not need.exists():
        raise RuntimeError(f"Нет файла: {need}. Выполните 01, 02 и 04.1-04.7.")
BACKUP.mkdir(parents=True, exist_ok=True)

def backup(path):
    if path.exists():
        shutil.copyfile(path, BACKUP / (str(path.relative_to(APP)).replace("/", "__") + ".bak"))

def write(rel, body):
    path = APP / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    backup(path)
    path.write_text(body, encoding="utf-8")
    print("[+]", rel)

# ============================================================
# 0. GROUND TRUTH: пакет + BtTransport
# ============================================================
def package_name(text):
    m = re.search(r"(?m)^name:\s*([a-zA-Z_0-9]+)\s*$", text)
    return m.group(1) if m else ""

pub = PUBSPEC.read_text(encoding="utf-8")
votes = {}
for path in sorted((APP / "test").glob("*.dart")):
    for m in re.finditer(r"(?m)^\s*import 'package:([a-zA-Z_0-9]+)/",
                         path.read_text(encoding="utf-8")):
        name = m.group(1)
        if not name.startswith(("flutter", "matcher", "meta")):
            votes[name] = votes.get(name, 0) + 1
real_name = max(votes.items(), key=lambda item: item[1])[0] if votes else package_name(pub)
if not real_name:
    raise RuntimeError("Не удалось определить name пакета")
print("[GROUND] package votes:", sorted(votes.items(), key=lambda x: -x[1]))

# Лечим только последствия предыдущих 04.8: name и bogus dep.
changed = []
if package_name(pub) != real_name:
    backup(PUBSPEC)
    old = package_name(pub)
    pub = re.sub(r"(?m)^name:\s*[a-zA-Z_0-9]+\s*$", "name: " + real_name, pub, count=1)
    changed.append(f"name {old} -> {real_name}")
for bogus in ("subaru_ssm2_fixed", "subaru_ssm2"):
    if bogus == real_name:
        continue
    new_pub = re.sub(r"(?m)^[ ]+" + re.escape(bogus) + r":\s*any\s*$\n?", "", pub)
    if new_pub != pub:
        backup(PUBSPEC)
        pub = new_pub
        changed.append("removed bogus dep " + bogus + ": any")
if changed:
    PUBSPEC.write_text(pub, encoding="utf-8")
    for item in changed:
        print("[repair pubspec]", item)
PKG = package_name(PUBSPEC.read_text(encoding="utf-8"))
print("[GROUND] package:", PKG)

bt_src = BT_PATH.read_text(encoding="utf-8")
required_contract = {
    "String get name", "bool get connected", "Stream<Uint8List> get data",
    "Stream<bool> get status", "Future<List<BtDevice>> paired()",
    "Future<void> connect(String address)", "Future<void> write(String ascii)",
    "Future<void> disconnect()", "Future<void> dispose()",
}
missing_contract = [sig for sig in required_contract if sig not in bt_src]
if missing_contract:
    print("\n===== lib/bt_transport.dart =====\n" + bt_src)
    raise RuntimeError("Контракт BtTransport изменился: " + str(missing_contract))
print("[GROUND] BtTransport: подтверждены 9 членов")

# ============================================================
# 1. Dart: настоящий OP2-транспорт и runtime-выбор
# ============================================================
OP2_IDS_DART = r'''// OpenPort 2.0 USB ids (original + known clone fallback).
const int kOp2Vid = 0x0403;
const List<int> kOp2Pids = <int>[0xCC4D, 0xCC4C];

bool looksLikeOp2(int vid, int pid) => vid == kOp2Vid && kOp2Pids.contains(pid);
String hex16(int v) => '0x' + v.toRadixString(16).toUpperCase().padLeft(4, '0');
'''

NATIVE_OP2_DART = r'''// OpenPort 2.0 over Android USB OTG.
import 'dart:async';
import 'package:flutter/services.dart';

import 'bt_transport.dart';

class NativeOp2Transport implements BtTransport {
  static const MethodChannel _method = MethodChannel('ssm2/op2');
  static const EventChannel _events = EventChannel('ssm2/op2_rx');

  final StreamController<bool> _status = StreamController<bool>.broadcast();
  bool _connected = false;

  @override
  String get name => 'OpenPort 2.0 (USB OTG)';
  @override
  bool get connected => _connected;
  @override
  Stream<Uint8List> get data => _events
      .receiveBroadcastStream()
      .where((Object? e) => e is Uint8List)
      .cast<Uint8List>();
  @override
  Stream<bool> get status => _status.stream;

  @override
  Future<List<BtDevice>> paired() async {
    final raw = await _method.invokeMethod<Map<dynamic, dynamic>>('op2/descriptor');
    final d = raw?.cast<String, Object?>() ?? const <String, Object?>{};
    if (d['found'] == true) {
      final pid = (d['pid'] ?? '0xCC4D').toString().toLowerCase().replaceAll('0x', '');
      return <BtDevice>[BtDevice('OpenPort 2.0 (USB OTG)', 'usb:0403:' + pid)];
    }
    return <BtDevice>[];
  }

  @override
  Future<void> connect(String address) async {
    final raw = await _method.invokeMethod<Map<dynamic, dynamic>>('op2/open');
    final d = raw?.cast<String, Object?>() ?? const <String, Object?>{};
    if (d['open'] != true) {
      throw StateError((d['error'] ?? 'OpenPort 2.0 не открыт').toString());
    }
    _connected = true;
    _status.add(true);
  }

  @override
  Future<void> write(String ascii) => _method.invokeMethod<void>('op2/write', ascii);

  @override
  Future<void> disconnect() async {
    await _method.invokeMethod<void>('op2/close');
    _connected = false;
    _status.add(false);
  }

  @override
  Future<void> dispose() async {
    await disconnect();
    await _status.close();
  }
}
'''

HYBRID_DART = r'''// Один APK: OpenPort 2.0 USB и Bluetooth ELM327 в одном списке.
// Базовый интерфейс BtTransport — implements (native_spp так и не требует extends).
import 'dart:async';
import 'dart:typed_data';

import 'bt_transport.dart';
import 'native_op2.dart';

class HybridTransport implements BtTransport {
  final BtTransport bluetooth;
  final BtTransport usb;
  final StreamController<Uint8List> _data = StreamController<Uint8List>.broadcast();
  final StreamController<bool> _status = StreamController<bool>.broadcast();
  StreamSubscription<Uint8List>? _dataSub;
  StreamSubscription<bool>? _statusSub;
  BtTransport? _active;

  HybridTransport(this.bluetooth, {BtTransport? usb})
      : usb = usb ?? NativeOp2Transport();

  @override
  String get name => _active?.name ?? 'Bluetooth / OpenPort 2.0';
  @override
  bool get connected => _active?.connected ?? false;
  @override
  Stream<Uint8List> get data => _data.stream;
  @override
  Stream<bool> get status => _status.stream;

  @override
  Future<List<BtDevice>> paired() async {
    final usbDevices = await usb.paired();
    final btDevices = await bluetooth.paired();
    return <BtDevice>[...usbDevices, ...btDevices];
  }

  @override
  Future<void> connect(String address) async {
    await disconnect();
    final target = address.toLowerCase().startsWith('usb:') ? usb : bluetooth;
    _active = target;
    _dataSub = target.data.listen(_data.add, onError: _data.addError);
    _statusSub = target.status.listen(_status.add, onError: _status.addError);
    try {
      await target.connect(address);
      _status.add(target.connected);
    } catch (_) {
      await _dataSub?.cancel();
      await _statusSub?.cancel();
      _dataSub = null;
      _statusSub = null;
      _active = null;
      rethrow;
    }
  }

  @override
  Future<void> write(String ascii) async {
    final target = _active;
    if (target == null) throw StateError('Адаптер не выбран');
    await target.write(ascii);
  }

  @override
  Future<void> disconnect() async {
    await _dataSub?.cancel();
    await _statusSub?.cancel();
    _dataSub = null;
    _statusSub = null;
    final target = _active;
    _active = null;
    if (target != null) await target.disconnect();
    _status.add(false);
  }

  @override
  Future<void> dispose() async {
    await disconnect();
    await bluetooth.dispose();
    await usb.dispose();
    await _data.close();
    await _status.close();
  }
}
'''

OP2_TEST = r'''import 'dart:async';
import 'dart:typed_data';

import 'package:flutter_test/flutter_test.dart';
import 'package:PKG/bt_transport.dart';
import 'package:PKG/hybrid_transport.dart';
import 'package:PKG/op2_ids.dart';

class FakeTransport extends BtTransport {
  final String fakeName;
  final List<BtDevice> devices;
  final StreamController<Uint8List> dc = StreamController<Uint8List>.broadcast();
  final StreamController<bool> sc = StreamController<bool>.broadcast();
  String? lastAddress;
  String? lastWrite;
  bool open = false;
  FakeTransport(this.fakeName, this.devices);
  @override String get name => fakeName;
  @override bool get connected => open;
  @override Stream<Uint8List> get data => dc.stream;
  @override Stream<bool> get status => sc.stream;
  @override Future<List<BtDevice>> paired() async => devices;
  @override Future<void> connect(String address) async { lastAddress = address; open = true; sc.add(true); }
  @override Future<void> write(String ascii) async { lastWrite = ascii; }
  @override Future<void> disconnect() async { open = false; sc.add(false); }
  @override Future<void> dispose() async { await dc.close(); await sc.close(); }
}

void main() {
  test('OP2 ids: original is 0403:CC4D', () {
    expect(looksLikeOp2(0x0403, 0xCC4D), isTrue);
    expect(looksLikeOp2(0x0403, 0xCC4C), isTrue);
  });
  test('Hybrid merges USB and Bluetooth choices', () async {
    final bt = FakeTransport('BT', <BtDevice>[BtDevice('ELM327', '00:11')]);
    final usb = FakeTransport('USB', <BtDevice>[BtDevice('OpenPort', 'usb:0403:cc4d')]);
    final hybrid = HybridTransport(bt, usb: usb);
    final all = await hybrid.paired();
    expect(all.length, 2);
    await hybrid.connect('usb:0403:cc4d');
    expect(usb.lastAddress, 'usb:0403:cc4d');
    await hybrid.write('A800000008');
    expect(usb.lastWrite, 'A800000008');
    await hybrid.dispose();
  });
}
'''.replace("PKG", PKG)

# ============================================================
# 2. Kotlin: OpenPort CDC + wire protocol + ELM compatibility
# ============================================================
OP2_KOTLIN = r'''package com.subaru.ssm2_fixed

import android.app.PendingIntent
import android.content.BroadcastReceiver
import android.content.Context
import android.content.Intent
import android.content.IntentFilter
import android.hardware.usb.UsbConstants
import android.hardware.usb.UsbDevice
import android.hardware.usb.UsbDeviceConnection
import android.hardware.usb.UsbEndpoint
import android.hardware.usb.UsbInterface
import android.hardware.usb.UsbManager
import android.os.Build
import android.os.Handler
import android.os.Looper
import io.flutter.embedding.engine.FlutterEngine
import io.flutter.plugin.common.EventChannel
import io.flutter.plugin.common.MethodChannel
import java.io.ByteArrayOutputStream
import java.util.concurrent.CompletableFuture
import java.util.concurrent.ConcurrentHashMap
import java.util.concurrent.Executors
import java.util.concurrent.ScheduledExecutorService
import java.util.concurrent.TimeUnit
import java.util.concurrent.atomic.AtomicInteger

object Op2Ids {
    const val VID = 0x0403
    val PIDS = intArrayOf(0xCC4D, 0xCC4C)
    fun matches(d: UsbDevice) = d.vendorId == VID && PIDS.contains(d.productId)
}

private class Op2Pipe(
    private val connection: UsbDeviceConnection,
    private val iface: UsbInterface,
    private val input: UsbEndpoint,
    private val output: UsbEndpoint,
) {
    @Synchronized fun write(bytes: ByteArray, timeout: Int = 1500): Int =
        connection.bulkTransfer(output, bytes, bytes.size, timeout)
    fun read(buffer: ByteArray, timeout: Int = 100): Int =
        connection.bulkTransfer(input, buffer, buffer.size, timeout)
    fun close() {
        runCatching { connection.releaseInterface(iface) }
        connection.close()
    }

    companion object {
        fun open(manager: UsbManager, device: UsbDevice): Op2Pipe {
            val connection = manager.openDevice(device)
                ?: error("UsbManager.openDevice вернул null")
            for (i in 0 until device.interfaceCount) {
                val iface = device.getInterface(i)
                var input: UsbEndpoint? = null
                var output: UsbEndpoint? = null
                for (e in 0 until iface.endpointCount) {
                    val ep = iface.getEndpoint(e)
                    if (ep.type != UsbConstants.USB_ENDPOINT_XFER_BULK) continue
                    if (ep.direction == UsbConstants.USB_DIR_IN) input = ep else output = ep
                }
                val inEp = input
                val outEp = output
                if (inEp != null && outEp != null && connection.claimInterface(iface, true)) {
                    return Op2Pipe(connection, iface, inEp, outEp)
                }
            }
            connection.close()
            error("CDC data interface с bulk IN/OUT не найден")
        }
    }
}

private class Op2Session(
    private val pipe: Op2Pipe,
    private val emit: (ByteArray) -> Unit,
) {
    private val sequence = AtomicInteger(1000)
    private val pending = ConcurrentHashMap<Int, CompletableFuture<String>>()
    private val timer: ScheduledExecutorService = Executors.newSingleThreadScheduledExecutor()
    @Volatile private var running = true
    @Volatile private var rx = ByteArray(0)
    @Volatile private var requestToken = 0
    private var partial: ByteArrayOutputStream? = null
    private var partialId = ByteArray(4)
    private var txId = 0x7E0
    private var showHeaders = true

    private val reader = Thread {
        val buffer = ByteArray(4096)
        while (running) {
            val n = runCatching { pipe.read(buffer) }.getOrDefault(-1)
            if (n > 0) feed(buffer.copyOfRange(0, n))
        }
    }.apply { isDaemon = true; name = "ssm2-op2-reader" }

    fun start() = reader.start()

    fun initialize() {
        pipe.write(byteArrayOf(13, 10, 13, 10))
        ok(command("atz", timeoutMs = 2500))
        ok(command("ata"))
        ok(command("ato6 0 500000 0"))
        // FLOW_CONTROL filter: mask 7FF, response 7E8, flow-control/request 7E0.
        val filter = be32(0x7FF) + be32(0x7E8) + be32(0x7E0)
        val reply = command("atf6 3 64 4", filter)
        if (reply.startsWith("are")) error("OP2 filter rejected: " + reply)
    }

    private fun ok(line: String) {
        if (line.startsWith("are")) error("OpenPort error: " + line)
    }

    private fun command(base: String, payload: ByteArray? = null, timeoutMs: Long = 1800): String {
        val seq = sequence.incrementAndGet()
        val future = CompletableFuture<String>()
        pending[seq] = future
        val line = (base + " " + seq + "\r\n").toByteArray(Charsets.US_ASCII)
        if (pipe.write(line) != line.size) error("USB write command failed")
        if (payload != null && pipe.write(payload) != payload.size) error("USB write payload failed")
        return try {
            future.get(timeoutMs, TimeUnit.MILLISECONDS)
        } finally {
            pending.remove(seq)
        }
    }

    fun elmWrite(raw: String) {
        val text = raw.trim().uppercase()
        if (text.isEmpty()) return
        if (text.startsWith("AT")) {
            if (text.startsWith("ATSH")) text.substring(4).trim().toIntOrNull(16)?.let { txId = it }
            if (text == "ATH0") showHeaders = false
            if (text == "ATH1") showHeaders = true
            emit("OK\r>".toByteArray(Charsets.US_ASCII))
            return
        }
        val compact = text.replace(Regex("[^0-9A-F]"), "")
        if (compact.isEmpty() || compact.length % 2 != 0) {
            emit("?\r>".toByteArray(Charsets.US_ASCII)); return
        }
        val payload = ByteArray(compact.length / 2) { i ->
            compact.substring(i * 2, i * 2 + 2).toInt(16).toByte()
        }
        val message = be32(txId) + payload
        val token = ++requestToken
        try {
            ok(command("att6 " + message.size + " 64 1000000", message, 2200))
            timer.schedule({
                if (requestToken == token) {
                    requestToken++
                    emit("NO DATA\r>".toByteArray(Charsets.US_ASCII))
                }
            }, 650, TimeUnit.MILLISECONDS)
        } catch (e: Exception) {
            if (requestToken == token) requestToken++
            emit("CAN ERROR\r>".toByteArray(Charsets.US_ASCII))
            throw e
        }
    }

    @Synchronized private fun feed(chunk: ByteArray) {
        rx = rx + chunk
        while (rx.size >= 2) {
            if (rx[0] != 'a'.code.toByte() || rx[1] != 'r'.code.toByte()) {
                rx = rx.copyOfRange(1, rx.size); continue
            }
            if (rx.size < 3) return
            val third = rx[2].toInt() and 0xFF
            if (third in '0'.code..'9'.code) {
                if (rx.size < 4) return
                val total = 4 + (rx[3].toInt() and 0xFF)
                if (rx.size < total) return
                val frame = rx.copyOfRange(0, total)
                rx = rx.copyOfRange(total, rx.size)
                handleFrame(frame)
            } else {
                val end = findCrlf(rx)
                if (end < 0) return
                val line = String(rx.copyOfRange(0, end), Charsets.US_ASCII)
                rx = rx.copyOfRange(end + 2, rx.size)
                val seq = line.trim().split(Regex("\\s+")).lastOrNull()?.toIntOrNull()
                if (seq != null) pending.remove(seq)?.complete(line)
            }
        }
    }

    private fun handleFrame(frame: ByteArray) {
        if (frame.size < 9) return
        val status = frame[4].toInt() and 0xFF
        if ((status and 0x10) != 0 || (status and 0x20) != 0) return // tx indication/loopback
        val payload = frame.copyOfRange(9, frame.size)
        if (payload.size < 4) return
        val start = (status and 0x80) != 0
        val end = (status and 0x40) != 0
        if (start && !end) {
            partialId = payload.copyOfRange(0, 4)
            partial = ByteArrayOutputStream()
            return
        }
        val part = partial
        if (part != null) {
            part.write(payload, 4, payload.size - 4)
            if (end) {
                emitElm(partialId, part.toByteArray())
                partial = null
            }
        } else if (end) {
            emitElm(payload.copyOfRange(0, 4), payload.copyOfRange(4, payload.size))
        }
    }

    private fun emitElm(idBytes: ByteArray, data: ByteArray) {
        requestToken++
        val id = ((idBytes[0].toInt() and 0xFF) shl 24) or
            ((idBytes[1].toInt() and 0xFF) shl 16) or
            ((idBytes[2].toInt() and 0xFF) shl 8) or (idBytes[3].toInt() and 0xFF)
        val hex = data.joinToString(" ") { "%02X".format(it.toInt() and 0xFF) }
        val line = (if (showHeaders) "%03X ".format(id) else "") + hex + "\r>"
        emit(line.toByteArray(Charsets.US_ASCII))
    }

    fun close() {
        // Reader должен жить, пока atc/atz ждут свои numbered replies.
        runCatching { command("atc6", timeoutMs = 300) }
        runCatching { command("atz", timeoutMs = 300) }
        running = false
        timer.shutdownNow()
        pending.values.forEach { it.completeExceptionally(IllegalStateException("OP2 closed")) }
        pending.clear()
        pipe.close()
        reader.join(300)
    }

    companion object {
        fun be32(value: Int) = byteArrayOf(
            (value ushr 24).toByte(), (value ushr 16).toByte(),
            (value ushr 8).toByte(), value.toByte())
        fun findCrlf(data: ByteArray): Int {
            for (i in 0 until data.size - 1) if (data[i] == 13.toByte() && data[i + 1] == 10.toByte()) return i
            return -1
        }
    }
}

object Op2Channel : MethodChannel.MethodCallHandler, EventChannel.StreamHandler {
    private const val ACTION = "com.subaru.ssm2_fixed.OP2_PERMISSION"
    private val main = Handler(Looper.getMainLooper())
    private val io = Executors.newSingleThreadExecutor()
    private lateinit var context: Context
    private lateinit var manager: UsbManager
    private var sink: EventChannel.EventSink? = null
    @Volatile private var session: Op2Session? = null
    private var permissionResult: MethodChannel.Result? = null

    private val receiver = object : BroadcastReceiver() {
        override fun onReceive(c: Context, intent: Intent) {
            if (intent.action != ACTION) return
            val granted = intent.getBooleanExtra(UsbManager.EXTRA_PERMISSION_GRANTED, false)
            val result = permissionResult ?: return
            permissionResult = null
            if (!granted) result.success(mapOf("open" to false, "error" to "USB permission denied"))
            else openAsync(result)
        }
    }

    fun register(ctx: Context, engine: FlutterEngine) {
        context = ctx.applicationContext
        manager = context.getSystemService(Context.USB_SERVICE) as UsbManager
        val filter = IntentFilter(ACTION)
        if (Build.VERSION.SDK_INT >= 33) {
            context.registerReceiver(receiver, filter, Context.RECEIVER_NOT_EXPORTED)
        } else {
            @Suppress("UnspecifiedRegisterReceiverFlag")
            context.registerReceiver(receiver, filter)
        }
        MethodChannel(engine.dartExecutor.binaryMessenger, "ssm2/op2").setMethodCallHandler(this)
        EventChannel(engine.dartExecutor.binaryMessenger, "ssm2/op2_rx").setStreamHandler(this)
    }

    private fun device(): UsbDevice? = manager.deviceList.values.firstOrNull(Op2Ids::matches)
    private fun descriptor(): Map<String, Any?> {
        val d = device()
        return mapOf("found" to (d != null),
            "vid" to d?.vendorId?.let { "0x%04X".format(it) },
            "pid" to d?.productId?.let { "0x%04X".format(it) },
            "product" to d?.productName,
            "hasPermission" to (d?.let(manager::hasPermission) ?: false))
    }

    override fun onMethodCall(call: io.flutter.plugin.common.MethodCall, result: MethodChannel.Result) {
        when (call.method) {
            "op2/descriptor" -> result.success(descriptor())
            "op2/open" -> {
                val d = device()
                if (d == null) { result.success(mapOf("open" to false, "error" to "OP2 not found")); return }
                if (!manager.hasPermission(d)) {
                    permissionResult = result
                    val pi = PendingIntent.getBroadcast(context, 0, Intent(ACTION),
                        PendingIntent.FLAG_UPDATE_CURRENT or PendingIntent.FLAG_IMMUTABLE)
                    manager.requestPermission(d, pi)
                } else openAsync(result)
            }
            "op2/write" -> {
                val ascii = call.arguments as? String
                if (ascii == null) { result.error("op2", "String expected", null); return }
                io.execute {
                    runCatching { session?.elmWrite(ascii) ?: error("OP2 not open") }
                        .onSuccess { main.post { result.success(null) } }
                        .onFailure { e -> main.post { result.error("op2", e.message, null) } }
                }
            }
            "op2/close" -> io.execute {
                session?.close(); session = null
                main.post { result.success(null) }
            }
            else -> result.notImplemented()
        }
    }

    private fun openAsync(result: MethodChannel.Result) = io.execute {
        runCatching {
            session?.close()
            val d = device() ?: error("OP2 disappeared")
            val p = Op2Pipe.open(manager, d)
            val s = Op2Session(p) { bytes -> main.post { sink?.success(bytes) } }
            s.start(); s.initialize(); session = s
            mapOf("open" to true, "vid" to "0x%04X".format(d.vendorId),
                "pid" to "0x%04X".format(d.productId))
        }.onSuccess { map -> main.post { result.success(map) } }
         .onFailure { e -> main.post { result.success(mapOf("open" to false, "error" to e.message)) } }
    }

    override fun onListen(arguments: Any?, events: EventChannel.EventSink?) { sink = events }
    override fun onCancel(arguments: Any?) { sink = null }
}
'''

# ============================================================
# 3. Manifest + MainActivity + factory
# ============================================================
FILTER_XML = '''<?xml version="1.0" encoding="utf-8"?>
<resources>
    <usb-device vendor-id="1027" product-id="52301" /> \x3C!-- 0403:CC4D original -->
    <usb-device vendor-id="1027" product-id="52300" /> \x3C!-- 0403:CC4C clone fallback -->
</resources>
'''

def patch_manifest():
    text = MANIFEST.read_text(encoding="utf-8")
    backup(MANIFEST)
    if "android.hardware.usb.host" not in text:
        text = text.replace("<application",
            '<uses-feature android:name="android.hardware.usb.host" android:required="false" />\n\n    <application', 1)
    if "USB_DEVICE_ATTACHED" not in text:
        m = re.search(r'(<activity[^>]*android:name="\.MainActivity"[^>]*>)', text)
        if not m: raise RuntimeError("MainActivity не найдена в manifest")
        hook = ('\n            <intent-filter>\n'
                '                <action android:name="android.hardware.usb.action.USB_DEVICE_ATTACHED" />\n'
                '            </intent-filter>\n'
                '            <meta-data android:name="android.hardware.usb.action.USB_DEVICE_ATTACHED"\n'
                '                android:resource="@xml/op2_device_filter" />')
        text = text[:m.end()] + hook + text[m.end():]
    MANIFEST.write_text(text, encoding="utf-8")

def patch_main():
    text = MAIN_ACTIVITY.read_text(encoding="utf-8")
    if "Op2Channel.register" in text: return
    backup(MAIN_ACTIVITY)
    m = re.search(r"super\.configureFlutterEngine\(flutterEngine\)", text)
    if not m: raise RuntimeError("configureFlutterEngine anchor не найден")
    text = text[:m.end()] + "\n        Op2Channel.register(this, flutterEngine)" + text[m.end():]
    MAIN_ACTIVITY.write_text(text, encoding="utf-8")

def function_span(text):
    m = re.search(r"(?m)^([A-Za-z_][\w<>,.\[\]?! ]*?)\s+createTransport\s*\([^)]*\)\s*", text)
    if not m: raise RuntimeError("createTransport не найдена")
    pos = m.end()
    if text.startswith("=>", pos):
        end = text.index(";", pos)
        return m.start(), end + 1, text[pos + 2:end].strip()
    brace = text.find("{", pos)
    if brace < 0: raise RuntimeError("тело createTransport не найдено")
    depth, i = 0, brace
    while i < len(text):
        if text[i] == "{": depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0: return m.start(), i + 1, text[brace + 1:i]
        i += 1
    raise RuntimeError("незакрытое тело createTransport")

def find_bt_class():
    """Фактический BtTransport-класс native_spp.dart: implements ИЛИ extends.
    04.1 мог их потому поменять при поддержке контракта задания контрактной
    реализации за локальной стороны."""
    src = NATIVE_SPP.read_text(encoding="utf-8")
    m = (re.search(r"class\s+(\w+)\s+extends\s+BtTransport\b", src)
         or re.search(r"class\s+(\w+)\s+implements\s+BtTransport\b", src))
    if not m:
        print("\n===== native_spp.dart HEAD =====\n" + src[:4000])
        raise RuntimeError("В native_spp.dart нет класса extends/implements BtTransport — "
                           "пришлите шапку файла")
    return m.group(1)

def extract_bt_expression(text):
    # Провер по body-фабрики: если там реконструирован конкретный
    # материал, используем его, иначе — распознанный класс из native_spp.
    _, _, body = function_span(text)
    assigns = {m.group(1): m.group(2).strip() for m in
        re.finditer(r"(?:final|var|BtTransport)\s+(\w+)\s*=\s*([^;]+);", body)}
    candidates = [m.group(1).strip() for m in re.finditer(r"return\s+([^;]+);", body)]
    candidates.append(body.strip())
    for expr in reversed(candidates):
        if expr in assigns: expr = assigns[expr]
        if any(bad in expr for bad in ("NativeOp2", "AutoTransport",
                                        "HybridTransport", "SelectedTransport", "null")):
            continue
        m = re.search(r"([A-Za-z_][A-Za-z0-9_]*Transport)\s*\(", expr)
        if m:
            cls = m.group(1)
            if cls == "Transport":
                # Движковая модификация: имя колнад OneRegion<DrugStorage>
                continue
            return expr
    return find_bt_class() + "()"

def patch_factory():
    old = TRANSPORT_SEL.read_text(encoding="utf-8")
    backup(TRANSPORT_SEL)
    bt_expr = extract_bt_expression(old)
    directives = re.findall(r"(?m)^(?:import|export)\s+[^;]+;$", old)
    directives = [d for d in directives if not any(x in d for x in
        ("native_op2.dart", "auto_transport.dart", "hybrid_transport.dart"))]
    # Также удаляем любые оставшиеся следы старых auto-hybrid прогонов:
    # объявления констант kOp2AndrBaud/Latency, kTransportName и т.д.,
    # встроенные helper-методы _createBtTransport и вызовы AutoTransport в
    # теле фабрики — они ломают итоговое референсное присвоение.
    def ensure(line):
        if line not in directives: directives.append(line)
    ensure("import 'bt_transport.dart';")
    ensure("import 'native_spp.dart';")
    ensure("import 'hybrid_transport.dart';")
    seen, clean = set(), []
    for d in directives:
        if d not in seen: clean.append(d); seen.add(d)
    body = "\n".join(clean) + "\n\n"
    body += "// v0.15: existing adapter picker lists USB OP2 + paired Bluetooth.\n"
    body += "BtTransport createTransport() => HybridTransport(" + bt_expr + ");\n"
    TRANSPORT_SEL.write_text(body, encoding="utf-8")
    print("[factory] BT =", bt_expr)
    print("[factory] runtime choice = HybridTransport")

# ============================================================
# 4. Apply
# ============================================================
print("SSM2 0.15 | OpenPort 2.0 REAL wire protocol + runtime adapter choice")
# Подчистка моих же предыдущих экспериментов: файл старого auto-facade и его
# импорт из transport_selected.dart конфликтуют с итоговой REAL-версией.
stale = APP / "lib/auto_transport.dart"
if stale.exists():
    shutil.move(str(stale), str(BACKUP / "lib__auto_transport.dart.stale_removed"))
    print("[cleanup] lib/auto_transport.dart → бэкап (устаревший r3-файл)")
write("lib/op2_ids.dart", OP2_IDS_DART)
write("lib/native_op2.dart", NATIVE_OP2_DART)
write("lib/hybrid_transport.dart", HYBRID_DART)
write("test/op2_transport_test.dart", OP2_TEST)
write("android/app/src/main/kotlin/com/subaru/ssm2_fixed/Op2Channel.kt", OP2_KOTLIN)
write("android/app/src/main/res/xml/op2_device_filter.xml", FILTER_XML)
write("tool/OPENPORT_PROTOCOL_NOTICE.md", """# OpenPort 2.0 protocol notice

Independent Android implementation using facts documented at:
https://github.com/bisak/openport-j2534/blob/main/docs/PROTOCOL.md

No source code from that project is copied into this application. OpenPort and
Tactrix are nominative marks of their respective owner. Hardware test required.
""")
patch_manifest()
patch_main()
patch_factory()

# ============================================================
# 5. Verify: pub get -> format -> analyze -> targeted tests
# ============================================================
checks = [
    (APP / "lib/hybrid_transport.dart", "Future<List<BtDevice>> paired()", "runtime picker missing"),
    (APP / "lib/hybrid_transport.dart", "usb:", "USB route missing"),
    (APP / "lib/native_op2.dart", "Stream<bool> get status", "BtTransport contract mismatch"),
    (APP / "android/app/src/main/kotlin/com/subaru/ssm2_fixed/Op2Channel.kt", "ato6 0 500000 0", "ISO15765 open missing"),
    (APP / "android/app/src/main/kotlin/com/subaru/ssm2_fixed/Op2Channel.kt", "atf6 3 64 4", "flow filter missing"),
    (APP / "android/app/src/main/kotlin/com/subaru/ssm2_fixed/Op2Channel.kt", "att6 ", "transmit missing"),
    (APP / "android/app/src/main/res/xml/op2_device_filter.xml", "52301", "PID CC4D missing"),
    (TRANSPORT_SEL, "HybridTransport", "factory not hybrid"),
]
for path, needle, message in checks:
    if needle not in path.read_text(encoding="utf-8"):
        raise RuntimeError("SELF-CHECK: " + message)
print("[OK] self-check:", len(checks), "проверок")

cfg = json.loads((APP / "build_config.json").read_text(encoding="utf-8"))
FLUTTER = Path(cfg["flutter"]) / "bin/flutter"
DART = Path(cfg["flutter"]) / "bin/dart"

def run(args, timeout=1200):
    p = subprocess.run([str(x) for x in args], cwd=APP, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=timeout)
    print(p.stdout[-10000:])
    if p.returncode: raise RuntimeError("Команда упала: " + " ".join(map(str, args)))

run([FLUTTER, "pub", "get"])
run([DART, "format", "lib/op2_ids.dart", "lib/native_op2.dart",
     "lib/hybrid_transport.dart", "lib/transport_selected.dart",
     "test/op2_transport_test.dart"])
run([FLUTTER, "analyze", "--no-pub", "--no-fatal-infos"])
run([FLUTTER, "test", "--no-pub", "test/op2_transport_test.dart", "--reporter", "expanded"])
# Kotlin/Manifest/USB API не проверяются Dart-анализатором — обязательный debug build.
run([FLUTTER, "build", "apk", "--debug", "--no-pub"], timeout=3600)

print("\n=== OK: OpenPort 2.0 REAL + выбор USB/Bluetooth установлен ===")
print("Выбор: подключите OP2 по OTG -> обновите список адаптеров ->")
print("        выберите 'OpenPort 2.0 (USB OTG)'.")
print("Bluetooth ELM327 остаётся в том же списке по MAC-адресу.")
print("Далее запускайте ячейку 05.")
print("HARDWARE_CONFIRMED оставьте False до 10-минутного стенда.")
print("Бэкапы:", BACKUP)


[GROUND] package votes: [('subaru_ssm2', 23)]
[GROUND] package: subaru_ssm2
[GROUND] BtTransport: подтверждены 9 членов
SSM2 0.15 | OpenPort 2.0 REAL wire protocol + runtime adapter choice
[+] lib/op2_ids.dart
[+] lib/native_op2.dart
[+] lib/hybrid_transport.dart
[+] test/op2_transport_test.dart
[+] android/app/src/main/kotlin/com/subaru/ssm2_fixed/Op2Channel.kt
[+] android/app/src/main/res/xml/op2_device_filter.xml
[+] tool/OPENPORT_PROTOCOL_NOTICE.md
[factory] BT = NativeSppTransport()
[factory] runtime choice = HybridTransport
[OK] self-check: 8 проверок
Resolving dependencies...
  characters 1.4.0 (1.4.1 available)
  clock 1.1.2 (1.1.3 available)
  cross_file 0.3.5+2 (0.4.0 available)
  matcher 0.12.17 (0.12.20 available)
  material_color_utilities 0.11.1 (0.13.1 available)
  meta 1.16.0 (1.19.0 available)
  path_provider 2.1.5 (2.1.6 available)
  path_provider_android 2.2.23 (2.3.1 available)
  path_provider_foundation 2.5.1 (2.6.0 available)
  path_provider_linux 2.2.1 (2.2.2 ava

In [ ]:
# @title 05 | SSM2 0.9 - guard зависимостей, строгий анализ, тесты, подпись APK { display-mode: "form" }
# v0.8 changelog:
#  - prepare_bt.py и вендоринг pub-плагинов УДАЛЕНЫ: транспорт — нативный RFCOMM-канал.
#  - Ячейка "автофикс импортов" удалена: вместо неё штатный `dart fix --apply`.
#  - flutter analyze снова строгий: предупреждения = ошибка сборки.
#  - Поддержка настоящей подписи release APK: keystore в base64 (параметр или Colab Secret
#    SSM2_KEYSTORE_B64) + apksigner --print-certs (SHA-256 сертификата идёт в отчёт).
#  - HARDWARE_CONFIRMED: чеклист стендовой проверки фиксируется в report.json.

import base64
import hashlib
import json
import os
from pathlib import Path
import re
import shutil
import subprocess
import sys
import time

BUILD_VARIANT = "release"  # @param ["release", "debug", "profile"]
CLEAN_BEFORE = True  # @param {type:"boolean"}
STRICT_ANALYZE = True  # @param {type:"boolean"}
HARDWARE_CONFIRMED = False  # @param {type:"boolean"}

# --- Подпись: заполните параметры ИЛИ создайте Colab Secrets с теми же именами ---
KEYSTORE_B64 = ""  # @param {type:"string"}
KEYSTORE_PASSWORD = ""  # @param {type:"string"}
KEY_ALIAS = "ssm2"  # @param {type:"string"}
KEY_PASSWORD = ""  # @param {type:"string"}


def colab_secret(name):
    """Секреты Colab (иконка ключа слева). Вне Colab — просто пусто."""
    try:
        from google.colab import userdata
        value = userdata.get(name)
        return value or ""
    except Exception:
        return ""


# --- 0. Очистка предыдущих сборок ---
print("=== 0. Очистка предыдущих сборок ===")
content_dir = Path("/content")
for old_apk in content_dir.glob("ssm2_fixed_*.apk"):
    try:
        old_apk.unlink()
        print(f"[Удалён старый APK] {old_apk.name}")
    except Exception as e:
        print(f"[Ошибка удаления {old_apk.name}]: {e}")
for old_item in content_dir.glob("ssm2_build_*"):
    try:
        if old_item.is_dir():
            shutil.rmtree(old_item, ignore_errors=True)
        else:
            old_item.unlink(missing_ok=True)
        print(f"[Удалён старый отчёт] {old_item.name}")
    except Exception as e:
        print(f"[Ошибка очистки {old_item.name}]: {e}")

APP = Path("/content/subaru_ssm2_fixed")
CONFIG = APP / "build_config.json"
if not CONFIG.exists():
    raise RuntimeError("Не найден проект. Выполните ячейки 01 и 02.")
cfg = json.loads(CONFIG.read_text(encoding="utf-8"))
FLUTTER = Path(cfg["flutter"])
SDK = Path(cfg["sdk"])
JAVA = Path(cfg["java"])
os.environ.update(JAVA_HOME=str(JAVA), ANDROID_HOME=str(SDK), ANDROID_SDK_ROOT=str(SDK))
os.environ["PATH"] = os.pathsep.join([
    str(JAVA / "bin"), str(FLUTTER / "bin"),
    str(SDK / "cmdline-tools/latest/bin"), os.environ.get("PATH", "")])
stamp = time.strftime("%Y%m%d_%H%M%S")
REPORT = Path(f"/content/ssm2_build_{stamp}")
REPORT.mkdir(parents=True, exist_ok=True)
report = {"revision": "0.9.2", "transport": cfg.get("bt_package", "native_spp"),
          "variant": BUILD_VARIANT, "flutter": cfg.get("flutter_version", "?"),
          "stages": {}, "signing": "debug"} | {"hardware_tested": HARDWARE_CONFIRMED}


def save_report():
    (REPORT / "report.json").write_text(
        json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")


def stage(name, args, timeout=3600):
    logfile = REPORT / f"{name}.log"
    print(f"\n=== {name} ===\n> {' '.join(map(str, args))}\nЖурнал: {logfile}")
    started = time.monotonic()
    try:
        with logfile.open("w", encoding="utf-8") as out:
            p = subprocess.run(list(map(str, args)), cwd=APP, text=True,
                               stdout=out, stderr=subprocess.STDOUT, timeout=timeout)
        text = logfile.read_text(encoding="utf-8", errors="replace")
        print(text[-7000:])
        report["stages"][name] = {"exit_code": p.returncode,
                                  "seconds": round(time.monotonic() - started, 1)}
        save_report()
        if p.returncode:
            raise RuntimeError(f"Этап {name} не пройден. Полный журнал: {logfile}")
        return text
    except subprocess.TimeoutExpired:
        report["stages"][name] = {"error": "timeout"}
        save_report()
        raise RuntimeError(f"Таймаут этапа {name}. Журнал: {logfile}")


print("SSM2 0.9.2 | Проверка перед сборкой")
required = [
    FLUTTER / "bin/flutter", FLUTTER / "bin/dart", JAVA / "bin/java",
    SDK / "platforms/android-36/android.jar", SDK / "build-tools/35.0.0/aapt",
    SDK / "build-tools/35.0.0/apksigner", SDK / "ndk/27.0.12077973/source.properties",
    APP / "lib/native_spp.dart", APP / "lib/transport_selected.dart",
    APP / "lib/analyzer.dart", APP / "lib/derived.dart", APP / "lib/identity.dart",
    APP / "lib/elm.dart", APP / "lib/engine.dart", APP / "lib/pids.dart",
    APP / "lib/maplab_link.dart", APP / "lib/maps_lab/maps_lab_page.dart",
    APP / "test/protocol_test.dart", APP / "test/maps_lab_test.dart",
    APP / "tool/calid_import.py", APP / "android/app/build.gradle.kts"]
for file in required:
    if not file.exists():
        raise RuntimeError(f"Отсутствует: {file}. Перегенерируйте проект ячейкой 02.")
    print("[OK]", file)

version_log = stage("flutter_version",
                    [FLUTTER / "bin/flutter", "--version", "--machine"], timeout=600)
actual = json.JSONDecoder().raw_decode(version_log[version_log.index("{"):])[0]
report["flutter"] = actual["frameworkVersion"]
save_report()

# --- 1. Подпись release APK ---
print("\n=== signing_setup ===")
keystore_b64 = KEYSTORE_B64.strip() or colab_secret("SSM2_KEYSTORE_B64").strip()
store_password = KEYSTORE_PASSWORD or colab_secret("SSM2_STORE_PASSWORD")
key_password = KEY_PASSWORD or colab_secret("SSM2_KEY_PASSWORD")
key_alias = (KEY_ALIAS or colab_secret("SSM2_KEY_ALIAS") or "ssm2").strip()
key_props = APP / "android/key.properties"
keystore_file = APP / "android/keystore_upload.jks"
key_props.unlink(missing_ok=True)
if keystore_b64:
    keystore_file.write_bytes(base64.b64decode(keystore_b64))
    key_props.write_text(
        f"storeFile=keystore_upload.jks\nstorePassword={store_password}\n"
        f"keyAlias={key_alias}\nkeyPassword={key_password}\n", encoding="utf-8")
    report["signing"] = "upload"
    print(f"[OK] Upload-keystore готов ({keystore_file.stat().st_size} байт), alias={key_alias}")
else:
    print("!!! ВНИМАНИЕ: keystore не задан — release APK будет подписан DEBUG-ключом.")
    print("!!! Прошивки/обновления это не сломает, но публикации и смены ключа не переживёт.")
    print("!!! Как сделать ключ: keytool -genkey -v -keystore ssm2.jks -alias ssm2 \\")
    print("!!!     -keyalg RSA -keysize 2048 -validity 10000 && base64 -w0 ssm2.jks")
    # v0.8.1: свежий Colab-рантайм может не иметь ~/.android/debug.keystore
    # (validateSigningRelease падает, если файла нет) — создаём сами стандартными параметрами.
    debug_keystore = Path.home() / ".android/debug.keystore"
    if debug_keystore.exists():
        print(f"[OK] Найден debug.keystore: {debug_keystore}")
    else:
        debug_keystore.parent.mkdir(parents=True, exist_ok=True)
        p = subprocess.run([
            str(JAVA / "bin/keytool"), "-genkeypair", "-v",
            "-keystore", str(debug_keystore),
            "-storepass", "android", "-keypass", "android",
            "-alias", "androiddebugkey",
            "-keyalg", "RSA", "-keysize", "2048", "-validity", "10000",
            "-dname", "CN=Android Debug,O=Android,C=US"],
            capture_output=True, text=True, timeout=300)
        if p.returncode or not debug_keystore.exists():
            raise RuntimeError(f"Не удалось создать debug.keystore: {p.stdout[-1500:]}")
        print(f"[OK] Создан debug.keystore: {debug_keystore}")
save_report()

properties_file = APP / "android/local.properties"
properties = {}
if properties_file.exists():
    for line in properties_file.read_text(encoding="utf-8").splitlines():
        if "=" in line and not line.lstrip().startswith("#"):
            key, value = line.split("=", 1)
            properties[key.strip()] = value
properties.update({"sdk.dir": str(SDK), "flutter.sdk": str(FLUTTER)})
properties_file.write_text(
    "\n".join(f"{key}={value}" for key, value in properties.items()) + "\n", encoding="utf-8")
gradlew = APP / "android/gradlew"
gradlew.chmod(gradlew.stat().st_mode | 0o111)

DART = FLUTTER / "bin/dart"
FLUTTER_BIN = FLUTTER / "bin/flutter"

if CLEAN_BEFORE:
    stage("clean", [FLUTTER_BIN, "clean"], timeout=600)

# --- v0.9: pubspec guard. Проверяем sdk-ограничения ДО pub get, чтобы не ловить
# "requires SDK version >=3.10" уже в середине сборки.
print("\n=== pubspec_guard ===")
dart_version = actual.get("dartSdkVersion", "0.0.0")


def _tuple(v):
    parts = re.findall(r"\d+", v)[:3]
    return tuple(int(x) for x in parts) + (0,) * (3 - len(parts))


def _min_sdk(constraint):
    m = re.search(r"\^\s*([\d.]+)", constraint) or re.search(r">=\s*([\d.]+)", constraint)
    return m.group(1) if m else "0.0.0"


pinned = {}
for line in (APP / "pubspec.yaml").read_text(encoding="utf-8").splitlines():
    m = re.match(r"^  ([a-z_0-9]+):\s*([\d][\w.+-]*)\s*$", line)
    if m:
        pinned[m.group(1)] = m.group(2)

violations = []
print(f"  Dart SDK рантайма: {dart_version}")
for name, version in sorted(pinned.items()):
    try:
        import urllib.request
        url = f"https://pub.dev/api/packages/{name}/versions/{version}"
        with urllib.request.urlopen(url, timeout=30) as resp:
            meta = json.loads(resp.read().decode("utf-8"))
        constraint = meta.get("pubspec", {}).get("environment", {}).get("sdk", "?")
        need = _min_sdk(constraint)
        ok = _tuple(dart_version) >= _tuple(need)
        print(f"  {'OK ' if ok else 'НЕТ'} {name} {version} -> Dart {constraint}")
        if not ok:
            violations.append(f"{name} {version} требует Dart {constraint}, есть {dart_version}")
    except Exception as e:
        print(f"  ??  {name} {version}: проверка недоступна ({e})")
report["pubspec_guard"] = {"dart": dart_version, "violations": violations}
save_report()
if violations:
    raise RuntimeError("Несовместимые зависимости:\n  - " + "\n  - ".join(violations))

stage("pub_get", [FLUTTER_BIN, "pub", "get"], timeout=1200)

# v0.8: штатный автофикс вместо regex-вырезания импортов
stage("dart_fix", [DART, "fix", "--apply"], timeout=900)
stage("format", [DART, "format", "lib", "test"], timeout=600)

# v0.8: строгий анализ — предупреждения = провал этапа (infos не фатальны)
analyze_args = [FLUTTER_BIN, "analyze", "--no-pub"]
if not STRICT_ANALYZE:
    analyze_args += ["--no-fatal-infos", "--no-fatal-warnings"]
else:
    analyze_args += ["--no-fatal-infos"]
stage("analyze", analyze_args, timeout=1200)

stage("tests", [FLUTTER_BIN, "test", "--no-pub", "--reporter", "expanded"], timeout=1200)

apk = APP / f"build/app/outputs/flutter-apk/app-{BUILD_VARIANT}.apk"
apk.unlink(missing_ok=True)
stage("build", [FLUTTER_BIN, "build", "apk", f"--{BUILD_VARIANT}", "--no-pub"], timeout=4800)
if not apk.exists() or apk.stat().st_size < 1024 * 1024:
    raise RuntimeError(f"Новый APK не найден: {apk}")

verify_log = stage("apk_signature",
                   [SDK / "build-tools/35.0.0/apksigner", "verify",
                    "--verbose", "--print-certs", apk], timeout=300)
if "Verifies" not in verify_log:
    raise RuntimeError("APK не прошёл проверку подписи")
if BUILD_VARIANT == "release":
    scheme_v2 = re.search(r"v2 scheme.*?: (true|false)", verify_log)
    if not scheme_v2 or scheme_v2.group(1) != "true":
        raise RuntimeError("Нет подписи v2 scheme — APK непригоден для установки")
digest_match = re.search(r"Signer #1 certificate SHA-256 digest: ([0-9a-f]+)", verify_log)
cert_sha256 = digest_match.group(1) if digest_match else None
report["cert_sha256"] = cert_sha256
print(f"[ПОДПИСЬ] {report['signing'].upper()} · cert SHA-256: {cert_sha256}")

digest = hashlib.sha256(apk.read_bytes()).hexdigest()
destination = Path(f"/content/ssm2_fixed_v09_{BUILD_VARIANT}.apk")
shutil.copy2(apk, destination)
report.update({"apk": str(destination), "sha256": digest, "apk_verified": True})

# --- v0.9: манифест воспроизводимости — sha256 всех исходников проекта ---
manifest = {}
for source in sorted(APP.rglob("*")):
    if not source.is_file():
        continue
    rel = source.relative_to(APP).as_posix()
    if rel.startswith(("build/", ".dart_tool/", "android/.gradle/")) or rel.endswith(".jks"):
        continue
    if source.suffix in {".dart", ".kts", ".gradle", ".yaml", ".yml", ".xml", ".kt", ".py", ".json", ".properties"}:
        manifest[rel] = hashlib.sha256(source.read_bytes()).hexdigest()
(REPORT / "sources_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
report["sources"] = {"files": len(manifest),
                     "manifest": str(REPORT / "sources_manifest.json")}
print(f"[манифест] {len(manifest)} исходников · {REPORT / 'sources_manifest.json'}")
save_report()

print(f"\n[OK] Проверки и сборка завершены: {destination}")
print(f"SHA256 APK: {digest}")

print("\n=== Чеклист стендовой проверки (hardware_tested) ===")
print("  [ ] 1. Адаптер сопряжён, подключение ≤ 10 с")
print("  [ ] 2. Базовый набор PID: качество ≥ 95% за 10 мин")
print("  [ ] 3. Ни одного mute-изоляции адресов за сессию")
print("  [ ] 4. Стоп/старт опроса без переподключения адаптера")
print("  [ ] 5. Расширенный набор активен при вводе своего ROM ID (CALID-гейт)")
print(f"Подтверждаете прохождение? Установите HARDWARE_CONFIRMED=True и пересоберите: "
      f"сейчас report.hardware_tested = {HARDWARE_CONFIRMED}")

# --- Автоматическое скачивание готового APK ---
try:
    from google.colab import files
    print(f"\n[АВТО-СКАЧИВАНИЕ] {destination.name} отправлен на ваш ПК/телефон...")
    files.download(str(destination))
except Exception:
    print(f"\nСкачайте файл из боковой панели «Файлы» Colab: {destination.name}")


=== 0. Очистка предыдущих сборок ===
[Удалён старый APK] ssm2_fixed_v09_release.apk
[Удалён старый отчёт] ssm2_build_20261006_114553
SSM2 0.9.2 | Проверка перед сборкой
[OK] /content/flutter/bin/flutter
[OK] /content/flutter/bin/dart
[OK] /usr/lib/jvm/java-17-openjdk-amd64/bin/java
[OK] /content/android-sdk/platforms/android-36/android.jar
[OK] /content/android-sdk/build-tools/35.0.0/aapt
[OK] /content/android-sdk/build-tools/35.0.0/apksigner
[OK] /content/android-sdk/ndk/27.0.12077973/source.properties
[OK] /content/subaru_ssm2_fixed/lib/native_spp.dart
[OK] /content/subaru_ssm2_fixed/lib/transport_selected.dart
[OK] /content/subaru_ssm2_fixed/lib/analyzer.dart
[OK] /content/subaru_ssm2_fixed/lib/derived.dart
[OK] /content/subaru_ssm2_fixed/lib/identity.dart
[OK] /content/subaru_ssm2_fixed/lib/elm.dart
[OK] /content/subaru_ssm2_fixed/lib/engine.dart
[OK] /content/subaru_ssm2_fixed/lib/pids.dart
[OK] /content/subaru_ssm2_fixed/lib/maplab_link.dart
[OK] /content/subaru_ssm2_fixed/lib/ma

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>